# Pan-cancer atlas: pretrained inference and MI interpretation

Apply the pretrained SpiderNet model to the annotated MERSCOPE Immuno-Oncology atlas.
Infer edge-level activities, export molecular loadings and pseudo-bulk summaries,
and evaluate cell-type pairs, malignant programs, spatial patterns, and CCC baselines.
Additional sections retain tumor-involvement, EcoTyper, gene-expression, and GO/KEGG analyses.

Run this notebook through `run_benchmarks.py --stage spatial`, which saves an executed
copy. See README.md for paper panels, prerequisites, paths, and cached plotting commands.


## Inputs, dependencies, and execution order

- Annotated `.h5ad` files under `DATA_ROOT / ADATA_FOLDER_NAME` (default:
  `adata_entire`) with `obsm["spatial"]`, `obs["subslice_id"]`,
  `obs["celltype_final"]`, and gene symbols in `var_names`. `SampleID` is
  constructed from the file prefix and sub-slice ID during preprocessing.
- A processed bundle in `PROCESSED_DATA_DIR`, including the AnnData objects,
  directed neighboring-cell graphs, batch IDs, LR lists, and training genes.
- A pretrained checkpoint and its matching LR/gene metadata. Reference
  features are searched in the result directory, reference processed-data
  directory, and full processed-data directory, in that order.
- CancerSEA marker text files for the malignant-program sections; the
  EcoTyper workbook files for the optional EcoTyper sections.

Use a Python 3 environment with SpiderNet and its scientific dependencies
(PyTorch/PyG, Scanpy/AnnData, NumPy, pandas, SciPy, Matplotlib, and seaborn).
The mixture-model section uses scikit-learn; GO/KEGG analyses use GSEApy and
Enrichr resources. Optional colormaps use cmcrameri, with a fallback in the code.

`RUN_PREPROCESSING=False` reuses an existing bundle. The setup cell configures all data, result, and reference paths.
`SPIDERNET_PANCANCER_RESULTS`, `SPIDERNET_PANCANCER_DATA`, and
`SPIDERNET_PANCANCER_OUTPUT` may override their default locations. Large processed objects and edge-level tables require
substantial memory. Downstream sections reuse variables and outputs from earlier
cells, so headings do not imply that every section is standalone.


## Dataset and reference-model configuration

Set the atlas, output, processed-bundle, and reference-checkpoint locations.
`DIM_ENVIR` must match the checkpoint; MI columns retain their reference order.
The default reference is the 11-MI, 40-sub-slice training run. Model training is
implemented in the separate `Pancancer_modeltraining` notebooks.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np

# ---------------------------------------------------------------------
# Dataset and output paths
# ---------------------------------------------------------------------
# Notebook execution uses this folder as its working directory.
TUTORIAL_DIR = Path.cwd().resolve()
_root_candidates = (TUTORIAL_DIR, *TUTORIAL_DIR.parents)
_default_root = next(
    (p for p in _root_candidates if (p / "Data").is_dir() and (p / "Results").is_dir()),
    TUTORIAL_DIR,
)
SPIDERNET_ROOT = Path(os.environ.get("SPIDERNET_ROOT", _default_root))
DATA_ROOT = Path(os.environ.get("SPIDERNET_PANCANCER_DATA", SPIDERNET_ROOT / "Data" / "Pancancer"))
RESULTS_DIR = Path(os.environ.get("SPIDERNET_PANCANCER_RESULTS", SPIDERNET_ROOT / "Results" / "Pancancer" / "V1" / "SpiderNet_Result_dim11"))
OUTPUT_ROOT = RESULTS_DIR.parent.parent
PANCANCER_OUTPUT = Path(os.environ.get("SPIDERNET_PANCANCER_OUTPUT", TUTORIAL_DIR / "output"))
PANCANCER_OUTPUT.mkdir(parents=True, exist_ok=True)

# The unified dataloader writes processed objects here.
# For the full pan-cancer inference dataset, keep this outside the model-training run directory.
PROCESSED_DATA_DIR = Path(os.environ.get("SPIDERNET_PANCANCER_PROCESSED", OUTPUT_ROOT / "ProcessedData_entire"))

# Folder under DATA_ROOT that stores the .h5ad files.
ADATA_FOLDER_NAME = "adata_entire"

# ---------------------------------------------------------------------
# Pancancer-specific AnnData fields
# ---------------------------------------------------------------------
SPECIES = "human"               # "human" or "mouse"
SAMPLE_COL = "SampleID"         # Constructed as <file_prefix>_subslice<subslice_id>
CELL_CLASS_COL = "celltype_final"
SPATIAL_KEY = "spatial"
PYG_EXTRA_OBS_FIELDS = {}

# Backward-compatible aliases used by some checks and downstream cells.
SAMPLE_ID_COL = SAMPLE_COL
CELL_TYPE_COL = CELL_CLASS_COL

# ---------------------------------------------------------------------
# Preprocessing parameters
# ---------------------------------------------------------------------
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 5

# Used only when no predefined LR list is supplied.
LR_CORR_THRESHOLD = 0.2

# Set this to False if PROCESSED_DATA_DIR already contains a complete processed bundle.
RUN_PREPROCESSING = False

# ---------------------------------------------------------------------
# Model/inference parameters
# ---------------------------------------------------------------------
DIM_ENVIR = 11
N_JOBS = 5
MAX_EPOCH = 20000
VERSION = "V1"

# Reference result directory from the subsampled 40-slice training run.
# This directory stores the trained model and inference outputs.
REFERENCE_RESULTS_DIR = RESULTS_DIR

# Reference processed-data directory from Pancancer_modeltraining.
# The unified pipeline saves LR_list.pkl and genenames_train.pkl here,
# not necessarily under REFERENCE_RESULTS_DIR.
REFERENCE_PROCESSED_DATA_DIR = OUTPUT_ROOT / "ProcessedData"

# Trained model checkpoint. Set to a specific file, or leave as None to auto-pick
# the latest model_epoch* checkpoint under REFERENCE_RESULTS_DIR / "Model".
TRAINED_MODEL_PATH = None

# Reuse the exact LR list and gene names saved in the reference training results.
USE_REFERENCE_LR_LIST = True
USE_REFERENCE_GENENAMES = True
REFERENCE_LR_LIST_FILENAME = "LR_list.pkl"
REFERENCE_GENENAME_CANDIDATES = ["genenames_train.pkl", "genenames.pkl"]

# ---------------------------------------------------------------------
# Save the high-level user-facing config.
# ---------------------------------------------------------------------
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "DATA_ROOT": str(DATA_ROOT),
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "PROCESSED_DATA_DIR": str(PROCESSED_DATA_DIR),
    "ADATA_FOLDER_NAME": ADATA_FOLDER_NAME,
    "SPECIES": SPECIES,
    "SAMPLE_COL": SAMPLE_COL,
    "CELL_CLASS_COL": CELL_CLASS_COL,
    "SPATIAL_KEY": SPATIAL_KEY,
    "N_HVG": N_HVG,
    "N_HVG_LR": N_HVG_LR,
    "NUM_NEIGHBORS": NUM_NEIGHBORS,
    "LR_CORR_THRESHOLD": LR_CORR_THRESHOLD,
    "RUN_PREPROCESSING": RUN_PREPROCESSING,
    "DIM_ENVIR": DIM_ENVIR,
    "N_JOBS": N_JOBS,
    "MAX_EPOCH": MAX_EPOCH,
    "VERSION": VERSION,
    "REFERENCE_RESULTS_DIR": str(REFERENCE_RESULTS_DIR),
    "REFERENCE_PROCESSED_DATA_DIR": str(REFERENCE_PROCESSED_DATA_DIR),
    "TRAINED_MODEL_PATH": str(TRAINED_MODEL_PATH) if TRAINED_MODEL_PATH is not None else None,
    "USE_REFERENCE_LR_LIST": USE_REFERENCE_LR_LIST,
    "USE_REFERENCE_GENENAMES": USE_REFERENCE_GENENAMES,
}

config_path = OUTPUT_ROOT / "Pancancer_analysis_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to: {config_path}")


## Imports and device setup

Load SpiderNet utilities, select CPU or CUDA, and define memory-release helpers.


In [ ]:
import gc
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from SpiderNet.utils import *
from SpiderNet.config import *
from SpiderNet.dataloading_unified import (
    prepare_processed_bundle_unified,
    preview_lr_corr_distribution,
)
from SpiderNet.utils import get_default_cellchat_db, get_default_scseqcomm_db

cuda_available = torch.cuda.is_available()
device = "cuda" if cuda_available else "cpu"
print(f"Using device: {device}")


def release_memory(*var_names, namespace=None, run_gc=True, clear_cuda=True, close_figures=False):
    """
    Delete variables from the provided namespace and optionally trigger
    Python garbage collection / CUDA cache cleanup.
    """
    if namespace is None:
        namespace = globals()

    for name in var_names:
        if name in namespace:
            try:
                del namespace[name]
            except Exception:
                namespace.pop(name, None)

    if close_figures:
        try:
            import matplotlib.pyplot as plt
            plt.close("all")
        except Exception:
            pass

    if run_gc:
        gc.collect()

    if clear_cuda and torch.cuda.is_available():
        torch.cuda.empty_cache()


## Resolve paths and reference features

Build output paths, resolve the reference checkpoint and feature files, and configure the pan-cancer AnnData hook.


In [ ]:
paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)



In [ ]:
CELLCHAT_DB = get_default_cellchat_db(species=SPECIES)
SCSEQCOMM_DB = get_default_scseqcomm_db(species=SPECIES)


def normalize_optional_path_local(path_str):
    if path_str is None:
        return None
    path_str = str(path_str).strip()
    if path_str.lower() in {"", "none", "null", "nan"}:
        return None
    return path_str


def resolve_existing_reference_file(search_dirs, candidates, label, required=True):
    """
    Resolve a reference file from several possible locations.

    The unified loader saves processed objects such as LR_list.pkl and
    genenames_train.pkl under PROCESSED_DATA_DIR / REFERENCE_PROCESSED_DATA_DIR,
    while the model checkpoint is usually under REFERENCE_RESULTS_DIR / Model.
    """
    if isinstance(candidates, (str, Path)):
        candidates = [candidates]

    checked = []
    for directory in search_dirs:
        directory = Path(directory)
        for candidate in candidates:
            candidate_path = directory / candidate
            checked.append(candidate_path)
            if candidate_path.exists():
                print(f"Resolved {label}: {candidate_path}")
                return candidate_path

    if required:
        checked_text = "\n  - " + "\n  - ".join(str(p) for p in checked)
        raise FileNotFoundError(
            f"Cannot find {label}. Checked:{checked_text}\n\n"
            "For Pancancer_analysis, the model is loaded from REFERENCE_RESULTS_DIR, "
            "but the matched LR/gene lists usually come from "
            "REFERENCE_PROCESSED_DATA_DIR, e.g. "
            "Results/Pancancer/ProcessedData/LR_list.pkl."
        )
    return None


def resolve_model_checkpoint(reference_results_dir, explicit_model_path=None):
    if explicit_model_path is not None:
        explicit_model_path = Path(explicit_model_path)
        if not explicit_model_path.exists():
            raise FileNotFoundError(f"Model checkpoint does not exist: {explicit_model_path}")
        return explicit_model_path

    search_dirs = []
    if (reference_results_dir / "Model").exists():
        search_dirs.append(reference_results_dir / "Model")
    search_dirs.append(reference_results_dir)

    candidates = []
    for search_dir in search_dirs:
        for pattern in ["model_epoch*", "*checkpoint*", "*.pt", "*.pth", "*.pkl"]:
            candidates.extend(search_dir.glob(pattern))

    candidates = [p for p in candidates if p.is_file()]
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Cannot find a trained model checkpoint under "
            f"{reference_results_dir} or {reference_results_dir / 'Model'}"
        )

    def extract_epoch(path):
        match = re.search(r"epoch(\d+)", path.stem)
        return int(match.group(1)) if match else -1

    candidates = sorted(
        candidates,
        key=lambda p: (extract_epoch(p), p.stat().st_mtime)
    )
    return candidates[-1]


# Search order:
# 1. REFERENCE_RESULTS_DIR: allows backward compatibility if files were exported there.
# 2. REFERENCE_PROCESSED_DATA_DIR: training-bundle metadata.
# 3. PROCESSED_DATA_DIR: fallback if the full analysis bundle was already generated.
REFERENCE_FEATURE_SEARCH_DIRS = [
    REFERENCE_RESULTS_DIR,
    REFERENCE_PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR,
]

reference_lr_list_path = (
    resolve_existing_reference_file(
        REFERENCE_FEATURE_SEARCH_DIRS,
        REFERENCE_LR_LIST_FILENAME,
        label="reference LR list",
        required=True,
    )
    if USE_REFERENCE_LR_LIST else None
)

reference_genename_path = (
    resolve_existing_reference_file(
        REFERENCE_FEATURE_SEARCH_DIRS,
        REFERENCE_GENENAME_CANDIDATES,
        label="reference gene-name file",
        required=True,
    )
    if USE_REFERENCE_GENENAMES else None
)

# The pretrained model's LR loading matrix must be interpreted with the same
# LR metadata used during Pancancer_modeltraining. These files should come from
# the reference training processed-data directory, not from the full inference
# processed-data directory if the latter was generated independently.
reference_lr_list_cellchatdb_path = resolve_existing_reference_file(
    REFERENCE_FEATURE_SEARCH_DIRS,
    "LR_list_cellchatdb.pkl",
    label="reference CellChat LR list",
    required=True,
)

reference_lr_meta_cellchatdb_path = resolve_existing_reference_file(
    REFERENCE_FEATURE_SEARCH_DIRS,
    "LR_meta_cellchatdb.pkl",
    label="reference CellChat LR metadata",
    required=True,
)

reference_model_path = resolve_model_checkpoint(
    REFERENCE_RESULTS_DIR,
    TRAINED_MODEL_PATH
)

LR_LIST_PATH = reference_lr_list_path
GENE_LIST_PATH = reference_genename_path


def pancancer_file_prefix(context):
    return Path(context["file_name"]).stem.split("_")[0]


def pancancer_per_file_hook(adata, context):
    """Match the original Pancancer loader while keeping raw .h5ad files unchanged."""
    sample_prefix = pancancer_file_prefix(context)

    if sp.issparse(adata.X):
        adata.X = adata.X.astype(np.float32)
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    adata.obs_names = [f"{sample_prefix}_{cellname}" for cellname in adata.obs_names]

    if "subslice_id" not in adata.obs.columns:
        raise KeyError(
            "Pancancer preprocessing expects adata.obs['subslice_id'] so that "
            "SampleID can be constructed as <file_prefix>_subslice<subslice_id>."
        )

    adata.obs["source_file_prefix"] = sample_prefix
    adata.obs[SAMPLE_COL] = [
        f"{sample_prefix}_subslice{subslice_id}"
        for subslice_id in adata.obs["subslice_id"].astype(str)
    ]

    if SPATIAL_KEY not in adata.obsm:
        raise KeyError(
            f"Pancancer preprocessing expects adata.obsm[{SPATIAL_KEY!r}] "
            "to contain spatial coordinates."
        )

    # Keep only the fields needed for SpiderNet preprocessing.
    adata.layers.clear()
    for key in ["X_pca", "X_umap"]:
        if key in adata.obsm:
            del adata.obsm[key]
    adata.obsm = {"spatial": np.asarray(adata.obsm[SPATIAL_KEY])}
    adata.varm.clear()
    adata.raw = None
    if adata.var.shape[1] > 0:
        adata.var = adata.var.iloc[:, []].copy()

    return adata


APPLY_HVG_SELECTION = normalize_optional_path_local(GENE_LIST_PATH) is None
APPLY_LR_CORR_FILTER = normalize_optional_path_local(LR_LIST_PATH) is None

DATA_REPRESENTATION_CONFIG = {
    "expression_source": {"kind": "X", "name": None},
    "normalize_strategy": "auto",
    "log1p": True,
    "remove_zero_count_cells": True,
    "spatial_source": {"kind": "obsm", "key": "spatial"},
    "apply_hvg_selection": APPLY_HVG_SELECTION,
}

base_config = {
    "data_path_main": DATA_ROOT,
    "output_dir": PROCESSED_DATA_DIR,
    "adata_folder_name": ADATA_FOLDER_NAME,
    "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
    "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,
    "sample_col": SAMPLE_COL,
    "sample_name_obs_col": "source_file_prefix",
    "sample_attr_obs_col": SAMPLE_COL,
    "cell_class_col": CELL_CLASS_COL,
    "pyg_obs_fields": PYG_EXTRA_OBS_FIELDS,
    "n_hvg": N_HVG,
    "n_hvg_lr": N_HVG_LR,
    "num_neighbors": NUM_NEIGHBORS,
    "per_file_hook": pancancer_per_file_hook,
    "apply_lr_corr_filter": APPLY_LR_CORR_FILTER,
    "skip_lr_filter_when_predefined": True,
    **DATA_REPRESENTATION_CONFIG,
}

lr_list_path_use = normalize_optional_path_local(LR_LIST_PATH)
gene_list_path_use = normalize_optional_path_local(GENE_LIST_PATH)

if lr_list_path_use is not None:
    base_config["lr_list_path"] = lr_list_path_use

if gene_list_path_use is not None:
    base_config["gene_list_path"] = gene_list_path_use

display(pd.DataFrame([
    {
        "item": "REFERENCE_RESULTS_DIR",
        "value": str(REFERENCE_RESULTS_DIR),
        "ok": REFERENCE_RESULTS_DIR.exists(),
    },
    {
        "item": "REFERENCE_PROCESSED_DATA_DIR",
        "value": str(REFERENCE_PROCESSED_DATA_DIR),
        "ok": REFERENCE_PROCESSED_DATA_DIR.exists(),
    },
    {
        "item": "reference_lr_list_path",
        "value": str(reference_lr_list_path) if reference_lr_list_path is not None else "None",
        "ok": True if reference_lr_list_path is None else reference_lr_list_path.exists(),
    },
    {
        "item": "reference_genename_path",
        "value": str(reference_genename_path) if reference_genename_path is not None else "None",
        "ok": True if reference_genename_path is None else reference_genename_path.exists(),
    },
    {
        "item": "reference_lr_list_cellchatdb_path",
        "value": str(reference_lr_list_cellchatdb_path),
        "ok": reference_lr_list_cellchatdb_path.exists(),
    },
    {
        "item": "reference_lr_meta_cellchatdb_path",
        "value": str(reference_lr_meta_cellchatdb_path),
        "ok": reference_lr_meta_cellchatdb_path.exists(),
    },
    {
        "item": "reference_model_path",
        "value": str(reference_model_path),
        "ok": reference_model_path.exists(),
    },
    {
        "item": "processed_data_dir",
        "value": str(PROCESSED_DATA_DIR),
        "ok": PROCESSED_DATA_DIR.exists(),
    },
]))



### Validate files and AnnData fields

Check configured paths and inspect the expected metadata and spatial coordinates in one input file.


In [ ]:
adata_dir = Path(DATA_ROOT) / ADATA_FOLDER_NAME
sample_files = sorted(adata_dir.glob("*.h5ad"))
sample_path = sample_files[0] if sample_files else None

rows = [
    {"item": "DATA_ROOT", "value": str(DATA_ROOT), "ok": Path(DATA_ROOT).exists()},
    {"item": "OUTPUT_ROOT", "value": str(OUTPUT_ROOT), "ok": Path(OUTPUT_ROOT).exists()},
    {"item": "PROCESSED_DATA_DIR", "value": str(PROCESSED_DATA_DIR), "ok": Path(PROCESSED_DATA_DIR).exists()},
    {"item": "CellChat DB", "value": str(CELLCHAT_DB), "ok": Path(CELLCHAT_DB).exists()},
    {"item": "scSeqComm DB", "value": str(SCSEQCOMM_DB), "ok": Path(SCSEQCOMM_DB).exists()},
    {"item": "adata folder", "value": str(adata_dir), "ok": adata_dir.exists()},
    {"item": "number of .h5ad files", "value": len(sample_files), "ok": len(sample_files) > 0},
    {"item": "REFERENCE_RESULTS_DIR", "value": str(REFERENCE_RESULTS_DIR), "ok": REFERENCE_RESULTS_DIR.exists()},
    {"item": "REFERENCE_PROCESSED_DATA_DIR", "value": str(REFERENCE_PROCESSED_DATA_DIR), "ok": REFERENCE_PROCESSED_DATA_DIR.exists()},
    {"item": "reference_lr_list_path", "value": str(reference_lr_list_path), "ok": reference_lr_list_path is None or reference_lr_list_path.exists()},
    {"item": "reference_genename_path", "value": str(reference_genename_path), "ok": reference_genename_path is None or reference_genename_path.exists()},
    {"item": "reference_lr_list_cellchatdb_path", "value": str(reference_lr_list_cellchatdb_path), "ok": reference_lr_list_cellchatdb_path.exists()},
    {"item": "reference_lr_meta_cellchatdb_path", "value": str(reference_lr_meta_cellchatdb_path), "ok": reference_lr_meta_cellchatdb_path.exists()},
    {"item": "reference_model_path", "value": str(reference_model_path), "ok": reference_model_path.exists()},
]

if sample_path is not None:
    adata_example = sc.read_h5ad(sample_path, backed="r")
    rows.extend([
        {"item": "example file", "value": sample_path.name, "ok": True},
        {"item": "obs['subslice_id']", "value": "subslice_id", "ok": "subslice_id" in adata_example.obs.columns},
        {"item": f"obs['{CELL_CLASS_COL}']", "value": CELL_CLASS_COL, "ok": CELL_CLASS_COL in adata_example.obs.columns},
        {"item": f"obsm['{SPATIAL_KEY}']", "value": SPATIAL_KEY, "ok": SPATIAL_KEY in adata_example.obsm_keys()},
        {"item": f"obs['{SAMPLE_COL}']", "value": "created during preprocessing", "ok": True},
    ])
    adata_example.file.close()
else:
    rows.append({"item": "example file", "value": "No .h5ad file found", "ok": False})

display(pd.DataFrame(rows))

# Release preview-only objects from this cell.
release_memory(
    "adata_example",
    "rows",
    "sample_files",
    "sample_path",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


## Configure preprocessing and checkpoint-compatible inference

Construct the configuration objects and result directories used by SpiderNet.


In [ ]:
preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS,
)

train_cfg = TrainingConfig(
    dim_envir=DIM_ENVIR,
    n_jobs=N_JOBS,
    max_epoch=MAX_EPOCH,
    version=VERSION,
)

run_dirs = paths.ensure_dirs(
    dim_envir=train_cfg.dim_envir,
)

run_dirs["run_dir"] = RESULTS_DIR
run_dirs["model_dir"] = RESULTS_DIR / "Model"
for _directory in run_dirs.values():
    Path(_directory).mkdir(parents=True, exist_ok=True)

processed_data_dir = Path(PROCESSED_DATA_DIR)
processed_data_dir.mkdir(parents=True, exist_ok=True)

print("Processed data directory:", processed_data_dir)
print("Model/result run directories:", run_dirs)

with open(paths.output_root / "run_dirs.json", "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in run_dirs.items()}, handle, indent=2)

with open(paths.output_root / "processed_data_dir.json", "w", encoding="utf-8") as handle:
    json.dump({"processed_data_dir": str(processed_data_dir)}, handle, indent=2)


In [ ]:
dataloading_config_preview = dict(base_config)
if APPLY_LR_CORR_FILTER and LR_CORR_THRESHOLD is not None:
    dataloading_config_preview["lr_corr_threshold"] = float(LR_CORR_THRESHOLD)

def json_safe_for_export(value):
    if callable(value):
        return getattr(value, "__name__", str(value))
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): json_safe_for_export(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe_for_export(v) for v in value]
    return value

with open(paths.output_root / "Pancancer_analysis_dataloading_config.json", "w", encoding="utf-8") as handle:
    json.dump(json_safe_for_export(dataloading_config_preview), handle, indent=2)


# Release LR-correlation preview objects after inspection.
# `base_config` is kept because it is used by the preprocessing cell.
release_memory(
    "dataloading_config_preview",
    "preview",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
    close_figures=True,
)


## Prepare or reuse the processed atlas

`prepare_processed_bundle_unified` applies the pan-cancer preprocessing hook
when preprocessing is enabled. Processed objects are stored in
`PROCESSED_DATA_DIR`; model and inference results use the separate run directory.


In [ ]:
if RUN_PREPROCESSING:
    dataloading_config = dict(base_config)

    if APPLY_LR_CORR_FILTER:
        if LR_CORR_THRESHOLD is None:
            preview = preview_lr_corr_distribution(base_config, show_plot=True)
            if preview.get("plot_path") is not None:
                print("LR-correlation preview saved to:", preview["plot_path"])
            raise ValueError(
                "LR_CORR_THRESHOLD is None. Inspect the preview plot, set LR_CORR_THRESHOLD, "
                "and rerun this cell."
            )
        dataloading_config["lr_corr_threshold"] = float(LR_CORR_THRESHOLD)

    bundle = prepare_processed_bundle_unified(dataloading_config)

    # Save one row per processed Pancancer batch. This file is used by optional downstream summaries.
    obs = bundle["adata"].obs.copy()
    metadata_sample = (
        obs.groupby(SAMPLE_COL, dropna=False)
          .agg(
              source_file_prefix=("source_file_prefix", "first"),
              subslice_id=("subslice_id", "first"),
              num_cells=(SAMPLE_COL, "size"),
              n_cell_types=(CELL_CLASS_COL, pd.Series.nunique),
          )
          .reset_index()
    )

    celltype_counts = pd.crosstab(obs[SAMPLE_COL], obs[CELL_CLASS_COL]).reset_index()
    celltype_counts.columns = [SAMPLE_COL] + [f"n_{col}" for col in celltype_counts.columns[1:]]

    metadata_sample = metadata_sample.merge(celltype_counts, on=SAMPLE_COL, how="left")
    metadata_sample = metadata_sample.sort_values(
        ["source_file_prefix", "subslice_id", SAMPLE_COL]
    ).reset_index(drop=True)

    metadata_sample_path = processed_data_dir / "metadata_sample.csv"
    metadata_sample.to_csv(metadata_sample_path, index=False)

    print("Processed bundle saved to:", bundle["output_dir"])
    print("Metadata saved to:", metadata_sample_path)
    display(metadata_sample.head())
else:
    print("Skipping preprocessing.")
    print("Using existing processed data directory:", processed_data_dir)

# Release preprocessing-only intermediates. The processed bundle is reloaded below
# through `load_processed_data`, so keeping `bundle` would duplicate memory.
release_memory(
    "bundle",
    "dataloading_config",
    "metadata_sample",
    "metadata_sample_path",
    "obs",
    "celltype_counts",
    "preview",
    namespace=globals(),
    run_gc=True,
    clear_cuda=True,
    close_figures=True,
)


### Load the processed bundle and check reference alignment

Load the AnnData and graph objects, verify training-gene order, and compare reference LR metadata with the processed bundle.


In [ ]:
from SpiderNet.io import load_processed_data, get_spidernet_pyg_list_path, load_spidernet_pyg_list, spidernet_pyg_list_exists

required_processed_files = [
    "adata_all.h5ad",
    "adata_list.pkl",
    "LR_list.pkl",
    "genenames_train.pkl",
]

missing_processed_files = [
    f for f in required_processed_files
    if not (processed_data_dir / f).exists()
]

if missing_processed_files or not spidernet_pyg_list_exists(processed_data_dir):
    raise FileNotFoundError(
        f"ProcessedData is incomplete. Missing files under {processed_data_dir}: "
        f"{missing_processed_files}. Run the unified preprocessing cell above first."
    )

processed = load_processed_data(processed_data_dir)

reference_lr_list = pd.read_pickle(reference_lr_list_path)
reference_genenames = pd.read_pickle(reference_genename_path)
reference_genenames_array = np.asarray(reference_genenames).astype(str)
processed_genenames_array = np.asarray(processed.genenames_train).astype(str)

print("Processed data directory:", processed_data_dir)
print("Number of batches:", len(processed.spidernet_data))
print("Number of LR pairs in full processed data:", len(processed.lr_list))
print("Number of LR pairs in reference training run:", len(reference_lr_list))
print("Number of training genes in full processed data:", processed.genenames_train.shape[0])
print("Number of training genes in reference training run:", reference_genenames_array.shape[0])

if processed.genenames_train.shape[0] != reference_genenames_array.shape[0]:
    raise ValueError(
        "The full processed data and the reference training model use different numbers "
        f"of genes: full={processed.genenames_train.shape[0]}, "
        f"reference={reference_genenames_array.shape[0]}. "
        "Rerun the full-data preprocessing with GENE_LIST_PATH set to the reference "
        "genenames_train.pkl."
    )

if not np.array_equal(processed_genenames_array, reference_genenames_array):
    raise ValueError(
        "The full processed data gene order does not match the reference training gene order. "
        "Rerun the full-data preprocessing with GENE_LIST_PATH set to the reference "
        "genenames_train.pkl. Matching gene order is required before transferring a trained model."
    )

if len(processed.lr_list) != len(reference_lr_list):
    print(
        "WARNING: The full processed data LR list length differs from the reference training LR list "
        f"({len(processed.lr_list)} vs {len(reference_lr_list)}). "
        "The pretrained model will be initialized from the reference checkpoint dimensions, "
        "so LR loading interpretation will use the reference LR metadata. "
        "For the cleanest full-data transfer, rerun preprocessing with RUN_PREPROCESSING=True "
        "so the full processed bundle is regenerated from the reference LR_list.pkl."
    )

# Release validation-only arrays; keep `processed` and reference paths.
release_memory(
    "processed_genenames_array",
    "reference_genenames",
    "reference_genenames_array",
    "required_processed_files",
    "missing_processed_files",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


In [ ]:
# ============================================================
# Unique cell-type label union across all tissues / cancer types
# ============================================================

celltype_union_records = []

for i, adata_i in enumerate(processed.adata_list):
    if CELL_CLASS_COL in adata_i.obs.columns:
        celltype_values = adata_i.obs[CELL_CLASS_COL].astype(str).values
    else:
        celltype_values = np.asarray(processed.spidernet_data[i]["cell_class"]).astype(str)

    if SAMPLE_COL in adata_i.obs.columns:
        sample_ids = adata_i.obs[SAMPLE_COL].astype(str).unique()
        sample_id = sample_ids[0] if len(sample_ids) == 1 else ";".join(sample_ids)
    else:
        sample_id = str(np.asarray(processed.spidernet_data[i]["sample"]).astype(str)[0])

    cancer_type = sample_id.split("_")[0]

    for ct in np.unique(celltype_values):
        celltype_union_records.append({
            "batch_index": i,
            "SampleID": sample_id,
            "CancerType": cancer_type,
            "cell_type_label": ct,
        })

celltype_union_df = (
    pd.DataFrame(celltype_union_records)
    .drop_duplicates()
    .sort_values(["CancerType", "cell_type_label", "SampleID"])
    .reset_index(drop=True)
)

celltype_label_union = sorted(celltype_union_df["cell_type_label"].unique())

print(f"Number of unique cell-type labels across all tissues: {len(celltype_label_union)}")
print("Union of unique cell-type labels:")
for ct in celltype_label_union:
    print(f"- {ct}")

display(
    pd.DataFrame({
        "cell_type_label_union": celltype_label_union
    })
)

# Optional: show which cancer types contain each cell-type label
celltype_by_cancer_df = (
    celltype_union_df
    .groupby("cell_type_label", as_index=False)
    .agg(
        n_cancer_types=("CancerType", "nunique"),
        CancerTypes=("CancerType", lambda x: "; ".join(sorted(pd.unique(x)))),
        n_batches=("SampleID", "nunique"),
    )
    .sort_values(["cell_type_label"])
    .reset_index(drop=True)
)

display(celltype_by_cancer_df)

### Validate the processed batches

Check that the bundle is nonempty and that graph cell indices agree with the AnnData cell counts before inference.


In [ ]:
summary_rows = []
for i in range(len(processed.adata_list)):
    adata_i = processed.adata_list[i]
    edge_index_max = torch.max(processed.spidernet_data[i]["edge_index"]).cpu().item()
    summary_rows.append(
        {
            "batch_index": i,
            "n_cells": adata_i.n_obs,
            "n_genes": adata_i.n_vars,
            "edge_index_max": edge_index_max,
            "edge_index_matches_n_cells": adata_i.n_obs == (edge_index_max + 1),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

issues = []
if len(processed.spidernet_data) == 0:
    issues.append("No processed batches were loaded.")
if len(processed.lr_list) == 0:
    issues.append("No ligand-receptor pairs were retained.")
if processed.genenames_train.shape[0] == 0:
    issues.append("No training genes were retained.")
if not summary_df["edge_index_matches_n_cells"].all():
    issues.append("At least one batch has an edge index / cell-count mismatch.")

if issues:
    print("Sanity check flagged the following issues:")
    for issue in issues:
        print("-", issue)
else:
    print("Sanity checks passed. The processed data appear internally consistent.")

# Release summary-table construction objects after display.
release_memory(
    "summary_rows",
    "summary_df",
    "issue",
    "issues",
    "adata_i",
    "edge_index_max",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


In [ ]:
# ============================================================
# Cell count summary: total and per sub-slice
# ============================================================

cell_count_rows = []

for i, adata_i in enumerate(processed.adata_list):
    # Get SampleID if available
    if SAMPLE_COL in adata_i.obs.columns:
        sample_ids = adata_i.obs[SAMPLE_COL].astype(str).unique()
        sample_id = sample_ids[0] if len(sample_ids) == 1 else ";".join(sample_ids)
    else:
        sample_id = str(np.asarray(processed.spidernet_data[i]["sample"]).astype(str)[0])

    cell_count_rows.append({
        "batch_index": i,
        "SampleID": sample_id,
        "CancerType": sample_id.split("_")[0],
        "n_cells": int(adata_i.n_obs),
        "n_genes": int(adata_i.n_vars),
    })

cell_count_df = pd.DataFrame(cell_count_rows)

print(f"Total number of cells: {cell_count_df['n_cells'].sum():,}")
print(f"Number of sub-slices / batches: {cell_count_df.shape[0]:,}")
print(f"Mean cells per sub-slice: {cell_count_df['n_cells'].mean():,.1f}")
print(f"Median cells per sub-slice: {cell_count_df['n_cells'].median():,.1f}")
print(f"Min cells per sub-slice: {cell_count_df['n_cells'].min():,}")
print(f"Max cells per sub-slice: {cell_count_df['n_cells'].max():,}")

display(cell_count_df)

# Optional: summarize by cancer type
cell_count_by_cancer = (
    cell_count_df
    .groupby("CancerType", as_index=False)
    .agg(
        n_slices=("SampleID", "nunique"),
        total_cells=("n_cells", "sum"),
        mean_cells_per_slice=("n_cells", "mean"),
        median_cells_per_slice=("n_cells", "median"),
        min_cells_per_slice=("n_cells", "min"),
        max_cells_per_slice=("n_cells", "max"),
    )
)

display(cell_count_by_cancer)

In [ ]:
# ============================================================
# Raw cell count summary: total and per original sub-slice
# This is BEFORE SpiderNet preprocessing / filtering
# ============================================================

raw_adata_dir = Path(DATA_ROOT) / ADATA_FOLDER_NAME
raw_h5ad_files = sorted(raw_adata_dir.glob("*.h5ad"))

if len(raw_h5ad_files) == 0:
    raise FileNotFoundError(f"No .h5ad files found under: {raw_adata_dir}")

raw_cell_count_rows = []

for file_index, h5ad_path in enumerate(raw_h5ad_files):
    print(f"Reading raw file {file_index + 1}/{len(raw_h5ad_files)}: {h5ad_path.name}")

    adata_raw = sc.read_h5ad(h5ad_path, backed="r")

    try:
        source_file_prefix = h5ad_path.stem.split("_")[0]
        obs_raw = adata_raw.obs.copy()

        # If raw data has subslice_id, summarize per original sub-slice.
        if "subslice_id" in obs_raw.columns:
            tmp_df = (
                obs_raw
                .groupby("subslice_id", dropna=False)
                .size()
                .reset_index(name="raw_n_cells")
            )

            tmp_df["source_file"] = h5ad_path.name
            tmp_df["source_file_prefix"] = source_file_prefix
            tmp_df[SAMPLE_COL] = (
                tmp_df["source_file_prefix"].astype(str)
                + "_subslice"
                + tmp_df["subslice_id"].astype(str)
            )
            tmp_df["CancerType"] = tmp_df["source_file_prefix"].astype(str)

            raw_cell_count_rows.append(
                tmp_df[
                    [
                        "source_file",
                        "source_file_prefix",
                        "subslice_id",
                        SAMPLE_COL,
                        "CancerType",
                        "raw_n_cells",
                    ]
                ]
            )

        # Fallback: if no subslice_id exists, summarize per file.
        else:
            raw_cell_count_rows.append(
                pd.DataFrame(
                    {
                        "source_file": [h5ad_path.name],
                        "source_file_prefix": [source_file_prefix],
                        "subslice_id": [np.nan],
                        SAMPLE_COL: [source_file_prefix],
                        "CancerType": [source_file_prefix],
                        "raw_n_cells": [int(adata_raw.n_obs)],
                    }
                )
            )

    finally:
        adata_raw.file.close()

raw_cell_count_df = pd.concat(raw_cell_count_rows, axis=0, ignore_index=True)

raw_cell_count_df = raw_cell_count_df.sort_values(
    ["source_file_prefix", "subslice_id", SAMPLE_COL],
    na_position="last"
).reset_index(drop=True)

print("\nRaw data cell-count summary")
print("=" * 60)
print(f"Raw total number of cells: {raw_cell_count_df['raw_n_cells'].sum():,}")
print(f"Number of raw .h5ad files: {len(raw_h5ad_files):,}")
print(f"Number of raw sub-slices: {raw_cell_count_df.shape[0]:,}")
print(f"Mean raw cells per sub-slice: {raw_cell_count_df['raw_n_cells'].mean():,.1f}")
print(f"Median raw cells per sub-slice: {raw_cell_count_df['raw_n_cells'].median():,.1f}")
print(f"Min raw cells per sub-slice: {raw_cell_count_df['raw_n_cells'].min():,}")
print(f"Max raw cells per sub-slice: {raw_cell_count_df['raw_n_cells'].max():,}")

display(raw_cell_count_df)

raw_cell_count_by_cancer = (
    raw_cell_count_df
    .groupby("CancerType", as_index=False)
    .agg(
        raw_n_slices=(SAMPLE_COL, "nunique"),
        raw_total_cells=("raw_n_cells", "sum"),
        raw_mean_cells_per_slice=("raw_n_cells", "mean"),
        raw_median_cells_per_slice=("raw_n_cells", "median"),
        raw_min_cells_per_slice=("raw_n_cells", "min"),
        raw_max_cells_per_slice=("raw_n_cells", "max"),
    )
)

display(raw_cell_count_by_cancer)

# Optional: save summaries
raw_cell_count_path = OUTPUT_ROOT / "raw_cell_count_per_subslice.csv"
raw_cell_count_by_cancer_path = OUTPUT_ROOT / "raw_cell_count_by_cancertype.csv"

raw_cell_count_df.to_csv(raw_cell_count_path, index=False)
raw_cell_count_by_cancer.to_csv(raw_cell_count_by_cancer_path, index=False)

print(f"Saved raw per-sub-slice cell counts to: {raw_cell_count_path}")
print(f"Saved raw cancer-type cell-count summary to: {raw_cell_count_by_cancer_path}")

## Build the model from the reference checkpoint

Read checkpoint dimensions before constructing the model. Check that the gene,
cell-class, LR, and MI dimensions are compatible with the processed atlas and
reference metadata. This section restores a trained model; it does not optimize
model parameters.


In [ ]:
from SpiderNet.model import SpiderNet_model


def extract_state_dict_from_checkpoint(checkpoint_obj):
    """Return a plain state_dict from common SpiderNet checkpoint formats."""
    if isinstance(checkpoint_obj, dict) and "model_state_dict" in checkpoint_obj:
        return checkpoint_obj["model_state_dict"]
    if isinstance(checkpoint_obj, dict) and "state_dict" in checkpoint_obj:
        return checkpoint_obj["state_dict"]
    if isinstance(checkpoint_obj, dict):
        return checkpoint_obj
    raise ValueError(
        "Unsupported checkpoint format. Expected a state-dict-like object, "
        f"but got: {type(checkpoint_obj)}"
    )


def infer_spidernet_dims_from_state_dict(state_dict):
    """Infer the exact model dimensions used during Pancancer_modeltraining."""
    required_keys = [
        "Loading_intrinsic_ori",
        "loading_receiver_ori",
        "loading_sender_ori",
        "loading_LR_ori",
        "enc_factor_envir_pre_receiver.0.weight",
    ]
    missing = [key for key in required_keys if key not in state_dict]
    if missing:
        raise KeyError(
            "The checkpoint is missing keys required to infer SpiderNet dimensions: "
            f"{missing}"
        )

    dim_intri, num_gene_from_intrinsic = state_dict["Loading_intrinsic_ori"].shape
    dim_envir, num_gene_from_receiver = state_dict["loading_receiver_ori"].shape
    dim_envir_lr, num_lr = state_dict["loading_LR_ori"].shape

    if dim_envir != dim_envir_lr:
        raise ValueError(
            "Checkpoint has inconsistent environmental dimensions: "
            f"receiver={dim_envir}, LR={dim_envir_lr}"
        )

    if num_gene_from_intrinsic != num_gene_from_receiver:
        raise ValueError(
            "Checkpoint has inconsistent gene dimensions between intrinsic and receiver loadings: "
            f"{num_gene_from_intrinsic} vs {num_gene_from_receiver}"
        )

    # In SpiderNet_model, enc_factor_envir_pre_receiver first-layer out_features = 2 * hidden_channels.
    hidden_channels = int(state_dict["enc_factor_envir_pre_receiver.0.weight"].shape[0] // 2)

    return {
        "num_gene": int(num_gene_from_receiver),
        "num_LR": int(num_lr),
        "dim_intri": int(dim_intri),
        "dim_envir": int(dim_envir),
        "hidden_channels": hidden_channels,
    }


# Load the checkpoint BEFORE model construction. The model must be initialized
# with the dimensions from Pancancer_modeltraining, not with the full processed
# data LR dimension if the full bundle was generated independently.
checkpoint = torch.load(reference_model_path, map_location="cpu")
state_dict = extract_state_dict_from_checkpoint(checkpoint)
checkpoint_dims = infer_spidernet_dims_from_state_dict(state_dict)

print("Checkpoint-derived SpiderNet dimensions:")
for key, value in checkpoint_dims.items():
    print(f"  {key}: {value}")

if checkpoint_dims["dim_envir"] != train_cfg.dim_envir:
    raise ValueError(
        f"DIM_ENVIR={train_cfg.dim_envir} does not match the checkpoint "
        f"dim_envir={checkpoint_dims['dim_envir']}."
    )

if checkpoint_dims["num_gene"] != processed.genenames_train.shape[0]:
    raise ValueError(
        "The checkpoint gene dimension does not match the full processed data gene dimension: "
        f"checkpoint={checkpoint_dims['num_gene']}, "
        f"processed={processed.genenames_train.shape[0]}. "
        "Rerun preprocessing with the reference genenames_train.pkl."
    )

processed_dim_intri = int(processed.spidernet_data[0]["cell_class_onehot"].shape[1])
if checkpoint_dims["dim_intri"] != processed_dim_intri:
    raise ValueError(
        "The checkpoint cell-class/intrinsic dimension does not match the full processed data: "
        f"checkpoint={checkpoint_dims['dim_intri']}, processed={processed_dim_intri}. "
        "The full data must use the same cell-class one-hot dimension/order as the training run."
    )

if checkpoint_dims["num_LR"] != len(reference_lr_list):
    raise ValueError(
        "The checkpoint LR dimension does not match the reference LR list length: "
        f"checkpoint={checkpoint_dims['num_LR']}, reference LR list={len(reference_lr_list)}."
    )

model = SpiderNet_model(
    num_gene=checkpoint_dims["num_gene"],
    num_LR=checkpoint_dims["num_LR"],
    hidden_channels=checkpoint_dims["hidden_channels"],
    Factor_mode="cell_class",
    dim_intri=checkpoint_dims["dim_intri"],
    dim_envir=checkpoint_dims["dim_envir"],
).to(device)

print(model)


### Restore pretrained weights

Load the reference state dictionary and switch the model to evaluation mode.


In [ ]:
load_result = model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

print(f"Loaded pretrained model from: {reference_model_path}")
print(load_result)


release_memory(
    "checkpoint",
    "state_dict",
    "load_result",
    "checkpoint_dims",
    "processed_dim_intri",
    "reference_lr_list",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False
)


### Infer MI activities and molecular loadings

Keep the processed graphs on CPU and pass one complete sub-slice at a time
to the standard inference API. This avoids placing the entire atlas on GPU.
The adapter first moves any graphs left on CUDA by an interrupted inference
back to CPU, and reports progress during inference.

The model forward pass and its precision policy are unchanged. Retain the
original batch and edge order, then call `normalize_outputs` once across
the complete atlas. The result dictionary remains compatible with the
following export cell. CPU RAM must still hold the processed bundle and
complete output arrays.

Use the configured device by default. If one sub-slice alone exceeds GPU
memory, set `INFERENCE_DEVICE = "cpu"` in this cell and rerun it. CPU
inference is slower and can differ numerically from CUDA under the model's
existing automatic mixed-precision policy.


In [ ]:
import gc
from copy import copy
from types import SimpleNamespace

import numpy as np
import torch

from SpiderNet.api import infer_meta_interactions, normalize_outputs, select_device


def infer_meta_interactions_streaming(model, processed, device=None):
    """Run the standard inference API on one sub-slice at a time."""
    device = select_device(device)
    batches = processed.spidernet_data
    if len(batches) == 0:
        raise ValueError("No processed sub-slices are available for inference.")

    # A previous all-at-once transfer may have left graphs partially on CUDA.
    for batch in batches:
        batch.to("cpu")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model = model.to(device)

    factor_envir_list = []
    for batch_index, batch in enumerate(batches):
        # PyG's shallow copy shares CPU tensors but has independent storage,
        # so the API's in-place device transfer leaves the source graph on CPU.
        device_batch = copy(batch)
        try:
            batch_results = infer_meta_interactions(
                model=model,
                processed=SimpleNamespace(spidernet_data=[device_batch]),
                device=device,
            )
        except Exception:
            # Release graph tensors even if an exception traceback retains them.
            device_batch.to("cpu")
            raise
        finally:
            del device_batch

        factor_envir_list.extend(batch_results["factor_envir_list"])
        # The API also stacks each batch; retain only its per-batch array here.
        del batch_results["factor_envir"]
        if batch_index == 0 or (batch_index + 1) % 10 == 0 or batch_index + 1 == len(batches):
            print(f"Inferred sub-slices: {batch_index + 1}/{len(batches)}", flush=True)

    # Preserve the API's batch/edge order and loadings from the final batch.
    batch_results["factor_envir_list"] = factor_envir_list
    batch_results["factor_envir"] = np.vstack(factor_envir_list)
    return batch_results


INFERENCE_DEVICE = device  # Set to "cpu" if a single sub-slice exceeds GPU memory.
results = infer_meta_interactions_streaming(
    model=model, processed=processed, device=INFERENCE_DEVICE
)
# Normalize once across the complete atlas, using the unchanged API.
results = normalize_outputs(results)

print("Factor_envir shape:", results["factor_envir"].shape)
print("LR loading shape:", results["loading_lr"].shape)
print("Receiver loading shape:", results["loading_receiver"].shape)
print("Sender loading shape:", results["loading_sender"].shape)


### Export inference results and provenance

Save the inferred arrays, loading tables, configuration files, and resolved reference inputs.


In [ ]:
from SpiderNet.api import export_results

export_results(
    results=results,
    processed=processed,
    precessed_data_dir=PROCESSED_DATA_DIR,
    output_dir=run_dirs["run_dir"],
)

with open(run_dirs["model_dir"] / "SpiderNet_model_config.json", "w", encoding="utf-8") as handle:
    json.dump(train_cfg.to_dict(), handle, indent=2)

with open(run_dirs["model_dir"] / "SpiderNet_preprocess_config.json", "w", encoding="utf-8") as handle:
    json.dump(preprocess_cfg.to_dict(), handle, indent=2)

dataloading_config_export = dict(base_config)
dataloading_config_export["lr_corr_threshold"] = LR_CORR_THRESHOLD
dataloading_config_export["processed_data_dir"] = str(processed_data_dir)
dataloading_config_export["run_dir"] = str(run_dirs["run_dir"])
dataloading_config_export["reference_results_dir"] = str(REFERENCE_RESULTS_DIR)
dataloading_config_export["reference_processed_data_dir"] = str(REFERENCE_PROCESSED_DATA_DIR)
dataloading_config_export["reference_lr_list_path"] = str(reference_lr_list_path) if reference_lr_list_path is not None else None
dataloading_config_export["reference_genename_path"] = str(reference_genename_path) if reference_genename_path is not None else None
dataloading_config_export["reference_lr_list_cellchatdb_path"] = str(reference_lr_list_cellchatdb_path)
dataloading_config_export["reference_lr_meta_cellchatdb_path"] = str(reference_lr_meta_cellchatdb_path)
dataloading_config_export["reference_model_path"] = str(reference_model_path)

with open(run_dirs["model_dir"] / "SpiderNet_dataloading_config.json", "w", encoding="utf-8") as handle:
    json.dump(json_safe_for_export(dataloading_config_export), handle, indent=2)

with open(run_dirs["run_dir"] / "Pancancer_inference_source_config.json", "w", encoding="utf-8") as handle:
    json.dump(json_safe_for_export(dataloading_config_export), handle, indent=2)

print(f"Results saved to: {run_dirs['run_dir']}")
print(f"Processed objects loaded from: {processed_data_dir}")


# The downstream analysis cells read results from disk, so the in-memory
# result dictionary and model object can be released here.
release_memory(
    "results",
    "model",
    namespace=globals(),
    run_gc=True,
    clear_cuda=True
)


### Major outputs

`Factor_envir_use.npy` stores edge-by-MI activities in processed-batch and edge
order. The `loading_LR_use`, `loading_sender_use`, and `loading_receiver_use`
arrays/tables describe LR bridges, sender regulators, and receiver targets.
Configuration JSON files record the preprocessing and checkpoint inputs.

Downstream sections also export `MI_mean_df.csv`, `geneexp_mean_df.csv`,
cell-type-pair summaries, CancerSEA profiles, MI-4 comparison tables, spatial
plots, and optional gene/GO/KEGG results under the run directory.


## Downstream interpretation

The following sections combine paper-associated analyses with additional
exploratory summaries. Keep their stated dependencies and execution order:
several plots use tables or MI ordering established in earlier cells.


### Sub-slice pseudo-bulk summaries

Average MI activity across directed edges and gene expression across cells in
each sub-slice. Export `MI_mean_df.csv` and `geneexp_mean_df.csv` for the separate
joint gene/LR projection and pseudo-bulk recovery analyses.


In [ ]:
##load data
Factor_envir_use = np.load(run_dirs['run_dir'] / "Factor_envir_use.npy")
batch_cell = pd.read_pickle(processed_data_dir / "batch_cell.pkl")

In [ ]:
batch_cell_unique = np.unique(batch_cell)


In [ ]:
##Get the mean MI strength
MI_mean_df = pd.DataFrame(
    0.0,
    columns=[f"MI_{i+1}" for i in range(Factor_envir_use.shape[1])],
    index=batch_cell_unique
)
for batch_cell_cur in batch_cell_unique:
    mask_cur = batch_cell == batch_cell_cur
    mask_cur_index = np.where(mask_cur)[0]
    MI_mean_df.loc[batch_cell_cur, :] = np.mean(Factor_envir_use[mask_cur_index, :], axis=0)

In [ ]:
subslice_id = MI_mean_df.index.to_series().str.extract(r'_subslice(\d+)$')[0].astype(int)

MI_mean_df["Split"] = np.where(subslice_id.isin([0, 1, 2, 3, 4]), "Train", "Test")

In [ ]:
##Save the MI_mean_df
MI_mean_df.to_csv(run_dirs['run_dir'] / "MI_mean_df.csv")
# MI_mean_df

# Release slice-level MI mean intermediates after saving to disk.
release_memory(
    "MI_mean_df",
    "batch_cell_cur",
    "mask_cur",
    "mask_cur_index",
    "subslice_id",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


In [ ]:
##Get the bulk level gene expression data
adata_list = pd.read_pickle(processed_data_dir / "adata_list.pkl")

In [ ]:
geneexp_mean_df = pd.DataFrame(
    0.0,
    columns=adata_list[0].var_names,
    index=batch_cell_unique
)
for batch_index in range(len(adata_list)):
    adata_i = adata_list[batch_index]
    SampleID_cur = adata_i.obs['SampleID'].unique()[0]
    geneexp_mean_df.loc[SampleID_cur, :] = np.mean(adata_i.X, axis=0)

In [ ]:
##Save the geneexp_mean_df
geneexp_mean_df.to_csv(run_dirs['run_dir'] / "geneexp_mean_df.csv")

# Release bulk-expression summary objects after saving to disk.
release_memory(
    "adata_list",
    "geneexp_mean_df",
    "adata_i",
    "SampleID_cur",
    "batch_index",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


### Validate the bulk-derived LR proxy

Compare the geometric-mean bulk LR proxy with spatial LR co-expression averaged
over directed edges. The pooled scatter is an additional diagnostic; the
per-LR Spearman distribution across sub-slices corresponds to Fig. S26a.


In [ ]:

# ============================================================
# Optional 0b. Within-dataset validation of bulk LR proxy
# against average spatial edge-level LR co-expression
# ============================================================
#
# For each sub-slice:
#   (1) bulk LR proxy:
#       sqrt(mean_bulk_ligand_expression * mean_bulk_receptor_expression)
#       using the same LR parsing rule as the R deconvolution script.
#   (2) spatial LR co-expression:
#       for each directed edge, use sender-cell ligand expression and
#       receiver-cell receptor expression; then average across edges.
#
# The final scatter plot uses one point per sub-slice:
#   x = mean bulk LR proxy across computable LR pairs
#   y = mean edge-level spatial LR co-expression across computable LR pairs

import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# Match the R setting:
# FALSE = use available subunits if at least one ligand and one receptor gene are present.
lr_require_all_genes = False

bulk_spatial_lr_output_dir = Path(run_dirs["run_dir"]) / "LR_bulk_spatial_validation"
bulk_spatial_lr_output_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def parse_lr_pair_name(lr_name):
    """
    Parse LR names like:
        "TGFB1 -> TGFBR1+TGFBR2"
    into ligand and receptor subunit lists.
    """
    lr_name = str(lr_name)
    parts = re.split(r"\s*->\s*", lr_name)
    if len(parts) != 2:
        raise ValueError(
            f"Cannot parse LR pair name '{lr_name}'. "
            "Expected format like 'LIGAND -> RECEPTOR' or 'LIG1+LIG2 -> REC1+REC2'."
        )

    ligands = [x.strip() for x in re.split(r"\+", parts[0]) if x.strip() != ""]
    receptors = [x.strip() for x in re.split(r"\+", parts[1]) if x.strip() != ""]

    if len(ligands) == 0 or len(receptors) == 0:
        raise ValueError(f"LR pair '{lr_name}' has empty ligand or receptor genes.")

    return list(dict.fromkeys(ligands)), list(dict.fromkeys(receptors))


def lr_record_to_name(record):
    """
    Convert common LR-list record formats into the same string format used by
    loading_LR_use.csv columns: 'ligand -> receptor'.
    """
    if isinstance(record, str):
        return record

    if isinstance(record, (tuple, list, np.ndarray)) and len(record) >= 2:
        return f"{record[0]} -> {record[1]}"

    if isinstance(record, dict):
        ligand_keys = ["ligand", "Ligand", "source", "Source", "ligand_gene_symbol", "ligand_symbol"]
        receptor_keys = ["receptor", "Receptor", "target", "Target", "receptor_gene_symbol", "receptor_symbol"]
        ligand_key = next((k for k in ligand_keys if k in record), None)
        receptor_key = next((k for k in receptor_keys if k in record), None)
        if ligand_key is not None and receptor_key is not None:
            return f"{record[ligand_key]} -> {record[receptor_key]}"

    raise ValueError(f"Unsupported LR-list record format: {type(record)} | {record}")


def get_lr_pair_names_for_validation():
    """
    Prefer loading_LR_use.csv because it is exactly the LR feature set used
    by the trained/deconvolution model. If it is not available, fall back
    to the reference LR list.
    """
    loading_lr_csv = Path(run_dirs["run_dir"]) / "loading_LR_use.csv"
    if loading_lr_csv.exists():
        loading_lr_df = pd.read_csv(loading_lr_csv, index_col=0)
        lr_names = list(loading_lr_df.columns.astype(str))
        print(f"Loaded {len(lr_names)} LR pairs from: {loading_lr_csv}")
        return lr_names

    # Fallback: use the reference LR list.
    if "reference_lr_list_path" not in globals() or reference_lr_list_path is None:
        raise FileNotFoundError(
            "Cannot find loading_LR_use.csv, and reference_lr_list_path is not available."
        )

    lr_obj = pd.read_pickle(reference_lr_list_path)

    if isinstance(lr_obj, pd.DataFrame):
        # First try columns that already contain 'LIG -> REC'.
        arrow_cols = [
            c for c in lr_obj.columns
            if lr_obj[c].astype(str).str.contains(r"\s*->\s*", regex=True).any()
        ]
        if len(arrow_cols) > 0:
            lr_names = lr_obj[arrow_cols[0]].astype(str).tolist()
        else:
            ligand_candidates = [
                "ligand", "Ligand", "source", "Source",
                "ligand_gene_symbol", "ligand_symbol", "interaction_name"
            ]
            receptor_candidates = [
                "receptor", "Receptor", "target", "Target",
                "receptor_gene_symbol", "receptor_symbol", "partner"
            ]
            ligand_col = next((c for c in ligand_candidates if c in lr_obj.columns), None)
            receptor_col = next((c for c in receptor_candidates if c in lr_obj.columns), None)
            if ligand_col is None or receptor_col is None:
                raise ValueError(
                    "Could not infer ligand/receptor columns from reference_lr_list. "
                    f"Available columns: {list(lr_obj.columns)}"
                )
            lr_names = [
                f"{lig} -> {rec}"
                for lig, rec in zip(lr_obj[ligand_col].astype(str), lr_obj[receptor_col].astype(str))
            ]
    else:
        lr_names = [lr_record_to_name(x) for x in list(lr_obj)]

    print(f"Loaded {len(lr_names)} LR pairs from: {reference_lr_list_path}")
    return lr_names


def build_bulk_lr_coexpression_matrix(expr_df, lr_pair_names, require_all_genes=False):
    """
    Python implementation of the R build_lr_coexpression_matrix() logic:
      - parse ligand/receptor genes from LR-pair names
      - if require_all_genes=False, use available subunits if at least
        one ligand and one receptor gene are present
      - bulk LR proxy = sqrt(mean ligand expression * mean receptor expression)
    """
    expr_df = expr_df.copy()
    expr_df.columns = expr_df.columns.astype(str)
    all_genes = set(expr_df.columns.astype(str))

    lr_values = {}
    metadata_rows = []
    skipped_rows = []

    for lr_name in lr_pair_names:
        ligands, receptors = parse_lr_pair_name(lr_name)

        ligand_present = [g in all_genes for g in ligands]
        receptor_present = [g in all_genes for g in receptors]

        if require_all_genes:
            keep_pair = all(ligand_present) and all(receptor_present)
        else:
            keep_pair = any(ligand_present) and any(receptor_present)

        if not keep_pair:
            skipped_rows.append({
                "lr_pair": lr_name,
                "ligands": "+".join(ligands),
                "receptors": "+".join(receptors),
                "missing_ligands": "+".join([g for g, present in zip(ligands, ligand_present) if not present]),
                "missing_receptors": "+".join([g for g, present in zip(receptors, receptor_present) if not present]),
            })
            continue

        ligands_use = [g for g, present in zip(ligands, ligand_present) if present]
        receptors_use = [g for g, present in zip(receptors, receptor_present) if present]

        ligand_expr = expr_df.loc[:, ligands_use].mean(axis=1).astype(float)
        receptor_expr = expr_df.loc[:, receptors_use].mean(axis=1).astype(float)

        prod = ligand_expr.to_numpy() * receptor_expr.to_numpy()
        if np.any(prod < 0):
            raise ValueError(f"Negative product detected for LR pair {lr_name}; expression should be non-negative.")

        lr_values[lr_name] = np.sqrt(prod)

        metadata_rows.append({
            "lr_pair": lr_name,
            "ligands": "+".join(ligands),
            "receptors": "+".join(receptors),
            "ligands_used": "+".join(ligands_use),
            "receptors_used": "+".join(receptors_use),
            "n_ligands_used": len(ligands_use),
            "n_receptors_used": len(receptors_use),
        })

    if len(lr_values) == 0:
        raise ValueError("No LR pairs could be computed from bulk expression.")

    lr_df = pd.DataFrame(lr_values, index=expr_df.index)
    metadata_df = pd.DataFrame(metadata_rows)
    skipped_df = pd.DataFrame(skipped_rows)

    return lr_df, metadata_df, skipped_df


def get_sample_id_from_batch(adata_i, data_i=None):
    """Return the unique sub-slice/sample ID for one processed batch."""
    if "SAMPLE_COL" in globals() and SAMPLE_COL in adata_i.obs.columns:
        vals = adata_i.obs[SAMPLE_COL].astype(str).unique()
    elif "SampleID" in adata_i.obs.columns:
        vals = adata_i.obs["SampleID"].astype(str).unique()
    elif data_i is not None and "sample" in data_i:
        vals = np.unique(np.asarray(data_i["sample"]).astype(str))
    else:
        raise KeyError("Cannot infer sample ID from AnnData.obs or spidernet_data['sample'].")

    if len(vals) != 1:
        raise ValueError(f"Expected one sample ID per batch, got {len(vals)}: {vals[:5]}")
    return str(vals[0])


def standardize_edge_index(edge_index):
    """Return edge_index as an E x 2 integer NumPy array."""
    if hasattr(edge_index, "detach"):
        edge_index = edge_index.detach().cpu().numpy()
    else:
        edge_index = np.asarray(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(np.int64)
    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64)

    raise ValueError(f"Cannot interpret edge_index shape {edge_index.shape}; expected E x 2 or 2 x E.")


def expression_to_dense_float32(X):
    """Convert AnnData.X to a dense float32 NumPy array for one sub-slice."""
    if sp.issparse(X):
        X = X.toarray()
    else:
        X = np.asarray(X)
    return np.asarray(X, dtype=np.float32)


def compute_spatial_lr_edge_mean_matrix(adata_list_use, spidernet_data_use, lr_metadata_df):
    """
    For each sub-slice and LR pair:
      mean_edges sqrt(sender_ligand_expr * receiver_receptor_expr)

    Uses the same directed edge orientation as SpiderNet:
      edge_index[:, 0] = sender
      edge_index[:, 1] = receiver
    """
    spatial_rows = {}
    n_edges_rows = {}

    for batch_idx, adata_i in enumerate(adata_list_use):
        data_i = spidernet_data_use[batch_idx]
        sample_id = get_sample_id_from_batch(adata_i, data_i)

        edge_index = standardize_edge_index(data_i["edge_index"])
        sender_idx = edge_index[:, 0]
        receiver_idx = edge_index[:, 1]

        X = expression_to_dense_float32(adata_i.X)
        gene_names = np.asarray(adata_i.var_names).astype(str)
        gene_to_col = {g: j for j, g in enumerate(gene_names)}

        if sender_idx.max(initial=-1) >= X.shape[0] or receiver_idx.max(initial=-1) >= X.shape[0]:
            raise ValueError(
                f"Edge index exceeds number of cells for {sample_id}: "
                f"max edge index={max(sender_idx.max(initial=-1), receiver_idx.max(initial=-1))}, "
                f"n_cells={X.shape[0]}"
            )

        avg_expr_cache = {}

        def get_avg_cell_expr(gene_str):
            genes = [g for g in str(gene_str).split("+") if g != ""]
            cols = tuple(gene_to_col[g] for g in genes if g in gene_to_col)

            if len(cols) == 0:
                return None

            if cols not in avg_expr_cache:
                if len(cols) == 1:
                    avg_expr_cache[cols] = X[:, cols[0]]
                else:
                    avg_expr_cache[cols] = X[:, list(cols)].mean(axis=1)

            return avg_expr_cache[cols]

        spatial_values = {}
        n_edges_used_values = {}

        for row in lr_metadata_df.itertuples(index=False):
            lr_pair = row.lr_pair
            ligand_cell_expr = get_avg_cell_expr(row.ligands_used)
            receptor_cell_expr = get_avg_cell_expr(row.receptors_used)

            if ligand_cell_expr is None or receptor_cell_expr is None or edge_index.shape[0] == 0:
                spatial_values[lr_pair] = np.nan
                n_edges_used_values[lr_pair] = 0
                continue

            prod_edge = ligand_cell_expr[sender_idx] * receptor_cell_expr[receiver_idx]
            prod_edge = np.clip(prod_edge, a_min=0, a_max=None)

            spatial_values[lr_pair] = float(np.mean(np.sqrt(prod_edge)))
            n_edges_used_values[lr_pair] = int(edge_index.shape[0])

        spatial_rows[sample_id] = spatial_values
        n_edges_rows[sample_id] = n_edges_used_values

        # Release the dense expression matrix for this batch before moving on.
        del X, avg_expr_cache

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(adata_list_use):
            print(f"Processed spatial LR co-expression for {batch_idx + 1}/{len(adata_list_use)} sub-slices")

    spatial_lr_df = pd.DataFrame.from_dict(spatial_rows, orient="index")
    n_edges_df = pd.DataFrame.from_dict(n_edges_rows, orient="index")

    return spatial_lr_df, n_edges_df


def format_pvalue(p):
    if not np.isfinite(p):
        return "P = NA"
    if p < 2.2e-16:
        return "P < 2.2e-16"
    if p < 0.001:
        return f"P = {p:.2e}"
    return f"P = {p:.3g}"


# ------------------------------------------------------------
# Step 1. Load / verify sub-slice bulk expression
# ------------------------------------------------------------
geneexp_mean_path = Path(run_dirs["run_dir"]) / "geneexp_mean_df.csv"

if geneexp_mean_path.exists():
    geneexp_mean_df = pd.read_csv(geneexp_mean_path, index_col=0)
else:
    # Fallback: regenerate if this cell is run before saving geneexp_mean_df.csv.
    adata_list_tmp = processed.adata_list if "processed" in globals() else pd.read_pickle(processed_data_dir / "adata_list.pkl")
    sample_ids_tmp = [
        get_sample_id_from_batch(adata_i, None)
        for adata_i in adata_list_tmp
    ]
    geneexp_mean_df = pd.DataFrame(
        0.0,
        columns=adata_list_tmp[0].var_names.astype(str),
        index=sample_ids_tmp
    )
    for adata_i, sample_id in zip(adata_list_tmp, sample_ids_tmp):
        X_tmp = adata_i.X
        if sp.issparse(X_tmp):
            geneexp_mean_df.loc[sample_id, :] = np.asarray(X_tmp.mean(axis=0)).ravel()
        else:
            geneexp_mean_df.loc[sample_id, :] = np.asarray(X_tmp).mean(axis=0)
    geneexp_mean_df.to_csv(geneexp_mean_path)

geneexp_mean_df.columns = geneexp_mean_df.columns.astype(str)
geneexp_mean_df.index = geneexp_mean_df.index.astype(str)

print(f"Loaded bulk expression matrix: {geneexp_mean_df.shape[0]} sub-slices x {geneexp_mean_df.shape[1]} genes")


# ------------------------------------------------------------
# Step 2. Compute bulk LR proxy, following the R logic
# ------------------------------------------------------------
lr_pair_names = get_lr_pair_names_for_validation()

bulk_lr_proxy_df, lr_metadata_df, lr_skipped_df = build_bulk_lr_coexpression_matrix(
    expr_df=geneexp_mean_df,
    lr_pair_names=lr_pair_names,
    require_all_genes=lr_require_all_genes,
)

bulk_lr_proxy_df.to_csv(bulk_spatial_lr_output_dir / "LR_coexpression_bulk_proxy_subslice_by_LR.csv")
lr_metadata_df.to_csv(bulk_spatial_lr_output_dir / "LR_pair_parse_metadata_for_bulk_spatial_validation.csv", index=False)
if len(lr_skipped_df) > 0:
    lr_skipped_df.to_csv(bulk_spatial_lr_output_dir / "LR_pair_skipped_missing_genes.csv", index=False)

print(
    "Bulk LR proxy computed:",
    f"{bulk_lr_proxy_df.shape[0]} sub-slices x {bulk_lr_proxy_df.shape[1]} LR pairs"
)
if len(lr_skipped_df) > 0:
    print(f"Skipped {len(lr_skipped_df)} LR pairs due to missing ligand/receptor genes.")


# ------------------------------------------------------------
# Step 3. Compute spatial edge-level LR co-expression
# ------------------------------------------------------------
adata_list_for_lr = processed.adata_list if "processed" in globals() else pd.read_pickle(processed_data_dir / "adata_list.pkl")
spidernet_data_for_lr = processed.spidernet_data if "processed" in globals() else load_spidernet_pyg_list(processed_data_dir)

spatial_lr_edge_mean_df, n_edges_by_lr_df = compute_spatial_lr_edge_mean_matrix(
    adata_list_use=adata_list_for_lr,
    spidernet_data_use=spidernet_data_for_lr,
    lr_metadata_df=lr_metadata_df,
)

spatial_lr_edge_mean_df.to_csv(
    bulk_spatial_lr_output_dir / "LR_coexpression_spatial_edge_mean_subslice_by_LR.csv"
)
n_edges_by_lr_df.to_csv(
    bulk_spatial_lr_output_dir / "LR_coexpression_spatial_n_edges_subslice_by_LR.csv"
)

print(
    "Spatial LR edge co-expression computed:",
    f"{spatial_lr_edge_mean_df.shape[0]} sub-slices x {spatial_lr_edge_mean_df.shape[1]} LR pairs"
)


# ------------------------------------------------------------
# Step 4. Align samples/LR pairs and make one summary value per sub-slice
# ------------------------------------------------------------
common_samples = [
    s for s in bulk_lr_proxy_df.index
    if s in spatial_lr_edge_mean_df.index
]
common_lr_pairs = [
    lr for lr in bulk_lr_proxy_df.columns
    if lr in spatial_lr_edge_mean_df.columns
]

if len(common_samples) < 3:
    raise ValueError("Too few common sub-slices between bulk and spatial LR matrices.")
if len(common_lr_pairs) < 2:
    raise ValueError("Too few common LR pairs between bulk and spatial LR matrices.")

bulk_lr_proxy_aligned = bulk_lr_proxy_df.loc[common_samples, common_lr_pairs]
spatial_lr_edge_mean_aligned = spatial_lr_edge_mean_df.loc[common_samples, common_lr_pairs]
# 
# bulk_lr_proxy_mean = bulk_lr_proxy_aligned.mean(axis=1, skipna=True)
# spatial_lr_edge_mean = spatial_lr_edge_mean_aligned.mean(axis=1, skipna=True)
# 
# summary_df = pd.DataFrame({
#     "sample_id": common_samples,
#     "CancerType": pd.Series(common_samples, index=common_samples).str.replace(r"_subslice\d+$", "", regex=True).values,
#     "bulk_LR_proxy_mean": bulk_lr_proxy_mean.values,
#     "spatial_edge_LR_coexpression_mean": spatial_lr_edge_mean.values,
#     "n_LR_pairs_used": spatial_lr_edge_mean_aligned.notna().sum(axis=1).values,
#     "n_edges": n_edges_by_lr_df.loc[common_samples, common_lr_pairs].max(axis=1).values,
# })
# 
# summary_df = summary_df[
#     np.isfinite(summary_df["bulk_LR_proxy_mean"])
#     & np.isfinite(summary_df["spatial_edge_LR_coexpression_mean"])
# ].copy()
# 
# rho, pval = spearmanr(
#     summary_df["bulk_LR_proxy_mean"],
#     summary_df["spatial_edge_LR_coexpression_mean"],
# )
# 
# summary_df["spearman_rho_all_subslices"] = rho
# summary_df["spearman_pvalue_all_subslices"] = pval
# summary_df.to_csv(
#     bulk_spatial_lr_output_dir / "LR_bulk_proxy_vs_spatial_edge_coexpression_summary.csv",
#     index=False
# )
# 
# print(f"Spearman rho = {rho:.3f}, {format_pvalue(pval)}, n = {summary_df.shape[0]} sub-slices")
# display(summary_df.head())
# 
# 
# # ------------------------------------------------------------
# # Step 5. Scatter plot: one point per sub-slice
# # ------------------------------------------------------------
# fig, ax = plt.subplots(figsize=(4.4, 4.0))
# 
# ax.scatter(
#     summary_df["bulk_LR_proxy_mean"],
#     summary_df["spatial_edge_LR_coexpression_mean"],
#     s=28,
#     alpha=0.75,
#     linewidths=0,
# )
# 
# ax.set_xlabel("Bulk LR co-expression proxy\n(mean across LR pairs)", fontsize=11)
# ax.set_ylabel("Spatial edge-level LR co-expression\n(mean across edges and LR pairs)", fontsize=11)
# 
# ax.text(
#     0.05,
#     0.95,
#     f"Spearman ρ = {rho:.3f}\n{format_pvalue(pval)}\nn = {summary_df.shape[0]}",
#     transform=ax.transAxes,
#     ha="left",
#     va="top",
#     fontsize=10,
# )
# 
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
# ax.tick_params(labelsize=10)
# 
# plt.tight_layout()
# 
# plot_pdf = bulk_spatial_lr_output_dir / "LR_bulk_proxy_vs_spatial_edge_coexpression_scatter.pdf"
# plot_png = bulk_spatial_lr_output_dir / "LR_bulk_proxy_vs_spatial_edge_coexpression_scatter.png"
# 
# plt.savefig(plot_pdf, bbox_inches="tight")
# plt.savefig(plot_png, bbox_inches="tight", dpi=300)
# plt.show()
# 
# print(f"Saved summary table to: {bulk_spatial_lr_output_dir / 'LR_bulk_proxy_vs_spatial_edge_coexpression_summary.csv'}")
# print(f"Saved scatter plot to: {plot_pdf}")


In [ ]:
from scipy.stats import spearmanr, linregress
# ------------------------------------------------------------
# Step 4. Flatten aligned matrices into paired observations
# Each point = one (sub-slice, LR pair)
# ------------------------------------------------------------
bulk_long = (
    bulk_lr_proxy_aligned.loc[common_samples, common_lr_pairs]
    .stack(dropna=False)
    .rename("bulk_LR_proxy")
)

spatial_long = (
    spatial_lr_edge_mean_aligned.loc[common_samples, common_lr_pairs]
    .stack(dropna=False)
    .rename("spatial_edge_LR_coexpression")
)

n_edges_long = (
    n_edges_by_lr_df.loc[common_samples, common_lr_pairs]
    .stack(dropna=False)
    .rename("n_edges")
)

summary_df = pd.concat([bulk_long, spatial_long, n_edges_long], axis=1).reset_index()
summary_df.columns = [
    "sample_id",
    "LR_pair",
    "bulk_LR_proxy",
    "spatial_edge_LR_coexpression",
    "n_edges",
]

summary_df["CancerType"] = summary_df["sample_id"].str.replace(
    r"_subslice\d+$", "", regex=True
)

# Keep only finite paired values
summary_df = summary_df[
    np.isfinite(summary_df["bulk_LR_proxy"])
    & np.isfinite(summary_df["spatial_edge_LR_coexpression"])
].copy()

# Optional exclusion of LR pairs without spatial edges.
summary_df = summary_df[
    np.isfinite(summary_df["n_edges"]) & (summary_df["n_edges"] > 0)
].copy()

# Spearman correlation
rho, pval = spearmanr(
    summary_df["bulk_LR_proxy"],
    summary_df["spatial_edge_LR_coexpression"],
)

# Linear regression line
lr_res = linregress(
    summary_df["bulk_LR_proxy"],
    summary_df["spatial_edge_LR_coexpression"],
)

summary_df["spearman_rho_all_points"] = rho
summary_df["spearman_pvalue_all_points"] = pval
summary_df["linear_slope"] = lr_res.slope
summary_df["linear_intercept"] = lr_res.intercept
summary_df["linear_rvalue"] = lr_res.rvalue
summary_df["linear_pvalue"] = lr_res.pvalue
summary_df["linear_stderr"] = lr_res.stderr

summary_csv = bulk_spatial_lr_output_dir / "LR_bulk_proxy_vs_spatial_edge_coexpression_long.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"Spearman rho = {rho:.3f}, {format_pvalue(pval)}, n = {summary_df.shape[0]} paired points")
display(summary_df.head())


# ------------------------------------------------------------
# Step 5. Scatter plot using flattened paired observations
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(4.8, 4.2))

ax.scatter(
    summary_df["bulk_LR_proxy"],
    summary_df["spatial_edge_LR_coexpression"],
    s=10,
    alpha=0.35,
    linewidths=0,
    rasterized=True,
)

# Regression line
x_min = summary_df["bulk_LR_proxy"].min()
x_max = summary_df["bulk_LR_proxy"].max()
x_line = np.linspace(x_min, x_max, 200)
y_line = lr_res.intercept + lr_res.slope * x_line

ax.plot(
    x_line,
    y_line,
    linewidth=2,
)

ax.set_xlabel("Bulk LR co-expression proxy", fontsize=11)
ax.set_ylabel("Spatial edge-level LR co-expression", fontsize=11)

ax.text(
    0.05,
    0.95,
    f"Spearman ρ = {rho:.3f}\n{format_pvalue(pval)}\nn = {summary_df.shape[0]}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=10)

plt.tight_layout()

plot_pdf = bulk_spatial_lr_output_dir / "LR_bulk_proxy_vs_spatial_edge_coexpression_scatter_flattened.pdf"
plot_png = bulk_spatial_lr_output_dir / "LR_bulk_proxy_vs_spatial_edge_coexpression_scatter_flattened.png"

plt.savefig(plot_pdf, bbox_inches="tight")
plt.savefig(plot_png, bbox_inches="tight", dpi=300)
plt.show()

print(f"Saved long-format summary table to: {summary_csv}")
print(f"Saved scatter plot to: {plot_pdf}")

In [ ]:
# ------------------------------------------------------------
# Step 6. Per-LR-pair Spearman correlation across sub-slices
# Each correlation = one LR pair
# ------------------------------------------------------------
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Nature / Illustrator-friendly settings
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.linewidth"] = 0.8

per_lr_corr_records = []

min_n_subslices = 5  # require at least this many paired sub-slices per LR pair

for lr_pair in common_lr_pairs:
    x = bulk_lr_proxy_aligned.loc[common_samples, lr_pair]
    y = spatial_lr_edge_mean_aligned.loc[common_samples, lr_pair]

    valid_mask = np.isfinite(x) & np.isfinite(y)
    x_use = x[valid_mask]
    y_use = y[valid_mask]

    n_valid = len(x_use)

    if n_valid < min_n_subslices:
        rho = np.nan
        pval = np.nan
        status = "too_few_subslices"
    elif x_use.nunique() < 2 or y_use.nunique() < 2:
        rho = np.nan
        pval = np.nan
        status = "constant_profile"
    else:
        rho, pval = spearmanr(x_use, y_use)
        status = "ok"

    per_lr_corr_records.append({
        "LR_pair": lr_pair,
        "spearman_rho": rho,
        "spearman_pvalue": pval,
        "n_subslices_used": n_valid,
        "status": status,
    })

per_lr_corr_df = pd.DataFrame(per_lr_corr_records)

per_lr_corr_csv = bulk_spatial_lr_output_dir / "LR_pairwise_bulk_proxy_vs_spatial_edge_spearman.csv"
per_lr_corr_df.to_csv(per_lr_corr_csv, index=False)

print(f"Saved per-LR-pair Spearman correlations to: {per_lr_corr_csv}")
display(per_lr_corr_df.head())

valid_corr_df = per_lr_corr_df[
    np.isfinite(per_lr_corr_df["spearman_rho"])
].copy()

print(
    f"Computed valid Spearman correlations for "
    f"{valid_corr_df.shape[0]} / {per_lr_corr_df.shape[0]} LR pairs."
)

print(valid_corr_df["spearman_rho"].describe())


# ------------------------------------------------------------
# Step 7. Nature-style boxplot of per-LR-pair Spearman correlations
# ------------------------------------------------------------
rho_values = valid_corr_df["spearman_rho"].dropna().values

if len(rho_values) == 0:
    raise ValueError("No valid per-LR-pair Spearman correlations to plot.")

median_rho = np.nanmedian(rho_values)
mean_rho = np.nanmean(rho_values)

fig, ax = plt.subplots(figsize=(2.4, 3.2))

# Transparent background
fig.patch.set_alpha(0)
ax.set_facecolor("none")

box = ax.boxplot(
    rho_values,
    positions=[1],
    widths=0.42,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(
        facecolor="none",
        edgecolor="black",
        linewidth=1.0,
    ),
    medianprops=dict(
        color="black",
        linewidth=1.2,
    ),
    whiskerprops=dict(
        color="black",
        linewidth=1.0,
    ),
    capprops=dict(
        color="black",
        linewidth=1.0,
    ),
)

# Overlay individual LR-pair correlations as normal vector dots
rng = np.random.default_rng(1)
x_jitter = 1 + rng.normal(0, 0.035, size=len(rho_values))

ax.scatter(
    x_jitter,
    rho_values,
    s=9,
    marker="o",
    color="black",
    alpha=0.45,
    edgecolors="none",
    linewidths=0,
    zorder=3,
)

# Optional: zero reference line
# ax.axhline(
#     0,
#     color="0.7",
#     linestyle="--",
#     linewidth=0.8,
#     zorder=0,
# )

ax.set_xlim(0.55, 1.45)
ax.set_xticks([1])
ax.set_xticklabels(["LR pairs"], fontsize=10)

ax.set_ylabel("Spearman ρ across sub-slices", fontsize=10)

# Explicitly restore x/y axes and ticks
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="y",
    which="major",
    left=True,
    right=False,
    labelleft=True,
    width=0.8,
    length=3,
    color="black",
    labelsize=9,
)

ax.tick_params(
    axis="x",
    which="major",
    bottom=True,
    top=False,
    labelbottom=True,
    width=0.8,
    length=3,
    color="black",
    labelsize=9,
)

# Use clean y-axis ticks
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

plt.tight_layout()

boxplot_pdf = bulk_spatial_lr_output_dir / "LR_pairwise_spearman_distribution_boxplot_nature_transparent.pdf"
boxplot_png = bulk_spatial_lr_output_dir / "LR_pairwise_spearman_distribution_boxplot_nature_transparent.png"

plt.savefig(
    boxplot_pdf,
    bbox_inches="tight",
    transparent=True,
)

plt.savefig(
    boxplot_png,
    bbox_inches="tight",
    dpi=600,
    transparent=True,
)

plt.show()

print(f"Saved Nature-style transparent boxplot to: {boxplot_pdf}")
print(f"Saved PNG to: {boxplot_png}")
print(f"Median Spearman rho = {median_rho:.3f}")
print(f"Mean Spearman rho = {mean_rho:.3f}")

### Correlations between MIs

Inspect relationships among inferred edge-level MI activities.


In [ ]:
from SpiderNet.analysis import MI_correlation

mi_results = MI_correlation(run_dirs['run_dir'], run_dirs['run_dir'] / "Factor_envir_use.npy", show=True)


release_memory('mi_results', namespace=globals(), run_gc=True, clear_cuda=False)


### CellChat LR pathway profiles

Summarize reference LR loadings by CellChat pathway with `LRLoading_enrichment`.
The current helper averages normalized LR loadings within pathways. Use the LR
list and CellChat metadata from the reference training run.


In [ ]:
from SpiderNet.analysis import LRLoading_enrichment

# LR loading comes from the pretrained Pancancer_modeltraining checkpoint, so
# interpret it with the reference training LR list and CellChat metadata.
LRLoading_enrichment(
    loading_LR_use_path=run_dirs['run_dir'] / "loading_LR_use.npy",
    lr_list_path=reference_lr_list_path,
    lr_list_cellchatdb_path=reference_lr_list_cellchatdb_path,
    lr_meta_cellchatdb_path=reference_lr_meta_cellchatdb_path,
    Factor_envir_use_path=run_dirs['run_dir'] / "Factor_envir_use.npy",
    file_savepath_main=run_dirs['run_dir'],
    show=True,
    min_lr_pairs_per_pathway=2
)


### Sender-receiver cell-type summaries

Compute tumor involvement and cell-type-pair activity profiles.


#### Tumor-involved versus non-tumor-involved MI activity

Compute sub-slice/sample summaries and log2 fold changes. The threshold-count
bar plot is an additional tumor-involvement summary.


In [ ]:
##load data
Factor_envir_use = np.load(run_dirs['run_dir'] / "Factor_envir_use.npy")
batch_cell = pd.read_pickle(processed_data_dir / "batch_cell.pkl")

In [ ]:
MI_intensity_threshold = 0.02

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from scipy.stats import wilcoxon


# ============================================================
# Step 1. Collect edge-level metadata from processed batches
# ============================================================
def collect_edge_metadata(processed):
    """
    Extract, for every edge across all batches:
    1) the slice/sample ID of the sender cell,
    2) the sender cell type,
    3) the receiver cell type.
    """
    edge_sample_list = []
    sender_celltype_list = []
    receiver_celltype_list = []

    for batch_idx in range(len(processed.spidernet_data)):
        data_cur = processed.spidernet_data[batch_idx]
        edge_index_cur = data_cur["edge_index"].cpu().numpy()

        sample_cur = np.asarray(data_cur["sample"]).astype(str)
        celltype_cur = np.asarray(data_cur["cell_class"]).astype(str)

        edge_sample_list.append(sample_cur[edge_index_cur[:, 0]])
        sender_celltype_list.append(celltype_cur[edge_index_cur[:, 0]])
        receiver_celltype_list.append(celltype_cur[edge_index_cur[:, 1]])

    edge_sample = np.hstack(edge_sample_list)
    sender_celltype = np.hstack(sender_celltype_list)
    receiver_celltype = np.hstack(receiver_celltype_list)

    return edge_sample, sender_celltype, receiver_celltype


# ============================================================
# Step 2. Summarize MI activity at the slice level
# ============================================================
def summarize_mi_by_slice(factor_matrix, edge_sample, edge_has_cancer_cell, use_tumor_edges=True):
    """
    Compute slice-level MI summaries using the median MI value across edges.

    Parameters
    ----------
    factor_matrix : np.ndarray
        Edge-by-MI matrix.
    edge_sample : np.ndarray
        Slice/sample ID for each edge.
    edge_has_cancer_cell : np.ndarray of bool
        Whether each edge involves at least one cancer cell.
    use_tumor_edges : bool
        If True, summarize edges that involve cancer cells.
        If False, summarize edges that do not involve cancer cells.
    """
    slice_ids = np.sort(np.unique(edge_sample))
    mi_summary_list = []

    base_mask = edge_has_cancer_cell if use_tumor_edges else ~edge_has_cancer_cell

    for slice_id in slice_ids:
        mask_cur = (edge_sample == slice_id) & base_mask

        if np.any(mask_cur):
            mi_summary = np.median(factor_matrix[mask_cur, :], axis=0)
        else:
            mi_summary = np.full(factor_matrix.shape[1], np.nan)

        mi_summary_list.append(mi_summary)

    mi_summary_df = pd.DataFrame(
        np.vstack(mi_summary_list),
        index=slice_ids,
        columns=[f"MI_{i + 1}" for i in range(factor_matrix.shape[1])]
    )

    return mi_summary_df


# ============================================================
# Step 3. Aggregate slice-level MI summaries to the sample level
# ============================================================
def aggregate_slice_to_sample(mi_slice_df, slice_cell_count):
    """
    Aggregate slice-level MI summaries into sample-level summaries using
    cell-count-weighted averaging across slices from the same sample.
    """
    slice_ids = mi_slice_df.index.astype(str).tolist()
    sample_ids = [slice_id.split("_")[0] for slice_id in slice_ids]
    unique_samples = np.sort(np.unique(sample_ids))

    mi_sample_df = pd.DataFrame(
        np.nan,
        index=unique_samples,
        columns=mi_slice_df.columns,
        dtype=float
    )

    for sample_id in unique_samples:
        slice_ids_cur = [sid for sid in slice_ids if sid.split("_")[0] == sample_id]
        weights_cur = slice_cell_count.loc[slice_ids_cur].values.astype(float)
        values_cur = mi_slice_df.loc[slice_ids_cur, :].values.astype(float)

        # Remove slices with all-NaN MI summaries
        valid_rows = ~np.isnan(values_cur).all(axis=1)
        values_cur = values_cur[valid_rows]
        weights_cur = weights_cur[valid_rows]

        if values_cur.shape[0] == 0:
            continue

        weighted_mean = (
            np.sum(values_cur * weights_cur[:, np.newaxis], axis=0) /
            (np.sum(weights_cur) + 1e-10)
        )
        mi_sample_df.loc[sample_id, :] = weighted_mean

    return mi_sample_df


# ============================================================
# Step 4. Compute tumor and non-tumor slice-level MI summaries
# ============================================================
Factor_envir_use_show = Factor_envir_use

edge_sample, sender_celltype, receiver_celltype = collect_edge_metadata(processed)

# Mark edges that involve at least one cancer cell
edge_has_cancer_cell = np.logical_or(
    np.char.find(sender_celltype, "-cancercell") >= 0,
    np.char.find(receiver_celltype, "-cancercell") >= 0
)

MI_mean_pd_tumor = summarize_mi_by_slice(
    factor_matrix=Factor_envir_use_show,
    edge_sample=edge_sample,
    edge_has_cancer_cell=edge_has_cancer_cell,
    use_tumor_edges=True
)

MI_mean_pd_nontumor = summarize_mi_by_slice(
    factor_matrix=Factor_envir_use_show,
    edge_sample=edge_sample,
    edge_has_cancer_cell=edge_has_cancer_cell,
    use_tumor_edges=False
)

print("Tumor-edge slice-level MI summary:")
display(MI_mean_pd_tumor)

print("Non-tumor-edge slice-level MI summary:")
display(MI_mean_pd_nontumor)


# ============================================================
# Step 5. Get the number of cells in each slice
# ============================================================
# This assumes processed.adata_list is batch-aligned with processed.spidernet_data.
slice_ids_processed = [
    str(np.asarray(processed.spidernet_data[i]["sample"])[0])
    for i in range(len(processed.spidernet_data))
]
numcell_list = [processed.adata_list[i].n_obs for i in range(len(processed.adata_list))]

slice_cell_count = pd.Series(numcell_list, index=slice_ids_processed, dtype=float)

print("Number of cells in each slice:")
display(slice_cell_count)


# ============================================================
# Step 6. Aggregate tumor and non-tumor MI summaries to sample level
# ============================================================
MI_mean_pd_tumor_sample = aggregate_slice_to_sample(
    mi_slice_df=MI_mean_pd_tumor,
    slice_cell_count=slice_cell_count
)

MI_mean_pd_nontumor_sample = aggregate_slice_to_sample(
    mi_slice_df=MI_mean_pd_nontumor,
    slice_cell_count=slice_cell_count
)

print("Tumor-edge sample-level MI summary:")
display(MI_mean_pd_tumor_sample)

print("Non-tumor-edge sample-level MI summary:")
display(MI_mean_pd_nontumor_sample)


# ============================================================
# Step 7. Compute log2(tumor / non-tumor) at the sample level
# ============================================================
MI_mean_pd_tumor_vs_nontumor = (
    MI_mean_pd_tumor_sample + 1e-6
) / (
    MI_mean_pd_nontumor_sample + 1e-6
)

MI_mean_pd_tumor_vs_nontumor_log2 = np.log2(MI_mean_pd_tumor_vs_nontumor)

# Order MIs by the mean log2 ratio across samples
colmean = MI_mean_pd_tumor_vs_nontumor_log2.mean(axis=0)
MI_mean_pd_tumor_vs_nontumor_log2 = MI_mean_pd_tumor_vs_nontumor_log2.loc[
    :, colmean.sort_values(ascending=False).index
]

print("Sample-level log2(tumor / non-tumor):")
display(MI_mean_pd_tumor_vs_nontumor_log2)

MI_mean_pd_tumor_sample_colmax = MI_mean_pd_tumor_sample.max(axis=0)
print("Maximum tumor-edge MI value across samples:")
display(MI_mean_pd_tumor_sample_colmax)


# ============================================================
# Step 8. Plot the maximum tumor-versus-non-tumor difference for each MI
# ============================================================
# Keep MIs with relatively large absolute tumor-edge activity for highlighting
MI_mean_pd_tumor_sample_colmax_large = MI_mean_pd_tumor_sample_colmax[
    MI_mean_pd_tumor_sample_colmax > MI_intensity_threshold
].index.tolist()

print("MIs with tumor-edge sample-level max > " + str(MI_intensity_threshold) + ":")
print(MI_mean_pd_tumor_sample_colmax_large)

df = MI_mean_pd_tumor_vs_nontumor_log2.copy()


# Release edge-level helper arrays after slice- and sample-level summaries
# are created. Keep the summary tables because they are reused below.
release_memory(
    "Factor_envir_use_show",
    "edge_sample",
    "sender_celltype",
    "receiver_celltype",
    "edge_has_cancer_cell",
    "slice_ids_processed",
    "numcell_list",
    "slice_cell_count",
    "colmean",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False
)


In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ============================================================
# Nature-style plotting settings
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

sns.set_theme(style="white", context="paper")


# ============================================================
# Step 1. Prepare log2FC matrix
# ============================================================
# Expected input:
#   df rows    = cancer sections / samples
#   df columns = MIs
#
# If df is accidentally MI x sample, transpose it automatically.
def _is_mi_like(x):
    x = str(x)
    return x.startswith("MI") or x.startswith("MetaI")


df_use = df.copy()

n_mi_like_cols = sum([_is_mi_like(x) for x in df_use.columns])
n_mi_like_index = sum([_is_mi_like(x) for x in df_use.index])

if n_mi_like_index > n_mi_like_cols:
    print("Detected df as MI x sample. Transposing to sample x MI.")
    df_use = df_use.T

df_use.columns = [str(x).replace("MI_", "MI-") for x in df_use.columns]
df_use = df_use.apply(pd.to_numeric, errors="coerce")


# ============================================================
# Step 2. Compute count of cancer sections with log2FC > 2
# ============================================================
log2fc_threshold = 2.0
tumor_edge_max_threshold = MI_intensity_threshold

count_log2fc_gt2 = (df_use > log2fc_threshold).sum(axis=0)
mean_log2fc = df_use.mean(axis=0, skipna=True)
max_log2fc = df_use.max(axis=0, skipna=True)

plot_df = pd.DataFrame({
    "MI": count_log2fc_gt2.index.astype(str),
    "count_log2FC_gt2": count_log2fc_gt2.values.astype(int),
    "mean_log2FC": mean_log2fc.loc[count_log2fc_gt2.index].values,
    "max_log2FC": max_log2fc.loc[count_log2fc_gt2.index].values,
})


# ============================================================
# Step 3. Compute tumor-edge max per MI and define red-highlight MIs
# ============================================================
tumor_edge_max_s = MI_mean_pd_tumor_sample.copy()
tumor_edge_max_s.columns = [
    str(x).replace("MI_", "MI-")
    for x in tumor_edge_max_s.columns
]
tumor_edge_max_s = tumor_edge_max_s.apply(pd.to_numeric, errors="coerce").max(axis=0, skipna=True)

plot_df["tumor_edge_max"] = plot_df["MI"].map(tumor_edge_max_s)
plot_df["tumor_edge_max"] = plot_df["tumor_edge_max"].fillna(0.0)

plot_df["highlight"] = plot_df["tumor_edge_max"] > tumor_edge_max_threshold


# ============================================================
# Step 4. Sort MIs
# Primary sorting: number of cancer sections with log2FC > 2
# Secondary sorting: mean log2FC across all cancer sections
# ============================================================
plot_df = (
    plot_df
    .sort_values(
        by=["count_log2FC_gt2", "mean_log2FC"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

mi_order = plot_df["MI"].tolist()
df_use = df_use.loc[:, mi_order]

print("MI plotting order:")
print(mi_order)

print("Summary table:")
display(plot_df)


# ============================================================
# Step 5. Plot barplot:
# Number of cancer sections with log2(tumor / non-tumor) > 2
# ============================================================
colors = [
    "red" if is_highlight else "lightgrey"
    for is_highlight in plot_df["highlight"].values
]

plt.close("all")
fig, ax = plt.subplots(figsize=(7.2, 5.2))

ax.barh(
    plot_df["MI"],
    plot_df["count_log2FC_gt2"],
    color=colors
)

ax.invert_yaxis()

ax.set_xlabel("Number of cancer sections with log2(tumor / non-tumor) > 2")
ax.set_ylabel("")

# Use integer x-axis ticks
max_count = int(plot_df["count_log2FC_gt2"].max())
ax.set_xlim(0, max_count + 0.8)
ax.set_xticks(np.arange(0, max_count + 1, 1))

# Add count labels to bars
for y_pos, count_value in enumerate(plot_df["count_log2FC_gt2"].values):
    ax.text(
        count_value + 0.08,
        y_pos,
        str(int(count_value)),
        va="center",
        ha="left",
        fontsize=9
    )

red_patch = mpatches.Patch(
    color="red",
    label=f"tumor-edge max > {tumor_edge_max_threshold}"
)

grey_patch = mpatches.Patch(
    color="lightgrey",
    label="other MIs"
)

ax.legend(
    handles=[red_patch, grey_patch],
    frameon=False,
    loc="lower right"
)

sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    str(run_dirs["run_dir"]) + "/MI_log2FC_gt2_cancer_section_count_barplot.pdf",
    bbox_inches="tight",
    transparent=True
)

fig.savefig(
    str(run_dirs["run_dir"]) + "/MI_log2FC_gt2_cancer_section_count_barplot.png",
    bbox_inches="tight",
    dpi=300,
    transparent=True
)

plt.show()
plt.close(fig)


# ============================================================
# Step 6. Save ordered summary table
# ============================================================
plot_df.to_csv(
    str(run_dirs["run_dir"]) + "/MI_log2FC_gt2_cancer_section_count_summary.csv",
    index=False
)

df_use.to_csv(
    str(run_dirs["run_dir"]) + "/MI_log2FC_matrix_ordered_by_count_gt2.csv",
    index=True
)


# ============================================================
# Step 7. Release memory
# ============================================================
release_memory(
    "df_use",
    "count_log2fc_gt2",
    "mean_log2fc",
    "max_log2fc",
    "tumor_edge_max_s",
    "plot_df",
    "colors",
    "red_patch",
    "grey_patch",
    "fig",
    "ax",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
    close_figures=True
)

#### Additional sample-level and cell-type-pair summaries

Generate mean-activity, row-normalized, pair-maximum, and tumor/non-tumor ratio
heatmaps. Gaussian mixture models explore activity thresholds. The final
top-10%-pair section provides the tumor-involved fractions and cancer-type
recurrence tables used in Fig. 6b; the preceding heatmaps are alternative summaries.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as patches


# ============================================================
# Step 1. Get the number of cells in each slice
# ============================================================
# This assumes that processed.adata_list is aligned with the slice-level MI summary table.
numcell_list = [processed.adata_list[i].n_obs for i in range(len(processed.adata_list))]
print("Number of cells in each slice:")
print(numcell_list)


# ============================================================
# Step 2. Aggregate slice-level MI summaries to the sample level
# ============================================================
def aggregate_slice_mi_to_sample(mi_slice_df, numcell_list):
    """
    Aggregate slice-level MI summaries into sample-level summaries
    using cell-count-weighted averaging.

    Parameters
    ----------
    mi_slice_df : pd.DataFrame
        Slice-level MI summary table.
        Rows are slices and columns are MI features.
    numcell_list : list
        Number of cells in each slice, aligned with mi_slice_df rows.

    Returns
    -------
    pd.DataFrame
        Sample-level MI summary table.
    """
    slice_names = mi_slice_df.index.tolist()
    sample_names = [slice_name.split("_")[0] for slice_name in slice_names]
    unique_samples = np.unique(sample_names)

    mi_sample_df = pd.DataFrame(
        0.0,
        index=unique_samples,
        columns=mi_slice_df.columns
    )

    for sample_name_cur in unique_samples:
        slice_index_cur = [
            i for i in range(len(slice_names))
            if sample_names[i] == sample_name_cur
        ]
        numcell_slice_cur = [numcell_list[i] for i in slice_index_cur]

        weighted_mean = (
            np.sum(
                mi_slice_df.iloc[slice_index_cur, :].values *
                np.array(numcell_slice_cur)[:, np.newaxis],
                axis=0
            ) / (np.sum(numcell_slice_cur) + 1e-10)
        )

        mi_sample_df.loc[sample_name_cur, :] = weighted_mean

    return mi_sample_df


MI_mean_pd_tumor_sample = aggregate_slice_mi_to_sample(
    mi_slice_df=MI_mean_pd_tumor,
    numcell_list=numcell_list
)

MI_mean_pd_nontumor_sample = aggregate_slice_mi_to_sample(
    mi_slice_df=MI_mean_pd_nontumor,
    numcell_list=numcell_list
)

print("Tumor sample-level MI summary:")
display(MI_mean_pd_tumor_sample)

print("Non-tumor sample-level MI summary:")
display(MI_mean_pd_nontumor_sample)

In [ ]:
import gc
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns


# ============================================================
# Step 3. Define reusable heatmap plotting functions
# ============================================================
def format_sample_labels(sample_ids):
    """
    Simplify sample labels for plotting.
    """
    sample_ids_new = []
    for sample_id in sample_ids:
        sample_id_new = str(sample_id)

        if "Patient" in sample_id_new:
            sample_id_new = sample_id_new.split("Patient")[0].rstrip("_- ")

        sample_id_new = sample_id_new.replace("Human", "").replace("Cancer", "").strip()
        sample_ids_new.append(sample_id_new)

    return sample_ids_new


def plot_mi_sample_heatmap(
    mi_sample_df,
    mi_order,
    xlabel,
    out_prefix,
    file_savepath_main,
    vmax=0.5,
    highlight_threshold=0.2,
    cmap="inferno",
    sort_sum_threshold=0.02,
    sample_order=None,
    return_plot_matrix=False,
):
    """
    Plot a sample-level MI heatmap and highlight cells above a threshold.

    Rows of mi_sample_df are samples.
    Columns of mi_sample_df are MI dimensions.

    If sample_order is None:
        samples are sorted by thresholded MI sum.
    If sample_order is provided:
        samples are forced to follow the provided order.

    Returns:
        data_plot: rows = MI, columns = samples
        sample_order_use: sample order used in the heatmap
    """
    mi_sample_df_use = mi_sample_df.copy()

    # Keep only MI columns in requested order
    mi_order_use = [mi for mi in mi_order if mi in mi_sample_df_use.columns]
    mi_sample_df_use = mi_sample_df_use.loc[:, mi_order_use]

    # Determine or reuse sample order
    if sample_order is None:
        sample_sort_score = (
            mi_sample_df_use
            .where(mi_sample_df_use > sort_sum_threshold, 0)
            .sum(axis=1, skipna=True)
        )

        sample_order_use = (
            sample_sort_score
            .sort_values(ascending=False)
            .index
            .tolist()
        )
    else:
        sample_order_use = [
            sample for sample in sample_order
            if sample in mi_sample_df_use.index
        ]

        sample_sort_score = (
            mi_sample_df_use
            .where(mi_sample_df_use > sort_sum_threshold, 0)
            .sum(axis=1, skipna=True)
        )

    mi_sample_df_use = mi_sample_df_use.loc[sample_order_use, :]

    # Save reordered matrix
    mi_sample_df_use.to_csv(
        file_savepath_main + "/" + f"{out_prefix}_matrix.csv",
        index=True
    )

    sample_sort_score.loc[sample_order_use].to_csv(
        file_savepath_main + "/" + f"{out_prefix}_sample_sort_score_threshold_{sort_sum_threshold}.csv",
        index=True,
        header=["thresholded_MI_sum"]
    )

    # Prepare plotting matrix: rows = MI, columns = samples
    data_plot = mi_sample_df_use.T

    ytick_labels = [
        str(mi).replace("MI_", "MI-")
        for mi in data_plot.index
    ]

    xtick_labels = format_sample_labels(data_plot.columns)

    fig, ax = plt.subplots(figsize=(5, 10))

    hm = sns.heatmap(
        data_plot,
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        ax=ax,
        linewidths=0.30,
        linecolor="#E3E5E6",
        cbar=True,
        rasterized=True,
        cbar_kws=dict(shrink=0.85, aspect=25, pad=0.03)
    )

    arr = np.asarray(data_plot.values, dtype=float)
    n_rows, n_cols = arr.shape

    for i in range(n_rows):
        for j in range(n_cols):
            if np.isfinite(arr[i, j]) and (arr[i, j] > highlight_threshold):
                rect = patches.Rectangle(
                    (j, i), 1, 1,
                    fill=False,
                    edgecolor="#00F7FF",
                    linewidth=2.0
                )
                ax.add_patch(rect)

    ax.set_yticklabels(ytick_labels, fontsize=12, rotation=0)
    ax.set_xticklabels(xtick_labels, fontsize=11, rotation=60, ha="left")

    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.tick_params(
        axis="x",
        top=True,
        bottom=False,
        labeltop=True,
        labelbottom=False,
        pad=2
    )
    ax.tick_params(axis="y", left=True, right=False, pad=2)

    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    ax.set_ylabel("Meta-interaction IDs", fontsize=14, labelpad=10)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10, width=0.8, length=3)

    fig.tight_layout()

    fig.savefig(
        file_savepath_main + "/" + f"{out_prefix}.pdf",
        format="pdf",
        bbox_inches="tight",
        transparent=True
    )

    fig.savefig(
        file_savepath_main + "/" + f"{out_prefix}.png",
        format="png",
        bbox_inches="tight",
        dpi=300,
        transparent=True
    )

    plt.show()
    plt.close(fig)

    data_plot_return = data_plot.copy()
    sample_order_return = list(sample_order_use)

    del arr, hm, fig, ax
    del mi_sample_df_use, ytick_labels, xtick_labels, mi_order_use
    del sample_sort_score
    gc.collect()

    if return_plot_matrix:
        return data_plot_return, sample_order_return

    return None


def plot_mi_log2_ratio_heatmap(
    numerator_plot_matrix,
    denominator_plot_matrix,
    xlabel,
    out_prefix,
    file_savepath_main,
    pseudocount=1e-4,
    vmax=None,
    cmap="vlag",
    highlight_log2fc=1.0,
):
    """
    Plot element-wise log2 ratio heatmap.

    numerator_plot_matrix and denominator_plot_matrix should both have:
        rows = MI
        columns = samples

    Example:
        log2(tumor / non-tumor)
    """
    common_mi = [
        mi for mi in numerator_plot_matrix.index
        if mi in denominator_plot_matrix.index
    ]

    common_samples = [
        sample for sample in numerator_plot_matrix.columns
        if sample in denominator_plot_matrix.columns
    ]

    numerator_use = numerator_plot_matrix.loc[common_mi, common_samples]
    denominator_use = denominator_plot_matrix.loc[common_mi, common_samples]

    log2_ratio = np.log2(
        (numerator_use.astype(float) + pseudocount)
        / (denominator_use.astype(float) + pseudocount)
    )

    log2_ratio = pd.DataFrame(
        log2_ratio,
        index=common_mi,
        columns=common_samples
    )

    log2_ratio.to_csv(
        file_savepath_main + "/" + f"{out_prefix}_matrix.csv",
        index=True
    )

    if vmax is None:
        vmax_use = np.nanpercentile(np.abs(log2_ratio.values), 98)
        vmax_use = max(float(vmax_use), 1.0)
    else:
        vmax_use = float(vmax)

    ytick_labels = [
        str(mi).replace("MI_", "MI-")
        for mi in log2_ratio.index
    ]

    xtick_labels = format_sample_labels(log2_ratio.columns)

    fig, ax = plt.subplots(figsize=(5, 10))

    hm = sns.heatmap(
        log2_ratio,
        cmap=cmap,
        vmin=-vmax_use,
        vmax=vmax_use,
        center=0,
        ax=ax,
        linewidths=0.30,
        linecolor="#E3E5E6",
        cbar=True,
        rasterized=True,
        cbar_kws=dict(shrink=0.85, aspect=25, pad=0.03)
    )

    arr = np.asarray(log2_ratio.values, dtype=float)
    n_rows, n_cols = arr.shape

    for i in range(n_rows):
        for j in range(n_cols):
            if np.isfinite(arr[i, j]) and ((arr[i, j]) > highlight_log2fc):
                rect = patches.Rectangle(
                    (j, i), 1, 1,
                    fill=False,
                    edgecolor="#00F7FF",
                    linewidth=2.0
                )
                ax.add_patch(rect)

    ax.set_yticklabels(ytick_labels, fontsize=12, rotation=0)
    ax.set_xticklabels(xtick_labels, fontsize=11, rotation=60, ha="left")

    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.tick_params(
        axis="x",
        top=True,
        bottom=False,
        labeltop=True,
        labelbottom=False,
        pad=2
    )
    ax.tick_params(axis="y", left=True, right=False, pad=2)

    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    ax.set_ylabel("Meta-interaction IDs", fontsize=14, labelpad=10)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10, width=0.8, length=3)
    cbar.set_label("log2(tumor / non-tumor)", fontsize=12)

    fig.tight_layout()

    fig.savefig(
        file_savepath_main + "/" + f"{out_prefix}.pdf",
        format="pdf",
        bbox_inches="tight",
        transparent=True
    )

    fig.savefig(
        file_savepath_main + "/" + f"{out_prefix}.png",
        format="png",
        bbox_inches="tight",
        dpi=300,
        transparent=True
    )

    plt.show()
    plt.close(fig)

    return log2_ratio


# ============================================================
# Step 4. Set plotting style
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

sns.set_theme(style="white", context="paper")

cmap = plt.get_cmap("YlOrRd")
# 
# 
# # ============================================================
# # Step 5. Plot tumor sample-level MI heatmap
# # ============================================================
# plot_mi_sample_heatmap(
#     mi_sample_df=MI_mean_pd_tumor_sample,
#     mi_order=mi_order,
#     xlabel="Cancer sample ID",
#     out_prefix="MI_heatmap_tumor_sampleagg",
#     file_savepath_main=str(run_dirs["run_dir"]),
#     vmax=0.2,
#     highlight_threshold=MI_intensity_threshold,
#     cmap=cmap,
#     sort_sum_threshold=0.03
# )

In [ ]:
##Replace the columns names "MI_" to "MI-" in both MI_mean_pd_tumor_sample and MI_mean_pd_nontumor_sample
MI_mean_pd_tumor_sample.columns = [
    str(col).replace("MI_", "MI-") for col in MI_mean_pd_tumor_sample.columns
]
MI_mean_pd_nontumor_sample.columns = [
    str(col).replace("MI_", "MI-") for col in MI_mean_pd_nontumor_sample.columns
]

In [ ]:
# ============================================================
# Step 4.5. Align tumor and non-tumor matrices before plotting
# ============================================================
mi_order_common = [
    mi for mi in mi_order
    if (mi in MI_mean_pd_tumor_sample.columns)
    and (mi in MI_mean_pd_nontumor_sample.columns)
]

common_sample_ids = [
    sample for sample in MI_mean_pd_tumor_sample.index
    if sample in MI_mean_pd_nontumor_sample.index
]

MI_mean_pd_tumor_sample_aligned = MI_mean_pd_tumor_sample.loc[
    common_sample_ids, mi_order_common
].copy()

MI_mean_pd_nontumor_sample_aligned = MI_mean_pd_nontumor_sample.loc[
    common_sample_ids, mi_order_common
].copy()


# ============================================================
# Step 5. Plot tumor sample-level MI heatmap
# ============================================================
MI_tumor_plot_matrix, shared_sample_order = plot_mi_sample_heatmap(
    mi_sample_df=MI_mean_pd_tumor_sample_aligned,
    mi_order=mi_order_common,
    xlabel="Cancer sample ID",
    out_prefix="MI_heatmap_tumor_sampleagg",
    file_savepath_main=str(run_dirs["run_dir"]),
    vmax=0.2,
    # vmax=None,
    # highlight_threshold=MI_intensity_threshold,
    highlight_threshold=1,
    cmap=cmap,
    sort_sum_threshold=0.03,
    sample_order=None,
    return_plot_matrix=True,
)

In [ ]:
# ============================================================
# Step 5.5. Tumor-involved sample-level MI heatmap using
#           max sender→receiver cell-type-pair mean
# ------------------------------------------------------------
# Previous tumor heatmap:
#   each element = mean MI over all tumor-involved edges in a sample
#
# Pair-maximum heatmap:
#   for each sample and each MI:
#       1) subset tumor-involved edges
#       2) group by sender→receiver cell-type pair
#       3) compute mean MI within each pair
#       4) take the maximum pair-level mean MI
#
# This avoids sample-level values being dominated by differences in
# cell-type-pair abundance.
#
# IMPORTANT:
#   Correct native MI mapping in this Pan-cancer notebook:
#       Factor_envir_use[:, 0] -> MI-1
#       Factor_envir_use[:, 1] -> MI-2
#       Factor_envir_use[:, 2] -> MI-3
#       ...
# ============================================================

import gc
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# Parameters
# ============================================================

MIN_EDGES_PER_CELLTYPE_PAIR_TUMOR_MAX = 20

TUMOR_PAIRMAX_OUT_PREFIX = "MI_heatmap_tumor_sampleagg_celltypepairmax_fixedMI_1based"

# Correct for this notebook:
# Factor_envir_use[:, 0] -> MI-1
NATIVE_MI_INDEX_BASE_TUMOR_PAIRMAX = 1

output_dir_tumor_pairmax = Path(run_dirs["run_dir"])


# ============================================================
# Helper functions
# ============================================================

def _to_numpy_tumor_pairmax(x):
    """Safely convert torch / numpy / list-like object to numpy array."""
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    elif hasattr(x, "cpu"):
        try:
            x = x.cpu().numpy()
        except Exception:
            pass
    return np.asarray(x)


def _standardize_mi_name_tumor_pairmax(mi):
    """
    Standardize MI label format without changing MI identity.
    Examples:
      MI_2 -> MI-2
      MI2  -> MI-2
      MI-2 -> MI-2
    """
    mi = str(mi)
    mi = mi.replace("MI_", "MI-")
    if mi.startswith("MI-"):
        return mi
    if mi.startswith("MI"):
        return mi.replace("MI", "MI-", 1)
    return mi


def _natural_mi_order_tumor_pairmax(mi_name):
    """Natural numeric ordering for MI labels."""
    try:
        return int(
            str(mi_name)
            .replace("MI-", "")
            .replace("MI_", "")
            .replace("MI", "")
        )
    except Exception:
        return 10**9


def ensure_edge_index_e_by_2_tumor_pairmax(edge_index):
    """Ensure edge_index has shape E x 2."""
    edge_index = _to_numpy_tumor_pairmax(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(np.int64, copy=False)

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def collect_edge_metadata_for_tumor_pairmax(processed):
    """
    Collect edge-level metadata aligned with Factor_envir_use.

    Returns
    -------
    edge_slice_id : np.ndarray
    edge_sample_id : np.ndarray
        sample_id = slice_id.split("_")[0], matching previous sample-level logic.
    sender_celltype : np.ndarray
    receiver_celltype : np.ndarray
    edge_pair : np.ndarray
        sender→receiver cell-type-pair label.
    edge_has_cancer_cell : np.ndarray[bool]
        Whether sender or receiver is a cancer cell.
    """
    edge_slice_list = []
    sender_celltype_list = []
    receiver_celltype_list = []

    for batch_idx in range(len(processed.spidernet_data)):
        data_cur = processed.spidernet_data[batch_idx]

        edge_index_cur = ensure_edge_index_e_by_2_tumor_pairmax(data_cur["edge_index"])

        sample_cur = _to_numpy_tumor_pairmax(data_cur["sample"]).astype(str).reshape(-1)
        celltype_cur = _to_numpy_tumor_pairmax(data_cur["cell_class"]).astype(str).reshape(-1)

        # Usually sample_cur is per cell. If it is one value per batch, broadcast it.
        if sample_cur.size == 1:
            edge_slice_cur = np.repeat(sample_cur[0], edge_index_cur.shape[0])
        else:
            edge_slice_cur = sample_cur[edge_index_cur[:, 0]]

        sender_celltype_cur = celltype_cur[edge_index_cur[:, 0]]
        receiver_celltype_cur = celltype_cur[edge_index_cur[:, 1]]

        edge_slice_list.append(edge_slice_cur)
        sender_celltype_list.append(sender_celltype_cur)
        receiver_celltype_list.append(receiver_celltype_cur)

    edge_slice_id = np.hstack(edge_slice_list).astype(str)
    sender_celltype = np.hstack(sender_celltype_list).astype(str)
    receiver_celltype = np.hstack(receiver_celltype_list).astype(str)

    edge_sample_id = np.asarray(
        [str(x).split("_")[0] for x in edge_slice_id],
        dtype=str,
    )

    edge_pair = np.char.add(
        np.char.add(sender_celltype.astype(str), " -> "),
        receiver_celltype.astype(str),
    )

    sender_lower = np.char.lower(sender_celltype.astype(str))
    receiver_lower = np.char.lower(receiver_celltype.astype(str))

    edge_has_cancer_cell = np.logical_or(
        np.char.find(sender_lower, "cancercell") >= 0,
        np.char.find(receiver_lower, "cancercell") >= 0,
    )

    return (
        edge_slice_id,
        edge_sample_id,
        sender_celltype,
        receiver_celltype,
        edge_pair,
        edge_has_cancer_cell,
    )


def resolve_native_mi_columns_tumor_pairmax(n_mi):
    """
    Define native MI labels according to Factor_envir_use column order.

    Correct mapping:
        column 0 -> MI-1
        column 1 -> MI-2
        column 2 -> MI-3
        ...
    """
    mi_columns = [
        f"MI-{i + int(NATIVE_MI_INDEX_BASE_TUMOR_PAIRMAX)}"
        for i in range(n_mi)
    ]

    print("Native Factor_envir_use column mapping for tumor pairmax heatmap:")
    for i, mi in enumerate(mi_columns[:20]):
        print(f"  column {i}: {mi}")
    if n_mi > 20:
        print(f"  ... total {n_mi} MIs")

    return mi_columns


def resolve_plot_mi_order_tumor_pairmax(mi_columns):
    """
    Resolve plotting order only.

    This function never changes MI identity. It only controls row order
    in the final heatmap.
    """
    native_set = set(mi_columns)
    candidate_order = None

    if "mi_order_common" in globals():
        candidate_order = list(mi_order_common)
        print("Using `mi_order_common` only as tumor-pairmax heatmap plotting order.")
    elif "mi_order" in globals():
        candidate_order = list(mi_order)
        print("Using `mi_order` only as tumor-pairmax heatmap plotting order.")
    elif "MI_tumor_plot_matrix" in globals():
        candidate_order = list(MI_tumor_plot_matrix.index)
        print("Using `MI_tumor_plot_matrix.index` only as tumor-pairmax heatmap plotting order.")

    if candidate_order is not None:
        candidate_order = [_standardize_mi_name_tumor_pairmax(x) for x in candidate_order]
        candidate_order = [x for x in candidate_order if x in native_set]

        missing = [x for x in mi_columns if x not in set(candidate_order)]
        missing = sorted(missing, key=_natural_mi_order_tumor_pairmax)

        mi_order_use = candidate_order + missing
    else:
        mi_order_use = sorted(mi_columns, key=_natural_mi_order_tumor_pairmax)

    print("\nTumor-pairmax heatmap MI plotting order:")
    print(mi_order_use)

    return mi_order_use


def compute_tumor_pairmax_mi_by_sample(
    factor_matrix,
    edge_sample_id,
    edge_pair,
    tumor_involved_mask,
    mi_columns,
    sample_order=None,
    min_edges_per_pair=20,
):
    """
    For each sample and each MI:
      1) subset tumor-involved edges;
      2) group by sender→receiver cell-type pair;
      3) compute mean MI for each pair;
      4) take maximum pair-level mean MI.

    Returns
    -------
    pairmax_df : sample x MI matrix
    selected_pair_df : sample x MI selected pair label
    selected_pair_nedge_df : sample x MI selected pair edge count
    selected_long_df : long-format annotation table
    """
    factor_matrix = np.asarray(factor_matrix)

    if sample_order is None:
        sample_ids = np.sort(np.unique(edge_sample_id.astype(str))).tolist()
    else:
        sample_ids = [
            str(x)
            for x in sample_order
            if str(x) in set(edge_sample_id.astype(str))
        ]

    n_mi = factor_matrix.shape[1]

    if len(mi_columns) != n_mi:
        raise ValueError(
            f"Length of mi_columns ({len(mi_columns)}) does not match "
            f"number of MI columns in factor_matrix ({n_mi})."
        )

    pairmax_df = pd.DataFrame(
        np.nan,
        index=sample_ids,
        columns=mi_columns,
        dtype=float,
    )

    selected_pair_df = pd.DataFrame(
        "",
        index=sample_ids,
        columns=mi_columns,
        dtype=object,
    )

    selected_pair_nedge_df = pd.DataFrame(
        0,
        index=sample_ids,
        columns=mi_columns,
        dtype=int,
    )

    selected_rows = []

    for sample_id in sample_ids:
        edge_idx_cur = np.where(
            (edge_sample_id.astype(str) == str(sample_id))
            & tumor_involved_mask
        )[0]

        if edge_idx_cur.size == 0:
            continue

        pair_cur = edge_pair[edge_idx_cur].astype(str)

        pair_names, pair_inverse = np.unique(pair_cur, return_inverse=True)
        pair_counts = np.bincount(pair_inverse, minlength=len(pair_names)).astype(int)

        valid_pair_mask = pair_counts >= int(min_edges_per_pair)

        if valid_pair_mask.sum() == 0:
            continue

        # Sum MI values within each sender→receiver pair.
        pair_sums = np.zeros((len(pair_names), n_mi), dtype=np.float64)
        np.add.at(pair_sums, pair_inverse, factor_matrix[edge_idx_cur, :])

        pair_means = pair_sums / np.maximum(pair_counts[:, np.newaxis], 1)

        # Remove rare pairs from max selection.
        pair_means[~valid_pair_mask, :] = np.nan

        finite_mi_mask = np.isfinite(pair_means).any(axis=0)

        for mi_idx, mi_name in enumerate(mi_columns):
            if not finite_mi_mask[mi_idx]:
                continue

            pair_mean_cur = pair_means[:, mi_idx]
            best_pair_idx = int(np.nanargmax(pair_mean_cur))

            best_value = float(pair_mean_cur[best_pair_idx])
            best_pair = str(pair_names[best_pair_idx])
            best_pair_nedge = int(pair_counts[best_pair_idx])

            pairmax_df.loc[sample_id, mi_name] = best_value
            selected_pair_df.loc[sample_id, mi_name] = best_pair
            selected_pair_nedge_df.loc[sample_id, mi_name] = best_pair_nedge

            selected_rows.append({
                "sample_id": sample_id,
                "MI": mi_name,
                "MI_native_column_index": int(mi_idx),
                "selected_sender_receiver_pair": best_pair,
                "selected_pair_mean_MI": best_value,
                "selected_pair_n_edges": best_pair_nedge,
                "min_edges_per_pair": int(min_edges_per_pair),
                "n_valid_pairs_in_sample": int(valid_pair_mask.sum()),
                "n_tumor_involved_edges_in_sample": int(edge_idx_cur.size),
            })

    selected_long_df = pd.DataFrame(selected_rows)

    return pairmax_df, selected_pair_df, selected_pair_nedge_df, selected_long_df


# ============================================================
# Step 1. Re-collect edge metadata and align Factor_envir_use
# ============================================================

(
    edge_slice_id_tumor_pairmax,
    edge_sample_id_tumor_pairmax,
    sender_celltype_tumor_pairmax,
    receiver_celltype_tumor_pairmax,
    edge_celltype_pair_tumor_pairmax,
    edge_has_cancer_cell_tumor_pairmax,
) = collect_edge_metadata_for_tumor_pairmax(processed)

factor_matrix_tumor_pairmax = np.asarray(Factor_envir_use)

if factor_matrix_tumor_pairmax.ndim != 2:
    raise ValueError(
        f"Factor_envir_use must be 2D, got shape {factor_matrix_tumor_pairmax.shape}"
    )

if factor_matrix_tumor_pairmax.shape[0] != edge_sample_id_tumor_pairmax.shape[0]:
    if factor_matrix_tumor_pairmax.shape[1] == edge_sample_id_tumor_pairmax.shape[0]:
        print("Detected Factor_envir_use as MI x edge. Transposing to edge x MI.")
        factor_matrix_tumor_pairmax = factor_matrix_tumor_pairmax.T
    else:
        raise ValueError(
            "Factor_envir_use is not aligned with collected edges.\n"
            f"Factor_envir_use shape: {factor_matrix_tumor_pairmax.shape}\n"
            f"Number of collected edges: {edge_sample_id_tumor_pairmax.shape[0]}"
        )

mi_columns_tumor_pairmax = resolve_native_mi_columns_tumor_pairmax(
    n_mi=factor_matrix_tumor_pairmax.shape[1]
)

mi_order_tumor_pairmax = resolve_plot_mi_order_tumor_pairmax(
    mi_columns=mi_columns_tumor_pairmax
)

# Use previous shared_sample_order whenever available.
if "shared_sample_order" in globals():
    sample_order_tumor_pairmax = [
        str(x)
        for x in shared_sample_order
        if str(x) in set(edge_sample_id_tumor_pairmax.astype(str))
    ]
else:
    sample_order_tumor_pairmax = np.sort(
        np.unique(edge_sample_id_tumor_pairmax.astype(str))
    ).tolist()

print("\nNumber of edges:", factor_matrix_tumor_pairmax.shape[0])
print("Number of tumor-involved edges:", int(edge_has_cancer_cell_tumor_pairmax.sum()))
print("Number of MIs:", factor_matrix_tumor_pairmax.shape[1])
print("Number of samples:", len(sample_order_tumor_pairmax))
print("MIN_EDGES_PER_CELLTYPE_PAIR_TUMOR_MAX:", MIN_EDGES_PER_CELLTYPE_PAIR_TUMOR_MAX)


# ============================================================
# Step 2. Compute sample-level tumor pairmax MI matrix
# ============================================================

(
    MI_mean_pd_tumor_pairmax_sample,
    MI_tumor_pairmax_selected_pair,
    MI_tumor_pairmax_selected_nedge,
    MI_tumor_pairmax_selected_long,
) = compute_tumor_pairmax_mi_by_sample(
    factor_matrix=factor_matrix_tumor_pairmax,
    edge_sample_id=edge_sample_id_tumor_pairmax,
    edge_pair=edge_celltype_pair_tumor_pairmax,
    tumor_involved_mask=edge_has_cancer_cell_tumor_pairmax,
    mi_columns=mi_columns_tumor_pairmax,
    sample_order=sample_order_tumor_pairmax,
    min_edges_per_pair=MIN_EDGES_PER_CELLTYPE_PAIR_TUMOR_MAX,
)


# ============================================================
# Step 3. Align matrix before plotting
# ============================================================

mi_order_tumor_pairmax = [
    str(mi)
    for mi in mi_order_tumor_pairmax
    if str(mi) in MI_mean_pd_tumor_pairmax_sample.columns
]

sample_order_tumor_pairmax = [
    str(sample)
    for sample in sample_order_tumor_pairmax
    if str(sample) in MI_mean_pd_tumor_pairmax_sample.index
]

MI_mean_pd_tumor_pairmax_sample_aligned = (
    MI_mean_pd_tumor_pairmax_sample
    .loc[sample_order_tumor_pairmax, mi_order_tumor_pairmax]
    .copy()
)



In [ ]:
# ============================================================
# Step 4. Plot tumor sample-level pairmax MI heatmap
# ============================================================

MI_tumor_pairmax_plot_matrix, shared_sample_order_tumor_pairmax = plot_mi_sample_heatmap(
    mi_sample_df=MI_mean_pd_tumor_pairmax_sample_aligned,
    mi_order=mi_order_tumor_pairmax,
    xlabel="Cancer sample ID",
    out_prefix=TUMOR_PAIRMAX_OUT_PREFIX,
    file_savepath_main=str(output_dir_tumor_pairmax),
    vmax=0.5,
    highlight_threshold=1,
    cmap=cmap,
    sort_sum_threshold=0.03,
    sample_order=sample_order_tumor_pairmax,
    return_plot_matrix=True,
)


# ============================================================
# Step 5. Save outputs
# ============================================================

MI_mean_pd_tumor_pairmax_sample.to_csv(
    output_dir_tumor_pairmax / "MI_tumor_involved_celltypepairmax_sample_matrix_fixedMI_1based.csv"
)

MI_mean_pd_tumor_pairmax_sample_aligned.to_csv(
    output_dir_tumor_pairmax / "MI_tumor_involved_celltypepairmax_sample_matrix_aligned_fixedMI_1based.csv"
)

MI_tumor_pairmax_selected_pair.to_csv(
    output_dir_tumor_pairmax / "MI_tumor_involved_celltypepairmax_selected_pair_fixedMI_1based.csv"
)

MI_tumor_pairmax_selected_nedge.to_csv(
    output_dir_tumor_pairmax / "MI_tumor_involved_celltypepairmax_selected_pair_nedges_fixedMI_1based.csv"
)

MI_tumor_pairmax_selected_long.to_csv(
    output_dir_tumor_pairmax / "MI_tumor_involved_celltypepairmax_selected_pair_long_table_fixedMI_1based.csv",
    index=False,
)

print("\nSaved tumor pairmax sample matrix and selected-pair annotations.")
print("Tumor pairmax sample matrix:")
display(MI_mean_pd_tumor_pairmax_sample_aligned)

print("Selected tumor-involved sender→receiver pair for each sample and MI:")
display(
    MI_tumor_pairmax_selected_pair
    .loc[sample_order_tumor_pairmax, mi_order_tumor_pairmax]
)

print("Selected tumor-involved pair edge counts for each sample and MI:")
display(
    MI_tumor_pairmax_selected_nedge
    .loc[sample_order_tumor_pairmax, mi_order_tumor_pairmax]
)


# ============================================================
# Optional diagnostics for key MIs
# ============================================================

diagnostic_mis_tumor_pairmax = [
    mi for mi in ["MI-4", "MI-6"]
    if mi in set(mi_columns_tumor_pairmax)
]

if len(diagnostic_mis_tumor_pairmax) > 0:
    print("\nDiagnostic selected tumor-involved pairs for key MIs:")
    display(
        MI_tumor_pairmax_selected_long
        .loc[MI_tumor_pairmax_selected_long["MI"].isin(diagnostic_mis_tumor_pairmax)]
        .sort_values(["MI", "sample_id"])
        .head(100)
    )


# ============================================================
# Optional cleanup
# ============================================================

if "release_memory" in globals():
    release_memory(
        "edge_slice_id_tumor_pairmax",
        "edge_sample_id_tumor_pairmax",
        "sender_celltype_tumor_pairmax",
        "receiver_celltype_tumor_pairmax",
        "edge_celltype_pair_tumor_pairmax",
        "edge_has_cancer_cell_tumor_pairmax",
        "factor_matrix_tumor_pairmax",
        namespace=globals(),
        run_gc=True,
        clear_cuda=False,
        close_figures=False,
    )
else:
    gc.collect()


In [ ]:
# ============================================================
# Step 4.5. Align tumor and non-tumor matrices before plotting
# ============================================================
mi_order_common = [
    mi for mi in mi_order
    if (mi in MI_mean_pd_tumor_sample.columns)
    and (mi in MI_mean_pd_nontumor_sample.columns)
]

common_sample_ids = [
    sample for sample in MI_mean_pd_tumor_sample.index
    if sample in MI_mean_pd_nontumor_sample.index
]

MI_mean_pd_tumor_sample_aligned = MI_mean_pd_tumor_sample.loc[
    common_sample_ids, mi_order_common
].copy()

MI_mean_pd_nontumor_sample_aligned = MI_mean_pd_nontumor_sample.loc[
    common_sample_ids, mi_order_common
].copy()


# ============================================================
# Step 4.6. Heatmap-row-wise max normalization
# Original matrix: sample x MI
# Plotted heatmap: MI x sample
# Therefore, normalize each MI column by its max across samples
# ============================================================
mi_col_max = MI_mean_pd_tumor_sample_aligned.max(axis=0).replace(0, np.nan)

MI_mean_pd_tumor_sample_aligned_rowmax = (
    MI_mean_pd_tumor_sample_aligned
    .div(mi_col_max, axis=1)
    .fillna(0)
)


shared_sample_order = ["HumanBreastCancerPatient1",
                                                                                     "HumanLiverCancerPatient2",
                                                                                     "HumanColonCancerPatient2",
                                                                                     "HumanOvarianCancerPatient2Slice3",
                                                                                     "HumanUterineCancerPatient1",
                                                                                     "HumanMelanomaCancerPatient1",
                                                                                     "HumanLungCancerPatient1",
                                                                                     "HumanProstateCancerPatient1"]
MI_mean_pd_tumor_sample_aligned_rowmax = MI_mean_pd_tumor_sample_aligned_rowmax.loc[shared_sample_order,:]
# ============================================================
# Step 5. Plot tumor sample-level MI heatmap
# ============================================================
MI_tumor_plot_matrix_max, shared_sample_order = plot_mi_sample_heatmap(
    mi_sample_df=MI_mean_pd_tumor_sample_aligned_rowmax,
    mi_order=mi_order_common,
    xlabel="Cancer sample ID",
    out_prefix="MI_heatmap_tumor_sampleagg_rowmax",
    file_savepath_main=str(run_dirs["run_dir"]),
    vmax=1,
    highlight_threshold=1,
    cmap=cmap,
    sort_sum_threshold=0.03,
    sample_order=shared_sample_order,
    return_plot_matrix=True,
)

In [ ]:
# ============================================================
# Step 5.5. Fit GMMs with 2, 3, 4, and 5 components to flattened
#          tumor heatmap elements and define thresholds by the
#          boundary where the two lowest-mean components have
#          pairwise proportions:
#              lowest : second-lowest = 1/1000 : 999/1000
# ============================================================
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from scipy.optimize import brentq

# ============================================================
# Nature-style plotting settings
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

sns.set_theme(style="white", context="paper")


# ============================================================
# Step 1. Flatten tumor heatmap matrix
# ============================================================
# MI_tumor_plot_matrix: rows = MI, columns = samples
tumor_heatmap_values = np.asarray(
    MI_tumor_plot_matrix.values,
    dtype=float
).reshape(-1)

tumor_heatmap_values = tumor_heatmap_values[np.isfinite(tumor_heatmap_values)]
tumor_heatmap_values = tumor_heatmap_values.reshape(-1, 1)

if tumor_heatmap_values.shape[0] < 10:
    raise ValueError(
        "Too few finite tumor heatmap values for GMM fitting."
    )

x_values = tumor_heatmap_values.reshape(-1)

print(f"Number of flattened heatmap elements: {x_values.size}")
print(f"Mean:   {np.mean(x_values):.4f}")
print(f"Median: {np.median(x_values):.4f}")
print(f"Min:    {np.min(x_values):.4f}")
print(f"Max:    {np.max(x_values):.4f}")


# ============================================================
# Step 2. Helper functions
# ============================================================
def normal_pdf(x, mean, sd):
    """
    Univariate Gaussian density.
    """
    sd = max(float(sd), 1e-8)
    return (
        1.0 / (sd * np.sqrt(2.0 * np.pi))
        * np.exp(-0.5 * ((x - mean) / sd) ** 2)
    )


def component_weighted_density(x, weight, mean, sd):
    """
    Weighted component density: pi_k * N(x | mu_k, sigma_k).
    """
    return float(weight) * normal_pdf(x, mean, sd)


def find_low_second_boundary(
    weights,
    means,
    stds,
    low_comp,
    second_low_comp,
    x_min,
    x_max,
    target_low_prop=1.0 / 1000.0,
    n_grid=50000,
):
    """
    Find the threshold between the two lowest-mean components where:

        prop_low    = target_low_prop
        prop_second = 1 - target_low_prop

    Specifically:
        prop_second = d_second / (d_low + d_second)

    and the target is:
        prop_second = 1 - target_low_prop

    With target_low_prop = 1/1000, this gives:
        prop_low : prop_second = 0.001 : 0.999

    This defines a stricter threshold than the previous 0.5 / 0.5 boundary.
    """
    target_second_prop = 1.0 - float(target_low_prop)

    low_mean = float(means[low_comp])
    second_mean = float(means[second_low_comp])

    def pairwise_second_prop(x):
        d_low = component_weighted_density(
            x,
            weights[low_comp],
            means[low_comp],
            stds[low_comp]
        )
        d_second = component_weighted_density(
            x,
            weights[second_low_comp],
            means[second_low_comp],
            stds[second_low_comp]
        )
        return d_second / max(d_low + d_second, 1e-300)

    def objective(x):
        return pairwise_second_prop(x) - target_second_prop

    # Search from the lower-mean component to the observed maximum.
    # This is more robust than only searching between the two means because
    # the 0.999 boundary is usually to the right of the 0.5 boundary.
    left = min(low_mean, second_mean)
    right = float(x_max)

    try:
        grid = np.linspace(left, right, n_grid)
        obj_grid = np.array([objective(x) for x in grid], dtype=float)

        finite_mask = np.isfinite(obj_grid)
        grid = grid[finite_mask]
        obj_grid = obj_grid[finite_mask]

        if grid.size < 2:
            raise ValueError("No finite objective values for threshold search.")

        sign_change_idx = np.where(obj_grid[:-1] * obj_grid[1:] <= 0)[0]

        if sign_change_idx.size > 0:
            # Use the first crossing after the lowest component.
            idx = int(sign_change_idx[0])
            threshold = brentq(objective, grid[idx], grid[idx + 1])
            method = (
                "brentq_pairwise_prop_low_0.001_second_0.999"
            )
        else:
            raise ValueError("No sign change found in observed value range.")

    except Exception:
        # Fallback: use the closest observed-range grid point to the target prop.
        grid = np.linspace(float(x_min), float(x_max), n_grid)

        d_low = weights[low_comp] * normal_pdf(
            grid,
            means[low_comp],
            stds[low_comp]
        )
        d_second = weights[second_low_comp] * normal_pdf(
            grid,
            means[second_low_comp],
            stds[second_low_comp]
        )

        pair_prop_second = d_second / np.maximum(d_low + d_second, 1e-300)

        threshold = float(
            grid[np.argmin(np.abs(pair_prop_second - target_second_prop))]
        )
        method = (
            "grid_closest_pairwise_prop_low_0.001_second_0.999"
        )

    return float(threshold), method


def extract_gmm_parameters(gmm):
    """
    Extract weights, means and standard deviations from a 1D full-covariance GMM.
    """
    weights = gmm.weights_.copy()
    means = gmm.means_.reshape(-1).copy()

    covariances = np.asarray(gmm.covariances_)
    if covariances.ndim == 3:
        variances = covariances[:, 0, 0]
    else:
        variances = covariances.reshape(-1)

    variances = np.maximum(variances, 1e-12)
    stds = np.sqrt(variances)

    return weights, means, stds


# ============================================================
# Step 3. Fit GMMs for K = 2, 3, 4, 5
# ============================================================
n_components_list = [2, 3, 4, 5]

# target_low_prop = 1.0 / 1000.0
target_low_prop = 1.0 / 10000.0
target_second_prop = 1.0 - target_low_prop

x_min = float(np.min(x_values))
x_max = float(np.max(x_values))
x_range = max(x_max - x_min, 1e-6)

x_grid = np.linspace(
    max(0, x_min - 0.05 * x_range),
    x_max + 0.05 * x_range,
    1000
)

# Component colors for up to 5 components after sorting by mean
component_colors = [
    "#377EB8",  # lowest mean component
    "#4DAF4A",  # second lowest mean component
    "#984EA3",
    "#FF7F00",
    "#E41A1C",
]

all_component_summary = []
all_threshold_summary = []
MI_intensity_threshold_by_gmm = {}

for n_components in n_components_list:

    print("\n" + "=" * 70)
    print(f"Fitting {n_components}-component GMM")
    print("=" * 70)

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        random_state=0,
        n_init=20,
        max_iter=1000
    )

    gmm.fit(tumor_heatmap_values)

    weights, means, stds = extract_gmm_parameters(gmm)

    component_order = np.argsort(means)
    low_comp = int(component_order[0])
    second_low_comp = int(component_order[1])

    threshold, threshold_method = find_low_second_boundary(
        weights=weights,
        means=means,
        stds=stds,
        low_comp=low_comp,
        second_low_comp=second_low_comp,
        x_min=x_min,
        x_max=x_max,
        target_low_prop=target_low_prop,
    )

    MI_intensity_threshold_by_gmm[n_components] = threshold

    bic_value = float(gmm.bic(tumor_heatmap_values))
    aic_value = float(gmm.aic(tumor_heatmap_values))

    print(f"BIC = {bic_value:.4f}")
    print(f"AIC = {aic_value:.4f}")
    print(
        "Threshold between the two lowest-mean components where "
        f"pairwise prop(lowest, second-lowest) = "
        f"({target_low_prop:.4g}, {target_second_prop:.4g}):"
    )
    print(f"  threshold = {threshold:.6f}")
    print(f"  method    = {threshold_method}")

    # ------------------------------------------------------------
    # Component summary
    # ------------------------------------------------------------
    for rank, comp in enumerate(component_order):
        comp = int(comp)

        if rank == 0:
            comp_label = "lowest-mean component"
        elif rank == 1:
            comp_label = "second-lowest-mean component"
        else:
            comp_label = f"higher-mean component {rank + 1}"

        all_component_summary.append({
            "n_components": int(n_components),
            "component_original_id": int(comp),
            "component_rank_by_mean": int(rank + 1),
            "component_label": comp_label,
            "weight": float(weights[comp]),
            "mean": float(means[comp]),
            "sd": float(stds[comp]),
            "bic": bic_value,
            "aic": aic_value,
            "target_lowest_component_prop": float(target_low_prop),
            "target_second_lowest_component_prop": float(target_second_prop),
            "threshold_low_vs_second_prop_0.001_0.999": float(threshold),
            "threshold_method": threshold_method,
        })

        print(
            f"  Rank {rank + 1} | Component {comp}: "
            f"weight={weights[comp]:.4f}, "
            f"mean={means[comp]:.4f}, "
            f"sd={stds[comp]:.4f}"
        )

    all_threshold_summary.append({
        "n_components": int(n_components),
        "bic": bic_value,
        "aic": aic_value,
        "lowest_mean_component": int(low_comp),
        "second_lowest_mean_component": int(second_low_comp),
        "lowest_component_mean": float(means[low_comp]),
        "second_lowest_component_mean": float(means[second_low_comp]),
        "target_lowest_component_prop": float(target_low_prop),
        "target_second_lowest_component_prop": float(target_second_prop),
        "threshold_low_vs_second_prop_0.001_0.999": float(threshold),
        "threshold_method": threshold_method,
    })

    # ------------------------------------------------------------
    # Density curves for each component
    # ------------------------------------------------------------
    component_density_dict = {}

    for rank, comp in enumerate(component_order):
        comp = int(comp)
        density_comp = weights[comp] * normal_pdf(
            x_grid,
            means[comp],
            stds[comp]
        )
        component_density_dict[comp] = density_comp

    density_low = component_density_dict[low_comp]
    density_second = component_density_dict[second_low_comp]

    pair_prop_low = density_low / np.maximum(
        density_low + density_second,
        1e-300
    )

    pair_prop_second_low = density_second / np.maximum(
        density_low + density_second,
        1e-300
    )

    # Full posterior probabilities over all components
    posterior_grid = gmm.predict_proba(x_grid.reshape(-1, 1))

    # ------------------------------------------------------------
    # Save posterior / density curve table
    # ------------------------------------------------------------
    posterior_df = pd.DataFrame({
        "x": x_grid,
        "pairwise_prop_lowest_among_two_lowest": pair_prop_low,
        "pairwise_prop_second_lowest_among_two_lowest": pair_prop_second_low,
        "target_lowest_component_prop": target_low_prop,
        "target_second_lowest_component_prop": target_second_prop,
        "threshold_low_vs_second_prop_0.001_0.999": threshold,
    })

    for rank, comp in enumerate(component_order):
        comp = int(comp)
        posterior_df[f"component_rank{rank + 1}_original_id"] = comp
        posterior_df[f"component_rank{rank + 1}_density"] = component_density_dict[comp]
        posterior_df[f"component_rank{rank + 1}_full_posterior"] = posterior_grid[:, comp]

    posterior_df.to_csv(
        str(run_dirs["run_dir"])
        + f"/MI_heatmap_tumor_sampleagg_flattened_GMM{n_components}_posterior_curve.csv",
        index=False
    )

    # ------------------------------------------------------------
    # Plot histogram + component densities
    # No dashed mixture-density line is drawn.
    # ------------------------------------------------------------
    plt.close("all")
    fig, ax = plt.subplots(figsize=(5.4, 3.9))

    sns.histplot(
        x_values,
        bins=40,
        stat="density",
        color="#D0D0D0",
        edgecolor="white",
        alpha=0.75,
        ax=ax
    )

    for rank, comp in enumerate(component_order):
        comp = int(comp)

        if rank == 0:
            label = "Lowest-mean component"
        elif rank == 1:
            label = "Second-lowest-mean component"
        else:
            label = f"Component rank {rank + 1}"

        ax.plot(
            x_grid,
            component_density_dict[comp],
            color=component_colors[rank],
            linewidth=2.0,
            label=label
        )

    ax.axvline(
        threshold,
        color="#7A0177",
        linestyle=":",
        linewidth=2.0,
        label=(
            "Pairwise prop(lowest, second-lowest)\n"
            f"= ({target_low_prop:.3g}, {target_second_prop:.3g})\n"
            f"threshold = {threshold:.3f}"
        )
    )

    ax.set_xlabel("Tumor heatmap element value")
    ax.set_ylabel("Density")
    ax.set_title(f"{n_components}-component GMM of flattened tumor heatmap values")

    ax.legend(frameon=False, fontsize=7.5)
    sns.despine(ax=ax)
    plt.tight_layout()

    fig.savefig(
        str(run_dirs["run_dir"])
        + f"/MI_heatmap_tumor_sampleagg_flattened_GMM{n_components}_density.pdf",
        bbox_inches="tight",
        transparent=True
    )

    fig.savefig(
        str(run_dirs["run_dir"])
        + f"/MI_heatmap_tumor_sampleagg_flattened_GMM{n_components}_density.png",
        bbox_inches="tight",
        dpi=300,
        transparent=True
    )

    plt.show()
    plt.close(fig)


# ============================================================
# Step 4. Save all summaries
# ============================================================
gmm_component_summary_df = pd.DataFrame(all_component_summary)
gmm_threshold_summary_df = pd.DataFrame(all_threshold_summary)

gmm_component_summary_df.to_csv(
    str(run_dirs["run_dir"])
    + "/MI_heatmap_tumor_sampleagg_flattened_GMM2to5_component_summary_prop001_999.csv",
    index=False
)

gmm_threshold_summary_df.to_csv(
    str(run_dirs["run_dir"])
    + "/MI_heatmap_tumor_sampleagg_flattened_GMM2to5_threshold_summary_prop001_999.csv",
    index=False
)

print("\nGMM threshold summary:")
display(gmm_threshold_summary_df)

print("\nMI_intensity_threshold_by_gmm:")
print(MI_intensity_threshold_by_gmm)


# ============================================================
# Step 5. Choose which GMM threshold to use for downstream heatmap highlighting
# ============================================================
# Option A: use the BIC-best GMM threshold.
selected_n_components_for_threshold = int(
    gmm_threshold_summary_df
    .sort_values("bic", ascending=True)
    .iloc[0]["n_components"]
)

# Option B: manually force a specific model, e.g.:
# selected_n_components_for_threshold = 2
# selected_n_components_for_threshold = 3
# selected_n_components_for_threshold = 4
# selected_n_components_for_threshold = 5

MI_intensity_threshold = float(
    MI_intensity_threshold_by_gmm[selected_n_components_for_threshold]
)

print(
    "\nSelected MI_intensity_threshold for downstream highlighting:"
)
print(f"selected_n_components_for_threshold = {selected_n_components_for_threshold}")
print(
    "threshold target pairwise prop(lowest, second-lowest) = "
    f"({target_low_prop:.6f}, {target_second_prop:.6f})"
)
print(f"MI_intensity_threshold = {MI_intensity_threshold:.6f}")

In [ ]:
# Step 6. Plot non-tumor sample-level MI heatmap
# ============================================================
MI_nontumor_plot_matrix, _ = plot_mi_sample_heatmap(
    mi_sample_df=MI_mean_pd_nontumor_sample_aligned,
    mi_order=mi_order_common,
    xlabel="Non-tumor sample ID",
    out_prefix="MI_heatmap_nontumor_sampleagg",
    file_savepath_main=str(run_dirs["run_dir"]),
    vmax=0.2,
    highlight_threshold=MI_intensity_threshold,
    cmap=cmap,
    sort_sum_threshold=0.03,
    sample_order=shared_sample_order,
    return_plot_matrix=True,
)

In [ ]:
# ============================================================
# Step 7. Plot element-wise log2 ratio heatmap: tumor / non-tumor
# ============================================================
MI_log2_tumor_vs_nontumor_sample = plot_mi_log2_ratio_heatmap(
    numerator_plot_matrix=MI_tumor_plot_matrix,
    denominator_plot_matrix=MI_nontumor_plot_matrix,
    xlabel="Sample ID",
    out_prefix="MI_heatmap_log2_tumor_over_nontumor_sampleagg",
    file_savepath_main=str(run_dirs["run_dir"]),
    pseudocount=1e-4,
    vmax=5,
    cmap="vlag",
    highlight_log2fc=2.0,
)

In [ ]:
MI_log2fc = np.log2(
    (MI_tumor_plot_matrix + 1e-4) / (MI_nontumor_plot_matrix + 1e-4)
)
##Save the MI_log2fc matrix
MI_log2fc_df = pd.DataFrame(
    MI_log2fc,
    index=mi_order_common,
    columns=shared_sample_order
)
MI_log2fc_df.to_csv(
    str(run_dirs["run_dir"])
    + "/MI_log2fc_tumor_over_nontumor_sampleagg.csv"
)

In [ ]:
# ============================================================
# Step 7.5. Cell-type-pair max-adjusted log2 ratio heatmap:
#           tumor-involved / non-tumor-involved
# ------------------------------------------------------------
# For each sample and each MI:
#   tumor-involved group:
#       group edges by sender→receiver cell-type pair
#       compute mean MI for each pair
#       use the maximum pair-level mean MI as the sample-level tumor value
#
#   non-tumor-involved group:
#       same procedure
#
#   final value:
#       log2((tumor pairmax MI + pseudocount) /
#            (non-tumor pairmax MI + pseudocount))
#
# IMPORTANT:
#   Correct native MI mapping in this Pan-cancer notebook:
#       Factor_envir_use[:, 0] -> MI-1
#       Factor_envir_use[:, 1] -> MI-2
#       Factor_envir_use[:, 2] -> MI-3
#       ...
#
#   Existing mi_order_common / heatmap index is used ONLY for plotting order,
#   NOT for naming Factor_envir_use columns.
# ============================================================

import gc
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# Parameters
# ============================================================

# To avoid unstable maxima from extremely rare sender→receiver pairs.
# A minimum of 1 includes every observed cell-type pair.
MIN_EDGES_PER_CELLTYPE_PAIR = 20

PAIRMAX_PSEUDOCOUNT = 1e-4
PAIRMAX_OUT_PREFIX = "MI_heatmap_log2_tumor_over_nontumor_celltypepairmax_sampleagg_fixedMI_1based"

# Correct for this notebook:
# Factor_envir_use[:, 0] -> MI-1
NATIVE_MI_INDEX_BASE_PAIRMAX = 1

output_dir_pairmax = Path(run_dirs["run_dir"])


# ============================================================
# Helper functions
# ============================================================

def _to_numpy_pairmax(x):
    """Safely convert torch / numpy / list-like object to numpy array."""
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    elif hasattr(x, "cpu"):
        try:
            x = x.cpu().numpy()
        except Exception:
            pass
    return np.asarray(x)


def _standardize_mi_name_pairmax(mi):
    """
    Standardize MI label format without changing identity.
    Examples:
      MI_2 -> MI-2
      MI2  -> MI-2
      MI-2 -> MI-2
    """
    mi = str(mi)
    mi = mi.replace("MI_", "MI-")
    if mi.startswith("MI-"):
        return mi
    if mi.startswith("MI"):
        return mi.replace("MI", "MI-", 1)
    return mi


def _natural_mi_order_pairmax(mi_name):
    """Natural numeric ordering for MI labels."""
    try:
        return int(
            str(mi_name)
            .replace("MI-", "")
            .replace("MI_", "")
            .replace("MI", "")
        )
    except Exception:
        return 10**9


def ensure_edge_index_e_by_2_pairmax(edge_index):
    """Ensure edge_index has shape E x 2."""
    edge_index = _to_numpy_pairmax(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(np.int64, copy=False)

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def collect_edge_metadata_for_celltype_pairmax(processed):
    """
    Collect edge-level metadata aligned with Factor_envir_use.

    Returns
    -------
    edge_slice_id : np.ndarray
        Slice ID for each edge.
    edge_sample_id : np.ndarray
        Sample ID for each edge, using the same split rule as the earlier cells:
        sample_id = slice_id.split("_")[0]
    sender_celltype : np.ndarray
        Sender cell type for each edge.
    receiver_celltype : np.ndarray
        Receiver cell type for each edge.
    edge_pair : np.ndarray
        Sender→receiver cell-type-pair label for each edge.
    edge_has_cancer_cell : np.ndarray[bool]
        Whether the edge involves at least one cancer cell.
    """
    edge_slice_list = []
    sender_celltype_list = []
    receiver_celltype_list = []

    for batch_idx in range(len(processed.spidernet_data)):
        data_cur = processed.spidernet_data[batch_idx]

        edge_index_cur = ensure_edge_index_e_by_2_pairmax(data_cur["edge_index"])

        sample_cur = _to_numpy_pairmax(data_cur["sample"]).astype(str).reshape(-1)
        celltype_cur = _to_numpy_pairmax(data_cur["cell_class"]).astype(str).reshape(-1)

        # Usually sample_cur is per cell. If it is one value per batch, broadcast it.
        if sample_cur.size == 1:
            edge_slice_cur = np.repeat(sample_cur[0], edge_index_cur.shape[0])
        else:
            edge_slice_cur = sample_cur[edge_index_cur[:, 0]]

        sender_celltype_cur = celltype_cur[edge_index_cur[:, 0]]
        receiver_celltype_cur = celltype_cur[edge_index_cur[:, 1]]

        edge_slice_list.append(edge_slice_cur)
        sender_celltype_list.append(sender_celltype_cur)
        receiver_celltype_list.append(receiver_celltype_cur)

    edge_slice_id = np.hstack(edge_slice_list).astype(str)
    sender_celltype = np.hstack(sender_celltype_list).astype(str)
    receiver_celltype = np.hstack(receiver_celltype_list).astype(str)

    edge_sample_id = np.asarray(
        [str(x).split("_")[0] for x in edge_slice_id],
        dtype=str
    )

    edge_pair = np.char.add(
        np.char.add(sender_celltype.astype(str), " -> "),
        receiver_celltype.astype(str)
    )

    sender_lower = np.char.lower(sender_celltype.astype(str))
    receiver_lower = np.char.lower(receiver_celltype.astype(str))

    edge_has_cancer_cell = np.logical_or(
        np.char.find(sender_lower, "cancercell") >= 0,
        np.char.find(receiver_lower, "cancercell") >= 0,
    )

    return (
        edge_slice_id,
        edge_sample_id,
        sender_celltype,
        receiver_celltype,
        edge_pair,
        edge_has_cancer_cell,
    )


def resolve_native_mi_columns_pairmax(n_mi):
    """
    Define native MI labels according to Factor_envir_use column order.

    Correct mapping:
        column 0 -> MI-1
        column 1 -> MI-2
        column 2 -> MI-3
        ...
    """
    mi_columns = [
        f"MI-{i + int(NATIVE_MI_INDEX_BASE_PAIRMAX)}"
        for i in range(n_mi)
    ]

    print("Native Factor_envir_use column mapping for pairmax analysis:")
    for i, mi in enumerate(mi_columns[:20]):
        print(f"  column {i}: {mi}")
    if n_mi > 20:
        print(f"  ... total {n_mi} MIs")

    return mi_columns


def resolve_plot_mi_order_pairmax(mi_columns):
    """
    Resolve plotting order only.

    This function never changes MI identity. It only controls the row order
    in the final heatmap.
    """
    native_set = set(mi_columns)
    candidate_order = None

    if "mi_order_common" in globals():
        candidate_order = list(mi_order_common)
        print("Using `mi_order_common` only as pairmax heatmap plotting order.")
    elif "mi_order" in globals():
        candidate_order = list(mi_order)
        print("Using `mi_order` only as pairmax heatmap plotting order.")
    elif "MI_log2_tumor_vs_nontumor_sample" in globals():
        candidate_order = list(MI_log2_tumor_vs_nontumor_sample.index)
        print("Using `MI_log2_tumor_vs_nontumor_sample.index` only as pairmax heatmap plotting order.")
    elif "MI_tumor_plot_matrix" in globals():
        candidate_order = list(MI_tumor_plot_matrix.index)
        print("Using `MI_tumor_plot_matrix.index` only as pairmax heatmap plotting order.")

    if candidate_order is not None:
        candidate_order = [_standardize_mi_name_pairmax(x) for x in candidate_order]
        candidate_order = [x for x in candidate_order if x in native_set]

        missing = [x for x in mi_columns if x not in set(candidate_order)]
        missing = sorted(missing, key=_natural_mi_order_pairmax)

        mi_order_use = candidate_order + missing
    else:
        mi_order_use = sorted(mi_columns, key=_natural_mi_order_pairmax)

    print("\nPairmax heatmap MI plotting order:")
    print(mi_order_use)

    return mi_order_use


def compute_pairmax_mi_by_sample(
    factor_matrix,
    edge_sample_id,
    edge_pair,
    group_mask,
    mi_columns,
    sample_order=None,
    min_edges_per_pair=20,
    group_label="tumor_involved",
):
    """
    For each sample and each MI:
      1) subset edges by group_mask;
      2) group edges by sender→receiver cell-type pair;
      3) compute mean MI for each pair;
      4) take the maximum mean MI across pairs.

    This removes the direct dependence on the abundance/proportion of
    different cell-type pairs, because each sample-level MI value is based
    on the strongest cell-type-pair mean rather than the pooled edge mean.
    """
    factor_matrix = np.asarray(factor_matrix, dtype=np.float32)

    if sample_order is None:
        sample_ids = np.sort(np.unique(edge_sample_id.astype(str))).tolist()
    else:
        sample_ids = [
            str(x)
            for x in sample_order
            if str(x) in set(edge_sample_id.astype(str))
        ]

    pairmax_df = pd.DataFrame(
        np.nan,
        index=sample_ids,
        columns=mi_columns,
        dtype=float,
    )

    selected_pair_df = pd.DataFrame(
        "",
        index=sample_ids,
        columns=mi_columns,
        dtype=object,
    )

    selected_pair_nedge_df = pd.DataFrame(
        0,
        index=sample_ids,
        columns=mi_columns,
        dtype=int,
    )

    selected_rows = []

    n_mi = factor_matrix.shape[1]

    if len(mi_columns) != n_mi:
        raise ValueError(
            f"Length of mi_columns ({len(mi_columns)}) does not match "
            f"number of MI columns in factor_matrix ({n_mi})."
        )

    for sample_id in sample_ids:
        edge_idx_cur = np.where(
            (edge_sample_id.astype(str) == str(sample_id)) & group_mask
        )[0]

        if edge_idx_cur.size == 0:
            continue

        pair_cur = edge_pair[edge_idx_cur].astype(str)

        pair_names, pair_inverse = np.unique(pair_cur, return_inverse=True)
        pair_counts = np.bincount(pair_inverse, minlength=len(pair_names)).astype(int)

        valid_pair_mask = pair_counts >= int(min_edges_per_pair)

        if valid_pair_mask.sum() == 0:
            continue

        # Sum MI values within each sender→receiver pair.
        pair_sums = np.zeros((len(pair_names), n_mi), dtype=np.float64)
        np.add.at(pair_sums, pair_inverse, factor_matrix[edge_idx_cur, :])

        pair_means = pair_sums / np.maximum(pair_counts[:, np.newaxis], 1)

        # Remove rare pairs from max selection.
        pair_means[~valid_pair_mask, :] = np.nan

        finite_mi_mask = np.isfinite(pair_means).any(axis=0)

        for mi_idx, mi_name in enumerate(mi_columns):
            if not finite_mi_mask[mi_idx]:
                continue

            pair_mean_cur = pair_means[:, mi_idx]
            best_pair_idx = int(np.nanargmax(pair_mean_cur))

            best_value = float(pair_mean_cur[best_pair_idx])
            best_pair = str(pair_names[best_pair_idx])
            best_pair_nedge = int(pair_counts[best_pair_idx])

            pairmax_df.loc[sample_id, mi_name] = best_value
            selected_pair_df.loc[sample_id, mi_name] = best_pair
            selected_pair_nedge_df.loc[sample_id, mi_name] = best_pair_nedge

            selected_rows.append({
                "group": group_label,
                "sample_id": sample_id,
                "MI": mi_name,
                "MI_native_column_index": int(mi_idx),
                "selected_sender_receiver_pair": best_pair,
                "selected_pair_mean_MI": best_value,
                "selected_pair_n_edges": best_pair_nedge,
                "min_edges_per_pair": int(min_edges_per_pair),
                "n_valid_pairs_in_sample_group": int(valid_pair_mask.sum()),
                "n_edges_in_sample_group": int(edge_idx_cur.size),
            })

    selected_long_df = pd.DataFrame(selected_rows)

    return pairmax_df, selected_pair_df, selected_pair_nedge_df, selected_long_df


# ============================================================
# Step 1. Re-collect edge metadata
# ============================================================

(
    edge_slice_id_pairmax,
    edge_sample_id_pairmax,
    sender_celltype_pairmax,
    receiver_celltype_pairmax,
    edge_celltype_pair_pairmax,
    edge_has_cancer_cell_pairmax,
) = collect_edge_metadata_for_celltype_pairmax(processed)

factor_matrix_pairmax = np.asarray(Factor_envir_use)

# Make sure Factor_envir_use is edge x MI.
if factor_matrix_pairmax.ndim != 2:
    raise ValueError(
        f"Factor_envir_use must be 2D, got shape {factor_matrix_pairmax.shape}"
    )

if factor_matrix_pairmax.shape[0] != edge_sample_id_pairmax.shape[0]:
    if factor_matrix_pairmax.shape[1] == edge_sample_id_pairmax.shape[0]:
        print("Detected Factor_envir_use as MI x edge. Transposing to edge x MI.")
        factor_matrix_pairmax = factor_matrix_pairmax.T
    else:
        raise ValueError(
            "Factor_envir_use is not aligned with collected edges.\n"
            f"Factor_envir_use shape: {factor_matrix_pairmax.shape}\n"
            f"Number of collected edges: {edge_sample_id_pairmax.shape[0]}"
        )

# Correct native MI labels.
# Do NOT use mi_order_common here.
mi_columns_pairmax = resolve_native_mi_columns_pairmax(
    n_mi=factor_matrix_pairmax.shape[1]
)

# Plotting order only.
mi_order_pairmax = resolve_plot_mi_order_pairmax(
    mi_columns=mi_columns_pairmax
)

# Use the same sample order as the current heatmap whenever available.
if "shared_sample_order" in globals():
    sample_order_pairmax = [
        str(x)
        for x in shared_sample_order
        if str(x) in set(edge_sample_id_pairmax.astype(str))
    ]
else:
    sample_order_pairmax = np.sort(
        np.unique(edge_sample_id_pairmax.astype(str))
    ).tolist()

print("\nNumber of edges:", factor_matrix_pairmax.shape[0])
print("Number of MIs:", factor_matrix_pairmax.shape[1])
print("Number of samples:", len(sample_order_pairmax))
print("MIN_EDGES_PER_CELLTYPE_PAIR:", MIN_EDGES_PER_CELLTYPE_PAIR)


# ============================================================
# Step 2. Compute sample-level max cell-type-pair mean MI
#         separately for tumor-involved and non-tumor-involved edges
# ============================================================

MI_mean_pd_tumor_pairmax_sample, MI_tumor_pairmax_selected_pair, MI_tumor_pairmax_selected_nedge, MI_tumor_pairmax_selected_long = (
    compute_pairmax_mi_by_sample(
        factor_matrix=factor_matrix_pairmax,
        edge_sample_id=edge_sample_id_pairmax,
        edge_pair=edge_celltype_pair_pairmax,
        group_mask=edge_has_cancer_cell_pairmax,
        mi_columns=mi_columns_pairmax,
        sample_order=sample_order_pairmax,
        min_edges_per_pair=MIN_EDGES_PER_CELLTYPE_PAIR,
        group_label="tumor_involved",
    )
)

MI_mean_pd_nontumor_pairmax_sample, MI_nontumor_pairmax_selected_pair, MI_nontumor_pairmax_selected_nedge, MI_nontumor_pairmax_selected_long = (
    compute_pairmax_mi_by_sample(
        factor_matrix=factor_matrix_pairmax,
        edge_sample_id=edge_sample_id_pairmax,
        edge_pair=edge_celltype_pair_pairmax,
        group_mask=~edge_has_cancer_cell_pairmax,
        mi_columns=mi_columns_pairmax,
        sample_order=sample_order_pairmax,
        min_edges_per_pair=MIN_EDGES_PER_CELLTYPE_PAIR,
        group_label="non_tumor_involved",
    )
)


# ============================================================
# Step 3. Align matrices and plot log2 ratio heatmap
# ============================================================

common_samples_pairmax = [
    s for s in sample_order_pairmax
    if (s in MI_mean_pd_tumor_pairmax_sample.index)
    and (s in MI_mean_pd_nontumor_pairmax_sample.index)
]

mi_order_pairmax = [
    str(mi)
    for mi in mi_order_pairmax
    if (str(mi) in MI_mean_pd_tumor_pairmax_sample.columns)
    and (str(mi) in MI_mean_pd_nontumor_pairmax_sample.columns)
]

MI_tumor_pairmax_plot_matrix = (
    MI_mean_pd_tumor_pairmax_sample
    .loc[common_samples_pairmax, mi_order_pairmax]
    .T
)

MI_nontumor_pairmax_plot_matrix = (
    MI_mean_pd_nontumor_pairmax_sample
    .loc[common_samples_pairmax, mi_order_pairmax]
    .T
)

MI_log2_tumor_vs_nontumor_pairmax_sample = plot_mi_log2_ratio_heatmap(
    numerator_plot_matrix=MI_tumor_pairmax_plot_matrix,
    denominator_plot_matrix=MI_nontumor_pairmax_plot_matrix,
    xlabel="Sample ID",
    out_prefix=PAIRMAX_OUT_PREFIX,
    file_savepath_main=str(output_dir_pairmax),
    pseudocount=PAIRMAX_PSEUDOCOUNT,
    vmax=2,
    cmap="vlag",
    highlight_log2fc=0.5,
)


# ============================================================
# Step 4. Save matrices and selected-pair annotations
# ============================================================

MI_mean_pd_tumor_pairmax_sample.to_csv(
    output_dir_pairmax / "MI_tumor_involved_celltypepairmax_sample_matrix_fixedMI_1based.csv"
)

MI_mean_pd_nontumor_pairmax_sample.to_csv(
    output_dir_pairmax / "MI_nontumor_involved_celltypepairmax_sample_matrix_fixedMI_1based.csv"
)

MI_tumor_pairmax_selected_pair.to_csv(
    output_dir_pairmax / "MI_tumor_involved_celltypepairmax_selected_pair_fixedMI_1based.csv"
)

MI_nontumor_pairmax_selected_pair.to_csv(
    output_dir_pairmax / "MI_nontumor_involved_celltypepairmax_selected_pair_fixedMI_1based.csv"
)

MI_tumor_pairmax_selected_nedge.to_csv(
    output_dir_pairmax / "MI_tumor_involved_celltypepairmax_selected_pair_nedges_fixedMI_1based.csv"
)

MI_nontumor_pairmax_selected_nedge.to_csv(
    output_dir_pairmax / "MI_nontumor_involved_celltypepairmax_selected_pair_nedges_fixedMI_1based.csv"
)

MI_pairmax_selected_long = pd.concat(
    [
        MI_tumor_pairmax_selected_long,
        MI_nontumor_pairmax_selected_long,
    ],
    axis=0,
    ignore_index=True,
)

MI_pairmax_selected_long.to_csv(
    output_dir_pairmax / "MI_celltypepairmax_selected_pair_long_table_fixedMI_1based.csv",
    index=False,
)


# ============================================================
# Step 5. Diagnostic table for MI-4 / MI-6
# ============================================================

diagnostic_mis_pairmax = [
    mi for mi in ["MI-4", "MI-6"]
    if mi in set(mi_columns_pairmax)
]

if len(diagnostic_mis_pairmax) > 0:
    print("\nTumor-involved selected pairs for diagnostic MIs:")
    display(
        MI_tumor_pairmax_selected_long
        .loc[MI_tumor_pairmax_selected_long["MI"].isin(diagnostic_mis_pairmax)]
        .sort_values(["MI", "sample_id"])
        .head(50)
    )

    print("\nNon-tumor-involved selected pairs for diagnostic MIs:")
    display(
        MI_nontumor_pairmax_selected_long
        .loc[MI_nontumor_pairmax_selected_long["MI"].isin(diagnostic_mis_pairmax)]
        .sort_values(["MI", "sample_id"])
        .head(50)
    )


# ============================================================
# Step 6. Show outputs
# ============================================================

print("\nCell-type-pair max-adjusted tumor matrix:")
display(
    MI_mean_pd_tumor_pairmax_sample
    .loc[common_samples_pairmax, mi_order_pairmax]
)

print("Cell-type-pair max-adjusted non-tumor matrix:")
display(
    MI_mean_pd_nontumor_pairmax_sample
    .loc[common_samples_pairmax, mi_order_pairmax]
)

print("log2(tumor-involved pairmax / non-tumor-involved pairmax):")
display(MI_log2_tumor_vs_nontumor_pairmax_sample)

print("Selected tumor-involved sender→receiver pair for each sample and MI:")
display(
    MI_tumor_pairmax_selected_pair
    .loc[common_samples_pairmax, mi_order_pairmax]
)

print("Selected non-tumor-involved sender→receiver pair for each sample and MI:")
display(
    MI_nontumor_pairmax_selected_pair
    .loc[common_samples_pairmax, mi_order_pairmax]
)


# ============================================================
# Optional cleanup
# ============================================================

if "release_memory" in globals():
    release_memory(
        "edge_slice_id_pairmax",
        "edge_sample_id_pairmax",
        "sender_celltype_pairmax",
        "receiver_celltype_pairmax",
        "edge_celltype_pair_pairmax",
        "edge_has_cancer_cell_pairmax",
        "factor_matrix_pairmax",
        namespace=globals(),
        run_gc=True,
        clear_cuda=False,
        close_figures=False,
    )
else:
    gc.collect()


In [ ]:
# ============================================================
# Pan-cancer global MI-specific sender→receiver cell-type pair plot
# Top-fraction + minimum-activity-filter version
# ------------------------------------------------------------
# Dot size  = raw mean MI activity for each sender→receiver pair
# Dot color = max-normalized MI score within each MI
#
# Selection rule:
#   For each MI:
#       1) rank all eligible sender→receiver cell-type pairs by mean MI activity
#       2) keep top TOP_PAIR_FRACTION pairs
#       3) further require mean MI activity > MIN_MEAN_MI_ACTIVITY
#
# Combined 3-panel figure:
#   Panel 1:
#       Count of selected sender→receiver pairs, with number annotation
#
#   Panel 2:
#       Fraction of selected pairs involving tumor cells.
#       Black vertical tick in each row marks the background denominator:
#           eligible tumor-involved pair fraction among all eligible pairs.
#       The x-axis is horizontally flipped.
#
#   Panel 3:
#       Tumor-type heatmap:
#           rows = MIs sorted by MAX-NORMALIZED heatmap row sum descending
#           columns = tumor types sorted by raw column sum descending
#           element = count of selected TUMOR-INVOLVED pairs related to each tumor type,
#                     max-normalized within each MI
#           colormap = YlOrRd
#           zero values are shown in gray
#           raw count values are NOT annotated on the heatmap
#
# Dot plot:
#   Sender→receiver pair order is rebuilt according to the final MI order.
#   For each MI in the final order, its selected top pairs are appended together,
#   producing a more focused/block-like dot plot.
#
# IMPORTANT:
#   In this Pan-cancer notebook:
#       Factor_envir_use[:, 0] -> MI-1
#       Factor_envir_use[:, 1] -> MI-2
#       Factor_envir_use[:, 2] -> MI-3
#       ...
#
#   Existing mi_order_common / heatmap index is used ONLY for initial plotting order,
#   NOT for naming Factor_envir_use columns.
# ============================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt


# ============================================================
# Parameters
# ============================================================

TOP_PAIR_FRACTION = 0.10
MIN_MEAN_MI_ACTIVITY = 0.10

MIN_EDGES_PER_PAIR = 30
EXCLUDE_SAME_CELL_TYPE = False

# Correct for this notebook:
# Factor_envir_use[:, 0] -> MI-1
NATIVE_MI_INDEX_BASE = 1

EPS = 1e-8

# Width multiplier for the three-panel summary figure.
FIG_WIDTH_SCALE = 0.5

BAR_FILL_COLOR = "#f8b129"

TUMOR_TYPE_ORDER = [
    "Breast",
    "Liver",
    "Colon",
    "Ovarian",
    "Uterine",
    "Melanoma",
    "Lung",
    "Prostate",
]

fraction_tag = f"top{int(round(TOP_PAIR_FRACTION * 100))}percent"
activity_tag = str(MIN_MEAN_MI_ACTIVITY).replace(".", "p")


# ============================================================
# Short and safe output paths
# Avoid Windows FileNotFoundError caused by overly long paths.
# ============================================================

BASE_RUN_DIR = Path(run_dirs["run_dir"]).resolve()

out_dir = BASE_RUN_DIR / f"MIpair_{fraction_tag}_gt{activity_tag}"
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {out_dir}")


def _safe_path(filename):
    """
    Build a short safe output path and ensure parent directory exists.
    """
    path = out_dir / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def _safe_to_csv(df, filename, index=True):
    """
    Save dataframe with a short filename.
    """
    path = _safe_path(filename)
    df.to_csv(path, index=index)
    return path


def _safe_savefig(fig, stem, dpi=300):
    """
    Save current figure as PDF and PNG using short filenames.
    """
    pdf_path = _safe_path(f"{stem}.pdf")
    png_path = _safe_path(f"{stem}.png")

    fig.savefig(pdf_path, bbox_inches="tight", dpi=dpi)
    fig.savefig(png_path, bbox_inches="tight", dpi=dpi)

    return pdf_path, png_path


# ============================================================
# Plot style
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

FONT_TITLE = 24
FONT_AXIS_LABEL = 20
FONT_XTICK = 14
FONT_YTICK = 15
FONT_CBAR_LABEL = 16
FONT_CBAR_TICK = 13

DOT_SIZE_MULTIPLIER = 2.0


# ============================================================
# Helper functions
# ============================================================

def _to_numpy(x):
    """Convert torch tensor / numpy-like object to numpy array."""
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    return np.asarray(x)


def _get_field(obj, key):
    """Get field from dict-like or PyG Data-like object."""
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    try:
        return obj[key]
    except Exception as e:
        raise KeyError(f"Cannot find field `{key}` in object of type {type(obj)}") from e


def _has_field(obj, key):
    if isinstance(obj, dict):
        return key in obj
    if hasattr(obj, key):
        return True
    try:
        obj[key]
        return True
    except Exception:
        return False


def _standardize_mi_name(mi):
    """
    Standardize MI label format without changing identity.
    Examples:
      MI_2 -> MI-2
      MI2  -> MI-2
      MI-2 -> MI-2
    """
    mi = str(mi)
    mi = mi.replace("MI_", "MI-")
    if mi.startswith("MI-"):
        return mi
    if mi.startswith("MI"):
        return mi.replace("MI", "MI-", 1)
    return mi


def _natural_mi_order(mi_name):
    try:
        return int(
            str(mi_name)
            .replace("MI-", "")
            .replace("MI_", "")
            .replace("MI", "")
        )
    except Exception:
        return 10**9


def _format_pair_label(label):
    """Format sender→receiver label clearly."""
    label = str(label)
    label = label.replace("-->", "→")
    label = label.replace("->", "→")
    label = label.replace("=>", "→")
    label = label.replace("→", " → ")
    label = " ".join(label.split())
    label = label.replace(" → ", "  →  ")
    return label


def _scale_size(values, global_values=None, min_size=55, max_size=760, clip=True):
    """Scale raw MI activity values to dot sizes."""
    values = np.asarray(values, dtype=float)

    if global_values is None:
        global_values = values

    global_values = np.asarray(global_values, dtype=float)
    global_values = global_values[np.isfinite(global_values)]

    if global_values.size == 0:
        return np.full_like(values, 160.0, dtype=float)

    vmin = np.nanmin(global_values)
    vmax = np.nanmax(global_values)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return np.full_like(values, 160.0, dtype=float)

    values_use = values.copy()

    if clip:
        values_use = np.clip(values_use, vmin, vmax)

    return min_size + (values_use - vmin) / (vmax - vmin) * (max_size - min_size)


def _is_tumor_celltype(celltype):
    """Whether one cell type label is a tumor/cancer cell type."""
    return "cancercell" in str(celltype).lower()


def _is_tumor_involved_pair(sender, receiver):
    """Whether sender or receiver cell type is tumor/cancer cell."""
    return _is_tumor_celltype(sender) or _is_tumor_celltype(receiver)


def _extract_tumor_type(celltype):
    """
    Extract tumor type from labels like:
        Breast-cancercell
        Colon-cancercell
    """
    s = str(celltype)

    if "cancercell" not in s.lower():
        return None

    s_clean = (
        s.replace("-cancercell", "")
         .replace("_cancercell", "")
         .replace(" cancercell", "")
         .replace("cancercell", "")
    )

    s_clean = s_clean.strip(" -_")

    if len(s_clean) == 0:
        return "Unknown"

    return s_clean


def _extract_tumor_types_from_pair(sender, receiver):
    """
    Return unique tumor types involved in a sender→receiver pair.
    If both sender and receiver are the same tumor type, count it once.
    """
    tumor_types = []

    sender_tumor = _extract_tumor_type(sender)
    receiver_tumor = _extract_tumor_type(receiver)

    if sender_tumor is not None:
        tumor_types.append(sender_tumor)

    if receiver_tumor is not None:
        tumor_types.append(receiver_tumor)

    return sorted(set(tumor_types))


def _ensure_processed_loaded():
    """Reload processed object if needed."""
    if "processed" in globals() and processed is not None:
        return processed

    if "processed_data_dir" not in globals():
        raise NameError(
            "`processed` is not in memory and `processed_data_dir` is not available. "
            "Please load processed data before running this cell."
        )

    from SpiderNet.io import load_processed_data
    return load_processed_data(processed_data_dir)


def _collect_edge_sender_receiver_from_processed(processed_obj):
    """
    Collect sender and receiver cell types for all edges across all batches.
    """
    sender_list = []
    receiver_list = []
    batch_list = []

    for batch_idx in range(len(processed_obj.spidernet_data)):
        data_cur = processed_obj.spidernet_data[batch_idx]

        edge_index_cur = _to_numpy(_get_field(data_cur, "edge_index"))

        if edge_index_cur.ndim != 2:
            raise ValueError(
                f"edge_index for batch {batch_idx} should be 2D, got {edge_index_cur.shape}"
            )

        if edge_index_cur.shape[1] == 2:
            edge_index_cur = edge_index_cur.astype(np.int64, copy=False)
        elif edge_index_cur.shape[0] == 2:
            edge_index_cur = edge_index_cur.T.astype(np.int64, copy=False)
        else:
            raise ValueError(
                f"edge_index for batch {batch_idx} should have shape [E, 2] or [2, E], "
                f"got {edge_index_cur.shape}"
            )

        if not _has_field(data_cur, "cell_class"):
            raise ValueError(
                f"Cannot find `cell_class` in processed.spidernet_data[{batch_idx}]."
            )

        celltype_cur = _to_numpy(_get_field(data_cur, "cell_class")).astype(str)

        if edge_index_cur.max() >= len(celltype_cur):
            raise ValueError(
                f"edge_index contains node index {edge_index_cur.max()}, "
                f"but batch {batch_idx} only has {len(celltype_cur)} cells."
            )

        sender_cur = celltype_cur[edge_index_cur[:, 0]]
        receiver_cur = celltype_cur[edge_index_cur[:, 1]]

        sender_list.append(sender_cur)
        receiver_list.append(receiver_cur)
        batch_list.append(np.repeat(batch_idx, edge_index_cur.shape[0]))

    sender_all = np.concatenate(sender_list).astype(str)
    receiver_all = np.concatenate(receiver_list).astype(str)
    batch_all = np.concatenate(batch_list).astype(int)

    pair_all = np.char.add(
        np.char.add(sender_all.astype(str), " → "),
        receiver_all.astype(str)
    )

    return sender_all, receiver_all, pair_all, batch_all


def _load_factor_matrix(n_edges_expected):
    """Load Factor_envir_use and ensure edge x MI orientation."""
    if "Factor_envir_use" in globals():
        factor_matrix = Factor_envir_use
    else:
        factor_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"
        if not factor_path.exists():
            raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {factor_path}")
        factor_matrix = np.load(factor_path, mmap_mode="r")

    factor_matrix = np.asarray(factor_matrix)

    if factor_matrix.ndim != 2:
        raise ValueError(f"Factor_envir_use must be 2D, got shape {factor_matrix.shape}")

    if factor_matrix.shape[0] != n_edges_expected:
        if factor_matrix.shape[1] == n_edges_expected:
            print("Detected Factor_envir_use as MI x edge. Transposing to edge x MI.")
            factor_matrix = factor_matrix.T
        else:
            raise ValueError(
                "Factor_envir_use is not aligned with reconstructed edges.\n"
                f"Factor_envir_use shape: {factor_matrix.shape}\n"
                f"Number of reconstructed edges: {n_edges_expected}"
            )

    return factor_matrix


def _resolve_native_mi_names_and_plot_order(n_mi):
    """
    Native MI names must follow Factor_envir_use column order.

    Correct mapping:
        column 0 -> MI-1
        column 1 -> MI-2
        column 2 -> MI-3
        ...
    """
    mi_names = [
        f"MI-{i + int(NATIVE_MI_INDEX_BASE)}"
        for i in range(n_mi)
    ]

    native_set = set(mi_names)

    candidate_order = None

    if "mi_order_common" in globals():
        candidate_order = list(mi_order_common)
        print("Using `mi_order_common` only as initial plotting order.")
    elif "mi_order" in globals():
        candidate_order = list(mi_order)
        print("Using `mi_order` only as initial plotting order.")
    elif "MI_log2_tumor_vs_nontumor_pairmax_sample" in globals():
        candidate_order = list(MI_log2_tumor_vs_nontumor_pairmax_sample.index)
        print("Using `MI_log2_tumor_vs_nontumor_pairmax_sample.index` only as initial plotting order.")
    elif "MI_log2_tumor_vs_nontumor_sample" in globals():
        candidate_order = list(MI_log2_tumor_vs_nontumor_sample.index)
        print("Using `MI_log2_tumor_vs_nontumor_sample.index` only as initial plotting order.")
    elif "MI_tumor_plot_matrix" in globals():
        candidate_order = list(MI_tumor_plot_matrix.index)
        print("Using `MI_tumor_plot_matrix.index` only as initial plotting order.")

    if candidate_order is not None:
        candidate_order = [_standardize_mi_name(x) for x in candidate_order]
        candidate_order = [x for x in candidate_order if x in native_set]

        missing = [x for x in mi_names if x not in set(candidate_order)]
        missing = sorted(missing, key=_natural_mi_order)

        mi_order_use = candidate_order + missing
    else:
        mi_order_use = sorted(mi_names, key=_natural_mi_order)

    print("\nNative Factor_envir_use column mapping:")
    for i in range(min(20, n_mi)):
        print(f"  column {i}: {mi_names[i]}")
    if n_mi > 20:
        print(f"  ... total {n_mi} MIs")

    print("\nInitial MI plotting order:")
    print(mi_order_use)

    return mi_names, mi_order_use


def _build_global_pair_metric_table(
    factor_matrix,
    sender_all,
    receiver_all,
    pair_all,
    mi_names,
):
    """
    Compute mean MI activity for each sender→receiver pair across all edges.
    """
    valid_mask = np.ones(len(pair_all), dtype=bool)

    if EXCLUDE_SAME_CELL_TYPE:
        valid_mask = valid_mask & (sender_all.astype(str) != receiver_all.astype(str))

    edge_idx = np.where(valid_mask)[0]

    if edge_idx.size == 0:
        raise ValueError("No valid edges available for sender→receiver pair analysis.")

    sender_use = sender_all[edge_idx].astype(str)
    receiver_use = receiver_all[edge_idx].astype(str)
    pair_use = pair_all[edge_idx].astype(str)

    pair_names, first_idx, pair_inverse = np.unique(
        pair_use,
        return_index=True,
        return_inverse=True,
    )

    num_pairs = len(pair_names)
    num_mi = factor_matrix.shape[1]

    pair_counts = np.bincount(pair_inverse, minlength=num_pairs).astype(int)

    sender_by_pair = pd.Series(
        sender_use[first_idx],
        index=pair_names,
        name="Sender",
    )

    receiver_by_pair = pd.Series(
        receiver_use[first_idx],
        index=pair_names,
        name="Receiver",
    )

    pair_count_s = pd.Series(
        pair_counts,
        index=pair_names,
        name="Pair_edge_count",
    )

    keep_pairs = pair_count_s.index[
        pair_count_s.values >= int(MIN_EDGES_PER_PAIR)
    ].tolist()

    if len(keep_pairs) == 0:
        raise ValueError(
            f"No sender→receiver pair has at least MIN_EDGES_PER_PAIR={MIN_EDGES_PER_PAIR} edges. "
            "Please lower MIN_EDGES_PER_PAIR."
        )

    print("\nEdge summary:")
    print("Total edges used:", edge_idx.size)
    print("Unique sender→receiver pairs:", len(pair_names))
    print("Pairs kept after edge-count filtering:", len(keep_pairs))

    sum_mat = np.zeros((num_pairs, num_mi), dtype=np.float64)
    count_mat = np.zeros((num_pairs, num_mi), dtype=np.float64)

    for mi_idx in range(num_mi):
        vals = np.asarray(factor_matrix[edge_idx, mi_idx], dtype=np.float64)
        finite_mask = np.isfinite(vals)

        if finite_mask.sum() == 0:
            continue

        sum_mat[:, mi_idx] = np.bincount(
            pair_inverse[finite_mask],
            weights=vals[finite_mask],
            minlength=num_pairs,
        )

        count_mat[:, mi_idx] = np.bincount(
            pair_inverse[finite_mask],
            minlength=num_pairs,
        )

    mean_mat = np.full_like(sum_mat, np.nan, dtype=np.float64)
    np.divide(
        sum_mat,
        count_mat,
        out=mean_mat,
        where=count_mat > 0,
    )

    activity_df = pd.DataFrame(
        mean_mat.T,
        index=mi_names,
        columns=pair_names,
    )

    activity_keep = activity_df.loc[:, keep_pairs].copy()

    mi_max_activity = activity_keep.max(axis=1, skipna=True)
    maxnorm_df = activity_keep.div(mi_max_activity + EPS, axis=0)
    maxnorm_df = maxnorm_df.replace([np.inf, -np.inf], np.nan).fillna(0)

    mi_sum_activity = activity_keep.sum(axis=1, skipna=True)
    contribution_df = activity_keep.div(mi_sum_activity + EPS, axis=0)
    contribution_df = contribution_df.replace([np.inf, -np.inf], np.nan).fillna(0)

    full_metric_rows = []

    for mi in mi_names:
        tmp = pd.DataFrame({
            "MI": mi,
            "MI_order": _natural_mi_order(mi),
            "Pair": keep_pairs,
            "Sender": sender_by_pair.loc[keep_pairs].values,
            "Receiver": receiver_by_pair.loc[keep_pairs].values,
            "Pair_edge_count": pair_count_s.loc[keep_pairs].values,
            "Activity": activity_keep.loc[mi, keep_pairs].values,
            "Max_normalized_score": maxnorm_df.loc[mi, keep_pairs].values,
            "MI_contribution": contribution_df.loc[mi, keep_pairs].values,
        })

        full_metric_rows.append(tmp)

    full_metric_df = pd.concat(full_metric_rows, axis=0, ignore_index=True)

    full_metric_df["Activity"] = pd.to_numeric(
        full_metric_df["Activity"],
        errors="coerce",
    )

    full_metric_df["Max_normalized_score"] = pd.to_numeric(
        full_metric_df["Max_normalized_score"],
        errors="coerce",
    ).fillna(0)

    full_metric_df["MI_contribution"] = pd.to_numeric(
        full_metric_df["MI_contribution"],
        errors="coerce",
    ).fillna(0)

    full_metric_df["Tumor_involved_pair"] = [
        _is_tumor_involved_pair(sender, receiver)
        for sender, receiver in zip(full_metric_df["Sender"], full_metric_df["Receiver"])
    ]

    full_metric_df["Tumor_types_in_pair"] = [
        _extract_tumor_types_from_pair(sender, receiver)
        for sender, receiver in zip(full_metric_df["Sender"], full_metric_df["Receiver"])
    ]

    full_metric_df["N_tumor_types_in_pair"] = [
        len(x) for x in full_metric_df["Tumor_types_in_pair"]
    ]

    return full_metric_df, activity_keep, maxnorm_df, contribution_df


def _build_top_fraction_activity_filtered_pairs(
    full_metric_df,
    mi_order_use,
    top_fraction,
    min_activity,
):
    """
    For each MI:
      1) rank sender→receiver pairs by mean MI activity;
      2) keep top_fraction candidate pairs;
      3) further require Activity > min_activity.
    """
    rows = []

    for mi in mi_order_use:
        tmp = full_metric_df.loc[full_metric_df["MI"] == mi].copy()
        tmp = tmp.loc[np.isfinite(tmp["Activity"].values)].copy()

        if tmp.shape[0] == 0:
            continue

        tmp = tmp.sort_values(
            ["Activity", "Max_normalized_score", "MI_contribution"],
            ascending=False,
        ).reset_index(drop=True)

        n_total_pairs = int(tmp.shape[0])
        n_top_candidate = int(np.ceil(float(top_fraction) * n_total_pairs))
        n_top_candidate = max(1, min(n_top_candidate, n_total_pairs))

        tmp_top = tmp.head(n_top_candidate).copy()
        tmp_sel = tmp_top.loc[
            tmp_top["Activity"] > float(min_activity)
        ].copy()

        if tmp_sel.shape[0] == 0:
            continue

        tmp_sel["Top_fraction_activity_rank_within_MI"] = np.arange(1, tmp_sel.shape[0] + 1)
        tmp_sel["Top_fraction"] = float(top_fraction)
        tmp_sel["Min_mean_MI_activity"] = float(min_activity)
        tmp_sel["Top_fraction_candidate_n"] = int(n_top_candidate)
        tmp_sel["Top_fraction_selected_n"] = int(tmp_sel.shape[0])
        tmp_sel["Total_eligible_pair_count"] = int(n_total_pairs)

        rows.append(tmp_sel)

    if len(rows) == 0:
        print(
            f"No sender→receiver pairs selected for top_fraction={top_fraction} "
            f"and Activity > {min_activity}."
        )
        return pd.DataFrame(
            columns=list(full_metric_df.columns)
            + [
                "Top_fraction_activity_rank_within_MI",
                "Top_fraction",
                "Min_mean_MI_activity",
                "Top_fraction_candidate_n",
                "Top_fraction_selected_n",
                "Total_eligible_pair_count",
                "MI_custom_order",
            ]
        )

    selected_df = pd.concat(rows, axis=0, ignore_index=True)

    mi_order_map = {mi: i for i, mi in enumerate(mi_order_use)}
    selected_df["MI_custom_order"] = selected_df["MI"].map(mi_order_map)

    return selected_df


def _compute_top_fraction_activity_filtered_summary_stats(
    full_metric_df,
    selected_df,
    mi_order_use,
    top_fraction,
    min_activity,
):
    """
    Compute per-MI summary statistics for selected pairs.
    """
    rows = []

    for mi in mi_order_use:
        tmp_all = full_metric_df.loc[full_metric_df["MI"] == mi].copy()
        tmp_all = tmp_all.loc[np.isfinite(tmp_all["Activity"].values)].copy()

        tmp_sel = selected_df.loc[selected_df["MI"] == mi].copy()

        n_total_pairs = int(tmp_all.shape[0])
        n_top_candidate = (
            max(1, min(int(np.ceil(float(top_fraction) * n_total_pairs)), n_total_pairs))
            if n_total_pairs > 0
            else 0
        )

        n_selected_pairs = int(tmp_sel.shape[0])

        n_selected_tumor_pairs = (
            int(tmp_sel["Tumor_involved_pair"].sum())
            if n_selected_pairs > 0
            else 0
        )

        n_all_tumor_pairs = (
            int(tmp_all["Tumor_involved_pair"].sum())
            if n_total_pairs > 0
            else 0
        )

        selected_pair_ratio = (
            n_selected_pairs / n_total_pairs
            if n_total_pairs > 0
            else np.nan
        )

        selected_tumor_pair_fraction = (
            n_selected_tumor_pairs / n_selected_pairs
            if n_selected_pairs > 0
            else np.nan
        )

        all_tumor_pair_fraction = (
            n_all_tumor_pairs / n_total_pairs
            if n_total_pairs > 0
            else np.nan
        )

        tumor_fraction_enrichment = (
            selected_tumor_pair_fraction / all_tumor_pair_fraction
            if (
                np.isfinite(selected_tumor_pair_fraction)
                and np.isfinite(all_tumor_pair_fraction)
                and all_tumor_pair_fraction > 0
            )
            else np.nan
        )

        rows.append({
            "MI": mi,
            "MI_custom_order": mi_order_use.index(mi),
            "Top_fraction": float(top_fraction),
            "Min_mean_MI_activity": float(min_activity),
            "Total_eligible_pair_count": n_total_pairs,
            "Eligible_tumor_involved_pair_count": n_all_tumor_pairs,
            "Eligible_tumor_involved_pair_fraction": all_tumor_pair_fraction,
            "Top_fraction_candidate_pair_count": n_top_candidate,
            "Selected_pair_count": n_selected_pairs,
            "Selected_pair_ratio": selected_pair_ratio,
            "Selected_tumor_involved_pair_count": n_selected_tumor_pairs,
            "Selected_tumor_involved_pair_fraction": selected_tumor_pair_fraction,
            "Selected_over_eligible_tumor_fraction_ratio": tumor_fraction_enrichment,
            "Selected_activity_mean": tmp_sel["Activity"].mean() if n_selected_pairs > 0 else np.nan,
            "Selected_activity_median": tmp_sel["Activity"].median() if n_selected_pairs > 0 else np.nan,
            "Selected_activity_min": tmp_sel["Activity"].min() if n_selected_pairs > 0 else np.nan,
            "Selected_activity_max": tmp_sel["Activity"].max() if n_selected_pairs > 0 else np.nan,
            "Min_edges_per_pair": int(MIN_EDGES_PER_PAIR),
        })

    stats_df = pd.DataFrame(rows)

    return stats_df


def _compute_selected_tumor_involved_type_heatmap_maxnorm(
    selected_df,
    mi_order_use,
    tumor_type_order,
):
    """
    Count tumor types ONLY from selected tumor-involved sender→receiver pairs.

    For each MI and tumor type:
      count selected pairs satisfying:
          Tumor_involved_pair == True
      and involving that tumor type.

    Outputs:
      tumor_involved_count_df:
          raw count matrix based only on selected tumor-involved pairs.
      tumor_involved_maxnorm_df:
          row-wise max-normalized count matrix.

    Columns are sorted by raw tumor-involved column sum descending.
    """
    if selected_df.shape[0] > 0 and "Tumor_involved_pair" in selected_df.columns:
        selected_tumor_df = selected_df.loc[
            selected_df["Tumor_involved_pair"].astype(bool)
        ].copy()
    else:
        selected_tumor_df = selected_df.iloc[0:0].copy()

    all_tumor_types = []

    if selected_tumor_df.shape[0] > 0 and "Tumor_types_in_pair" in selected_tumor_df.columns:
        for tumor_types in selected_tumor_df["Tumor_types_in_pair"].tolist():
            all_tumor_types.extend(tumor_types)

    all_tumor_types = sorted(set(all_tumor_types))

    tumor_type_cols = [
        x for x in tumor_type_order
        if x in set(all_tumor_types)
    ]

    extra_tumor_types = [
        x for x in all_tumor_types
        if x not in set(tumor_type_cols)
    ]

    tumor_type_cols = tumor_type_cols + extra_tumor_types

    if len(tumor_type_cols) == 0:
        tumor_type_cols = tumor_type_order.copy()

    tumor_involved_count_df = pd.DataFrame(
        0.0,
        index=mi_order_use,
        columns=tumor_type_cols,
    )

    for _, row in selected_tumor_df.iterrows():
        mi = str(row["MI"])
        tumor_types = row["Tumor_types_in_pair"]

        if mi not in tumor_involved_count_df.index:
            continue

        for tumor_type in tumor_types:
            if tumor_type in tumor_involved_count_df.columns:
                tumor_involved_count_df.loc[mi, tumor_type] += 1.0

    col_order = (
        tumor_involved_count_df
        .sum(axis=0)
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    tumor_involved_count_df = tumor_involved_count_df.loc[:, col_order]

    row_max = tumor_involved_count_df.max(axis=1).replace(0, np.nan)

    tumor_involved_maxnorm_df = (
        tumor_involved_count_df
        .div(row_max, axis=0)
        .fillna(0.0)
    )

    return tumor_involved_count_df, tumor_involved_maxnorm_df


def _sort_mi_order_by_heatmap_maxnorm_row_sum(
    mi_order_use,
    tumor_type_maxnorm_df,
):
    """
    Sort MI rows by MAX-NORMALIZED heatmap row sum descending.
    """
    row_sum = (
        tumor_type_maxnorm_df
        .reindex(mi_order_use)
        .fillna(0.0)
        .sum(axis=1)
    )

    original_rank = pd.Series(
        np.arange(len(mi_order_use)),
        index=mi_order_use,
        name="original_rank",
    )

    sort_df = pd.DataFrame({
        "maxnorm_row_sum": row_sum,
        "original_rank": original_rank,
    })

    sort_df = sort_df.sort_values(
        ["maxnorm_row_sum", "original_rank"],
        ascending=[False, True],
    )

    sorted_mi_order = sort_df.index.astype(str).tolist()

    print("\nMI row order sorted by heatmap max-normalized row sum descending:")
    print(sorted_mi_order)

    return sorted_mi_order


def _build_focused_pair_order_for_dotplot(
    selected_df,
    mi_order_use,
):
    """
    Build sender→receiver pair order for the dot plot using final MI order.

    For each MI in final mi_order_use:
      - sort its selected pairs by within-MI rank / Activity
      - append pairs not already used
    """
    pair_order = []
    seen_pairs = set()

    for mi in mi_order_use:
        tmp = selected_df.loc[selected_df["MI"].astype(str) == str(mi)].copy()

        if tmp.shape[0] == 0:
            continue

        sort_cols = []
        ascending = []

        if "Top_fraction_activity_rank_within_MI" in tmp.columns:
            sort_cols.append("Top_fraction_activity_rank_within_MI")
            ascending.append(True)

        sort_cols.extend(["Activity", "Max_normalized_score", "MI_contribution", "Pair"])
        ascending.extend([False, False, False, True])

        tmp = tmp.sort_values(sort_cols, ascending=ascending)

        for pair in tmp["Pair"].astype(str).tolist():
            if pair not in seen_pairs:
                pair_order.append(pair)
                seen_pairs.add(pair)

    remaining_pairs = [
        pair for pair in selected_df["Pair"].astype(str).tolist()
        if pair not in seen_pairs
    ]

    for pair in remaining_pairs:
        if pair not in seen_pairs:
            pair_order.append(pair)
            seen_pairs.add(pair)

    return pair_order


def _plot_selected_bubble_plot(
    full_metric_df,
    selected_df,
    mi_order_use,
    top_fraction,
    min_activity,
):
    """
    Bubble plot showing selected MI-pair dots.
    Row order is rebuilt according to final MI order so the selected
    sender→receiver pairs form more focused MI-specific blocks.
    """
    if selected_df.shape[0] == 0:
        print("Skipping selected-pair bubble plot because no pairs were selected.")
        return None, None

    pair_order = _build_focused_pair_order_for_dotplot(
        selected_df=selected_df,
        mi_order_use=mi_order_use,
    )

    if len(pair_order) == 0:
        print("Skipping selected-pair bubble plot because pair_order is empty.")
        return None, None

    plot_df = selected_df.copy()

    mi_order_map = {mi: i for i, mi in enumerate(mi_order_use)}
    plot_df["MI_custom_order_final"] = plot_df["MI"].astype(str).map(mi_order_map)

    plot_df["MI"] = pd.Categorical(
        plot_df["MI"].astype(str),
        categories=mi_order_use,
        ordered=True,
    )

    plot_df["Pair"] = pd.Categorical(
        plot_df["Pair"].astype(str),
        categories=pair_order,
        ordered=True,
    )

    plot_df["x"] = plot_df["MI"].cat.codes
    plot_df["y"] = plot_df["Pair"].cat.codes

    plot_df = plot_df.loc[
        (plot_df["x"] >= 0) & (plot_df["y"] >= 0)
    ].copy()

    norm = plt.Normalize(vmin=0, vmax=1)

    fig_width = max(18, 0.95 * len(mi_order_use) + 8)
    fig_height = max(10, 0.42 * len(pair_order) + 6)

    plt.close("all")
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor="white")
    ax.set_facecolor("white")

    sizes = _scale_size(
        plot_df["Activity"].values,
        global_values=full_metric_df["Activity"].values,
        min_size=55,
        max_size=760,
    ) * DOT_SIZE_MULTIPLIER

    sc = ax.scatter(
        plot_df["x"],
        plot_df["y"],
        s=sizes,
        c=plot_df["Max_normalized_score"].values,
        cmap="Reds",
        norm=norm,
        edgecolors="face",
        linewidths=1.0,
        alpha=0.95,
    )

    ax.set_xticks(np.arange(len(mi_order_use)))
    ax.set_xticklabels(
        mi_order_use,
        rotation=90,
        ha="center",
        va="top",
        fontsize=FONT_XTICK,
    )

    ax.set_yticks(np.arange(len(pair_order)))
    ax.set_yticklabels(
        [_format_pair_label(p) for p in pair_order],
        fontsize=max(8, FONT_YTICK - 3),
    )

    ax.set_ylim(len(pair_order) - 0.75, -0.25)

    ax.set_xlabel("Meta-interaction IDs", fontsize=FONT_AXIS_LABEL, labelpad=25)
    ax.set_ylabel("Sender  →  receiver cell-type pairs", fontsize=FONT_AXIS_LABEL, labelpad=25)

    ax.set_title(
        f"Pan-cancer MI-specific sender→receiver interactions\n"
        f"Showing top {top_fraction:.0%} pairs per MI with mean MI activity > {min_activity}",
        fontsize=FONT_TITLE,
        pad=35,
    )

    ax.grid(axis="both", color="#E6E6E6", linewidth=1.0)
    ax.set_axisbelow(True)

    block_boundaries = []
    used_pairs = set()
    running_count = 0

    for mi in mi_order_use:
        tmp = selected_df.loc[selected_df["MI"].astype(str) == str(mi)].copy()

        if tmp.shape[0] == 0:
            continue

        tmp = tmp.sort_values(
            ["Top_fraction_activity_rank_within_MI", "Activity"],
            ascending=[True, False],
        )

        tmp_pairs = tmp["Pair"].astype(str).tolist()
        new_pairs = [p for p in tmp_pairs if p not in used_pairs]

        if len(new_pairs) > 0:
            running_count += len(new_pairs)
            block_boundaries.append(running_count - 0.5)

            for p in new_pairs:
                used_pairs.add(p)

    for boundary in block_boundaries[:-1]:
        ax.axhline(
            boundary,
            color="#BDBDBD",
            linewidth=0.8,
            linestyle="--",
            alpha=0.7,
        )

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
        spine.set_color("#333333")

    cbar = plt.colorbar(sc, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label(
        "Max-normalized MI score\nwithin each MI",
        fontsize=FONT_CBAR_LABEL,
        labelpad=25,
    )
    cbar.ax.tick_params(labelsize=FONT_CBAR_TICK)

    plt.tight_layout()

    pdf_path, png_path = _safe_savefig(
        fig=fig,
        stem="dotplot_blocks",
        dpi=300,
    )

    plt.show()
    plt.close()

    print(f"Saved focused-block selected-pair bubble plot PDF: {pdf_path}")
    print(f"Saved focused-block selected-pair bubble plot PNG: {png_path}")

    return pdf_path, png_path


def _annotate_barh_counts(ax, values, x_offset_frac=0.02, fontsize=12):
    """
    Annotate horizontal bars with integer counts.
    """
    values = np.asarray(values, dtype=float)
    finite_values = values[np.isfinite(values)]
    xmax = np.nanmax(finite_values) if finite_values.size > 0 else 1.0

    offset = max(xmax * x_offset_frac, 0.05)

    for i, val in enumerate(values):
        if not np.isfinite(val):
            continue

        ax.text(
            val + offset,
            i,
            f"{int(round(val))}",
            va="center",
            ha="left",
            fontsize=fontsize,
        )


HEATMAP_GRID_COLOR = "#b7b9bb"

def _plot_combined_selected_summary_figure(
    selected_df,
    selected_stats_df,
    tumor_type_tumor_involved_count_df,
    tumor_type_tumor_involved_maxnorm_df,
    mi_order_use,
    top_fraction,
    min_activity,
):
    """
    Three panels arranged horizontally with MIs as rows:
      1) selected-pair count, annotated
      2) tumor-involved fraction among selected pairs,
         with per-MI vertical tick marking eligible all-pair tumor fraction
         x-axis is flipped left-right
      3) tumor-type max-normalized heatmap based on selected tumor-involved pair counts,
         WITHOUT raw count annotation.
         Heatmap cell borders are shown in HEATMAP_GRID_COLOR.
    """
    stats_plot = (
        selected_stats_df
        .set_index("MI")
        .reindex(mi_order_use)
        .copy()
    )

    count_df = (
        tumor_type_tumor_involved_count_df
        .reindex(mi_order_use)
        .fillna(0.0)
        .copy()
    )

    heatmap_df = (
        tumor_type_tumor_involved_maxnorm_df
        .reindex(mi_order_use)
        .fillna(0.0)
        .copy()
    )

    count_df = count_df.loc[:, heatmap_df.columns]

    n_mi = len(mi_order_use)
    y = np.arange(n_mi)

    previous_fig_width = max(22, 1.2 * len(heatmap_df.columns) + 15)
    fig_width = max(8.5, previous_fig_width * FIG_WIDTH_SCALE)
    fig_height = max(8, 0.52 * n_mi + 3)

    plt.close("all")
    fig, axes = plt.subplots(
        nrows=1,
        ncols=3,
        figsize=(fig_width, fig_height),
        facecolor="white",
        gridspec_kw={
            "width_ratios": [
                1.05,
                1.25,
                max(2.3, 0.52 * len(heatmap_df.columns)),
            ]
        },
        sharey=True,
    )

    ax_count, ax_tumor_frac, ax_heatmap = axes

    # --------------------------------------------------------
    # Panel 1: selected pair count
    # --------------------------------------------------------
    count_values = stats_plot["Selected_pair_count"].fillna(0).values.astype(float)

    ax_count.barh(
        y,
        count_values,
        color=BAR_FILL_COLOR,
        edgecolor="none",
        linewidth=0,
    )

    _annotate_barh_counts(
        ax=ax_count,
        values=count_values,
        x_offset_frac=0.03,
        fontsize=max(9, FONT_XTICK - 3),
    )

    ax_count.set_title(
        "Selected pair count",
        fontsize=FONT_AXIS_LABEL,
        pad=15,
    )

    ax_count.set_xlabel("Count", fontsize=FONT_AXIS_LABEL)
    ax_count.set_yticks(y)
    ax_count.set_yticklabels(mi_order_use, fontsize=FONT_YTICK)
    ax_count.set_ylim(n_mi - 0.5, -0.5)

    count_xmax = np.nanmax(count_values) if np.isfinite(count_values).any() else 1.0
    ax_count.set_xlim(0, max(1.0, count_xmax * 1.25))

    ax_count.grid(axis="x", color="#E6E6E6", linewidth=1.0)
    ax_count.set_axisbelow(True)

    # --------------------------------------------------------
    # Panel 2: tumor-involved fraction among selected pairs
    # plus background eligible tumor-involved fraction as per-MI tick
    # x-axis is horizontally flipped.
    # --------------------------------------------------------
    tumor_frac_values = (
        stats_plot["Selected_tumor_involved_pair_fraction"]
        .fillna(0)
        .values
        .astype(float)
    )

    eligible_tumor_frac_values = (
        stats_plot["Eligible_tumor_involved_pair_fraction"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .values
        .astype(float)
    )

    ax_tumor_frac.barh(
        y,
        tumor_frac_values,
        color=BAR_FILL_COLOR,
        edgecolor="none",
        linewidth=0,
        label="Selected pairs",
    )

    tick_half_height = 0.36
    ax_tumor_frac.vlines(
        x=eligible_tumor_frac_values,
        ymin=y - tick_half_height,
        ymax=y + tick_half_height,
        colors="black",
        linewidth=2.0,
        label="All eligible pairs",
    )

    ax_tumor_frac.set_title(
        "Tumor-involved fraction\nselected vs. all eligible",
        fontsize=FONT_AXIS_LABEL,
        pad=15,
    )

    ax_tumor_frac.set_xlabel("Fraction", fontsize=FONT_AXIS_LABEL)

    # Left-right flip.
    ax_tumor_frac.set_xlim(1.0, 0.0)

    ax_tumor_frac.grid(axis="x", color="#E6E6E6", linewidth=1.0)
    ax_tumor_frac.set_axisbelow(True)

    ax_tumor_frac.legend(
        frameon=False,
        loc="lower left",
        fontsize=max(8, FONT_XTICK - 4),
        handlelength=1.4,
    )

    # --------------------------------------------------------
    # Panel 3: tumor-type max-normalized heatmap
    # based on selected tumor-involved pair counts
    # 0 values are shown as gray.
    # Raw count values are NOT annotated.
    # Cell borders are shown in HEATMAP_GRID_COLOR.
    # --------------------------------------------------------
    heatmap_values = heatmap_df.values.astype(float)
    heatmap_masked = np.ma.masked_where(heatmap_values <= 0, heatmap_values)

    ylorrd_cmap = mpl.colormaps["YlOrRd"].copy()
    ylorrd_cmap.set_bad("#D9D9D9")

    im = ax_heatmap.imshow(
        heatmap_masked,
        aspect="auto",
        interpolation="nearest",
        vmin=0,
        vmax=1,
        cmap=ylorrd_cmap,
    )

    ax_heatmap.set_title(
        "Tumor-type composition\nselected tumor-involved pairs",
        fontsize=FONT_AXIS_LABEL,
        pad=15,
    )

    ax_heatmap.set_xticks(np.arange(len(heatmap_df.columns)))
    ax_heatmap.set_xticklabels(
        heatmap_df.columns,
        rotation=90,
        ha="center",
        va="top",
        fontsize=FONT_XTICK,
    )

    ax_heatmap.set_yticks(y)
    ax_heatmap.set_yticklabels(mi_order_use, fontsize=FONT_YTICK)

    ax_heatmap.set_xlim(-0.5, len(heatmap_df.columns) - 0.5)
    ax_heatmap.set_ylim(n_mi - 0.5, -0.5)

    # Add heatmap cell borders.
    ax_heatmap.set_xticks(
        np.arange(-0.5, len(heatmap_df.columns), 1),
        minor=True,
    )
    ax_heatmap.set_yticks(
        np.arange(-0.5, n_mi, 1),
        minor=True,
    )
    ax_heatmap.grid(
        which="minor",
        color=HEATMAP_GRID_COLOR,
        linestyle="-",
        linewidth=0.8,
    )
    ax_heatmap.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    # Raw count annotation intentionally removed.

    cbar = plt.colorbar(im, ax=ax_heatmap, fraction=0.046, pad=0.03)
    cbar.set_label(
        "Max-normalized\ncount",
        fontsize=FONT_CBAR_LABEL,
    )
    cbar.ax.tick_params(labelsize=FONT_CBAR_TICK)

    # --------------------------------------------------------
    # Shared styling
    # --------------------------------------------------------
    for ax in axes:
        ax.tick_params(axis="both", labelsize=FONT_XTICK)
        for spine in ax.spines.values():
            spine.set_linewidth(1.0)
            spine.set_color("#333333")

    ax_count.tick_params(axis="y", labelsize=FONT_YTICK)
    ax_tumor_frac.tick_params(axis="y", labelleft=False)
    ax_heatmap.tick_params(axis="y", labelleft=False)

    fig.suptitle(
        f"Top {top_fraction:.0%} sender→receiver pairs with mean MI activity > {min_activity}",
        fontsize=FONT_TITLE,
        y=1.02,
    )

    plt.tight_layout()

    pdf_path, png_path = _safe_savefig(
        fig=fig,
        stem="summary_3panel",
        dpi=300,
    )

    plt.show()
    plt.close()

    print(f"Saved combined 3-panel summary PDF: {pdf_path}")
    print(f"Saved combined 3-panel summary PNG: {png_path}")

    return pdf_path, png_path


# ============================================================
# Step 1. Load / reconstruct edge metadata and MI matrix
# ============================================================

processed = _ensure_processed_loaded()

sender_all_pc, receiver_all_pc, pair_all_pc, batch_all_pc = (
    _collect_edge_sender_receiver_from_processed(processed)
)

factor_matrix_pc = _load_factor_matrix(n_edges_expected=len(sender_all_pc))

n_edges_pc, n_mi_pc = factor_matrix_pc.shape

mi_names_pc, mi_order_pc_initial = _resolve_native_mi_names_and_plot_order(n_mi_pc)

print("\nFactor_envir_use shape:", factor_matrix_pc.shape)
print("Number of reconstructed edges:", len(sender_all_pc))
print("Number of MIs:", n_mi_pc)


# ============================================================
# Step 2. Compute global sender→receiver pair metrics
# ============================================================

full_metric_df, activity_df, maxnorm_df, contribution_df = (
    _build_global_pair_metric_table(
        factor_matrix=factor_matrix_pc,
        sender_all=sender_all_pc,
        receiver_all=receiver_all_pc,
        pair_all=pair_all_pc,
        mi_names=mi_names_pc,
    )
)


# ============================================================
# Step 3. Select top 10% pairs per MI with Activity > 0.1
# ============================================================

selected_pair_df = _build_top_fraction_activity_filtered_pairs(
    full_metric_df=full_metric_df,
    mi_order_use=mi_order_pc_initial,
    top_fraction=TOP_PAIR_FRACTION,
    min_activity=MIN_MEAN_MI_ACTIVITY,
)

print(
    f"\nNumber of selected MI-pair entries "
    f"(top {TOP_PAIR_FRACTION:.0%} and Activity > {MIN_MEAN_MI_ACTIVITY}): "
    f"{selected_pair_df.shape[0]}"
)

display(
    selected_pair_df
    .sort_values(["MI_custom_order", "Top_fraction_activity_rank_within_MI"])
    .head(100)
)


# ============================================================
# Step 4. Compute selected-pair summary statistics and heatmap matrix
# ============================================================

selected_stats_df = _compute_top_fraction_activity_filtered_summary_stats(
    full_metric_df=full_metric_df,
    selected_df=selected_pair_df,
    mi_order_use=mi_order_pc_initial,
    top_fraction=TOP_PAIR_FRACTION,
    min_activity=MIN_MEAN_MI_ACTIVITY,
)

tumor_type_tumor_involved_count_df, tumor_type_tumor_involved_maxnorm_df = (
    _compute_selected_tumor_involved_type_heatmap_maxnorm(
        selected_df=selected_pair_df,
        mi_order_use=mi_order_pc_initial,
        tumor_type_order=TUMOR_TYPE_ORDER,
    )
)

# Final row order for the figure:
# sort MIs by max-normalized heatmap row sum descending.
mi_order_pc = _sort_mi_order_by_heatmap_maxnorm_row_sum(
    mi_order_use=mi_order_pc_initial,
    tumor_type_maxnorm_df=tumor_type_tumor_involved_maxnorm_df,
)

# Update selected-pair MI order after final row ordering.
final_mi_order_map = {mi: i for i, mi in enumerate(mi_order_pc)}
selected_pair_df["MI_custom_order_final"] = (
    selected_pair_df["MI"].astype(str).map(final_mi_order_map)
)


# ============================================================
# Step 5. Save tables
# Short filenames to avoid Windows path-length errors
# ============================================================

full_metric_path = _safe_to_csv(
    full_metric_df,
    "full_metric.csv",
    index=False,
)

selected_pair_path = _safe_to_csv(
    selected_pair_df,
    "sel_pairs.csv",
    index=False,
)

selected_pair_sorted_path = _safe_to_csv(
    selected_pair_df.sort_values(
        ["MI_custom_order_final", "Top_fraction_activity_rank_within_MI", "Activity"],
        ascending=[True, True, False],
    ),
    "sel_pairs_sorted.csv",
    index=False,
)

selected_stats_path = _safe_to_csv(
    selected_stats_df,
    "stats.csv",
    index=False,
)

selected_stats_sorted_path = _safe_to_csv(
    selected_stats_df
    .set_index("MI")
    .reindex(mi_order_pc)
    .reset_index(),
    "stats_sorted.csv",
    index=False,
)

tumor_type_tumor_involved_count_path = _safe_to_csv(
    tumor_type_tumor_involved_count_df,
    "tumor_count.csv",
    index=True,
)

tumor_type_tumor_involved_count_sorted_rows_path = _safe_to_csv(
    tumor_type_tumor_involved_count_df.reindex(mi_order_pc),
    "tumor_count_sorted.csv",
    index=True,
)

tumor_type_tumor_involved_maxnorm_path = _safe_to_csv(
    tumor_type_tumor_involved_maxnorm_df,
    "tumor_maxnorm.csv",
    index=True,
)

tumor_type_tumor_involved_maxnorm_sorted_rows_path = _safe_to_csv(
    tumor_type_tumor_involved_maxnorm_df.reindex(mi_order_pc),
    "tumor_maxnorm_sorted.csv",
    index=True,
)

activity_matrix_path = _safe_to_csv(
    activity_df,
    "activity_mat.csv",
    index=True,
)

maxnorm_matrix_path = _safe_to_csv(
    maxnorm_df,
    "maxnorm_mat.csv",
    index=True,
)

contribution_matrix_path = _safe_to_csv(
    contribution_df,
    "contrib_mat.csv",
    index=True,
)

print(f"\nSaved full metric table: {full_metric_path}")
print(f"Saved selected pairs: {selected_pair_path}")
print(f"Saved selected pairs sorted by max-normalized heatmap row sum: {selected_pair_sorted_path}")
print(f"Saved selected-pair summary stats: {selected_stats_path}")
print(f"Saved sorted selected-pair summary stats: {selected_stats_sorted_path}")
print(f"Saved tumor-involved pair tumor-type count matrix: {tumor_type_tumor_involved_count_path}")
print(f"Saved row-sorted tumor-involved pair tumor-type count matrix: {tumor_type_tumor_involved_count_sorted_rows_path}")
print(f"Saved tumor-involved pair tumor-type max-normalized matrix: {tumor_type_tumor_involved_maxnorm_path}")
print(f"Saved row-sorted tumor-involved pair tumor-type max-normalized matrix: {tumor_type_tumor_involved_maxnorm_sorted_rows_path}")


# ============================================================
# Step 6. Selected-pair focused-block dot plot
# ============================================================

selected_bubble_pdf, selected_bubble_png = _plot_selected_bubble_plot(
    full_metric_df=full_metric_df,
    selected_df=selected_pair_df,
    mi_order_use=mi_order_pc,
    top_fraction=TOP_PAIR_FRACTION,
    min_activity=MIN_MEAN_MI_ACTIVITY,
)


# ============================================================
# Step 7. Combined 3-panel figure
# ============================================================

combined_summary_pdf, combined_summary_png = _plot_combined_selected_summary_figure(
    selected_df=selected_pair_df,
    selected_stats_df=selected_stats_df,
    tumor_type_tumor_involved_count_df=tumor_type_tumor_involved_count_df,
    tumor_type_tumor_involved_maxnorm_df=tumor_type_tumor_involved_maxnorm_df,
    mi_order_use=mi_order_pc,
    top_fraction=TOP_PAIR_FRACTION,
    min_activity=MIN_MEAN_MI_ACTIVITY,
)


# ============================================================
# Step 8. Display summary tables
# ============================================================

print("\nSelected-pair summary statistics, sorted by max-normalized heatmap row sum:")
display(
    selected_stats_df
    .set_index("MI")
    .reindex(mi_order_pc)
    .reset_index()
)

print("\nSelected pairs, sorted by final MI order:")
display(
    selected_pair_df
    .sort_values(
        ["MI_custom_order_final", "Top_fraction_activity_rank_within_MI", "Activity"],
        ascending=[True, True, False],
    )
    .head(100)
)

print("\nTumor-type raw count matrix based on selected tumor-involved pairs, rows sorted by max-normalized row sum and columns sorted by raw column sum:")
display(
    tumor_type_tumor_involved_count_df
    .reindex(mi_order_pc)
)

print("\nTumor-type max-normalized matrix based on selected tumor-involved pairs, rows sorted by max-normalized row sum and columns sorted by raw column sum:")
display(
    tumor_type_tumor_involved_maxnorm_df
    .reindex(mi_order_pc)
)


# ============================================================
# Optional cleanup
# ============================================================

if "release_memory" in globals():
    release_memory(
        "sender_all_pc",
        "receiver_all_pc",
        "pair_all_pc",
        "batch_all_pc",
        "factor_matrix_pc",
        namespace=globals(),
        run_gc=True,
        clear_cuda=False,
        close_figures=False,
    )
else:
    gc.collect()

### Order the LR pathway heatmap

Align pathway rows to the MI ordering from the top-pair analysis. The subsequent LR-pair stem plot is an additional molecular-loading view.


In [ ]:
mi_order = list(final_mi_order_map.keys())


In [ ]:
##Load LR_loading_pathway
# LR_loading_pathway.to_csv(os.path.join(file_savepath_main, "LR_loading_pathway.csv"))
LR_loading_pathway = pd.read_csv(run_dirs['run_dir'] / "LR_loading_pathway.csv",index_col=0)
MI_orderbyage = mi_order
##Change the "MI_" to "MI-" in the index of LR_loading_pathway
MI_orderbyage = [x.replace("MI_", "MI-") for x in MI_orderbyage]
LR_loading_pathway = LR_loading_pathway.loc[MI_orderbyage, :]
LR_loading_pathway = LR_loading_pathway.iloc[:,
                     np.argsort(np.array(np.max(LR_loading_pathway, axis=0)))[::-1]
                     ]
# Refined pathway ordering
argmax_1 = np.argmax(np.array(LR_loading_pathway), axis=0)
max_1 = np.max(np.array(LR_loading_pathway), axis=0)
order_index_LRpathway = []
for argmax_1_cur in np.sort(np.unique(argmax_1)):
    index_cur = np.where(argmax_1 == argmax_1_cur)[0]
    index_cur = index_cur[np.argsort(max_1[index_cur])[::-1]]
    index_cur = index_cur.tolist()
    order_index_LRpathway.extend(index_cur)

LR_loading_pathway = LR_loading_pathway.iloc[:, order_index_LRpathway]


In [ ]:
# Visualization
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import os
plt.close()
fig, ax = plt.subplots(figsize=(14, 8))
# cmap = LinearSegmentedColormap.from_list(
#     "white_red", ["white", "#FFDFEF", "#EABDE6", "#D69ADE", "#AA60C8"], N=256
# )
cmap = LinearSegmentedColormap.from_list(
    "white_red", ["#FCF5F0", "#F9B2BC", "#F6689F", "#C31988", "#510269"], N=256
)

data = LR_loading_pathway.values
vmin = data.min()
# vmax = min(0.4, np.max(data) * 0.7)
vmax = np.max(data) * 0.6
norm = TwoSlopeNorm(vmin=vmin, vcenter=(vmin + vmax) / 2, vmax=vmax)

mesh = ax.pcolormesh(
    np.arange(data.shape[1] + 1),
    np.arange(data.shape[0] + 1),
    data,
    cmap=cmap,
    norm=norm,
    edgecolors="#B6B9BA",
    linewidth=1.0
)

ax.set_xticks(np.arange(data.shape[1]) + 0.5)
# ax.set_xticklabels(LR_loading_pathway.columns, rotation=60, fontsize=22)
ax.set_xticklabels(
    LR_loading_pathway.columns,
    rotation=60,
    fontsize=22,
    ha="left",
    rotation_mode="anchor"
)
ax.set_yticks(np.arange(data.shape[0]) + 0.5)
ax.set_yticklabels(LR_loading_pathway.index, fontsize=22)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.invert_yaxis()

plt.colorbar(mesh, ax=ax)
plt.tight_layout()

save_png = os.path.join(run_dirs['run_dir'], "LR_loading_pathway.png")
plt.savefig(save_png, format="png", bbox_inches="tight", dpi=300)

save_pdf = os.path.join(run_dirs['run_dir'], "LR_loading_pathway.pdf")
plt.savefig(save_pdf, format="pdf", bbox_inches="tight", dpi=300)

plt.show()
plt.close()


release_memory(
    "data",
    "vmin",
    "vmax",
    "norm",
    "mesh",
    "fig",
    "ax",
    "LR_loading_pathway",
    "MI_orderbyage",
    "argmax_1",
    "max_1",
    "order_index_LRpathway",
    "index_cur",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
    close_figures=True
)


In [ ]:
# ------------------------------------------------------------
# Stem plot: top LR pairs for one MI of interest
# Read LR-pair loadings from loading_LR_use / loading_LR_use.csv,
# rather than the pathway-aggregated LR_loading_pathway.csv.
# ------------------------------------------------------------
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Illustrator-friendly settings
# ------------------------------------------------------------
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["xtick.major.width"] = 0.8
plt.rcParams["ytick.major.width"] = 0.8
plt.rcParams["xtick.major.size"] = 3
plt.rcParams["ytick.major.size"] = 3

MI_OI = "MI4"
top_n_lr = 10

run_dir = Path(run_dirs["run_dir"])


# ------------------------------------------------------------
# Load LR-pair loading matrix
# Expected matrix:
#   rows    = MI dimensions
#   columns = LR pairs
# ------------------------------------------------------------
if "loading_LR_use" in globals():
    loading_LR_stem = loading_LR_use.copy()
    print("Using existing variable: loading_LR_use")

elif "Loading_LR_use" in globals():
    loading_LR_stem = Loading_LR_use.copy()
    print("Using existing variable: Loading_LR_use")

else:
    candidate_paths = [
        run_dir / "loading_LR_use.csv",
        run_dir / "Loading_LR_use.csv",
        run_dir / "LR_loading_use.csv",
        run_dir / "loading_LR.csv",
        run_dir / "LR_loading.csv",
    ]

    loading_lr_path = None
    for p in candidate_paths:
        if p.exists():
            loading_lr_path = p
            break

    if loading_lr_path is None:
        raise FileNotFoundError(
            "Cannot find LR-pair loading file. Tried:\n"
            + "\n".join([str(p) for p in candidate_paths])
            + "\n\nPlease check where loading_LR_use.csv was saved."
        )

    loading_LR_stem = pd.read_csv(
        loading_lr_path,
        index_col=0
    )

    print(f"Loaded LR-pair loading matrix from: {loading_lr_path}")

# Make numeric while preserving LR-pair column names
loading_LR_stem = loading_LR_stem.apply(pd.to_numeric, errors="coerce")

print("LR loading matrix shape:", loading_LR_stem.shape)
print("First 5 rows:", list(loading_LR_stem.index[:5]))
print("First 5 columns:", list(loading_LR_stem.columns[:5]))


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def _mi_name_candidates(mi_name):
    """
    Return possible MI row/column names for inputs such as:
    MI2, MI-2, MI_2, or 2.
    """
    mi_name = str(mi_name)
    digits = re.findall(r"\d+", mi_name)

    candidates = [
        mi_name,
        mi_name.replace("_", "-"),
        mi_name.replace("-", "_"),
    ]

    if len(digits) > 0:
        d = digits[-1]
        candidates.extend([
            f"MI{d}",
            f"MI-{d}",
            f"MI_{d}",
            f"Meta-interaction {d}",
            f"meta-interaction {d}",
            str(d),
        ])

    out = []
    for x in candidates:
        if x not in out:
            out.append(x)

    return out


def _extract_mi_lr_vector(df, mi_name):
    """
    Robustly extract LR-pair loading vector for one MI.

    Preferred orientation:
        rows    = MIs
        columns = LR pairs

    Fallback orientation:
        rows    = LR pairs
        columns = MIs
    """
    candidates = _mi_name_candidates(mi_name)

    # Case 1: rows are MIs, columns are LR pairs
    for cand in candidates:
        if cand in df.index:
            vec = df.loc[cand, :].copy()
            orientation = "rows_are_MIs"
            mi_used = cand
            return vec, orientation, mi_used

    # Case 2: columns are MIs, rows are LR pairs
    for cand in candidates:
        if cand in df.columns:
            vec = df.loc[:, cand].copy()
            orientation = "columns_are_MIs"
            mi_used = cand
            return vec, orientation, mi_used

    raise ValueError(
        f"Cannot find MI_OI={mi_name!r} in loading_LR_stem.\n"
        f"Tried candidates: {candidates}\n"
        f"Available first 10 rows: {list(df.index[:10])}\n"
        f"Available first 10 columns: {list(df.columns[:10])}"
    )


# ------------------------------------------------------------
# Extract top LR pairs for MI_OI
# ------------------------------------------------------------
lr_loading_vec, orientation_used, mi_name_used = _extract_mi_lr_vector(
    loading_LR_stem,
    MI_OI
)

lr_loading_vec = (
    pd.to_numeric(lr_loading_vec, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

if lr_loading_vec.shape[0] == 0:
    raise ValueError(f"No finite LR-pair loading values found for {MI_OI}.")

# Remove possible non-LR metadata-like entries if present
metadata_like_names = [
    "mi",
    "mi_id",
    "pathway",
    "pathway_name",
    "annotation",
    "name",
    "index",
]

lr_loading_vec = lr_loading_vec[
    ~lr_loading_vec.index.astype(str).str.lower().isin(metadata_like_names)
]

if lr_loading_vec.shape[0] == 0:
    raise ValueError(f"No valid LR-pair loading values remained for {MI_OI}.")

# Sort LR pairs by raw loading and select top N
top_lr_loading = (
    lr_loading_vec
    .sort_values(ascending=False)
    .head(top_n_lr)
)

# Normalize by max LR-pair loading within this MI
mi_max_loading = float(np.nanmax(lr_loading_vec.values))

if not np.isfinite(mi_max_loading) or mi_max_loading <= 0:
    raise ValueError(f"Invalid max LR loading for {MI_OI}: {mi_max_loading}")

top_lr_df = pd.DataFrame({
    "rank": np.arange(1, len(top_lr_loading) + 1),
    "MI_input": MI_OI,
    "MI_name_used": mi_name_used,
    "orientation_used": orientation_used,
    "LR_pair": top_lr_loading.index.astype(str),
    "raw_LR_loading": top_lr_loading.values.astype(float),
})

top_lr_df["normalized_LR_loading"] = (
    top_lr_df["raw_LR_loading"] / mi_max_loading
)

top_lr_df = (
    top_lr_df
    .sort_values("normalized_LR_loading", ascending=False)
    .reset_index(drop=True)
)

top_lr_df["rank"] = np.arange(1, top_lr_df.shape[0] + 1)

top_lr_csv = run_dir / f"{MI_OI}_top{top_n_lr}_LR_pair_loading_stem_table.csv"
top_lr_df.to_csv(top_lr_csv, index=False)

print(f"MI used: {mi_name_used}")
print(f"Orientation used: {orientation_used}")
print(f"Saved top LR-pair loading table to: {top_lr_csv}")
display(top_lr_df)


# ------------------------------------------------------------
# Horizontal stem / lollipop plot
# Highest normalized loading shown at the top
# ------------------------------------------------------------
plot_df = (
    top_lr_df
    .sort_values("normalized_LR_loading", ascending=True)
    .reset_index(drop=True)
)

y_pos = np.arange(plot_df.shape[0])
x_val = plot_df["normalized_LR_loading"].values

fig_height = max(3.2, 0.38 * plot_df.shape[0] + 1.0)
fig, ax = plt.subplots(figsize=(5.0, fig_height))

stem_color = "#510269"

# Stem lines
ax.hlines(
    y=y_pos,
    xmin=0,
    xmax=x_val,
    color=stem_color,
    linewidth=1.2,
    zorder=1,
)

# Stem points
ax.scatter(
    x_val,
    y_pos,
    s=38,
    color=stem_color,
    edgecolors="none",
    zorder=2,
)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["LR_pair"].values, fontsize=9)

ax.set_xlabel("Normalized LR-pair loading", fontsize=10)
ax.set_title(f"Top {top_n_lr} LR pairs for {MI_OI}", fontsize=11, pad=6)

xmax = max(1.0, float(np.nanmax(x_val)) * 1.08)
ax.set_xlim(0, xmax)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=9,
    width=0.8,
    length=3,
    color="black",
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)
ax.grid(False)

plt.tight_layout()

stem_pdf = run_dir / f"{MI_OI}_top{top_n_lr}_LR_pair_loading_stem.pdf"
stem_png = run_dir / f"{MI_OI}_top{top_n_lr}_LR_pair_loading_stem.png"

plt.savefig(stem_pdf, bbox_inches="tight")
plt.savefig(stem_png, bbox_inches="tight", dpi=600)

plt.show()
plt.close()

print(f"Saved LR-pair stem plot PDF to: {stem_pdf}")
print(f"Saved LR-pair stem plot PNG to: {stem_png}")


# ------------------------------------------------------------
# Release memory
# ------------------------------------------------------------
if "release_memory" in globals():
    release_memory(
        "loading_LR_stem",
        "lr_loading_vec",
        "top_lr_loading",
        "top_lr_df",
        "plot_df",
        "fig",
        "ax",
        namespace=globals(),
        run_gc=True,
        clear_cuda=False,
        close_figures=True,
    )
else:
    del loading_LR_stem, lr_loading_vec, top_lr_loading, top_lr_df, plot_df, fig, ax

### CancerSEA regulator and target profiles

Normalize gene loadings, compute within-gene-set mean ranks, and plot the
sender and receiver profiles separately and as a split-triangle heatmap.
The CancerSEA marker files must be available at the configured resource path.


In [ ]:
## Identify candidate regulator and target genes for knockout
loading_receiver_use = np.load(run_dirs['run_dir'] / "loading_receiver_use.npy")
loading_sender_use = np.load(run_dirs['run_dir'] / "loading_sender_use.npy")
loading_receiver_use_df = pd.DataFrame(loading_receiver_use, index=["MI" + str(i+1) for i in range(loading_receiver_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_sender_use_df = pd.DataFrame(loading_sender_use, index=["MI" + str(i+1) for i in range(loading_sender_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)
loading_receiver_use_df_norm = loading_receiver_use_df.div(loading_receiver_use_df_colsum, axis=1)
loading_sender_use_df_norm = loading_sender_use_df.div(loading_sender_use_df_colsum, axis=1)

In [ ]:
## Load CancerSEA gene sets (txt files)
cancerSEA_path = str(DATA_ROOT / "CancerSEA_marker") + os.sep
##Get the list of txt files in the cancerSEA_path
cancerSEA_files = [f for f in os.listdir(cancerSEA_path) if f.endswith(".txt")]
cancerSEA_gene_sets = {}
for file in cancerSEA_files:
    gene_set_name = file.replace(".txt", "")
    ##Open the txt file and presented as dataframe (sep by '\t')
    gene_set_df = pd.read_csv(os.path.join(cancerSEA_path, file), sep='\t', header=0)
    gene_set_cur = gene_set_df['GeneName'].tolist()
    ##intersect the gene_set_cur with the columns of loading_receiver_use_df_norm and loading_sender_use_df_norm
    gene_set_cur = list(set(gene_set_cur) & set(loading_receiver_use_df_norm.columns) & set(loading_sender_use_df_norm.columns))
    cancerSEA_gene_sets[gene_set_name] = gene_set_cur
    
##Print the number of genes in each gene set
for gene_set_name, gene_set_cur in cancerSEA_gene_sets.items():
    print(f"{gene_set_name}: {len(gene_set_cur)} genes")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Step 1. Load receiver/sender loadings and normalize by column sum
# ============================================================
loading_receiver_use = np.load(run_dirs['run_dir'] / "loading_receiver_use.npy")
loading_sender_use = np.load(run_dirs['run_dir'] / "loading_sender_use.npy")

loading_receiver_use_df = pd.DataFrame(
    loading_receiver_use,
    index=["MI-" + str(i + 1) for i in range(loading_receiver_use.shape[0])],
    columns=processed.adata_list[0].var_names
)

loading_sender_use_df = pd.DataFrame(
    loading_sender_use,
    index=["MI-" + str(i + 1) for i in range(loading_sender_use.shape[0])],
    columns=processed.adata_list[0].var_names
)

loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)

loading_receiver_use_df_norm = loading_receiver_use_df.div(
    loading_receiver_use_df_colsum.replace(0, np.nan), axis=1
)
loading_sender_use_df_norm = loading_sender_use_df.div(
    loading_sender_use_df_colsum.replace(0, np.nan), axis=1
)

loading_receiver_use_df_norm = loading_receiver_use_df_norm.fillna(0)
loading_sender_use_df_norm = loading_sender_use_df_norm.fillna(0)


# ============================================================
# Step 2. Rank loading_receiver_use_df_norm by row
# Larger value -> larger rank
# ============================================================
loading_receiver_use_df_norm_rank = loading_receiver_use_df_norm.rank(
    axis=1,
    method="average",
    ascending=True
)

loading_receiver_use_df_norm_rank.to_csv(
    run_dirs['run_dir'] / "loading_receiver_use_df_norm_rank.csv"
)


# ============================================================
# Step 3. Load CancerSEA gene sets
# ============================================================
cancerSEA_path = str(DATA_ROOT / "CancerSEA_marker") + os.sep

cancerSEA_files = [f for f in os.listdir(cancerSEA_path) if f.endswith(".txt")]
cancerSEA_gene_sets = {}

for file in cancerSEA_files:
    gene_set_name = file.replace(".txt", "")
    gene_set_df = pd.read_csv(os.path.join(cancerSEA_path, file), sep="\t", header=0)

    gene_set_cur = gene_set_df["GeneName"].astype(str).tolist()

    gene_set_cur = list(
        set(gene_set_cur)
        & set(loading_receiver_use_df_norm.columns)
        & set(loading_sender_use_df_norm.columns)
    )

    cancerSEA_gene_sets[gene_set_name] = sorted(gene_set_cur)

print("Number of genes in each CancerSEA gene set:")
for gene_set_name, gene_set_cur in cancerSEA_gene_sets.items():
    print(f"{gene_set_name}: {len(gene_set_cur)} genes")


# ============================================================
# Step 4. Compute MI x CancerSEA gene-set mean-rank profile
# ============================================================
mean_rank_profile_dict = {}

for gene_set_name, gene_set_genes in cancerSEA_gene_sets.items():
    if len(gene_set_genes) == 0:
        mean_rank_profile_dict[gene_set_name] = pd.Series(
            np.nan,
            index=loading_receiver_use_df_norm_rank.index
        )
    else:
        mean_rank_profile_dict[gene_set_name] = loading_receiver_use_df_norm_rank[
            gene_set_genes
        ].mean(axis=1)

MI_cancerSEA_meanrank_df = pd.DataFrame(mean_rank_profile_dict)

# Normalize rank to 0-1 scale
MI_cancerSEA_meanrank_df = (
    MI_cancerSEA_meanrank_df / loading_receiver_use_df_norm_rank.shape[1]
)

# Remove empty gene sets
MI_cancerSEA_meanrank_df = MI_cancerSEA_meanrank_df.loc[
    :,
    MI_cancerSEA_meanrank_df.notna().any(axis=0)
]

# Sort rows by existing mi_order
# Convert mi_order to MI- format, compatible with row names like MI-1, MI-2, ...
mi_order_use = [
    str(mi).replace("MI_", "MI-").replace("MI", "MI-")
    if not str(mi).startswith("MI-")
    else str(mi)
    for mi in mi_order
]

# Normalize repeated hyphens in MI labels.
mi_order_use = [
    mi.replace("MI--", "MI-")
    for mi in mi_order_use
]

mi_order_use = [
    mi for mi in mi_order_use
    if mi in MI_cancerSEA_meanrank_df.index
]

MI_cancerSEA_meanrank_df = MI_cancerSEA_meanrank_df.loc[mi_order_use, :]

# Optional: sort columns by average mean-rank across MIs
MI_cancerSEA_meanrank_df = MI_cancerSEA_meanrank_df.loc[
    :,
    MI_cancerSEA_meanrank_df.mean(axis=0).sort_values(ascending=False).index
]

MI_cancerSEA_meanrank_df.to_csv(
    run_dirs['run_dir'] / "MI_cancerSEA_meanrank_profile_receiver.csv"
)

print("Row max of MI_cancerSEA_meanrank_receiver_df:")
MI_cancerSEA_meanrank_receiver_df_max = MI_cancerSEA_meanrank_df.max(axis=1)
print(MI_cancerSEA_meanrank_receiver_df_max)

# ============================================================
# Step 5. Plot heatmap
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

sns.set_theme(style="white", context="paper")

plt.close("all")

fig_w = max(8, 0.45 * MI_cancerSEA_meanrank_df.shape[1])
fig_h = max(5, 0.35 * MI_cancerSEA_meanrank_df.shape[0])

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.heatmap(
    MI_cancerSEA_meanrank_df,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    ax=ax,
    linewidths=0.3,
    linecolor="white",
    cbar=True,
    cbar_kws={"shrink": 0.8, "label": "Normalized mean rank"},
    rasterized=True
)

ax.set_xlabel("CancerSEA gene set", fontsize=12)
ax.set_ylabel("Meta-interaction", fontsize=12)
ax.set_title("Receiver loading mean-rank profile across CancerSEA gene sets", fontsize=13)

ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right", fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()

plt.savefig(
    run_dirs['run_dir'] / "MI_cancerSEA_meanrank_profile_receiver_heatmap.pdf",
    bbox_inches="tight",
    transparent=True
)

plt.savefig(
    run_dirs['run_dir'] / "MI_cancerSEA_meanrank_profile_receiver_heatmap.png",
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

plt.show()
plt.close(fig)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Step 1. Load receiver/sender loadings and normalize by column sum
# ============================================================
loading_receiver_use = np.load(run_dirs['run_dir'] / "loading_receiver_use.npy")
loading_sender_use = np.load(run_dirs['run_dir'] / "loading_sender_use.npy")

loading_receiver_use_df = pd.DataFrame(
    loading_receiver_use,
    index=["MI-" + str(i + 1) for i in range(loading_receiver_use.shape[0])],
    columns=processed.adata_list[0].var_names
)

loading_sender_use_df = pd.DataFrame(
    loading_sender_use,
    index=["MI-" + str(i + 1) for i in range(loading_sender_use.shape[0])],
    columns=processed.adata_list[0].var_names
)

loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)

loading_receiver_use_df_norm = loading_receiver_use_df.div(
    loading_receiver_use_df_colsum.replace(0, np.nan), axis=1
)
loading_sender_use_df_norm = loading_sender_use_df.div(
    loading_sender_use_df_colsum.replace(0, np.nan), axis=1
)

loading_receiver_use_df_norm = loading_receiver_use_df_norm.fillna(0)
loading_sender_use_df_norm = loading_sender_use_df_norm.fillna(0)


# ============================================================
# Step 2. Rank loading_sender_use_df_norm by row
# Larger value -> larger rank
# ============================================================
loading_sender_use_df_norm_rank = loading_sender_use_df_norm.rank(
    axis=1,
    method="average",
    ascending=True
)

loading_sender_use_df_norm_rank.to_csv(
    run_dirs['run_dir'] / "loading_sender_use_df_norm_rank.csv"
)

# print("loading_sender_use_df_norm_rank shape:")
# print(loading_sender_use_df_norm_rank.shape)
# display(loading_sender_use_df_norm_rank.iloc[:5, :5])


# ============================================================
# Step 3. Load CancerSEA gene sets
# ============================================================
cancerSEA_path = str(DATA_ROOT / "CancerSEA_marker") + os.sep

cancerSEA_files = [f for f in os.listdir(cancerSEA_path) if f.endswith(".txt")]
cancerSEA_gene_sets = {}

for file in cancerSEA_files:
    gene_set_name = file.replace(".txt", "")
    gene_set_df = pd.read_csv(os.path.join(cancerSEA_path, file), sep="\t", header=0)

    gene_set_cur = gene_set_df["GeneName"].astype(str).tolist()

    gene_set_cur = list(
        set(gene_set_cur)
        & set(loading_receiver_use_df_norm.columns)
        & set(loading_sender_use_df_norm.columns)
    )

    cancerSEA_gene_sets[gene_set_name] = sorted(gene_set_cur)

print("Number of genes in each CancerSEA gene set:")
for gene_set_name, gene_set_cur in cancerSEA_gene_sets.items():
    print(f"{gene_set_name}: {len(gene_set_cur)} genes")


# ============================================================
# Step 4. Compute MI x CancerSEA gene-set mean-rank profile
#         based on sender loading
# ============================================================
mean_rank_profile_dict = {}

for gene_set_name, gene_set_genes in cancerSEA_gene_sets.items():
    if len(gene_set_genes) == 0:
        mean_rank_profile_dict[gene_set_name] = pd.Series(
            np.nan,
            index=loading_sender_use_df_norm_rank.index
        )
    else:
        mean_rank_profile_dict[gene_set_name] = loading_sender_use_df_norm_rank[
            gene_set_genes
        ].mean(axis=1)

MI_cancerSEA_meanrank_sender_df = pd.DataFrame(mean_rank_profile_dict)

# Normalize rank to 0-1 scale
MI_cancerSEA_meanrank_sender_df = (
    MI_cancerSEA_meanrank_sender_df / loading_sender_use_df_norm_rank.shape[1]
)

# Remove empty gene sets
MI_cancerSEA_meanrank_sender_df = MI_cancerSEA_meanrank_sender_df.loc[
    :,
    MI_cancerSEA_meanrank_sender_df.notna().any(axis=0)
]

# Sort rows by existing mi_order
# Convert mi_order to MI- format, compatible with row names like MI-1, MI-2, ...
mi_order_use = [
    str(mi).replace("MI_", "MI-").replace("MI", "MI-")
    if not str(mi).startswith("MI-")
    else str(mi)
    for mi in mi_order
]

# Normalize repeated hyphens in MI labels.
mi_order_use = [
    mi.replace("MI--", "MI-")
    for mi in mi_order_use
]

mi_order_use = [
    mi for mi in mi_order_use
    if mi in MI_cancerSEA_meanrank_sender_df.index
]

MI_cancerSEA_meanrank_sender_df = MI_cancerSEA_meanrank_sender_df.loc[mi_order_use, :]

# Optional: sort columns by average mean-rank across MIs
MI_cancerSEA_meanrank_sender_df = MI_cancerSEA_meanrank_sender_df.loc[
    :,
    MI_cancerSEA_meanrank_sender_df.mean(axis=0).sort_values(ascending=False).index
]

MI_cancerSEA_meanrank_sender_df.to_csv(
    run_dirs['run_dir'] / "MI_cancerSEA_meanrank_profile_sender.csv"
)

# print("MI x CancerSEA sender mean-rank profile shape:")
# print(MI_cancerSEA_meanrank_sender_df.shape)
# display(MI_cancerSEA_meanrank_sender_df.iloc[:5, :5])
##Print the row max of MI_cancerSEA_meanrank_sender_df
print("Row max of MI_cancerSEA_meanrank_sender_df:")
MI_cancerSEA_meanrank_sender_df_max = MI_cancerSEA_meanrank_sender_df.max(axis=1)
print(MI_cancerSEA_meanrank_sender_df_max)

# ============================================================
# Step 5. Plot sender heatmap
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

sns.set_theme(style="white", context="paper")

plt.close("all")

fig_w = max(8, 0.45 * MI_cancerSEA_meanrank_sender_df.shape[1])
fig_h = max(5, 0.35 * MI_cancerSEA_meanrank_sender_df.shape[0])

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.heatmap(
    MI_cancerSEA_meanrank_sender_df,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    ax=ax,
    linewidths=0.3,
    linecolor="white",
    cbar=True,
    cbar_kws={"shrink": 0.8, "label": "Normalized mean rank"},
    rasterized=True
)

ax.set_xlabel("CancerSEA gene set", fontsize=12)
ax.set_ylabel("Meta-interaction", fontsize=12)
ax.set_title("Sender loading mean-rank profile across CancerSEA gene sets", fontsize=13)

ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right", fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()

plt.savefig(
    run_dirs['run_dir'] / "MI_cancerSEA_meanrank_profile_sender_heatmap.pdf",
    bbox_inches="tight",
    transparent=True
)

plt.savefig(
    run_dirs['run_dir'] / "MI_cancerSEA_meanrank_profile_sender_heatmap.png",
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

plt.show()
plt.close(fig)

In [ ]:
##Get the max of the receiver and sender mean-rank profiles for each MI
MI_cancerSEA_meanrank_df_max_agg = pd.DataFrame({
    "receiver_mean_rank_max": MI_cancerSEA_meanrank_receiver_df_max,
    "sender_mean_rank_max": MI_cancerSEA_meanrank_sender_df_max
})
MI_cancerSEA_meanrank_df_max_agg["overall_mean_rank_max"] = MI_cancerSEA_meanrank_df_max_agg[[
    "receiver_mean_rank_max", "sender_mean_rank_max"
]].max(axis=1)


In [ ]:
MI_cancerSEA_meanrank_receiver_df = MI_cancerSEA_meanrank_df.copy()

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import LineCollection
import seaborn as sns

# Optional, required for managua colormap if not already registered
try:
    import cmcrameri.cm as cmc
except ImportError:
    cmc = None


# ============================================================
# Step 1. Make sure receiver matrix has a clear name
# ============================================================
if "MI_cancerSEA_meanrank_receiver_df" not in globals():
    MI_cancerSEA_meanrank_receiver_df = MI_cancerSEA_meanrank_df.copy()


# ============================================================
# Step 2. Align sender and receiver mean-rank matrices
# ============================================================
def standardize_mi_name(mi):
    mi = str(mi)
    mi = mi.replace("MI_", "MI-")
    if mi.startswith("MI-"):
        return mi
    if mi.startswith("MI"):
        return mi.replace("MI", "MI-", 1)
    return mi


def get_mi_order_use(mi_order, available_mis):
    if "mi_order" in globals():
        order = [standardize_mi_name(mi) for mi in mi_order]
        order = [mi for mi in order if mi in available_mis]
        remaining = [mi for mi in available_mis if mi not in order]
        return order + remaining
    return list(available_mis)


common_mis = [
    mi for mi in MI_cancerSEA_meanrank_sender_df.index
    if mi in MI_cancerSEA_meanrank_receiver_df.index
]

common_gene_sets = [
    gs for gs in MI_cancerSEA_meanrank_sender_df.columns
    if gs in MI_cancerSEA_meanrank_receiver_df.columns
]

mi_order_use = get_mi_order_use(mi_order, common_mis)


# ============================================================
# Step 2.5. Sort columns by peak-row index in max(sender, receiver)
# ------------------------------------------------------------
# For each MI x gene-set cell:
#     triangle_max = max(sender value, receiver value)
#
# For each gene-set column:
#     1) find the MI row where triangle_max is largest
#     2) get that MI's row index in mi_order_use
#     3) sort columns by this row index from small to large
#
# Tie-breaks:
#     - within the same peak row, sort by peak value from high to low
#     - then by column sum from high to low
#     - then by gene-set name alphabetically for stability
# ============================================================

sender_for_order = MI_cancerSEA_meanrank_sender_df.loc[
    mi_order_use, common_gene_sets
].copy()

receiver_for_order = MI_cancerSEA_meanrank_receiver_df.loc[
    mi_order_use, common_gene_sets
].copy()

# Element-wise max of upper triangle value and lower triangle value.
# np.fmax ignores one-sided NaN when the other side is finite.
triangle_max_for_order = pd.DataFrame(
    np.fmax(
        sender_for_order.to_numpy(dtype=float),
        receiver_for_order.to_numpy(dtype=float),
    ),
    index=sender_for_order.index,
    columns=sender_for_order.columns,
)

mi_to_row_index = {
    mi: i
    for i, mi in enumerate(mi_order_use)
}

column_order_records = []

for gs in triangle_max_for_order.columns:
    col_vals = triangle_max_for_order[gs]

    finite_col_vals = col_vals.replace([np.inf, -np.inf], np.nan)

    if finite_col_vals.notna().sum() == 0:
        peak_mi = None
        peak_row_index = 10**9
        peak_value = np.nan
        column_sum = 0.0
    else:
        peak_mi = finite_col_vals.idxmax()
        peak_row_index = mi_to_row_index.get(peak_mi, 10**9)
        peak_value = float(finite_col_vals.loc[peak_mi])
        column_sum = float(finite_col_vals.sum(skipna=True))

    column_order_records.append({
        "gene_set": gs,
        "peak_mi": peak_mi,
        "peak_row_index": peak_row_index,
        "peak_value": peak_value,
        "triangle_max_column_sum": column_sum,
    })

column_order_df = pd.DataFrame(column_order_records).set_index("gene_set")

column_order_df = column_order_df.sort_values(
    by=[
        "peak_row_index",
        "peak_value",
        "triangle_max_column_sum",
        "gene_set",
    ],
    ascending=[
        True,
        False,
        False,
        True,
    ],
)

gene_set_order_for_plot = column_order_df.index.tolist()

column_order_df.to_csv(
    run_dirs["run_dir"] / "CancerSEA_gene_set_column_order_by_triangle_max_peak_row.csv"
)

triangle_max_for_order.loc[
    mi_order_use,
    gene_set_order_for_plot
].to_csv(
    run_dirs["run_dir"] / "CancerSEA_triangle_max_sender_receiver_for_column_order.csv"
)

sender_plot_df = MI_cancerSEA_meanrank_sender_df.loc[
    mi_order_use, gene_set_order_for_plot
].copy()

receiver_plot_df = MI_cancerSEA_meanrank_receiver_df.loc[
    mi_order_use, gene_set_order_for_plot
].copy()

sender_plot_df.to_csv(
    run_dirs["run_dir"] / "MI_cancerSEA_meanrank_sender_reordered_for_split_heatmap.csv"
)

receiver_plot_df.to_csv(
    run_dirs["run_dir"] / "MI_cancerSEA_meanrank_receiver_reordered_for_split_heatmap.csv"
)


# ============================================================
# Step 3. Plot split-triangle mean-rank heatmap
#   upper triangle = sender
#   lower triangle = receiver
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

sns.set_theme(style="white", context="paper")


def resolve_cmap(cmap_name):
    """
    Resolve colormap name.

    For cmcrameri colormaps, matplotlib may register them as either:
        "managua" or "cmc.managua"
    depending on the package/version.
    """
    if cmap_name in plt.colormaps():
        return mpl.colormaps.get_cmap(cmap_name)

    cmc_name = f"cmc.{cmap_name}"
    if cmc_name in plt.colormaps():
        return mpl.colormaps.get_cmap(cmc_name)

    if cmc is not None and hasattr(cmc, cmap_name):
        return getattr(cmc, cmap_name)

    raise ValueError(
        f"Colormap '{cmap_name}' is not available. "
        "Please install cmcrameri first: pip install cmcrameri"
    )


def plot_split_triangle_heatmap(
    sender_df,
    receiver_df,
    out_prefix,
    title,
    cmap="PiYG_r",
    vmin=0,
    vmax=1,
    annotate=False,
    annot_fmt=".2f",
    highlight_threshold=0.7,
    highlight_edgecolor="#FFD700",
    highlight_linewidth=1.8,
    grid_color="#D9D9D9",
    grid_linewidth=0.55,
    rasterize_grid=True,
):
    """
    Plot split-triangle heatmap:
        upper triangle = sender
        lower triangle = receiver

    Key plotting layers:
        zorder=1  : colored sender/receiver triangles
        zorder=5  : light-gray cell borders, rasterized in PDF/SVG
        zorder=20 : highlight borders, vector and on top
    """
    sender_df = sender_df.copy()
    receiver_df = receiver_df.loc[sender_df.index, sender_df.columns].copy()

    n_rows, n_cols = sender_df.shape

    fig_w = max(8, 0.55 * n_cols)
    fig_h = max(5, 0.38 * n_rows)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cmap_obj = resolve_cmap(cmap)

    highlight_patches = []

    for i, mi in enumerate(sender_df.index):
        for j, gs in enumerate(sender_df.columns):
            s_val = sender_df.loc[mi, gs]
            r_val = receiver_df.loc[mi, gs]

            x0, x1 = j, j + 1
            y0, y1 = i, i + 1

            sender_coords = [[x0, y0], [x1, y0], [x1, y1]]
            receiver_coords = [[x0, y0], [x0, y1], [x1, y1]]

            # ----------------------------------------------------
            # Upper triangle: sender
            # ----------------------------------------------------
            if pd.notna(s_val):
                s_val_float = float(s_val)

                tri_sender = patches.Polygon(
                    sender_coords,
                    closed=True,
                    facecolor=cmap_obj(norm(s_val_float)),
                    edgecolor="none",
                    linewidth=0,
                    zorder=1
                )
                ax.add_patch(tri_sender)

                if s_val_float > highlight_threshold:
                    tri_sender_highlight = patches.Polygon(
                        sender_coords,
                        closed=True,
                        facecolor="none",
                        edgecolor=highlight_edgecolor,
                        linewidth=highlight_linewidth,
                        joinstyle="miter",
                        zorder=20
                    )
                    highlight_patches.append(tri_sender_highlight)

            # ----------------------------------------------------
            # Lower triangle: receiver
            # ----------------------------------------------------
            if pd.notna(r_val):
                r_val_float = float(r_val)

                tri_receiver = patches.Polygon(
                    receiver_coords,
                    closed=True,
                    facecolor=cmap_obj(norm(r_val_float)),
                    edgecolor="none",
                    linewidth=0,
                    zorder=1
                )
                ax.add_patch(tri_receiver)

                if r_val_float > highlight_threshold:
                    tri_receiver_highlight = patches.Polygon(
                        receiver_coords,
                        closed=True,
                        facecolor="none",
                        edgecolor=highlight_edgecolor,
                        linewidth=highlight_linewidth,
                        joinstyle="miter",
                        zorder=20
                    )
                    highlight_patches.append(tri_receiver_highlight)

            # ----------------------------------------------------
            # Optional annotation
            # ----------------------------------------------------
            if annotate:
                if pd.notna(s_val):
                    ax.text(
                        j + 0.72, i + 0.28,
                        format(float(s_val), annot_fmt),
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color="black",
                        zorder=25
                    )

                if pd.notna(r_val):
                    ax.text(
                        j + 0.28, i + 0.72,
                        format(float(r_val), annot_fmt),
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color="black",
                        zorder=25
                    )

    # ------------------------------------------------------------
    # Light-gray cell borders
    # Rasterized so these dense grid borders are not saved as many
    # individual vector objects in PDF/SVG.
    # ------------------------------------------------------------
    grid_segments = []

    for x in range(n_cols + 1):
        grid_segments.append([(x, 0), (x, n_rows)])

    for y in range(n_rows + 1):
        grid_segments.append([(0, y), (n_cols, y)])

    grid_collection = LineCollection(
        grid_segments,
        colors=grid_color,
        linewidths=grid_linewidth,
        zorder=5,
        rasterized=rasterize_grid
    )
    ax.add_collection(grid_collection)

    # ------------------------------------------------------------
    # Add highlight borders as the top independent layer
    # ------------------------------------------------------------
    for hp in highlight_patches:
        ax.add_patch(hp)

    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.invert_yaxis()

    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_yticks(np.arange(n_rows) + 0.5)

    ax.set_xticklabels(
        sender_df.columns,
        rotation=60,
        ha="right",
        fontsize=10
    )

    ax.set_yticklabels(
        sender_df.index,
        rotation=0,
        fontsize=10
    )

    ax.set_xlabel("CancerSEA gene set", fontsize=12)
    ax.set_ylabel("Meta-interaction", fontsize=12)
    ax.set_title(title, fontsize=13)

    ax.tick_params(length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    sm = mpl.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        fraction=0.03,
        pad=0.02,
        shrink=0.85
    )

    cbar.set_label(
        "Normalized mean rank",
        fontsize=11
    )
    cbar.ax.tick_params(labelsize=9)

    fig.text(
        0.99,
        0.98,
        (
            "Upper triangle: sender\n"
            "Lower triangle: receiver\n"
            f"Border: value > {highlight_threshold}"
        ),
        ha="right",
        va="top",
        fontsize=10
    )

    plt.tight_layout()

    fig.savefig(
        run_dirs["run_dir"] / f"{out_prefix}.pdf",
        bbox_inches="tight",
        transparent=True
    )

    fig.savefig(
        run_dirs["run_dir"] / f"{out_prefix}.png",
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

    plt.show()
    plt.close(fig)


plot_split_triangle_heatmap(
    sender_df=sender_plot_df,
    receiver_df=receiver_plot_df,
    out_prefix="MI_cancerSEA_meanrank_sender_receiver_split_triangle_heatmap_peakrow_PiYG_lightgray_grid",
    title="CancerSEA mean-rank profile\nUpper triangle = sender; lower triangle = receiver",
    cmap="PiYG_r",
    vmin=0,
    vmax=1,
    annotate=False,
    annot_fmt=".2f",
    highlight_threshold=0.6,
    highlight_edgecolor="#A6A6A6",
    highlight_linewidth=3,
    grid_color="#D9D9D9",
    grid_linewidth=0.55,
    rasterize_grid=True,
)

### Additional EcoTyper cell-state profiles

Read the EcoTyper LR/CE and cell-state marker workbooks, build a deduplicated
gene-to-ecotype table, and summarize regulator/target loadings by ecotype.
These supplementary molecular-profile views are separate from the CancerSEA
programs displayed in Fig. 6b.


In [ ]:
# ============================================================
# Build CE-level Ecotyper gene table from:
#   1) LR_CE.xlsx
#   2) Marker_Cellstate.xlsx
#
# Input files:
#   DATA_ROOT / Ecotyper / LR_CE.xlsx
#   DATA_ROOT / Ecotyper / Marker_Cellstate.xlsx
#
# Output dataframe:
#   Ecotyper_gene_CE_df
#
# Columns:
#   Gene
#   CE
#   Type
#
# Important:
#   Final table is deduplicated by CE + Gene.
#   If the same Gene appears in the same CE as Ligand/Receptor/Marker,
#   Type is collapsed as "Ligand;Receptor;Marker".
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd


# ============================================================
# Parameters
# ============================================================

ecotyper_dir = DATA_ROOT / "Ecotyper"

lr_ce_path = ecotyper_dir / "LR_CE.xlsx"
marker_cellstate_path = ecotyper_dir / "Marker_Cellstate.xlsx"

raw_out_path = ecotyper_dir / "Ecotyper_gene_CE_table_raw_with_type_duplicates.csv"
out_path = ecotyper_dir / "Ecotyper_gene_CE_table_deduplicated_by_CE_Gene.csv"


# ============================================================
# Helper functions
# ============================================================

def _clean_text(x):
    """Clean text values from Excel cells."""
    if pd.isna(x):
        return np.nan

    x = str(x)
    x = x.replace("\n", " ")
    x = x.replace("\r", " ")
    x = re.sub(r"\s+", " ", x)
    x = x.strip()

    if x == "":
        return np.nan

    return x


def _normalize_colname(x):
    """Normalize column names for robust matching."""
    if pd.isna(x):
        return ""

    x = str(x)
    x = x.replace("\n", " ")
    x = x.replace("\r", " ")
    x = re.sub(r"\s+", " ", x)
    x = x.strip()

    return x


def _clean_gene_symbol(x):
    """Clean one gene symbol."""
    x = _clean_text(x)

    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.strip(",")
    x = x.strip(";")
    x = x.strip()

    if x == "":
        return np.nan

    return x


def _split_marker_genes(x):
    """
    Split marker gene list.

    Expected format:
        "MS4A1, TCL1A"

    Also tolerates semicolon, slash, or Chinese comma.
    """
    x = _clean_text(x)

    if pd.isna(x):
        return []

    x = str(x).replace("，", ",")
    parts = re.split(r"[,;/]", x)

    genes = []
    for p in parts:
        g = _clean_gene_symbol(p)
        if pd.notna(g):
            genes.append(g)

    return genes


def _find_col(df, candidates, required=True):
    """
    Find a column by normalized name.
    """
    norm_map = {
        _normalize_colname(c).lower(): c
        for c in df.columns
    }

    for cand in candidates:
        cand_norm = _normalize_colname(cand).lower()
        if cand_norm in norm_map:
            return norm_map[cand_norm]

    if required:
        raise KeyError(
            "Cannot find required column. Tried candidates:\n"
            f"{candidates}\n\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return None


def _standardize_yes_no(x):
    """Convert yes/no-like strings to normalized Yes/No."""
    x = _clean_text(x)

    if pd.isna(x):
        return np.nan

    x_low = str(x).strip().lower()

    if x_low in {"yes", "y", "true", "1"}:
        return "Yes"

    if x_low in {"no", "n", "false", "0"}:
        return "No"

    return str(x).strip()


def _standardize_ce(x):
    """Clean CE labels."""
    x = _clean_text(x)

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x.upper() in {"NA", "N/A", "NONE", "NAN"}:
        return np.nan

    return x


def _collapse_type(types):
    """
    Collapse duplicated Type values within the same CE + Gene.
    """
    type_order = {
        "Ligand": 0,
        "Receptor": 1,
        "Marker": 2,
    }

    clean_types = []

    for t in types:
        t = _clean_text(t)
        if pd.notna(t):
            clean_types.append(t)

    clean_types = sorted(
        set(clean_types),
        key=lambda x: type_order.get(x, 999)
    )

    return ";".join(clean_types)


# ============================================================
# LR_CE.xlsx reader
# ============================================================

def _flatten_lr_multiindex_columns(lr_raw):
    """
    Flatten LR_CE.xlsx columns.

    The expected layout is:
        Ligand block:
            Gene / Cell type / Cell state / CE
        Receptor block:
            Gene / Cell type / Cell state / CE
        Same ecotype?
        Significantly Enriched?a

    Because merged headers can be read inconsistently by pandas,
    this function uses both header names and column positions.
    """
    flat_cols = []

    for idx, col in enumerate(lr_raw.columns):
        if isinstance(col, tuple):
            top, bottom = col
            top_clean = _normalize_colname(top)
            bottom_clean = _normalize_colname(bottom)
        else:
            top_clean = ""
            bottom_clean = _normalize_colname(col)

        top_low = top_clean.lower()
        bottom_low = bottom_clean.lower()

        if top_low.startswith("unnamed"):
            top_clean = ""

        if bottom_low.startswith("unnamed"):
            bottom_clean = ""

        # Positional fallback based on the known file layout.
        if idx in [0, 1, 2, 3]:
            prefix = "Ligand"
            if bottom_clean == "":
                bottom_clean = ["Gene", "Cell type", "Cell state", "CE"][idx]
            flat_cols.append(f"{prefix}_{bottom_clean}")

        elif idx in [4, 5, 6, 7]:
            prefix = "Receptor"
            if bottom_clean == "":
                bottom_clean = ["Gene", "Cell type", "Cell state", "CE"][idx - 4]
            flat_cols.append(f"{prefix}_{bottom_clean}")

        else:
            # Same ecotype? and Significantly Enriched?a
            if bottom_clean != "":
                flat_cols.append(bottom_clean)
            elif top_clean != "":
                flat_cols.append(top_clean)
            else:
                flat_cols.append(f"Column_{idx}")

    flat_cols = [_normalize_colname(c) for c in flat_cols]

    return flat_cols


def _read_lr_ce_table(lr_ce_path):
    """
    Read LR_CE.xlsx and return clean LR table with columns:
        Ligand_Gene
        Ligand_CE
        Receptor_Gene
        Receptor_CE
        Same_ecotype
        Significantly_enriched
    """
    if not lr_ce_path.exists():
        raise FileNotFoundError(f"Cannot find LR_CE file:\n{lr_ce_path}")

    lr_raw = pd.read_excel(lr_ce_path, header=[0, 1], engine="openpyxl")
    lr_raw = lr_raw.dropna(how="all").copy()

    lr_raw.columns = _flatten_lr_multiindex_columns(lr_raw)

    ligand_gene_col = _find_col(
        lr_raw,
        ["Ligand_Gene", "Ligand Gene"],
        required=True,
    )

    ligand_ce_col = _find_col(
        lr_raw,
        ["Ligand_CE", "Ligand CE"],
        required=True,
    )

    receptor_gene_col = _find_col(
        lr_raw,
        ["Receptor_Gene", "Receptor Gene"],
        required=True,
    )

    receptor_ce_col = _find_col(
        lr_raw,
        ["Receptor_CE", "Receptor CE"],
        required=True,
    )

    same_ecotype_col = _find_col(
        lr_raw,
        ["Same ecotype?", "Same ecotype"],
        required=True,
    )

    enriched_col = _find_col(
        lr_raw,
        [
            "Significantly Enriched?a",
            "Significantly Enriched?",
            "Significantly Enriched",
            "Significantly enriched?",
            "Significantly enriched",
        ],
        required=True,
    )

    lr_df = lr_raw.rename(
        columns={
            ligand_gene_col: "Ligand_Gene",
            ligand_ce_col: "Ligand_CE",
            receptor_gene_col: "Receptor_Gene",
            receptor_ce_col: "Receptor_CE",
            same_ecotype_col: "Same_ecotype",
            enriched_col: "Significantly_enriched",
        }
    ).copy()

    keep_cols = [
        "Ligand_Gene",
        "Ligand_CE",
        "Receptor_Gene",
        "Receptor_CE",
        "Same_ecotype",
        "Significantly_enriched",
    ]

    lr_df = lr_df[keep_cols].copy()

    for col in keep_cols:
        lr_df[col] = lr_df[col].map(_clean_text)

    lr_df["Ligand_Gene"] = lr_df["Ligand_Gene"].map(_clean_gene_symbol)
    lr_df["Receptor_Gene"] = lr_df["Receptor_Gene"].map(_clean_gene_symbol)

    lr_df["Ligand_CE"] = lr_df["Ligand_CE"].map(_standardize_ce)
    lr_df["Receptor_CE"] = lr_df["Receptor_CE"].map(_standardize_ce)

    lr_df["Same_ecotype"] = lr_df["Same_ecotype"].map(_standardize_yes_no)
    lr_df["Significantly_enriched"] = lr_df["Significantly_enriched"].map(_standardize_yes_no)

    return lr_df


def _build_lr_gene_ce_df(lr_df):
    """
    From LR table, filter:
        Same_ecotype == Yes
        Significantly_enriched == Yes

    Then build long table:
        Gene, CE, Type
    """
    lr_keep = lr_df.loc[
        (lr_df["Same_ecotype"] == "Yes")
        & (lr_df["Significantly_enriched"] == "Yes")
    ].copy()

    ligand_df = (
        lr_keep[["Ligand_Gene", "Ligand_CE"]]
        .rename(columns={"Ligand_Gene": "Gene", "Ligand_CE": "CE"})
        .copy()
    )
    ligand_df["Type"] = "Ligand"

    receptor_df = (
        lr_keep[["Receptor_Gene", "Receptor_CE"]]
        .rename(columns={"Receptor_Gene": "Gene", "Receptor_CE": "CE"})
        .copy()
    )
    receptor_df["Type"] = "Receptor"

    lr_gene_ce_df = pd.concat(
        [ligand_df, receptor_df],
        axis=0,
        ignore_index=True,
    )

    lr_gene_ce_df["Gene"] = lr_gene_ce_df["Gene"].map(_clean_gene_symbol)
    lr_gene_ce_df["CE"] = lr_gene_ce_df["CE"].map(_standardize_ce)
    lr_gene_ce_df["Type"] = lr_gene_ce_df["Type"].map(_clean_text)

    lr_gene_ce_df = lr_gene_ce_df.dropna(subset=["Gene", "CE", "Type"]).copy()

    lr_gene_ce_df = (
        lr_gene_ce_df
        .drop_duplicates(["Gene", "CE", "Type"])
        .sort_values(["CE", "Type", "Gene"])
        .reset_index(drop=True)
    )

    return lr_gene_ce_df, lr_keep


# ============================================================
# Marker_Cellstate.xlsx reader
# ============================================================

def _read_marker_cellstate_table(marker_cellstate_path):
    """
    Read Marker_Cellstate.xlsx.

    Required columns:
        Carcinoma ecotype
        Key marker genes
    """
    if not marker_cellstate_path.exists():
        raise FileNotFoundError(f"Cannot find Marker_Cellstate file:\n{marker_cellstate_path}")

    marker_raw = pd.read_excel(marker_cellstate_path, header=0, engine="openpyxl")
    marker_raw = marker_raw.dropna(how="all").copy()

    marker_raw.columns = [_normalize_colname(c) for c in marker_raw.columns]

    ce_col = _find_col(
        marker_raw,
        [
            "Carcinoma ecotype",
            "Carcinoma ecotyper",
            "Carcinoma ecotype ",
            "Carcinoma ecotyper ",
        ],
        required=True,
    )

    marker_gene_col = _find_col(
        marker_raw,
        [
            "Key marker genes",
            "Marker genes",
            "Key markers",
            "Key marker gene",
        ],
        required=True,
    )

    marker_df = marker_raw.rename(
        columns={
            ce_col: "CE",
            marker_gene_col: "Key_marker_genes",
        }
    ).copy()

    marker_df["CE"] = marker_df["CE"].map(_standardize_ce)
    marker_df["Key_marker_genes"] = marker_df["Key_marker_genes"].map(_clean_text)

    return marker_df


def _build_marker_gene_ce_df(marker_df):
    """
    From Marker_Cellstate table, build:
        Gene, CE, Type
    """
    rows = []

    for _, row in marker_df.iterrows():
        ce = _standardize_ce(row.get("CE", np.nan))

        if pd.isna(ce):
            continue

        genes = _split_marker_genes(row.get("Key_marker_genes", np.nan))

        for gene in genes:
            rows.append({
                "Gene": gene,
                "CE": ce,
                "Type": "Marker",
            })

    marker_gene_ce_df = pd.DataFrame(rows, columns=["Gene", "CE", "Type"])

    if marker_gene_ce_df.shape[0] == 0:
        return pd.DataFrame(columns=["Gene", "CE", "Type"])

    marker_gene_ce_df["Gene"] = marker_gene_ce_df["Gene"].map(_clean_gene_symbol)
    marker_gene_ce_df["CE"] = marker_gene_ce_df["CE"].map(_standardize_ce)
    marker_gene_ce_df["Type"] = marker_gene_ce_df["Type"].map(_clean_text)

    marker_gene_ce_df = marker_gene_ce_df.dropna(subset=["Gene", "CE", "Type"]).copy()

    marker_gene_ce_df = (
        marker_gene_ce_df
        .drop_duplicates(["Gene", "CE", "Type"])
        .sort_values(["CE", "Type", "Gene"])
        .reset_index(drop=True)
    )

    return marker_gene_ce_df


# ============================================================
# Step 1. Read LR_CE.xlsx
# ============================================================

lr_df = _read_lr_ce_table(lr_ce_path)

lr_gene_ce_df, lr_keep_df = _build_lr_gene_ce_df(lr_df)

print("LR_CE total rows:", lr_df.shape[0])
print("LR_CE filtered rows: Same ecotype == Yes & Significantly enriched == Yes:", lr_keep_df.shape[0])
print("LR-derived Gene-CE-Type rows before CE-Gene collapse:", lr_gene_ce_df.shape[0])

display(lr_gene_ce_df.head(30))


# ============================================================
# Step 2. Read Marker_Cellstate.xlsx
# ============================================================

marker_df = _read_marker_cellstate_table(marker_cellstate_path)

marker_gene_ce_df = _build_marker_gene_ce_df(marker_df)

print("Marker_Cellstate total rows:", marker_df.shape[0])
print("Marker-derived Gene-CE-Type rows before CE-Gene collapse:", marker_gene_ce_df.shape[0])

display(marker_gene_ce_df.head(30))


# ============================================================
# Step 3. Combine LR genes and marker genes
# Keep raw Type-specific table first
# ============================================================

Ecotyper_gene_CE_df_raw = pd.concat(
    [
        lr_gene_ce_df,
        marker_gene_ce_df,
    ],
    axis=0,
    ignore_index=True,
)

Ecotyper_gene_CE_df_raw["Gene"] = Ecotyper_gene_CE_df_raw["Gene"].map(_clean_gene_symbol)
Ecotyper_gene_CE_df_raw["CE"] = Ecotyper_gene_CE_df_raw["CE"].map(_standardize_ce)
Ecotyper_gene_CE_df_raw["Type"] = Ecotyper_gene_CE_df_raw["Type"].map(_clean_text)

Ecotyper_gene_CE_df_raw = Ecotyper_gene_CE_df_raw.dropna(
    subset=["Gene", "CE", "Type"]
).copy()

Ecotyper_gene_CE_df_raw = (
    Ecotyper_gene_CE_df_raw
    .drop_duplicates(["Gene", "CE", "Type"])
    .sort_values(["CE", "Type", "Gene"])
    .reset_index(drop=True)
)


# ============================================================
# Step 4. Deduplicate by CE + Gene
# If one Gene appears in the same CE with multiple Types,
# collapse Type as "Ligand;Receptor;Marker"
# ============================================================

Ecotyper_gene_CE_df = (
    Ecotyper_gene_CE_df_raw
    .groupby(["CE", "Gene"], as_index=False)
    .agg({"Type": _collapse_type})
    .loc[:, ["Gene", "CE", "Type"]]
    .sort_values(["CE", "Gene"])
    .reset_index(drop=True)
)

print("Raw Ecotyper Gene-CE-Type rows:", Ecotyper_gene_CE_df_raw.shape[0])
print("Deduplicated Ecotyper CE-Gene rows:", Ecotyper_gene_CE_df.shape[0])
print(
    "Duplicated CE-Gene pairs after deduplication:",
    Ecotyper_gene_CE_df.duplicated(["CE", "Gene"]).sum()
)

display(Ecotyper_gene_CE_df.head(80))


# ============================================================
# Step 5. Save
# ============================================================

Ecotyper_gene_CE_df_raw.to_csv(raw_out_path, index=False)
Ecotyper_gene_CE_df.to_csv(out_path, index=False)

print(f"Saved raw Ecotyper Gene-CE-Type table to:\n{raw_out_path}")
print(f"Saved CE-Gene deduplicated Ecotyper table to:\n{out_path}")


# ============================================================
# Optional summary
# ============================================================

print("\nNumber of unique genes per CE:")
display(
    Ecotyper_gene_CE_df
    .groupby("CE")["Gene"]
    .nunique()
    .reset_index(name="n_unique_genes")
    .sort_values("CE")
)

print("\nNumber of genes per CE and collapsed Type:")
display(
    Ecotyper_gene_CE_df
    .groupby(["CE", "Type"])["Gene"]
    .nunique()
    .reset_index(name="n_unique_genes")
    .sort_values(["CE", "Type"])
)

In [ ]:
# ============================================================
# Cell 1. Load sender/receiver loadings and build Ecotyper CE gene sets
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import LineCollection
import seaborn as sns
from pathlib import Path

# Optional cmcrameri support
try:
    import cmcrameri.cm as cmc
except ImportError:
    cmc = None


# ============================================================
# Step 1. Load receiver/sender loadings
# ============================================================

loading_receiver_use = np.load(run_dirs["run_dir"] / "loading_receiver_use.npy")
loading_sender_use = np.load(run_dirs["run_dir"] / "loading_sender_use.npy")

gene_names_use = pd.Index(processed.adata_list[0].var_names.astype(str))

loading_receiver_use_df = pd.DataFrame(
    loading_receiver_use,
    index=["MI-" + str(i + 1) for i in range(loading_receiver_use.shape[0])],
    columns=gene_names_use,
)

loading_sender_use_df = pd.DataFrame(
    loading_sender_use,
    index=["MI-" + str(i + 1) for i in range(loading_sender_use.shape[0])],
    columns=gene_names_use,
)


# ============================================================
# Step 2. Normalize each gene column by absolute column sum
# ============================================================

loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)

loading_receiver_use_df_norm = loading_receiver_use_df.div(
    loading_receiver_use_df_colsum.replace(0, np.nan),
    axis=1,
).fillna(0)

loading_sender_use_df_norm = loading_sender_use_df.div(
    loading_sender_use_df_colsum.replace(0, np.nan),
    axis=1,
).fillna(0)


# ============================================================
# Step 3. Rank normalized loading values within each MI row
# Larger loading value -> larger rank
# ============================================================

loading_receiver_use_df_norm_rank = loading_receiver_use_df_norm.rank(
    axis=1,
    method="average",
    ascending=True,
)

loading_sender_use_df_norm_rank = loading_sender_use_df_norm.rank(
    axis=1,
    method="average",
    ascending=True,
)

loading_receiver_use_df_norm_rank.to_csv(
    run_dirs["run_dir"] / "Ecotyper_loading_receiver_use_df_norm_rank.csv"
)

loading_sender_use_df_norm_rank.to_csv(
    run_dirs["run_dir"] / "Ecotyper_loading_sender_use_df_norm_rank.csv"
)


# ============================================================
# Step 4. Build Ecotyper CE gene sets from Ecotyper_gene_CE_df
# ============================================================

def _clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x != "" else np.nan


def _clean_gene_symbol(x):
    x = _clean_text(x)
    if pd.isna(x):
        return np.nan
    x = str(x).strip().strip(",").strip(";").strip()
    return x if x != "" else np.nan


def _standardize_mi_name(mi):
    mi = str(mi)
    mi = mi.replace("MI_", "MI-")
    if mi.startswith("MI-"):
        return mi
    if mi.startswith("MI"):
        return mi.replace("MI", "MI-", 1).replace("MI--", "MI-")
    return mi


def _get_mi_order_use(available_mis):
    if "mi_order" in globals():
        order = [_standardize_mi_name(mi) for mi in mi_order]
        order = [mi.replace("MI--", "MI-") for mi in order]
        order = [mi for mi in order if mi in available_mis]
        remaining = [mi for mi in available_mis if mi not in order]
        return order + remaining
    return list(available_mis)


# If Ecotyper_gene_CE_df is not in memory, optionally load the saved version.
if "Ecotyper_gene_CE_df" not in globals():
    ecotyper_gene_ce_path = DATA_ROOT / "Ecotyper" / "Ecotyper_gene_CE_table_deduplicated_by_CE_Gene.csv"
    if not ecotyper_gene_ce_path.exists():
        raise NameError(
            "`Ecotyper_gene_CE_df` is not in memory, and saved CSV was not found:\n"
            f"{ecotyper_gene_ce_path}"
        )
    Ecotyper_gene_CE_df = pd.read_csv(ecotyper_gene_ce_path)

required_cols = {"Gene", "CE", "Type"}
missing_cols = required_cols - set(Ecotyper_gene_CE_df.columns)
if len(missing_cols) > 0:
    raise ValueError(f"Ecotyper_gene_CE_df is missing required columns: {missing_cols}")

Ecotyper_gene_CE_df_use = Ecotyper_gene_CE_df.copy()
Ecotyper_gene_CE_df_use["Gene"] = Ecotyper_gene_CE_df_use["Gene"].map(_clean_gene_symbol)
Ecotyper_gene_CE_df_use["CE"] = Ecotyper_gene_CE_df_use["CE"].map(_clean_text)
Ecotyper_gene_CE_df_use["Type"] = Ecotyper_gene_CE_df_use["Type"].map(_clean_text)

Ecotyper_gene_CE_df_use = Ecotyper_gene_CE_df_use.dropna(
    subset=["Gene", "CE"]
).copy()

# Deduplicate once more by CE + Gene for safety.
Ecotyper_gene_CE_df_use = (
    Ecotyper_gene_CE_df_use
    .drop_duplicates(["CE", "Gene"])
    .reset_index(drop=True)
)

available_genes = (
    set(loading_receiver_use_df_norm.columns)
    & set(loading_sender_use_df_norm.columns)
)

ecotyper_CE_gene_sets = {}

for ce, sub_df in Ecotyper_gene_CE_df_use.groupby("CE"):
    genes_cur = (
        sub_df["Gene"]
        .dropna()
        .astype(str)
        .map(_clean_gene_symbol)
        .dropna()
        .unique()
        .tolist()
    )

    genes_cur = sorted(set(genes_cur) & available_genes)
    ecotyper_CE_gene_sets[ce] = genes_cur

# Remove empty CE gene sets
ecotyper_CE_gene_sets = {
    ce: genes
    for ce, genes in ecotyper_CE_gene_sets.items()
    if len(genes) > 0
}

print("Number of genes in each Ecotyper CE gene set after intersection:")
for ce, genes in ecotyper_CE_gene_sets.items():
    print(f"{ce}: {len(genes)} genes")

pd.DataFrame({
    "CE": list(ecotyper_CE_gene_sets.keys()),
    "n_genes": [len(v) for v in ecotyper_CE_gene_sets.values()],
}).to_csv(
    run_dirs["run_dir"] / "Ecotyper_CE_gene_set_sizes_after_intersection.csv",
    index=False,
)

In [ ]:
# ============================================================
# Cell 2. Compute MI x Ecotyper-CE mean-rank profiles
# ============================================================

# ============================================================
# Step 1. Compute receiver mean-rank profile
# ============================================================

receiver_mean_rank_profile_dict = {}

for ce, genes in ecotyper_CE_gene_sets.items():
    if len(genes) == 0:
        receiver_mean_rank_profile_dict[ce] = pd.Series(
            np.nan,
            index=loading_receiver_use_df_norm_rank.index,
        )
    else:
        receiver_mean_rank_profile_dict[ce] = (
            loading_receiver_use_df_norm_rank[genes].mean(axis=1)
        )

MI_ecotyper_CE_meanrank_receiver_df = pd.DataFrame(receiver_mean_rank_profile_dict)

# Normalize rank to 0-1 scale
MI_ecotyper_CE_meanrank_receiver_df = (
    MI_ecotyper_CE_meanrank_receiver_df
    / loading_receiver_use_df_norm_rank.shape[1]
)

# Remove empty CE columns
MI_ecotyper_CE_meanrank_receiver_df = MI_ecotyper_CE_meanrank_receiver_df.loc[
    :,
    MI_ecotyper_CE_meanrank_receiver_df.notna().any(axis=0),
]


# ============================================================
# Step 2. Compute sender mean-rank profile
# ============================================================

sender_mean_rank_profile_dict = {}

for ce, genes in ecotyper_CE_gene_sets.items():
    if len(genes) == 0:
        sender_mean_rank_profile_dict[ce] = pd.Series(
            np.nan,
            index=loading_sender_use_df_norm_rank.index,
        )
    else:
        sender_mean_rank_profile_dict[ce] = (
            loading_sender_use_df_norm_rank[genes].mean(axis=1)
        )

MI_ecotyper_CE_meanrank_sender_df = pd.DataFrame(sender_mean_rank_profile_dict)

# Normalize rank to 0-1 scale
MI_ecotyper_CE_meanrank_sender_df = (
    MI_ecotyper_CE_meanrank_sender_df
    / loading_sender_use_df_norm_rank.shape[1]
)

# Remove empty CE columns
MI_ecotyper_CE_meanrank_sender_df = MI_ecotyper_CE_meanrank_sender_df.loc[
    :,
    MI_ecotyper_CE_meanrank_sender_df.notna().any(axis=0),
]


# ============================================================
# Step 3. Align rows and columns
# ============================================================

common_mis = [
    mi for mi in MI_ecotyper_CE_meanrank_sender_df.index
    if mi in MI_ecotyper_CE_meanrank_receiver_df.index
]

common_CEs = [
    ce for ce in MI_ecotyper_CE_meanrank_sender_df.columns
    if ce in MI_ecotyper_CE_meanrank_receiver_df.columns
]

mi_order_use = _get_mi_order_use(common_mis)

MI_ecotyper_CE_meanrank_sender_df = MI_ecotyper_CE_meanrank_sender_df.loc[
    mi_order_use,
    common_CEs,
].copy()

MI_ecotyper_CE_meanrank_receiver_df = MI_ecotyper_CE_meanrank_receiver_df.loc[
    mi_order_use,
    common_CEs,
].copy()


# ============================================================
# Step 4. Summary max values
# ============================================================

MI_ecotyper_CE_meanrank_receiver_df_max = MI_ecotyper_CE_meanrank_receiver_df.max(axis=1)
MI_ecotyper_CE_meanrank_sender_df_max = MI_ecotyper_CE_meanrank_sender_df.max(axis=1)

MI_ecotyper_CE_meanrank_df_max_agg = pd.DataFrame({
    "receiver_mean_rank_max": MI_ecotyper_CE_meanrank_receiver_df_max,
    "sender_mean_rank_max": MI_ecotyper_CE_meanrank_sender_df_max,
})

MI_ecotyper_CE_meanrank_df_max_agg["overall_mean_rank_max"] = (
    MI_ecotyper_CE_meanrank_df_max_agg[
        ["receiver_mean_rank_max", "sender_mean_rank_max"]
    ].max(axis=1)
)

display(MI_ecotyper_CE_meanrank_df_max_agg)


# ============================================================
# Step 5. Save profiles
# ============================================================

MI_ecotyper_CE_meanrank_sender_df.to_csv(
    run_dirs["run_dir"] / "MI_ecotyper_CE_meanrank_profile_sender.csv"
)

MI_ecotyper_CE_meanrank_receiver_df.to_csv(
    run_dirs["run_dir"] / "MI_ecotyper_CE_meanrank_profile_receiver.csv"
)

MI_ecotyper_CE_meanrank_df_max_agg.to_csv(
    run_dirs["run_dir"] / "MI_ecotyper_CE_meanrank_profile_max_summary.csv"
)

print("MI_ecotyper_CE_meanrank_sender_df shape:", MI_ecotyper_CE_meanrank_sender_df.shape)
print("MI_ecotyper_CE_meanrank_receiver_df shape:", MI_ecotyper_CE_meanrank_receiver_df.shape)

In [ ]:
# ============================================================
# Cell 3. Plot Ecotyper CE sender/receiver split-triangle heatmap
# ============================================================

# ============================================================
# Step 1. Sort CE columns by peak-row index in max(sender, receiver)
# ------------------------------------------------------------
# For each MI x CE:
#     triangle_max = max(sender mean-rank, receiver mean-rank)
#
# For each CE column:
#     1) find the MI row where triangle_max is largest
#     2) get that MI's row index in mi_order_use
#     3) sort columns by this row index from small to large
#
# Tie-breaks:
#     - within the same peak row, sort by peak value from high to low
#     - then by column sum from high to low
#     - then by CE name alphabetically
# ============================================================

sender_for_order = MI_ecotyper_CE_meanrank_sender_df.loc[
    mi_order_use,
    common_CEs,
].copy()

receiver_for_order = MI_ecotyper_CE_meanrank_receiver_df.loc[
    mi_order_use,
    common_CEs,
].copy()

triangle_max_for_order = pd.DataFrame(
    np.fmax(
        sender_for_order.to_numpy(dtype=float),
        receiver_for_order.to_numpy(dtype=float),
    ),
    index=sender_for_order.index,
    columns=sender_for_order.columns,
)

mi_to_row_index = {
    mi: i
    for i, mi in enumerate(mi_order_use)
}

column_order_records = []

for ce in triangle_max_for_order.columns:
    col_vals = triangle_max_for_order[ce].replace([np.inf, -np.inf], np.nan)

    if col_vals.notna().sum() == 0:
        peak_mi = None
        peak_row_index = 10**9
        peak_value = np.nan
        column_sum = 0.0
    else:
        peak_mi = col_vals.idxmax()
        peak_row_index = mi_to_row_index.get(peak_mi, 10**9)
        peak_value = float(col_vals.loc[peak_mi])
        column_sum = float(col_vals.sum(skipna=True))

    column_order_records.append({
        "CE": ce,
        "peak_mi": peak_mi,
        "peak_row_index": peak_row_index,
        "peak_value": peak_value,
        "triangle_max_column_sum": column_sum,
    })

ecotyper_CE_column_order_df = pd.DataFrame(column_order_records).set_index("CE")

ecotyper_CE_column_order_df = ecotyper_CE_column_order_df.sort_values(
    by=[
        "peak_row_index",
        "peak_value",
        "triangle_max_column_sum",
        "CE",
    ],
    ascending=[
        True,
        False,
        False,
        True,
    ],
)

CE_order_for_plot = ecotyper_CE_column_order_df.index.tolist()

ecotyper_CE_column_order_df.to_csv(
    run_dirs["run_dir"] / "Ecotyper_CE_column_order_by_triangle_max_peak_row.csv"
)

triangle_max_for_order.loc[
    mi_order_use,
    CE_order_for_plot,
].to_csv(
    run_dirs["run_dir"] / "Ecotyper_CE_triangle_max_sender_receiver_for_column_order.csv"
)

sender_plot_df = MI_ecotyper_CE_meanrank_sender_df.loc[
    mi_order_use,
    CE_order_for_plot,
].copy()

receiver_plot_df = MI_ecotyper_CE_meanrank_receiver_df.loc[
    mi_order_use,
    CE_order_for_plot,
].copy()

sender_plot_df.to_csv(
    run_dirs["run_dir"] / "MI_ecotyper_CE_meanrank_sender_reordered_for_split_heatmap.csv"
)

receiver_plot_df.to_csv(
    run_dirs["run_dir"] / "MI_ecotyper_CE_meanrank_receiver_reordered_for_split_heatmap.csv"
)


# ============================================================
# Step 2. Plot split-triangle heatmap
#   upper triangle = sender
#   lower triangle = receiver
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

sns.set_theme(style="white", context="paper")


def resolve_cmap(cmap_name):
    """
    Resolve colormap name.

    For cmcrameri colormaps, matplotlib may register them as either:
        "managua" or "cmc.managua"
    depending on the package/version.
    """
    if cmap_name in plt.colormaps():
        return mpl.colormaps.get_cmap(cmap_name)

    cmc_name = f"cmc.{cmap_name}"
    if cmc_name in plt.colormaps():
        return mpl.colormaps.get_cmap(cmc_name)

    if cmc is not None and hasattr(cmc, cmap_name):
        return getattr(cmc, cmap_name)

    raise ValueError(
        f"Colormap '{cmap_name}' is not available. "
        "Please install cmcrameri first: pip install cmcrameri"
    )


def plot_split_triangle_heatmap(
    sender_df,
    receiver_df,
    out_prefix,
    title,
    xlabel="Ecotyper carcinoma ecotype",
    ylabel="Meta-interaction",
    cmap="PiYG_r",
    vmin=0,
    vmax=1,
    annotate=False,
    annot_fmt=".2f",
    highlight_threshold=0.6,
    highlight_edgecolor="#A6A6A6",
    highlight_linewidth=3,
    grid_color="#D9D9D9",
    grid_linewidth=0.55,
    rasterize_grid=True,
):
    """
    Plot split-triangle heatmap:
        upper triangle = sender
        lower triangle = receiver
    """
    sender_df = sender_df.copy()
    receiver_df = receiver_df.loc[sender_df.index, sender_df.columns].copy()

    n_rows, n_cols = sender_df.shape

    fig_w = max(7, 0.55 * n_cols)
    fig_h = max(5, 0.38 * n_rows)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cmap_obj = resolve_cmap(cmap)

    highlight_patches = []

    for i, mi in enumerate(sender_df.index):
        for j, ce in enumerate(sender_df.columns):
            s_val = sender_df.loc[mi, ce]
            r_val = receiver_df.loc[mi, ce]

            x0, x1 = j, j + 1
            y0, y1 = i, i + 1

            sender_coords = [[x0, y0], [x1, y0], [x1, y1]]
            receiver_coords = [[x0, y0], [x0, y1], [x1, y1]]

            # Upper triangle: sender
            if pd.notna(s_val):
                s_val_float = float(s_val)

                tri_sender = patches.Polygon(
                    sender_coords,
                    closed=True,
                    facecolor=cmap_obj(norm(s_val_float)),
                    edgecolor="none",
                    linewidth=0,
                    zorder=1,
                )
                ax.add_patch(tri_sender)

                if s_val_float > highlight_threshold:
                    tri_sender_highlight = patches.Polygon(
                        sender_coords,
                        closed=True,
                        facecolor="none",
                        edgecolor=highlight_edgecolor,
                        linewidth=highlight_linewidth,
                        joinstyle="miter",
                        zorder=20,
                    )
                    highlight_patches.append(tri_sender_highlight)

            # Lower triangle: receiver
            if pd.notna(r_val):
                r_val_float = float(r_val)

                tri_receiver = patches.Polygon(
                    receiver_coords,
                    closed=True,
                    facecolor=cmap_obj(norm(r_val_float)),
                    edgecolor="none",
                    linewidth=0,
                    zorder=1,
                )
                ax.add_patch(tri_receiver)

                if r_val_float > highlight_threshold:
                    tri_receiver_highlight = patches.Polygon(
                        receiver_coords,
                        closed=True,
                        facecolor="none",
                        edgecolor=highlight_edgecolor,
                        linewidth=highlight_linewidth,
                        joinstyle="miter",
                        zorder=20,
                    )
                    highlight_patches.append(tri_receiver_highlight)

            # Optional numeric annotation
            if annotate:
                if pd.notna(s_val):
                    ax.text(
                        j + 0.72,
                        i + 0.28,
                        format(float(s_val), annot_fmt),
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color="black",
                        zorder=25,
                    )

                if pd.notna(r_val):
                    ax.text(
                        j + 0.28,
                        i + 0.72,
                        format(float(r_val), annot_fmt),
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color="black",
                        zorder=25,
                    )

    # Grid borders
    grid_segments = []

    for x in range(n_cols + 1):
        grid_segments.append([(x, 0), (x, n_rows)])

    for y in range(n_rows + 1):
        grid_segments.append([(0, y), (n_cols, y)])

    grid_collection = LineCollection(
        grid_segments,
        colors=grid_color,
        linewidths=grid_linewidth,
        zorder=5,
        rasterized=rasterize_grid,
    )
    ax.add_collection(grid_collection)

    for hp in highlight_patches:
        ax.add_patch(hp)

    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.invert_yaxis()

    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_yticks(np.arange(n_rows) + 0.5)

    ax.set_xticklabels(
        sender_df.columns,
        rotation=60,
        ha="right",
        fontsize=10,
    )

    ax.set_yticklabels(
        sender_df.index,
        rotation=0,
        fontsize=10,
    )

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13)

    ax.tick_params(length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    sm = mpl.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        fraction=0.03,
        pad=0.02,
        shrink=0.85,
    )

    cbar.set_label(
        "Normalized mean rank",
        fontsize=11,
    )
    cbar.ax.tick_params(labelsize=9)

    fig.text(
        0.99,
        0.98,
        (
            "Upper triangle: sender\n"
            "Lower triangle: receiver\n"
            f"Border: value > {highlight_threshold}"
        ),
        ha="right",
        va="top",
        fontsize=10,
    )

    plt.tight_layout()

    pdf_path = run_dirs["run_dir"] / f"{out_prefix}.pdf"
    png_path = run_dirs["run_dir"] / f"{out_prefix}.png"

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        transparent=True,
    )

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

    plt.show()
    plt.close(fig)

    print(f"Saved PDF: {pdf_path}")
    print(f"Saved PNG: {png_path}")


plot_split_triangle_heatmap(
    sender_df=sender_plot_df,
    receiver_df=receiver_plot_df,
    out_prefix="MI_ecotyper_CE_meanrank_sender_receiver_split_triangle_heatmap_peakrow_PiYG_lightgray_grid",
    title="Ecotyper CE mean-rank profile\nUpper triangle = sender; lower triangle = receiver",
    xlabel="Ecotyper carcinoma ecotype",
    ylabel="Meta-interaction",
    cmap="PiYG_r",
    vmin=0,
    vmax=1,
    annotate=False,
    annot_fmt=".2f",
    highlight_threshold=0.6,
    highlight_edgecolor="#A6A6A6",
    highlight_linewidth=3,
    grid_color="#D9D9D9",
    grid_linewidth=0.55,
    rasterize_grid=True,
)

In [ ]:
# Release large intermediate objects before the in-situ plotting section.
# Keep `processed`, `run_dirs`, and `mi_order` because they are still needed.
release_memory(
    "Factor_envir_use",
    "batch_cell",
    "MI_mean_pd_tumor",
    "MI_mean_pd_nontumor",
    "MI_mean_pd_tumor_vs_nontumor",
    "MI_mean_pd_tumor_vs_nontumor_log2",
    "MI_mean_pd_tumor_sample",
    "MI_mean_pd_nontumor_sample",
    "MI_mean_pd_tumor_sample_colmax",
    "MI_mean_pd_tumor_sample_colmax_large",
    "pval_s",
    "LR_loading_pathway",
    namespace=globals(),
    run_gc=True,
    clear_cuda=True,
    close_figures=True
)


### Fibroblast-to-tumor MI-4 and malignant programs

Sum incoming fibroblast-to-tumor MI-4 activities for receiver tumor cells and
compare CancerSEA invasion, angiogenesis, and hypoxia scores between the
configured high/low groups. Export full analysis tables and violin/box/SMD plots.


In [ ]:
# ============================================================
# Cell-level aggregated Fibroblast -> tumor MI-4 and CancerSEA module scores
# ------------------------------------------------------------
# Insert this cell immediately after the release_memory(...) cell.
#
# Output:
#   mi4_fibro_to_tumor_celllevel_wide_df
#   CSV:
#     MI4_fibro_to_tumor_celllevel_CancerSEA_scores_wide.csv
#     MI4_fibro_to_tumor_celllevel_CancerSEA_scores_wide_<MODULE_SCORE_METHOD>.csv
# ============================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

# -----------------------------
# User parameters
# -----------------------------
MI_OI = "MI4"
MI_OI_INDEX = 3  # Factor_envir_use[:, 3] -> MI-4 because MI-1 is column 0

SENDER_CELLTYPE = "Fibroblast"
RECEIVER_TUMOR_SUFFIX = "-cancercell"

CANCERSEA_PROGRAMS = ["Invasion", "Angiogenesis", "Hypoxia"]
CANCERSEA_PATH = DATA_ROOT / "CancerSEA_marker"

# Keep only tumor cells that have at least one spatial Fibroblast -> tumor edge.
# The MI-4 level itself is aggregated by sum over those Fibroblast -> tumor edges.
KEEP_ONLY_FIBROBLAST_NEIGHBOR_TUMOR_CELLS = True

# Expression source. Set to a layer name if needed, e.g. "log1p_norm".
EXPRESSION_LAYER = None

# Module score method:
#
#   zscore_mean_within_cancertype:
#       For each CancerType, compute gene-wise mean/std across selected tumor cells
#       from that cancer type, then average z-scored genes in each CancerSEA program.
#
#   zscore_mean_global:
#       Pool selected tumor cells from all cancer types together, compute global
#       gene-wise mean/std, then average z-scored genes in each CancerSEA program.
#
# MODULE_SCORE_METHOD = "zscore_mean_within_cancertype"
MODULE_SCORE_METHOD = "zscore_mean_global"

VALID_MODULE_SCORE_METHODS = {
    "zscore_mean_within_cancertype",
    "zscore_mean_global",
}

if MODULE_SCORE_METHOD not in VALID_MODULE_SCORE_METHODS:
    raise ValueError(
        f"Unsupported MODULE_SCORE_METHOD={MODULE_SCORE_METHOD}. "
        f"Choose from {sorted(VALID_MODULE_SCORE_METHODS)}."
    )

# Output directory
mi4_program_outdir = Path(run_dirs["run_dir"]) / "MI4_fibroblast_to_tumor_celllevel_CancerSEA"
mi4_program_outdir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helper functions
# -----------------------------
def _to_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def _get_field(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    return obj[key]


def _edge_index_to_numpy(edge_index_obj):
    edge_index = _to_numpy(edge_index_obj).astype(np.int64, copy=False)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"edge_index should have shape [E, 2] or [2, E], got {edge_index.shape}")


def _edge_count_from_data(data_obj):
    edge_index = _get_field(data_obj, "edge_index")
    edge_index_np = _to_numpy(edge_index)

    if edge_index_np.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {edge_index_np.shape}")

    if edge_index_np.shape[1] == 2:
        return int(edge_index_np.shape[0])

    if edge_index_np.shape[0] == 2:
        return int(edge_index_np.shape[1])

    raise ValueError(f"edge_index should have shape [E, 2] or [2, E], got {edge_index_np.shape}")


def _get_celltypes(adata, data_obj):
    if "celltype_final" in adata.obs.columns:
        return adata.obs["celltype_final"].astype(str).to_numpy()

    if "cell_class" in data_obj:
        return _to_numpy(data_obj["cell_class"]).astype(str)

    if hasattr(data_obj, "cell_class"):
        return _to_numpy(data_obj.cell_class).astype(str)

    raise KeyError(
        "Cannot find celltype labels. "
        "Expected adata.obs['celltype_final'] or data_obj['cell_class']."
    )


def _get_sample_id(adata, sample_index):
    sample_col_candidates = [
        "SampleID", "sample_id", "sample", "samples", "Sample",
        "slice", "slide", "library_id"
    ]

    for col in sample_col_candidates:
        if col in adata.obs.columns:
            vals = pd.Series(adata.obs[col]).dropna().astype(str).unique()
            if len(vals) > 0:
                return str(vals[0])

    return f"sample_{sample_index}"


def _extract_cancer_type_from_celltype(celltype):
    s = str(celltype)
    if RECEIVER_TUMOR_SUFFIX not in s:
        return None
    return s.replace(RECEIVER_TUMOR_SUFFIX, "").strip(" -_")


def _get_barcode_values(adata, cell_indices):
    barcode_candidates = ["barcode", "Barcode", "cell_id", "cell", "CellID"]

    for col in barcode_candidates:
        if col in adata.obs.columns:
            return adata.obs.iloc[cell_indices][col].astype(str).to_numpy()

    return adata.obs_names[cell_indices].astype(str).to_numpy()


def _get_expr_matrix(adata):
    if EXPRESSION_LAYER is not None:
        if EXPRESSION_LAYER not in adata.layers:
            raise KeyError(f"EXPRESSION_LAYER='{EXPRESSION_LAYER}' not found in adata.layers.")
        return adata.layers[EXPRESSION_LAYER]
    return adata.X


def _expr_rows_cols_to_numpy(adata, rows, cols):
    X = _get_expr_matrix(adata)
    sub = X[rows, :][:, cols]

    if sparse.issparse(sub):
        sub = sub.toarray()
    else:
        sub = np.asarray(sub)

    if sub.ndim == 1:
        sub = sub.reshape(-1, 1)

    return sub.astype(np.float32, copy=False)


def _resolve_gene_indices(var_names, genes):
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_idx = {}

    for i, g in enumerate(var_names):
        upper_to_idx[str(g).upper()] = i

    idx = []
    used = []

    for g in genes:
        g = str(g).strip()
        if len(g) == 0:
            continue

        if g in var_names:
            idx_cur = int(var_names.get_loc(g))
        else:
            idx_cur = upper_to_idx.get(g.upper(), None)

        if idx_cur is not None:
            idx.append(int(idx_cur))
            used.append(str(var_names[idx_cur]))

    seen = set()
    idx_unique = []
    used_unique = []

    for i, g in zip(idx, used):
        if i not in seen:
            seen.add(i)
            idx_unique.append(i)
            used_unique.append(g)

    return np.asarray(idx_unique, dtype=np.int64), used_unique


def _load_cancersea_gene_sets(programs, cancersea_path, var_names):
    gene_sets = {}

    for program in programs:
        path = cancersea_path / f"{program}.txt"
        if not path.exists():
            raise FileNotFoundError(f"Cannot find CancerSEA marker file: {path}")

        df = pd.read_csv(path, sep="\t", header=0)

        if "GeneName" not in df.columns:
            raise KeyError(f"{path} does not contain a 'GeneName' column.")

        genes_raw = (
            df["GeneName"]
            .dropna()
            .astype(str)
            .str.strip()
            .loc[lambda x: x.ne("")]
            .drop_duplicates()
            .tolist()
        )

        gene_idx, genes_used = _resolve_gene_indices(var_names, genes_raw)

        if len(gene_idx) == 0:
            raise ValueError(f"No genes from CancerSEA {program} are found in adata.var_names.")

        gene_sets[program] = {
            "genes_raw": genes_raw,
            "gene_idx": gene_idx,
            "genes_used": genes_used,
        }

    return gene_sets


def _build_gene_reference_from_sums(gene_sum, gene_sumsq, gene_n):
    """
    Build gene-wise mean/std reference from running sum/sumsq/count.
    """
    if gene_n <= 0:
        raise ValueError("Cannot build gene reference with gene_n <= 0.")

    mean = gene_sum / gene_n
    var = (gene_sumsq / gene_n) - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-8))

    std[~np.isfinite(std)] = 1.0
    std[std < 1e-6] = 1.0

    return {
        "mean": mean.astype(np.float32),
        "std": std.astype(np.float32),
        "n_ref": int(gene_n),
    }


# -----------------------------
# Prepare Factor_envir_use memory map and edge offsets
# -----------------------------
factor_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"

if not factor_path.exists():
    raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {factor_path}")

Factor_envir_use_mmap = np.load(factor_path, mmap_mode="r")

edge_counts = np.asarray(
    [_edge_count_from_data(processed.spidernet_data[i]) for i in range(len(processed.spidernet_data))],
    dtype=np.int64,
)

edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))

if Factor_envir_use_mmap.shape[0] != int(edge_offsets[-1]):
    if Factor_envir_use_mmap.shape[1] == int(edge_offsets[-1]):
        raise ValueError(
            "Factor_envir_use appears to be MI x edge. "
            "This cell expects edge x MI. Please transpose and save before running."
        )
    raise ValueError(
        f"Factor_envir_use shape {Factor_envir_use_mmap.shape} is not aligned "
        f"with total edge count {edge_offsets[-1]}."
    )

if MI_OI_INDEX >= Factor_envir_use_mmap.shape[1]:
    raise ValueError(
        f"MI_OI_INDEX={MI_OI_INDEX} is out of range for Factor_envir_use with "
        f"{Factor_envir_use_mmap.shape[1]} MI columns."
    )

# -----------------------------
# Load CancerSEA gene sets against the first adata var_names
# -----------------------------
first_adata = processed.adata_list[0]
cancersea_gene_sets = _load_cancersea_gene_sets(
    programs=CANCERSEA_PROGRAMS,
    cancersea_path=CANCERSEA_PATH,
    var_names=first_adata.var_names,
)

all_gene_idx = np.unique(
    np.concatenate([v["gene_idx"] for v in cancersea_gene_sets.values()])
).astype(np.int64)

gene_idx_to_pos = {int(gidx): pos for pos, gidx in enumerate(all_gene_idx)}

program_gene_pos = {
    program: np.asarray([gene_idx_to_pos[int(i)] for i in info["gene_idx"]], dtype=np.int64)
    for program, info in cancersea_gene_sets.items()
}

print("CancerSEA genes used:")
for program, info in cancersea_gene_sets.items():
    print(f"  {program}: {len(info['genes_used'])} genes")

# -----------------------------
# First pass:
#   1) aggregate incoming Fibroblast -> tumor MI-4 by sum;
#   2) store selected cell metadata;
#   3) compute gene-wise mean/std references:
#        - within cancer type
#        - global across all cancer types
# -----------------------------
metadata_frames = []

gene_sum_by_ct = {}
gene_sumsq_by_ct = {}
gene_n_by_ct = {}

gene_sum_global = np.zeros(len(all_gene_idx), dtype=np.float64)
gene_sumsq_global = np.zeros(len(all_gene_idx), dtype=np.float64)
gene_n_global = 0

for sample_index, (adata, data_obj) in enumerate(zip(processed.adata_list, processed.spidernet_data)):
    if sample_index % 10 == 0:
        print(f"[First pass] sample {sample_index + 1}/{len(processed.adata_list)}")

    edge_index = _edge_index_to_numpy(_get_field(data_obj, "edge_index"))
    celltypes = _get_celltypes(adata, data_obj)

    if edge_index.shape[0] != edge_counts[sample_index]:
        raise ValueError(
            f"Edge count mismatch for sample {sample_index}: "
            f"{edge_index.shape[0]} vs expected {edge_counts[sample_index]}"
        )

    start = int(edge_offsets[sample_index])
    end = int(edge_offsets[sample_index + 1])

    mi4_edge = np.asarray(
        Factor_envir_use_mmap[start:end, MI_OI_INDEX],
        dtype=np.float32
    )

    src = edge_index[:, 0]
    dst = edge_index[:, 1]

    sender_ct = celltypes[src].astype(str)
    receiver_ct = celltypes[dst].astype(str)

    tumor_receiver_edge = np.char.find(receiver_ct.astype(str), RECEIVER_TUMOR_SUFFIX) >= 0
    fibro_to_tumor_edge = (sender_ct == SENDER_CELLTYPE) & tumor_receiver_edge

    n_cells = adata.n_obs

    mi4_incoming_sum = np.bincount(
        dst[fibro_to_tumor_edge],
        weights=mi4_edge[fibro_to_tumor_edge],
        minlength=n_cells,
    ).astype(np.float32)

    fibro_incoming_edge_count = np.bincount(
        dst[fibro_to_tumor_edge],
        minlength=n_cells,
    ).astype(np.int32)

    tumor_cell_mask = np.char.find(celltypes.astype(str), RECEIVER_TUMOR_SUFFIX) >= 0

    if KEEP_ONLY_FIBROBLAST_NEIGHBOR_TUMOR_CELLS:
        keep_mask = tumor_cell_mask & (fibro_incoming_edge_count > 0)
    else:
        keep_mask = tumor_cell_mask

    cell_idx = np.where(keep_mask)[0].astype(np.int64)

    if len(cell_idx) == 0:
        continue

    celltype_sel = celltypes[cell_idx].astype(str)
    cancer_type_sel = np.asarray([
        _extract_cancer_type_from_celltype(x) for x in celltype_sel
    ], dtype=object)

    valid_ct = pd.notna(cancer_type_sel)
    cell_idx = cell_idx[valid_ct]
    celltype_sel = celltype_sel[valid_ct]
    cancer_type_sel = cancer_type_sel[valid_ct].astype(str)

    if len(cell_idx) == 0:
        continue

    barcode_sel = _get_barcode_values(adata, cell_idx)
    sample_id = _get_sample_id(adata, sample_index)

    meta_df = pd.DataFrame({
        "sample_index": sample_index,
        "sample_id": sample_id,
        "barcode": barcode_sel,
        "cell_index": cell_idx,
        "celltype_final": celltype_sel,
        "CancerType": cancer_type_sel,
        "MI": MI_OI,
        "MI4_Fibroblast_to_tumor_sum": mi4_incoming_sum[cell_idx],
        "Fibroblast_to_tumor_edge_count": fibro_incoming_edge_count[cell_idx],
    })

    metadata_frames.append(meta_df)

    # Expression matrix for selected tumor cells and CancerSEA genes.
    X_sel = _expr_rows_cols_to_numpy(adata, cell_idx, all_gene_idx)
    X_sel64 = X_sel.astype(np.float64, copy=False)

    # Global reference stats across all selected tumor cells.
    gene_sum_global += np.nansum(X_sel64, axis=0)
    gene_sumsq_global += np.nansum(X_sel64 ** 2, axis=0)
    gene_n_global += int(X_sel64.shape[0])

    # Cancer-type-specific reference stats.
    for cancer_type in np.unique(cancer_type_sel):
        rows = np.where(cancer_type_sel == cancer_type)[0]

        if len(rows) == 0:
            continue

        X_ct = X_sel64[rows, :]

        if cancer_type not in gene_sum_by_ct:
            gene_sum_by_ct[cancer_type] = np.zeros(X_ct.shape[1], dtype=np.float64)
            gene_sumsq_by_ct[cancer_type] = np.zeros(X_ct.shape[1], dtype=np.float64)
            gene_n_by_ct[cancer_type] = 0

        gene_sum_by_ct[cancer_type] += np.nansum(X_ct, axis=0)
        gene_sumsq_by_ct[cancer_type] += np.nansum(X_ct ** 2, axis=0)
        gene_n_by_ct[cancer_type] += int(X_ct.shape[0])

    del X_sel, X_sel64
    gc.collect()

if len(metadata_frames) == 0:
    raise ValueError("No selected tumor cells were found for Fibroblast -> tumor MI-4 aggregation.")

mi4_fibro_to_tumor_meta_df = pd.concat(metadata_frames, axis=0, ignore_index=True)

print("Selected cell counts by cancer type:")
display(
    mi4_fibro_to_tumor_meta_df
    .groupby("CancerType", as_index=False)
    .agg(n_cells=("cell_index", "size"))
    .sort_values("CancerType")
)

# -----------------------------
# Build reference mean/std
# -----------------------------
gene_ref_by_ct = {}

for cancer_type in sorted(gene_sum_by_ct.keys()):
    n = gene_n_by_ct[cancer_type]

    if n <= 0:
        continue

    gene_ref_by_ct[cancer_type] = _build_gene_reference_from_sums(
        gene_sum=gene_sum_by_ct[cancer_type],
        gene_sumsq=gene_sumsq_by_ct[cancer_type],
        gene_n=n,
    )

gene_ref_global = _build_gene_reference_from_sums(
    gene_sum=gene_sum_global,
    gene_sumsq=gene_sumsq_global,
    gene_n=gene_n_global,
)

print(f"Global z-score reference cells: n = {gene_ref_global['n_ref']:,}")

if MODULE_SCORE_METHOD == "zscore_mean_within_cancertype":
    print("Using cancer-type-specific z-score references.")
    for cancer_type in sorted(gene_ref_by_ct.keys()):
        print(f"  {cancer_type}: n_ref = {gene_ref_by_ct[cancer_type]['n_ref']:,}")

elif MODULE_SCORE_METHOD == "zscore_mean_global":
    print("Using global z-score reference pooled across all selected tumor cells.")

# -----------------------------
# Second pass:
#   compute CancerSEA scores for the selected cells.
# -----------------------------
score_frames = []

for sample_index, meta_sub in mi4_fibro_to_tumor_meta_df.groupby("sample_index", sort=True):
    sample_index = int(sample_index)

    if sample_index % 10 == 0:
        print(f"[Second pass] sample {sample_index + 1}/{len(processed.adata_list)}")

    adata = processed.adata_list[sample_index]

    cell_idx = meta_sub["cell_index"].to_numpy(dtype=np.int64)
    X_sel = _expr_rows_cols_to_numpy(adata, cell_idx, all_gene_idx)

    score_df = meta_sub.copy().reset_index(drop=True)

    for program in CANCERSEA_PROGRAMS:
        score_df[f"{program}_score"] = np.nan
        score_df[f"{program}_n_genes_used"] = int(len(program_gene_pos[program]))

    if MODULE_SCORE_METHOD == "zscore_mean_global":
        ref = gene_ref_global
        X_z = (X_sel - ref["mean"][None, :]) / ref["std"][None, :]

        for program in CANCERSEA_PROGRAMS:
            cols = program_gene_pos[program]
            score_df[f"{program}_score"] = np.nanmean(X_z[:, cols], axis=1)

        score_df["zscore_reference"] = "global"
        score_df["zscore_reference_n"] = int(ref["n_ref"])

    elif MODULE_SCORE_METHOD == "zscore_mean_within_cancertype":
        cancer_type_arr = score_df["CancerType"].astype(str).to_numpy()

        for cancer_type in np.unique(cancer_type_arr):
            if cancer_type not in gene_ref_by_ct:
                continue

            rows = np.where(cancer_type_arr == cancer_type)[0]

            ref = gene_ref_by_ct[cancer_type]
            X_z = (X_sel[rows, :] - ref["mean"][None, :]) / ref["std"][None, :]

            for program in CANCERSEA_PROGRAMS:
                cols = program_gene_pos[program]
                score_df.loc[rows, f"{program}_score"] = np.nanmean(X_z[:, cols], axis=1)

            score_df.loc[rows, "zscore_reference"] = str(cancer_type)
            score_df.loc[rows, "zscore_reference_n"] = int(ref["n_ref"])

    else:
        raise ValueError(
            f"Unsupported MODULE_SCORE_METHOD={MODULE_SCORE_METHOD}. "
            f"Choose from {sorted(VALID_MODULE_SCORE_METHODS)}."
        )

    score_df["module_score_method"] = MODULE_SCORE_METHOD
    score_frames.append(score_df)

    del X_sel
    if "X_z" in locals():
        del X_z
    gc.collect()

mi4_fibro_to_tumor_celllevel_wide_df = pd.concat(score_frames, axis=0, ignore_index=True)

# -----------------------------
# Save outputs
# -----------------------------
wide_out = mi4_program_outdir / "MI4_fibro_to_tumor_celllevel_CancerSEA_scores_wide.csv"
wide_out_method = mi4_program_outdir / (
    f"MI4_fibro_to_tumor_celllevel_CancerSEA_scores_wide_"
    f"{MODULE_SCORE_METHOD}.csv"
)

mi4_fibro_to_tumor_celllevel_wide_df.to_csv(wide_out, index=False)
mi4_fibro_to_tumor_celllevel_wide_df.to_csv(wide_out_method, index=False)

gene_counts_out = mi4_program_outdir / "CancerSEA_genes_used_MI4_program_scores.csv"
pd.DataFrame([
    {
        "Program": program,
        "n_genes_used": len(info["genes_used"]),
        "genes_used": ";".join(info["genes_used"]),
    }
    for program, info in cancersea_gene_sets.items()
]).to_csv(gene_counts_out, index=False)

zscore_ref_out = mi4_program_outdir / (
    f"MI4_CancerSEA_zscore_reference_summary_"
    f"{MODULE_SCORE_METHOD}.csv"
)

zscore_ref_rows = []

if MODULE_SCORE_METHOD == "zscore_mean_global":
    zscore_ref_rows.append({
        "reference": "global",
        "CancerType": "all",
        "n_ref": int(gene_ref_global["n_ref"]),
        "module_score_method": MODULE_SCORE_METHOD,
    })

elif MODULE_SCORE_METHOD == "zscore_mean_within_cancertype":
    for cancer_type in sorted(gene_ref_by_ct.keys()):
        zscore_ref_rows.append({
            "reference": str(cancer_type),
            "CancerType": str(cancer_type),
            "n_ref": int(gene_ref_by_ct[cancer_type]["n_ref"]),
            "module_score_method": MODULE_SCORE_METHOD,
        })

pd.DataFrame(zscore_ref_rows).to_csv(zscore_ref_out, index=False)

print(f"Saved wide cell-level MI-4/CancerSEA table: {wide_out}")
print(f"Saved method-specific wide table: {wide_out_method}")
print(f"Saved CancerSEA genes used: {gene_counts_out}")
print(f"Saved z-score reference summary: {zscore_ref_out}")

display(mi4_fibro_to_tumor_celllevel_wide_df.head())

In [ ]:
# ============================================================
# Split tumor cells into MI-4-high / MI-4-low groups and compute SMD
# ------------------------------------------------------------
# Output:
#   mi4_program_long_df
#   mi4_program_stats_df
#   CSVs in mi4_program_outdir
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# -----------------------------
# User parameters
# -----------------------------
# Recommended default for pan-cancer faceting:
#   median_by_cancertype gives both high and low groups within each cancer type.
#
# Optional fixed-threshold grouping:
#   MI4_GROUPING_METHOD = "fixed_threshold"
#   MI4_AGG_FIXED_THRESHOLD = 0.5
MI4_GROUPING_METHOD = "median_by_cancertype"  # "median_by_cancertype" or "fixed_threshold"
MI4_AGG_FIXED_THRESHOLD = 0.5

GROUP_LOW = "MI-4 low"
GROUP_HIGH = "MI-4 high"
GROUP_ORDER = [GROUP_LOW, GROUP_HIGH]

# Mann–Whitney direction for testing program score in high > low.
TEST_ALTERNATIVE = "greater"

# Order in final 3 x 8 figure
CANCERTYPE_ORDER = [
    "Breast",
    "Colon",
    "Liver",
    "Lung",
    "Melanoma",
    "Ovarian",
    "Prostate",
    "Uterine",
]

PROGRAM_ORDER = ["Invasion", "Angiogenesis", "Hypoxia"]

# -----------------------------
# Helpers
# -----------------------------
def _p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _smd_high_minus_low(high_vals, low_vals):
    high_vals = np.asarray(high_vals, dtype=float)
    low_vals = np.asarray(low_vals, dtype=float)

    high_vals = high_vals[np.isfinite(high_vals)]
    low_vals = low_vals[np.isfinite(low_vals)]

    if len(high_vals) == 0 or len(low_vals) == 0:
        return np.nan

    mean_high = np.mean(high_vals)
    mean_low = np.mean(low_vals)

    sd_high = np.std(high_vals, ddof=1) if len(high_vals) > 1 else np.nan
    sd_low = np.std(low_vals, ddof=1) if len(low_vals) > 1 else np.nan

    if len(high_vals) > 1 and len(low_vals) > 1 and np.isfinite(sd_high) and np.isfinite(sd_low):
        pooled_sd = np.sqrt(
            ((len(high_vals) - 1) * sd_high ** 2 + (len(low_vals) - 1) * sd_low ** 2)
            / (len(high_vals) + len(low_vals) - 2)
        )
    else:
        pooled_sd = np.nan

    if not np.isfinite(pooled_sd) or pooled_sd <= 0:
        return np.nan

    return (mean_high - mean_low) / pooled_sd


def _split_mi4_groups(df):
    """Add MI4_group and MI4_group_threshold columns."""
    df = df.copy()
    df["MI4_group"] = pd.NA
    df["MI4_group_threshold"] = np.nan

    if MI4_GROUPING_METHOD == "fixed_threshold":
        thr = float(MI4_AGG_FIXED_THRESHOLD)
        df["MI4_group_threshold"] = thr
        df["MI4_group"] = np.where(
            df["MI4_Fibroblast_to_tumor_sum"] >= thr,
            GROUP_HIGH,
            GROUP_LOW,
        )
        return df

    if MI4_GROUPING_METHOD != "median_by_cancertype":
        raise ValueError("MI4_GROUPING_METHOD must be 'median_by_cancertype' or 'fixed_threshold'.")

    # Within-cancer-type median split. If ties make one group empty,
    # use rank-based half split as a fallback.
    for cancer_type, idx in df.groupby("CancerType").groups.items():
        vals = pd.to_numeric(
            df.loc[idx, "MI4_Fibroblast_to_tumor_sum"],
            errors="coerce"
        ).to_numpy(dtype=float)

        finite_mask = np.isfinite(vals)
        if finite_mask.sum() == 0:
            continue

        thr = float(np.nanmedian(vals[finite_mask]))
        groups = np.where(vals >= thr, GROUP_HIGH, GROUP_LOW)

        # Tie-heavy fallback: split by rank to ensure both groups when possible.
        if len(np.unique(groups[finite_mask])) < 2 and finite_mask.sum() >= 2:
            ranks = pd.Series(vals).rank(method="first", na_option="keep").to_numpy()
            median_rank = np.nanmedian(ranks[finite_mask])
            groups = np.where(ranks > median_rank, GROUP_HIGH, GROUP_LOW)

        df.loc[idx, "MI4_group_threshold"] = thr
        df.loc[idx, "MI4_group"] = groups

    return df


# -----------------------------
# Add group labels to wide table
# -----------------------------
if "mi4_fibro_to_tumor_celllevel_wide_df" not in globals():
    wide_path = mi4_program_outdir / "MI4_fibro_to_tumor_celllevel_CancerSEA_scores_wide.csv"
    if not wide_path.exists():
        raise FileNotFoundError(
            "mi4_fibro_to_tumor_celllevel_wide_df is not in memory and CSV was not found:\n"
            f"{wide_path}"
        )
    mi4_fibro_to_tumor_celllevel_wide_df = pd.read_csv(wide_path)

mi4_fibro_to_tumor_celllevel_wide_df = _split_mi4_groups(mi4_fibro_to_tumor_celllevel_wide_df)
mi4_fibro_to_tumor_celllevel_wide_df["MI4_group"] = pd.Categorical(
    mi4_fibro_to_tumor_celllevel_wide_df["MI4_group"],
    categories=GROUP_ORDER,
    ordered=True,
)

# -----------------------------
# Convert to long format
# -----------------------------
long_frames = []

for program in PROGRAM_ORDER:
    score_col = f"{program}_score"
    n_gene_col = f"{program}_n_genes_used"

    if score_col not in mi4_fibro_to_tumor_celllevel_wide_df.columns:
        raise KeyError(f"Cannot find score column: {score_col}")

    cols = [
        "sample_index",
        "sample_id",
        "barcode",
        "cell_index",
        "celltype_final",
        "CancerType",
        "MI",
        "MI4_Fibroblast_to_tumor_sum",
        "Fibroblast_to_tumor_edge_count",
        "MI4_group",
        "MI4_group_threshold",
        score_col,
    ]

    if n_gene_col in mi4_fibro_to_tumor_celllevel_wide_df.columns:
        cols.append(n_gene_col)

    tmp = mi4_fibro_to_tumor_celllevel_wide_df[cols].copy()
    tmp = tmp.rename(columns={score_col: "State_Score", n_gene_col: "n_genes_used"})
    tmp["Program"] = program

    long_frames.append(tmp)

mi4_program_long_df = pd.concat(long_frames, axis=0, ignore_index=True)

mi4_program_long_df["CancerType"] = pd.Categorical(
    mi4_program_long_df["CancerType"].astype(str),
    categories=CANCERTYPE_ORDER,
    ordered=True,
)

mi4_program_long_df["Program"] = pd.Categorical(
    mi4_program_long_df["Program"].astype(str),
    categories=PROGRAM_ORDER,
    ordered=True,
)

mi4_program_long_df["MI4_group"] = pd.Categorical(
    mi4_program_long_df["MI4_group"].astype(str),
    categories=GROUP_ORDER,
    ordered=True,
)

mi4_program_long_df["State_Score"] = pd.to_numeric(
    mi4_program_long_df["State_Score"],
    errors="coerce",
)

mi4_program_long_df = mi4_program_long_df.loc[
    mi4_program_long_df["CancerType"].notna()
    & mi4_program_long_df["Program"].notna()
    & mi4_program_long_df["MI4_group"].notna()
    & np.isfinite(mi4_program_long_df["State_Score"].to_numpy(dtype=float))
].copy()

# -----------------------------
# Compute per cancer type x program statistics
# -----------------------------
stats_rows = []

for (cancer_type, program), sub in mi4_program_long_df.groupby(
    ["CancerType", "Program"],
    observed=True,
):
    high_vals = sub.loc[sub["MI4_group"] == GROUP_HIGH, "State_Score"].to_numpy(dtype=float)
    low_vals = sub.loc[sub["MI4_group"] == GROUP_LOW, "State_Score"].to_numpy(dtype=float)

    high_vals = high_vals[np.isfinite(high_vals)]
    low_vals = low_vals[np.isfinite(low_vals)]

    p_value = np.nan
    if len(high_vals) > 0 and len(low_vals) > 0:
        try:
            p_value = mannwhitneyu(
                high_vals,
                low_vals,
                alternative=TEST_ALTERNATIVE,
                method="asymptotic",
            ).pvalue
        except TypeError:
            p_value = mannwhitneyu(
                high_vals,
                low_vals,
                alternative=TEST_ALTERNATIVE,
            ).pvalue

    smd = _smd_high_minus_low(high_vals, low_vals)

    mean_high = np.nanmean(high_vals) if len(high_vals) else np.nan
    mean_low = np.nanmean(low_vals) if len(low_vals) else np.nan

    threshold_vals = pd.to_numeric(
        sub["MI4_group_threshold"],
        errors="coerce"
    ).dropna().unique()

    threshold_used = threshold_vals[0] if len(threshold_vals) > 0 else np.nan

    stats_rows.append({
        "CancerType": str(cancer_type),
        "Program": str(program),
        "MI": MI_OI,
        "MI4_grouping_method": MI4_GROUPING_METHOD,
        "MI4_group_threshold": threshold_used,
        "test": "Mann-Whitney U",
        "alternative": TEST_ALTERNATIVE,
        "n_high": int(len(high_vals)),
        "n_low": int(len(low_vals)),
        "mean_high": float(mean_high) if np.isfinite(mean_high) else np.nan,
        "mean_low": float(mean_low) if np.isfinite(mean_low) else np.nan,
        "mean_diff_high_minus_low": (
            float(mean_high - mean_low)
            if np.isfinite(mean_high) and np.isfinite(mean_low)
            else np.nan
        ),
        "SMD_high_minus_low": float(smd) if np.isfinite(smd) else np.nan,
        "p_value": float(p_value) if np.isfinite(p_value) else np.nan,
        "p_star": _p_to_star(p_value),
    })

mi4_program_stats_df = pd.DataFrame(stats_rows)

mi4_program_stats_df["CancerType"] = pd.Categorical(
    mi4_program_stats_df["CancerType"].astype(str),
    categories=CANCERTYPE_ORDER,
    ordered=True,
)

mi4_program_stats_df["Program"] = pd.Categorical(
    mi4_program_stats_df["Program"].astype(str),
    categories=PROGRAM_ORDER,
    ordered=True,
)

mi4_program_stats_df = mi4_program_stats_df.sort_values(
    ["Program", "CancerType"]
).reset_index(drop=True)

# -----------------------------
# Save outputs
# -----------------------------
long_out = mi4_program_outdir / (
    f"MI4_fibro_to_tumor_celllevel_CancerSEA_scores_long_"
    f"{MI4_GROUPING_METHOD}.csv"
)
stats_out = mi4_program_outdir / (
    f"MI4_fibro_to_tumor_CancerSEA_SMD_stats_"
    f"{MI4_GROUPING_METHOD}.csv"
)
wide_grouped_out = mi4_program_outdir / (
    f"MI4_fibro_to_tumor_celllevel_CancerSEA_scores_wide_"
    f"{MI4_GROUPING_METHOD}.csv"
)

mi4_program_long_df.to_csv(long_out, index=False)
mi4_program_stats_df.to_csv(stats_out, index=False)
mi4_fibro_to_tumor_celllevel_wide_df.to_csv(wide_grouped_out, index=False)

print(f"Saved long table: {long_out}")
print(f"Saved SMD/statistics table: {stats_out}")
print(f"Saved grouped wide table: {wide_grouped_out}")

display(mi4_program_stats_df)

In [ ]:
# ============================================================
# Final 3 programs x 8 cancer types faceted violin + boxplot with SMD
# ------------------------------------------------------------
# Statistics are computed in Cell 2 using all cells.
# For visualization only, optionally downsample cells per panel/group.
#
# Style:
#   - High first, Low second
#   - High violin: edge #e93732, fill #e93732 with alpha = 50%
#   - Low violin:  edge #d9adac, fill #d9adac with alpha = 50%
#   - Boxplot edge follows group color
#   - Boxplot median line is white
#   - Remove boxplot upper/lower horizontal caps
#
# Output:
#   MI4_fibro_to_tumor_CancerSEA_3x8_violin_boxplot_SMD_*.pdf/png
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba

# -----------------------------
# Plot parameters
# -----------------------------
PLOT_MAX_CELLS_PER_PANEL_GROUP = 5000
PLOT_RANDOM_STATE = 2026

# High first, Low second
PLOT_GROUP_ORDER = [GROUP_HIGH, GROUP_LOW]

GROUP_EDGE_COLORS = {
    GROUP_HIGH: "#e93732",
    GROUP_LOW:  "#d9adac",
}

GROUP_FILL_COLORS = {
    GROUP_HIGH: "#e93732",
    GROUP_LOW:  "#d9adac",
}

GROUP_XTICK_LABELS = ["High", "Low"]

VIOLIN_ALPHA = 0.50
BOX_FACE_ALPHA = 0.92
BOX_WIDTH = 0.30

MEDIAN_COLOR = "white"
ANNOTATION_COLOR = "black"

# PDF-friendly settings
plt.close("all")
plt.style.use("default")

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.45,
    "xtick.major.width": 0.45,
    "ytick.major.width": 0.45,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

sns.set_theme(style="white", context="paper")

# -----------------------------
# Load data if needed
# -----------------------------
if "mi4_program_long_df" not in globals():
    long_path = mi4_program_outdir / (
        f"MI4_fibro_to_tumor_celllevel_CancerSEA_scores_long_"
        f"{MI4_GROUPING_METHOD}.csv"
    )
    if not long_path.exists():
        raise FileNotFoundError(f"Cannot find long table: {long_path}")
    mi4_program_long_df = pd.read_csv(long_path)

if "mi4_program_stats_df" not in globals():
    stats_path = mi4_program_outdir / (
        f"MI4_fibro_to_tumor_CancerSEA_SMD_stats_"
        f"{MI4_GROUPING_METHOD}.csv"
    )
    if not stats_path.exists():
        raise FileNotFoundError(f"Cannot find statistics table: {stats_path}")
    mi4_program_stats_df = pd.read_csv(stats_path)

# -----------------------------
# Re-apply categorical orders
# -----------------------------
mi4_program_long_df["CancerType"] = pd.Categorical(
    mi4_program_long_df["CancerType"].astype(str),
    categories=CANCERTYPE_ORDER,
    ordered=True,
)

mi4_program_long_df["Program"] = pd.Categorical(
    mi4_program_long_df["Program"].astype(str),
    categories=PROGRAM_ORDER,
    ordered=True,
)

# Important: force plotting order as High -> Low
mi4_program_long_df["MI4_group"] = pd.Categorical(
    mi4_program_long_df["MI4_group"].astype(str),
    categories=PLOT_GROUP_ORDER,
    ordered=True,
)

mi4_program_stats_df["CancerType"] = pd.Categorical(
    mi4_program_stats_df["CancerType"].astype(str),
    categories=CANCERTYPE_ORDER,
    ordered=True,
)

mi4_program_stats_df["Program"] = pd.Categorical(
    mi4_program_stats_df["Program"].astype(str),
    categories=PROGRAM_ORDER,
    ordered=True,
)

# -----------------------------
# Downsample for plotting only
# -----------------------------
plot_df = (
    mi4_program_long_df
    .dropna(subset=["State_Score", "MI4_group", "CancerType", "Program"])
    .copy()
)

plot_df["State_Score"] = pd.to_numeric(plot_df["State_Score"], errors="coerce")
plot_df = plot_df.loc[
    np.isfinite(plot_df["State_Score"].to_numpy(dtype=float))
].copy()

plot_df["MI4_group"] = pd.Categorical(
    plot_df["MI4_group"].astype(str),
    categories=PLOT_GROUP_ORDER,
    ordered=True,
)

if PLOT_MAX_CELLS_PER_PANEL_GROUP is not None:
    plot_df = (
        plot_df
        .groupby(["Program", "CancerType", "MI4_group"], observed=True, group_keys=False)
        .apply(
            lambda x: x.sample(
                n=min(len(x), int(PLOT_MAX_CELLS_PER_PANEL_GROUP)),
                random_state=PLOT_RANDOM_STATE
            )
        )
        .reset_index(drop=True)
    )

    plot_df["MI4_group"] = pd.Categorical(
        plot_df["MI4_group"].astype(str),
        categories=PLOT_GROUP_ORDER,
        ordered=True,
    )

plot_df_out = mi4_program_outdir / (
    f"MI4_fibro_to_tumor_CancerSEA_3x8_plotting_table_"
    f"{MI4_GROUPING_METHOD}_max{PLOT_MAX_CELLS_PER_PANEL_GROUP}.csv"
)
plot_df.to_csv(plot_df_out, index=False)
print(f"Saved plotting table: {plot_df_out}")

# -----------------------------
# Plot helpers
# -----------------------------
def _format_smd_label(smd):
    if pd.isna(smd) or not np.isfinite(float(smd)):
        return "SMD = NA"
    return f"SMD = {float(smd):.2f}"


def _format_panel_stat_label(row):
    smd_label = _format_smd_label(row["SMD_high_minus_low"])
    p_star = row["p_star"] if "p_star" in row and pd.notna(row["p_star"]) else "NA"
    return f"{p_star}\n{smd_label}"


def _present_group_order(sub_df):
    """Return groups present in this panel, preserving High -> Low order."""
    present = set(sub_df["MI4_group"].astype(str))
    return [g for g in PLOT_GROUP_ORDER if str(g) in present]


def _style_violin_collections(ax, violin_collections, group_order_present):
    """
    Apply group-specific edge color and 50% fill alpha to violin bodies.
    """
    for coll, group in zip(violin_collections, group_order_present):
        fill_color = GROUP_FILL_COLORS[group]
        edge_color = GROUP_EDGE_COLORS[group]

        try:
            coll.set_facecolor(to_rgba(fill_color, VIOLIN_ALPHA))
            coll.set_edgecolor(edge_color)
            coll.set_linewidth(0.45)
            coll.set_alpha(None)
        except Exception:
            pass


def _style_boxplot_artists(ax, before_patch_count, group_order_present):
    """
    Apply group-specific box edge/fill colors after seaborn.boxplot.
    Also removes caps by using showcaps=False in sns.boxplot.
    """
    new_patches = ax.patches[before_patch_count:]

    for patch, group in zip(new_patches, group_order_present):
        edge_color = GROUP_EDGE_COLORS[group]
        fill_color = GROUP_FILL_COLORS[group]

        try:
            patch.set_facecolor(to_rgba(fill_color, BOX_FACE_ALPHA))
            patch.set_edgecolor(edge_color)
            patch.set_linewidth(0.55)
        except Exception:
            pass

    # In seaborn/matplotlib, boxplot lines are added in order per box:
    # whisker1, whisker2, cap1, cap2, median, plus sometimes extra lines.
    # Because showcaps=False, cap lines should not be drawn.
    # Here we still recolor whiskers and medians robustly.
    lines = ax.lines

    # Approximate from the end: each visible box usually contributes
    # two whiskers + one median when showcaps=False.
    n_groups = len(group_order_present)
    if n_groups > 0:
        candidate_lines = lines[-3 * n_groups:]

        for i, group in enumerate(group_order_present):
            edge_color = GROUP_EDGE_COLORS[group]
            group_lines = candidate_lines[(3 * i):(3 * i + 3)]

            for j, line in enumerate(group_lines):
                try:
                    # First two are whiskers; third is median.
                    if j < 2:
                        line.set_color(edge_color)
                        line.set_linewidth(0.55)
                    else:
                        line.set_color(MEDIAN_COLOR)
                        line.set_linewidth(0.90)
                except Exception:
                    pass


# -----------------------------
# Make 3 x 8 figure
# -----------------------------
n_rows = len(PROGRAM_ORDER)
n_cols = len(CANCERTYPE_ORDER)

fig_width = 1.35 * n_cols + 1.2
fig_height = 1.55 * n_rows + 1.0

plt.close("all")
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(fig_width, fig_height),
    sharex=True,
    sharey=False,
    facecolor="white",
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)
if n_cols == 1:
    axes = np.expand_dims(axes, axis=1)

for r, program in enumerate(PROGRAM_ORDER):
    for c, cancer_type in enumerate(CANCERTYPE_ORDER):
        ax = axes[r, c]

        sub = plot_df.loc[
            (plot_df["Program"].astype(str) == str(program))
            & (plot_df["CancerType"].astype(str) == str(cancer_type))
        ].copy()

        stat_sub = mi4_program_stats_df.loc[
            (mi4_program_stats_df["Program"].astype(str) == str(program))
            & (mi4_program_stats_df["CancerType"].astype(str) == str(cancer_type))
        ].copy()

        if sub.empty:
            ax.text(
                0.5,
                0.5,
                "No cells",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=7,
                color="#777777",
            )
            ax.set_xticks([0, 1])
            ax.set_xticklabels(GROUP_XTICK_LABELS, rotation=0)
            ax.set_yticks([])

        else:
            sub["MI4_group"] = pd.Categorical(
                sub["MI4_group"].astype(str),
                categories=PLOT_GROUP_ORDER,
                ordered=True,
            )

            group_order_present = _present_group_order(sub)

            # -----------------------------
            # Violin plot
            # -----------------------------
            before_collections = len(ax.collections)

            sns.violinplot(
                data=sub,
                x="MI4_group",
                y="State_Score",
                order=PLOT_GROUP_ORDER,
                palette=GROUP_FILL_COLORS,
                inner=None,
                cut=0,
                linewidth=0.45,
                saturation=1,
                ax=ax,
            )

            violin_collections = ax.collections[before_collections:]
            _style_violin_collections(
                ax=ax,
                violin_collections=violin_collections,
                group_order_present=group_order_present,
            )

            # -----------------------------
            # Boxplot overlay
            # -----------------------------
            before_patch_count = len(ax.patches)

            sns.boxplot(
                data=sub,
                x="MI4_group",
                y="State_Score",
                order=PLOT_GROUP_ORDER,
                palette=GROUP_FILL_COLORS,
                width=BOX_WIDTH,
                showfliers=False,
                showcaps=False,   # remove upper/lower horizontal cap lines
                saturation=1,
                boxprops={
                    "linewidth": 0.55,
                    "alpha": BOX_FACE_ALPHA,
                },
                whiskerprops={
                    "linewidth": 0.55,
                },
                medianprops={
                    "color": MEDIAN_COLOR,
                    "linewidth": 0.90,
                },
                ax=ax,
            )

            _style_boxplot_artists(
                ax=ax,
                before_patch_count=before_patch_count,
                group_order_present=group_order_present,
            )

            # -----------------------------
            # Annotation: p-star + SMD from all cells,
            # not downsampled cells.
            # -----------------------------
            y_values = sub["State_Score"].to_numpy(dtype=float)
            y_values = y_values[np.isfinite(y_values)]

            if y_values.size > 0:
                y_min = np.nanmin(y_values)
                y_max = np.nanmax(y_values)
                y_span = max(y_max - y_min, 1e-6)

                if not stat_sub.empty:
                    stat_row = stat_sub.iloc[0]
                    label = _format_panel_stat_label(stat_row)

                    y_line = y_max + 0.10 * y_span
                    y_text = y_max + 0.14 * y_span
                    y_tick = 0.035 * y_span

                    ax.plot(
                        [0, 1],
                        [y_line, y_line],
                        color=ANNOTATION_COLOR,
                        linewidth=0.45,
                    )
                    ax.plot(
                        [0, 0],
                        [y_line, y_line - y_tick],
                        color=ANNOTATION_COLOR,
                        linewidth=0.45,
                    )
                    ax.plot(
                        [1, 1],
                        [y_line, y_line - y_tick],
                        color=ANNOTATION_COLOR,
                        linewidth=0.45,
                    )

                    ax.text(
                        0.5,
                        y_text,
                        label,
                        ha="center",
                        va="bottom",
                        fontsize=6.6,
                        color=ANNOTATION_COLOR,
                        linespacing=0.95,
                    )

                    ax.set_ylim(y_min - 0.05 * y_span, y_max + 0.32 * y_span)

        # Column titles
        if r == 0:
            ax.set_title(str(cancer_type), fontsize=9, pad=5, color="black")

        # Row labels
        if c == 0:
            ax.set_ylabel(str(program), fontsize=9, color="black")
        else:
            ax.set_ylabel("")

        # X-axis labels only on bottom row
        if r == n_rows - 1:
            ax.set_xticks([0, 1])
            ax.set_xticklabels(GROUP_XTICK_LABELS, fontsize=7, color="black")
        else:
            ax.set_xticks([0, 1])
            ax.set_xticklabels([])

        ax.set_xlabel("")

        # Axis style
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        for spine in ["left", "bottom"]:
            ax.spines[spine].set_linewidth(0.45)
            ax.spines[spine].set_color("black")

        ax.tick_params(axis="both", width=0.45, length=2.0, colors="black")
        ax.grid(False)

fig.supxlabel(
    f"Aggregated Fibroblast→tumor {MI_OI} group",
    fontsize=9,
    y=0.02,
)

fig.supylabel(
    "CancerSEA module score",
    fontsize=9,
    x=0.006,
)

fig.suptitle(
    f"Receiver tumor programs by cell-level Fibroblast→tumor {MI_OI} level",
    fontsize=10,
    y=0.995,
)

plt.tight_layout(rect=[0.025, 0.045, 1.0, 0.965])

plot_stem = (
    f"MI4_fibro_to_tumor_CancerSEA_3x8_violin_boxplot_SMD_"
    f"{MI4_GROUPING_METHOD}_HGSOCstyle_highLow_noCaps"
)

pdf_out = mi4_program_outdir / f"{plot_stem}.pdf"
png_out = mi4_program_outdir / f"{plot_stem}.png"

fig.savefig(pdf_out, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(png_out, dpi=300, bbox_inches="tight", facecolor="white")

print(f"Saved 3 x 8 PDF: {pdf_out}")
print(f"Saved 3 x 8 PNG: {png_out}")

plt.show()

### CCC method comparison

Compare SpiderNet MI-4, COMMOT and scCChain by the high-versus-low SMD of
Invasion, Angiogenesis and Hypoxia scores across eight cancer types. This
section contains the required Python workflow and launches the companion
`ScCChain_Pancancer_runner.jl` directly; the separate CCC notebook is not needed.

Run these cells in order. Reuse the current `processed` bundle and exported
factor matrix. Existing compatible SMD scans bypass CancerSEA recomputation
and baseline execution. When scans are missing, COMMOT runs slice by slice,
and the Julia cell runs scCChain with its existing resume logic. Keep the Julia
runner beside this notebook, or set its path in the settings below. Fresh runs
require Python `commot` and Julia with `ScCChain`, `CSV`, and `DataFrames`.

The `%%fig6d` cells use a separate namespace, so their variables and plotting
settings do not replace the surrounding analysis state. The setup cell must
run first. Tables and the final figure are exposed as `fig6d_results` at the end.
New summaries and figures are written under `Fig6d_in_analysis_V2` inside the
original CCC results directory; existing baseline outputs are reused there.

**Scientific settings retained from the CCC workflow:** incoming `sum`,
within-cancer-type gene z-scores, a ranked upper-half split, positive incoming
SpiderNet values versus all finite baseline values, and baseline feature
selection by mean rank across panels. This differs from the preceding violin
section's global gene z-scores. The previously noted manuscript descriptions
of maximum aggregation and strongest mean coupling remain unresolved; this
integration preserves the implemented choices. NMF-LR is optional in the
settings and remains excluded from the CCC plot.

Caches assume the same data, checkpoint, marker genes and settings that produced
them. If these inputs changed, set `FORCE_RECOMPUTE_METHODS` to the three methods
and rerun the section; regenerate baseline outputs separately if their inputs
changed. A missing requested method stops the comparison rather than yielding
an incomplete figure.


In [ ]:
from pathlib import Path as _Fig6dPath
from IPython.display import display as _fig6d_display
import gc as _fig6d_gc
import os as _fig6d_os
import sys as _fig6d_sys
import tempfile as _fig6d_tempfile
import matplotlib as _fig6d_mpl
import pandas as _fig6d_pd

# Reuse the current processed bundle; do not load a second copy of the atlas.
if "processed" not in globals() or "run_dirs" not in globals():
    raise RuntimeError("Run the processed-data and inference/export sections first.")

fig6d_state = {
    "__name__": "pancancer_fig6d",
    "display": _fig6d_display,
    "processed": processed,
    "_analysis_data_root": _Fig6dPath(DATA_ROOT),
    "_analysis_output_root": _Fig6dPath(OUTPUT_ROOT),
    "_analysis_processed_dir": _Fig6dPath(processed_data_dir),
    "_analysis_run_dir": _Fig6dPath(run_dirs["run_dir"]),
    "_analysis_celltype_col": CELL_CLASS_COL,
    "_analysis_sample_col": SAMPLE_COL,
}


def _run_fig6d_cell(line, cell):
    """Run a visible Python cell in the persistent Fig. 6d namespace."""
    if "processed" not in fig6d_state:
        raise RuntimeError("Run the Fig. 6d setup cell again before rerunning this section.")
    environment = {key: _fig6d_os.environ.get(key) for key in ("TMP", "TEMP", "TMPDIR")}
    tempdir = _fig6d_tempfile.tempdir
    search_path = _fig6d_sys.path.copy()
    try:
        with _fig6d_mpl.rc_context(fig6d_state.get("_plot_style", {})):
            with _fig6d_pd.option_context("display.max_columns", _fig6d_pd.get_option("display.max_columns")):
                try:
                    exec(compile(cell, f"<Fig.6d:{line.strip()}>", "exec"), fig6d_state)
                finally:
                    fig6d_state["_plot_style"] = dict(_fig6d_mpl.rcParams)
    finally:
        for key, value in environment.items():
            if value is None:
                _fig6d_os.environ.pop(key, None)
            else:
                _fig6d_os.environ[key] = value
        _fig6d_tempfile.tempdir = tempdir
        _fig6d_sys.path[:] = search_path
        _fig6d_gc.collect()


get_ipython().register_magic_function(_run_fig6d_cell, magic_kind="cell", magic_name="fig6d")
print("Fig. 6d setup ready. Run the following cells in order.")


In [ ]:
%%fig6d settings
# ============================================================
# 1. User settings
# ============================================================
from pathlib import Path

# ---------------------------------------------------------------------
# Dataset and SpiderNet result paths, matching Pancancer_analysis_V2
# ---------------------------------------------------------------------
DATA_ROOT = _analysis_data_root
OUTPUT_ROOT = _analysis_output_root
PROCESSED_DATA_DIR = _analysis_processed_dir
VERSION = "V1"
DIM_ENVIR = 11
RUN_DIR = _analysis_run_dir

# Optional local SpiderNet package path
SPIDERNET_PROJECT_DIR = Path.cwd()

# ---------------------------------------------------------------------
# Pan-cancer AnnData fields
# ---------------------------------------------------------------------
CELLTYPE_COL = _analysis_celltype_col
SAMPLE_COL = _analysis_sample_col
SENDER_CELLTYPE = "Fibroblast"
RECEIVER_TUMOR_SUFFIX = "-cancercell"

CANCERTYPE_ORDER = [
    "Breast",
    "Colon",
    "Liver",
    "Lung",
    "Melanoma",
    "Ovarian",
    "Prostate",
    "Uterine",
]

# ---------------------------------------------------------------------
# CancerSEA programs
# ---------------------------------------------------------------------
CANCERSEA_PROGRAMS = ["Invasion", "Angiogenesis", "Hypoxia"]
PROGRAM_ORDER = CANCERSEA_PROGRAMS.copy()
CANCERSEA_PATH = DATA_ROOT / "CancerSEA_marker"

# Expression source for CancerSEA module scores.
# Set to a layer name if needed, e.g. "log1p_norm".
EXPRESSION_LAYER = None

# Recommended for within-cancer-type SMD comparison.
# Set to "zscore_mean_global" if you want to match the earlier global-zscore MI-4 cell.
MODULE_SCORE_METHOD = "zscore_mean_within_cancertype"
VALID_MODULE_SCORE_METHODS = {"zscore_mean_within_cancertype", "zscore_mean_global"}

# To save time, set False after the first successful run to reuse the cached CancerSEA table.
RECOMPUTE_CANCERSEA_SCORE_TABLE = True

# Keep only tumor cells with at least one spatial Fibroblast -> tumor edge.
KEEP_ONLY_FIBROBLAST_NEIGHBOR_TUMOR_CELLS = True

# ---------------------------------------------------------------------
# CCC methods to compare
# ---------------------------------------------------------------------
METHODS_TO_RUN = ["SpiderNet", "COMMOT", "ScCChain"]
INCLUDE_SPACIA = False
if INCLUDE_SPACIA and "Spacia" not in METHODS_TO_RUN:
    METHODS_TO_RUN.append("Spacia")

# If True, missing/failed baseline methods are skipped so the notebook can finish with available methods.
# This is useful because some CCC baselines may fail or may not have finished yet.
SKIP_MISSING_BASELINES = False
REQUIRE_METHODS_TO_LOAD = ["SpiderNet"] if not SKIP_MISSING_BASELINES else []

# If True, a method-level scan error is recorded and the notebook continues with other methods.
# Set False for strict debugging.
CONTINUE_ON_METHOD_FAILURE = False

# Baseline result directories.
# COMMOT and ScCChain outputs generated by this notebook are stored under OUT_DIR/baseline_CCC_outputs.
# These variables are reassigned after OUT_DIR is created below.
COMMOT_PATH_MAIN = OUTPUT_ROOT / "COMMOT"
SC_CCHAIN_PATH_MAIN = OUTPUT_ROOT / "ScCChain"
SPACIA_PATH_MAIN = OUTPUT_ROOT / "Spacia"
NMF_LR_PATH_MAIN = OUTPUT_ROOT / "NMF-LR"

# NMF-LR: first try precomputed factors. If unavailable, recompute with an out-of-core MiniBatchNMF backend.
NMF_PRECOMPUTED_CANDIDATES = [
    NMF_LR_PATH_MAIN / "Factor_LR_list.pkl",
    NMF_LR_PATH_MAIN / "Factor_LR_use.npy",
    RUN_DIR / "Factor_LR_list.pkl",
    RUN_DIR / "Factor_LR_use.npy",
]
ALLOW_RECOMPUTE_NMF_LR = True

# "minibatch" is recommended for this pan-cancer run because the full dense LR matrix is too large.
# Set to "dense" only if you explicitly want the original all-at-once sklearn NMF.
NMF_RECOMPUTE_BACKEND = "minibatch"
NMF_RANDOM_STATE = 0
NMF_MAX_ITER = 1000
NMF_RANK = DIM_ENVIR

# MiniBatchNMF settings. With 114 LR features, 100k rows is usually memory-safe.
NMF_MINIBATCH_BATCH_SIZE = 100_000
NMF_MINIBATCH_TRANSFORM_BATCH_SIZE = 200_000
NMF_MINIBATCH_EPOCHS = 3
NMF_MINIBATCH_INIT = "nndsvda"
NMF_MINIBATCH_FORGET_FACTOR = 0.7
NMF_MINIBATCH_MAX_NO_IMPROVEMENT = 20
NMF_MINIBATCH_TRANSFORM_MAX_ITER = 200
REUSE_NMF_LR_FACTOR_CACHE = True

# ---------------------------------------------------------------------
# SpiderNet fixed MI for selected summary and final barplot
# ---------------------------------------------------------------------
# For this pan-cancer Fibroblast -> tumor analysis, SpiderNet is explicitly fixed to MI-4.
SPIDERNET_FIXED_DIM_ONE_BASED = 4
SPIDERNET_FIXED_FEATURE_NAME = f"MI-{SPIDERNET_FIXED_DIM_ONE_BASED}"

# If True, only MI-4 is loaded/scanned for SpiderNet, instead of scanning all MIs and selecting MI-4 later.
# This saves memory and makes the SpiderNet comparison explicitly MI-4-specific.
SPIDERNET_ONLY_FIXED_MI = True

# ---------------------------------------------------------------------
# Grouping and SMD settings
# ---------------------------------------------------------------------
# For Fibroblast -> tumor, tumor cells are receivers, so the grouping strength is incoming.
GROUPING_SOURCE = "incoming_Fibroblast_to_tumor_aggregate"
INCOMING_AGG = "sum"   # options: "sum", "mean", "max"

SPLIT_RULE = "median_rank_split_upper_half"  # robust to ties
MIN_CELLS_PER_GROUP = 20
TEST_ALTERNATIVE = "greater"  # high > low

# Evaluation filters:
# - SpiderNet MI-4 uses positive incoming signal.
# - Baselines use all finite tumor-cell values by default, avoiding sparse baselines being dropped.
SPIDERNET_EVAL_FILTER = "positive_incoming"  # "positive_incoming" or "all_finite"
BASELINE_EVAL_FILTER = "all_finite"          # "positive_incoming" or "all_finite"

# Baseline feature selection for the final barplot.
# Options:
#   "mean_rank_across_all_panels": rank axes within each Method x Program x CancerType, then select lowest mean rank.
#   "mean_smd_across_all_panels": select axis with largest mean SMD across Program x CancerType.
BASELINE_FEATURE_SELECTION_MODE = "mean_rank_across_all_panels"
REQUIRE_ALL_PANELS_FOR_FEATURE_SELECTION = False

# ---------------------------------------------------------------------
# Outputs and method-level cache/checkpoint settings
# ---------------------------------------------------------------------
SOURCE_CCC_DIR = RUN_DIR / "Pancancer_Fibroblast_to_tumor_CCC_method_CancerSEA_SMD_comparison"
OUT_DIR = SOURCE_CCC_DIR / "Fig6d_in_analysis_V2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Cache path for out-of-core NMF-LR factors generated by MiniBatchNMF.
# It is prepended to the candidate list so future runs load it directly instead of refitting.
NMF_LR_BATCH_CACHE_NPY = OUT_DIR / "Factor_LR_use_recomputed_minibatch.npy"
NMF_LR_BATCH_COMPONENTS_NPY = OUT_DIR / "Factor_LR_components_recomputed_minibatch.npy"
NMF_LR_BATCH_META_JSON = OUT_DIR / "Factor_LR_use_recomputed_minibatch_meta.json"
NMF_PRECOMPUTED_CANDIDATES = [NMF_LR_BATCH_CACHE_NPY] + [
    p for p in NMF_PRECOMPUTED_CANDIDATES
    if Path(p) != NMF_LR_BATCH_CACHE_NPY
]

# Method-level cache: each method saves its all-axis SMD scan immediately after finishing.
# This prevents re-running completed methods if a later method fails.
METHOD_CACHE_DIR = OUT_DIR / "per_method_cache"
METHOD_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# COMMOT and ScCChain baseline-output locations
# ---------------------------------------------------------------------
BASELINE_CCC_OUTDIR = SOURCE_CCC_DIR / "baseline_CCC_outputs"
COMMOT_OUTDIR = BASELINE_CCC_OUTDIR / "COMMOT"
SCCCHAIN_INPUT_DIR = BASELINE_CCC_OUTDIR / "ScCChain_input_h5ad"
SCCCHAIN_OUTDIR = BASELINE_CCC_OUTDIR / "ScCChain"
SCCCHAIN_MANIFEST_PATH = BASELINE_CCC_OUTDIR / "ScCChain_input_manifest.csv"
SCCCHAIN_LR_DB_PATH = BASELINE_CCC_OUTDIR / "Pancancer_lr_db_for_scCChain.csv"
COMMOT_SAMPLE_MANIFEST_PATH = COMMOT_OUTDIR / "COMMOT_sample_manifest.csv"

# Make Cell 6 read the outputs generated by the new COMMOT / ScCChain cells.
COMMOT_PATH_MAIN = COMMOT_OUTDIR
SC_CCHAIN_PATH_MAIN = SCCCHAIN_OUTDIR

for _p in [BASELINE_CCC_OUTDIR, COMMOT_OUTDIR, SCCCHAIN_INPUT_DIR, SCCCHAIN_OUTDIR]:
    _p.mkdir(parents=True, exist_ok=True)

# COMMOT can be slow. If outputs already exist, this cell skips them.
RUN_COMMOT_IF_MISSING = True
COMMOT_COT_NITERMAX = 2000
COMMOT_NORMALIZE_EDGE_SCORES = True
COMMOT_CONTINUE_ON_SAMPLE_FAILURE = True

# ScCChain settings for the paired Julia runner.
SCCCHAIN_N_PROGRAMS = DIM_ENVIR
SCCCHAIN_KNN_K = 10
SCCCHAIN_ALPHA = 0.00002
SCCCHAIN_SEED = 42

# Reuse finished method-level scans when available.
REUSE_METHOD_SMD_CACHE = True

# Set True to ignore all existing method-level caches and recompute everything.
OVERWRITE_METHOD_SMD_CACHE = False

# Recompute only selected methods, e.g. ["COMMOT"] or ["NMF-LR", "ScCChain"].
FORCE_RECOMPUTE_METHODS = []

# If a previous run only produced the old combined all-method CSV, allow extracting one method from it.
# For new runs, the per-method cache is preferred and is safer for resume-after-failure.
ALLOW_LEGACY_COMBINED_CACHE = True

# Cache is keyed by core settings that change the numerical SMD table.
METHOD_CACHE_TAG = "__".join([
    MODULE_SCORE_METHOD,
    f"fibNeighbor{int(KEEP_ONLY_FIBROBLAST_NEIGHBOR_TUMOR_CELLS)}",
    f"agg-{INCOMING_AGG}",
    f"split-{SPLIT_RULE}",
    f"spiderMI{SPIDERNET_FIXED_DIM_ONE_BASED}",
])

print("DATA_ROOT:", DATA_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("RUN_DIR:", RUN_DIR)
print("OUT_DIR:", OUT_DIR)
print("METHODS_TO_RUN:", METHODS_TO_RUN)
print("NMF_RECOMPUTE_BACKEND:", NMF_RECOMPUTE_BACKEND)
print("ALLOW_RECOMPUTE_NMF_LR:", ALLOW_RECOMPUTE_NMF_LR)
print("NMF_LR_BATCH_CACHE_NPY:", NMF_LR_BATCH_CACHE_NPY)
print("SPIDERNET_FIXED_FEATURE_NAME:", SPIDERNET_FIXED_FEATURE_NAME)
print("SPIDERNET_ONLY_FIXED_MI:", SPIDERNET_ONLY_FIXED_MI)
print("GROUPING_SOURCE:", GROUPING_SOURCE)
print("MODULE_SCORE_METHOD:", MODULE_SCORE_METHOD)
print("METHOD_CACHE_DIR:", METHOD_CACHE_DIR)
print("BASELINE_CCC_OUTDIR:", BASELINE_CCC_OUTDIR)
print("COMMOT_OUTDIR:", COMMOT_OUTDIR)
print("SCCCHAIN_INPUT_DIR:", SCCCHAIN_INPUT_DIR)
print("SCCCHAIN_OUTDIR:", SCCCHAIN_OUTDIR)
print("REUSE_METHOD_SMD_CACHE:", REUSE_METHOD_SMD_CACHE)
print("FORCE_RECOMPUTE_METHODS:", FORCE_RECOMPUTE_METHODS)

# Julia is launched only when scCChain needs computation and outputs are missing.
JULIA_EXECUTABLE = "julia"  # An absolute executable path is also accepted.
JULIA_PROJECT = None       # Set to an existing Julia project directory if needed.
SCCCHAIN_RUNNER_PATH = None  # Auto-locate ScCChain_Pancancer_runner.jl.
RUN_SCCCHAIN_IF_MISSING = True
SCCCHAIN_OVERWRITE = False
SCCCHAIN_FORCE_LAUNCH = False  # Usually unnecessary; preserves runner resume checks.


In [ ]:
%%fig6d imports
# ============================================================
# 2. Imports and environment setup
# ============================================================
import os
import sys
import gc
import json
import pickle
import warnings
import inspect
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy import sparse
from scipy.stats import mannwhitneyu
from sklearn.decomposition import NMF
try:
    from sklearn.decomposition import MiniBatchNMF
except Exception as e:
    MiniBatchNMF = None
    print("[Warning] MiniBatchNMF is unavailable; batch NMF-LR recomputation will not work:", e)

import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    import torch
except Exception as e:
    torch = None
    print("[Warning] torch is unavailable:", e)

if SPIDERNET_PROJECT_DIR.exists() and str(SPIDERNET_PROJECT_DIR) not in sys.path:
    sys.path.append(str(SPIDERNET_PROJECT_DIR))

try:
    from SpiderNet.io import load_processed_data, spidernet_pyg_list_exists
except Exception as e:
    load_processed_data = None
    spidernet_pyg_list_exists = None
    print("[Warning] Could not import SpiderNet.io helpers. Will fall back to pickle files where possible.")
    print(e)

pd.set_option("display.max_columns", 200)

plt.close("all")
plt.style.use("default")
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})


In [ ]:
%%fig6d helpers
# ============================================================
# 3. Helper functions
# ============================================================

def p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def to_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if sp.issparse(x):
        return x.toarray()
    return np.asarray(x)


def get_field(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    return obj[key]


def edge_index_to_e2(edge_index_obj):
    arr = to_numpy(edge_index_obj).astype(np.int64, copy=False)
    if arr.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {arr.shape}")
    if arr.shape[1] == 2:
        return arr
    if arr.shape[0] == 2:
        return arr.T.astype(np.int64, copy=False)
    raise ValueError(f"edge_index should have shape [E, 2] or [2, E], got {arr.shape}")


def get_edge_count(data_obj):
    return int(edge_index_to_e2(get_field(data_obj, "edge_index")).shape[0])


def get_num_cells_from_data(data_obj, adata):
    try:
        return int(get_field(data_obj, "x").shape[0])
    except Exception:
        return int(adata.n_obs)


def get_celltypes(adata, data_obj=None):
    if CELLTYPE_COL in adata.obs.columns:
        return adata.obs[CELLTYPE_COL].astype(str).to_numpy()
    if data_obj is not None:
        for key in ["cell_class", "celltype", CELLTYPE_COL]:
            try:
                return to_numpy(get_field(data_obj, key)).astype(str)
            except Exception:
                pass
    raise KeyError(f"Cannot find cell-type labels. Expected adata.obs[{CELLTYPE_COL!r}].")


def get_sample_id(adata, sample_index):
    candidates = [SAMPLE_COL, "SampleID", "sample_id", "sample", "samples", "Sample", "slice", "slide", "library_id"]
    for col in candidates:
        if col in adata.obs.columns:
            vals = pd.Series(adata.obs[col]).dropna().astype(str).str.strip().unique()
            vals = [v for v in vals if v != "" and v.lower() != "nan"]
            if len(vals) > 0:
                return str(vals[0])
    return f"sample_{sample_index}"


def extract_cancer_type_from_celltype(celltype):
    s = str(celltype)
    if RECEIVER_TUMOR_SUFFIX not in s:
        return None
    return s.replace(RECEIVER_TUMOR_SUFFIX, "").strip(" -_")


def get_barcode_values(adata, cell_indices):
    for col in ["barcode", "Barcode", "cell_id", "cell", "CellID"]:
        if col in adata.obs.columns:
            return adata.obs.iloc[cell_indices][col].astype(str).to_numpy()
    return adata.obs_names[cell_indices].astype(str).to_numpy()


def get_expr_matrix(adata):
    if EXPRESSION_LAYER is not None:
        if EXPRESSION_LAYER not in adata.layers:
            raise KeyError(f"EXPRESSION_LAYER={EXPRESSION_LAYER!r} not found in adata.layers.")
        return adata.layers[EXPRESSION_LAYER]
    return adata.X


def expr_rows_cols_to_numpy(adata, rows, cols):
    X = get_expr_matrix(adata)
    sub = X[rows, :][:, cols]
    if sp.issparse(sub):
        sub = sub.toarray()
    else:
        sub = np.asarray(sub)
    if sub.ndim == 1:
        sub = sub.reshape(-1, 1)
    return sub.astype(np.float32, copy=False)


def resolve_gene_indices(var_names, genes):
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_idx = {str(g).upper(): i for i, g in enumerate(var_names)}
    idx = []
    used = []
    for g in genes:
        g = str(g).strip()
        if len(g) == 0:
            continue
        if g in var_names:
            idx_cur = int(var_names.get_loc(g))
        else:
            idx_cur = upper_to_idx.get(g.upper(), None)
        if idx_cur is not None:
            idx.append(int(idx_cur))
            used.append(str(var_names[idx_cur]))
    seen = set()
    idx_unique = []
    used_unique = []
    for i, g in zip(idx, used):
        if i not in seen:
            seen.add(i)
            idx_unique.append(i)
            used_unique.append(g)
    return np.asarray(idx_unique, dtype=np.int64), used_unique


def load_cancersea_gene_sets(programs, cancersea_path, var_names):
    gene_sets = {}
    for program in programs:
        path = Path(cancersea_path) / f"{program}.txt"
        if not path.exists():
            raise FileNotFoundError(f"Cannot find CancerSEA marker file: {path}")
        df = pd.read_csv(path, sep="\t", header=0)
        if "GeneName" not in df.columns:
            raise KeyError(f"{path} does not contain a 'GeneName' column.")
        genes_raw = (
            df["GeneName"].dropna().astype(str).str.strip()
            .loc[lambda x: x.ne("")].drop_duplicates().tolist()
        )
        gene_idx, genes_used = resolve_gene_indices(var_names, genes_raw)
        if len(gene_idx) == 0:
            raise ValueError(f"No genes from CancerSEA {program} are found in adata.var_names.")
        gene_sets[program] = {"genes_raw": genes_raw, "gene_idx": gene_idx, "genes_used": genes_used}
    return gene_sets


def build_gene_reference_from_sums(gene_sum, gene_sumsq, gene_n):
    if gene_n <= 0:
        raise ValueError("Cannot build gene reference with gene_n <= 0.")
    mean = gene_sum / gene_n
    var = (gene_sumsq / gene_n) - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-8))
    std[~np.isfinite(std)] = 1.0
    std[std < 1e-6] = 1.0
    return {"mean": mean.astype(np.float32), "std": std.astype(np.float32), "n_ref": int(gene_n)}


def high_mask_from_split_rule(values, split_rule=SPLIT_RULE):
    values = np.asarray(values, dtype=float)
    high_mask = np.zeros(values.shape[0], dtype=bool)
    finite_mask = np.isfinite(values)
    if not np.any(finite_mask):
        return high_mask
    finite_idx = np.where(finite_mask)[0]
    vals = values[finite_idx]
    if split_rule == "median_rank_split_upper_half":
        order = np.lexsort((finite_idx, vals))
        high_rel = order[len(order) // 2:]
        high_mask[finite_idx[high_rel]] = True
        return high_mask
    if split_rule == "median_ge":
        med = np.nanmedian(vals)
        high_mask[finite_idx] = vals >= med
        return high_mask
    raise ValueError(f"Unknown split_rule={split_rule}")


def build_high_low_masks(values_all, eval_mask, split_rule=SPLIT_RULE):
    values_all = np.asarray(values_all, dtype=float)
    eval_mask = np.asarray(eval_mask, dtype=bool)
    high = np.zeros(values_all.shape[0], dtype=bool)
    low = np.zeros(values_all.shape[0], dtype=bool)
    vals_eval = values_all[eval_mask]
    if vals_eval.size == 0:
        return high, low, np.nan
    high_rel = high_mask_from_split_rule(vals_eval, split_rule=split_rule)
    eval_idx = np.where(eval_mask)[0]
    high[eval_idx[high_rel]] = True
    low[eval_idx[~high_rel]] = True
    return high, low, float(np.nanmedian(vals_eval))


def compute_smd_from_groups(high_scores, low_scores, min_cells_per_group=MIN_CELLS_PER_GROUP):
    high = pd.to_numeric(pd.Series(high_scores), errors="coerce").dropna().to_numpy(dtype=float)
    low = pd.to_numeric(pd.Series(low_scores), errors="coerce").dropna().to_numpy(dtype=float)
    high = high[np.isfinite(high)]
    low = low[np.isfinite(low)]
    out = {
        "n_high": int(len(high)),
        "n_low": int(len(low)),
        "mean_high": np.nan,
        "mean_low": np.nan,
        "mean_diff_high_minus_low": np.nan,
        "SMD_high_minus_low": np.nan,
        "p_value": np.nan,
        "p_star": "NA",
    }
    if len(high) == 0 or len(low) == 0:
        return out
    out["mean_high"] = float(np.mean(high))
    out["mean_low"] = float(np.mean(low))
    out["mean_diff_high_minus_low"] = float(out["mean_high"] - out["mean_low"])
    if len(high) < min_cells_per_group or len(low) < min_cells_per_group:
        return out
    sd_high = np.std(high, ddof=1) if len(high) > 1 else np.nan
    sd_low = np.std(low, ddof=1) if len(low) > 1 else np.nan
    if len(high) > 1 and len(low) > 1 and np.isfinite(sd_high) and np.isfinite(sd_low):
        pooled = np.sqrt(((len(high) - 1) * sd_high ** 2 + (len(low) - 1) * sd_low ** 2) / (len(high) + len(low) - 2))
        if np.isfinite(pooled) and pooled > 0:
            out["SMD_high_minus_low"] = float((out["mean_high"] - out["mean_low"]) / pooled)
    try:
        out["p_value"] = float(mannwhitneyu(high, low, alternative=TEST_ALTERNATIVE, method="asymptotic").pvalue)
    except TypeError:
        out["p_value"] = float(mannwhitneyu(high, low, alternative=TEST_ALTERNATIVE).pvalue)
    out["p_star"] = p_to_star(out["p_value"])
    return out


def edge_matrix_from_square(square_mat, rows, cols):
    if sp.issparse(square_mat):
        square_mat = square_mat.tocsr()
        return square_mat[rows, cols].A1.astype(np.float32, copy=False)
    return np.asarray(square_mat)[rows, cols].astype(np.float32, copy=False)


def clean_nonnegative_matrix_for_nmf(X):
    if torch is not None and torch.is_tensor(X):
        X = X.detach().cpu().numpy()
    if sp.issparse(X):
        X = X.tocsr(copy=True)
        X.data = np.nan_to_num(X.data, nan=0.0, posinf=0.0, neginf=0.0)
        X.data[X.data < 0] = 0.0
        return X
    X = np.asarray(X, dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[X < 0] = 0.0
    return X


def aggregate_incoming_for_selected_receivers(edge_features, edge_index_e2, celltypes, selected_cell_indices, n_cells, agg=INCOMING_AGG):
    """Aggregate Fibroblast -> tumor edge features into selected tumor receiver cells."""
    edge_features = np.asarray(edge_features, dtype=np.float32)
    edge_index_e2 = np.asarray(edge_index_e2, dtype=np.int64)
    src = edge_index_e2[:, 0]
    dst = edge_index_e2[:, 1]
    valid = (src >= 0) & (src < n_cells) & (dst >= 0) & (dst < n_cells)
    src = src[valid]
    dst = dst[valid]
    ef = edge_features[valid, :]

    sender_ct = np.asarray(celltypes, dtype=str)[src]
    receiver_ct = np.asarray(celltypes, dtype=str)[dst]
    edge_mask = (sender_ct == SENDER_CELLTYPE) & (np.char.find(receiver_ct.astype(str), RECEIVER_TUMOR_SUFFIX) >= 0)

    K = ef.shape[1]
    incoming = np.zeros((n_cells, K), dtype=np.float32)
    if np.any(edge_mask):
        dst_use = dst[edge_mask]
        ef_use = ef[edge_mask, :]
        if agg in ["sum", "mean"]:
            np.add.at(incoming, dst_use, ef_use)
            if agg == "mean":
                count = np.zeros((n_cells, K), dtype=np.float32)
                np.add.at(count, dst_use, (ef_use > 0).astype(np.float32))
                incoming = np.divide(incoming, count, out=np.zeros_like(incoming), where=count > 0)
        elif agg == "max":
            np.maximum.at(incoming, dst_use, ef_use)
        else:
            raise ValueError(f"Unknown INCOMING_AGG={agg}")
    return incoming[np.asarray(selected_cell_indices, dtype=np.int64), :]


In [ ]:
%%fig6d metadata
# Reuse the already-loaded graph and AnnData objects in their original order.
adata_list = processed.adata_list
spidernet_data_list = processed.spidernet_data
if not adata_list or len(adata_list) != len(spidernet_data_list):
    raise ValueError("Processed AnnData and graph lists must be nonempty and aligned.")
edge_counts = np.asarray([get_edge_count(d) for d in spidernet_data_list], dtype=np.int64)
edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))
cell_offsets = np.concatenate(([0], np.cumsum([ad.n_obs for ad in adata_list])))
factor_path = RUN_DIR / "Factor_envir_use.npy"
if not factor_path.exists():
    raise FileNotFoundError(f"Export the current SpiderNet inference results first: {factor_path}")
factor_preview = np.load(factor_path, mmap_mode="r")
if factor_preview.ndim != 2 or factor_preview.shape[0] != int(edge_offsets[-1]):
    raise ValueError("The factor matrix is not aligned with the current processed edge order.")
if factor_preview.shape[1] < SPIDERNET_FIXED_DIM_ONE_BASED:
    raise ValueError("The factor matrix does not contain MI-4.")
del factor_preview
print("Sub-slices:", len(adata_list))
print("Cells:", f"{int(cell_offsets[-1]):,}")
print("Directed edges:", f"{int(edge_offsets[-1]):,}")


In [ ]:
%%fig6d source-functions
# ============================================================
# 6. Prepare CCC method edge-feature sources, with method-level cache checks
# ============================================================
# Each source returns E x K edge-feature matrices aligned to the SpiderNet edge_index for a given sample_index.
# If a method-level SMD scan already exists, that method is not prepared again.

sample_info_df = pd.DataFrame({
    "sample_index": np.arange(len(adata_list), dtype=int),
    "sample_id": [get_sample_id(adata_list[i], i) for i in range(len(adata_list))],
    "n_obs": [adata_list[i].n_obs for i in range(len(adata_list))],
    "n_edges": edge_counts,
})

# Be robust when this cell is re-run independently from Cell 1.
OUT_DIR.mkdir(parents=True, exist_ok=True)
METHOD_CACHE_DIR.mkdir(parents=True, exist_ok=True)



def _safe_token(x):
    x = str(x)
    for ch in ["/", "\\", " ", ":", ";", "|", "*", "?", "\"", "<", ">"]:
        x = x.replace(ch, "_")
    x = x.replace("→", "to").replace("-", "_")
    return x


def method_cache_paths(method):
    safe_method = _safe_token(method)
    safe_tag = _safe_token(METHOD_CACHE_TAG)
    prefix = METHOD_CACHE_DIR / f"{safe_method}__{safe_tag}"
    return {
        "scan": Path(str(prefix) + "__SMD_scan.csv"),
        "strength": Path(str(prefix) + "__incoming_strength_summary.csv"),
        "meta": Path(str(prefix) + "__meta.json"),
    }


def _validate_scan_cache(scan_df, method):
    required = {
        "Method", "Feature_Dim", "Feature_Name", "CancerType", "Program", "SMD",
        "Grouping_Source", "Incoming_Agg", "module_score_method",
    }
    missing = required.difference(scan_df.columns)
    if missing:
        return False, f"missing columns: {sorted(missing)}"
    if scan_df.empty:
        return False, "empty scan table"
    methods = set(scan_df["Method"].astype(str))
    if methods != {str(method)}:
        return False, f"method mismatch: {methods}"
    programs_found = set(scan_df["Program"].astype(str))
    if not set(PROGRAM_ORDER).issubset(programs_found):
        return False, f"programs missing: {sorted(set(PROGRAM_ORDER) - programs_found)}"
    if method == "SpiderNet":
        if SPIDERNET_FIXED_FEATURE_NAME not in set(scan_df["Feature_Name"].astype(str)):
            return False, f"SpiderNet cache does not contain {SPIDERNET_FIXED_FEATURE_NAME}"
    return True, "ok"


def try_load_cached_method_scan(method):
    if not REUSE_METHOD_SMD_CACHE:
        return None, None, "cache disabled"
    if OVERWRITE_METHOD_SMD_CACHE:
        return None, None, "overwrite requested"
    if method in set(FORCE_RECOMPUTE_METHODS):
        return None, None, f"{method} is in FORCE_RECOMPUTE_METHODS"

    paths = method_cache_paths(method)
    if paths["scan"].exists():
        scan_df = pd.read_csv(paths["scan"])
        ok, reason = _validate_scan_cache(scan_df, method)
        if ok:
            strength_df = pd.read_csv(paths["strength"]) if paths["strength"].exists() else pd.DataFrame()
            print(f"[{method}] Loaded method-level cached SMD scan: {paths['scan']}")
            return scan_df, strength_df, "method-level cache"
        print(f"[{method}] Ignoring method-level cache: {reason}")

    # Backward compatibility: if an earlier run only wrote the combined all-method CSV, extract this method from it.
    legacy_scan = OUT_DIR / "Pancancer_CCC_all_axis_Fibroblast_to_tumor_CancerSEA_SMD_scan.csv"
    legacy_strength = OUT_DIR / "Pancancer_CCC_all_axis_incoming_strength_summary.csv"
    if ALLOW_LEGACY_COMBINED_CACHE and legacy_scan.exists():
        legacy_df = pd.read_csv(legacy_scan)
        if "Method" in legacy_df.columns:
            scan_df = legacy_df.loc[legacy_df["Method"].astype(str).eq(str(method))].copy()
            ok, reason = _validate_scan_cache(scan_df, method)
            if ok:
                strength_df = pd.DataFrame()
                if legacy_strength.exists():
                    tmp = pd.read_csv(legacy_strength)
                    if "Method" in tmp.columns:
                        strength_df = tmp.loc[tmp["Method"].astype(str).eq(str(method))].copy()
                print(f"[{method}] Loaded cached scan from legacy combined CSV: {legacy_scan}")
                return scan_df, strength_df, "legacy combined cache"
            print(f"[{method}] Legacy combined cache not usable: {reason}")

    return None, None, "no usable cache"


def save_method_scan_cache(method, scan_df, strength_summary_df):
    paths = method_cache_paths(method)

    # Ensure cache/output directories exist immediately before writing.
    # This avoids FileNotFoundError if the notebook was resumed from a later cell
    # or if the per_method_cache directory was removed after Cell 1.
    METHOD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for p in paths.values():
        Path(p).parent.mkdir(parents=True, exist_ok=True)

    scan_df.to_csv(paths["scan"], index=False)
    strength_summary_df.to_csv(paths["strength"], index=False)
    meta = {
        "method": method,
        "cache_tag": METHOD_CACHE_TAG,
        "module_score_method": MODULE_SCORE_METHOD,
        "grouping_source": GROUPING_SOURCE,
        "incoming_agg": INCOMING_AGG,
        "split_rule": SPLIT_RULE,
        "spidernet_fixed_feature_name": SPIDERNET_FIXED_FEATURE_NAME,
        "spidernet_only_fixed_mi": bool(SPIDERNET_ONLY_FIXED_MI),
        "program_order": list(PROGRAM_ORDER),
        "cancertype_order": list(CANCERTYPE_ORDER),
        "n_scan_rows": int(scan_df.shape[0]),
        "n_strength_rows": int(strength_summary_df.shape[0]),
    }
    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f"[{method}] Saved method-level cache: {paths['scan']}")


def factor_envir_path_candidates():
    return [
        RUN_DIR / "Factor_envir_use.npy",
        RUN_DIR / "Factor_envir_list.pkl",
    ]


def prepare_spidernet_source():
    npy_path = RUN_DIR / "Factor_envir_use.npy"
    pkl_path = RUN_DIR / "Factor_envir_list.pkl"
    if npy_path.exists():
        factor = np.load(npy_path, mmap_mode="r")
        total_edges = int(edge_offsets[-1])
        if factor.shape[0] != total_edges:
            if factor.ndim == 2 and factor.shape[1] == total_edges:
                raise ValueError(
                    f"{npy_path} appears to be MI x edge. Save it as edge x MI before running this notebook."
                )
            raise ValueError(f"Factor_envir_use shape {factor.shape} is not aligned with total_edges={total_edges}")
        K = int(factor.shape[1])
        names = [f"MI-{j+1}" for j in range(K)]
        def getter(sample_index):
            start = int(edge_offsets[sample_index])
            end = int(edge_offsets[sample_index + 1])
            return np.asarray(factor[start:end, :], dtype=np.float32)
        return names, getter
    if pkl_path.exists():
        factor_list = pd.read_pickle(pkl_path)
        K = int(np.asarray(factor_list[0]).shape[1])
        names = [f"MI-{j+1}" for j in range(K)]
        def getter(sample_index):
            return np.asarray(factor_list[sample_index], dtype=np.float32)
        return names, getter
    raise FileNotFoundError(f"Cannot find SpiderNet factor outputs under {RUN_DIR}")


def restrict_spidernet_to_fixed_mi_if_needed(names, getter):
    if not SPIDERNET_ONLY_FIXED_MI:
        dims = list(range(1, len(names) + 1))
        return names, getter, dims
    fixed_dim = int(SPIDERNET_FIXED_DIM_ONE_BASED)
    fixed_zero = fixed_dim - 1
    if fixed_zero < 0 or fixed_zero >= len(names):
        raise ValueError(f"Requested {SPIDERNET_FIXED_FEATURE_NAME}, but SpiderNet has only {len(names)} dimensions.")
    base_getter = getter
    def fixed_getter(sample_index, base_getter=base_getter, fixed_zero=fixed_zero):
        arr = base_getter(sample_index)
        return np.asarray(arr[:, [fixed_zero]], dtype=np.float32)
    print(f"SpiderNet is restricted to {SPIDERNET_FIXED_FEATURE_NAME} only.")
    return [SPIDERNET_FIXED_FEATURE_NAME], fixed_getter, [fixed_dim]



def _nmflr_source_from_factor_list(factor_list):
    K = int(np.asarray(factor_list[0]).shape[1])
    names = [f"NMF-LR-{j+1}" for j in range(K)]
    def getter(sample_index):
        return np.asarray(factor_list[sample_index], dtype=np.float32)
    return names, getter


def _nmflr_source_from_npy(npy_path):
    factor = np.load(npy_path, mmap_mode="r")
    total_edges = int(edge_offsets[-1])
    if factor.shape[0] != total_edges:
        raise ValueError(f"Precomputed NMF-LR npy shape {factor.shape} is not aligned with total_edges={total_edges}")
    K = int(factor.shape[1])
    names = [f"NMF-LR-{j+1}" for j in range(K)]
    def getter(sample_index):
        start = int(edge_offsets[sample_index])
        end = int(edge_offsets[sample_index + 1])
        return np.asarray(factor[start:end, :], dtype=np.float32)
    return names, getter


def _get_lr_matrix_for_nmflr(sample_index):
    data_obj = spidernet_data_list[sample_index]
    if "cellpair_LRpair_neigh" not in data_obj:
        raise KeyError("processed graph contains no 'cellpair_LRpair_neigh'; NMF-LR cannot be recomputed.")
    return clean_nonnegative_matrix_for_nmf(data_obj["cellpair_LRpair_neigh"])


def _to_dense_float32_nmf_batch(X_batch):
    if sp.issparse(X_batch):
        X_batch = X_batch.toarray()
    else:
        X_batch = np.asarray(X_batch)
    X_batch = np.asarray(X_batch, dtype=np.float32, order="C")
    np.nan_to_num(X_batch, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    X_batch[X_batch < 0] = 0.0
    return X_batch


def _iter_nmf_row_batches(X, batch_size):
    n = int(X.shape[0])
    for start in range(0, n, int(batch_size)):
        end = min(start + int(batch_size), n)
        yield start, end, _to_dense_float32_nmf_batch(X[start:end])


def _make_minibatch_nmf_model():
    if MiniBatchNMF is None:
        raise ImportError(
            "MiniBatchNMF is unavailable in this sklearn installation. "
            "Use scikit-learn >= 1.1 or set NMF_RECOMPUTE_BACKEND='dense'."
        )

    # Filter kwargs for compatibility across sklearn versions.
    kwargs = {
        "n_components": int(NMF_RANK),
        "init": NMF_MINIBATCH_INIT,
        "batch_size": int(NMF_MINIBATCH_BATCH_SIZE),
        "max_iter": int(NMF_MAX_ITER),
        "max_no_improvement": int(NMF_MINIBATCH_MAX_NO_IMPROVEMENT),
        "forget_factor": float(NMF_MINIBATCH_FORGET_FACTOR),
        "transform_max_iter": int(NMF_MINIBATCH_TRANSFORM_MAX_ITER),
        "random_state": int(NMF_RANDOM_STATE),
        "verbose": 0,
    }
    sig = inspect.signature(MiniBatchNMF)
    kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}
    return MiniBatchNMF(**kwargs)


def build_global_nmflr_factors_minibatch():
    """Fit global NMF-LR out-of-core and save edge x NMF factor matrix as a memmap-backed .npy.

    This avoids constructing the full dense LR matrix of shape total_edges x n_LR_features.
    It makes two passes over samples:
      1. partial_fit MiniBatchNMF on dense row batches.
      2. transform dense row batches and write factors directly to disk.
    """
    out_npy = Path(NMF_LR_BATCH_CACHE_NPY)
    comp_npy = Path(NMF_LR_BATCH_COMPONENTS_NPY)
    meta_json = Path(NMF_LR_BATCH_META_JSON)
    out_npy.parent.mkdir(parents=True, exist_ok=True)

    total_edges = int(edge_offsets[-1])
    if REUSE_NMF_LR_FACTOR_CACHE and out_npy.exists():
        factor = np.load(out_npy, mmap_mode="r")
        if factor.shape == (total_edges, int(NMF_RANK)):
            print(f"Reusing existing MiniBatch NMF-LR factor cache: {out_npy}")
            del factor
            return out_npy
        print(f"Ignoring existing MiniBatch NMF-LR cache with incompatible shape {factor.shape}; expected {(total_edges, int(NMF_RANK))}.")
        del factor

    print("Fitting global NMF-LR with MiniBatchNMF")
    print("  total_edges:", total_edges)
    print("  rank:", NMF_RANK)
    print("  batch_size:", NMF_MINIBATCH_BATCH_SIZE)
    print("  epochs:", NMF_MINIBATCH_EPOCHS)

    nmf = _make_minibatch_nmf_model()

    n_batches = 0
    for epoch in range(int(NMF_MINIBATCH_EPOCHS)):
        print(f"[NMF-LR] MiniBatchNMF epoch {epoch + 1}/{int(NMF_MINIBATCH_EPOCHS)}")
        for sample_index in range(len(spidernet_data_list)):
            X_cur = _get_lr_matrix_for_nmflr(sample_index)
            print(f"  fit sample {sample_index + 1}/{len(spidernet_data_list)}: {X_cur.shape}")
            for _, _, X_batch in _iter_nmf_row_batches(X_cur, NMF_MINIBATCH_BATCH_SIZE):
                if X_batch.shape[0] == 0:
                    continue
                # Avoid initializing on a completely empty all-zero batch.
                if n_batches == 0 and not np.any(X_batch > 0):
                    continue
                nmf.partial_fit(X_batch)
                n_batches += 1
            del X_cur
            gc.collect()

    if n_batches == 0:
        raise RuntimeError("MiniBatchNMF saw no non-zero LR batches; cannot fit NMF-LR.")

    np.save(comp_npy, np.asarray(nmf.components_, dtype=np.float32))
    print(f"Saved MiniBatchNMF components: {comp_npy}")

    # Second pass: transform and write W directly to a memmap-backed .npy.
    factor_mem = np.lib.format.open_memmap(
        out_npy, mode="w+", dtype=np.float32, shape=(total_edges, int(NMF_RANK))
    )
    colmax = np.zeros(int(NMF_RANK), dtype=np.float64)

    global_start = 0
    for sample_index in range(len(spidernet_data_list)):
        X_cur = _get_lr_matrix_for_nmflr(sample_index)
        n_rows = int(X_cur.shape[0])
        expected_rows = int(edge_offsets[sample_index + 1] - edge_offsets[sample_index])
        if n_rows != expected_rows:
            raise ValueError(
                f"NMF-LR LR matrix rows for sample {sample_index} are {n_rows}, "
                f"but edge_offsets expect {expected_rows}."
            )
        print(f"  transform sample {sample_index + 1}/{len(spidernet_data_list)}: {X_cur.shape}")
        for local_start, local_end, X_batch in _iter_nmf_row_batches(X_cur, NMF_MINIBATCH_TRANSFORM_BATCH_SIZE):
            W = nmf.transform(X_batch).astype(np.float32, copy=False)
            np.nan_to_num(W, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
            W[W < 0] = 0.0
            factor_mem[global_start + local_start:global_start + local_end, :] = W
            if W.size:
                colmax = np.maximum(colmax, np.nanmax(W, axis=0))
            del W
        global_start += n_rows
        del X_cur
        gc.collect()

    colmax[~np.isfinite(colmax)] = 0.0
    colmax[colmax <= 0] = 1.0

    # Normalize columns in place by global column max, matching the original code.
    for start in range(0, total_edges, int(NMF_MINIBATCH_TRANSFORM_BATCH_SIZE)):
        end = min(start + int(NMF_MINIBATCH_TRANSFORM_BATCH_SIZE), total_edges)
        factor_mem[start:end, :] = (factor_mem[start:end, :] / colmax.reshape(1, -1)).astype(np.float32, copy=False)

    factor_mem.flush()
    del factor_mem

    meta = {
        "backend": "MiniBatchNMF",
        "n_edges": int(total_edges),
        "n_components": int(NMF_RANK),
        "batch_size": int(NMF_MINIBATCH_BATCH_SIZE),
        "transform_batch_size": int(NMF_MINIBATCH_TRANSFORM_BATCH_SIZE),
        "epochs": int(NMF_MINIBATCH_EPOCHS),
        "random_state": int(NMF_RANDOM_STATE),
        "colmax_before_normalization": colmax.tolist(),
        "output_npy": str(out_npy),
        "components_npy": str(comp_npy),
    }
    with open(meta_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved MiniBatch NMF-LR factors: {out_npy}")
    print(f"Saved MiniBatch NMF-LR metadata: {meta_json}")
    return out_npy


def build_global_nmflr_factors_dense():
    """Original all-at-once NMF-LR implementation. This can require very large RAM."""
    lr_mats = []
    local_edge_counts = []
    for sample_index, data_obj in enumerate(spidernet_data_list):
        if "cellpair_LRpair_neigh" not in data_obj:
            raise KeyError("processed graph contains no 'cellpair_LRpair_neigh'; NMF-LR cannot be recomputed.")
        X_cur = clean_nonnegative_matrix_for_nmf(data_obj["cellpair_LRpair_neigh"])
        lr_mats.append(X_cur)
        local_edge_counts.append(X_cur.shape[0])
        print(f"Collected LR coexpression sample {sample_index + 1}/{len(spidernet_data_list)}: {X_cur.shape}")
    if any(sp.issparse(X) for X in lr_mats):
        lr_all = sp.vstack(lr_mats, format="csr")
    else:
        lr_all = np.vstack(lr_mats)
    print("Fitting global dense NMF-LR:", lr_all.shape, "rank=", NMF_RANK)
    nmf = NMF(n_components=NMF_RANK, init="nndsvda", random_state=NMF_RANDOM_STATE, max_iter=NMF_MAX_ITER)
    factor_all = nmf.fit_transform(lr_all)
    colmax = np.max(factor_all, axis=0)
    colmax[colmax <= 0] = 1.0
    factor_all = (factor_all / colmax.reshape(1, -1)).astype(np.float32, copy=False)
    factor_list = []
    start = 0
    for n_edges in local_edge_counts:
        end = start + n_edges
        factor_list.append(factor_all[start:end].copy())
        start = end
    return factor_list


def prepare_nmflr_source():
    existing = next((p for p in NMF_PRECOMPUTED_CANDIDATES if Path(p).exists()), None)
    if existing is not None:
        existing = Path(existing)
        print(f"Loading precomputed NMF-LR factors: {existing}")
        if existing.suffix == ".pkl":
            return _nmflr_source_from_factor_list(pd.read_pickle(existing))
        if existing.suffix == ".npy":
            return _nmflr_source_from_npy(existing)
        raise ValueError(f"Unsupported NMF-LR factor cache suffix: {existing.suffix}")

    if not ALLOW_RECOMPUTE_NMF_LR:
        raise FileNotFoundError(
            "No precomputed NMF-LR factors were found and ALLOW_RECOMPUTE_NMF_LR=False. "
            f"Checked: {[str(p) for p in NMF_PRECOMPUTED_CANDIDATES]}"
        )

    if str(NMF_RECOMPUTE_BACKEND).lower() == "minibatch":
        factor_npy = build_global_nmflr_factors_minibatch()
        return _nmflr_source_from_npy(factor_npy)

    if str(NMF_RECOMPUTE_BACKEND).lower() == "dense":
        factor_list = build_global_nmflr_factors_dense()
        nmf_cache_path = OUT_DIR / "Factor_LR_list_recomputed_for_CCC_SMD.pkl"
        pd.to_pickle(factor_list, nmf_cache_path)
        print(f"Saved recomputed dense NMF-LR factor list: {nmf_cache_path}")
        return _nmflr_source_from_factor_list(factor_list)

    raise ValueError(f"Unknown NMF_RECOMPUTE_BACKEND={NMF_RECOMPUTE_BACKEND!r}. Use 'minibatch' or 'dense'.")


def list_h5ad_files(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Directory does not exist: {path}")
    return sorted(path.rglob("*.h5ad"))


def build_h5ad_match_by_sample_or_nobs(h5ad_files, source_name):
    # Prefer filename containing sample_id; fall back to n_obs matching.
    nobs_cache = {}
    index_map = {}
    used_files = set()
    for _, row in sample_info_df.iterrows():
        sample_index = int(row["sample_index"])
        sample_id = str(row["sample_id"])
        n_obs = int(row["n_obs"])
        candidates = [p for p in h5ad_files if sample_id in p.stem]
        candidates = [p for p in candidates if p not in used_files]
        if len(candidates) == 0:
            candidates = []
            for p in h5ad_files:
                if p in used_files:
                    continue
                if p not in nobs_cache:
                    ad = sc.read_h5ad(p, backed="r")
                    nobs_cache[p] = int(ad.n_obs)
                    del ad
                if nobs_cache[p] == n_obs:
                    candidates.append(p)
        if len(candidates) == 0:
            raise ValueError(f"Cannot match {source_name} h5ad for sample_index={sample_index}, sample_id={sample_id}, n_obs={n_obs}")
        chosen = candidates[0]
        index_map[sample_index] = chosen
        used_files.add(chosen)
    match_df = pd.DataFrame([
        {"sample_index": k, "sample_id": sample_info_df.loc[k, "sample_id"], "matched_path": str(v)}
        for k, v in index_map.items()
    ])
    match_df.to_csv(OUT_DIR / f"{source_name}_matched_h5ad_files.csv", index=False)
    return index_map


def get_commot_pathway_keys(commot_adata):
    keys = list(commot_adata.obsp.keys())
    total_key = "commot-cellchat-total-total"
    pathway_keys = [
        k for k in keys
        if k.startswith("commot-cellchat-") and len(k.split("-")) == 3 and k != total_key
    ]
    return sorted(pathway_keys)



def _safe_colmax_normalize_feature_matrix(X):
    X = np.asarray(X, dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[X < 0] = 0.0
    if X.size == 0:
        return X
    colmax = np.nanmax(X, axis=0)
    colmax[~np.isfinite(colmax)] = 0.0
    colmax[colmax <= 0] = 1.0
    return (X / colmax.reshape(1, -1)).astype(np.float32, copy=False)


# Metadata columns written by Cell 6A/other edge-score exporters.
# Exclude metadata columns from numeric COMMOT pathway features.
# In particular, sample_tag contains string identifiers such as "s000";
# including them in feature_union would prevent conversion to float.
EDGE_SCORE_METADATA_COLS = {
    "sample_index", "sample_id", "sample_name", "sample_tag",
    "batch_index", "local_edge_index", "global_edge_index",
    "sender_index", "receiver_index",
    "status", "message", "elapsed_min", "n_obs", "n_edges",
    "score_csv", "sample_outdir", "legacy_sample_outdir",
    "existing_score_csv", "legacy_score_csv",
}


def _is_edge_score_metadata_col(col):
    s = str(col).strip()
    lower = s.lower()
    if lower.startswith("unnamed:"):
        return True
    return s in EDGE_SCORE_METADATA_COLS or lower in {x.lower() for x in EDGE_SCORE_METADATA_COLS}


def _feature_cols_from_edge_score_csv(csv_path):
    """
    Return numeric edge-score feature columns from an exported baseline CSV.

    This is intentionally conservative:
    - exclude known metadata columns, including sample_tag;
    - keep only columns that are numeric in a small preview.
    """
    preview = pd.read_csv(csv_path, nrows=200)
    feature_cols = []

    for c in preview.columns:
        if _is_edge_score_metadata_col(c):
            continue

        # Avoid treating string metadata as pathway features.
        s = pd.to_numeric(preview[c], errors="coerce")
        n_nonmissing = int(preview[c].notna().sum())
        n_numeric = int(s.notna().sum())

        if n_nonmissing == 0:
            continue
        if n_numeric == n_nonmissing:
            feature_cols.append(c)
        else:
            print(
                f"[edge-score parser] Ignoring non-numeric column in {csv_path}: "
                f"{c!r}; numeric={n_numeric}/{n_nonmissing}"
            )

    return feature_cols


def _numeric_score_values(df, score_cols, csv_path):
    """Convert selected score columns to float32, coercing bad values to 0."""
    if len(score_cols) == 0:
        return np.zeros((df.shape[0], 0), dtype=np.float32)

    vals_df = df.loc[:, score_cols].apply(pd.to_numeric, errors="coerce")
    n_bad = int(vals_df.isna().sum().sum())

    vals = vals_df.to_numpy(dtype=np.float32, copy=True)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)

    if n_bad > 0:
        print(
            f"[edge-score parser] Coerced {n_bad} non-numeric/NA score entries to 0 "
            f"while reading {csv_path}"
        )

    return vals


def _align_edge_score_csv_by_local_edge_index(csv_path, sample_index, feature_union, normalize=False):
    edge_index = edge_index_to_e2(get_field(spidernet_data_list[sample_index], "edge_index"))
    E = int(edge_index.shape[0])
    feature_to_col = {c: i for i, c in enumerate(feature_union)}

    score_df = pd.read_csv(csv_path)
    score_cols = [
        c for c in score_df.columns
        if (c in feature_to_col) and (not _is_edge_score_metadata_col(c))
    ]
    arr = np.zeros((E, len(feature_union)), dtype=np.float32)

    if len(score_cols) == 0:
        return arr

    if "local_edge_index" in score_df.columns:
        score_df["local_edge_index"] = pd.to_numeric(score_df["local_edge_index"], errors="coerce")
        score_df = score_df.dropna(subset=["local_edge_index"]).copy()
        score_df["local_edge_index"] = score_df["local_edge_index"].astype(np.int64)
        score_df = score_df.drop_duplicates("local_edge_index", keep="last")
        score_df = score_df.set_index("local_edge_index")
        score_df = score_df.reindex(np.arange(E, dtype=np.int64))
        vals = _numeric_score_values(score_df, score_cols, csv_path)
    else:
        vals = _numeric_score_values(score_df, score_cols, csv_path)
        if vals.shape[0] != E:
            raise ValueError(
                f"{csv_path}: rows={vals.shape[0]} but sample {sample_index} has E={E}; "
                "local_edge_index is required for alignment."
            )

    vals[vals < 0] = 0.0
    if normalize:
        vals = _safe_colmax_normalize_feature_matrix(vals)
    arr[:, [feature_to_col[c] for c in score_cols]] = vals
    return arr


def _read_commot_sample_manifest():
    manifest_path = globals().get("COMMOT_SAMPLE_MANIFEST_PATH", COMMOT_PATH_MAIN / "COMMOT_sample_manifest.csv")
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Missing COMMOT sample manifest: {manifest_path}. Run Cell 6A to generate COMMOT outputs first."
        )
    df = pd.read_csv(manifest_path)
    if not {"sample_index", "score_csv"}.issubset(df.columns):
        raise ValueError(f"COMMOT manifest must contain sample_index and score_csv columns: {manifest_path}")
    return df


def prepare_commot_source():
    manifest_df = _read_commot_sample_manifest()
    score_path_by_sample = {}
    missing = []
    for sample_index in range(len(adata_list)):
        sub = manifest_df.loc[manifest_df["sample_index"].astype(int).eq(int(sample_index))]
        if sub.empty:
            missing.append((sample_index, "not in manifest"))
            continue
        p = Path(str(sub.iloc[0]["score_csv"]))
        if not p.exists():
            missing.append((sample_index, str(p)))
            continue
        score_path_by_sample[sample_index] = p

    if missing:
        raise FileNotFoundError(
            "COMMOT outputs are incomplete. Run Cell 6A first. "
            f"Missing {len(missing)}/{len(adata_list)} samples; first missing entries: {missing[:10]}"
        )

    feature_union = []
    for sample_index in range(len(adata_list)):
        for c in _feature_cols_from_edge_score_csv(score_path_by_sample[sample_index]):
            if c not in feature_union:
                feature_union.append(c)

    if len(feature_union) == 0:
        raise ValueError("No COMMOT feature columns were found in COMMOT edge-score CSVs.")

    feature_union = sorted(feature_union)
    pd.DataFrame([
        {
            "sample_index": i,
            "sample_id": sample_info_df.loc[i, "sample_id"],
            "score_csv": str(score_path_by_sample[i]),
        }
        for i in range(len(adata_list))
    ]).to_csv(OUT_DIR / "COMMOT_matched_edge_score_files.csv", index=False)

    names = [f"COMMOT::{c}" for c in feature_union]

    def getter(sample_index):
        return _align_edge_score_csv_by_local_edge_index(
            csv_path=score_path_by_sample[sample_index],
            sample_index=sample_index,
            feature_union=feature_union,
            normalize=False,
        )

    return names, getter


def _validate_sccchain_score_csv(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return False, "missing"
    try:
        cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    except Exception as e:
        return False, f"cannot read header: {repr(e)}"
    if len(cols) < 3:
        return False, "fewer than 3 columns"
    if not any(str(c).startswith("program_") for c in cols[2:]):
        return False, "no program_* columns"
    return True, "ok"


def _find_sccchain_csv(sample_result_root):
    sample_result_root = Path(sample_result_root)
    files = sorted(sample_result_root.glob("*_ScCChain_edge_program_scores.csv"))
    for p in files:
        ok, _ = _validate_sccchain_score_csv(p)
        if ok:
            return p
    return None


def _read_sccchain_manifest():
    manifest_path = globals().get("SCCCHAIN_MANIFEST_PATH", OUT_DIR / "baseline_CCC_outputs" / "ScCChain_input_manifest.csv")
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Missing ScCChain manifest: {manifest_path}. Run Cell 6B to export inputs, run the Julia runner, then rerun Cell 6."
        )
    df = pd.read_csv(manifest_path)
    if not {"sample_index", "sample_name"}.issubset(df.columns):
        raise ValueError(f"ScCChain manifest must contain sample_index and sample_name columns: {manifest_path}")
    return df


def _candidate_sccchain_csv_paths(row, result_root):
    """Candidate ScCChain output paths, ordered from preferred short path to legacy path."""
    result_root = Path(result_root)
    candidates = []

    def add_path(x):
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return
        s = str(x).strip()
        if not s or s.lower() in {"nan", "none", "missing", "null"}:
            return
        p = Path(s)
        if p not in candidates:
            candidates.append(p)

    # New resume-safe manifest columns from Cell 6B.
    add_path(row.get("score_csv", None))
    add_path(row.get("existing_score_csv", None))
    add_path(row.get("legacy_score_csv", None))

    sample_index = int(row["sample_index"])
    sample_name = str(row["sample_name"])
    sample_tag = str(row.get("sample_tag", f"s{sample_index:03d}"))

    # New short-path convention.
    add_path(result_root / sample_tag / f"{sample_tag}_ScCChain_edge_program_scores.csv")

    # Legacy long-path convention from the original runner.
    add_path(result_root / sample_name / f"{sample_name}_ScCChain_edge_program_scores.csv")

    # Also allow any matching CSV inside declared output dirs.
    for dir_col in ["sample_outdir", "legacy_sample_outdir"]:
        d = row.get(dir_col, None)
        if d is not None and not (isinstance(d, float) and np.isnan(d)):
            d = Path(str(d))
            if d.exists():
                for p in sorted(d.glob("*_ScCChain_edge_program_scores.csv")):
                    add_path(p)

    return candidates


def prepare_sccchain_source():
    manifest_df = _read_sccchain_manifest()
    result_root = Path(globals().get("SCCCHAIN_OUTDIR", SC_CCHAIN_PATH_MAIN))

    score_path_by_sample = {}
    missing = []
    checked_rows = []

    for sample_index in range(len(adata_list)):
        sub = manifest_df.loc[manifest_df["sample_index"].astype(int).eq(int(sample_index))]
        if sub.empty:
            missing.append((sample_index, "not in manifest"))
            continue

        row = sub.iloc[0]
        candidates = _candidate_sccchain_csv_paths(row, result_root)
        csv_path = None
        candidate_messages = []
        for p in candidates:
            ok, reason = _validate_sccchain_score_csv(p)
            candidate_messages.append(f"{p} => {reason}")
            if ok:
                csv_path = Path(p)
                break

        if csv_path is None:
            sample_name = str(row["sample_name"])
            missing.append((sample_index, sample_name, candidate_messages[:4]))
            continue

        score_path_by_sample[sample_index] = Path(csv_path)
        checked_rows.append({
            "sample_index": sample_index,
            "sample_id": sample_info_df.loc[sample_index, "sample_id"],
            "score_csv": str(csv_path),
            "n_candidates_checked": len(candidates),
        })

    if missing:
        pd.DataFrame(checked_rows).to_csv(OUT_DIR / "ScCChain_matched_edge_score_files_partial.csv", index=False)
        raise FileNotFoundError(
            "ScCChain outputs are incomplete. Run Cell 6B, then the Julia runner, then rerun Cell 6. "
            f"Missing {len(missing)}/{len(adata_list)} samples; first missing entries: {missing[:10]}"
        )

    feature_union = []
    for sample_index in range(len(adata_list)):
        cols = pd.read_csv(score_path_by_sample[sample_index], nrows=0).columns.tolist()
        program_cols = [c for c in cols[2:] if str(c).startswith("program_")]
        for c in program_cols:
            if c not in feature_union:
                feature_union.append(c)

    if len(feature_union) == 0:
        raise ValueError("No ScCChain program columns were found in ScCChain output CSVs.")

    feature_to_col = {c: i for i, c in enumerate(feature_union)}

    pd.DataFrame(checked_rows).to_csv(OUT_DIR / "ScCChain_matched_edge_score_files.csv", index=False)

    names = [f"ScCChain::{c}" for c in feature_union]

    def getter(sample_index):
        csv_path = score_path_by_sample[sample_index]
        score_df = pd.read_csv(csv_path)
        if score_df.shape[1] < 3:
            raise ValueError(f"Unexpected ScCChain output format: {csv_path}")

        sender_col, receiver_col = score_df.columns[:2]
        program_cols = [c for c in score_df.columns[2:] if c in feature_to_col]

        score_df = score_df.rename(columns={sender_col: "sender_index", receiver_col: "receiver_index"}).copy()
        score_df["sender_index"] = pd.to_numeric(score_df["sender_index"], errors="coerce")
        score_df["receiver_index"] = pd.to_numeric(score_df["receiver_index"], errors="coerce")
        score_df = score_df.dropna(subset=["sender_index", "receiver_index"]).copy()
        score_df["sender_index"] = score_df["sender_index"].astype(int)
        score_df["receiver_index"] = score_df["receiver_index"].astype(int)

        # Julia ScCChain usually returns 1-based indices. Convert if it looks 1-based.
        n_cells = int(adata_list[sample_index].n_obs)
        idx_min = score_df[["sender_index", "receiver_index"]].min().min()
        idx_max = score_df[["sender_index", "receiver_index"]].max().max()
        if idx_min >= 1 and idx_max <= n_cells:
            score_df["sender_index"] -= 1
            score_df["receiver_index"] -= 1

        for c in program_cols:
            score_df[c] = pd.to_numeric(score_df[c], errors="coerce")

        edge_index = edge_index_to_e2(get_field(spidernet_data_list[sample_index], "edge_index"))
        E = int(edge_index.shape[0])
        ref_pairs = pd.DataFrame(edge_index, columns=["sender_index", "receiver_index"])
        ref_index = pd.MultiIndex.from_frame(ref_pairs[["sender_index", "receiver_index"]])

        sc_indexed = (
            score_df
            .drop_duplicates(["sender_index", "receiver_index"], keep="first")
            .set_index(["sender_index", "receiver_index"])
        )

        vals = sc_indexed.reindex(ref_index)[program_cols].fillna(0.0).to_numpy(dtype=np.float32, copy=False)
        vals = _safe_colmax_normalize_feature_matrix(vals)

        arr = np.zeros((E, len(feature_union)), dtype=np.float32)
        arr[:, [feature_to_col[c] for c in program_cols]] = vals
        del score_df, sc_indexed, vals
        return arr

    return names, getter


def prepare_spacia_source():
    spacia_dir = SPACIA_PATH_MAIN / "spacia_outputs"
    files = list_h5ad_files(spacia_dir)
    if len(files) == 0:
        raise FileNotFoundError(f"No Spacia .h5ad files found under {spacia_dir}")
    path_map = build_h5ad_match_by_sample_or_nobs(files, "Spacia")
    first = sc.read_h5ad(path_map[0], backed="r")
    del first
    first = sc.read_h5ad(path_map[0])
    n_programs = int(first.obsp["interaction_scores"].shape[2])
    del first
    names = [f"Spacia-{j+1}" for j in range(n_programs)]
    def getter(sample_index):
        ad = sc.read_h5ad(path_map[sample_index])
        edge_index = edge_index_to_e2(get_field(spidernet_data_list[sample_index], "edge_index"))
        rows = edge_index[:, 0].astype(np.int64)
        cols = edge_index[:, 1].astype(np.int64)
        scores = ad.obsp["interaction_scores"]
        arr = np.zeros((edge_index.shape[0], n_programs), dtype=np.float32)
        for j in range(n_programs):
            arr[:, j] = edge_matrix_from_square(scores[:, :, j], rows, cols)
        return arr
    return names, getter



_fig6d_base_cache_loader = try_load_cached_method_scan


In [ ]:
%%fig6d cache-plan
# Reuse compatible original CCC scans without overwriting the original reports.
_fig6d_local_cache_loader = _fig6d_base_cache_loader


def _fig6d_cache_matches_settings(scan, method):
    ok, reason = _validate_scan_cache(scan, method)
    if not ok:
        return False, reason
    expected = {
        "Incoming_Agg": INCOMING_AGG,
        "module_score_method": MODULE_SCORE_METHOD,
        "Grouping_Rule": SPLIT_RULE,
        "Grouping_Source": GROUPING_SOURCE,
        "Eval_Filter": SPIDERNET_EVAL_FILTER if method == "SpiderNet" else BASELINE_EVAL_FILTER,
    }
    for column, value in expected.items():
        if column not in scan or not scan[column].astype(str).eq(str(value)).all():
            return False, f"{column} does not match {value!r}"
    return True, "matching recorded settings"


def try_load_cached_method_scan(method):
    scan, strength, reason = _fig6d_local_cache_loader(method)
    if scan is not None:
        ok, why = _fig6d_cache_matches_settings(scan, method)
        if ok:
            return scan, strength, reason
        print(f"[{method}] Ignoring incompatible local cache: {why}")
    if (not REUSE_METHOD_SMD_CACHE or OVERWRITE_METHOD_SMD_CACHE
            or method in FORCE_RECOMPUTE_METHODS or not ALLOW_LEGACY_COMBINED_CACHE):
        return None, None, "recomputation requested"
    reference = SOURCE_CCC_DIR / "Pancancer_CCC_all_axis_Fibroblast_to_tumor_CancerSEA_SMD_scan.csv"
    if not reference.exists():
        return None, None, "no original CCC scan"
    combined = pd.read_csv(reference)
    if "Method" not in combined:
        return None, None, "original CCC scan has no Method column"
    scan = combined.loc[combined["Method"].astype(str).eq(method)].copy()
    ok, why = _fig6d_cache_matches_settings(scan, method)
    if not ok:
        return None, None, why
    strength_path = SOURCE_CCC_DIR / "Pancancer_CCC_all_axis_incoming_strength_summary.csv"
    strength = pd.DataFrame()
    if strength_path.exists():
        combined_strength = pd.read_csv(strength_path)
        if "Method" in combined_strength:
            strength = combined_strength.loc[combined_strength["Method"].astype(str).eq(method)].copy()
    print(f"[{method}] Reusing original CCC scan: {reference}")
    return scan, strength, "original CCC scan with matching recorded settings"


FIG6D_METHODS_NEED_SCAN = []
fig6d_cache_plan_rows = []
for method in METHODS_TO_RUN:
    scan, strength, reason = try_load_cached_method_scan(method)
    if scan is None:
        FIG6D_METHODS_NEED_SCAN.append(method)
    fig6d_cache_plan_rows.append({"Method": method, "Recompute_SMD": scan is None, "Source": reason})
display(pd.DataFrame(fig6d_cache_plan_rows))
del scan, strength
print("New Fig. 6d reports will be saved under:", OUT_DIR)


#### CancerSEA scores and baseline computation

The following cells skip computation for methods with reusable SMD scans.
CancerSEA scores are computed independently using the CCC settings above;
the preceding MI-4 violin table is not substituted. Existing baseline CSVs
are checked before reuse. The Julia cell waits for completion, displays its
log, and stops on failure. Interrupting that cell also stops its Julia process.


In [ ]:
%%fig6d cancersea
if FIG6D_METHODS_NEED_SCAN:
    # ============================================================
    # 5. Build tumor-cell CancerSEA module-score table
    # ============================================================
    # This table is method-independent. It defines the tumor cells evaluated in all CCC-method SMD scans.

    if MODULE_SCORE_METHOD not in VALID_MODULE_SCORE_METHODS:
        raise ValueError(f"Unsupported MODULE_SCORE_METHOD={MODULE_SCORE_METHOD}. Choose from {sorted(VALID_MODULE_SCORE_METHODS)}")

    base_table_path = OUT_DIR / f"Pancancer_fibroblast_neighbor_tumor_CancerSEA_scores_wide_{MODULE_SCORE_METHOD}.csv"
    genes_used_path = OUT_DIR / "CancerSEA_genes_used_for_panCCC_SMD.csv"
    zref_summary_path = OUT_DIR / f"CancerSEA_zscore_reference_summary_{MODULE_SCORE_METHOD}.csv"

    if (not RECOMPUTE_CANCERSEA_SCORE_TABLE) and base_table_path.exists():
        tumor_cancersea_df = pd.read_csv(base_table_path)
        print(f"Loaded cached CancerSEA table: {base_table_path}")
    else:
        first_adata = adata_list[0]
        cancersea_gene_sets = load_cancersea_gene_sets(CANCERSEA_PROGRAMS, CANCERSEA_PATH, first_adata.var_names)
        all_gene_idx = np.unique(np.concatenate([v["gene_idx"] for v in cancersea_gene_sets.values()])).astype(np.int64)
        gene_idx_to_pos = {int(gidx): pos for pos, gidx in enumerate(all_gene_idx)}
        program_gene_pos = {
            program: np.asarray([gene_idx_to_pos[int(i)] for i in info["gene_idx"]], dtype=np.int64)
            for program, info in cancersea_gene_sets.items()
        }

        print("CancerSEA genes used:")
        for program, info in cancersea_gene_sets.items():
            print(f"  {program}: {len(info['genes_used'])} genes")

        metadata_frames = []
        gene_sum_by_ct = {}
        gene_sumsq_by_ct = {}
        gene_n_by_ct = {}
        gene_sum_global = np.zeros(len(all_gene_idx), dtype=np.float64)
        gene_sumsq_global = np.zeros(len(all_gene_idx), dtype=np.float64)
        gene_n_global = 0

        for sample_index, (adata, data_obj) in enumerate(zip(adata_list, spidernet_data_list)):
            if sample_index % 10 == 0:
                print(f"[CancerSEA first pass] sample {sample_index + 1}/{len(adata_list)}")
            edge_index = edge_index_to_e2(get_field(data_obj, "edge_index"))
            celltypes = get_celltypes(adata, data_obj)
            n_cells = int(adata.n_obs)

            src = edge_index[:, 0]
            dst = edge_index[:, 1]
            sender_ct = celltypes[src].astype(str)
            receiver_ct = celltypes[dst].astype(str)
            fibro_to_tumor_edge = (sender_ct == SENDER_CELLTYPE) & (np.char.find(receiver_ct.astype(str), RECEIVER_TUMOR_SUFFIX) >= 0)
            fibro_to_tumor_edge_count = np.bincount(dst[fibro_to_tumor_edge], minlength=n_cells).astype(np.int32)

            tumor_cell_mask = np.char.find(celltypes.astype(str), RECEIVER_TUMOR_SUFFIX) >= 0
            if KEEP_ONLY_FIBROBLAST_NEIGHBOR_TUMOR_CELLS:
                keep_mask = tumor_cell_mask & (fibro_to_tumor_edge_count > 0)
            else:
                keep_mask = tumor_cell_mask

            cell_idx = np.where(keep_mask)[0].astype(np.int64)
            if len(cell_idx) == 0:
                continue

            celltype_sel = celltypes[cell_idx].astype(str)
            cancer_type_sel = np.asarray([extract_cancer_type_from_celltype(x) for x in celltype_sel], dtype=object)
            valid_ct = pd.notna(cancer_type_sel)
            cell_idx = cell_idx[valid_ct]
            celltype_sel = celltype_sel[valid_ct]
            cancer_type_sel = cancer_type_sel[valid_ct].astype(str)
            if len(cell_idx) == 0:
                continue

            sample_id = get_sample_id(adata, sample_index)
            barcode_sel = get_barcode_values(adata, cell_idx)
            meta_df = pd.DataFrame({
                "sample_index": sample_index,
                "sample_id": sample_id,
                "barcode": barcode_sel,
                "cell_index": cell_idx,
                "global_cell_index": cell_offsets[sample_index] + cell_idx,
                CELLTYPE_COL: celltype_sel,
                "CancerType": cancer_type_sel,
                "Fibroblast_to_tumor_edge_count": fibro_to_tumor_edge_count[cell_idx],
            })
            metadata_frames.append(meta_df)

            X_sel = expr_rows_cols_to_numpy(adata, cell_idx, all_gene_idx)
            X_sel64 = X_sel.astype(np.float64, copy=False)
            gene_sum_global += np.nansum(X_sel64, axis=0)
            gene_sumsq_global += np.nansum(X_sel64 ** 2, axis=0)
            gene_n_global += int(X_sel64.shape[0])

            for cancer_type in np.unique(cancer_type_sel):
                rows = np.where(cancer_type_sel == cancer_type)[0]
                if len(rows) == 0:
                    continue
                X_ct = X_sel64[rows, :]
                if cancer_type not in gene_sum_by_ct:
                    gene_sum_by_ct[cancer_type] = np.zeros(X_ct.shape[1], dtype=np.float64)
                    gene_sumsq_by_ct[cancer_type] = np.zeros(X_ct.shape[1], dtype=np.float64)
                    gene_n_by_ct[cancer_type] = 0
                gene_sum_by_ct[cancer_type] += np.nansum(X_ct, axis=0)
                gene_sumsq_by_ct[cancer_type] += np.nansum(X_ct ** 2, axis=0)
                gene_n_by_ct[cancer_type] += int(X_ct.shape[0])

            del X_sel, X_sel64
            gc.collect()

        if len(metadata_frames) == 0:
            raise ValueError("No selected tumor cells were found for Fibroblast -> tumor evaluation.")

        tumor_meta_df = pd.concat(metadata_frames, axis=0, ignore_index=True)
        print("Selected tumor-cell counts by cancer type:")
        display(tumor_meta_df.groupby("CancerType", as_index=False).size().sort_values("CancerType"))

        gene_ref_by_ct = {
            ct: build_gene_reference_from_sums(gene_sum_by_ct[ct], gene_sumsq_by_ct[ct], gene_n_by_ct[ct])
            for ct in sorted(gene_sum_by_ct.keys()) if gene_n_by_ct[ct] > 0
        }
        gene_ref_global = build_gene_reference_from_sums(gene_sum_global, gene_sumsq_global, gene_n_global)

        zref_rows = []
        if MODULE_SCORE_METHOD == "zscore_mean_global":
            zref_rows.append({"reference": "global", "CancerType": "all", "n_ref": int(gene_ref_global["n_ref"]), "module_score_method": MODULE_SCORE_METHOD})
        else:
            for ct in sorted(gene_ref_by_ct):
                zref_rows.append({"reference": str(ct), "CancerType": str(ct), "n_ref": int(gene_ref_by_ct[ct]["n_ref"]), "module_score_method": MODULE_SCORE_METHOD})

        score_frames = []
        for sample_index, meta_sub in tumor_meta_df.groupby("sample_index", sort=True):
            sample_index = int(sample_index)
            if sample_index % 10 == 0:
                print(f"[CancerSEA second pass] sample {sample_index + 1}/{len(adata_list)}")
            adata = adata_list[sample_index]
            cell_idx = meta_sub["cell_index"].to_numpy(dtype=np.int64)
            X_sel = expr_rows_cols_to_numpy(adata, cell_idx, all_gene_idx)
            score_df = meta_sub.copy().reset_index(drop=True)
            for program in CANCERSEA_PROGRAMS:
                score_df[f"{program}_score"] = np.nan
                score_df[f"{program}_n_genes_used"] = int(len(program_gene_pos[program]))

            if MODULE_SCORE_METHOD == "zscore_mean_global":
                ref = gene_ref_global
                X_z = (X_sel - ref["mean"][None, :]) / ref["std"][None, :]
                for program in CANCERSEA_PROGRAMS:
                    score_df[f"{program}_score"] = np.nanmean(X_z[:, program_gene_pos[program]], axis=1)
                score_df["zscore_reference"] = "global"
                score_df["zscore_reference_n"] = int(ref["n_ref"])
            else:
                cancer_type_arr = score_df["CancerType"].astype(str).to_numpy()
                for ct in np.unique(cancer_type_arr):
                    if ct not in gene_ref_by_ct:
                        continue
                    rows = np.where(cancer_type_arr == ct)[0]
                    ref = gene_ref_by_ct[ct]
                    X_z = (X_sel[rows, :] - ref["mean"][None, :]) / ref["std"][None, :]
                    for program in CANCERSEA_PROGRAMS:
                        score_df.loc[rows, f"{program}_score"] = np.nanmean(X_z[:, program_gene_pos[program]], axis=1)
                    score_df.loc[rows, "zscore_reference"] = str(ct)
                    score_df.loc[rows, "zscore_reference_n"] = int(ref["n_ref"])

            score_df["module_score_method"] = MODULE_SCORE_METHOD
            score_frames.append(score_df)
            del X_sel
            if "X_z" in locals():
                del X_z
            gc.collect()

        tumor_cancersea_df = pd.concat(score_frames, axis=0, ignore_index=True)
        tumor_cancersea_df.to_csv(base_table_path, index=False)
        pd.DataFrame([
            {"Program": program, "n_genes_used": len(info["genes_used"]), "genes_used": ";".join(info["genes_used"])}
            for program, info in cancersea_gene_sets.items()
        ]).to_csv(genes_used_path, index=False)
        pd.DataFrame(zref_rows).to_csv(zref_summary_path, index=False)

        print(f"Saved tumor CancerSEA table: {base_table_path}")
        print(f"Saved CancerSEA gene table: {genes_used_path}")
        print(f"Saved z-score reference summary: {zref_summary_path}")

    # Re-apply orders and basic type cleaning.
    tumor_cancersea_df["CancerType"] = pd.Categorical(
        tumor_cancersea_df["CancerType"].astype(str), categories=CANCERTYPE_ORDER, ordered=True
    )
    tumor_cancersea_df = tumor_cancersea_df.loc[tumor_cancersea_df["CancerType"].notna()].copy()
    tumor_cancersea_df["sample_index"] = tumor_cancersea_df["sample_index"].astype(int)
    tumor_cancersea_df["cell_index"] = tumor_cancersea_df["cell_index"].astype(int)

    print("Tumor CancerSEA table shape:", tumor_cancersea_df.shape)
    display(tumor_cancersea_df.head())

else:
    tumor_cancersea_df = pd.DataFrame()
    print("All requested SMD scans are cached; CancerSEA recomputation is unnecessary.")


In [ ]:
%%fig6d commot-functions
# ============================================================
# 6A. Run / export COMMOT edge scores for pan-cancer slices
# ------------------------------------------------------------
# Robust Windows-safe version:
#   1. Uses user_database namespace for the custom LR table.
#   2. Uses short output folders/files: s000/s000_COMMOT_edge_scores.csv.
#   3. Keeps full sample_id/sample_name/sample_tag in the manifest and CSV.
#   4. Writes traceback to COMMOT/_errors using short filenames.
#   5. Forces adata.X to in-memory CSR matrix because COMMOT uses .toarray().
#   6. Validates existing outputs and supports partial reruns.
#   7. Resumes from previous manifest, fallback CSV folder, and legacy long-path folders.
#   8. Writes the manifest incrementally after every slice.
# ============================================================

import os
import re
import gc
import traceback
import tempfile
from pathlib import Path
import time

import numpy as np
import pandas as pd
from scipy import sparse

COMMOT_OUTDIR.mkdir(parents=True, exist_ok=True)

# Use a user-defined database name because df_ligrec is built manually.
COMMOT_DB_NAME = "user_database"

# Avoid hidden FileNotFoundError from invalid system temp dirs on Windows / cluster sessions.
COMMOT_TMPDIR = COMMOT_OUTDIR / "_tmp"
COMMOT_TMPDIR.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(COMMOT_TMPDIR)
os.environ["TEMP"] = str(COMMOT_TMPDIR)
os.environ["TMP"] = str(COMMOT_TMPDIR)
tempfile.tempdir = str(COMMOT_TMPDIR)

# Optional debugging controls.
# Set to 1 first if you only want to test the first slice.
COMMOT_MAX_SAMPLES = None
COMMOT_FAIL_FAST = not COMMOT_CONTINUE_ON_SAMPLE_FAILURE

# Windows-safe short output names.
# Full biological/sample names are still stored in the manifest and CSV columns.
COMMOT_USE_SHORT_PATHS = True


def _sanitize_baseline_name(x):
    x = str(x)
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x if x else "sample"


def _commot_sample_tag(sample_index):
    return f"s{int(sample_index):03d}"


def _get_commot_sample_outdir_and_csv(sample_index, sample_name):
    """
    Return Windows-safe output paths.

    Old long path:
        COMMOT/sample000_HumanBreastCancerPatient1_subslice0/
        sample000_HumanBreastCancerPatient1_subslice0_COMMOT_edge_scores.csv

    New short path:
        COMMOT/s000/s000_COMMOT_edge_scores.csv
    """
    if COMMOT_USE_SHORT_PATHS:
        sample_tag = _commot_sample_tag(sample_index)
        sample_outdir = COMMOT_OUTDIR / sample_tag
        out_csv = sample_outdir / f"{sample_tag}_COMMOT_edge_scores.csv"
    else:
        sample_tag = sample_name
        sample_outdir = COMMOT_OUTDIR / sample_name
        out_csv = sample_outdir / f"{sample_name}_COMMOT_edge_scores.csv"

    sample_outdir.mkdir(parents=True, exist_ok=True)
    return sample_tag, sample_outdir, out_csv


def _flatten_lr_side_for_baseline(x):
    if isinstance(x, str):
        return x
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        vals = []
        for y in x:
            if isinstance(y, (list, tuple, set, np.ndarray, pd.Series)):
                vals.extend([str(z) for z in y])
            else:
                vals.append(str(y))
        vals = [v for v in vals if len(str(v).strip()) > 0]
        return ";".join(vals)
    return str(x)


def _first_gene_for_baseline(x):
    """Use a single representative gene for COMMOT / ScCChain LR baselines."""
    if isinstance(x, str):
        if ";" in x:
            return x.split(";")[0].strip()
        return x.strip()
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        vals = list(x)
        if len(vals) == 0:
            return ""
        return _first_gene_for_baseline(vals[0])
    return str(x).strip()


def _lr_pair_to_ligand_receptor_for_baseline(lr):
    if isinstance(lr, (list, tuple, np.ndarray, pd.Series)) and len(lr) >= 2:
        return _first_gene_for_baseline(lr[0]), _first_gene_for_baseline(lr[1])

    lr_str = str(lr)

    for sep in ["—", "|", "~", ":", "_"]:
        if sep in lr_str:
            parts = lr_str.split(sep)
            if len(parts) >= 2:
                return parts[0].strip(), parts[1].strip()

    # Avoid splitting gene symbols containing hyphens unless no other separator exists.
    if "-" in lr_str:
        parts = lr_str.split("-")
        if len(parts) >= 2:
            return parts[0].strip(), parts[1].strip()

    raise ValueError(f"Cannot parse LR pair: {lr}")


def _safe_colmax_normalize_for_baseline(X):
    X = np.asarray(X, dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[X < 0] = 0.0

    if X.size == 0:
        return X

    colmax = np.nanmax(X, axis=0)
    colmax[~np.isfinite(colmax)] = 0.0
    colmax[colmax <= 0] = 1.0

    return (X / colmax.reshape(1, -1)).astype(np.float32, copy=False)


def _ensure_spatial_for_baseline(adata_cur):
    if "spatial" in adata_cur.obsm:
        return adata_cur

    for cols in [
        ("x", "y"),
        ("X", "Y"),
        ("spatial_x", "spatial_y"),
        ("array_row", "array_col"),
    ]:
        if all(c in adata_cur.obs.columns for c in cols):
            adata_cur.obsm["spatial"] = adata_cur.obs[list(cols)].to_numpy(dtype=float)
            return adata_cur

    raise KeyError(
        "Baseline CCC requires adata.obsm['spatial'] or spatial coordinate columns."
    )


def _ensure_commot_compatible_adata(adata_cur):
    """
    COMMOT internally uses adata[:, genes].X.toarray().
    Therefore X should be sparse or another object with .toarray().
    This function also makes sure the data are loaded in memory.
    """
    adata_commot = adata_cur.copy()
    adata_commot = _ensure_spatial_for_baseline(adata_commot)

    # Make var names unique to avoid ambiguous subsetting.
    try:
        adata_commot.var_names_make_unique()
    except Exception:
        pass

    X = adata_commot.X

    # Convert backed / lazy / dense objects to in-memory CSR.
    if sparse.issparse(X):
        adata_commot.X = X.tocsr().astype(np.float32)
    else:
        X_arr = np.asarray(X, dtype=np.float32)
        X_arr = np.nan_to_num(X_arr, nan=0.0, posinf=0.0, neginf=0.0)
        X_arr[X_arr < 0] = 0.0
        adata_commot.X = sparse.csr_matrix(X_arr)

    # COMMOT assumes non-negative abundance-like values.
    if sparse.issparse(adata_commot.X):
        adata_commot.X.data = np.nan_to_num(
            adata_commot.X.data, nan=0.0, posinf=0.0, neginf=0.0
        )
        adata_commot.X.data[adata_commot.X.data < 0] = 0.0

    adata_commot.obsm["spatial"] = np.asarray(
        adata_commot.obsm["spatial"], dtype=np.float64
    )

    return adata_commot


def _build_commot_lr_table_for_adata_pancancer(adata_cur):
    genes = set(map(str, adata_cur.var_names))
    rows = []

    for idx, lr in enumerate(processed.lr_list):
        ligand, receptor = _lr_pair_to_ligand_receptor_for_baseline(lr)

        ligand = str(ligand).strip()
        receptor = str(receptor).strip()

        if ligand in genes and receptor in genes:
            # Safe pathway name for AnnData obsp key.
            pathway = _sanitize_baseline_name(f"LR_{idx:04d}_{ligand}_{receptor}")
            rows.append([ligand, receptor, pathway])

    if len(rows) == 0:
        raise ValueError(
            "No LR pairs are present in the current adata genes for COMMOT."
        )

    df_ligrec = pd.DataFrame(rows, columns=[0, 1, 2]).drop_duplicates()
    return df_ligrec


def _write_traceback_file(outdir, sample_tag, sample_name, exc):
    """
    Write traceback to a short path. This avoids failing again when the
    original sample_name-based path is too long on Windows.
    """
    tb = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))

    error_dir = COMMOT_OUTDIR / "_errors"
    error_dir.mkdir(parents=True, exist_ok=True)

    tb_path = error_dir / f"{sample_tag}_COMMOT_error_traceback.txt"

    with open(tb_path, "w", encoding="utf-8") as f:
        f.write(f"sample_tag: {sample_tag}\n")
        f.write(f"sample_name: {sample_name}\n")
        f.write("=" * 80 + "\n")
        f.write(tb)

    return tb_path, tb


def _safe_to_csv_with_fallback(df, out_csv, sample_tag):
    """
    Save CSV to the intended short path. If even that path fails on Windows,
    fall back to a shared short directory and return the actual path.
    """
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    try:
        df.to_csv(out_csv, index=False)
        return out_csv
    except FileNotFoundError:
        fallback_dir = COMMOT_OUTDIR / "_edge_score_csv"
        fallback_dir.mkdir(parents=True, exist_ok=True)
        fallback_csv = fallback_dir / f"{sample_tag}_COMMOT_edge_scores.csv"
        df.to_csv(fallback_csv, index=False)
        print(
            f"[COMMOT] Warning: failed to write intended path, "
            f"used fallback CSV instead: {fallback_csv}"
        )
        return fallback_csv



# ------------------------------------------------------------
# Partial-rerun / resume helpers
# ------------------------------------------------------------
# A previous run may have finished some slices but stopped before writing
# COMMOT_sample_manifest.csv, or may have written valid CSVs to the fallback
# folder / an older long sample-name folder.  These helpers discover and
# validate any completed CSV before deciding to re-run a slice.

_COMMOT_METADATA_COLS = {
    "sample_index",
    "sample_id",
    "sample_name",
    "sample_tag",
    "batch_index",
    "local_edge_index",
    "global_edge_index",
    "sender_index",
    "receiver_index",
}


def _atomic_write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)


def _validate_commot_edge_score_csv(csv_path, sample_index):
    """
    Check that an existing COMMOT CSV is likely complete and aligned.

    A valid file must:
      - be readable as CSV;
      - contain at least one non-metadata feature column;
      - contain the expected number of rows / local_edge_index entries.
    """
    csv_path = Path(csv_path)

    if not csv_path.exists():
        return False, "file does not exist", 0, 0

    try:
        header_cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    except Exception as e:
        return False, f"cannot read CSV header: {repr(e)}", 0, 0

    if len(header_cols) == 0:
        return False, "CSV has no columns", 0, 0

    feature_cols = [c for c in header_cols if c not in _COMMOT_METADATA_COLS]
    if len(feature_cols) == 0:
        return False, "CSV has no COMMOT feature columns", 0, 0

    try:
        expected_edges = int(edge_counts[int(sample_index)])
    except Exception:
        expected_edges = None

    try:
        if "local_edge_index" in header_cols:
            idx_df = pd.read_csv(csv_path, usecols=["local_edge_index"])
            n_rows = int(idx_df.shape[0])

            if expected_edges is not None and n_rows < expected_edges:
                return (
                    False,
                    f"incomplete rows: {n_rows} < expected_edges={expected_edges}",
                    n_rows,
                    len(feature_cols),
                )

            loc = pd.to_numeric(idx_df["local_edge_index"], errors="coerce")
            loc = loc.dropna().astype(np.int64)

            if loc.shape[0] == 0:
                return False, "local_edge_index column is empty/non-numeric", n_rows, len(feature_cols)

            if expected_edges is not None:
                if int(loc.min()) < 0 or int(loc.max()) >= expected_edges:
                    return (
                        False,
                        f"local_edge_index out of range [{int(loc.min())}, {int(loc.max())}] for expected_edges={expected_edges}",
                        n_rows,
                        len(feature_cols),
                    )

                n_unique = int(loc.nunique())
                if n_unique < expected_edges:
                    return (
                        False,
                        f"incomplete local_edge_index coverage: {n_unique} unique < expected_edges={expected_edges}",
                        n_rows,
                        len(feature_cols),
                    )
        else:
            # Fallback count without loading all feature columns.
            with open(csv_path, "rb") as f:
                n_rows = max(sum(1 for _ in f) - 1, 0)
            if expected_edges is not None and n_rows != expected_edges:
                return (
                    False,
                    f"row count mismatch without local_edge_index: rows={n_rows}, expected_edges={expected_edges}",
                    n_rows,
                    len(feature_cols),
                )

    except Exception as e:
        return False, f"CSV validation failed: {repr(e)}", 0, len(feature_cols)

    return True, "valid complete CSV", n_rows, len(feature_cols)


def _load_previous_commot_manifest_if_available():
    manifest_path = Path(COMMOT_SAMPLE_MANIFEST_PATH)
    if not manifest_path.exists():
        return pd.DataFrame()

    try:
        old = pd.read_csv(manifest_path)
        if "sample_index" not in old.columns:
            print(f"[COMMOT] Ignoring old manifest without sample_index: {manifest_path}")
            return pd.DataFrame()
        old["sample_index"] = pd.to_numeric(old["sample_index"], errors="coerce")
        old = old.dropna(subset=["sample_index"]).copy()
        old["sample_index"] = old["sample_index"].astype(int)
        return old
    except Exception as e:
        print(f"[COMMOT] Warning: could not read existing manifest {manifest_path}: {repr(e)}")
        return pd.DataFrame()


_COMMOT_PREVIOUS_MANIFEST_DF = _load_previous_commot_manifest_if_available()


def _commot_existing_csv_candidates(sample_index, sample_name, sample_tag, expected_csv):
    """
    Return candidate existing CSV paths from:
      1. previous manifest;
      2. current short path;
      3. fallback CSV directory;
      4. old long sample-name path;
      5. globbed legacy folders.
    """
    candidates = []

    # 1) Previous manifest path, if a prior run wrote one.
    if _COMMOT_PREVIOUS_MANIFEST_DF is not None and not _COMMOT_PREVIOUS_MANIFEST_DF.empty:
        sub = _COMMOT_PREVIOUS_MANIFEST_DF.loc[
            _COMMOT_PREVIOUS_MANIFEST_DF["sample_index"].astype(int).eq(int(sample_index))
        ]
        if not sub.empty and "score_csv" in sub.columns:
            for x in sub["score_csv"].dropna().astype(str).tolist():
                if len(x.strip()) > 0:
                    candidates.append(Path(x))

    # 2) Expected current short path.
    candidates.append(Path(expected_csv))

    # 3) Fallback path used by _safe_to_csv_with_fallback.
    candidates.append(COMMOT_OUTDIR / "_edge_score_csv" / f"{sample_tag}_COMMOT_edge_scores.csv")

    # 4) Old long sample-name path.
    candidates.append(COMMOT_OUTDIR / sample_name / f"{sample_name}_COMMOT_edge_scores.csv")

    # 5) Legacy/globbed folders that may have been created by older code.
    try:
        candidates.extend(sorted(COMMOT_OUTDIR.glob(f"sample{int(sample_index):03d}_*/*_COMMOT_edge_scores.csv")))
    except Exception:
        pass

    try:
        candidates.extend(sorted(COMMOT_OUTDIR.glob(f"{sample_tag}/*COMMOT_edge_scores.csv")))
    except Exception:
        pass

    # De-duplicate while preserving order.
    seen = set()
    unique_candidates = []
    for p in candidates:
        p = Path(p)
        key = str(p)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(p)

    return unique_candidates


def _find_existing_valid_commot_csv(sample_index, sample_name, sample_tag, expected_csv):
    invalid_notes = []

    for p in _commot_existing_csv_candidates(sample_index, sample_name, sample_tag, expected_csv):
        ok, reason, n_rows, n_features = _validate_commot_edge_score_csv(p, sample_index)
        if ok:
            msg = (
                f"loaded existing complete output; rows={n_rows}; "
                f"n_features={n_features}; path={p}"
            )
            print(f"[COMMOT] Existing complete output for sample {sample_index} ({sample_tag}): {p}")
            return p, "existing_loaded", msg

        if p.exists():
            invalid_notes.append(f"{p} -> {reason}")

    if len(invalid_notes) > 0:
        return None, "missing", "no valid existing CSV; invalid candidates: " + " | ".join(invalid_notes[:5])

    return None, "missing", "no existing CSV found"


def _merge_with_previous_commot_manifest(new_rows_df):
    """
    Merge newly discovered/generated rows into the previous manifest so that
    interrupted reruns still preserve completed samples.
    """
    new_rows_df = new_rows_df.copy()
    if not new_rows_df.empty:
        new_rows_df["sample_index"] = new_rows_df["sample_index"].astype(int)

    old_manifest = _COMMOT_PREVIOUS_MANIFEST_DF.copy()
    if old_manifest is not None and not old_manifest.empty and not new_rows_df.empty:
        old_manifest = old_manifest.loc[
            ~old_manifest["sample_index"].astype(int).isin(new_rows_df["sample_index"].astype(int))
        ].copy()
        merged = pd.concat([old_manifest, new_rows_df], axis=0, ignore_index=True, sort=False)
    elif old_manifest is not None and not old_manifest.empty:
        merged = old_manifest.copy()
    else:
        merged = new_rows_df.copy()

    if not merged.empty and "sample_index" in merged.columns:
        merged["sample_index"] = pd.to_numeric(merged["sample_index"], errors="coerce")
        merged = merged.dropna(subset=["sample_index"]).copy()
        merged["sample_index"] = merged["sample_index"].astype(int)
        merged = merged.sort_values("sample_index").reset_index(drop=True)

    return merged


def _write_commot_manifest_progress(new_rows):
    """
    Write manifest after every slice so another interruption does not lose
    the resume information.
    """
    if len(new_rows) == 0:
        return
    new_df = pd.DataFrame(new_rows)
    merged = _merge_with_previous_commot_manifest(new_df)
    _atomic_write_csv(merged, COMMOT_SAMPLE_MANIFEST_PATH)


def _run_commot_one_sample_pancancer(sample_index, sample_name, sample_tag, adata_cur, outdir, out_csv):
    try:
        import commot as ct
    except Exception as e:
        raise ImportError(
            "commot is required to run COMMOT. Install/import commot or provide existing COMMOT CSV outputs."
        ) from e

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    out_csv = Path(out_csv)

    if out_csv.exists():
        print(f"[COMMOT] Skipping existing output: {out_csv}")
        return out_csv, "existing", "existing output found"

    adata_commot = _ensure_commot_compatible_adata(adata_cur)
    edge_index_cur = edge_index_to_e2(
        get_field(spidernet_data_list[sample_index], "edge_index")
    )

    df_ligrec = _build_commot_lr_table_for_adata_pancancer(adata_commot)

    coords = np.asarray(adata_commot.obsm["spatial"], dtype=float)
    if coords.ndim != 2 or coords.shape[0] != adata_commot.n_obs:
        raise ValueError(
            f"Invalid spatial coordinates for sample {sample_index}: "
            f"coords.shape={coords.shape}, n_obs={adata_commot.n_obs}"
        )

    edge_d = np.sqrt(
        np.sum(
            (coords[edge_index_cur[:, 0]] - coords[edge_index_cur[:, 1]]) ** 2,
            axis=1,
        )
    )
    edge_d = edge_d[np.isfinite(edge_d)]

    if len(edge_d) == 0:
        raise ValueError(
            f"Cannot estimate COMMOT distance threshold for sample {sample_index}: no finite edge distances."
        )

    # Keep original logic: distance threshold = median SpiderNet edge distance.
    dis_thr = float(np.nanmedian(edge_d))

    print(
        f"[COMMOT] Running sample {sample_index + 1}/{len(adata_list)}: {sample_name}; "
        f"sample_tag={sample_tag}, n_obs={adata_commot.n_obs}, "
        f"n_edges={edge_index_cur.shape[0]}, LR pairs={df_ligrec.shape[0]}, "
        f"dis_thr={dis_thr:.4f}"
    )

    ct.tl.spatial_communication(
        adata_commot,
        database_name=COMMOT_DB_NAME,
        df_ligrec=df_ligrec,
        dis_thr=dis_thr,
        heteromeric=False,          # We already reduce each LR side to one representative gene.
        pathway_sum=True,
        cot_nitermax=COMMOT_COT_NITERMAX,
    )

    score_cols = []
    score_mat_parts = []

    rows = edge_index_cur[:, 0].astype(np.int64)
    cols = edge_index_cur[:, 1].astype(np.int64)

    # COMMOT stores pathway-summed matrices as:
    # adata.obsp[f"commot-{database_name}-{pathway}"]
    for pathway in df_ligrec[2].astype(str).unique().tolist():
        key = f"commot-{COMMOT_DB_NAME}-{pathway}"

        if key not in adata_commot.obsp:
            continue

        vals = edge_matrix_from_square(adata_commot.obsp[key], rows, cols)
        vals = np.asarray(vals, dtype=np.float32)

        score_cols.append(pathway)
        score_mat_parts.append(vals)

    if len(score_mat_parts) == 0:
        available_obsp = list(adata_commot.obsp.keys())[:20]
        raise RuntimeError(
            f"COMMOT finished for sample {sample_name}, but no edge-score matrices were found. "
            f"Expected keys like commot-{COMMOT_DB_NAME}-<pathway>. "
            f"First available obsp keys: {available_obsp}"
        )

    X = np.vstack(score_mat_parts).T.astype(np.float32, copy=False)

    if COMMOT_NORMALIZE_EDGE_SCORES:
        X = _safe_colmax_normalize_for_baseline(X)

    out = pd.DataFrame(X, columns=score_cols)
    out.insert(
        0,
        "global_edge_index",
        edge_offsets[sample_index] + np.arange(edge_index_cur.shape[0], dtype=np.int64),
    )
    out.insert(0, "local_edge_index", np.arange(edge_index_cur.shape[0], dtype=np.int64))
    out.insert(0, "sample_name", sample_name)
    out.insert(0, "sample_tag", sample_tag)
    out.insert(0, "sample_id", get_sample_id(adata_cur, sample_index))
    out.insert(0, "sample_index", sample_index)

    actual_csv = _safe_to_csv_with_fallback(out, out_csv, sample_tag)

    n_features = len(score_cols)
    n_nonzero = int(np.count_nonzero(X))

    del adata_commot, X, out, score_mat_parts
    gc.collect()

    msg = f"generated; n_features={n_features}; n_nonzero={n_nonzero}"
    print(f"[COMMOT] Saved: {actual_csv} ({msg})")

    return actual_csv, "generated", msg




In [ ]:
%%fig6d commot
if "COMMOT" in FIG6D_METHODS_NEED_SCAN:
    commot_manifest_rows = []
    commot_cell_t0 = time.perf_counter()

    sample_iter = list(enumerate(adata_list))
    if COMMOT_MAX_SAMPLES is not None:
        sample_iter = sample_iter[: int(COMMOT_MAX_SAMPLES)]

    n_samples_to_process = len(sample_iter)

    for sample_counter, (sample_index, adata_cur) in enumerate(sample_iter, start=1):
        sample_t0 = time.perf_counter()

        sample_id = get_sample_id(adata_cur, sample_index)
        sample_name = f"sample{sample_index:03d}_{_sanitize_baseline_name(sample_id)}"

        sample_tag, sample_outdir, expected_csv = _get_commot_sample_outdir_and_csv(
            sample_index=sample_index,
            sample_name=sample_name,
        )

        status = "missing"
        message = ""
        out_csv = expected_csv

        # Used only when COMMOT_FAIL_FAST=True. We still write manifest/timing first.
        raise_after_progress = False
        tb_to_print = None
        exc_to_raise = None

        # Resume logic:
        #   - first search previous manifest / short path / fallback path / old long path;
        #   - validate that the CSV is complete for the current slice;
        #   - if valid, keep it in the manifest and do not rerun COMMOT.
        existing_csv, existing_status, existing_message = _find_existing_valid_commot_csv(
            sample_index=sample_index,
            sample_name=sample_name,
            sample_tag=sample_tag,
            expected_csv=expected_csv,
        )

        if existing_csv is not None:
            out_csv = existing_csv
            status = existing_status
            message = existing_message

        elif RUN_COMMOT_IF_MISSING:
            if existing_message:
                print(f"[COMMOT] No reusable output for sample {sample_index} ({sample_tag}): {existing_message}")

            try:
                out_csv, status, message = _run_commot_one_sample_pancancer(
                    sample_index=sample_index,
                    sample_name=sample_name,
                    sample_tag=sample_tag,
                    adata_cur=adata_cur,
                    outdir=sample_outdir,
                    out_csv=expected_csv,
                )
            except Exception as e:
                status = "failed"
                tb_path, tb = _write_traceback_file(sample_outdir, sample_tag, sample_name, e)
                message = f"{repr(e)} | traceback: {tb_path}"

                print(
                    f"[Warning] COMMOT failed for sample {sample_index} ({sample_name}).\n"
                    f"Traceback saved to: {tb_path}\n"
                    f"Last error: {repr(e)}"
                )

                if COMMOT_FAIL_FAST:
                    raise_after_progress = True
                    tb_to_print = tb
                    exc_to_raise = e

        else:
            message = (
                existing_message
                if existing_message
                else "RUN_COMMOT_IF_MISSING=False and output does not exist"
            )

        # ------------------------------------------------------------
        # Timing summary for this sample
        # ------------------------------------------------------------
        sample_elapsed_min = (time.perf_counter() - sample_t0) / 60.0

        if message is None:
            message = ""
        message = f"{message} | elapsed_min={sample_elapsed_min:.3f}"

        print(
            f"[COMMOT] Sample {sample_counter}/{n_samples_to_process} finished; "
            f"sample_index={sample_index}; sample_tag={sample_tag}; "
            f"status={status}; elapsed_min={sample_elapsed_min:.3f}"
        )

        commot_manifest_rows.append(
            {
                "sample_index": int(sample_index),
                "sample_id": sample_id,
                "sample_name": sample_name,
                "sample_tag": sample_tag,
                "score_csv": str(out_csv),
                "status": status,
                "message": message,
                "elapsed_min": float(sample_elapsed_min),
                "n_obs": int(adata_cur.n_obs),
                "n_edges": int(edge_counts[sample_index]),
            }
        )

        # Save progress after every slice. If the notebook/kernel stops later,
        # already completed slices will still be discoverable on the next rerun.
        _write_commot_manifest_progress(commot_manifest_rows)

        if raise_after_progress:
            if tb_to_print is not None:
                print(tb_to_print)
            raise exc_to_raise

    # If debugging only a subset, keep previous manifest rows for samples not touched if available.
    commot_sample_manifest_df_new = pd.DataFrame(commot_manifest_rows)

    if COMMOT_MAX_SAMPLES is not None:
        commot_sample_manifest_df = _merge_with_previous_commot_manifest(commot_sample_manifest_df_new)
    else:
        # For full runs, still merge with previous manifest to preserve rows from
        # samples not reached if an earlier interrupted run left valid entries.
        commot_sample_manifest_df = _merge_with_previous_commot_manifest(commot_sample_manifest_df_new)

    COMMOT_SAMPLE_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    _atomic_write_csv(commot_sample_manifest_df, COMMOT_SAMPLE_MANIFEST_PATH)

    commot_total_elapsed_min = (time.perf_counter() - commot_cell_t0) / 60.0

    print("Saved COMMOT sample manifest:", COMMOT_SAMPLE_MANIFEST_PATH)
    print(
        f"[COMMOT] Cell finished; processed_samples={n_samples_to_process}; "
        f"total_elapsed_min={commot_total_elapsed_min:.3f}"
    )

    display(commot_sample_manifest_df.groupby("status", as_index=False).size())

    if "elapsed_min" in commot_sample_manifest_df.columns:
        display(
            commot_sample_manifest_df[
                ["sample_index", "sample_tag", "status", "elapsed_min", "n_obs", "n_edges", "score_csv"]
            ].head()
        )
    else:
        display(commot_sample_manifest_df.head())
else:
    print("COMMOT SMD cache is available; baseline computation is unnecessary.")


In [ ]:
%%fig6d sccchain-inputs
if "ScCChain" in FIG6D_METHODS_NEED_SCAN:
    import re
    import gc
    from pathlib import Path

    SCCCHAIN_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    SCCCHAIN_OUTDIR.mkdir(parents=True, exist_ok=True)
    BASELINE_CCC_OUTDIR.mkdir(parents=True, exist_ok=True)

    # Companion Julia runner supplied beside this notebook.
    SCCCHAIN_JULIA_RUNNER_NAME = "ScCChain_Pancancer_runner.jl"


    def _sanitize_sccchain_name(x):
        x = str(x)
        x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
        x = re.sub(r"_+", "_", x).strip("_")
        return x if x else "sample"


    def _sccchain_sample_tag(sample_index):
        return f"s{int(sample_index):03d}"


    def _sccchain_expected_paths(sample_index, sample_name):
        """Return short output paths plus legacy long-path fallback."""
        sample_tag = _sccchain_sample_tag(sample_index)
        sample_outdir = SCCCHAIN_OUTDIR / sample_tag
        score_csv = sample_outdir / f"{sample_tag}_ScCChain_edge_program_scores.csv"

        legacy_outdir = SCCCHAIN_OUTDIR / sample_name
        legacy_score_csv = legacy_outdir / f"{sample_name}_ScCChain_edge_program_scores.csv"
        return sample_tag, sample_outdir, score_csv, legacy_outdir, legacy_score_csv


    def _looks_like_sccchain_score_csv(path):
        path = Path(path)
        if not path.exists():
            return False, "missing"
        try:
            header = pd.read_csv(path, nrows=0).columns.tolist()
        except Exception as e:
            return False, f"cannot read header: {repr(e)}"
        if len(header) < 3:
            return False, "fewer than 3 columns"
        program_cols = [c for c in header[2:] if str(c).startswith("program_")]
        if len(program_cols) == 0:
            return False, "no program_* columns"
        return True, f"ok; n_program_cols={len(program_cols)}"


    def _ensure_spatial_for_sccchain_export(adata_cur):
        if "spatial" in adata_cur.obsm:
            return adata_cur
        for cols in [("x", "y"), ("X", "Y"), ("spatial_x", "spatial_y"), ("array_row", "array_col")]:
            if all(c in adata_cur.obs.columns for c in cols):
                adata_cur.obsm["spatial"] = adata_cur.obs[list(cols)].to_numpy(dtype=float)
                return adata_cur
        raise KeyError("ScCChain export requires adata.obsm['spatial'] or spatial coordinate columns.")


    def _estimate_radius_from_spidernet_edges(sample_index, adata_cur):
        adata_cur = _ensure_spatial_for_sccchain_export(adata_cur)
        coords = np.asarray(adata_cur.obsm["spatial"], dtype=float)
        edge_index_cur = edge_index_to_e2(get_field(spidernet_data_list[sample_index], "edge_index"))
        d = np.sqrt(np.sum((coords[edge_index_cur[:, 0]] - coords[edge_index_cur[:, 1]]) ** 2, axis=1))
        d = d[np.isfinite(d)]
        if len(d) == 0:
            return np.nan
        return float(np.nanmedian(d))


    manifest_rows = []
    existing_score_rows = []

    for sample_index, adata_cur in enumerate(adata_list):
        sample_id = get_sample_id(adata_cur, sample_index)
        sample_name = f"sample{sample_index:03d}_{_sanitize_sccchain_name(sample_id)}"
        sample_tag, sample_outdir, score_csv, legacy_outdir, legacy_score_csv = _sccchain_expected_paths(sample_index, sample_name)

        # Keep h5ad names unchanged for backward compatibility with already exported inputs.
        h5ad_path = SCCCHAIN_INPUT_DIR / f"{sample_name}.h5ad"

        radius = _estimate_radius_from_spidernet_edges(sample_index, adata_cur)

        if not h5ad_path.exists():
            print(f"[ScCChain export] Writing sample {sample_index + 1}/{len(adata_list)}: {h5ad_path}")
            adata_export = adata_cur.copy()
            adata_export = _ensure_spatial_for_sccchain_export(adata_export)
            adata_export.var["Gene_name"] = adata_export.var_names.astype(str)
            try:
                adata_export.write_h5ad(h5ad_path, compression="gzip")
            except TypeError:
                adata_export.write_h5ad(h5ad_path)
            del adata_export
            gc.collect()
        else:
            print(f"[ScCChain export] Existing h5ad: {h5ad_path}")

        # Check both short and legacy output locations for resume awareness.
        short_ok, short_msg = _looks_like_sccchain_score_csv(score_csv)
        legacy_ok, legacy_msg = _looks_like_sccchain_score_csv(legacy_score_csv)
        existing_score_csv = ""
        existing_status = "missing"
        if short_ok:
            existing_score_csv = str(score_csv)
            existing_status = "existing_short"
        elif legacy_ok:
            existing_score_csv = str(legacy_score_csv)
            existing_status = "existing_legacy"

        existing_score_rows.append({
            "sample_index": int(sample_index),
            "sample_id": sample_id,
            "sample_name": sample_name,
            "sample_tag": sample_tag,
            "score_csv": str(score_csv),
            "legacy_score_csv": str(legacy_score_csv),
            "existing_score_csv": existing_score_csv,
            "existing_status": existing_status,
            "short_check": short_msg,
            "legacy_check": legacy_msg,
        })

        manifest_rows.append({
            "sample_index": int(sample_index),
            "sample_id": sample_id,
            "sample_name": sample_name,
            "sample_tag": sample_tag,
            "h5ad_path": str(h5ad_path),
            "radius": radius,
            "n_obs": int(adata_cur.n_obs),
            "n_edges": int(edge_counts[sample_index]),
            "sample_outdir": str(sample_outdir),
            "score_csv": str(score_csv),
            "legacy_sample_outdir": str(legacy_outdir),
            "legacy_score_csv": str(legacy_score_csv),
        })

    scchain_manifest_df = pd.DataFrame(manifest_rows)
    scchain_manifest_df.to_csv(SCCCHAIN_MANIFEST_PATH, index=False)

    scchain_existing_score_df = pd.DataFrame(existing_score_rows)
    scchain_existing_score_df.to_csv(BASELINE_CCC_OUTDIR / "ScCChain_existing_output_check.csv", index=False)

    # Export LR DB used by the Julia ScCChain runner.
    lr_rows = []
    for i, lr in enumerate(processed.lr_list):
        ligand, receptor = _lr_pair_to_ligand_receptor_for_baseline(lr) if "_lr_pair_to_ligand_receptor_for_baseline" in globals() else (str(lr[0]), str(lr[1]))
        lr_rows.append({
            "lr_index": i,
            "ligand": ligand,
            "receptor": receptor,
            "pathway": f"LR_{i:04d}_{ligand}_{receptor}",
        })

    pd.DataFrame(lr_rows).to_csv(SCCCHAIN_LR_DB_PATH, index=False)

    print("Saved ScCChain manifest:", SCCCHAIN_MANIFEST_PATH)
    display(scchain_manifest_df.head())
    print("Saved ScCChain existing-output check:", BASELINE_CCC_OUTDIR / "ScCChain_existing_output_check.csv")
    display(scchain_existing_score_df.groupby("existing_status", as_index=False).size())
    print("Saved ScCChain LR DB:", SCCCHAIN_LR_DB_PATH)
    print("ScCChain output root:", SCCCHAIN_OUTDIR)

    print("Run the next Julia cell to complete missing scCChain outputs.")

else:
    print("ScCChain SMD cache is available; input export is unnecessary.")


In [ ]:
%%fig6d julia
import shutil
import subprocess
import time


def _fig6d_resolve_julia_runner():
    if SCCCHAIN_RUNNER_PATH is not None:
        runner = Path(SCCCHAIN_RUNNER_PATH).expanduser().resolve()
        if not runner.is_file():
            raise FileNotFoundError(f"Julia runner not found: {runner}")
        return runner
    relative_dirs = [
        Path("."), Path("Tutorial/Pancancer"), Path("SpiderNet_Project/Tutorial/Pancancer"),
        Path("SpiderNet_proj/SpiderNet_Project/Tutorial/Pancancer"),
    ]
    for base in [Path.cwd(), *Path.cwd().parents]:
        for relative_dir in relative_dirs:
            candidate = base / relative_dir / "ScCChain_Pancancer_runner.jl"
            if candidate.is_file():
                return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate ScCChain_Pancancer_runner.jl. Set SCCCHAIN_RUNNER_PATH "
        "in the Fig. 6d settings cell to the companion file beside this notebook."
    )


def _fig6d_sccchain_output_status(manifest):
    rows = []
    for _, row in manifest.iterrows():
        paths = [row.get("score_csv"), row.get("legacy_score_csv")]
        available = None
        for value in paths:
            if value is None or pd.isna(value):
                continue
            path = Path(str(value))
            if not path.is_file():
                continue
            try:
                preview = pd.read_csv(path, nrows=1)
                programs = [c for c in preview.columns[2:] if str(c).startswith("program_")]
                if not preview.empty and len(programs) == SCCCHAIN_N_PROGRAMS:
                    available = path
                    break
            except (OSError, ValueError, pd.errors.ParserError):
                continue
        rows.append({
            "sample_index": int(row["sample_index"]),
            "sample_id": str(row["sample_id"]),
            "ready": available is not None,
            "score_csv": str(available) if available is not None else str(row["score_csv"]),
        })
    return pd.DataFrame(rows)


def run_fig6d_julia():
    """Run the companion script here, retaining its parameters and resume logic."""
    manifest = pd.read_csv(SCCCHAIN_MANIFEST_PATH)
    before = _fig6d_sccchain_output_status(manifest)
    if before["ready"].all() and not SCCCHAIN_OVERWRITE and not SCCCHAIN_FORCE_LAUNCH:
        print(f"ScCChain: all {len(before)} outputs are present; Julia execution is unnecessary.")
        return before
    if not RUN_SCCCHAIN_IF_MISSING:
        raise RuntimeError("ScCChain outputs are incomplete. Enable RUN_SCCCHAIN_IF_MISSING and rerun this cell.")
    runner = _fig6d_resolve_julia_runner()
    executable = shutil.which(str(JULIA_EXECUTABLE))
    if executable is None and Path(str(JULIA_EXECUTABLE)).is_file():
        executable = str(Path(JULIA_EXECUTABLE).resolve())
    if executable is None:
        raise FileNotFoundError("Julia was not found. Set JULIA_EXECUTABLE to the installed Julia executable.")
    command = [executable, "--startup-file=no"]
    if JULIA_PROJECT is not None:
        command.append(f"--project={Path(JULIA_PROJECT).expanduser().resolve()}")
    command += [
        str(runner), "--manifest", str(SCCCHAIN_MANIFEST_PATH.resolve()),
        "--lr-db-csv", str(SCCCHAIN_LR_DB_PATH.resolve()),
        "--result-root", str(SCCCHAIN_OUTDIR.resolve()),
        "--n-programs", str(SCCCHAIN_N_PROGRAMS), "--seed", str(SCCCHAIN_SEED),
        "--knn-k", str(SCCCHAIN_KNN_K), "--alpha", str(SCCCHAIN_ALPHA),
        "--no-if-test", "--fail-fast",
    ]
    if SCCCHAIN_OVERWRITE:
        command.append("--overwrite")
    log_path = OUT_DIR / "ScCChain_Julia_runner.log"
    print("Running:", subprocess.list2cmdline(command))
    print("Julia log:", log_path)
    # A file-backed log avoids filling a pipe during a long Julia run.
    with open(log_path, "w", encoding="utf-8") as output:
        process = subprocess.Popen(
            command, cwd=str(runner.parent), stdin=subprocess.DEVNULL,
            stdout=output, stderr=subprocess.STDOUT,
            creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
        )
        started = time.monotonic()
        last_progress = started
        try:
            with open(log_path, "r", encoding="utf-8", errors="replace") as reader:
                while process.poll() is None:
                    chunk = reader.read(65536)
                    if chunk:
                        print(chunk, end="", flush=True)
                    if time.monotonic() - last_progress >= 60:
                        print(f"Julia running: {(time.monotonic() - started) / 60:.1f} min", flush=True)
                        last_progress = time.monotonic()
                    time.sleep(1)
                print(reader.read(), end="", flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if process.returncode != 0:
        raise RuntimeError(f"Julia exited with code {process.returncode}. See {log_path}")
    after = _fig6d_sccchain_output_status(manifest)
    if not after["ready"].all():
        display(after.loc[~after["ready"]])
        raise RuntimeError("ScCChain outputs remain incomplete; the SMD comparison was not run.")
    return after


if "ScCChain" in FIG6D_METHODS_NEED_SCAN:
    fig6d_sccchain_status = run_fig6d_julia()
    display(fig6d_sccchain_status.groupby("ready", as_index=False).size())
else:
    print("ScCChain SMD cache is available; Julia execution is unnecessary.")


In [ ]:
%%fig6d sccchain-check
if "ScCChain" in FIG6D_METHODS_NEED_SCAN:
    # ============================================================
    # 6B-test. Sanity check the first ScCChain output before Cell 6
    # ------------------------------------------------------------
    # Run after:
    #   1) Cell 6B
    #   2) Julia runner with --if-test
    #
    # Purpose:
    #   Verify that the first sample output:
    #     ScCChain/s000/s000_ScCChain_edge_program_scores.csv
    #   is readable, has valid program columns, and can be aligned to
    #   the SpiderNet edge_index used downstream in Cell 6.
    # ============================================================

    import numpy as np
    import pandas as pd
    from pathlib import Path
    import matplotlib.pyplot as plt

    TEST_SAMPLE_INDEX = 0
    TEST_REQUIRE_MIN_SPN_COVERAGE = 0.50   # only warning threshold; not a hard failure
    TEST_REQUIRE_MIN_SC_OVERLAP = 0.50     # only warning threshold; not a hard failure
    TEST_PLOT_HIST = True


    def _to_e2_local(edge_index):
        """Robustly convert edge_index to E x 2 numpy array."""
        arr = np.asarray(edge_index)
        if arr.ndim != 2:
            raise ValueError(f"edge_index must be 2D, got shape={arr.shape}")
        if arr.shape[1] == 2:
            return arr.astype(np.int64, copy=False)
        if arr.shape[0] == 2:
            return arr.T.astype(np.int64, copy=False)
        raise ValueError(f"Cannot interpret edge_index shape={arr.shape} as E x 2 or 2 x E.")


    def _get_spidernet_edge_index_e2(sample_index):
        raw = get_field(spidernet_data_list[sample_index], "edge_index")
        if "edge_index_to_e2" in globals():
            return edge_index_to_e2(raw).astype(np.int64, copy=False)
        return _to_e2_local(raw)


    def _validate_sccchain_score_csv_local(csv_path):
        csv_path = Path(csv_path)
        if not csv_path.exists():
            return False, "missing"
        try:
            cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
        except Exception as e:
            return False, f"cannot read header: {repr(e)}"
        if len(cols) < 3:
            return False, "fewer than 3 columns"
        program_cols = [c for c in cols[2:] if str(c).startswith("program_")]
        if len(program_cols) == 0:
            return False, "no program_* columns"
        return True, f"ok; n_program_cols={len(program_cols)}"


    def _candidate_sccchain_first_csv_paths(sample_index=0):
        manifest_path = Path(SCCCHAIN_MANIFEST_PATH)
        if not manifest_path.exists():
            raise FileNotFoundError(f"Missing ScCChain manifest: {manifest_path}. Run the scCChain input-export cell first.")

        manifest_df = pd.read_csv(manifest_path)
        sub = manifest_df.loc[manifest_df["sample_index"].astype(int).eq(int(sample_index))]
        if sub.empty:
            raise ValueError(f"Sample index {sample_index} not found in ScCChain manifest: {manifest_path}")

        row = sub.iloc[0]
        sample_name = str(row["sample_name"])
        sample_tag = str(row.get("sample_tag", f"s{sample_index:03d}"))

        candidates = []

        def add_path(x):
            if x is None:
                return
            if isinstance(x, float) and np.isnan(x):
                return
            s = str(x).strip()
            if not s or s.lower() in {"nan", "none", "missing", "null"}:
                return
            p = Path(s)
            if p not in candidates:
                candidates.append(p)

        # Preferred short-path output from v7 runner.
        add_path(row.get("score_csv", None))

        # Legacy / fallback columns if present.
        add_path(row.get("existing_score_csv", None))
        add_path(row.get("legacy_score_csv", None))

        # Reconstruct expected paths just in case.
        result_root = Path(globals().get("SCCCHAIN_OUTDIR", SC_CCHAIN_PATH_MAIN))
        add_path(result_root / sample_tag / f"{sample_tag}_ScCChain_edge_program_scores.csv")
        add_path(result_root / sample_name / f"{sample_name}_ScCChain_edge_program_scores.csv")

        return row, candidates


    # ----------------------------
    # 1) Locate first sample output
    # ----------------------------
    row0, candidate_paths = _candidate_sccchain_first_csv_paths(TEST_SAMPLE_INDEX)

    print("First ScCChain manifest row:")
    display(pd.DataFrame([row0.to_dict()]))

    checked = []
    score_csv = None
    for p in candidate_paths:
        ok, msg = _validate_sccchain_score_csv_local(p)
        checked.append({"candidate_path": str(p), "exists": Path(p).exists(), "check": msg})
        if ok and score_csv is None:
            score_csv = Path(p)

    print("Candidate ScCChain output paths:")
    display(pd.DataFrame(checked))

    if score_csv is None:
        raise FileNotFoundError(
            "No usable ScCChain output CSV found for the first sample. "
            "Run the preceding Julia cell first."
        )

    print(f"[OK] Using ScCChain output CSV:\n{score_csv}")


    # ----------------------------
    # 2) Read and inspect output CSV
    # ----------------------------
    score_df = pd.read_csv(score_csv)
    print(f"Raw ScCChain output shape: {score_df.shape}")
    display(score_df.head())

    if score_df.shape[1] < 3:
        raise ValueError(f"Unexpected ScCChain output format: {score_csv}")

    sender_col, receiver_col = score_df.columns[:2]
    program_cols = [c for c in score_df.columns[2:] if str(c).startswith("program_")]

    print("Sender/receiver columns:", sender_col, receiver_col)
    print("Program columns:", program_cols)

    if len(program_cols) == 0:
        raise ValueError("No program_* columns found in the ScCChain output.")

    # Numeric conversion
    score_df = score_df.rename(columns={sender_col: "sender_index", receiver_col: "receiver_index"}).copy()
    score_df["sender_index"] = pd.to_numeric(score_df["sender_index"], errors="coerce")
    score_df["receiver_index"] = pd.to_numeric(score_df["receiver_index"], errors="coerce")

    n_bad_index = int(score_df[["sender_index", "receiver_index"]].isna().any(axis=1).sum())
    if n_bad_index > 0:
        print(f"[Warning] Dropping {n_bad_index} rows with invalid sender/receiver indices.")

    score_df = score_df.dropna(subset=["sender_index", "receiver_index"]).copy()
    score_df["sender_index"] = score_df["sender_index"].astype(int)
    score_df["receiver_index"] = score_df["receiver_index"].astype(int)

    for c in program_cols:
        score_df[c] = pd.to_numeric(score_df[c], errors="coerce")

    # Detect whether indices are Julia 1-based and convert to Python 0-based.
    n_cells = int(adata_list[TEST_SAMPLE_INDEX].n_obs)
    idx_min = int(score_df[["sender_index", "receiver_index"]].min().min())
    idx_max = int(score_df[["sender_index", "receiver_index"]].max().max())

    index_base = "unknown"
    if idx_min >= 1 and idx_max <= n_cells:
        index_base = "1-based; converted to 0-based"
        score_df["sender_index"] -= 1
        score_df["receiver_index"] -= 1
    elif idx_min >= 0 and idx_max < n_cells:
        index_base = "0-based; no conversion"
    else:
        index_base = f"unexpected index range before conversion: min={idx_min}, max={idx_max}, n_cells={n_cells}"

    print("Detected ScCChain index base:", index_base)
    print("n_cells:", n_cells)


    # ----------------------------
    # 3) Basic score statistics
    # ----------------------------
    program_stat_rows = []
    for c in program_cols:
        vals = score_df[c].to_numpy(dtype=float)
        finite = np.isfinite(vals)
        vals_finite = vals[finite]
        program_stat_rows.append({
            "program": c,
            "n_finite": int(finite.sum()),
            "n_nan_or_inf": int((~finite).sum()),
            "min": float(np.nanmin(vals_finite)) if vals_finite.size else np.nan,
            "max": float(np.nanmax(vals_finite)) if vals_finite.size else np.nan,
            "mean": float(np.nanmean(vals_finite)) if vals_finite.size else np.nan,
            "std": float(np.nanstd(vals_finite)) if vals_finite.size else np.nan,
            "nonzero_fraction": float(np.mean(vals_finite != 0)) if vals_finite.size else np.nan,
        })

    program_stats_df = pd.DataFrame(program_stat_rows)
    print("Program score summary:")
    display(program_stats_df)

    if not np.all(np.isfinite(score_df[program_cols].to_numpy(dtype=float))):
        print("[Warning] Some program values are NaN/Inf before downstream fillna.")


    # ----------------------------
    # 4) Compare ScCChain graph edges with SpiderNet edge_index
    # ----------------------------
    spn_edge_index = _get_spidernet_edge_index_e2(TEST_SAMPLE_INDEX)
    E_spn = int(spn_edge_index.shape[0])
    E_scc_raw = int(score_df.shape[0])

    sc_pairs = pd.DataFrame({
        "sender_index": score_df["sender_index"].to_numpy(dtype=np.int64),
        "receiver_index": score_df["receiver_index"].to_numpy(dtype=np.int64),
    }).drop_duplicates()

    spn_pairs = pd.DataFrame(spn_edge_index, columns=["sender_index", "receiver_index"]).drop_duplicates()

    sc_index = pd.MultiIndex.from_frame(sc_pairs[["sender_index", "receiver_index"]])
    spn_index = pd.MultiIndex.from_frame(spn_pairs[["sender_index", "receiver_index"]])

    common_index = sc_index.intersection(spn_index)
    sc_overlap = len(common_index) / max(len(sc_index), 1)
    spn_coverage = len(common_index) / max(len(spn_index), 1)

    overlap_summary = pd.DataFrame([{
        "sample_index": TEST_SAMPLE_INDEX,
        "sample_id": get_sample_id(adata_list[TEST_SAMPLE_INDEX], TEST_SAMPLE_INDEX),
        "n_cells": n_cells,
        "SpiderNet_edges_E": E_spn,
        "ScCChain_rows_raw": E_scc_raw,
        "ScCChain_unique_pairs": len(sc_index),
        "SpiderNet_unique_pairs": len(spn_index),
        "common_unique_pairs": len(common_index),
        "ScCChain_pairs_overlapping_SpiderNet_fraction": sc_overlap,
        "SpiderNet_pairs_covered_by_ScCChain_fraction": spn_coverage,
    }])

    print("Pair overlap summary:")
    display(overlap_summary)

    if spn_coverage < TEST_REQUIRE_MIN_SPN_COVERAGE:
        print(
            f"[Warning] SpiderNet edge coverage by ScCChain is low: {spn_coverage:.3f}. "
            "This may be expected only if ScCChain build_cell_graph uses a substantially different graph."
        )
    if sc_overlap < TEST_REQUIRE_MIN_SC_OVERLAP:
        print(
            f"[Warning] ScCChain-to-SpiderNet pair overlap is low: {sc_overlap:.3f}. "
            "Check whether radius / coordinates / index conversion are correct."
        )


    # ----------------------------
    # 5) Simulate Cell 6 alignment to SpiderNet edge_index
    # ----------------------------
    ref_pairs = pd.DataFrame(spn_edge_index, columns=["sender_index", "receiver_index"])
    ref_index = pd.MultiIndex.from_frame(ref_pairs[["sender_index", "receiver_index"]])

    sc_indexed = (
        score_df
        .drop_duplicates(["sender_index", "receiver_index"], keep="first")
        .set_index(["sender_index", "receiver_index"])
    )

    aligned_df = sc_indexed.reindex(ref_index)[program_cols]
    matched_mask = aligned_df.notna().any(axis=1).to_numpy()
    aligned_mat_raw = aligned_df.fillna(0.0).to_numpy(dtype=np.float32, copy=False)

    # Same column-wise nonnegative normalization style used by Cell 6 helper.
    aligned_mat = np.nan_to_num(aligned_mat_raw, nan=0.0, posinf=0.0, neginf=0.0)
    aligned_mat[aligned_mat < 0] = 0.0
    colmax = np.nanmax(aligned_mat, axis=0)
    colmax[~np.isfinite(colmax)] = 0.0
    colmax[colmax <= 0] = 1.0
    aligned_mat_norm = (aligned_mat / colmax.reshape(1, -1)).astype(np.float32, copy=False)

    alignment_summary = pd.DataFrame([{
        "aligned_matrix_shape": str(aligned_mat_norm.shape),
        "expected_shape": str((E_spn, len(program_cols))),
        "n_spidernet_edges_matched_to_sccchain": int(matched_mask.sum()),
        "n_spidernet_edges_missing_from_sccchain": int((~matched_mask).sum()),
        "matched_fraction": float(matched_mask.mean()),
        "aligned_nonzero_fraction_after_norm": float(np.mean(aligned_mat_norm != 0)),
        "aligned_min_after_norm": float(np.min(aligned_mat_norm)),
        "aligned_max_after_norm": float(np.max(aligned_mat_norm)),
    }])

    print("Cell-6-style alignment summary:")
    display(alignment_summary)

    if aligned_mat_norm.shape != (E_spn, len(program_cols)):
        raise ValueError(
            f"Aligned matrix shape {aligned_mat_norm.shape} != expected {(E_spn, len(program_cols))}"
        )

    print("[OK] First sample ScCChain output can be read and aligned to SpiderNet edges.")


    # ----------------------------
    # 6) Show matched/missing examples
    # ----------------------------
    ref_pairs_check = ref_pairs.copy()
    ref_pairs_check["matched_in_sccchain"] = matched_mask

    print("Example SpiderNet edges matched in ScCChain:")
    display(ref_pairs_check.loc[ref_pairs_check["matched_in_sccchain"]].head(10))

    print("Example SpiderNet edges missing from ScCChain:")
    display(ref_pairs_check.loc[~ref_pairs_check["matched_in_sccchain"]].head(10))


    # ----------------------------
    # 7) Optional quick diagnostic plot
    # ----------------------------
    if TEST_PLOT_HIST:
        row_sum = aligned_mat_norm.sum(axis=1)
        plt.figure(figsize=(4.2, 3.0))
        plt.hist(row_sum, bins=60)
        plt.xlabel("Aligned ScCChain row-sum score across programs")
        plt.ylabel("Number of SpiderNet edges")
        plt.title("First sample ScCChain aligned scores")
        plt.tight_layout()
        plt.show()
else:
    print("ScCChain cached SMD scan is being reused; edge-alignment diagnostics are unnecessary.")


#### Scan, select and plot the CCC comparison

The original incoming aggregation, high/low split, pooled-standard-deviation
SMD, one-sided Mann-Whitney test and feature selection are retained. All cells
contribute to the statistics; the earlier violin plotting subsample is not used.
The final plot contains SpiderNet MI-4, COMMOT and scCChain in the original style.


In [ ]:
%%fig6d sources
method_feature_names = {}
method_feature_dims = {}
method_feature_getters = {}
method_load_errors = {}
cached_method_scan_dfs = {}
cached_strength_summary_dfs = {}
method_cache_sources = {}

for method in METHODS_TO_RUN:
    cached_scan_df, cached_strength_df, cache_reason = try_load_cached_method_scan(method)
    if cached_scan_df is not None:
        cached_method_scan_dfs[method] = cached_scan_df
        cached_strength_summary_dfs[method] = cached_strength_df
        method_cache_sources[method] = cache_reason
        continue

    try:
        print("\n==============================")
        print("Preparing method:", method)
        print(f"Cache status for {method}: {cache_reason}")
        if method == "SpiderNet":
            names, getter = prepare_spidernet_source()
            names, getter, dims = restrict_spidernet_to_fixed_mi_if_needed(names, getter)
        elif method == "NMF-LR":
            names, getter = prepare_nmflr_source()
            dims = list(range(1, len(names) + 1))
        elif method == "COMMOT":
            names, getter = prepare_commot_source()
            dims = list(range(1, len(names) + 1))
        elif method == "ScCChain":
            names, getter = prepare_sccchain_source()
            dims = list(range(1, len(names) + 1))
        elif method == "Spacia":
            names, getter = prepare_spacia_source()
            dims = list(range(1, len(names) + 1))
        else:
            raise ValueError(f"Unknown method: {method}")
        # Lightweight validation on the first slice.
        first_arr = getter(0)
        if first_arr.shape[0] != edge_counts[0]:
            raise ValueError(f"{method}: first edge-feature rows {first_arr.shape[0]} != graph edges {edge_counts[0]}")
        if first_arr.shape[1] != len(names):
            raise ValueError(f"{method}: first edge-feature cols {first_arr.shape[1]} != feature names {len(names)}")
        if len(dims) != len(names):
            raise ValueError(f"{method}: len(feature_dims) != len(feature_names)")
        method_feature_names[method] = names
        method_feature_dims[method] = dims
        method_feature_getters[method] = getter
        print(f"{method}: ready; n_features={len(names)}; feature_dims={dims[:5]}{'...' if len(dims) > 5 else ''}; first shape={first_arr.shape}")
        del first_arr
        gc.collect()
    except Exception as e:
        method_load_errors[method] = repr(e)
        if method in REQUIRE_METHODS_TO_LOAD or not SKIP_MISSING_BASELINES:
            raise RuntimeError(f"{method} was requested but could not be prepared. Original error: {repr(e)}") from e
        print(f"[Warning] Skipping {method}: {e}")

method_order_use = [m for m in METHODS_TO_RUN if (m in cached_method_scan_dfs) or (m in method_feature_getters)]
print("\nCached methods:", list(cached_method_scan_dfs.keys()))
print("Prepared methods:", list(method_feature_getters.keys()))
print("Methods to include:", method_order_use)
if method_load_errors:
    print("Method load errors:")
    for m, err in method_load_errors.items():
        print(f"  {m}: {err}")


In [ ]:
%%fig6d scan
# ============================================================
# 7. Scan all communication axes and compute CancerSEA SMD
#    Robust version with detailed diagnostics
# ============================================================

import gc
import json
import traceback
import numpy as np
import pandas as pd

# -----------------------------
# Helper: eval filter
# -----------------------------
def eval_filter_for_method(method):
    return SPIDERNET_EVAL_FILTER if method == "SpiderNet" else BASELINE_EVAL_FILTER


# -----------------------------
# Helper: validate global objects before scanning
# -----------------------------
def _check_required_globals_for_smd_scan():
    required_names = [
        "tumor_cancersea_df",
        "adata_list",
        "spidernet_data_list",
        "method_order_use",
        "method_feature_names",
        "method_feature_getters",
        "method_feature_dims",
        "cached_method_scan_dfs",
        "cached_strength_summary_dfs",
        "method_cache_sources",
        "PROGRAM_ORDER",
        "CANCERTYPE_ORDER",
        "SPIDERNET_EVAL_FILTER",
        "BASELINE_EVAL_FILTER",
        "INCOMING_AGG",
        "SPLIT_RULE",
        "GROUPING_SOURCE",
        "SENDER_CELLTYPE",
        "RECEIVER_TUMOR_SUFFIX",
        "MODULE_SCORE_METHOD",
        "MIN_CELLS_PER_GROUP",
        "OUT_DIR",
    ]

    missing = [x for x in required_names if x not in globals()]
    if len(missing) > 0:
        raise NameError(
            "The following required objects are missing before SMD scan:\n"
            + "\n".join([f"  - {x}" for x in missing])
            + "\n\nPlease run the upstream setup/cache-loading cells first."
        )

    if len(method_order_use) == 0:
        raise ValueError(
            "`method_order_use` is empty. No method will be scanned.\n"
            "Please check the upstream method-loading cell."
        )

    print("============================================================")
    print("SMD scan preflight check")
    print("============================================================")
    print("Number of tumor cells in tumor_cancersea_df:", tumor_cancersea_df.shape[0])
    print("Number of samples in adata_list:", len(adata_list))
    print("Number of samples in spidernet_data_list:", len(spidernet_data_list))
    print("method_order_use:", list(method_order_use))
    print("cached_method_scan_dfs keys:", list(cached_method_scan_dfs.keys()))
    print("method_feature_names keys:", list(method_feature_names.keys()))
    print("method_feature_getters keys:", list(method_feature_getters.keys()))
    print("method_feature_dims keys:", list(method_feature_dims.keys()))
    print("CONTINUE_ON_METHOD_FAILURE:", CONTINUE_ON_METHOD_FAILURE)
    print("============================================================")


# -----------------------------
# Main scan function
# -----------------------------
def scan_one_method(method, feature_names, getter, feature_dims=None):
    K = len(feature_names)

    if feature_dims is None:
        feature_dims = list(range(1, K + 1))

    if len(feature_dims) != K:
        raise ValueError(
            f"{method}: len(feature_dims)={len(feature_dims)} "
            f"but len(feature_names)={K}"
        )

    if K == 0:
        raise ValueError(f"{method}: feature_names is empty.")

    n_rows = tumor_cancersea_df.shape[0]
    strength_all = np.full((n_rows, K), np.nan, dtype=np.float32)

    # Fill feature values in exactly the same row order as tumor_cancersea_df.
    for sample_index, idx in tumor_cancersea_df.groupby("sample_index", sort=True).groups.items():
        sample_index = int(sample_index)

        if sample_index % 10 == 0:
            print(
                f"[{method}] aggregating incoming strengths "
                f"sample {sample_index + 1}/{len(adata_list)}"
            )

        idx_arr = np.asarray(list(idx), dtype=np.int64)
        selected_cell_indices = tumor_cancersea_df.iloc[idx_arr]["cell_index"].to_numpy(dtype=np.int64)

        adata = adata_list[sample_index]
        data_obj = spidernet_data_list[sample_index]

        edge_index = edge_index_to_e2(get_field(data_obj, "edge_index"))
        celltypes = get_celltypes(adata, data_obj)

        edge_features = getter(sample_index)

        if edge_features is None:
            raise ValueError(f"{method} sample {sample_index}: getter returned None.")

        edge_features = np.asarray(edge_features)

        if edge_features.ndim == 1:
            edge_features = edge_features.reshape(-1, 1)

        if edge_features.shape[0] != edge_index.shape[0]:
            raise ValueError(
                f"{method} sample {sample_index}: "
                f"feature rows {edge_features.shape[0]} != edges {edge_index.shape[0]}"
            )

        if edge_features.shape[1] != K:
            raise ValueError(
                f"{method} sample {sample_index}: "
                f"feature cols {edge_features.shape[1]} != K={K}. "
                f"Feature names length may not match loaded edge feature matrix."
            )

        incoming_sel = aggregate_incoming_for_selected_receivers(
            edge_features=edge_features,
            edge_index_e2=edge_index,
            celltypes=celltypes,
            selected_cell_indices=selected_cell_indices,
            n_cells=adata.n_obs,
            agg=INCOMING_AGG,
        )

        if incoming_sel.shape != (len(idx_arr), K):
            raise ValueError(
                f"{method} sample {sample_index}: incoming_sel shape "
                f"{incoming_sel.shape} != expected {(len(idx_arr), K)}"
            )

        strength_all[idx_arr, :] = incoming_sel

        del edge_features, incoming_sel
        gc.collect()

    rows = []
    eval_filter = eval_filter_for_method(method)

    scores_by_program = {
        program: pd.to_numeric(
            tumor_cancersea_df[f"{program}_score"],
            errors="coerce",
        ).to_numpy(dtype=float)
        for program in PROGRAM_ORDER
    }

    for j, feature_name in enumerate(feature_names):
        values = strength_all[:, j].astype(float)
        feature_dim_actual = int(feature_dims[j])

        for cancer_type in CANCERTYPE_ORDER:
            ct_mask = tumor_cancersea_df["CancerType"].astype(str).eq(cancer_type).to_numpy()

            if eval_filter == "positive_incoming":
                eval_mask = ct_mask & np.isfinite(values) & (values > 0)
            elif eval_filter == "all_finite":
                eval_mask = ct_mask & np.isfinite(values)
            else:
                raise ValueError(f"Unknown eval_filter={eval_filter}")

            high_mask, low_mask, threshold = build_high_low_masks(
                values,
                eval_mask,
                split_rule=SPLIT_RULE,
            )

            for program in PROGRAM_ORDER:
                stats = compute_smd_from_groups(
                    scores_by_program[program][high_mask],
                    scores_by_program[program][low_mask],
                    min_cells_per_group=MIN_CELLS_PER_GROUP,
                )

                rows.append({
                    "Method": method,
                    "Feature_Dim": feature_dim_actual,
                    "Feature_Name": feature_name,
                    "Feature_Local_Index": j + 1,
                    "CancerType": cancer_type,
                    "Program": program,
                    "SMD": stats["SMD_high_minus_low"],
                    "Threshold": threshold,
                    "Threshold_Mode": "median",
                    "Grouping_Rule": SPLIT_RULE,
                    "Grouping_Source": GROUPING_SOURCE,
                    "Incoming_Agg": INCOMING_AGG,
                    "Eval_Filter": eval_filter,
                    "Sender_Celltype": SENDER_CELLTYPE,
                    "Receiver_Celltype_Filter": f"*{RECEIVER_TUMOR_SUFFIX}",
                    "module_score_method": MODULE_SCORE_METHOD,
                    **stats,
                })

    strength_summary = []

    for j, feature_name in enumerate(feature_names):
        vals = strength_all[:, j].astype(float)

        strength_summary.append({
            "Method": method,
            "Feature_Dim": int(feature_dims[j]),
            "Feature_Name": feature_name,
            "Feature_Local_Index": j + 1,
            "n_finite": int(np.isfinite(vals).sum()),
            "n_positive": int((np.isfinite(vals) & (vals > 0)).sum()),
            "mean_strength": float(np.nanmean(vals)) if np.isfinite(vals).any() else np.nan,
            "median_strength": float(np.nanmedian(vals)) if np.isfinite(vals).any() else np.nan,
        })

    return pd.DataFrame(rows), pd.DataFrame(strength_summary)


# -----------------------------
# Run scan / load cache
# -----------------------------
_check_required_globals_for_smd_scan()
OUT_DIR.mkdir(parents=True, exist_ok=True)
METHOD_CACHE_DIR.mkdir(parents=True, exist_ok=True)

all_scan_parts = []
strength_summary_parts = []
method_scan_errors = {}

for method in method_order_use:
    print("\n==============================")
    print("Scanning/loading method:", method)

    # --------------------------------------------------------
    # 1) Use cached scan if available
    # --------------------------------------------------------
    if method in cached_method_scan_dfs:
        scan_df = cached_method_scan_dfs[method].copy()
        strength_summary_df = cached_strength_summary_dfs.get(method, pd.DataFrame()).copy()

        print(
            f"[{method}] Using cached scan "
            f"({method_cache_sources.get(method, 'cache')}); rows={scan_df.shape[0]}"
        )

        if scan_df.empty:
            method_scan_errors[method] = (
                "Cached scan_df exists but is empty. "
                "Please delete this method cache or force recompute."
            )
            print(f"[Warning] {method}: cached scan_df is empty and will be skipped.")
            continue

        all_scan_parts.append(scan_df)

        if not strength_summary_df.empty:
            strength_summary_parts.append(strength_summary_df)

        continue

    # --------------------------------------------------------
    # 2) Validate method configuration before computing
    # --------------------------------------------------------
    missing_config = []

    if method not in method_feature_names:
        missing_config.append("method_feature_names")

    if method not in method_feature_getters:
        missing_config.append("method_feature_getters")

    if len(missing_config) > 0:
        msg = (
            f"{method}: missing configuration in {missing_config}. "
            f"Available method_feature_names keys={list(method_feature_names.keys())}; "
            f"available method_feature_getters keys={list(method_feature_getters.keys())}."
        )

        method_scan_errors[method] = msg
        print(f"[Warning] {msg}")

        if not CONTINUE_ON_METHOD_FAILURE:
            raise RuntimeError(msg)

        continue

    # --------------------------------------------------------
    # 3) Compute scan
    # --------------------------------------------------------
    try:
        scan_df, strength_summary_df = scan_one_method(
            method=method,
            feature_names=method_feature_names[method],
            feature_dims=method_feature_dims.get(method),
            getter=method_feature_getters[method],
        )

        if scan_df.empty:
            raise ValueError(f"{method}: scan_one_method returned an empty scan_df.")

        # Append results before writing cache so a cache-write problem does not
        # make a successfully computed method disappear from this run.
        all_scan_parts.append(scan_df)

        if not strength_summary_df.empty:
            strength_summary_parts.append(strength_summary_df)

        try:
            save_method_scan_cache(method, scan_df, strength_summary_df)
        except Exception as cache_e:
            cache_tb = traceback.format_exc()
            method_scan_errors[f"{method}__cache_save"] = cache_tb
            print(f"[Warning] {method} finished scanning but cache saving failed.")
            print(cache_tb)

        print(f"[{method}] Finished scan successfully; rows={scan_df.shape[0]}")

    except Exception as e:
        tb = traceback.format_exc()
        method_scan_errors[method] = tb

        print(f"[Warning] {method} failed during SMD scanning and will be skipped.")
        print("Original error:")
        print(tb)

        if not CONTINUE_ON_METHOD_FAILURE:
            raise RuntimeError(
                f"{method} failed during SMD scanning. Original error: {repr(e)}"
            ) from e

        gc.collect()


# -----------------------------
# Stop with detailed error if no method succeeded
# -----------------------------
if len(all_scan_parts) == 0:
    error_out = OUT_DIR / "Pancancer_CCC_method_scan_errors.json"

    with open(error_out, "w", encoding="utf-8") as f:
        json.dump(method_scan_errors, f, indent=2)

    print("\n============================================================")
    print("No method produced or loaded an SMD scan.")
    print("Saved detailed method errors to:")
    print(error_out)
    print("============================================================")

    print("Methods requested:")
    print(list(method_order_use))

    print("\nCached scan keys:")
    print(list(cached_method_scan_dfs.keys()))

    print("\nFeature-name keys:")
    print(list(method_feature_names.keys()))

    print("\nGetter keys:")
    print(list(method_feature_getters.keys()))

    print("\nMethod-specific errors:")
    for m, err in method_scan_errors.items():
        print("\n------------------------------")
        print(m)
        print("------------------------------")
        print(err)

    raise RuntimeError(
        "No method produced or loaded an SMD scan. "
        f"See detailed errors above and in {error_out}"
    )


# -----------------------------
# Concatenate results
# -----------------------------
all_scan_df = pd.concat(all_scan_parts, ignore_index=True)

method_strength_summary_df = (
    pd.concat(strength_summary_parts, ignore_index=True)
    if len(strength_summary_parts) > 0
    else pd.DataFrame()
)

# Rank axes within each Method x Program x CancerType. Larger positive SMD is better.
all_scan_df["SMD_rank_within_panel"] = np.nan

for (method, program, cancer_type), idx in all_scan_df.groupby(
    ["Method", "Program", "CancerType"]
).groups.items():
    all_scan_df.loc[idx, "SMD_rank_within_panel"] = all_scan_df.loc[idx, "SMD"].rank(
        method="average",
        ascending=False,
        na_option="keep",
    )

# -----------------------------
# Save outputs
# -----------------------------
scan_out = OUT_DIR / "Pancancer_CCC_all_axis_Fibroblast_to_tumor_CancerSEA_SMD_scan.csv"
strength_out = OUT_DIR / "Pancancer_CCC_all_axis_incoming_strength_summary.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)
scan_out.parent.mkdir(parents=True, exist_ok=True)
strength_out.parent.mkdir(parents=True, exist_ok=True)

all_scan_df.to_csv(scan_out, index=False)
method_strength_summary_df.to_csv(strength_out, index=False)

if method_scan_errors:
    error_out = OUT_DIR / "Pancancer_CCC_method_scan_errors.json"
    with open(error_out, "w", encoding="utf-8") as f:
        json.dump(method_scan_errors, f, indent=2)
    print(f"Saved method scan errors: {error_out}")

print(f"Saved all-axis SMD scan: {scan_out}")
print(f"Saved incoming-strength summary: {strength_out}")
print("Methods included in all_scan_df:", sorted(all_scan_df["Method"].astype(str).unique()))

display(all_scan_df.head())

In [ ]:
%%fig6d select
# Require all three CCC methods before selecting or exporting a comparison.
_required_methods = {"SpiderNet", "COMMOT", "ScCChain"}
_available_methods = set(all_scan_df["Method"].astype(str))
if not _required_methods.issubset(_available_methods):
    raise RuntimeError(f"Fig. 6d methods are missing: {sorted(_required_methods - _available_methods)}")

# ============================================================
# 8. Select one feature per method and build final SMD table
# ============================================================

def select_feature_for_method(method, df_method):
    if method == "SpiderNet":
        fixed = SPIDERNET_FIXED_FEATURE_NAME
        sub = df_method[df_method["Feature_Name"].astype(str).eq(fixed)].copy()
        if sub.empty:
            raise ValueError(f"SpiderNet fixed feature {fixed} was not found.")
        return {
            "Method": method,
            "Selected_Feature_Name": fixed,
            "Selected_Feature_Dim": int(sub["Feature_Dim"].iloc[0]),
            "Selection_Mode": "fixed_SpiderNet_MI4",
            "mean_SMD": float(np.nanmean(sub["SMD"])),
            "mean_rank": float(np.nanmean(sub["SMD_rank_within_panel"])),
            "n_panels_available": int(sub["SMD"].notna().sum()),
        }

    rows = []
    expected_panels = len(PROGRAM_ORDER) * len(CANCERTYPE_ORDER)
    for (fdim, fname), sub in df_method.groupby(["Feature_Dim", "Feature_Name"], sort=False):
        n_avail = int(sub["SMD"].notna().sum())
        if REQUIRE_ALL_PANELS_FOR_FEATURE_SELECTION and n_avail < expected_panels:
            continue
        rows.append({
            "Feature_Dim": int(fdim),
            "Feature_Name": str(fname),
            "mean_SMD": float(np.nanmean(sub["SMD"])),
            "median_SMD": float(np.nanmedian(sub["SMD"])),
            "mean_rank": float(np.nanmean(sub["SMD_rank_within_panel"])),
            "n_panels_available": n_avail,
        })
    summary = pd.DataFrame(rows)
    if summary.empty:
        raise ValueError(f"No eligible feature dimensions for {method}.")
    if BASELINE_FEATURE_SELECTION_MODE == "mean_rank_across_all_panels":
        summary = summary.sort_values(["mean_rank", "mean_SMD"], ascending=[True, False])
        mode = BASELINE_FEATURE_SELECTION_MODE
    elif BASELINE_FEATURE_SELECTION_MODE == "mean_smd_across_all_panels":
        summary = summary.sort_values(["mean_SMD", "mean_rank"], ascending=[False, True])
        mode = BASELINE_FEATURE_SELECTION_MODE
    else:
        raise ValueError(f"Unknown BASELINE_FEATURE_SELECTION_MODE={BASELINE_FEATURE_SELECTION_MODE}")
    best = summary.iloc[0]
    return {
        "Method": method,
        "Selected_Feature_Name": str(best["Feature_Name"]),
        "Selected_Feature_Dim": int(best["Feature_Dim"]),
        "Selection_Mode": mode,
        "mean_SMD": float(best["mean_SMD"]),
        "mean_rank": float(best["mean_rank"]),
        "n_panels_available": int(best["n_panels_available"]),
    }


selected_rows = []
method_order_available = [m for m in method_order_use if m in set(all_scan_df["Method"].astype(str))]
for method in method_order_available:
    selected_rows.append(select_feature_for_method(method, all_scan_df[all_scan_df["Method"].astype(str).eq(str(method))].copy()))

selected_feature_summary_df = pd.DataFrame(selected_rows)

selected_parts = []
for _, row in selected_feature_summary_df.iterrows():
    method = row["Method"]
    feature_name = row["Selected_Feature_Name"]
    sub = all_scan_df[
        all_scan_df["Method"].eq(method)
        & all_scan_df["Feature_Name"].astype(str).eq(str(feature_name))
    ].copy()
    sub["Selected_Feature_Name"] = feature_name
    sub["Selected_Feature_Dim"] = int(row["Selected_Feature_Dim"])
    sub["Selection_Mode"] = row["Selection_Mode"]
    selected_parts.append(sub)

if len(selected_parts) == 0:
    raise RuntimeError("No selected SMD rows were generated. Check all_scan_df and method selection errors.")
selected_smd_df = pd.concat(selected_parts, ignore_index=True)
selected_smd_df["Method"] = pd.Categorical(selected_smd_df["Method"], categories=method_order_available, ordered=True)
selected_smd_df["Program"] = pd.Categorical(selected_smd_df["Program"], categories=PROGRAM_ORDER, ordered=True)
selected_smd_df["CancerType"] = pd.Categorical(selected_smd_df["CancerType"], categories=CANCERTYPE_ORDER, ordered=True)
selected_smd_df = selected_smd_df.sort_values(["Program", "CancerType", "Method"]).reset_index(drop=True)

selected_summary_out = OUT_DIR / "Pancancer_CCC_selected_feature_summary.csv"
selected_smd_out = OUT_DIR / "Pancancer_CCC_selected_feature_Fibroblast_to_tumor_CancerSEA_SMD.csv"
selected_feature_summary_df.to_csv(selected_summary_out, index=False)
selected_smd_df.to_csv(selected_smd_out, index=False)

print(f"Saved selected feature summary: {selected_summary_out}")
print(f"Saved selected-feature SMD table: {selected_smd_out}")
display(selected_feature_summary_df)
display(selected_smd_df.head())


In [ ]:
%%fig6d plot
# ============================================================
# 9. Final grouped barplot: rows = programs; x-axis grouped by cancer type
#    Display the three CCC methods without bar annotations
#    No repeated cross-panel axes; highlight highest bar within each cancer-type group
# ============================================================

plot_df = selected_smd_df.copy()
plot_df["SMD"] = pd.to_numeric(plot_df["SMD"], errors="coerce")

# Remove NMF-LR from plotting
plot_df = plot_df[~plot_df["Method"].astype(str).eq("NMF-LR")].copy()

method_order_plot = [
    m for m in method_order_available
    if m != "NMF-LR" and m in set(plot_df["Method"].astype(str))
]

method_fill_colors = {
    "SpiderNet": "#e1b6a7",
    "COMMOT": "#b9cec7",
    "ScCChain": "#a6a2b9",
    "Spacia": "#F28E2B",
}

method_edge_colors = {
    "SpiderNet": "#a03c39",
    "COMMOT": "#529384",
    "ScCChain": "#656592",
    "Spacia": "#A65E1B",
}

def get_method_fill_color(m):
    return method_fill_colors.get(str(m), "#d1d3d4")

def get_method_edge_color(m):
    return method_edge_colors.get(str(m), "#6b6b6b")

def get_ylim_from_values(y):
    y = np.asarray(y, dtype=float)
    finite_y = y[np.isfinite(y)]

    if finite_y.size == 0:
        return (-1.0, 1.0)

    y_min = min(0.0, float(np.nanmin(finite_y)))
    y_max = max(0.0, float(np.nanmax(finite_y)))

    if y_min == y_max:
        base = abs(y_max) if y_max != 0 else 1.0
        return (-1.15 * base, 1.15 * base)

    span = y_max - y_min
    pad = 0.18 * span
    return (y_min - pad, y_max + pad)


# ---------------------------
# Layout settings
# ---------------------------
n_rows = len(PROGRAM_ORDER)
n_cancers = len(CANCERTYPE_ORDER)
n_methods = len(method_order_plot)

bar_width = 0.72
within_group_step = 1.0
between_group_gap = 0.6

group_width = (n_methods - 1) * within_group_step if n_methods > 1 else 0.0
group_step = group_width + between_group_gap + 1.0

group_centers = np.arange(n_cancers, dtype=float) * group_step
method_offsets = (
    np.arange(n_methods, dtype=float) - (n_methods - 1) / 2.0
) * within_group_step

x_positions_by_cancer = {
    cancer_type: group_centers[j] + method_offsets
    for j, cancer_type in enumerate(CANCERTYPE_ORDER)
}

x_min = group_centers[0] + method_offsets[0] - 0.75 if n_methods > 0 else -0.75
x_max = group_centers[-1] + method_offsets[-1] + 0.75 if n_methods > 0 else 0.75

fig_width = max(10.5, 1.35 * n_cancers + 0.38 * n_cancers * n_methods)
fig_height = 1.85 * n_rows + 1.35

plt.close("all")
fig, axes = plt.subplots(
    n_rows,
    1,
    figsize=(fig_width, fig_height),
    sharex=True,
    sharey=False,
)

if n_rows == 1:
    axes = np.asarray([axes])

for i, program in enumerate(PROGRAM_ORDER):
    ax = axes[i]

    row_values = plot_df.loc[
        plot_df["Program"].astype(str).eq(program)
        & plot_df["CancerType"].astype(str).isin(CANCERTYPE_ORDER)
        & plot_df["Method"].astype(str).isin(method_order_plot),
        "SMD"
    ].to_numpy(dtype=float)

    ax.set_ylim(get_ylim_from_values(row_values))

    for j, cancer_type in enumerate(CANCERTYPE_ORDER):
        sub = plot_df[
            plot_df["Program"].astype(str).eq(program)
            & plot_df["CancerType"].astype(str).eq(cancer_type)
        ].copy()

        sub = sub.set_index("Method").reindex(method_order_plot).reset_index()

        x = x_positions_by_cancer[cancer_type]
        y = pd.to_numeric(sub["SMD"], errors="coerce").to_numpy(dtype=float)

        fill_colors = [get_method_fill_color(m) for m in method_order_plot]
        edge_colors = [get_method_edge_color(m) for m in method_order_plot]

        bars = ax.bar(
            x,
            y,
            width=bar_width,
            color=fill_colors,
            edgecolor=edge_colors,
            linewidth=0.7,
            zorder=3,
        )

        # Highlight the highest bar within this cancer-type group.
        finite_mask = np.isfinite(y)
        if finite_mask.any():
            y_max = np.nanmax(y)
            for bar, yi in zip(bars, y):
                if np.isfinite(yi) and np.isclose(yi, y_max, rtol=1e-8, atol=1e-10):
                    bar.set_linewidth(1.8)
                else:
                    bar.set_linewidth(0.7)

        # Add subtle vertical separator between cancer-type groups.
        if j < n_cancers - 1:
            sep_x = (group_centers[j] + group_centers[j + 1]) / 2.0
            ax.axvline(sep_x, color="#D9D9D9", linewidth=0.5, zorder=1)

    ax.axhline(0, color="black", linewidth=0.55, zorder=2)
    ax.set_xlim(x_min, x_max)

    ax.grid(axis="y", color="#D9D9D9", linewidth=0.35, alpha=0.8, zorder=0)
    ax.set_axisbelow(True)

    ax.set_ylabel(f"{program}\nSMD", fontsize=8)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    if i < n_rows - 1:
        ax.tick_params(axis="x", labelbottom=False)
    else:
        ax.set_xticks(group_centers)
        ax.set_xticklabels(CANCERTYPE_ORDER, rotation=35, ha="right", fontsize=8)

# ---------------------------
# Method legend
# ---------------------------
legend_handles = [
    mpl.patches.Patch(
        facecolor=get_method_fill_color(m),
        edgecolor=get_method_edge_color(m),
        linewidth=0.9,
        label=m,
    )
    for m in method_order_plot
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    ncol=max(1, len(method_order_plot)),
    frameon=False,
    fontsize=8,
    handlelength=1.2,
    columnspacing=1.0,
)

fig.suptitle(
    "Fibroblast→tumor incoming communication strength vs CancerSEA program scores",
    y=1.035,
    fontsize=10,
)

fig.tight_layout(rect=[0, 0, 1, 0.92], h_pad=0.8)

plot_pdf = OUT_DIR / "Pancancer_CCC_selected_feature_CancerSEA_SMD_3x8_barplot_no_NMFLR_no_annotation.pdf"
plot_png = OUT_DIR / "Pancancer_CCC_selected_feature_CancerSEA_SMD_3x8_barplot_no_NMFLR_no_annotation.png"

fig.savefig(plot_pdf, bbox_inches="tight")
fig.savefig(plot_png, bbox_inches="tight")

print(f"Saved final PDF: {plot_pdf}")
print(f"Saved final PNG: {plot_png}")

plt.show()

In [ ]:
# Keep the final tables and paths, then release the isolated workflow state.
fig6d_results = {
    "selected_features": fig6d_state["selected_feature_summary_df"].copy(),
    "selected_smd": fig6d_state["selected_smd_df"].copy(),
    "output_dir": fig6d_state["OUT_DIR"],
    "pdf": fig6d_state["plot_pdf"],
    "png": fig6d_state["plot_png"],
}
fig6d_state.clear()
_fig6d_gc.collect()
_fig6d_display(fig6d_results["selected_features"])
print("Fig. 6d PDF:", fig6d_results["pdf"])
print("Fig. 6d PNG:", fig6d_results["png"])


### Edge-level MI-4 distributions and ranked cell-type pairs

The edge-distribution plot supports S24b; the median-based cell-type-pair stem
plot supports S24a. The high-MI4 fibroblast-to-tumor fraction and mean-based stem
plot provide additional views and use different summary quantities.


In [ ]:
# ============================================================
# Additional MI-4 summary
# Edge-level Fibroblast -> tumor cell MI-4 strength distribution
# across cancer types
# ------------------------------------------------------------
# y-axis:
#   Edge-level Fibroblast -> tumor MI-4 strength
#
# x-axis:
#   Cancer type, defined by receiver tumor-cell label:
#       Breast-cancercell -> Breast
#       Liver-cancercell  -> Liver
#       ...
#
# Output:
#   MI4_fibro_to_tumor_edgelevel_strength_by_cancertype_violin_boxplot.pdf/png
# ============================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# -----------------------------
# Parameters
# -----------------------------
MI_OI = "MI4"
MI_OI_INDEX = 3  # Factor_envir_use[:, 3] -> MI-4 if MI-1 is column 0

SENDER_CELLTYPE = "Fibroblast"
RECEIVER_TUMOR_SUFFIX = "-cancercell"

PLOT_MAX_EDGES_PER_CANCERTYPE = 30000
PLOT_RANDOM_STATE = 2026

VIOLIN_FILL_COLOR = "#f8b129"
VIOLIN_EDGE_COLOR = "#b7b9bb"
BOX_EDGE_COLOR = "black"
VIOLIN_ALPHA = 0.82
BOX_WIDTH = 0.18

# Output directory
if "mi4_program_outdir" not in globals():
    mi4_program_outdir = Path(run_dirs["run_dir"]) / "MI4_fibroblast_to_tumor_celllevel_CancerSEA"
    mi4_program_outdir.mkdir(parents=True, exist_ok=True)

# Cancer-type order
if "CANCERTYPE_ORDER" not in globals():
    CANCERTYPE_ORDER = [
        "Breast",
        "Colon",
        "Liver",
        "Lung",
        "Melanoma",
        "Ovarian",
        "Prostate",
        "Uterine",
    ]

# -----------------------------
# Helper functions
# -----------------------------
def _to_numpy_local(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def _get_field_local(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    return obj[key]


def _has_field_local(obj, key):
    if isinstance(obj, dict):
        return key in obj
    if hasattr(obj, key):
        return True
    try:
        obj[key]
        return True
    except Exception:
        return False


def _edge_index_to_numpy_local(edge_index_obj):
    edge_index = _to_numpy_local(edge_index_obj).astype(np.int64, copy=False)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(
        f"edge_index should have shape [E, 2] or [2, E], got {edge_index.shape}"
    )


def _edge_count_from_data_local(data_obj):
    edge_index = _edge_index_to_numpy_local(_get_field_local(data_obj, "edge_index"))
    return int(edge_index.shape[0])


def _get_celltypes_local(adata, data_obj):
    if adata is not None and hasattr(adata, "obs") and "celltype_final" in adata.obs.columns:
        return adata.obs["celltype_final"].astype(str).to_numpy()

    if _has_field_local(data_obj, "cell_class"):
        return _to_numpy_local(_get_field_local(data_obj, "cell_class")).astype(str)

    raise KeyError(
        "Cannot find celltype labels. Expected adata.obs['celltype_final'] "
        "or data_obj['cell_class']."
    )


def _get_sample_id_local(adata, sample_index):
    if adata is not None and hasattr(adata, "obs"):
        sample_col_candidates = [
            "SampleID", "sample_id", "sample", "samples", "Sample",
            "slice", "slide", "library_id"
        ]

        for col in sample_col_candidates:
            if col in adata.obs.columns:
                vals = pd.Series(adata.obs[col]).dropna().astype(str).unique()
                if len(vals) > 0:
                    return str(vals[0])

    return f"sample_{sample_index}"


def _extract_cancer_type_from_tumor_celltype(celltype):
    s = str(celltype)
    if RECEIVER_TUMOR_SUFFIX not in s:
        return None
    return s.replace(RECEIVER_TUMOR_SUFFIX, "").strip(" -_")


def _get_processed_adata_list_and_data_list():
    """
    Robustly get adata_list and spidernet_data from the current notebook.
    """
    if "processed" in globals() and processed is not None:
        data_list = processed.spidernet_data

        if hasattr(processed, "adata_list"):
            adata_list_use = processed.adata_list
        elif "adata_list" in globals():
            adata_list_use = adata_list
        else:
            adata_list_use = [None] * len(data_list)

        return adata_list_use, data_list

    if "adata_list" in globals() and "spidernet_data" in globals():
        return adata_list, spidernet_data

    raise NameError(
        "Cannot find `processed` or (`adata_list` and `spidernet_data`) in memory."
    )


# -----------------------------
# Load processed data handles and Factor_envir_use
# -----------------------------
adata_list_use, spidernet_data_list_use = _get_processed_adata_list_and_data_list()

factor_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"

if not factor_path.exists():
    raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {factor_path}")

factor_mmap = np.load(factor_path, mmap_mode="r")

edge_counts = np.asarray(
    [_edge_count_from_data_local(spidernet_data_list_use[i]) for i in range(len(spidernet_data_list_use))],
    dtype=np.int64,
)

edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))
n_total_edges = int(edge_offsets[-1])

# Ensure edge x MI orientation.
if factor_mmap.ndim != 2:
    raise ValueError(f"Factor_envir_use must be 2D, got shape {factor_mmap.shape}")

if factor_mmap.shape[0] == n_total_edges:
    factor_edge_by_mi = factor_mmap
elif factor_mmap.shape[1] == n_total_edges:
    raise ValueError(
        "Factor_envir_use appears to be MI x edge. "
        "This plotting cell expects edge x MI. Please transpose before using."
    )
else:
    raise ValueError(
        f"Factor_envir_use shape {factor_mmap.shape} is not aligned with "
        f"total edge count {n_total_edges}."
    )

if MI_OI_INDEX >= factor_edge_by_mi.shape[1]:
    raise ValueError(
        f"MI_OI_INDEX={MI_OI_INDEX} is out of range for Factor_envir_use with "
        f"{factor_edge_by_mi.shape[1]} MI columns."
    )

# -----------------------------
# Collect edge-level Fibroblast -> tumor MI-4 strengths
# -----------------------------
edge_records = []

for sample_index, data_obj in enumerate(spidernet_data_list_use):
    if sample_index % 10 == 0:
        print(f"Processing sample {sample_index + 1}/{len(spidernet_data_list_use)}")

    adata = adata_list_use[sample_index] if sample_index < len(adata_list_use) else None

    edge_index = _edge_index_to_numpy_local(_get_field_local(data_obj, "edge_index"))
    celltypes = _get_celltypes_local(adata, data_obj)

    if edge_index.shape[0] != edge_counts[sample_index]:
        raise ValueError(
            f"Edge count mismatch for sample {sample_index}: "
            f"{edge_index.shape[0]} vs expected {edge_counts[sample_index]}"
        )

    start = int(edge_offsets[sample_index])
    end = int(edge_offsets[sample_index + 1])

    mi4_edge_strength = np.asarray(
        factor_edge_by_mi[start:end, MI_OI_INDEX],
        dtype=np.float32,
    )

    src = edge_index[:, 0]
    dst = edge_index[:, 1]

    sender_ct = celltypes[src].astype(str)
    receiver_ct = celltypes[dst].astype(str)

    is_fibro_to_tumor = (
        (sender_ct == SENDER_CELLTYPE)
        & np.char.endswith(receiver_ct.astype(str), RECEIVER_TUMOR_SUFFIX)
    )

    if not np.any(is_fibro_to_tumor):
        continue

    edge_local_idx = np.where(is_fibro_to_tumor)[0].astype(np.int64)
    receiver_tumor_ct = receiver_ct[edge_local_idx].astype(str)

    cancer_type = np.asarray([
        _extract_cancer_type_from_tumor_celltype(x)
        for x in receiver_tumor_ct
    ], dtype=object)

    valid = pd.notna(cancer_type)
    if valid.sum() == 0:
        continue

    edge_local_idx = edge_local_idx[valid]
    receiver_tumor_ct = receiver_tumor_ct[valid]
    cancer_type = cancer_type[valid].astype(str)

    sample_id = _get_sample_id_local(adata, sample_index)

    edge_records.append(pd.DataFrame({
        "sample_index": int(sample_index),
        "sample_id": sample_id,
        "edge_local_index": edge_local_idx,
        "edge_global_index": start + edge_local_idx,
        "sender_cell_index": src[edge_local_idx],
        "receiver_cell_index": dst[edge_local_idx],
        "Sender": SENDER_CELLTYPE,
        "Receiver": receiver_tumor_ct,
        "CancerType": cancer_type,
        "MI": MI_OI,
        "MI4_edge_strength": mi4_edge_strength[edge_local_idx],
    }))

if len(edge_records) == 0:
    raise ValueError("No Fibroblast -> tumor-cell edges were found.")

mi4_fibro_to_tumor_edgelevel_df = pd.concat(edge_records, axis=0, ignore_index=True)

mi4_fibro_to_tumor_edgelevel_df["MI4_edge_strength"] = pd.to_numeric(
    mi4_fibro_to_tumor_edgelevel_df["MI4_edge_strength"],
    errors="coerce",
)

mi4_fibro_to_tumor_edgelevel_df = mi4_fibro_to_tumor_edgelevel_df.loc[
    mi4_fibro_to_tumor_edgelevel_df["CancerType"].notna()
    & np.isfinite(mi4_fibro_to_tumor_edgelevel_df["MI4_edge_strength"].to_numpy(dtype=float))
].copy()

# Apply cancer-type order.
cancer_type_order_use = [
    ct for ct in CANCERTYPE_ORDER
    if ct in set(mi4_fibro_to_tumor_edgelevel_df["CancerType"].astype(str))
]

extra_ct = sorted(
    set(mi4_fibro_to_tumor_edgelevel_df["CancerType"].astype(str))
    - set(cancer_type_order_use)
)

cancer_type_order_use = cancer_type_order_use + extra_ct

mi4_fibro_to_tumor_edgelevel_df["CancerType"] = pd.Categorical(
    mi4_fibro_to_tumor_edgelevel_df["CancerType"].astype(str),
    categories=cancer_type_order_use,
    ordered=True,
)

# -----------------------------
# Save full edge-level table and summary statistics
# -----------------------------
edge_full_out = mi4_program_outdir / "MI4_fibro_to_tumor_edgelevel_strength_by_cancertype_full.csv"
mi4_fibro_to_tumor_edgelevel_df.to_csv(edge_full_out, index=False)

mi4_edge_summary_df = (
    mi4_fibro_to_tumor_edgelevel_df
    .groupby("CancerType", observed=True)
    .agg(
        n_edges=("MI4_edge_strength", "size"),
        n_samples=("sample_id", "nunique"),
        mean_MI4=("MI4_edge_strength", "mean"),
        median_MI4=("MI4_edge_strength", "median"),
        q25_MI4=("MI4_edge_strength", lambda x: np.nanquantile(x, 0.25)),
        q75_MI4=("MI4_edge_strength", lambda x: np.nanquantile(x, 0.75)),
        max_MI4=("MI4_edge_strength", "max"),
    )
    .reset_index()
)

summary_out = mi4_program_outdir / "MI4_fibro_to_tumor_edgelevel_strength_by_cancertype_summary.csv"
mi4_edge_summary_df.to_csv(summary_out, index=False)

print(f"Saved full edge-level table: {edge_full_out}")
print(f"Saved summary table: {summary_out}")
display(mi4_edge_summary_df)

# -----------------------------
# Downsample for plotting only
# -----------------------------
if PLOT_MAX_EDGES_PER_CANCERTYPE is not None:
    mi4_edge_plot_df = (
        mi4_fibro_to_tumor_edgelevel_df
        .groupby("CancerType", observed=True, group_keys=False)
        .apply(
            lambda x: x.sample(
                n=min(len(x), int(PLOT_MAX_EDGES_PER_CANCERTYPE)),
                random_state=PLOT_RANDOM_STATE,
            )
        )
        .reset_index(drop=True)
    )
else:
    mi4_edge_plot_df = mi4_fibro_to_tumor_edgelevel_df.copy()

plotting_table_out = mi4_program_outdir / (
    f"MI4_fibro_to_tumor_edgelevel_strength_by_cancertype_plotting_table_"
    f"max{PLOT_MAX_EDGES_PER_CANCERTYPE}.csv"
)
mi4_edge_plot_df.to_csv(plotting_table_out, index=False)
print(f"Saved plotting table: {plotting_table_out}")

# -----------------------------
# Plot style
# -----------------------------
plt.close("all")
plt.style.use("default")

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

sns.set_theme(style="white", context="paper")

# -----------------------------
# Plot violin + boxplot
# -----------------------------
fig_width = max(6.5, 0.75 * len(cancer_type_order_use) + 2.2)
fig_height = 3.2

plt.close("all")
fig, ax = plt.subplots(
    figsize=(fig_width, fig_height),
    facecolor="white",
)

sns.violinplot(
    data=mi4_edge_plot_df,
    x="CancerType",
    y="MI4_edge_strength",
    order=cancer_type_order_use,
    color=VIOLIN_FILL_COLOR,
    inner=None,
    cut=0,
    linewidth=0.55,
    saturation=1,
    ax=ax,
)

# Violin alpha and edge color.
for coll in ax.collections:
    try:
        coll.set_alpha(VIOLIN_ALPHA)
        coll.set_edgecolor(VIOLIN_EDGE_COLOR)
        coll.set_linewidth(0.55)
    except Exception:
        pass

sns.boxplot(
    data=mi4_edge_plot_df,
    x="CancerType",
    y="MI4_edge_strength",
    order=cancer_type_order_use,
    width=BOX_WIDTH,
    showfliers=False,
    showcaps=True,
    boxprops={
        "facecolor": "white",
        "edgecolor": BOX_EDGE_COLOR,
        "linewidth": 0.65,
        "alpha": 0.88,
    },
    whiskerprops={"color": BOX_EDGE_COLOR, "linewidth": 0.65},
    capprops={"color": BOX_EDGE_COLOR, "linewidth": 0.65},
    medianprops={"color": BOX_EDGE_COLOR, "linewidth": 0.85},
    ax=ax,
)

# Add n-edge labels.
y_values = mi4_edge_plot_df["MI4_edge_strength"].to_numpy(dtype=float)
y_min = np.nanmin(y_values)
y_max = np.nanmax(y_values)
y_span = max(y_max - y_min, 1e-6)

for i, cancer_type in enumerate(cancer_type_order_use):
    n_cur = int(
        mi4_edge_summary_df.loc[
            mi4_edge_summary_df["CancerType"].astype(str) == str(cancer_type),
            "n_edges"
        ].iloc[0]
    )

    y_annot = (
        mi4_edge_plot_df.loc[
            mi4_edge_plot_df["CancerType"].astype(str) == str(cancer_type),
            "MI4_edge_strength"
        ].max()
        + 0.035 * y_span
    )

    ax.text(
        i,
        y_annot,
        f"n={n_cur:,}",
        ha="center",
        va="bottom",
        fontsize=6.8,
        color="black",
        rotation=90,
    )

ax.set_xlabel("")
ax.set_ylabel(f"Edge-level Fibroblast→tumor {MI_OI} strength", fontsize=9)
ax.set_title(
    f"Fibroblast→tumor cell {MI_OI} edge strength across cancer types",
    fontsize=10,
    pad=8,
)

ax.set_xticklabels(
    cancer_type_order_use,
    rotation=45,
    ha="right",
    fontsize=8,
    color="black",
)

ax.set_ylim(y_min - 0.03 * y_span, y_max + 0.18 * y_span)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(0.6)
    ax.spines[spine].set_color("black")

ax.tick_params(axis="both", width=0.6, length=2.5, colors="black")
ax.grid(axis="y", color="#E6E6E6", linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

plot_stem = "MI4_fibro_to_tumor_edgelevel_strength_by_cancertype_violin_boxplot"

pdf_out = mi4_program_outdir / f"{plot_stem}.pdf"
png_out = mi4_program_outdir / f"{plot_stem}.png"

fig.savefig(pdf_out, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(png_out, dpi=300, bbox_inches="tight", facecolor="white")

print(f"Saved PDF: {pdf_out}")
print(f"Saved PNG: {png_out}")

plt.show()

# Optional cleanup
del factor_mmap, factor_edge_by_mi
gc.collect()

In [ ]:
# ============================================================
# Additional MI-4 summary
# Fraction of high-MI4 edges that are Fibroblast -> tumor cell
# across cancer types
# ------------------------------------------------------------
# For each cancer type:
#   denominator = all directed edges with MI-4 strength > 0.5
#   numerator   = among these high-MI4 edges, edges where
#                 sender = Fibroblast and receiver = *-cancercell
#
# y-axis:
#   Proportion of Fibroblast -> tumor edges among MI4-high edges
#
# Output:
#   MI4_high_edges_fibro_to_tumor_fraction_by_cancertype_barplot.pdf/png
#   MI4_high_edges_fibro_to_tumor_fraction_by_sample.csv
#   MI4_high_edges_fibro_to_tumor_fraction_by_cancertype_summary.csv
# ============================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# -----------------------------
# Parameters
# -----------------------------
MI_OI = "MI4"
MI_OI_INDEX = 3  # Factor_envir_use[:, 3] -> MI-4 if MI-1 is column 0
MI4_HIGH_THRESHOLD = 0.5

SENDER_CELLTYPE = "Fibroblast"
RECEIVER_TUMOR_SUFFIX = "-cancercell"

# If True, numerator requires the receiver tumor label to match the inferred sample cancer type.
# Usually this should be True for this pan-cancer dataset.
REQUIRE_RECEIVER_TUMOR_MATCH_SAMPLE_CANCERTYPE = True

BAR_FILL_COLOR = "#f8b129"
BAR_EDGE_COLOR = "#b7b9bb"
BAR_ALPHA = 0.90

# Output directory
if "mi4_program_outdir" not in globals():
    mi4_program_outdir = Path(run_dirs["run_dir"]) / "MI4_fibroblast_to_tumor_celllevel_CancerSEA"
    mi4_program_outdir.mkdir(parents=True, exist_ok=True)

# Cancer-type order
if "CANCERTYPE_ORDER" not in globals():
    CANCERTYPE_ORDER = [
        "Breast",
        "Colon",
        "Liver",
        "Lung",
        "Melanoma",
        "Ovarian",
        "Prostate",
        "Uterine",
    ]

# -----------------------------
# Helper functions
# -----------------------------
def _to_numpy_local(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def _get_field_local(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    return obj[key]


def _has_field_local(obj, key):
    if isinstance(obj, dict):
        return key in obj
    if hasattr(obj, key):
        return True
    try:
        obj[key]
        return True
    except Exception:
        return False


def _edge_index_to_numpy_local(edge_index_obj):
    edge_index = _to_numpy_local(edge_index_obj).astype(np.int64, copy=False)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(
        f"edge_index should have shape [E, 2] or [2, E], got {edge_index.shape}"
    )


def _edge_count_from_data_local(data_obj):
    edge_index = _edge_index_to_numpy_local(_get_field_local(data_obj, "edge_index"))
    return int(edge_index.shape[0])


def _get_celltypes_local(adata, data_obj):
    if adata is not None and hasattr(adata, "obs") and "celltype_final" in adata.obs.columns:
        return adata.obs["celltype_final"].astype(str).to_numpy()

    if _has_field_local(data_obj, "cell_class"):
        return _to_numpy_local(_get_field_local(data_obj, "cell_class")).astype(str)

    raise KeyError(
        "Cannot find celltype labels. Expected adata.obs['celltype_final'] "
        "or data_obj['cell_class']."
    )


def _get_sample_id_local(adata, sample_index):
    if adata is not None and hasattr(adata, "obs"):
        sample_col_candidates = [
            "SampleID", "sample_id", "sample", "samples", "Sample",
            "slice", "slide", "library_id",
        ]

        for col in sample_col_candidates:
            if col in adata.obs.columns:
                vals = pd.Series(adata.obs[col]).dropna().astype(str).unique()
                if len(vals) > 0:
                    return str(vals[0])

    return f"sample_{sample_index}"


def _extract_cancer_type_from_tumor_celltype(celltype):
    s = str(celltype)
    if RECEIVER_TUMOR_SUFFIX not in s:
        return None
    return s.replace(RECEIVER_TUMOR_SUFFIX, "").strip(" -_")


def _infer_sample_cancer_type_local(adata, data_obj, celltypes, sample_index):
    """
    Infer cancer type for one sample/sub-slice.

    Primary rule:
      Use the tumor-cell label present in celltype_final, e.g.
      Liver-cancercell -> Liver.

    Fallback:
      Search common adata.obs metadata columns or sample_id string.
    """
    celltype_s = pd.Series(celltypes).astype(str)
    tumor_celltypes = celltype_s[celltype_s.str.endswith(RECEIVER_TUMOR_SUFFIX, na=False)]

    if len(tumor_celltypes) > 0:
        inferred = (
            tumor_celltypes
            .map(_extract_cancer_type_from_tumor_celltype)
            .dropna()
            .astype(str)
        )
        if len(inferred) > 0:
            return inferred.value_counts().idxmax()

    if adata is not None and hasattr(adata, "obs"):
        cancer_col_candidates = [
            "CancerType", "cancer_type", "Cancer_Type", "cancer",
            "tumor_type", "TumorType", "tissue", "Tissue",
            "sample", "Sample", "SampleID", "sample_id", "library_id",
        ]

        for col in cancer_col_candidates:
            if col in adata.obs.columns:
                vals = pd.Series(adata.obs[col]).dropna().astype(str)
                if len(vals) == 0:
                    continue

                vals_unique = vals.unique().tolist()

                # Exact match to CANCERTYPE_ORDER
                for val in vals_unique:
                    for ct in CANCERTYPE_ORDER:
                        if str(val).lower() == str(ct).lower():
                            return ct

                # Substring match to CANCERTYPE_ORDER
                joined_vals = " ".join(vals_unique).lower()
                for ct in CANCERTYPE_ORDER:
                    if str(ct).lower() in joined_vals:
                        return ct

    sample_id = _get_sample_id_local(adata, sample_index)
    sample_id_lower = str(sample_id).lower()
    for ct in CANCERTYPE_ORDER:
        if str(ct).lower() in sample_id_lower:
            return ct

    return f"Unknown_sample_{sample_index}"


def _get_processed_adata_list_and_data_list():
    """
    Robustly get adata_list and spidernet_data from the current notebook.
    """
    if "processed" in globals() and processed is not None:
        data_list = processed.spidernet_data

        if hasattr(processed, "adata_list"):
            adata_list_use = processed.adata_list
        elif "adata_list" in globals():
            adata_list_use = adata_list
        else:
            adata_list_use = [None] * len(data_list)

        return adata_list_use, data_list

    if "adata_list" in globals() and "spidernet_data" in globals():
        return adata_list, spidernet_data

    raise NameError(
        "Cannot find `processed` or (`adata_list` and `spidernet_data`) in memory."
    )


# -----------------------------
# Load processed data handles and Factor_envir_use
# -----------------------------
adata_list_use, spidernet_data_list_use = _get_processed_adata_list_and_data_list()

factor_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"

if not factor_path.exists():
    raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {factor_path}")

factor_mmap = np.load(factor_path, mmap_mode="r")

edge_counts = np.asarray(
    [
        _edge_count_from_data_local(spidernet_data_list_use[i])
        for i in range(len(spidernet_data_list_use))
    ],
    dtype=np.int64,
)

edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))
n_total_edges = int(edge_offsets[-1])

# Ensure edge x MI orientation.
if factor_mmap.ndim != 2:
    raise ValueError(f"Factor_envir_use must be 2D, got shape {factor_mmap.shape}")

if factor_mmap.shape[0] == n_total_edges:
    factor_edge_by_mi = factor_mmap
elif factor_mmap.shape[1] == n_total_edges:
    # Allow MI x edge orientation as a memory-efficient view.
    factor_edge_by_mi = factor_mmap.T
else:
    raise ValueError(
        f"Factor_envir_use shape {factor_mmap.shape} is not aligned with "
        f"total edge count {n_total_edges}."
    )

if MI_OI_INDEX >= factor_edge_by_mi.shape[1]:
    raise ValueError(
        f"MI_OI_INDEX={MI_OI_INDEX} is out of range for Factor_envir_use with "
        f"{factor_edge_by_mi.shape[1]} MI columns."
    )

# -----------------------------
# Count MI4-high edges and Fibroblast -> tumor subset
# -----------------------------
sample_fraction_records = []

for sample_index, data_obj in enumerate(spidernet_data_list_use):
    if sample_index % 10 == 0:
        print(f"Processing sample {sample_index + 1}/{len(spidernet_data_list_use)}")

    adata = adata_list_use[sample_index] if sample_index < len(adata_list_use) else None

    edge_index = _edge_index_to_numpy_local(_get_field_local(data_obj, "edge_index"))
    celltypes = _get_celltypes_local(adata, data_obj)

    if edge_index.shape[0] != edge_counts[sample_index]:
        raise ValueError(
            f"Edge count mismatch for sample {sample_index}: "
            f"{edge_index.shape[0]} vs expected {edge_counts[sample_index]}"
        )

    sample_id = _get_sample_id_local(adata, sample_index)
    sample_cancer_type = _infer_sample_cancer_type_local(
        adata=adata,
        data_obj=data_obj,
        celltypes=celltypes,
        sample_index=sample_index,
    )

    start = int(edge_offsets[sample_index])
    end = int(edge_offsets[sample_index + 1])

    mi4_edge_strength = np.asarray(
        factor_edge_by_mi[start:end, MI_OI_INDEX],
        dtype=np.float32,
    )

    src = edge_index[:, 0]
    dst = edge_index[:, 1]

    sender_ct = celltypes[src].astype(str)
    receiver_ct = celltypes[dst].astype(str)

    receiver_is_tumor = (
        pd.Series(receiver_ct)
        .astype(str)
        .str.endswith(RECEIVER_TUMOR_SUFFIX, na=False)
        .to_numpy()
    )

    receiver_cancer_type = np.asarray(
        [
            _extract_cancer_type_from_tumor_celltype(x)
            if is_tumor else None
            for x, is_tumor in zip(receiver_ct, receiver_is_tumor)
        ],
        dtype=object,
    )

    is_mi4_high = np.isfinite(mi4_edge_strength) & (mi4_edge_strength > MI4_HIGH_THRESHOLD)

    is_fibro_to_tumor = (
        (sender_ct == SENDER_CELLTYPE)
        & receiver_is_tumor
    )

    if REQUIRE_RECEIVER_TUMOR_MATCH_SAMPLE_CANCERTYPE:
        is_fibro_to_tumor = (
            is_fibro_to_tumor
            & (receiver_cancer_type.astype(str) == str(sample_cancer_type))
        )

    is_fibro_to_tumor_and_mi4_high = is_mi4_high & is_fibro_to_tumor

    n_total_edges = int(edge_index.shape[0])
    n_mi4_high_edges = int(is_mi4_high.sum())
    n_fibro_to_tumor_mi4_high_edges = int(is_fibro_to_tumor_and_mi4_high.sum())

    fraction_fibro_to_tumor_among_mi4_high = (
        n_fibro_to_tumor_mi4_high_edges / n_mi4_high_edges
        if n_mi4_high_edges > 0
        else np.nan
    )

    sample_fraction_records.append({
        "sample_index": int(sample_index),
        "sample_id": sample_id,
        "CancerType": sample_cancer_type,
        "MI": MI_OI,
        "MI4_high_threshold": MI4_HIGH_THRESHOLD,
        "n_total_edges": n_total_edges,
        "n_MI4_high_edges": n_mi4_high_edges,
        "n_Fibroblast_to_tumor_MI4_high_edges": n_fibro_to_tumor_mi4_high_edges,
        "fraction_Fibroblast_to_tumor_among_MI4_high_edges": fraction_fibro_to_tumor_among_mi4_high,
    })

mi4_high_edge_fraction_by_sample_df = pd.DataFrame(sample_fraction_records)

if mi4_high_edge_fraction_by_sample_df.empty:
    raise ValueError("No samples were processed. Please check spidernet_data_list_use.")

sample_out = mi4_program_outdir / "MI4_high_edges_fibro_to_tumor_fraction_by_sample.csv"
mi4_high_edge_fraction_by_sample_df.to_csv(sample_out, index=False)
print(f"Saved sample-level fraction table: {sample_out}")

# -----------------------------
# Summarize by cancer type
# -----------------------------
mi4_high_edge_fraction_by_cancertype_df = (
    mi4_high_edge_fraction_by_sample_df
    .groupby("CancerType", observed=True)
    .agg(
        n_samples=("sample_id", "nunique"),
        n_total_edges=("n_total_edges", "sum"),
        n_MI4_high_edges=("n_MI4_high_edges", "sum"),
        n_Fibroblast_to_tumor_MI4_high_edges=(
            "n_Fibroblast_to_tumor_MI4_high_edges", "sum"
        ),
        mean_sample_fraction_Fibroblast_to_tumor_among_MI4_high_edges=(
            "fraction_Fibroblast_to_tumor_among_MI4_high_edges", "mean"
        ),
        median_sample_fraction_Fibroblast_to_tumor_among_MI4_high_edges=(
            "fraction_Fibroblast_to_tumor_among_MI4_high_edges", "median"
        ),
    )
    .reset_index()
)

mi4_high_edge_fraction_by_cancertype_df[
    "fraction_Fibroblast_to_tumor_among_MI4_high_edges"
] = (
    mi4_high_edge_fraction_by_cancertype_df[
        "n_Fibroblast_to_tumor_MI4_high_edges"
    ]
    / mi4_high_edge_fraction_by_cancertype_df["n_MI4_high_edges"].replace(0, np.nan)
)

# Apply cancer-type order.
cancer_type_order_use = [
    ct for ct in CANCERTYPE_ORDER
    if ct in set(mi4_high_edge_fraction_by_cancertype_df["CancerType"].astype(str))
]

extra_ct = sorted(
    set(mi4_high_edge_fraction_by_cancertype_df["CancerType"].astype(str))
    - set(cancer_type_order_use)
)

cancer_type_order_use = cancer_type_order_use + extra_ct

mi4_high_edge_fraction_by_cancertype_df["CancerType"] = pd.Categorical(
    mi4_high_edge_fraction_by_cancertype_df["CancerType"].astype(str),
    categories=cancer_type_order_use,
    ordered=True,
)

mi4_high_edge_fraction_by_cancertype_df = (
    mi4_high_edge_fraction_by_cancertype_df
    .sort_values("CancerType")
    .reset_index(drop=True)
)

summary_out = mi4_program_outdir / (
    "MI4_high_edges_fibro_to_tumor_fraction_by_cancertype_summary.csv"
)
mi4_high_edge_fraction_by_cancertype_df.to_csv(summary_out, index=False)
print(f"Saved cancer-type summary table: {summary_out}")

display(mi4_high_edge_fraction_by_cancertype_df)

# -----------------------------
# Plot style
# -----------------------------
plt.close("all")
plt.style.use("default")

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

sns.set_theme(style="white", context="paper")

# -----------------------------
# Barplot
# -----------------------------
plot_df = mi4_high_edge_fraction_by_cancertype_df.copy()
plot_df = plot_df.loc[
    plot_df["CancerType"].astype(str).isin(cancer_type_order_use)
].copy()

x = np.arange(len(cancer_type_order_use))

y = (
    plot_df
    .set_index(plot_df["CancerType"].astype(str))
    .loc[cancer_type_order_use, "fraction_Fibroblast_to_tumor_among_MI4_high_edges"]
    .to_numpy(dtype=float)
)

n_num = (
    plot_df
    .set_index(plot_df["CancerType"].astype(str))
    .loc[cancer_type_order_use, "n_Fibroblast_to_tumor_MI4_high_edges"]
    .to_numpy(dtype=float)
)

n_den = (
    plot_df
    .set_index(plot_df["CancerType"].astype(str))
    .loc[cancer_type_order_use, "n_MI4_high_edges"]
    .to_numpy(dtype=float)
)

fig_width = max(6.5, 0.75 * len(cancer_type_order_use) + 2.2)
fig_height = 3.2

plt.close("all")
fig, ax = plt.subplots(
    figsize=(fig_width, fig_height),
    facecolor="white",
)

bars = ax.bar(
    x,
    y,
    width=0.62,
    color=BAR_FILL_COLOR,
    edgecolor=BAR_EDGE_COLOR,
    linewidth=0.75,
    alpha=BAR_ALPHA,
)

# Add labels: percentage + numerator/denominator.
finite_y = y[np.isfinite(y)]
y_max = np.nanmax(finite_y) if len(finite_y) > 0 else 0.0
ylim_top = min(1.08, max(0.12, y_max + 0.16))

for i, (yi, num_i, den_i) in enumerate(zip(y, n_num, n_den)):
    if not np.isfinite(yi):
        label = "NA\n0/0"
        y_text = 0.01
    else:
        label = f"{yi * 100:.1f}%\n{int(num_i):,}/{int(den_i):,}"
        y_text = min(yi + 0.025, ylim_top - 0.025)

    ax.text(
        i,
        y_text,
        label,
        ha="center",
        va="bottom",
        fontsize=6.8,
        color="black",
        rotation=90,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    cancer_type_order_use,
    rotation=45,
    ha="right",
    fontsize=8,
    color="black",
)

ax.set_xlabel("")
ax.set_ylabel(
    f"Fibroblast→tumor edges / {MI_OI}> {MI4_HIGH_THRESHOLD:g} edges",
    fontsize=9,
)
ax.set_title(
    f"Fraction of {MI_OI}-high edges that are Fibroblast→tumor cell",
    fontsize=10,
    pad=8,
)

ax.set_ylim(0, ylim_top)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(0.6)
    ax.spines[spine].set_color("black")

ax.tick_params(axis="both", width=0.6, length=2.5, colors="black")
ax.grid(axis="y", color="#E6E6E6", linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

plot_stem = "MI4_high_edges_fibro_to_tumor_fraction_by_cancertype_barplot"

pdf_out = mi4_program_outdir / f"{plot_stem}.pdf"
png_out = mi4_program_outdir / f"{plot_stem}.png"

fig.savefig(pdf_out, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(png_out, dpi=300, bbox_inches="tight", facecolor="white")

print(f"Saved PDF: {pdf_out}")
print(f"Saved PNG: {png_out}")

plt.show()

# Optional cleanup
del factor_mmap, factor_edge_by_mi
gc.collect()

In [ ]:
# ============================================================
# Additional MI-4 summary
# Stem plot of top sender→receiver cell-type pairs by MI-4 activity
# ------------------------------------------------------------
# x-axis:
#   sender→receiver cell-type pair
#
# y-axis:
#   mean MI-4 interacting strength across all cancer types
#   i.e., full_metric_df["Activity"] for MI-4
#
# This is the same raw value used as dot size in dotplot_blocks.
#
# Output:
#   MI4_top30_sender_receiver_pair_mean_activity_stemplot.pdf/png
#   MI4_top30_sender_receiver_pair_mean_activity.csv
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# -----------------------------
# Parameters
# -----------------------------
MI_OI = "MI-4"
TOP_N_PAIRS = 30

# If True, restrict to pairs that were actually selected/shown in dotplot_blocks
# by the top-fraction + min-activity filter.
# If False, rank all eligible sender→receiver pairs in full_metric_df.
USE_SELECTED_DOTPLOT_PAIRS_ONLY = False

STEM_LINE_COLOR = "#b7b9bb"
MARKER_FILL_COLOR = "#f8b129"
MARKER_EDGE_COLOR = "black"
BASELINE_COLOR = "black"

# Highlight Fibroblast -> *-cancercell x-axis labels
FIBRO_TO_CANCERCELL_XTICK_COLOR = "#e93732"
DEFAULT_XTICK_COLOR = "black"

FIG_WIDTH = 10.5
FIG_HEIGHT = 4.2

# -----------------------------
# Helper
# -----------------------------
def _standardize_mi_name_local(x):
    """
    Convert MI4, MI-4, mi_4, 4 -> MI-4.
    """
    s = str(x).strip()
    digits = "".join([ch for ch in s if ch.isdigit()])
    if digits == "":
        return s
    return f"MI-{int(digits)}"


def _is_fibroblast_to_cancercell_pair(sender, receiver):
    """
    Return True for Fibroblast -> *-cancercell pairs.
    """
    sender_s = str(sender).strip().lower()
    receiver_s = str(receiver).strip().lower()

    return (
        sender_s == "fibroblast"
        and receiver_s.endswith("-cancercell")
    )


# -----------------------------
# Check required input from previous dotplot_blocks cell
# -----------------------------
if "full_metric_df" not in globals():
    raise NameError(
        "`full_metric_df` is not found. Please run the dotplot_blocks cell first, "
        "because this cell uses full_metric_df['Activity'], which defines the dot size."
    )

required_cols = {"MI", "Pair", "Sender", "Receiver", "Pair_edge_count", "Activity"}
missing_cols = required_cols - set(full_metric_df.columns)
if len(missing_cols) > 0:
    raise ValueError(
        f"`full_metric_df` is missing required columns: {sorted(missing_cols)}"
    )

# -----------------------------
# Select MI-4 pair activity table
# -----------------------------
metric_df = full_metric_df.copy()
metric_df["MI_standardized"] = metric_df["MI"].map(_standardize_mi_name_local)

mi_oi_std = _standardize_mi_name_local(MI_OI)

mi4_pair_activity_df = metric_df.loc[
    metric_df["MI_standardized"] == mi_oi_std
].copy()

if mi4_pair_activity_df.empty:
    raise ValueError(
        f"No rows found for {MI_OI}. Available MIs are: "
        f"{sorted(metric_df['MI_standardized'].dropna().unique().tolist())}"
    )

# Optional: restrict to selected dotplot pairs only.
if USE_SELECTED_DOTPLOT_PAIRS_ONLY:
    if "selected_pair_df" not in globals():
        raise NameError(
            "`selected_pair_df` is not found. Please run the dotplot_blocks cell first, "
            "or set USE_SELECTED_DOTPLOT_PAIRS_ONLY = False."
        )

    selected_tmp = selected_pair_df.copy()
    selected_tmp["MI_standardized"] = selected_tmp["MI"].map(_standardize_mi_name_local)

    selected_mi4_pairs = set(
        selected_tmp.loc[
            selected_tmp["MI_standardized"] == mi_oi_std,
            "Pair"
        ].astype(str)
    )

    mi4_pair_activity_df = mi4_pair_activity_df.loc[
        mi4_pair_activity_df["Pair"].astype(str).isin(selected_mi4_pairs)
    ].copy()

    if mi4_pair_activity_df.empty:
        raise ValueError(
            f"No selected dotplot pairs found for {MI_OI}. "
            "Try setting USE_SELECTED_DOTPLOT_PAIRS_ONLY = False."
        )

mi4_pair_activity_df["Activity"] = pd.to_numeric(
    mi4_pair_activity_df["Activity"],
    errors="coerce",
)

mi4_pair_activity_df = mi4_pair_activity_df.loc[
    np.isfinite(mi4_pair_activity_df["Activity"].to_numpy(dtype=float))
].copy()

mi4_pair_activity_df = (
    mi4_pair_activity_df
    .sort_values(
        ["Activity", "Pair_edge_count"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

mi4_pair_activity_df["Rank"] = np.arange(1, mi4_pair_activity_df.shape[0] + 1)

mi4_top_pair_activity_df = mi4_pair_activity_df.head(TOP_N_PAIRS).copy()

# Cleaner x-axis label.
mi4_top_pair_activity_df["Pair_label"] = (
    mi4_top_pair_activity_df["Sender"].astype(str)
    + "→"
    + mi4_top_pair_activity_df["Receiver"].astype(str)
)

# Mark Fibroblast -> *-cancercell pairs.
mi4_top_pair_activity_df["Is_Fibroblast_to_cancercell"] = [
    _is_fibroblast_to_cancercell_pair(sender, receiver)
    for sender, receiver in zip(
        mi4_top_pair_activity_df["Sender"],
        mi4_top_pair_activity_df["Receiver"],
    )
]

# -----------------------------
# Output directory
# -----------------------------
if "out_dir" in globals():
    stem_out_dir = Path(out_dir)
elif "run_dirs" in globals() and "run_dir" in run_dirs:
    stem_out_dir = Path(run_dirs["run_dir"]) / "MI4_pair_activity_stemplot"
else:
    stem_out_dir = Path(".") / "MI4_pair_activity_stemplot"

stem_out_dir.mkdir(parents=True, exist_ok=True)

table_out = stem_out_dir / "MI4_top30_sender_receiver_pair_mean_activity.csv"
mi4_top_pair_activity_df.to_csv(table_out, index=False)
print(f"Saved top-pair table: {table_out}")

display_cols = [
    "Rank",
    "MI",
    "Pair",
    "Sender",
    "Receiver",
    "Pair_edge_count",
    "Activity",
    "Is_Fibroblast_to_cancercell",
]

for optional_col in ["Max_normalized_score", "MI_contribution"]:
    if optional_col in mi4_top_pair_activity_df.columns:
        display_cols.append(optional_col)

display(mi4_top_pair_activity_df[display_cols])

# -----------------------------
# Plot style
# -----------------------------
plt.close("all")
plt.style.use("default")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# -----------------------------
# Stem plot
# -----------------------------
plot_df = mi4_top_pair_activity_df.copy()

x = np.arange(plot_df.shape[0])
y = plot_df["Activity"].to_numpy(dtype=float)
x_labels = plot_df["Pair_label"].astype(str).tolist()
highlight_flags = plot_df["Is_Fibroblast_to_cancercell"].to_numpy(dtype=bool)

fig, ax = plt.subplots(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    facecolor="white",
)

# Stem lines
ax.vlines(
    x,
    ymin=0,
    ymax=y,
    color=STEM_LINE_COLOR,
    linewidth=1.2,
    zorder=1,
)

# Points
ax.scatter(
    x,
    y,
    s=55,
    facecolor=MARKER_FILL_COLOR,
    edgecolor=MARKER_EDGE_COLOR,
    linewidth=0.65,
    zorder=3,
)

# Baseline
ax.axhline(
    0,
    color=BASELINE_COLOR,
    linewidth=0.8,
    zorder=0,
)

# No point-value annotation.

ax.set_xticks(x)
ax.set_xticklabels(
    x_labels,
    rotation=75,
    ha="right",
    fontsize=7.2,
)

# Highlight Fibroblast -> *-cancercell x-axis labels in red.
for tick_label, is_highlight in zip(ax.get_xticklabels(), highlight_flags):
    if is_highlight:
        tick_label.set_color(FIBRO_TO_CANCERCELL_XTICK_COLOR)
        tick_label.set_fontweight("bold")
    else:
        tick_label.set_color(DEFAULT_XTICK_COLOR)
        tick_label.set_fontweight("normal")

ax.set_xlabel("Sender→receiver cell-type pair", fontsize=9)
ax.set_ylabel("Mean MI-4 interacting strength", fontsize=9)

title_suffix = "selected dotplot pairs only" if USE_SELECTED_DOTPLOT_PAIRS_ONLY else "all eligible pairs"
ax.set_title(
    f"Top {TOP_N_PAIRS} sender→receiver pairs by mean MI-4 activity ({title_suffix})",
    fontsize=10,
    pad=8,
)

y_max = np.nanmax(y) if len(y) > 0 else 0.0

ax.set_xlim(-0.7, len(x) - 0.3)
ax.set_ylim(0, y_max * 1.08 if y_max > 0 else 1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(0.7)
    ax.spines[spine].set_color("black")

ax.tick_params(axis="both", width=0.7, length=3, colors="black")

# tick_params would reset tick label color, so re-apply x-label highlight after tick styling.
for tick_label, is_highlight in zip(ax.get_xticklabels(), highlight_flags):
    if is_highlight:
        tick_label.set_color(FIBRO_TO_CANCERCELL_XTICK_COLOR)
        tick_label.set_fontweight("bold")
    else:
        tick_label.set_color(DEFAULT_XTICK_COLOR)
        tick_label.set_fontweight("normal")

ax.grid(axis="y", color="#E6E6E6", linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

plot_stem = "MI4_top30_sender_receiver_pair_mean_activity_stemplot"

pdf_out = stem_out_dir / f"{plot_stem}.pdf"
png_out = stem_out_dir / f"{plot_stem}.png"

fig.savefig(pdf_out, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(png_out, dpi=300, bbox_inches="tight", facecolor="white")

print(f"Saved PDF: {pdf_out}")
print(f"Saved PNG: {png_out}")

plt.show()

In [ ]:
# ============================================================
# Additional MI-4 summary
# Stem plot of top sender→receiver cell-type pairs by MEDIAN MI-4 activity
# ------------------------------------------------------------
# x-axis:
#   sender→receiver cell-type pair
#
# y-axis:
#   median MI-4 interacting strength across all cancer types
#   computed directly from edge-level Factor_envir_use[:, MI4]
#
# Difference from dotplot_blocks:
#   dotplot_blocks dot size used mean MI activity.
#   This cell computes edge-level median MI-4 activity.
#
# Output:
#   MI4_top30_sender_receiver_pair_median_activity_stemplot.pdf/png
#   MI4_all_sender_receiver_pair_median_activity.csv
#   MI4_top30_sender_receiver_pair_median_activity.csv
# ============================================================

import gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import sparse

import matplotlib as mpl
import matplotlib.pyplot as plt

# -----------------------------
# Parameters
# -----------------------------
MI_OI = "MI-4"
MI_OI_INDEX = 3  # Factor_envir_use[:, 3] -> MI-4 if MI-1 is column 0
TOP_N_PAIRS = 20

PAIR_DELIM = "|||"

STEM_LINE_COLOR = "#b7b9bb"
MARKER_FILL_COLOR = "#f8b129"
MARKER_EDGE_COLOR = "black"
BASELINE_COLOR = "black"

# Highlight Fibroblast -> *-cancercell x-axis labels
FIBRO_TO_CANCERCELL_XTICK_COLOR = "#e93732"
DEFAULT_XTICK_COLOR = "black"

FIG_WIDTH = 10.5
FIG_HEIGHT = 4.2

# -----------------------------
# Helper functions
# -----------------------------
def _to_numpy_local(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def _get_field_local(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    return obj[key]


def _has_field_local(obj, key):
    if isinstance(obj, dict):
        return key in obj
    if hasattr(obj, key):
        return True
    try:
        obj[key]
        return True
    except Exception:
        return False


def _edge_index_to_numpy_local(edge_index_obj):
    edge_index = _to_numpy_local(edge_index_obj).astype(np.int64, copy=False)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(
        f"edge_index should have shape [E, 2] or [2, E], got {edge_index.shape}"
    )


def _edge_count_from_data_local(data_obj):
    edge_index = _edge_index_to_numpy_local(_get_field_local(data_obj, "edge_index"))
    return int(edge_index.shape[0])


def _get_celltypes_local(adata, data_obj):
    if adata is not None and hasattr(adata, "obs") and "celltype_final" in adata.obs.columns:
        return adata.obs["celltype_final"].astype(str).to_numpy()

    if _has_field_local(data_obj, "cell_class"):
        return _to_numpy_local(_get_field_local(data_obj, "cell_class")).astype(str)

    raise KeyError(
        "Cannot find celltype labels. Expected adata.obs['celltype_final'] "
        "or data_obj['cell_class']."
    )


def _get_processed_adata_list_and_data_list():
    """
    Robustly get adata_list and spidernet_data from the current notebook.
    """
    if "processed" in globals() and processed is not None:
        data_list = processed.spidernet_data

        if hasattr(processed, "adata_list"):
            adata_list_use = processed.adata_list
        elif "adata_list" in globals():
            adata_list_use = adata_list
        else:
            adata_list_use = [None] * len(data_list)

        return adata_list_use, data_list

    if "adata_list" in globals() and "spidernet_data" in globals():
        return adata_list, spidernet_data

    raise NameError(
        "Cannot find `processed` or (`adata_list` and `spidernet_data`) in memory."
    )


def _standardize_mi_name_local(x):
    """
    Convert MI4, MI-4, mi_4, 4 -> MI-4.
    """
    s = str(x).strip()
    digits = "".join([ch for ch in s if ch.isdigit()])
    if digits == "":
        return s
    return f"MI-{int(digits)}"


def _is_fibroblast_to_cancercell_pair(sender, receiver):
    """
    Return True for Fibroblast -> *-cancercell pairs.
    """
    sender_s = str(sender).strip().lower()
    receiver_s = str(receiver).strip().lower()

    return (
        sender_s == "fibroblast"
        and receiver_s.endswith("-cancercell")
    )


# -----------------------------
# Load processed data handles
# -----------------------------
adata_list_use, spidernet_data_list_use = _get_processed_adata_list_and_data_list()

edge_counts = np.asarray(
    [
        _edge_count_from_data_local(spidernet_data_list_use[i])
        for i in range(len(spidernet_data_list_use))
    ],
    dtype=np.int64,
)

edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))
n_total_edges_all = int(edge_offsets[-1])

# -----------------------------
# Load Factor_envir_use
# -----------------------------
if "Factor_envir_use" in globals():
    factor_obj = Factor_envir_use
else:
    if "run_dirs" not in globals() or "run_dir" not in run_dirs:
        raise NameError(
            "`Factor_envir_use` is not in memory and `run_dirs['run_dir']` is unavailable."
        )

    factor_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"

    if not factor_path.exists():
        raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {factor_path}")

    factor_obj = np.load(factor_path, mmap_mode="r")

factor_arr = _to_numpy_local(factor_obj)

if factor_arr.ndim != 2:
    raise ValueError(f"Factor_envir_use must be 2D, got shape {factor_arr.shape}")

# Ensure edge x MI orientation.
if factor_arr.shape[0] == n_total_edges_all:
    factor_edge_by_mi = factor_arr
elif factor_arr.shape[1] == n_total_edges_all:
    factor_edge_by_mi = factor_arr.T
else:
    raise ValueError(
        f"Factor_envir_use shape {factor_arr.shape} is not aligned with "
        f"total edge count {n_total_edges_all}."
    )

if MI_OI_INDEX >= factor_edge_by_mi.shape[1]:
    raise ValueError(
        f"MI_OI_INDEX={MI_OI_INDEX} is out of range for Factor_envir_use with "
        f"{factor_edge_by_mi.shape[1]} MI columns."
    )

# -----------------------------
# Collect edge-level MI-4 values by sender→receiver pair
# -----------------------------
pair_value_chunks = defaultdict(list)
pair_sender_receiver = {}

for sample_index, data_obj in enumerate(spidernet_data_list_use):
    if sample_index % 10 == 0:
        print(f"Processing sample {sample_index + 1}/{len(spidernet_data_list_use)}")

    adata = adata_list_use[sample_index] if sample_index < len(adata_list_use) else None

    edge_index = _edge_index_to_numpy_local(_get_field_local(data_obj, "edge_index"))
    celltypes = _get_celltypes_local(adata, data_obj)

    if edge_index.shape[0] != edge_counts[sample_index]:
        raise ValueError(
            f"Edge count mismatch for sample {sample_index}: "
            f"{edge_index.shape[0]} vs expected {edge_counts[sample_index]}"
        )

    start = int(edge_offsets[sample_index])
    end = int(edge_offsets[sample_index + 1])

    mi4_edge_strength = np.asarray(
        factor_edge_by_mi[start:end, MI_OI_INDEX],
        dtype=np.float32,
    )

    valid_edge = np.isfinite(mi4_edge_strength)

    if valid_edge.sum() == 0:
        continue

    src = edge_index[:, 0]
    dst = edge_index[:, 1]

    sender_ct = celltypes[src].astype(str)
    receiver_ct = celltypes[dst].astype(str)

    sender_valid = sender_ct[valid_edge]
    receiver_valid = receiver_ct[valid_edge]
    mi4_valid = mi4_edge_strength[valid_edge]

    pair_keys = np.asarray(
        [
            f"{s}{PAIR_DELIM}{r}"
            for s, r in zip(sender_valid, receiver_valid)
        ],
        dtype=object,
    )

    unique_pairs, inv = np.unique(pair_keys, return_inverse=True)

    for pair_i, pair_key in enumerate(unique_pairs):
        vals = mi4_valid[inv == pair_i]
        if vals.size == 0:
            continue

        pair_value_chunks[pair_key].append(vals.astype(np.float32, copy=True))

        if pair_key not in pair_sender_receiver:
            sender, receiver = str(pair_key).split(PAIR_DELIM, 1)
            pair_sender_receiver[pair_key] = (sender, receiver)

if len(pair_value_chunks) == 0:
    raise ValueError("No valid MI-4 edge-level values were collected.")

# -----------------------------
# Compute exact edge-level median by pair
# -----------------------------
pair_summary_records = []

for pair_key, chunks in pair_value_chunks.items():
    values = np.concatenate(chunks).astype(np.float32, copy=False)
    sender, receiver = pair_sender_receiver[pair_key]

    pair_summary_records.append({
        "MI": MI_OI,
        "Pair": f"{sender}->{receiver}",
        "Sender": sender,
        "Receiver": receiver,
        "Pair_edge_count": int(values.size),
        "Median_activity": float(np.nanmedian(values)),
        "Mean_activity": float(np.nanmean(values)),
        "Q25_activity": float(np.nanquantile(values, 0.25)),
        "Q75_activity": float(np.nanquantile(values, 0.75)),
        "Is_Fibroblast_to_cancercell": _is_fibroblast_to_cancercell_pair(sender, receiver),
    })

mi4_pair_median_activity_df = pd.DataFrame(pair_summary_records)

mi4_pair_median_activity_df = (
    mi4_pair_median_activity_df
    .sort_values(
        ["Median_activity", "Pair_edge_count"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

mi4_pair_median_activity_df["Rank"] = np.arange(
    1,
    mi4_pair_median_activity_df.shape[0] + 1,
)

mi4_top_pair_median_activity_df = mi4_pair_median_activity_df.head(TOP_N_PAIRS).copy()

# Cleaner x-axis label.
mi4_top_pair_median_activity_df["Pair_label"] = (
    mi4_top_pair_median_activity_df["Sender"].astype(str)
    + "→"
    + mi4_top_pair_median_activity_df["Receiver"].astype(str)
)

# -----------------------------
# Output directory
# -----------------------------
if "out_dir" in globals():
    stem_out_dir = Path(out_dir)
elif "run_dirs" in globals() and "run_dir" in run_dirs:
    stem_out_dir = Path(run_dirs["run_dir"]) / "MI4_pair_median_activity_stemplot"
else:
    stem_out_dir = Path(".") / "MI4_pair_median_activity_stemplot"

stem_out_dir.mkdir(parents=True, exist_ok=True)

all_table_out = stem_out_dir / "MI4_all_sender_receiver_pair_median_activity.csv"
top_table_out = stem_out_dir / "MI4_top30_sender_receiver_pair_median_activity.csv"

mi4_pair_median_activity_df.to_csv(all_table_out, index=False)
mi4_top_pair_median_activity_df.to_csv(top_table_out, index=False)

print(f"Saved all-pair median table: {all_table_out}")
print(f"Saved top-pair median table: {top_table_out}")

display(
    mi4_top_pair_median_activity_df[
        [
            "Rank",
            "MI",
            "Pair",
            "Sender",
            "Receiver",
            "Pair_edge_count",
            "Median_activity",
            "Mean_activity",
            "Q25_activity",
            "Q75_activity",
            "Is_Fibroblast_to_cancercell",
        ]
    ]
)

# -----------------------------
# Plot style
# -----------------------------
plt.close("all")
plt.style.use("default")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# -----------------------------
# Stem plot
# -----------------------------
plot_df = mi4_top_pair_median_activity_df.copy()

x = np.arange(plot_df.shape[0])
y = plot_df["Median_activity"].to_numpy(dtype=float)
x_labels = plot_df["Pair_label"].astype(str).tolist()
highlight_flags = plot_df["Is_Fibroblast_to_cancercell"].to_numpy(dtype=bool)

fig, ax = plt.subplots(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    facecolor="white",
)

# Stem lines
ax.vlines(
    x,
    ymin=0,
    ymax=y,
    color=STEM_LINE_COLOR,
    linewidth=1.2,
    zorder=1,
)

# Points
ax.scatter(
    x,
    y,
    s=55,
    facecolor=MARKER_FILL_COLOR,
    edgecolor=MARKER_EDGE_COLOR,
    linewidth=0.65,
    zorder=3,
)

# Baseline
ax.axhline(
    0,
    color=BASELINE_COLOR,
    linewidth=0.8,
    zorder=0,
)

ax.set_xticks(x)
ax.set_xticklabels(
    x_labels,
    rotation=75,
    ha="right",
    fontsize=7.2,
)

# Highlight Fibroblast -> *-cancercell x-axis labels in red.
for tick_label, is_highlight in zip(ax.get_xticklabels(), highlight_flags):
    if is_highlight:
        tick_label.set_color(FIBRO_TO_CANCERCELL_XTICK_COLOR)
        tick_label.set_fontweight("bold")
    else:
        tick_label.set_color(DEFAULT_XTICK_COLOR)
        tick_label.set_fontweight("normal")

ax.set_xlabel("Sender→receiver cell-type pair", fontsize=9)
ax.set_ylabel("Median MI-4 interacting strength", fontsize=9)

ax.set_title(
    f"Top {TOP_N_PAIRS} sender→receiver pairs by median MI-4 activity",
    fontsize=10,
    pad=8,
)

y_max = np.nanmax(y) if len(y) > 0 else 0.0

ax.set_xlim(-0.7, len(x) - 0.3)
ax.set_ylim(0, y_max * 1.08 if y_max > 0 else 1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(0.7)
    ax.spines[spine].set_color("black")

ax.tick_params(axis="both", width=0.7, length=3, colors="black")

# tick_params would reset tick label color, so re-apply x-label highlight after tick styling.
for tick_label, is_highlight in zip(ax.get_xticklabels(), highlight_flags):
    if is_highlight:
        tick_label.set_color(FIBRO_TO_CANCERCELL_XTICK_COLOR)
        tick_label.set_fontweight("bold")
    else:
        tick_label.set_color(DEFAULT_XTICK_COLOR)
        tick_label.set_fontweight("normal")

ax.grid(axis="y", color="#E6E6E6", linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

plot_stem = "MI4_top30_sender_receiver_pair_median_activity_stemplot"

pdf_out = stem_out_dir / f"{plot_stem}.pdf"
png_out = stem_out_dir / f"{plot_stem}.png"

fig.savefig(pdf_out, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(png_out, dpi=300, bbox_inches="tight", facecolor="white")

print(f"Saved PDF: {pdf_out}")
print(f"Saved PNG: {png_out}")

plt.show()

# Optional cleanup
del pair_value_chunks
gc.collect()

### Fibroblast-to-tumor MI-4 in situ

Export whole-sub-slice maps with fibroblast senders and tumor receivers.
The current display selects edges above 0.3, samples at most 2,000 edges with
a fixed seed, and uses within-panel color limits. ROI crops, connectors, and
the final manuscript layout are assembled separately.


In [ ]:
##Show the in-situ meta-interaction plot
save_path_insituMI = str(run_dirs['run_dir']) + "/In_situ_meta_interaction/"
if not os.path.exists(save_path_insituMI):
    os.makedirs(save_path_insituMI)

In [ ]:
##MI of interest
MI_OI = "MI-4"

In [ ]:
MI_index = int(MI_OI.replace("MI-", "")) - 1
##
gray_other_cells = True 
##
# vis_mode = 2
vis_mode = 1

In [ ]:
# ============================================================
# In-situ plot of one meta-interaction dimension
# - Plot only one MI at a time
# - Read MI strengths from a memory-mapped .npy file
# - Release per-sample temporary objects immediately after saving
# - Release renderer buffers between exports without changing DPI
# - Print progress only every 10% of slices
# ============================================================

Sender_celltype_list_choose = [
    "Fibroblast"
]

Receiver_celltype_list_choose = [
    "Breast-cancercell",
    "Colon-cancercell",
    "Liver-cancercell",
    "Lung-cancercell",
    "Melanoma-cancercell",
    "Ovarian-cancercell",
    "Prostate-cancercell",
    "Uterine-cancercell",
]

import gc
import os
import math
import sys
import traceback
from pathlib import Path
from plot_spatial import save_drawing_inputs, write_drawing_manifest
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import FancyArrowPatch
from matplotlib.backend_bases import FigureCanvasBase
from matplotlib.backends.backend_agg import FigureCanvasAgg


def save_insitu_figure(fig, out_png, out_pdf, dpi_png, dpi_pdf):
    """Export both formats without retaining the PNG renderer during PDF export."""
    # Disconnect the pyplot manager before replacing the drawing canvas.
    plt.close(fig)
    for output_path, output_dpi in ((out_png, dpi_png), (out_pdf, dpi_pdf)):
        FigureCanvasAgg(fig)
        try:
            fig.savefig(output_path, dpi=output_dpi, bbox_inches="tight", facecolor="white")
        finally:
            # A base canvas holds no Agg pixel buffer. Collect renderer cycles
            # before allocating the next full-resolution raster canvas.
            FigureCanvasBase(fig)
            gc.collect()

# IPython retains exception frames; release their arrays after a memory error.
_last_error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
if isinstance(_last_error, MemoryError):
    traceback.clear_frames(_last_error.__traceback__)
del _last_error

# Drop figure references left by an interrupted plotting loop.
release_memory(
    "fig", "ax", "arr", "spine",
    namespace=globals(), run_gc=True, clear_cuda=False, close_figures=True,
)

batch_cell_unique = np.asarray(pd.read_pickle(processed_data_dir / "batch_cell_unique.pkl"))
factor_envir_path = run_dirs["run_dir"] / "Factor_envir_use.npy"
Factor_envir_use_mmap = np.load(factor_envir_path, mmap_mode="r")

# Pre-compute edge offsets so each sample can read only its own MI slice.
edge_counts = np.asarray(
    [processed.spidernet_data[i]["edge_index"].shape[0] for i in range(len(processed.spidernet_data))],
    dtype=np.int64
)
edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))

edge_sample_random_state = 2026

if vis_mode == 1:
    base_size = 18
    ratio_size = 1
    arrow_lw_base = 1.2
    arrow_ms = 10
    figure_size = (24, 20)
    dpi_png = 250
    dpi_pdf = 300
elif vis_mode == 2:
    base_size = 10
    ratio_size = 1
    arrow_lw_base = 0.8
    arrow_ms = 8
    figure_size = (24, 20)
    dpi_png = 300
    dpi_pdf = 300
else:
    base_size = 14
    ratio_size = 1
    arrow_lw_base = 1.0
    arrow_ms = 9
    figure_size = (24, 20)
    dpi_png = 250
    dpi_pdf = 300

dot_base = base_size * (2.0 / 3.0)
arrow_ms_eff = arrow_ms * 0.5

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

cell_types = [
    "B cell", "Breast-cancercell", "CD4_T", "CD8_T/NK", "Colon-cancercell", "DC",
    "Endothelial", "Epithelial", "Fibroblast", "Liver-cancercell", "Lung-cancercell",
    "Macrophage", "Mast cell", "Melanoma-cancercell", "Ovarian-cancercell",
    "Prostate-cancercell", "Treg", "Uterine-cancercell", "low_exp",
]
color_set = [
    "#8894F1", "#7F3480", "#FF259E", "#FF52DB", "#F7B2EF", "#2C5E1A",
    "#029991", "#E384FF", "#944E13", "#FF8B6C", "#B37166", "#74650F",
    "#7228B8", "#009D4E", "#DD0303", "#028DCE", "#DE0878", "#FAB12F", "#BBC58F",
]
celltype_to_color = dict(zip(cell_types, color_set))
default_gray = "#BFBFBF"
cmap_edge = LinearSegmentedColormap.from_list("white_to_red", ["white", "red"], N=256)

os.makedirs(save_path_insituMI, exist_ok=True)

# Print progress only at 10%, 20%, ..., 100%
n_samples_total = len(batch_cell_unique)
progress_points = sorted(set([
    max(1, math.ceil(n_samples_total * frac / 10))
    for frac in range(1, 11)
]))

drawing_cache_dir = Path(save_path_insituMI) / "drawing_inputs" / f"MI{MI_index + 1}_Mode{vis_mode}"
drawing_cache_frames = []
write_drawing_manifest(drawing_cache_dir, [], n_samples_total, complete=False)

for sample_index_cur, sample_cur in enumerate(batch_cell_unique):
    sample_num = sample_index_cur + 1
    should_report = sample_num in progress_points

    try:
        adata_cursample = processed.adata_list[sample_index_cur]
        data_pyg_cursample = processed.spidernet_data[sample_index_cur]

        spatial_xy = np.asarray(adata_cursample.obsm["spatial"], dtype=np.float32)
        edge_index = data_pyg_cursample["edge_index"].cpu().numpy().astype(np.int32, copy=False)

        if edge_index.ndim != 2 or edge_index.shape[1] != 2:
            raise ValueError(f"edge_index expected shape (E, 2), got {edge_index.shape}")

        start_idx = edge_offsets[sample_index_cur]
        end_idx = edge_offsets[sample_index_cur + 1]
        factor_edge = np.asarray(Factor_envir_use_mmap[start_idx:end_idx, MI_index], dtype=np.float32)

        if factor_edge.shape[0] != edge_index.shape[0]:
            raise ValueError(
                f"Mismatch between edge count and MI edge weights for sample {sample_cur}: "
                f"{edge_index.shape[0]} edges versus {factor_edge.shape[0]} MI values."
            )

        cellclass = np.asarray(data_pyg_cursample["cell_class"]).astype(str)

        src = edge_index[:, 0]
        dst = edge_index[:, 1]

        factor_thr = 0.3
        # Test labels once per cell instead of copying strings for every edge.
        mask_sr = (
            np.isin(cellclass, Sender_celltype_list_choose)[src]
            & np.isin(cellclass, Receiver_celltype_list_choose)[dst]
        )
        mask_thr = factor_edge > factor_thr
        passed_edge_idx = np.flatnonzero(mask_sr & mask_thr)

        n_pass = passed_edge_idx.size
        if n_pass > 2000:
            rng = np.random.default_rng(edge_sample_random_state + sample_index_cur)
            passed_edge_idx = np.sort(rng.choice(passed_edge_idx, size=2000, replace=False))

        if should_report:
            progress_pct = int(round(sample_num / n_samples_total * 100))
            print(f"Progress: {progress_pct}% ({sample_num}/{n_samples_total})")
            # print(f"  passed edges before sampling: {n_pass}")
            # print(f"  edges drawn (after sampling): {passed_edge_idx.size}")

        if passed_edge_idx.size > 0:
            passed_vals = factor_edge[passed_edge_idx]
            if passed_vals.size >= 20:
                vmin = float(np.percentile(passed_vals, 5))
                vmax = float(np.percentile(passed_vals, 95))
            else:
                vmin = float(np.min(passed_vals))
                vmax = float(np.max(passed_vals))
        else:
            passed_vals = factor_edge
            if passed_vals.size == 0:
                vmin, vmax = 0.0, 1.0
            else:
                vmin = float(np.min(passed_vals))
                vmax = float(np.max(passed_vals))

        if np.isclose(vmin, vmax):
            vmax = vmin + 1e-8

        # Store precisely the edges already selected for drawing; replay never resamples.
        drawing_cache_path = save_drawing_inputs(
            drawing_cache_dir,
            f"Insitu_sample{sample_cur}_MI{MI_index + 1}_celltype_Mode{vis_mode}",
            "mi4",
            {"spatial_xy": spatial_xy, "labels": cellclass,
             "src": src[passed_edge_idx], "dst": dst[passed_edge_idx],
             "factor_edge": factor_edge[passed_edge_idx]},
            {"sample_cur": str(sample_cur), "MI_index": int(MI_index), "vis_mode": int(vis_mode),
             "vmin": float(vmin), "vmax": float(vmax), "figure_size": list(figure_size),
             "dpi_png": dpi_png, "dpi_pdf": dpi_pdf, "dot_base": dot_base,
             "ratio_size": ratio_size, "arrow_ms_eff": arrow_ms_eff,
             "arrow_lw_base": arrow_lw_base, "cell_types": cell_types,
             "celltype_to_color": celltype_to_color, "default_gray": default_gray},
        )

        norm_edge = Normalize(vmin=vmin, vmax=vmax, clip=True)

        plt.close("all")
        fig, ax = plt.subplots(figsize=figure_size, facecolor="white")
        ax.set_facecolor("white")

        node_in_edge_mask = np.zeros(len(cellclass), dtype=bool)
        if passed_edge_idx.size > 0:
            involved_nodes = np.unique(np.concatenate([src[passed_edge_idx], dst[passed_edge_idx]]))
            node_in_edge_mask[involved_nodes] = True
        else:
            involved_nodes = np.array([], dtype=np.int32)

        sizes = np.full(len(cellclass), dot_base, dtype=np.float32)
        sizes[node_in_edge_mask] = dot_base * ratio_size

        non_involved_mask = ~node_in_edge_mask
        if np.any(non_involved_mask):
            ax.scatter(
                spatial_xy[non_involved_mask, 0],
                spatial_xy[non_involved_mask, 1],
                c=default_gray,
                s=sizes[non_involved_mask],
                zorder=2,
                rasterized=True,
                alpha=1.0,
                edgecolors="none",
                linewidths=0,
            )

        for ct in cell_types:
            mask_ct = node_in_edge_mask & (cellclass == ct)
            if not np.any(mask_ct):
                continue
            ax.scatter(
                spatial_xy[mask_ct, 0],
                spatial_xy[mask_ct, 1],
                c=celltype_to_color.get(ct, default_gray),
                s=sizes[mask_ct],
                zorder=3,
                rasterized=True,
                alpha=1.0,
                edgecolors="none",
                linewidths=0,
            )

        arrowstyle = "-|>"
        edge_zorder = 10

        for ei in passed_edge_idx:
            s = int(src[ei])
            t = int(dst[ei])

            x0, y0 = spatial_xy[s, 0], spatial_xy[s, 1]
            x1, y1 = spatial_xy[t, 0], spatial_xy[t, 1]

            w01 = float(norm_edge(float(factor_edge[ei])))
            edge_color = cmap_edge(w01)

            arr = FancyArrowPatch(
                (x0, y0),
                (x1, y1),
                arrowstyle=arrowstyle,
                mutation_scale=arrow_ms_eff,
                linewidth=arrow_lw_base * 0.8,
                color=edge_color,
                alpha=0.35 + 0.55 * w01,
                shrinkA=0,
                shrinkB=0,
                capstyle="round",
                joinstyle="round",
                zorder=edge_zorder,
            )
            ax.add_patch(arr)

        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

        out_png = os.path.join(
            save_path_insituMI,
            f"Insitu_sample{sample_cur}_MI{MI_index + 1}_celltype_Mode{vis_mode}.png"
        )
        out_pdf = os.path.join(
            save_path_insituMI,
            f"Insitu_sample{sample_cur}_MI{MI_index + 1}_celltype_Mode{vis_mode}.pdf"
        )

        save_insitu_figure(fig, out_png, out_pdf, dpi_png, dpi_pdf)
        drawing_cache_frames.append(drawing_cache_path)

    finally:
        plt.close("all")
        for _name in [
            "adata_cursample",
            "data_pyg_cursample",
            "spatial_xy",
            "edge_index",
            "factor_edge",
            "cellclass",
            "src",
            "dst",
            "mask_sr",
            "mask_thr",
            "passed_edge_idx",
            "passed_vals",
            "norm_edge",
            "node_in_edge_mask",
            "involved_nodes",
            "sizes",
            "non_involved_mask",
            "fig",
            "ax",
            "arr",
            "spine",
            "mask_ct",
        ]:
            if _name in globals():
                del globals()[_name]

        gc.collect()

write_drawing_manifest(drawing_cache_dir, drawing_cache_frames, n_samples_total, complete=True)

release_memory(
    "Factor_envir_use_mmap",
    "edge_counts",
    "edge_offsets",
    "batch_cell_unique",
    "Sender_celltype_list_choose",
    "Receiver_celltype_list_choose",
    "cell_types",
    "color_set",
    "celltype_to_color",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
    close_figures=True
)

#### Representative MI-2 sub-slices across cancer types

Rank all eligible sub-slices within each cancer type by **total MI-2 activity**,
the sum over the complete set of fibroblast-to-tumor edges. This is an overall
strength criterion; it is not normalized by tissue area or cell count. Ties are
resolved by sample ID and then batch index. The ranking table also records mean
MI-2 activity and the number of edges above the display threshold.

This preview uses MI-2 (column 1), while the manuscript MI-4 maps above retain
their original configuration. The complete MI-4 edge-distribution table supplies
edge identities; MI-2 weights are read from the current inferred factor matrix.
Use the full table, not a downsampled plotting table. The code checks sample
ordering, saved MI-4 weights, and cell labels before displaying the maps.

The highest-total sub-slice from each cancer type is displayed below. Ranking
uses all eligible edges; only rendering applies the 0.3 threshold and seeded
2,000-edge limit. Tables remain available as `mi2_subslice_ranking` and
`mi2_selected_subslices`. This section displays its results inline and does not
overwrite the MI-4 exports.


In [ ]:
# MI-2 representative sub-slices: rank complete fibroblast-to-tumor edges
# before applying the display threshold or sampling edges for rendering.

MI2_PREVIEW_EDGE_THRESHOLD = 0.3
MI2_PREVIEW_MAX_EDGES = 2000
MI2_PREVIEW_RANDOM_SEED = 2026


def select_mi2_representative_subslices(edge_table, factor_path, batch_ids, edge_threshold=0.3):
    """Rank each cancer type's sub-slices by the sum of MI-2 edge activities.

    ``edge_table`` is the complete fibroblast-to-tumor table from the MI-4
    edge-distribution section, not its plotting subsample. Only its edge
    identities are reused for ranking; MI-2 values are read from column 1 of
    ``Factor_envir_use.npy``. Saved MI-4 values provide an alignment check.
    """
    import numpy as np
    import pandas as pd

    columns = [
        "sample_index", "sample_id", "edge_local_index", "edge_global_index",
        "sender_cell_index", "receiver_cell_index", "Sender", "Receiver",
        "CancerType", "MI4_edge_strength",
    ]
    missing = set(columns) - set(edge_table.columns)
    if missing:
        raise ValueError(f"The complete edge table is missing columns: {sorted(missing)}")
    edges = edge_table.loc[:, columns].copy()
    if edges.empty:
        raise ValueError("The fibroblast-to-tumor edge table is empty.")
    if not edges["Sender"].eq("Fibroblast").all():
        raise ValueError("The input must contain only Fibroblast sender edges.")
    if not edges["Receiver"].astype(str).str.endswith("-cancercell").all():
        raise ValueError("The input must contain only tumor receiver edges.")
    if edges["edge_global_index"].duplicated().any():
        raise ValueError("Duplicate global edge indices would inflate sub-slice MI-2 totals.")

    for column in [
        "sample_index", "edge_local_index", "edge_global_index",
        "sender_cell_index", "receiver_cell_index",
    ]:
        values = pd.to_numeric(edges[column], errors="raise").to_numpy()
        if not np.isfinite(values).all() or np.any(values < 0) or np.any(values != np.floor(values)):
            raise ValueError(f"{column} must contain non-negative integer indices.")
        edges[column] = values.astype(np.int64)

    batch_ids = np.asarray(batch_ids).astype(str)
    sample_index = edges["sample_index"].to_numpy()
    if np.any(sample_index >= len(batch_ids)):
        raise ValueError("The edge table references a batch outside the processed bundle.")
    if not np.array_equal(batch_ids[sample_index], edges["sample_id"].astype(str).to_numpy()):
        raise ValueError("The edge table and processed bundle have different sample ordering.")
    receiver_cancer = edges["Receiver"].astype(str).str.removesuffix("-cancercell")
    if not np.array_equal(receiver_cancer.to_numpy(), edges["CancerType"].astype(str).to_numpy()):
        raise ValueError("CancerType must match the tumor receiver annotation.")

    factors = np.load(factor_path, mmap_mode="r")
    global_index = edges["edge_global_index"].to_numpy()
    if factors.ndim != 2 or factors.shape[1] < 4:
        raise ValueError("Expected an edge-by-MI matrix containing MI-2 and MI-4.")
    if global_index.max() >= factors.shape[0]:
        raise ValueError("The edge table references rows outside Factor_envir_use.npy.")
    saved_mi4 = edges["MI4_edge_strength"].to_numpy(dtype=np.float32)
    current_mi4 = np.asarray(factors[global_index, 3], dtype=np.float32)
    if not np.allclose(saved_mi4, current_mi4, rtol=1e-6, atol=1e-7):
        raise ValueError("The edge table is stale or misaligned; regenerate the full edge table.")
    mi2 = np.asarray(factors[global_index, 1], dtype=np.float64)
    if not np.isfinite(mi2).all():
        raise ValueError("Non-finite MI-2 values prevent a complete sub-slice ranking.")
    edges["MI2_strength"] = mi2
    edges["above_display_threshold"] = mi2 > edge_threshold
    del factors

    summary = (
        edges.groupby(["CancerType", "sample_index", "sample_id"], observed=True)
        .agg(
            total_MI2=("MI2_strength", "sum"),
            mean_MI2=("MI2_strength", "mean"),
            n_fibroblast_tumor_edges=("MI2_strength", "size"),
            n_edges_above_display_threshold=("above_display_threshold", "sum"),
        )
        .reset_index()
    )
    cancer_order = ["Breast", "Colon", "Liver", "Lung", "Melanoma", "Ovarian", "Prostate", "Uterine"]
    missing_cancers = set(cancer_order) - set(summary["CancerType"].astype(str))
    if missing_cancers:
        raise ValueError(f"No eligible edges were found for: {sorted(missing_cancers)}")
    extra_cancers = sorted(set(summary["CancerType"].astype(str)) - set(cancer_order))
    summary["CancerType"] = pd.Categorical(
        summary["CancerType"], categories=cancer_order + extra_cancers, ordered=True
    )
    # Resolve equal totals by sample ID, then by the original batch index.
    summary = summary.sort_values(
        ["CancerType", "total_MI2", "sample_id", "sample_index"],
        ascending=[True, False, True, True], kind="mergesort",
    ).reset_index(drop=True)
    summary["rank_within_cancer"] = summary.groupby("CancerType", observed=True).cumcount() + 1
    selected = summary.loc[summary["rank_within_cancer"].eq(1)].reset_index(drop=True)
    return edges, summary, selected


def show_mi2_representative_subslices(
    processed_bundle, run_directory, processed_directory, edge_table=None,
    edge_threshold=0.3, max_edges=2000, random_seed=2026,
):
    """Display one selected MI-2 sub-slice per cancer type and its ranking table.

    Rendering uses the existing in-situ convention: fibroblast-to-tumor edges
    above 0.3, at most 2,000 edges per panel, and within-panel 5th/95th
    percentile color limits. Ranking always uses all eligible edges.
    """
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap, Normalize
    from matplotlib.lines import Line2D
    from matplotlib.patches import FancyArrowPatch
    from IPython.display import display
    from plot_spatial import save_drawing_inputs, write_drawing_manifest

    if max_edges < 1:
        raise ValueError("max_edges must be positive.")
    if edge_table is None:
        edge_path = (
            Path(run_directory) / "MI4_fibroblast_to_tumor_celllevel_CancerSEA"
            / "MI4_fibro_to_tumor_edgelevel_strength_by_cancertype_full.csv"
        )
        if not edge_path.is_file():
            raise FileNotFoundError("Run the MI-4 edge-distribution section to export the complete edge table.")
        edge_table = pd.read_csv(edge_path)
    batch_ids = pd.read_pickle(Path(processed_directory) / "batch_cell_unique.pkl")
    edges, summary, selected = select_mi2_representative_subslices(
        edge_table, Path(run_directory) / "Factor_envir_use.npy", batch_ids, edge_threshold
    )
    display(selected)
    preview_dir = Path(run_directory) / "In_situ_meta_interaction" / "MI2_representative"
    preview_dir.mkdir(parents=True, exist_ok=True)
    drawing_cache_dir = Path(run_directory) / "In_situ_meta_interaction" / "drawing_inputs" / "MI2_representative"
    drawing_cache_frames = []
    write_drawing_manifest(drawing_cache_dir, [], len(selected), complete=False)
    summary.to_csv(preview_dir / "MI2_subslice_ranking.csv", index=False)
    selected.to_csv(preview_dir / "MI2_selected_subslices.csv", index=False)

    tumor_colors = {
        "Breast": "#7F3480", "Colon": "#F7B2EF", "Liver": "#FF8B6C", "Lung": "#B37166",
        "Melanoma": "#009D4E", "Ovarian": "#DD0303", "Prostate": "#028DCE", "Uterine": "#FAB12F",
    }
    cmap = LinearSegmentedColormap.from_list("mi2_white_to_red", ["white", "red"], N=256)
    with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"]}):
        for row in selected.itertuples(index=False):
            sample_edges = edges.loc[edges["sample_index"].eq(row.sample_index)].copy()
            adata = processed_bundle.adata_list[int(row.sample_index)]
            coords = np.asarray(adata.obsm["spatial"], dtype=np.float32)
            labels = adata.obs["celltype_final"].astype(str).to_numpy()
            sample_labels = adata.obs["SampleID"].astype(str).unique()
            if len(sample_labels) != 1 or sample_labels[0] != row.sample_id:
                raise ValueError(f"AnnData sample identity does not match {row.sample_id}.")
            if coords.ndim != 2 or coords.shape[1] != 2 or coords.shape[0] != len(labels):
                raise ValueError(f"Invalid spatial coordinates for {row.sample_id}.")
            src_all = sample_edges["sender_cell_index"].to_numpy(dtype=np.int64)
            dst_all = sample_edges["receiver_cell_index"].to_numpy(dtype=np.int64)
            if max(src_all.max(), dst_all.max()) >= len(labels):
                raise ValueError(f"Cell indices are outside the AnnData for {row.sample_id}.")
            if not np.array_equal(labels[src_all], sample_edges["Sender"].to_numpy()):
                raise ValueError(f"Sender annotations do not match {row.sample_id}.")
            if not np.array_equal(labels[dst_all], sample_edges["Receiver"].to_numpy()):
                raise ValueError(f"Receiver annotations do not match {row.sample_id}.")

            passed = sample_edges.loc[sample_edges["MI2_strength"] > edge_threshold]
            if len(passed) > max_edges:
                rng = np.random.default_rng(random_seed + int(row.sample_index))
                positions = np.sort(rng.choice(len(passed), size=max_edges, replace=False))
                passed = passed.iloc[positions]
            src = passed["sender_cell_index"].to_numpy(dtype=np.int64)
            dst = passed["receiver_cell_index"].to_numpy(dtype=np.int64)
            strengths = passed["MI2_strength"].to_numpy()
            if len(strengths) >= 20:
                vmin, vmax = np.percentile(strengths, [5, 95])
            elif len(strengths):
                vmin, vmax = strengths.min(), strengths.max()
            else:
                vmin, vmax = 0.0, 1.0
            if np.isclose(vmin, vmax):
                vmax = vmin + 1e-8
            preview_stem = f"MI2_representative_{row.CancerType}_{row.sample_id}"
            drawing_cache_path = save_drawing_inputs(
                drawing_cache_dir, preview_stem, "mi2",
                {"coords": coords, "labels": labels, "src": src, "dst": dst, "strengths": strengths},
                {"row_values": {"CancerType": str(row.CancerType), "sample_id": str(row.sample_id),
                                "total_MI2": float(row.total_MI2),
                                "n_edges_above_display_threshold": int(row.n_edges_above_display_threshold)},
                 "vmin": float(vmin), "vmax": float(vmax), "edge_threshold": float(edge_threshold),
                 "tumor_colors": tumor_colors, "preview_stem": preview_stem},
            )
            drawing_cache_frames.append(drawing_cache_path)
            norm = Normalize(vmin=vmin, vmax=vmax, clip=True)
            involved = np.zeros(len(labels), dtype=bool)
            involved[src] = True
            involved[dst] = True
            fig, ax = plt.subplots(figsize=(12, 10), dpi=120, facecolor="white")
            try:
                ax.scatter(*coords[~involved].T, c="#BFBFBF", s=3, edgecolors="none", rasterized=True)
                for cell_label, color in [
                    ("Fibroblast", "#944E13"),
                    (f"{row.CancerType}-cancercell", tumor_colors.get(str(row.CancerType), "#7F3480")),
                ]:
                    mask = involved & (labels == cell_label)
                    ax.scatter(*coords[mask].T, c=color, s=3, edgecolors="none", rasterized=True, zorder=3)
                for sender, receiver, strength in zip(src, dst, strengths):
                    scaled = float(norm(strength))
                    ax.add_patch(FancyArrowPatch(
                        coords[sender], coords[receiver], arrowstyle="-|>", mutation_scale=2.5,
                        linewidth=0.48, color=cmap(scaled), alpha=0.35 + 0.55 * scaled,
                        shrinkA=0, shrinkB=0, capstyle="round", joinstyle="round", zorder=10,
                    ))
                ax.set_aspect("equal")
                ax.set_axis_off()
                ax.set_title(
                    f"{row.CancerType} | {row.sample_id}\n"
                    f"Fibroblast → tumor MI-2 | total activity = {row.total_MI2:,.2f}\n"
                    f"{len(passed):,} / {row.n_edges_above_display_threshold:,} edges shown above {edge_threshold:g}",
                    fontsize=12, pad=14,
                )
                colorbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.025, pad=0.02)
                colorbar.set_label("MI-2 activity (within-panel color limits)", fontsize=9)
                ax.legend(handles=[
                    Line2D([], [], marker="o", linestyle="", color="#944E13", label="Fibroblast"),
                    Line2D([], [], marker="o", linestyle="", color=tumor_colors.get(str(row.CancerType), "#7F3480"), label="Tumor cell"),
                    Line2D([], [], marker="o", linestyle="", color="#BFBFBF", label="Other cells"),
                ], loc="lower center", bbox_to_anchor=(0.5, -0.04), ncol=3, frameon=False)
                fig.tight_layout()
                fig.savefig(preview_dir / f"{preview_stem}.png", dpi=120, bbox_inches="tight", facecolor="white")
                fig.savefig(preview_dir / f"{preview_stem}.pdf", dpi=120, bbox_inches="tight", facecolor="white")
                display(fig)
            finally:
                plt.close(fig)
    write_drawing_manifest(drawing_cache_dir, drawing_cache_frames, len(selected), complete=True)
    return summary, selected


mi2_subslice_ranking, mi2_selected_subslices = show_mi2_representative_subslices(
    processed_bundle=processed,
    run_directory=run_dirs["run_dir"],
    processed_directory=processed_data_dir,
    edge_table=globals().get("mi4_fibro_to_tumor_edgelevel_df"),
    edge_threshold=MI2_PREVIEW_EDGE_THRESHOLD,
    max_edges=MI2_PREVIEW_MAX_EDGES,
    random_seed=MI2_PREVIEW_RANDOM_SEED,
)


### Additional cell-type-pair link table

Estimate cell-type-pair means from a seeded 20% edge sample and export the
MI-4 fibroblast-to-tumor subset. This table can support circular interaction
plots; such a plot is not part of the current pan-cancer figure panels.


In [ ]:

# Prepare cell-type-pair metadata without materializing edge-level sender/receiver arrays.
# The next cell streams through sampled edges by batch, which avoids holding two
# full-length object arrays for all edges in memory.

cellclass_unique = sorted(
    pd.unique(
        pd.concat(
            [
                pd.Series(adata_i.obs["celltype_final"].astype(str).values)
                for adata_i in processed.adata_list
            ],
            ignore_index=True,
        )
    ).tolist()
)

target_cellclass_pair_df_all = pd.DataFrame(
    {
        "Sender": np.repeat(cellclass_unique, len(cellclass_unique)),
        "Receiver": np.tile(cellclass_unique, len(cellclass_unique)),
    }
)

def _edge_count_for_cellpair(edge_index_obj):
    """Return edge count for either E x 2 or 2 x E edge-index layout."""
    shape = tuple(edge_index_obj.shape)
    if len(shape) != 2:
        raise ValueError(f"Unexpected edge_index shape: {shape}")
    if shape[1] == 2:
        return int(shape[0])
    if shape[0] == 2:
        return int(shape[1])
    raise ValueError(f"Unexpected edge_index shape: {shape}")


edge_counts_cellpair = np.asarray(
    [
        _edge_count_for_cellpair(processed.spidernet_data[i]["edge_index"])
        for i in range(len(processed.spidernet_data))
    ],
    dtype=np.int64,
)
edge_offsets_cellpair = np.concatenate(([0], np.cumsum(edge_counts_cellpair)))

print(f"Number of cell classes: {len(cellclass_unique)}")
print(f"Number of candidate sender-receiver pairs: {target_cellclass_pair_df_all.shape[0]}")
print(f"Total number of edges: {int(edge_offsets_cellpair[-1])}")

release_memory(
    "cellclass_unique",
    "_edge_count_for_cellpair",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


In [ ]:
import gc
import numpy as np
import pandas as pd
import torch

# --------------------------------------------------
# Memory-safe mean aggregation by (sender, receiver) pair
# using only a random subset of edges.
#
# Use string arrays for pair labels and release batch-local intermediates.
# --------------------------------------------------

def safe_release_memory(*var_names, namespace=None, run_gc=True, clear_cuda=False):
    """
    Safe variable cleanup.
    If your notebook already defines release_memory(), this function will use it.
    Otherwise, it falls back to deleting variables from namespace.
    """
    if namespace is None:
        namespace = globals()

    if "release_memory" in namespace and callable(namespace["release_memory"]):
        namespace["release_memory"](
            *var_names,
            namespace=namespace,
            run_gc=run_gc,
            clear_cuda=clear_cuda,
        )
    else:
        for name in var_names:
            if name in namespace:
                del namespace[name]

        if run_gc:
            gc.collect()

        if clear_cuda and torch.cuda.is_available():
            torch.cuda.empty_cache()


# -----------------------------
# Parameters
# -----------------------------
sample_ratio = 0.20
random_seed = 2026
celltype_col = "celltype_final"

factor_path = run_dirs["run_dir"] / "Factor_envir_use.npy"

# mmap avoids loading the full Factor_envir_use.npy into memory
Factor_envir_use_mmap = np.load(factor_path, mmap_mode="r")
E, M = Factor_envir_use_mmap.shape

n_sample = max(1, int(E * sample_ratio))
rng = np.random.default_rng(random_seed)

edge_idx_sample = np.sort(
    rng.choice(E, size=n_sample, replace=False)
).astype(np.int64)

print(f"Original number of edges: {E}")
print(f"Sampled number of edges: {n_sample}")


# -----------------------------
# Build target pair index
# Important: force unicode/string dtype here
# -----------------------------
sender_pair = target_cellclass_pair_df_all["Sender"].astype(str).to_numpy(dtype=str)
receiver_pair = target_cellclass_pair_df_all["Receiver"].astype(str).to_numpy(dtype=str)

pair_list = np.char.add(
    np.char.add(sender_pair, "_to_"),
    receiver_pair,
).astype(str)

pair_index = pd.Index(pair_list)
P = len(pair_list)

sums = np.zeros((P, M), dtype=np.float64)
counts = np.zeros(P, dtype=np.int64)


# -----------------------------
# Map sampled global edges to batch IDs
# edge_offsets_cellpair should have length n_batch + 1
# -----------------------------
edge_offsets_cellpair = np.asarray(edge_offsets_cellpair, dtype=np.int64)

sample_batch_ids = np.searchsorted(
    edge_offsets_cellpair[1:],
    edge_idx_sample,
    side="right",
).astype(np.int64)


# -----------------------------
# Stream through batches
# -----------------------------
for batch_idx in np.unique(sample_batch_ids):
    batch_idx = int(batch_idx)

    mask_batch = sample_batch_ids == batch_idx
    global_edge_idx_cur = edge_idx_sample[mask_batch]
    local_edge_idx_cur = (
        global_edge_idx_cur - edge_offsets_cellpair[batch_idx]
    ).astype(np.int64)

    data_pyg_cur = processed.spidernet_data[batch_idx]
    edge_index_cur = data_pyg_cur["edge_index"]

    # --------------------------------------------------
    # Select only sampled edges from current batch
    # Support both E x 2 and 2 x E layouts.
    # --------------------------------------------------
    if torch.is_tensor(edge_index_cur):
        local_edge_idx_t = torch.as_tensor(
            local_edge_idx_cur,
            dtype=torch.long,
            device=edge_index_cur.device,
        )

        if edge_index_cur.ndim == 2 and edge_index_cur.shape[1] == 2:
            edge_index_sel = (
                edge_index_cur[local_edge_idx_t, :]
                .detach()
                .cpu()
                .numpy()
            )
        elif edge_index_cur.ndim == 2 and edge_index_cur.shape[0] == 2:
            edge_index_sel = (
                edge_index_cur[:, local_edge_idx_t]
                .T
                .detach()
                .cpu()
                .numpy()
            )
        else:
            raise ValueError(
                f"Unexpected edge_index shape: {tuple(edge_index_cur.shape)}"
            )

    else:
        edge_index_cur = np.asarray(edge_index_cur)

        if edge_index_cur.ndim == 2 and edge_index_cur.shape[1] == 2:
            edge_index_sel = edge_index_cur[local_edge_idx_cur, :]
        elif edge_index_cur.ndim == 2 and edge_index_cur.shape[0] == 2:
            edge_index_sel = edge_index_cur[:, local_edge_idx_cur].T
        else:
            raise ValueError(
                f"Unexpected edge_index shape: {edge_index_cur.shape}"
            )

    edge_index_sel = np.asarray(edge_index_sel, dtype=np.int64)

    # --------------------------------------------------
    # Get cell type strings.
    # String arrays keep cell-type labels compatible with NumPy string operations.
    # --------------------------------------------------
    cellclass_cur = (
        processed.adata_list[batch_idx]
        .obs[celltype_col]
        .astype(str)
        .to_numpy(dtype=str)
    )

    sender_str = np.asarray(
        cellclass_cur[edge_index_sel[:, 0]],
        dtype=str,
    )
    receiver_str = np.asarray(
        cellclass_cur[edge_index_sel[:, 1]],
        dtype=str,
    )

    # --------------------------------------------------
    # Build sender_to_receiver strings.
    # Concatenate sender and receiver labels as NumPy string arrays.
    # --------------------------------------------------
    edge_pairs = np.char.add(
        np.char.add(sender_str, "_to_"),
        receiver_str,
    ).astype(str)

    codes = pair_index.get_indexer(edge_pairs).astype(np.int64)
    mask_valid = codes >= 0

    if np.any(mask_valid):
        factor_vals = np.asarray(
            Factor_envir_use_mmap[global_edge_idx_cur[mask_valid], :],
            dtype=np.float32,
        )

        np.add.at(sums, codes[mask_valid], factor_vals)
        np.add.at(counts, codes[mask_valid], 1)

        safe_release_memory(
            "factor_vals",
            namespace=globals(),
            run_gc=False,
            clear_cuda=False,
        )

    # Release per-batch temporaries immediately
    safe_release_memory(
        "data_pyg_cur",
        "edge_index_cur",
        "edge_index_sel",
        "cellclass_cur",
        "sender_str",
        "receiver_str",
        "edge_pairs",
        "codes",
        "mask_valid",
        "global_edge_idx_cur",
        "local_edge_idx_cur",
        "mask_batch",
        namespace=globals(),
        run_gc=True,
        clear_cuda=False,
    )


# -----------------------------
# Compute mean MI value per cell-type pair
# -----------------------------
means = np.full((P, M), np.nan, dtype=np.float32)

np.divide(
    sums,
    counts[:, None],
    out=means,
    where=(counts[:, None] > 0),
)

mean_MI_cellclasspair_all = pd.DataFrame(
    means.T,
    index=[f"MI-{i + 1}" for i in range(M)],
    columns=pair_list,
)


# -----------------------------
# Save MI-4 Fibroblast -> cancer-cell cell-type-pair table
# -----------------------------
df_MI4_fibro_to_cancer_all = pd.DataFrame(
    {
        "MI": np.repeat(
            mean_MI_cellclasspair_all.index,
            mean_MI_cellclasspair_all.shape[1],
        ),
        "Pair": np.tile(
            mean_MI_cellclasspair_all.columns,
            mean_MI_cellclasspair_all.shape[0],
        ),
        "Mean": mean_MI_cellclasspair_all.values.ravel(),
    }
)

df_MI4_fibro_to_cancer_all["Sender"] = df_MI4_fibro_to_cancer_all["Pair"].str.split("_to_").str[0]
df_MI4_fibro_to_cancer_all["Receiver"] = df_MI4_fibro_to_cancer_all["Pair"].str.split("_to_").str[1]

# Focus on the downstream axis of interest:
#   Fibroblast --(MI-4)--> cancer cell
# First filter by the MI dimension, then keep only Fibroblast sender and
# cancer-cell receivers.
df_MI4_fibro_to_cancer_all = df_MI4_fibro_to_cancer_all[
    (df_MI4_fibro_to_cancer_all["MI"] == "MI-4")
    & (df_MI4_fibro_to_cancer_all["Sender"].astype(str).eq("Fibroblast"))
    & (df_MI4_fibro_to_cancer_all["Receiver"].astype(str).str.endswith("-cancercell"))
].copy()

csv_path = str(run_dirs["run_dir"] / "MI4_fibroblast_to_cancercell_links.csv")
df_MI4_fibro_to_cancer_all.to_csv(csv_path, index=False)

print(f"Saved: {csv_path}")


# -----------------------------
# Final cleanup
# -----------------------------
safe_release_memory(
    "Factor_envir_use_mmap",
    "edge_idx_sample",
    "sample_batch_ids",
    "pair_index",
    "pair_list",
    "sender_pair",
    "receiver_pair",
    "sums",
    "counts",
    "means",
    "mean_MI_cellclasspair_all",
    "df_MI4_fibro_to_cancer_all",
    "target_cellclass_pair_df_all",
    "edge_counts_cellpair",
    "edge_offsets_cellpair",
    "factor_path",
    "rng",
    "csv_path",
    namespace=globals(),
    run_gc=True,
    clear_cuda=True,
)


### Additional gene-expression associations with MI-4

Correlate fibroblast sender and tumor receiver gene expression with cell-level
MI-4 on fibroblast-to-tumor edges. The default correlation is Spearman and the
default cell-level aggregation in this section is the mean over eligible edges.


In [ ]:
import gc
import numpy as np
import pandas as pd

try:
    import scipy.sparse as sp
except ImportError:
    sp = None

try:
    from scipy.stats import rankdata
except ImportError:
    rankdata = None


# ============================================================
# User settings
# ============================================================
MI_OI = "MI4"
mi_index = int(MI_OI.replace("MI", "")) - 1

TUMOR_SUFFIX = "-cancercell"
FIBROBLAST_LABEL = "Fibroblast"

# Choose correlation method: "pearson" or "spearman"
# CORR_METHOD = "pearson"
CORR_METHOD = "spearman"

CORR_METHOD = CORR_METHOD.lower()
if CORR_METHOD not in {"pearson", "spearman"}:
    raise ValueError("CORR_METHOD must be either 'pearson' or 'spearman'.")

if CORR_METHOD == "spearman" and rankdata is None:
    raise ImportError("scipy.stats.rankdata is required for Spearman correlation.")

# Whether cell-level MI uses summed edge MI or average edge MI
# False = average over valid Fibroblast->cancer-cell edges
# True  = sum over valid Fibroblast->cancer-cell edges
CELL_MI_IF_SUM = False

# Process expression in chunks to limit peak memory use.
EXPR_CHUNK_SIZE = 100_000

# For Spearman only: process genes block by block
SPEARMAN_GENE_BLOCK_SIZE = 50


In [ ]:
# ============================================================
# Helper functions
# ============================================================
def ensure_edge_index_e_by_2(edge_index):
    """Convert edge_index to shape (E, 2) if needed."""
    edge_index = np.asarray(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(np.int64, copy=False)

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def to_numpy_maybe_torch(x):
    """Convert torch tensor / numpy array / list-like object to numpy array."""
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    if hasattr(x, "cpu"):
        return x.cpu().numpy()
    return np.asarray(x)


def get_sample_id(sample_obj):
    """Extract sample id string from processed.spidernet_data[i]['sample']."""
    arr = to_numpy_maybe_torch(sample_obj)
    arr = np.asarray(arr).astype(str).ravel()

    if arr.size == 0:
        raise ValueError("Empty sample field.")

    return str(arr[0])


def compute_cell_level_mi(
    mi_edge,
    edge_index,
    n_cells,
    valid_edge_mask=None,
    if_sum=False,
):
    """
    Aggregate edge-level MI values into cell-level sending and receiving scores.

    If valid_edge_mask is provided, only edges with valid_edge_mask == True
    are used for the aggregation.

    if_sum=False: average MI over valid edges for each cell.
    if_sum=True : sum MI over valid edges for each cell.
    """
    edge_index = np.asarray(edge_index)
    mi_edge = np.asarray(mi_edge, dtype=np.float32)

    if valid_edge_mask is None:
        valid_edge_mask = np.ones(edge_index.shape[0], dtype=bool)
    else:
        valid_edge_mask = np.asarray(valid_edge_mask, dtype=bool)

    src = edge_index[valid_edge_mask, 0]
    dst = edge_index[valid_edge_mask, 1]
    mi_edge_use = mi_edge[valid_edge_mask]

    send_sum = np.bincount(
        src,
        weights=mi_edge_use,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    send_cnt = np.bincount(
        src,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    recv_sum = np.bincount(
        dst,
        weights=mi_edge_use,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    recv_cnt = np.bincount(
        dst,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    send_score = np.full(n_cells, np.nan, dtype=np.float32)
    recv_score = np.full(n_cells, np.nan, dtype=np.float32)

    if if_sum:
        send_score = send_sum.astype(np.float32, copy=False)
        recv_score = recv_sum.astype(np.float32, copy=False)
    else:
        np.divide(send_sum, send_cnt, out=send_score, where=send_cnt > 0)
        np.divide(recv_sum, recv_cnt, out=recv_score, where=recv_cnt > 0)

    return send_score, recv_score


# ============================================================
# Pearson correlation: streaming sufficient statistics
# ============================================================
def init_pearson_accumulator(n_genes):
    return {
        "n": 0.0,
        "sum_x": 0.0,
        "sum_x2": 0.0,
        "sum_y": np.zeros(n_genes, dtype=np.float64),
        "sum_y2": np.zeros(n_genes, dtype=np.float64),
        "sum_xy": np.zeros(n_genes, dtype=np.float64),
    }


def update_pearson_accumulator_from_rows(
    acc,
    x_values,
    X_full,
    row_indices,
    chunk_size=100_000,
):
    """
    Update Pearson-correlation sufficient statistics using selected rows.

    This avoids materializing all selected expression rows at once.
    """
    x_values = np.asarray(x_values, dtype=np.float64).ravel()
    row_indices = np.asarray(row_indices, dtype=np.int64).ravel()

    if x_values.shape[0] != row_indices.shape[0]:
        raise ValueError(
            f"x_values and row_indices length mismatch: "
            f"{x_values.shape[0]} vs {row_indices.shape[0]}"
        )

    valid_x_mask = np.isfinite(x_values)

    if valid_x_mask.sum() == 0:
        return acc

    x_values = x_values[valid_x_mask]
    row_indices = row_indices[valid_x_mask]

    n_valid = x_values.shape[0]

    for start in range(0, n_valid, chunk_size):
        end = min(start + chunk_size, n_valid)

        rows_chunk = row_indices[start:end]
        x_chunk = x_values[start:end].astype(np.float64, copy=False)

        X_chunk = X_full[rows_chunk, :]

        if sp is not None and sp.issparse(X_chunk):
            X_chunk = X_chunk.tocsr()

            y_sum = np.asarray(X_chunk.sum(axis=0)).ravel().astype(np.float64, copy=False)
            y2_sum = np.asarray(X_chunk.multiply(X_chunk).sum(axis=0)).ravel().astype(np.float64, copy=False)
            xy_sum = np.asarray(X_chunk.T.dot(x_chunk)).ravel().astype(np.float64, copy=False)

        else:
            X_chunk = np.asarray(X_chunk, dtype=np.float64)

            y_sum = X_chunk.sum(axis=0, dtype=np.float64)
            y2_sum = np.einsum("ij,ij->j", X_chunk, X_chunk)
            xy_sum = x_chunk @ X_chunk

        acc["n"] += float(x_chunk.shape[0])
        acc["sum_x"] += float(x_chunk.sum())
        acc["sum_x2"] += float(np.dot(x_chunk, x_chunk))
        acc["sum_y"] += y_sum
        acc["sum_y2"] += y2_sum
        acc["sum_xy"] += xy_sum

        del rows_chunk, x_chunk, X_chunk, y_sum, y2_sum, xy_sum
        gc.collect()

    return acc


def finalize_pearson_accumulator(acc):
    """Convert sufficient statistics to Pearson correlations."""
    n = acc["n"]
    n_genes = acc["sum_y"].shape[0]

    corr = np.full(n_genes, np.nan, dtype=np.float64)

    if n < 3:
        return corr

    cov_xy = acc["sum_xy"] - (acc["sum_x"] * acc["sum_y"] / n)
    var_x = acc["sum_x2"] - (acc["sum_x"] ** 2 / n)
    var_y = acc["sum_y2"] - (acc["sum_y"] ** 2 / n)

    var_y = np.maximum(var_y, 0.0)

    denom = np.sqrt(var_x * var_y)

    valid = (
        np.isfinite(cov_xy)
        & np.isfinite(denom)
        & (denom > 0)
        & (var_x > 0)
        & (var_y > 0)
    )

    corr[valid] = cov_xy[valid] / denom[valid]

    return corr


# Backward-compatible aliases
init_corr_accumulator = init_pearson_accumulator
update_corr_accumulator_from_rows = update_pearson_accumulator_from_rows
finalize_corr_accumulator = finalize_pearson_accumulator


# ============================================================
# Spearman correlation: rank x and each gene across concatenated cells
# ============================================================
def pearson_corr_1d(x, y):
    """Pearson correlation between two 1D arrays with finite values only."""
    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 3:
        return np.nan

    x = x[valid]
    y = y[valid]

    x = x - x.mean()
    y = y - y.mean()

    denom = np.sqrt(np.dot(x, x) * np.dot(y, y))

    if denom <= 0 or not np.isfinite(denom):
        return np.nan

    return float(np.dot(x, y) / denom)


def load_expression_gene_block_from_entries(
    entries,
    adata_list,
    gene_start,
    gene_end,
):
    """
    Load expression rows for a list of entries and a gene block.

    entries must be ordered exactly as x_values are concatenated.
    """
    X_blocks = []

    for entry in entries:
        batch_idx = entry["batch_idx"]
        row_indices = entry["row_indices"]

        if row_indices.size == 0:
            continue

        X_full = adata_list[batch_idx].X
        X_sub = X_full[row_indices, gene_start:gene_end]

        if sp is not None and sp.issparse(X_sub):
            X_sub = X_sub.toarray()
        else:
            X_sub = np.asarray(X_sub)

        X_blocks.append(np.asarray(X_sub, dtype=np.float64))

    if len(X_blocks) == 0:
        return np.empty((0, gene_end - gene_start), dtype=np.float64)

    X_block = np.vstack(X_blocks)

    for X_sub in X_blocks:
        del X_sub
    del X_blocks
    gc.collect()

    return X_block


def compute_spearman_corr_from_entries(
    entries,
    adata_list,
    n_genes,
    gene_block_size=50,
):
    """
    Compute Spearman correlation between concatenated MI values and each gene.

    This is equivalent to:
        spearmanr(x_concat, X_concat[:, gene])
    for each gene, but processes genes in blocks to reduce memory use.
    """
    corr = np.full(n_genes, np.nan, dtype=np.float64)

    if len(entries) == 0:
        return corr

    x_concat = np.concatenate([
        np.asarray(entry["x_values"], dtype=np.float64).ravel()
        for entry in entries
        if len(entry["x_values"]) > 0
    ])

    if x_concat.size < 3:
        return corr

    for gene_start in range(0, n_genes, gene_block_size):
        gene_end = min(gene_start + gene_block_size, n_genes)

        print(f"    Spearman gene block: {gene_start}:{gene_end}")

        X_block = load_expression_gene_block_from_entries(
            entries=entries,
            adata_list=adata_list,
            gene_start=gene_start,
            gene_end=gene_end,
        )

        if X_block.shape[0] != x_concat.shape[0]:
            raise ValueError(
                f"Spearman X/x length mismatch for gene block {gene_start}:{gene_end}: "
                f"X has {X_block.shape[0]} rows, x has {x_concat.shape[0]} values."
            )

        for local_j in range(X_block.shape[1]):
            gene_j = gene_start + local_j
            y = X_block[:, local_j]

            valid = np.isfinite(x_concat) & np.isfinite(y)

            if valid.sum() < 3:
                corr[gene_j] = np.nan
                continue

            # Exact Spearman: rank over the pairwise-valid cells for this gene
            rx = rankdata(x_concat[valid], method="average")
            ry = rankdata(y[valid], method="average")

            corr[gene_j] = pearson_corr_1d(rx, ry)

        del X_block
        gc.collect()

    del x_concat
    gc.collect()

    return corr


#### Median-split gene-expression fold changes and GO/KEGG summaries

The next section uses summed cell-level MI-4 to define high/low groups and
exports gene-expression log2 fold changes. Alternative heatmaps display
z-scored absolute fold changes or signed fold changes. GO/KEGG enrichment also
exports reference gene sets for downstream cascade analyses. These analyses
extend beyond the CancerSEA comparisons in Fig. S25a.


In [ ]:
import gc
import numpy as np
import pandas as pd

try:
    import scipy.sparse as sp
except ImportError:
    sp = None


# ============================================================
# User settings
# ============================================================
MI_OI = "MI4"
mi_index = int(MI_OI.replace("MI", "")) - 1

TUMOR_SUFFIX = "-cancercell"
FIBROBLAST_LABEL = "Fibroblast"

# Whether cell-level MI uses summed edge MI or average edge MI
# False = average over valid Fibroblast->cancer-cell edges
# True  = sum over valid Fibroblast->cancer-cell edges
# CELL_MI_IF_SUM = False
CELL_MI_IF_SUM = True

# Median split mode:
# "rank_half": split cells into two similarly sized groups by MI rank
# "threshold": high = MI > median, low = MI <= median; falls back to rank_half if one group is empty
MEDIAN_SPLIT_MODE = "rank_half"

# log2FC pseudocount
LFC_PSEUDOCOUNT = 1e-6

# Process expression in chunks to limit peak memory use.
EXPR_CHUNK_SIZE = 50_000


# ============================================================
# Helper functions
# ============================================================
def ensure_edge_index_e_by_2(edge_index):
    """Convert edge_index to shape (E, 2) if needed."""
    edge_index = np.asarray(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(np.int64, copy=False)

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def to_numpy_maybe_torch(x):
    """Convert torch tensor / numpy array / list-like object to numpy array."""
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    if hasattr(x, "cpu"):
        return x.cpu().numpy()
    return np.asarray(x)


def get_sample_id(sample_obj):
    """Extract sample id string from processed.spidernet_data[i]['sample']."""
    arr = to_numpy_maybe_torch(sample_obj)
    arr = np.asarray(arr).astype(str).ravel()

    if arr.size == 0:
        raise ValueError("Empty sample field.")

    return str(arr[0])


def compute_cell_level_mi(
    mi_edge,
    edge_index,
    n_cells,
    valid_edge_mask=None,
    if_sum=False,
):
    """
    Aggregate edge-level MI values into cell-level sending and receiving scores.

    if_sum=False: average MI over valid edges for each cell.
    if_sum=True : sum MI over valid edges for each cell.
    """
    edge_index = np.asarray(edge_index)
    mi_edge = np.asarray(mi_edge, dtype=np.float32)

    if valid_edge_mask is None:
        valid_edge_mask = np.ones(edge_index.shape[0], dtype=bool)
    else:
        valid_edge_mask = np.asarray(valid_edge_mask, dtype=bool)

    src = edge_index[valid_edge_mask, 0]
    dst = edge_index[valid_edge_mask, 1]
    mi_edge_use = mi_edge[valid_edge_mask]

    send_sum = np.bincount(
        src,
        weights=mi_edge_use,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    send_cnt = np.bincount(
        src,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    recv_sum = np.bincount(
        dst,
        weights=mi_edge_use,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    recv_cnt = np.bincount(
        dst,
        minlength=n_cells
    ).astype(np.float64, copy=False)

    send_score = np.full(n_cells, np.nan, dtype=np.float32)
    recv_score = np.full(n_cells, np.nan, dtype=np.float32)

    if if_sum:
        send_score = send_sum.astype(np.float32, copy=False)
        recv_score = recv_sum.astype(np.float32, copy=False)
    else:
        np.divide(send_sum, send_cnt, out=send_score, where=send_cnt > 0)
        np.divide(recv_sum, recv_cnt, out=recv_score, where=recv_cnt > 0)

    return send_score, recv_score


def init_lfc_accumulator(n_genes):
    """Initialize expression sum accumulators for high-MI and low-MI groups."""
    return {
        "n_high": 0,
        "n_low": 0,
        "sum_high": np.zeros(n_genes, dtype=np.float64),
        "sum_low": np.zeros(n_genes, dtype=np.float64),
    }


def update_lfc_accumulator_from_rows(
    acc,
    X_full,
    row_indices,
    high_mask,
    chunk_size=50_000,
):
    """
    Update expression sums for high-MI and low-MI groups.

    row_indices: selected cell indices in X_full.
    high_mask: boolean array, same length as row_indices.
               True = high-MI group, False = low-MI group.
    """
    row_indices = np.asarray(row_indices, dtype=np.int64).ravel()
    high_mask = np.asarray(high_mask, dtype=bool).ravel()

    if row_indices.shape[0] != high_mask.shape[0]:
        raise ValueError(
            f"row_indices and high_mask length mismatch: "
            f"{row_indices.shape[0]} vs {high_mask.shape[0]}"
        )

    n_rows = row_indices.shape[0]

    if n_rows == 0:
        return acc

    for start in range(0, n_rows, chunk_size):
        end = min(start + chunk_size, n_rows)

        rows_chunk = row_indices[start:end]
        high_chunk = high_mask[start:end]
        low_chunk = ~high_chunk

        X_chunk = X_full[rows_chunk, :]

        if sp is not None and sp.issparse(X_chunk):
            X_chunk = X_chunk.tocsr()

            if np.any(high_chunk):
                high_sum = np.asarray(
                    X_chunk[high_chunk, :].sum(axis=0)
                ).ravel().astype(np.float64, copy=False)
                acc["sum_high"] += high_sum
                acc["n_high"] += int(high_chunk.sum())
                del high_sum

            if np.any(low_chunk):
                low_sum = np.asarray(
                    X_chunk[low_chunk, :].sum(axis=0)
                ).ravel().astype(np.float64, copy=False)
                acc["sum_low"] += low_sum
                acc["n_low"] += int(low_chunk.sum())
                del low_sum

        else:
            X_chunk = np.asarray(X_chunk)

            if np.any(high_chunk):
                acc["sum_high"] += X_chunk[high_chunk, :].sum(
                    axis=0,
                    dtype=np.float64
                )
                acc["n_high"] += int(high_chunk.sum())

            if np.any(low_chunk):
                acc["sum_low"] += X_chunk[low_chunk, :].sum(
                    axis=0,
                    dtype=np.float64
                )
                acc["n_low"] += int(low_chunk.sum())

        del rows_chunk, high_chunk, low_chunk, X_chunk
        gc.collect()

    return acc


def finalize_lfc_accumulator(acc, pseudocount=1e-6):
    """Convert high/low expression sums into gene-wise log2 fold change."""
    n_genes = acc["sum_high"].shape[0]
    log2fc = np.full(n_genes, np.nan, dtype=np.float64)

    n_high = int(acc["n_high"])
    n_low = int(acc["n_low"])

    if n_high == 0 or n_low == 0:
        return log2fc

    mean_high = acc["sum_high"] / n_high
    mean_low = acc["sum_low"] / n_low

    log2fc = np.log2(
        (mean_high + pseudocount) /
        (mean_low + pseudocount)
    )

    return log2fc


def make_median_split_high_mask(x_values, mode="rank_half"):
    """
    Generate high/low group labels from MI values.

    Returns
    -------
    high_mask : bool array
        True for high-MI group.
    median_value : float
        Median of x_values.
    split_info : dict
        Additional summary info.
    """
    x_values = np.asarray(x_values, dtype=np.float64).ravel()

    valid = np.isfinite(x_values)
    if valid.sum() != x_values.size:
        raise ValueError("x_values should already be finite before median splitting.")

    n = x_values.size

    if n < 2:
        high_mask = np.zeros(n, dtype=bool)
        return high_mask, np.nan, {
            "split_mode_used": mode,
            "n_values": n,
            "n_high": int(high_mask.sum()),
            "n_low": int((~high_mask).sum()),
        }

    median_value = float(np.median(x_values))

    if mode == "threshold":
        high_mask = x_values > median_value

        # If all values are tied around the median, fallback to rank-half split
        if high_mask.sum() == 0 or (~high_mask).sum() == 0:
            mode_used = "rank_half_fallback"
        else:
            mode_used = "threshold"
            return high_mask, median_value, {
                "split_mode_used": mode_used,
                "n_values": n,
                "n_high": int(high_mask.sum()),
                "n_low": int((~high_mask).sum()),
            }

    elif mode == "rank_half":
        mode_used = "rank_half"

    else:
        raise ValueError("MEDIAN_SPLIT_MODE must be either 'rank_half' or 'threshold'.")

    # rank-half median split: lower half = low, upper half = high
    order = np.argsort(x_values, kind="mergesort")
    split = n // 2

    high_mask = np.zeros(n, dtype=bool)
    high_mask[order[split:]] = True

    return high_mask, median_value, {
        "split_mode_used": mode_used,
        "n_values": n,
        "n_high": int(high_mask.sum()),
        "n_low": int((~high_mask).sum()),
    }


def compute_log2fc_from_entries(
    entries,
    adata_list,
    n_genes,
    pseudocount=1e-6,
    split_mode="rank_half",
    chunk_size=50_000,
):
    """
    For one cancer type and one MI direction, split cells by median MI value,
    then compute gene expression log2FC: high-MI vs low-MI.

    entries is a list of dicts:
        {
            "batch_idx": int,
            "row_indices": tumor cell row indices in that batch,
            "x_values": MI values corresponding to row_indices
        }
    """
    log2fc = np.full(n_genes, np.nan, dtype=np.float64)

    if len(entries) == 0:
        return log2fc, {
            "median_mi": np.nan,
            "split_mode_used": split_mode,
            "n_cells_for_lfc": 0,
            "n_high": 0,
            "n_low": 0,
        }

    x_concat = np.concatenate([
        np.asarray(entry["x_values"], dtype=np.float64).ravel()
        for entry in entries
        if len(entry["x_values"]) > 0
    ])

    if x_concat.size < 2:
        return log2fc, {
            "median_mi": np.nan if x_concat.size == 0 else float(np.median(x_concat)),
            "split_mode_used": split_mode,
            "n_cells_for_lfc": int(x_concat.size),
            "n_high": 0,
            "n_low": int(x_concat.size),
        }

    high_concat_mask, median_mi, split_info = make_median_split_high_mask(
        x_concat,
        mode=split_mode,
    )

    acc = init_lfc_accumulator(n_genes)

    offset = 0

    for entry in entries:
        x_values = np.asarray(entry["x_values"], dtype=np.float64).ravel()
        row_indices = np.asarray(entry["row_indices"], dtype=np.int64).ravel()

        n_entry = x_values.size

        if n_entry == 0:
            continue

        high_mask_entry = high_concat_mask[offset:offset + n_entry]

        update_lfc_accumulator_from_rows(
            acc=acc,
            X_full=adata_list[entry["batch_idx"]].X,
            row_indices=row_indices,
            high_mask=high_mask_entry,
            chunk_size=chunk_size,
        )

        offset += n_entry

        del x_values, row_indices, high_mask_entry
        gc.collect()

    if offset != x_concat.size:
        raise ValueError(
            f"Internal offset mismatch: offset={offset}, x_concat.size={x_concat.size}"
        )

    log2fc = finalize_lfc_accumulator(
        acc,
        pseudocount=pseudocount,
    )

    stats = {
        "median_mi": median_mi,
        "split_mode_used": split_info["split_mode_used"],
        "n_cells_for_lfc": int(split_info["n_values"]),
        "n_high": int(acc["n_high"]),
        "n_low": int(acc["n_low"]),
    }

    del x_concat, high_concat_mask, acc
    gc.collect()

    return log2fc, stats


# ============================================================
# Prepare edge offsets and MI mmap
# ============================================================
n_batches = len(processed.spidernet_data)

edge_counts = np.asarray(
    [
        ensure_edge_index_e_by_2(
            to_numpy_maybe_torch(processed.spidernet_data[i]["edge_index"])
        ).shape[0]
        for i in range(n_batches)
    ],
    dtype=np.int64,
)

edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))

factor_path = run_dirs["run_dir"] / "Factor_envir_use.npy"
Factor_envir_use_mmap = np.load(factor_path, mmap_mode="r")

if mi_index < 0 or mi_index >= Factor_envir_use_mmap.shape[1]:
    raise ValueError(
        f"{MI_OI} gives mi_index={mi_index}, but Factor_envir_use has "
        f"{Factor_envir_use_mmap.shape[1]} MI dimensions."
    )

if edge_offsets[-1] > Factor_envir_use_mmap.shape[0]:
    raise ValueError(
        f"Total edge count from processed data is {edge_offsets[-1]}, "
        f"but Factor_envir_use has only {Factor_envir_use_mmap.shape[0]} rows."
    )

batch_sample_ids = [
    get_sample_id(processed.spidernet_data[i]["sample"])
    for i in range(n_batches)
]

batch_cancer_types = [
    sample_id.split("_")[0]
    for sample_id in batch_sample_ids
]

cancer_type_order = list(dict.fromkeys(batch_cancer_types))

cancer_type_to_batch_indices = {
    cancer_type: [
        i for i, ct in enumerate(batch_cancer_types)
        if ct == cancer_type
    ]
    for cancer_type in cancer_type_order
}

print("[Cancer type order]")
print(cancer_type_order)

print("\n[Number of batches per cancer type]")
for ct in cancer_type_order:
    print(f"{ct}: {len(cancer_type_to_batch_indices[ct])}")


# ============================================================
# Check gene order
# ============================================================
gene_names = np.asarray(processed.adata_list[0].var_names.astype(str))
n_genes = len(gene_names)

for i in range(len(processed.adata_list)):
    gene_names_i = np.asarray(processed.adata_list[i].var_names.astype(str))

    if len(gene_names_i) != n_genes or not np.array_equal(gene_names_i, gene_names):
        raise ValueError(
            f"Gene order mismatch in batch {i}. "
            "Please align genes across adata_list before computing log fold changes."
        )


# ============================================================
# Main loop: cancer type -> batch indices -> median split log2FC
# ============================================================
sending_lfc_records = []
receiver_lfc_records = []

summary_records = []

for cancer_type_cur in cancer_type_order:

    print(f"\n==============================")
    print(f"Processing cancer type: {cancer_type_cur}")
    print(f"==============================")

    batch_indices_cur = cancer_type_to_batch_indices[cancer_type_cur]

    sending_entries = []
    receiver_entries = []

    total_fibroblast_cells = 0
    total_tumor_cells = 0
    total_valid_sending_cells = 0
    total_valid_receiver_cells = 0
    total_fibroblast_to_tumor_edges = 0

    for batch_idx in batch_indices_cur:

        adata_batch = processed.adata_list[batch_idx]
        sample_id_cur = batch_sample_ids[batch_idx]

        print(f"  Batch {batch_idx}: {sample_id_cur}")

        cell_types_batch = (
            adata_batch.obs[CELL_TYPE_COL]
            .astype(str)
            .to_numpy()
        )
        cell_types_batch_str = cell_types_batch.astype(str)

        tumor_cell_mask = np.char.endswith(cell_types_batch_str, TUMOR_SUFFIX)
        fibroblast_cell_mask = cell_types_batch_str == FIBROBLAST_LABEL

        tumor_idx = np.flatnonzero(tumor_cell_mask)
        fibroblast_idx = np.flatnonzero(fibroblast_cell_mask)

        if tumor_idx.size == 0 or fibroblast_idx.size == 0:
            print("    Missing tumor cells or Fibroblast cells. Skipping.")
            del adata_batch, cell_types_batch, cell_types_batch_str, tumor_cell_mask, fibroblast_cell_mask, tumor_idx, fibroblast_idx
            gc.collect()
            continue

        edge_index_cur = ensure_edge_index_e_by_2(
            to_numpy_maybe_torch(processed.spidernet_data[batch_idx]["edge_index"])
        )

        start_idx = edge_offsets[batch_idx]
        end_idx = edge_offsets[batch_idx + 1]

        mi_edge_cur = np.asarray(
            Factor_envir_use_mmap[start_idx:end_idx, mi_index],
            dtype=np.float32
        )

        if mi_edge_cur.shape[0] != edge_index_cur.shape[0]:
            raise ValueError(
                f"Mismatch between edge counts and MI values for batch {batch_idx}: "
                f"{edge_index_cur.shape[0]} edges versus {mi_edge_cur.shape[0]} MI rows."
            )

        src_idx = edge_index_cur[:, 0]
        dst_idx = edge_index_cur[:, 1]

        src_is_fibroblast = fibroblast_cell_mask[src_idx]
        dst_is_tumor = tumor_cell_mask[dst_idx]

        fibroblast_to_tumor_edge_mask = src_is_fibroblast & dst_is_tumor
        n_fibroblast_to_tumor_edges = int(fibroblast_to_tumor_edge_mask.sum())

        if n_fibroblast_to_tumor_edges == 0:
            print("    No Fibroblast->cancer-cell edges found. Skipping MI aggregation.")
            del (
                adata_batch,
                cell_types_batch,
                cell_types_batch_str,
                tumor_cell_mask,
                fibroblast_cell_mask,
                tumor_idx,
                fibroblast_idx,
                edge_index_cur,
                mi_edge_cur,
                src_idx,
                dst_idx,
                src_is_fibroblast,
                dst_is_tumor,
                fibroblast_to_tumor_edge_mask,
            )
            gc.collect()
            continue

        sending_mi_cur, receiver_mi_cur = compute_cell_level_mi(
            mi_edge=mi_edge_cur,
            edge_index=edge_index_cur,
            n_cells=adata_batch.n_obs,
            valid_edge_mask=fibroblast_to_tumor_edge_mask,
            if_sum=CELL_MI_IF_SUM,
        )

        # Sender-side analysis uses Fibroblast cells; receiver-side analysis uses cancer cells.
        sending_mi_fibroblast = sending_mi_cur[fibroblast_idx]
        receiver_mi_tumor = receiver_mi_cur[tumor_idx]

        valid_sending_mask = np.isfinite(sending_mi_fibroblast)
        valid_receiver_mask = np.isfinite(receiver_mi_tumor)

        n_valid_sending = int(valid_sending_mask.sum())
        n_valid_receiver = int(valid_receiver_mask.sum())

        print(
            f"    fibroblast cells: {fibroblast_idx.size:,}; "
            f"tumor cells: {tumor_idx.size:,}; "
            f"Fibroblast->tumor edges: {n_fibroblast_to_tumor_edges:,}; "
            f"valid sender Fibroblasts: {n_valid_sending:,}; "
            f"valid receiver tumor cells: {n_valid_receiver:,}"
        )

        total_fibroblast_cells += int(fibroblast_idx.size)
        total_tumor_cells += int(tumor_idx.size)
        total_fibroblast_to_tumor_edges += n_fibroblast_to_tumor_edges
        total_valid_sending_cells += n_valid_sending
        total_valid_receiver_cells += n_valid_receiver

        if n_valid_sending > 0:
            sending_entries.append({
                "batch_idx": batch_idx,
                "row_indices": fibroblast_idx[valid_sending_mask].astype(np.int64, copy=True),
                "x_values": sending_mi_fibroblast[valid_sending_mask].astype(np.float64, copy=True),
            })

        if n_valid_receiver > 0:
            receiver_entries.append({
                "batch_idx": batch_idx,
                "row_indices": tumor_idx[valid_receiver_mask].astype(np.int64, copy=True),
                "x_values": receiver_mi_tumor[valid_receiver_mask].astype(np.float64, copy=True),
            })

        del (
            adata_batch,
            cell_types_batch,
            cell_types_batch_str,
            tumor_cell_mask,
            fibroblast_cell_mask,
            tumor_idx,
            fibroblast_idx,
            edge_index_cur,
            mi_edge_cur,
            src_idx,
            dst_idx,
            src_is_fibroblast,
            dst_is_tumor,
            fibroblast_to_tumor_edge_mask,
            sending_mi_cur,
            receiver_mi_cur,
            sending_mi_fibroblast,
            receiver_mi_tumor,
            valid_sending_mask,
            valid_receiver_mask,
        )

        gc.collect()


    print("  Computing sending median-split log2FC...")
    sending_lfc, sending_stats = compute_log2fc_from_entries(
        entries=sending_entries,
        adata_list=processed.adata_list,
        n_genes=n_genes,
        pseudocount=LFC_PSEUDOCOUNT,
        split_mode=MEDIAN_SPLIT_MODE,
        chunk_size=EXPR_CHUNK_SIZE,
    )

    print("  Computing receiver median-split log2FC...")
    receiver_lfc, receiver_stats = compute_log2fc_from_entries(
        entries=receiver_entries,
        adata_list=processed.adata_list,
        n_genes=n_genes,
        pseudocount=LFC_PSEUDOCOUNT,
        split_mode=MEDIAN_SPLIT_MODE,
        chunk_size=EXPR_CHUNK_SIZE,
    )

    sending_lfc_records.append(
        pd.Series(sending_lfc, index=gene_names, name=cancer_type_cur)
    )

    receiver_lfc_records.append(
        pd.Series(receiver_lfc, index=gene_names, name=cancer_type_cur)
    )

    summary_records.append({
        "CancerType": cancer_type_cur,
        "MI_OI": MI_OI,
        "mi_index": mi_index,
        "cell_mi_if_sum": CELL_MI_IF_SUM,
        "median_split_mode_requested": MEDIAN_SPLIT_MODE,
        "lfc_pseudocount": LFC_PSEUDOCOUNT,
        "n_batches": len(batch_indices_cur),
        "n_fibroblast_cells_total": total_fibroblast_cells,
        "n_tumor_cells_total": total_tumor_cells,
        "n_fibroblast_to_tumor_edges_total": total_fibroblast_to_tumor_edges,
        "n_valid_sending_cells_total": total_valid_sending_cells,
        "n_valid_receiver_cells_total": total_valid_receiver_cells,

        "sending_median_mi": sending_stats["median_mi"],
        "sending_split_mode_used": sending_stats["split_mode_used"],
        "sending_n_cells_for_lfc": sending_stats["n_cells_for_lfc"],
        "sending_n_high": sending_stats["n_high"],
        "sending_n_low": sending_stats["n_low"],

        "receiver_median_mi": receiver_stats["median_mi"],
        "receiver_split_mode_used": receiver_stats["split_mode_used"],
        "receiver_n_cells_for_lfc": receiver_stats["n_cells_for_lfc"],
        "receiver_n_high": receiver_stats["n_high"],
        "receiver_n_low": receiver_stats["n_low"],
    })

    del (
        sending_entries,
        receiver_entries,
        sending_lfc,
        receiver_lfc,
        sending_stats,
        receiver_stats,
    )

    gc.collect()


# ============================================================
# Final outputs:
# cancer types x genes log2FC matrices
# ============================================================
sending_mi_gene_log2fc_df = pd.DataFrame(sending_lfc_records)
receiver_mi_gene_log2fc_df = pd.DataFrame(receiver_lfc_records)

sending_mi_gene_log2fc_df = sending_mi_gene_log2fc_df.loc[cancer_type_order, gene_names]
receiver_mi_gene_log2fc_df = receiver_mi_gene_log2fc_df.loc[cancer_type_order, gene_names]

mi_gene_log2fc_summary_df = pd.DataFrame(summary_records).set_index("CancerType")
mi_gene_log2fc_summary_df = mi_gene_log2fc_summary_df.loc[cancer_type_order]

print("\n[Done]")
print("sending_mi_gene_log2fc_df shape:", sending_mi_gene_log2fc_df.shape)
print("receiver_mi_gene_log2fc_df shape:", receiver_mi_gene_log2fc_df.shape)


In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch, Rectangle
from pathlib import Path

# ============================================================
# User settings
# ============================================================
SENDING_DF = sending_mi_gene_log2fc_df.copy()
RECEIVING_DF = receiver_mi_gene_log2fc_df.copy()

# ============================================================
# Sorting/filtering is based on ORIGINAL signed log2FC
# ============================================================
ABS_LOG2FC_SHOW_THRESHOLD = 0.20

# Maximum number of displayed columns assigned to each cancer type
# Each column is one Sending|gene or Receiving|gene feature
MAX_COLUMNS_PER_CANCER_TYPE = 50

# ============================================================
# Add gaps between top-gene blocks
# Block is defined by max_abs_log2fc_cancer_type
# ============================================================
ADD_GAP_BETWEEN_TOPGENE_BLOCKS = True
GAP_SIZE = 2
GAP_COLOR = "#FFFFFF"

# ============================================================
# Display is column-wise z-score of ABS(original log2FC)
# Values > Z_VMAX will be treated as Z_VMAX
# Values < Z_VMIN will be treated as Z_VMIN
# ============================================================
Z_VMIN = -2
Z_VCENTER = 0
Z_VMAX = 2

if not (Z_VMIN < Z_VCENTER < Z_VMAX):
    raise ValueError(
        f"For TwoSlopeNorm, need Z_VMIN < Z_VCENTER < Z_VMAX, "
        f"but got Z_VMIN={Z_VMIN}, Z_VCENTER={Z_VCENTER}, Z_VMAX={Z_VMAX}."
    )

SHOW_GENE_LABELS = False
MAX_GENE_LABELS_TO_SHOW = 80

# Top annotation colors
SENDING_COLOR = "#7dd2ec"
RECEIVING_COLOR = "#dc7b7f"

# Cancer-type color palette
CANCER_CMAP_NAME = "tab20"

# ============================================================
# Save path
# Short filenames to avoid Windows path-length FileNotFoundError
# ============================================================
sum_or_mean_tag = "sumMI" if CELL_MI_IF_SUM else "meanMI"

outdir = run_dirs["run_dir"] / f"{MI_OI}_lfcHM"
outdir.mkdir(parents=True, exist_ok=True)

threshold_tag = str(ABS_LOG2FC_SHOW_THRESHOLD).replace(".", "p")
topn_tag = f"top{MAX_COLUMNS_PER_CANCER_TYPE}"
gap_tag = f"gap{GAP_SIZE}" if ADD_GAP_BETWEEN_TOPGENE_BLOCKS else "nogap"

prefix = f"{MI_OI}_{sum_or_mean_tag}_absz_{threshold_tag}_{topn_tag}_{gap_tag}"

pdf_path = outdir / f"{prefix}.pdf"
svg_path = outdir / f"{prefix}.svg"

csv_path_original = outdir / f"{prefix}_orig.csv"
csv_path_abs = outdir / f"{prefix}_abs.csv"
csv_path_abs_zscore = outdir / f"{prefix}_absz.csv"
csv_path_abs_zscore_with_gaps = outdir / f"{prefix}_absz_gap.csv"

col_meta_path = outdir / f"{prefix}_colmeta.csv"
col_meta_with_gaps_path = outdir / f"{prefix}_colmeta_gap.csv"
all_col_meta_path = outdir / f"{prefix}_allcol.csv"
filtered_before_topn_meta_path = outdir / f"{prefix}_threshold.csv"
row_order_path = outdir / f"{prefix}_row.csv"
cancer_color_map_path = outdir / f"{prefix}_colors.csv"

print("Output directory:")
print(outdir)

print("\n[Output path lengths]")
for p in [
    pdf_path,
    svg_path,
    csv_path_original,
    csv_path_abs,
    csv_path_abs_zscore,
    csv_path_abs_zscore_with_gaps,
    col_meta_path,
    col_meta_with_gaps_path,
    all_col_meta_path,
    filtered_before_topn_meta_path,
    row_order_path,
    cancer_color_map_path,
]:
    print(len(str(p)), p)

# ============================================================
# Illustrator-friendly plotting settings
# ============================================================
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "Arial"

# ============================================================
# Helper functions
# ============================================================
def prepare_lfc_df(df, df_name):
    df = df.copy()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)

    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")

    if df.shape[0] < 1 or df.shape[1] < 1:
        raise ValueError(
            f"{df_name} has no valid rows or columns after removing all-NaN entries."
        )

    return df


def add_direction_prefix(df, direction):
    df_prefixed = df.copy()
    df_prefixed.columns = [f"{direction}|{str(g)}" for g in df.columns]
    return df_prefixed


def build_mixed_column_order(
    combined_df,
    threshold,
    max_columns_per_cancer_type=50,
):
    """
    Sorting/filtering is based on the ORIGINAL signed log2FC matrix.

    For each combined column, e.g. Sending|gene or Receiving|gene:
      1. compute max absolute log2FC across rows
      2. find row index where absolute log2FC is maximal
      3. keep if max abs log2FC > threshold
      4. within each cancer type / argmax row, keep top N columns by max abs log2FC
      5. sort all kept Sending and Receiving columns together by:
         - abs-argmax row index ascending
         - max abs log2FC descending
         - original global column position ascending
    """
    records = []

    for original_col_pos, combined_col in enumerate(combined_df.columns):
        vals = combined_df[combined_col].to_numpy(dtype=float)

        if "|" in combined_col:
            direction, gene = combined_col.split("|", 1)
        else:
            direction, gene = "Unknown", combined_col

        if np.all(np.isnan(vals)):
            abs_argmax_row_index = np.inf
            max_abs_log2fc_value = np.nan
            signed_log2fc_at_absmax = np.nan
            max_abs_log2fc_cancer_type = None
            pass_threshold = False
        else:
            abs_vals = np.abs(vals)
            abs_argmax_row_index = int(np.nanargmax(abs_vals))
            max_abs_log2fc_value = float(abs_vals[abs_argmax_row_index])
            signed_log2fc_at_absmax = float(vals[abs_argmax_row_index])
            max_abs_log2fc_cancer_type = combined_df.index[abs_argmax_row_index]
            pass_threshold = bool(max_abs_log2fc_value > threshold)

        records.append({
            "combined_column_id": combined_col,
            "Direction": direction,
            "Gene": gene,
            "abs_argmax_row_index_no_row_clustering": abs_argmax_row_index,
            "max_abs_log2fc_value": max_abs_log2fc_value,
            "signed_log2fc_at_absmax": signed_log2fc_at_absmax,
            "max_abs_log2fc_cancer_type": max_abs_log2fc_cancer_type,
            "pass_abs_log2fc_threshold": pass_threshold,
            "abs_log2fc_show_threshold": threshold,
            "original_global_col_pos": original_col_pos,
        })

    all_col_meta_df = pd.DataFrame(records)

    threshold_passed_df = all_col_meta_df.loc[
        all_col_meta_df["pass_abs_log2fc_threshold"] == True
    ].copy()

    if threshold_passed_df.empty:
        raise ValueError(
            f"No combined Sending/Receiving columns passed "
            f"ABS_LOG2FC_SHOW_THRESHOLD={threshold}. Please lower the threshold."
        )

    threshold_passed_sorted_df = threshold_passed_df.sort_values(
        by=[
            "abs_argmax_row_index_no_row_clustering",
            "max_abs_log2fc_value",
            "original_global_col_pos",
        ],
        ascending=[True, False, True],
        kind="mergesort"
    ).copy()

    if max_columns_per_cancer_type is not None:
        shown_col_meta_df = (
            threshold_passed_sorted_df
            .groupby(
                "abs_argmax_row_index_no_row_clustering",
                sort=False,
                group_keys=False
            )
            .head(max_columns_per_cancer_type)
            .copy()
        )
    else:
        shown_col_meta_df = threshold_passed_sorted_df.copy()

    shown_col_meta_df = shown_col_meta_df.sort_values(
        by=[
            "abs_argmax_row_index_no_row_clustering",
            "max_abs_log2fc_value",
            "original_global_col_pos",
        ],
        ascending=[True, False, True],
        kind="mergesort"
    ).reset_index(drop=True)

    shown_col_meta_df["display_col_index"] = np.arange(shown_col_meta_df.shape[0])

    ordered_cols = shown_col_meta_df["combined_column_id"].tolist()

    return ordered_cols, shown_col_meta_df, all_col_meta_df, threshold_passed_sorted_df


def columnwise_zscore(df):
    """
    Column-wise z-score normalization.
    NaNs are ignored in mean/std computation.
    If a column has std == 0 or all-NaN, valid entries are set to 0.
    """
    z_df = df.copy().astype(float)

    for col in z_df.columns:
        vals = z_df[col].to_numpy(dtype=float)
        valid = np.isfinite(vals)

        z_vals = np.full(vals.shape, np.nan, dtype=float)

        if valid.sum() == 0:
            pass
        else:
            mean_val = np.nanmean(vals)
            std_val = np.nanstd(vals)

            if (not np.isfinite(std_val)) or (std_val <= 0):
                z_vals[valid] = 0.0
            else:
                z_vals[valid] = (vals[valid] - mean_val) / std_val

        z_df[col] = z_vals

    return z_df


def make_discrete_color_map(labels, cmap_name="tab20"):
    """
    Generate deterministic colors for categorical labels.
    Returns label_to_color and a dataframe.
    """
    labels = list(labels)
    n = len(labels)

    base_cmap = plt.get_cmap(cmap_name)

    if n <= 1:
        color_list = [base_cmap(0)]
    else:
        color_list = [base_cmap(i / max(n - 1, 1)) for i in range(n)]

    label_to_color = {
        label: color_list[i]
        for i, label in enumerate(labels)
    }

    color_df = pd.DataFrame({
        "CancerType": labels,
        "color_rgba": [str(label_to_color[label]) for label in labels],
    })

    return label_to_color, color_df


def insert_gaps_between_column_blocks(
    data_df,
    col_meta_df,
    block_col="max_abs_log2fc_cancer_type",
    gap_size=2,
    gap_prefix="__GAP__",
):
    """
    Insert NaN columns between consecutive blocks defined by block_col.

    data_df columns must match col_meta_df['combined_column_id'].
    """
    if gap_size is None or gap_size <= 0:
        meta_no_gap = col_meta_df.copy()
        meta_no_gap["is_gap"] = False
        meta_no_gap["display_col_index_with_gap"] = np.arange(meta_no_gap.shape[0])
        return data_df.copy(), meta_no_gap, []

    mat_parts = []
    col_names = []
    meta_rows = []
    gap_positions = []

    prev_block = None
    gap_counter = 0
    n_rows = data_df.shape[0]

    for _, row in col_meta_df.iterrows():
        cur_block = str(row[block_col])
        cur_col = row["combined_column_id"]

        if prev_block is not None and cur_block != prev_block:
            gap_start_pos = len(col_names)
            gap_positions.append(gap_start_pos)

            gap_mat = np.full((n_rows, gap_size), np.nan, dtype=float)
            mat_parts.append(gap_mat)

            for j in range(gap_size):
                gap_col = f"{gap_prefix}_{gap_counter}_{j}"
                col_names.append(gap_col)

                gap_row = {c: np.nan for c in col_meta_df.columns}
                gap_row.update({
                    "combined_column_id": gap_col,
                    "Direction": "Gap",
                    "Gene": "",
                    "max_abs_log2fc_cancer_type": "Gap",
                    "is_gap": True,
                    "display_col_index_with_gap": len(col_names) - 1,
                    "gap_group_index": gap_counter,
                })
                meta_rows.append(gap_row)

            gap_counter += 1

        if cur_col not in data_df.columns:
            raise ValueError(f"Column {cur_col} from col_meta_df is not in data_df.")

        mat_parts.append(data_df[[cur_col]].to_numpy(dtype=float))
        col_names.append(cur_col)

        row_dict = row.to_dict()
        row_dict["is_gap"] = False
        row_dict["display_col_index_with_gap"] = len(col_names) - 1
        row_dict["gap_group_index"] = np.nan
        meta_rows.append(row_dict)

        prev_block = cur_block

    data_mat_with_gaps = np.concatenate(mat_parts, axis=1)

    data_df_with_gaps = pd.DataFrame(
        data_mat_with_gaps,
        index=data_df.index,
        columns=col_names,
    )

    col_meta_with_gaps = pd.DataFrame(meta_rows)

    return data_df_with_gaps, col_meta_with_gaps, gap_positions


def draw_vector_annotation_bar(
    ax,
    labels,
    label_to_color,
    ylabel=None,
    gap_label="Gap",
    gap_color="#FFFFFF",
):
    """
    Draw an annotation bar using vector Rectangle patches instead of imshow.

    This avoids Adobe Illustrator issues where imshow-based annotation bars
    may be interpreted as a single raster/image object or lose categorical colors.
    """
    labels = [str(x) for x in labels]
    n = len(labels)

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.5, 0.5)

    for i, lab in enumerate(labels):
        color = label_to_color.get(lab, gap_color)

        rect = Rectangle(
            (i - 0.5, -0.5),
            1.0,
            1.0,
            facecolor=color,
            edgecolor="none",
            linewidth=0,
            antialiased=False,
        )
        ax.add_patch(rect)

    ax.set_xticks([])
    ax.set_yticks([])

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            rotation=0,
            ha="right",
            va="center",
            fontsize=8,
            labelpad=18,
        )

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_facecolor(gap_color)


# ============================================================
# Prepare matrices
# ============================================================
sending_df = prepare_lfc_df(SENDING_DF, "sending_mi_gene_log2fc_df")
receiving_df = prepare_lfc_df(RECEIVING_DF, "receiver_mi_gene_log2fc_df")

# ============================================================
# Align row order
# Keep original row order from sending_df and use common cancer types
# ============================================================
common_rows = [idx for idx in sending_df.index if idx in receiving_df.index]

if len(common_rows) == 0:
    raise ValueError("No common cancer types found between sending and receiving matrices.")

sending_df = sending_df.loc[common_rows, :]
receiving_df = receiving_df.loc[common_rows, :]

row_order_df = pd.DataFrame({
    "CancerType": common_rows,
    "row_index": np.arange(len(common_rows))
})
row_order_df.to_csv(row_order_path, index=False)

print(f"[Sending input matrix after dropping all-NaN rows/columns] {sending_df.shape}")
print(f"[Receiving input matrix after dropping all-NaN rows/columns] {receiving_df.shape}")
print(f"[Common cancer types used for combined heatmap] {len(common_rows)}")

# ============================================================
# Combine Sending and Receiving BEFORE sorting
# Sorting/filtering is based on ORIGINAL signed log2FC
# ============================================================
sending_prefixed_df = add_direction_prefix(sending_df, "Sending")
receiving_prefixed_df = add_direction_prefix(receiving_df, "Receiving")

combined_all_df = pd.concat(
    [sending_prefixed_df, receiving_prefixed_df],
    axis=1
)

# ============================================================
# Mixed column filtering and sorting
# First threshold, then top N per cancer type / argmax row
# ============================================================
ordered_cols, col_meta_df, all_col_meta_df, threshold_passed_df = build_mixed_column_order(
    combined_df=combined_all_df,
    threshold=ABS_LOG2FC_SHOW_THRESHOLD,
    max_columns_per_cancer_type=MAX_COLUMNS_PER_CANCER_TYPE,
)

combined_plot_df_original = combined_all_df.loc[:, ordered_cols].copy()

combined_plot_df_original.to_csv(csv_path_original)
col_meta_df.to_csv(col_meta_path, index=False)
all_col_meta_df.to_csv(all_col_meta_path, index=False)
threshold_passed_df.to_csv(filtered_before_topn_meta_path, index=False)

n_send_shown = int((col_meta_df["Direction"] == "Sending").sum())
n_recv_shown = int((col_meta_df["Direction"] == "Receiving").sum())

print(
    f"[Combined columns passing max |log2FC| threshold > {ABS_LOG2FC_SHOW_THRESHOLD}] "
    f"{threshold_passed_df.shape[0]} / {combined_all_df.shape[1]}"
)
print(
    f"[After keeping at most {MAX_COLUMNS_PER_CANCER_TYPE} columns per cancer type] "
    f"{combined_plot_df_original.shape[1]} columns shown"
)
print(f"  Sending shown: {n_send_shown}")
print(f"  Receiving shown: {n_recv_shown}")

print("\n[Shown columns per cancer type / abs-argmax row]")
display(
    col_meta_df
    .groupby("max_abs_log2fc_cancer_type")
    .size()
    .rename("n_columns_shown")
    .to_frame()
)

# ============================================================
# Display matrix = column-wise z-score of ABS(original log2FC)
# ============================================================
combined_plot_df_abs = combined_plot_df_original.abs()
combined_plot_df_abs.to_csv(csv_path_abs)

combined_plot_df_abs_zscore = columnwise_zscore(combined_plot_df_abs)
combined_plot_df_abs_zscore.to_csv(csv_path_abs_zscore)

# ============================================================
# Insert gaps between top-gene blocks
# ============================================================
if ADD_GAP_BETWEEN_TOPGENE_BLOCKS:
    combined_plot_df_abs_zscore_plot, col_meta_plot_df, gap_positions = (
        insert_gaps_between_column_blocks(
            data_df=combined_plot_df_abs_zscore,
            col_meta_df=col_meta_df,
            block_col="max_abs_log2fc_cancer_type",
            gap_size=GAP_SIZE,
            gap_prefix="__TOPGENE_BLOCK_GAP__",
        )
    )
else:
    combined_plot_df_abs_zscore_plot = combined_plot_df_abs_zscore.copy()
    col_meta_plot_df = col_meta_df.copy()
    col_meta_plot_df["is_gap"] = False
    col_meta_plot_df["display_col_index_with_gap"] = np.arange(col_meta_plot_df.shape[0])
    gap_positions = []

combined_plot_df_abs_zscore_plot.to_csv(csv_path_abs_zscore_with_gaps)
col_meta_plot_df.to_csv(col_meta_with_gaps_path, index=False)

print(f"[Gap columns inserted] {len(gap_positions) * GAP_SIZE}")
print(f"[Gap start positions] {gap_positions}")

display_mat = combined_plot_df_abs_zscore_plot.to_numpy(dtype=float)

# Clip z-scored abs(log2FC) values to [Z_VMIN, Z_VMAX]
display_mat_clipped = np.clip(display_mat, Z_VMIN, Z_VMAX)
display_mat_clipped[np.isnan(display_mat)] = np.nan

# ============================================================
# Top annotation bars metadata
# Gap columns are rendered white.
# ============================================================
cancer_types_for_columns = (
    col_meta_plot_df["max_abs_log2fc_cancer_type"]
    .astype(str)
    .tolist()
)

cancer_type_order_for_colors = [
    ct for ct in common_rows
    if ct in set(cancer_types_for_columns)
]

for ct in cancer_types_for_columns:
    if ct not in cancer_type_order_for_colors and ct != "Gap":
        cancer_type_order_for_colors.append(ct)

cancer_label_to_color, cancer_color_df = make_discrete_color_map(
    cancer_type_order_for_colors,
    cmap_name=CANCER_CMAP_NAME
)
cancer_color_df.to_csv(cancer_color_map_path, index=False)

# ============================================================
# Colormap for column-wise z-scored abs(log2FC)
# ============================================================
abs_z_cmap = plt.get_cmap("RdYlBu_r").copy()
abs_z_cmap.set_bad(GAP_COLOR)

norm = TwoSlopeNorm(
    vmin=Z_VMIN,
    vcenter=Z_VCENTER,
    vmax=Z_VMAX
)

masked_mat = np.ma.masked_invalid(display_mat_clipped)

# ============================================================
# Figure size
# ============================================================
n_rows, n_cols = combined_plot_df_abs_zscore_plot.shape

fig_width = max(9.0, min(24, 0.18 * n_cols + 5.0))
fig_height = max(5.0, min(11.0, 0.45 * n_rows + 3.8))

fig = plt.figure(figsize=(fig_width, fig_height))

gs = GridSpec(
    nrows=3,
    ncols=2,
    height_ratios=[0.22, 0.22, 6.0],
    width_ratios=[6.5, 0.45],
    hspace=0.04,
    wspace=0.06
)

ax_top_cancer = fig.add_subplot(gs[0, 0])
ax_top_direction = fig.add_subplot(gs[1, 0])
ax_heatmap = fig.add_subplot(gs[2, 0])
ax_cbar = fig.add_subplot(gs[2, 1])

# ============================================================
# Top annotation bar 1: cancer type
# Illustrator-friendly: vector rectangles, not imshow
# ============================================================
cancer_bar_labels = (
    col_meta_plot_df["max_abs_log2fc_cancer_type"]
    .astype(str)
    .tolist()
)

cancer_bar_color_map = {"Gap": GAP_COLOR}
cancer_bar_color_map.update(cancer_label_to_color)

draw_vector_annotation_bar(
    ax=ax_top_cancer,
    labels=cancer_bar_labels,
    label_to_color=cancer_bar_color_map,
    ylabel="Top\ncancer",
    gap_label="Gap",
    gap_color=GAP_COLOR,
)

# ============================================================
# Top annotation bar 2: sending / receiving
# Illustrator-friendly: vector rectangles, not imshow
# ============================================================
direction_bar_labels = (
    col_meta_plot_df["Direction"]
    .astype(str)
    .tolist()
)

direction_bar_color_map = {
    "Gap": GAP_COLOR,
    "Sending": SENDING_COLOR,
    "Receiving": RECEIVING_COLOR,
}

draw_vector_annotation_bar(
    ax=ax_top_direction,
    labels=direction_bar_labels,
    label_to_color=direction_bar_color_map,
    ylabel="MI\nside",
    gap_label="Gap",
    gap_color=GAP_COLOR,
)

# ============================================================
# Heatmap
# ============================================================
im = ax_heatmap.imshow(
    masked_mat,
    aspect="auto",
    interpolation="nearest",
    cmap=abs_z_cmap,
    norm=norm,
    rasterized=False,
)

ax_heatmap.set_yticks(np.arange(n_rows))
ax_heatmap.set_yticklabels(
    combined_plot_df_abs_zscore_plot.index.tolist(),
    fontsize=10
)

if SHOW_GENE_LABELS and n_cols <= MAX_GENE_LABELS_TO_SHOW:
    xlabels = [
        "" if bool(is_gap) else str(gene)
        for is_gap, gene in zip(
            col_meta_plot_df["is_gap"].tolist(),
            col_meta_plot_df["Gene"].tolist()
        )
    ]
    ax_heatmap.set_xticks(np.arange(n_cols))
    ax_heatmap.set_xticklabels(
        xlabels,
        rotation=90,
        fontsize=7
    )
else:
    ax_heatmap.set_xticks([])

ax_heatmap.set_xlabel(
    f"Top {MAX_COLUMNS_PER_CANCER_TYPE} Sending/Receiving columns per cancer type "
    f"with max |log2FC| > {ABS_LOG2FC_SHOW_THRESHOLD}; "
    f"ordered by row of maximal |log2FC|",
    fontsize=12
)
ax_heatmap.set_ylabel("Cancer type", fontsize=12)

ax_heatmap.tick_params(axis="both", length=0)

for spine in ax_heatmap.spines.values():
    spine.set_visible(False)

# ============================================================
# Colorbar
# ============================================================
cbar = fig.colorbar(
    im,
    cax=ax_cbar,
    extend="both"
)

cbar_ticks = np.linspace(Z_VMIN, Z_VMAX, 5)
cbar.set_ticks(cbar_ticks)
cbar.set_ticklabels([f"{x:.2g}" for x in cbar_ticks])
cbar.set_label("Column-wise z-score of |log2FC|", fontsize=11)
cbar.ax.tick_params(labelsize=9)

cbar.outline.set_visible(True)
cbar.outline.set_linewidth(0.6)

# ============================================================
# Legends
# ============================================================
direction_legend_handles = [
    Patch(facecolor=SENDING_COLOR, edgecolor="none", label="Sending"),
    Patch(facecolor=RECEIVING_COLOR, edgecolor="none", label="Receiving"),
]

direction_legend = ax_heatmap.legend(
    handles=direction_legend_handles,
    title="MI side",
    loc="upper left",
    bbox_to_anchor=(1.18, 1.02),
    frameon=False,
    fontsize=9,
    title_fontsize=10
)

ax_heatmap.add_artist(direction_legend)

cancer_legend_handles = [
    Patch(
        facecolor=cancer_label_to_color[ct],
        edgecolor="none",
        label=ct
    )
    for ct in cancer_type_order_for_colors
]

ax_heatmap.legend(
    handles=cancer_legend_handles,
    title="Top gene cancer type",
    loc="upper left",
    bbox_to_anchor=(1.18, 0.70),
    frameon=False,
    fontsize=8,
    title_fontsize=10
)

# ============================================================
# Title
# ============================================================
fig.suptitle(
    f"{MI_OI} median-split gene-expression pattern: Sending + Receiving "
    f"(top {MAX_COLUMNS_PER_CANCER_TYPE} per cancer type; shown as column-wise z-score of |log2FC|)",
    fontsize=13,
    y=1.01
)

# ============================================================
# Save
# ============================================================
fig.savefig(
    pdf_path,
    format="pdf",
    bbox_inches="tight",
    transparent=True
)

fig.savefig(
    svg_path,
    format="svg",
    bbox_inches="tight",
    transparent=True
)

plt.show()
plt.close(fig)

gc.collect()

print("Saved:")
print(pdf_path)
print(svg_path)
print(csv_path_original)
print(csv_path_abs)
print(csv_path_abs_zscore)
print(csv_path_abs_zscore_with_gaps)
print(col_meta_path)
print(col_meta_with_gaps_path)
print(all_col_meta_path)
print(filtered_before_topn_meta_path)
print(row_order_path)
print(cancer_color_map_path)

print("\nFinal combined plotted matrix shape without gaps:")
print(combined_plot_df_abs_zscore.shape)

print("\nFinal plotted matrix shape with gaps:")
print(combined_plot_df_abs_zscore_plot.shape)

print("\nShown mixed column metadata preview without gaps:")
display(col_meta_df.head(30))

print("\nShown mixed column metadata preview with gaps:")
display(col_meta_plot_df.head(40))

print("\nAll mixed column metadata preview:")
display(
    all_col_meta_df
    .sort_values("max_abs_log2fc_value", ascending=False)
    .head(30)
)

print("\nCancer type color map:")
display(cancer_color_df)

In [ ]:
# ============================================================
# GO / KEGG enrichment for Fibroblast-side and Tumor-cell-side
# top genes from Fibroblast -> tumor MI-4 heatmap
# ------------------------------------------------------------
# Input:
#   col_meta_df from the previous heatmap cell.
#
# Expected col_meta_df columns from current heatmap:
#   Gene
#   Direction
#   max_abs_log2fc_cancer_type
#   max_abs_log2fc_value
#
# Biological mapping:
#   Direction == "Sending"   -> Fibroblast-side genes
#   Direction == "Receiving" -> Tumor-cell-side genes
#
# Enrichment reference genes:
#   Enrichr term reference genes are retrieved from the corresponding
#   Enrichr library and saved for each CancerType x CellSide x GO/KEGG term.
#   These reference-gene files are used later by the cascade-triplet DEG
#   enrichment analysis.
# ============================================================

import gc
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize

# ============================================================
# User settings
# ============================================================

COL_META_DF = col_meta_df.copy()

DIRECTION_TO_CELL_SIDE = {
    "Sending": "Fibroblast",
    "Receiving": "TumorCell",
}

CELL_SIDE_DISPLAY = {
    "Fibroblast": "Fibroblast-side",
    "TumorCell": "Tumor-cell-side",
}

UPPERCASE_GENES = True
MIN_GENES_FOR_ENRICHMENT = 5

DEBUG_FAST_RUN = False
DEBUG_N_GENE_LISTS = None

# If True, use requested names directly. If False, query Enrichr for available names.
DEBUG_SKIP_LIBRARY_RESOLUTION = True

ENRICHR_LIBRARIES_REQUESTED = [
    "GO_Biological_Process_2023",
    "KEGG_2021_Human",
]

ORGANISM = "human"

MAX_ENRICHR_RETRIES = 2
ENRICHR_SLEEP_SECONDS = 2

# Visualization
TOP_TERMS_PER_CANCER_PER_LIBRARY = 4
MAX_UNIQUE_TERMS_PER_LIBRARY = 60
PLOT_ADJ_PVALUE_CUTOFF = 1.0

COLOR_MAX_NEGLOG10 = 10
SIZE_BY = "gene_ratio_input"  # "gene_ratio_input" or "overlap_n"

# ============================================================
# Output paths
# ============================================================

if "MI_OI" in globals():
    mi_name_for_out = MI_OI
elif "MI_OI_RERUN" in globals():
    mi_name_for_out = MI_OI_RERUN
else:
    mi_name_for_out = "MI4"

if "CELL_MI_IF_SUM" in globals():
    sum_or_mean_tag = "sumMI" if CELL_MI_IF_SUM else "meanMI"
elif "CELL_MI_IF_SUM_RERUN" in globals():
    sum_or_mean_tag = "sumMI" if CELL_MI_IF_SUM_RERUN else "meanMI"
else:
    sum_or_mean_tag = "sumMI"

if "outdir" in globals():
    heatmap_outdir = Path(outdir)
else:
    heatmap_outdir = Path(run_dirs["run_dir"]) / f"{mi_name_for_out}_lfcHM"

enrich_outdir = heatmap_outdir / "TopGene_Enrichment_GO_KEGG_Fibroblast_TumorCell"
enrich_outdir.mkdir(parents=True, exist_ok=True)

mode_tag = "Fibroblast_TumorCell_separate"

gene_list_path = enrich_outdir / f"{mi_name_for_out}_{sum_or_mean_tag}_{mode_tag}_topGenes_byCancerType.csv"
enrichr_result_path = enrich_outdir / f"{mi_name_for_out}_{sum_or_mean_tag}_{mode_tag}_GO_KEGG_Enrichr_results.csv"
plot_table_path = enrich_outdir / f"{mi_name_for_out}_{sum_or_mean_tag}_{mode_tag}_GO_KEGG_dotplot_table.csv"

# Export full reference gene sets from the Enrichr libraries.
term_reference_wide_path = enrich_outdir / f"{mi_name_for_out}_{sum_or_mean_tag}_{mode_tag}_GO_KEGG_term_reference_genes_wide.csv"
term_reference_long_path = enrich_outdir / f"{mi_name_for_out}_{sum_or_mean_tag}_{mode_tag}_GO_KEGG_term_reference_genes_long.csv"
term_reference_missing_path = enrich_outdir / f"{mi_name_for_out}_{sum_or_mean_tag}_{mode_tag}_GO_KEGG_term_reference_genes_missing.csv"

print("Enrichment output directory:")
print(enrich_outdir)

# ============================================================
# Plotting style
# ============================================================

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "Arial"

dot_cmap = LinearSegmentedColormap.from_list(
    "BluePurpleRed",
    ["#1400FC", "#CA0089", "#FE0006"]
)

# ============================================================
# Helper functions
# ============================================================

def clean_gene_symbol(gene):
    gene = str(gene).strip()
    if UPPERCASE_GENES:
        gene = gene.upper()
    return gene


def unique_preserve_order(items):
    seen = set()
    out = []

    for x in items:
        if pd.isna(x):
            continue

        x = clean_gene_symbol(x)

        if x == "" or x in seen:
            continue

        seen.add(x)
        out.append(x)

    return out


def get_available_enrichr_libraries():
    try:
        import gseapy as gp
        libs = gp.get_library_name(organism=ORGANISM)
        return list(libs)

    except Exception as e:
        print("[Warning] Could not fetch Enrichr library names. Using requested names directly.", flush=True)
        print("Reason:", repr(e), flush=True)
        return None


def resolve_enrichr_libraries(requested_libraries, available_libraries=None):
    if available_libraries is None:
        return requested_libraries

    available_set = set(available_libraries)
    resolved = []

    for lib in requested_libraries:
        if lib in available_set:
            resolved.append(lib)
            continue

        lib_lower = lib.lower()

        if "go_biological_process" in lib_lower:
            candidates = [
                x for x in available_libraries
                if "go_biological_process" in x.lower()
            ]

        elif "kegg" in lib_lower:
            candidates = [
                x for x in available_libraries
                if "kegg" in x.lower() and "human" in x.lower()
            ]

        else:
            candidates = [
                x for x in available_libraries
                if lib_lower in x.lower()
            ]

        if len(candidates) == 0:
            print(f"[Warning] Could not resolve library: {lib}. Keeping requested name.", flush=True)
            resolved.append(lib)
        else:
            chosen = sorted(candidates)[-1]
            print(f"[Library resolved] {lib} -> {chosen}", flush=True)
            resolved.append(chosen)

    return resolved


def run_enrichr_with_retry(gene_list, library_name, gene_list_name):
    try:
        import gseapy as gp
    except ImportError:
        raise ImportError(
            "gseapy is required for Enrichr analysis. Install with: pip install gseapy"
        )

    last_error = None

    for attempt in range(1, MAX_ENRICHR_RETRIES + 1):
        try:
            print(
                f"    Enrichr request start | {gene_list_name} | {library_name} | "
                f"attempt {attempt}/{MAX_ENRICHR_RETRIES}",
                flush=True,
            )

            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=library_name,
                organism=ORGANISM,
                outdir=None,
                cutoff=1.0,
                no_plot=True,
            )

            if enr.results is None or enr.results.empty:
                print(
                    f"    Empty result | {gene_list_name} | {library_name}",
                    flush=True,
                )
                return pd.DataFrame()

            res = enr.results.copy()
            res["Library"] = library_name
            res["GeneListName"] = gene_list_name
            res["n_input_genes"] = len(gene_list)

            print(
                f"    Done | {gene_list_name} | {library_name} | {res.shape[0]} rows",
                flush=True,
            )

            return res

        except Exception as e:
            last_error = e
            print(
                f"[Enrichr warning] {gene_list_name} | {library_name} | "
                f"attempt {attempt}/{MAX_ENRICHR_RETRIES} failed: {repr(e)}",
                flush=True,
            )

            if attempt < MAX_ENRICHR_RETRIES:
                sleep_time = ENRICHR_SLEEP_SECONDS * attempt
                print(f"    Sleeping {sleep_time} seconds before retry...", flush=True)
                time.sleep(sleep_time)

    print(
        f"[Enrichr failed] {gene_list_name} | {library_name}. "
        f"Last error: {repr(last_error)}",
        flush=True,
    )

    return pd.DataFrame()


def parse_overlap(overlap):
    if pd.isna(overlap):
        return np.nan, np.nan

    overlap = str(overlap)

    if "/" not in overlap:
        return np.nan, np.nan

    a, b = overlap.split("/", 1)

    try:
        return int(a), int(b)
    except Exception:
        return np.nan, np.nan


def wrap_term(term, width=52):
    return "\n".join(textwrap.wrap(str(term), width=width))


def infer_cancer_type_order(meta_df):
    """
    Use the same row order as the heatmap whenever possible.
    """
    if "combined_plot_df_abs_zscore_plot" in globals():
        candidate_order = [str(x) for x in combined_plot_df_abs_zscore_plot.index]
    elif "combined_plot_df_abs_zscore" in globals():
        candidate_order = [str(x) for x in combined_plot_df_abs_zscore.index]
    elif "combined_plot_df_original" in globals():
        candidate_order = [str(x) for x in combined_plot_df_original.index]
    elif "common_rows" in globals():
        candidate_order = [str(x) for x in common_rows]
    elif "CANCERTYPE_ORDER" in globals():
        candidate_order = [str(x) for x in CANCERTYPE_ORDER]
    else:
        candidate_order = []

    cancer_types_present = set(meta_df["CancerType"].astype(str))
    cancer_type_order = [ct for ct in candidate_order if ct in cancer_types_present]

    for ct in meta_df["CancerType"].astype(str).tolist():
        if ct not in cancer_type_order:
            cancer_type_order.append(ct)

    return cancer_type_order


def _load_enrichr_library_reference_genes(library_name):
    """
    Return {Term: [reference genes]} for one Enrichr library.
    The returned genes are standardized with clean_gene_symbol().
    """
    try:
        import gseapy as gp
    except ImportError:
        raise ImportError("gseapy is required to retrieve Enrichr reference gene sets.")

    try:
        lib = gp.get_library(name=library_name, organism=ORGANISM)
    except TypeError:
        # Compatibility with older gseapy versions.
        lib = gp.get_library(library_name, organism=ORGANISM)

    out = {}
    for term, genes in lib.items():
        out[str(term)] = unique_preserve_order(genes)
    return out


def attach_reference_genes_to_enrichr_results(enrichr_df):
    """
    For each enriched Term, retrieve the full reference genes from the
    corresponding Enrichr library. Save both wide and long reference-gene tables.
    """
    if enrichr_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    reference_cache = {}
    wide_records = []
    long_records = []
    missing_records = []

    needed_libraries = sorted(enrichr_df["Library"].dropna().astype(str).unique())

    for lib in needed_libraries:
        try:
            reference_cache[lib] = _load_enrichr_library_reference_genes(lib)
            print(f"[Reference library loaded] {lib}: {len(reference_cache[lib])} terms", flush=True)
        except Exception as e:
            print(f"[Reference library warning] Failed to load {lib}: {repr(e)}", flush=True)
            reference_cache[lib] = {}

    for _, row in enrichr_df.iterrows():
        library_name = str(row["Library"])
        term = str(row["Term"])
        reference_genes = reference_cache.get(library_name, {}).get(term, [])

        base = {
            "CancerType": row.get("CancerType", np.nan),
            "CellSide": row.get("CellSide", np.nan),
            "Direction": row.get("Direction", np.nan),
            "GeneListName": row.get("GeneListName", np.nan),
            "Library": library_name,
            "Term": term,
            "Adjusted P-value": row.get("Adjusted P-value", np.nan),
            "P-value": row.get("P-value", np.nan),
            "Overlap": row.get("Overlap", np.nan),
            "n_input_genes": row.get("n_input_genes", np.nan),
            "overlap_n": row.get("overlap_n", np.nan),
            "gene_set_size_reported_by_enrichr": row.get("gene_set_size", np.nan),
        }

        if len(reference_genes) == 0:
            missing_records.append(base.copy())

        wide_records.append({
            **base,
            "n_reference_genes": int(len(reference_genes)),
            "ReferenceGenes": ";".join(reference_genes),
        })

        for gene in reference_genes:
            long_records.append({
                **base,
                "ReferenceGene": gene,
            })

    wide_df = pd.DataFrame(wide_records)
    long_df = pd.DataFrame(long_records)
    missing_df = pd.DataFrame(missing_records)

    return wide_df, long_df, missing_df


def make_dotplot_for_library_and_cell_side(plot_df, library_name, cell_side, cancer_type_order, output_prefix):
    """
    Dotplot for one library and one biological side.

    x = cancer type
    y = enriched term
    color = -log10 adjusted P-value
    size = gene ratio or overlap count
    """
    df = plot_df.loc[
        (plot_df["Library"] == library_name)
        & (plot_df["CellSide"] == cell_side)
    ].copy()

    if df.empty:
        print(f"[Skip plot] No enrichment terms for {library_name} | {cell_side}", flush=True)
        return

    df["CancerType"] = df["CancerType"].astype(str)
    df["Term"] = df["Term"].astype(str)

    x_order = [str(x) for x in cancer_type_order if str(x) in set(df["CancerType"])]
    for x in df["CancerType"].unique():
        if x not in x_order:
            x_order.append(x)

    if len(x_order) == 0:
        print(f"[Skip plot] No matching cancer types for {library_name} | {cell_side}", flush=True)
        return

    x_to_idx = {x: i for i, x in enumerate(x_order)}
    df["x"] = df["CancerType"].map(x_to_idx)
    df = df.dropna(subset=["x", "Term", "neglog10_adj_p"]).copy()

    if df.empty:
        print(f"[Skip plot] No valid dotplot entries for {library_name} | {cell_side}", flush=True)
        return

    term_order_records = []
    for term, sub_df in df.groupby("Term", sort=False):
        sub_df = sub_df.copy()
        idx_max = sub_df["neglog10_adj_p"].idxmax()
        argmax_cancer_type = str(sub_df.loc[idx_max, "CancerType"])
        argmax_cancer_type_index = int(x_to_idx.get(argmax_cancer_type, 10**9))
        max_neglog10 = float(sub_df["neglog10_adj_p"].max())
        min_adj_p = float(sub_df["Adjusted P-value"].min())
        n_cancer_types = int(sub_df["CancerType"].nunique())

        term_order_records.append({
            "Term": term,
            "CellSide": cell_side,
            "argmax_cancer_type": argmax_cancer_type,
            "argmax_cancer_type_index": argmax_cancer_type_index,
            "max_neglog10_adj_p": max_neglog10,
            "min_adj_p": min_adj_p,
            "n_cancer_types": n_cancer_types,
        })

    term_order_df = pd.DataFrame(term_order_records)
    term_order_df = term_order_df.sort_values(
        by=["argmax_cancer_type_index", "max_neglog10_adj_p", "min_adj_p", "Term"],
        ascending=[True, False, True, True],
        kind="mergesort",
    )

    keep_terms = term_order_df["Term"].head(MAX_UNIQUE_TERMS_PER_LIBRARY).tolist()
    df = df.loc[df["Term"].isin(keep_terms)].copy()

    term_order_top_to_bottom = [term for term in keep_terms if term in set(df["Term"])]
    if len(term_order_top_to_bottom) == 0:
        print(f"[Skip plot] No terms remain for {library_name} | {cell_side}", flush=True)
        return

    term_order_for_axis = term_order_top_to_bottom[::-1]
    y_to_idx = {t: i for i, t in enumerate(term_order_for_axis)}
    df["y"] = df["Term"].map(y_to_idx)
    df = df.dropna(subset=["x", "y"]).copy()

    if df.empty:
        print(f"[Skip plot] No mappable terms for {library_name} | {cell_side}", flush=True)
        return

    safe_lib_name = str(library_name).replace("/", "_").replace(" ", "_").replace(":", "_")
    safe_cell_side = str(cell_side).replace("/", "_").replace(" ", "_")
    term_order_path = enrich_outdir / f"{output_prefix}_{safe_cell_side}_{safe_lib_name}_termOrder.csv"
    term_order_df.loc[term_order_df["Term"].isin(keep_terms)].to_csv(term_order_path, index=False)

    if SIZE_BY == "overlap_n":
        size_values = df["overlap_n"].astype(float).values
        size_label = "Overlap genes"
        size_ref_values = np.array([2, 5, 10], dtype=float)
        size_area = 35 + 35 * size_values
        def ref_to_area(v):
            return 35 + 35 * v
    else:
        size_values = df["gene_ratio_input"].astype(float).values
        size_label = "Gene ratio"
        size_ref_values = np.array([0.05, 0.10, 0.20], dtype=float)
        size_area = 80 + 900 * size_values
        def ref_to_area(v):
            return 80 + 900 * v

    color_values = df["neglog10_adj_p"].clip(upper=COLOR_MAX_NEGLOG10).values

    fig_width = max(7.0, 0.58 * len(x_order) + 4.5)
    fig_height = max(5.0, 0.30 * len(term_order_for_axis) + 2.5)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    sc = ax.scatter(
        df["x"],
        df["y"],
        s=size_area,
        c=color_values,
        cmap=dot_cmap,
        norm=Normalize(vmin=0, vmax=COLOR_MAX_NEGLOG10),
        edgecolor="black",
        linewidth=0.35,
    )

    ax.set_xticks(np.arange(len(x_order)))
    ax.set_xticklabels(x_order, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(np.arange(len(term_order_for_axis)))
    ax.set_yticklabels([wrap_term(t, width=52) for t in term_order_for_axis], fontsize=8)
    ax.set_xlabel(f"Cancer type ({CELL_SIDE_DISPLAY.get(cell_side, cell_side)} top genes)", fontsize=11)
    ax.set_ylabel(library_name, fontsize=11)
    ax.tick_params(axis="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label("-log10 adjusted P-value", fontsize=10)
    cbar.set_ticks(np.linspace(0, COLOR_MAX_NEGLOG10, 6))
    cbar.ax.tick_params(labelsize=8)

    size_handles = []
    for v in size_ref_values:
        size_handles.append(
            plt.Line2D(
                [0], [0],
                marker="o",
                linestyle="",
                markerfacecolor="white",
                markeredgecolor="black",
                markersize=np.sqrt(ref_to_area(v)),
                label=f"{v:g}",
            )
        )

    ax.legend(
        handles=size_handles,
        title=size_label,
        loc="upper left",
        bbox_to_anchor=(1.22, 0.55),
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        labelspacing=1.2,
    )

    ax.set_title(
        f"{library_name} enrichment of {mi_name_for_out} "
        f"{CELL_SIDE_DISPLAY.get(cell_side, cell_side)} top genes",
        fontsize=12,
        pad=10,
    )

    plt.tight_layout()

    pdf_path = enrich_outdir / f"{output_prefix}_{safe_cell_side}_{safe_lib_name}_dotplot.pdf"
    svg_path = enrich_outdir / f"{output_prefix}_{safe_cell_side}_{safe_lib_name}_dotplot.svg"

    fig.savefig(pdf_path, format="pdf", bbox_inches="tight", transparent=True)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", transparent=True)
    plt.show()
    plt.close(fig)

    print("Saved plot:", flush=True)
    print(pdf_path, flush=True)
    print(svg_path, flush=True)
    print("Saved term order:", flush=True)
    print(term_order_path, flush=True)


# ============================================================
# Build top gene lists from current col_meta_df
# ============================================================

required_cols = {
    "Gene",
    "Direction",
    "max_abs_log2fc_cancer_type",
    "max_abs_log2fc_value",
}

missing_cols = required_cols - set(COL_META_DF.columns)
if len(missing_cols) > 0:
    raise ValueError(
        f"COL_META_DF is missing required columns: {missing_cols}. "
        "This cell expects col_meta_df from the current abs-log2FC heatmap code. "
        f"Required columns: {sorted(required_cols)}."
    )

meta_df = COL_META_DF.copy()
meta_df = meta_df.loc[meta_df["Direction"].astype(str).isin(DIRECTION_TO_CELL_SIDE.keys())].copy()
meta_df["Gene"] = meta_df["Gene"].astype(str)
meta_df["Direction"] = meta_df["Direction"].astype(str)
meta_df["CellSide"] = meta_df["Direction"].map(DIRECTION_TO_CELL_SIDE)
meta_df["CancerType"] = meta_df["max_abs_log2fc_cancer_type"].astype(str)
meta_df["max_abs_log2fc_value"] = pd.to_numeric(meta_df["max_abs_log2fc_value"], errors="coerce")

meta_df = meta_df.dropna(
    subset=["CancerType", "Gene", "Direction", "CellSide", "max_abs_log2fc_value"]
).copy()

meta_df = meta_df.loc[
    meta_df["Gene"].ne("")
    & meta_df["CancerType"].ne("Gap")
    & meta_df["Direction"].ne("Gap")
    & meta_df["CellSide"].notna()
].copy()

if meta_df.empty:
    raise ValueError("No valid Fibroblast/Tumor-cell top genes found in col_meta_df.")

cancer_type_order = infer_cancer_type_order(meta_df)
print("[Cancer type order used for enrichment and dotplots]")
print(cancer_type_order)

gene_list_records = []
gene_lists = {}

for cell_side in ["Fibroblast", "TumorCell"]:
    for cancer_type in cancer_type_order:
        sub_df = meta_df.loc[
            (meta_df["CancerType"].astype(str) == str(cancer_type))
            & (meta_df["CellSide"].astype(str) == str(cell_side))
        ].copy()

        if sub_df.empty:
            continue

        sub_df = sub_df.sort_values(by=["max_abs_log2fc_value"], ascending=False, kind="mergesort")
        genes = unique_preserve_order(sub_df["Gene"].tolist())
        gene_list_name = f"{cancer_type}_{cell_side}"

        gene_lists[gene_list_name] = {
            "CancerType": str(cancer_type),
            "CellSide": str(cell_side),
            "Direction": "Sending" if cell_side == "Fibroblast" else "Receiving",
            "GeneListName": gene_list_name,
            "Genes": genes,
        }

        for rank, gene in enumerate(genes, start=1):
            gene_list_records.append({
                "CancerType": str(cancer_type),
                "CellSide": str(cell_side),
                "Direction": "Sending" if cell_side == "Fibroblast" else "Receiving",
                "GeneListName": gene_list_name,
                "Gene": gene,
                "RankInGeneList": rank,
                "n_genes_in_list": len(genes),
            })

top_gene_list_df = pd.DataFrame(gene_list_records)
top_gene_list_df.to_csv(gene_list_path, index=False)

print("[Top gene lists by CancerType x CellSide]", flush=True)
if top_gene_list_df.empty:
    raise ValueError("No gene lists were constructed from col_meta_df.")

gene_list_summary_df = (
    top_gene_list_df
    .groupby(["CancerType", "CellSide", "Direction", "GeneListName"], as_index=False)
    .agg(n_genes=("Gene", "nunique"))
    .sort_values(["CellSide", "CancerType"])
)

display(gene_list_summary_df)
print(f"[Saved top gene list] {gene_list_path}", flush=True)

# ============================================================
# Resolve Enrichr libraries
# ============================================================

if DEBUG_SKIP_LIBRARY_RESOLUTION:
    ENRICHR_LIBRARIES = ENRICHR_LIBRARIES_REQUESTED
    print("[Info] Skipping Enrichr library resolution.", flush=True)
else:
    available_libraries = get_available_enrichr_libraries()
    ENRICHR_LIBRARIES = resolve_enrichr_libraries(
        ENRICHR_LIBRARIES_REQUESTED,
        available_libraries=available_libraries,
    )

print("\n[Enrichr libraries to use]", flush=True)
print(ENRICHR_LIBRARIES, flush=True)

# ============================================================
# Run GO / KEGG enrichment
# ============================================================

gene_list_items = list(gene_lists.items())
if DEBUG_FAST_RUN and DEBUG_N_GENE_LISTS is not None:
    gene_list_items = gene_list_items[:DEBUG_N_GENE_LISTS]
    print(f"[Debug] Running only first {len(gene_list_items)} gene list(s).", flush=True)

enrichr_results = []

for gene_list_name, info in gene_list_items:
    genes = info["Genes"]

    if len(genes) < MIN_GENES_FOR_ENRICHMENT:
        print(
            f"[Skip] {gene_list_name}: only {len(genes)} genes "
            f"(< MIN_GENES_FOR_ENRICHMENT={MIN_GENES_FOR_ENRICHMENT})",
            flush=True,
        )
        continue

    print("\n" + "=" * 60, flush=True)
    print(
        f"[Enrichment] {gene_list_name}: "
        f"{info['CancerType']} | {info['CellSide']} | {len(genes)} genes",
        flush=True,
    )
    print("=" * 60, flush=True)

    for lib in ENRICHR_LIBRARIES:
        print(f"  Running {lib} ...", flush=True)
        res = run_enrichr_with_retry(
            gene_list=genes,
            library_name=lib,
            gene_list_name=gene_list_name,
        )
        print(f"  Done {lib}; result rows = {res.shape[0]}", flush=True)

        if res.empty:
            continue

        res["CancerType"] = info["CancerType"]
        res["CellSide"] = info["CellSide"]
        res["Direction"] = info["Direction"]
        enrichr_results.append(res)

        if ENRICHR_SLEEP_SECONDS > 0:
            time.sleep(ENRICHR_SLEEP_SECONDS)

# ============================================================
# Standardize, attach reference genes, and save Enrichr results
# ============================================================

if len(enrichr_results) == 0:
    all_enrichr_results_df = pd.DataFrame()
    plot_df = pd.DataFrame()
    term_reference_wide_df = pd.DataFrame()
    term_reference_long_df = pd.DataFrame()
    term_reference_missing_df = pd.DataFrame()

    all_enrichr_results_df.to_csv(enrichr_result_path, index=False)
    plot_df.to_csv(plot_table_path, index=False)
    term_reference_wide_df.to_csv(term_reference_wide_path, index=False)
    term_reference_long_df.to_csv(term_reference_long_path, index=False)
    term_reference_missing_df.to_csv(term_reference_missing_path, index=False)

    print("\n[No Enrichr results returned]", flush=True)
    print("Saved empty result table:", enrichr_result_path, flush=True)
    print("Saved empty plot table:", plot_table_path, flush=True)
    print("Saved empty term-reference files:", term_reference_wide_path, term_reference_long_path, flush=True)

else:
    all_enrichr_results_df = pd.concat(enrichr_results, axis=0, ignore_index=True)

    if "Adjusted P-value" not in all_enrichr_results_df.columns:
        raise ValueError("Enrichr result does not contain 'Adjusted P-value' column.")

    all_enrichr_results_df["Adjusted P-value"] = pd.to_numeric(
        all_enrichr_results_df["Adjusted P-value"], errors="coerce"
    )
    all_enrichr_results_df["P-value"] = pd.to_numeric(
        all_enrichr_results_df.get("P-value", np.nan), errors="coerce"
    )
    all_enrichr_results_df["neglog10_adj_p"] = -np.log10(
        all_enrichr_results_df["Adjusted P-value"].clip(lower=1e-300)
    )

    overlap_parsed = all_enrichr_results_df["Overlap"].apply(parse_overlap)
    all_enrichr_results_df["overlap_n"] = [x[0] for x in overlap_parsed]
    all_enrichr_results_df["gene_set_size"] = [x[1] for x in overlap_parsed]
    all_enrichr_results_df["gene_ratio_input"] = (
        all_enrichr_results_df["overlap_n"] / all_enrichr_results_df["n_input_genes"]
    )
    all_enrichr_results_df["gene_ratio_set"] = (
        all_enrichr_results_df["overlap_n"] / all_enrichr_results_df["gene_set_size"]
    )

    all_enrichr_results_df = all_enrichr_results_df.sort_values(
        by=["Library", "CellSide", "CancerType", "Adjusted P-value", "P-value"],
        ascending=[True, True, True, True, True],
        kind="mergesort",
    )

    # Attach and save full reference genes for each enriched term.
    term_reference_wide_df, term_reference_long_df, term_reference_missing_df = (
        attach_reference_genes_to_enrichr_results(all_enrichr_results_df)
    )
    term_reference_wide_df.to_csv(term_reference_wide_path, index=False)
    term_reference_long_df.to_csv(term_reference_long_path, index=False)
    term_reference_missing_df.to_csv(term_reference_missing_path, index=False)

    all_enrichr_results_df.to_csv(enrichr_result_path, index=False)

    print(f"\n[Saved Enrichr results] {enrichr_result_path}", flush=True)
    print(f"[Saved term reference genes wide] {term_reference_wide_path}", flush=True)
    print(f"[Saved term reference genes long] {term_reference_long_path}", flush=True)
    print(f"[Saved missing term-reference lookup table] {term_reference_missing_path}", flush=True)
    display(all_enrichr_results_df.head(20))
    display(term_reference_wide_df.head(20))

    # ========================================================
    # Build visualization table
    # ========================================================
    plot_source_df = all_enrichr_results_df.copy()
    plot_source_df = plot_source_df.dropna(subset=["Adjusted P-value", "Term"]).copy()
    plot_source_df_use = plot_source_df.loc[
        plot_source_df["Adjusted P-value"] <= PLOT_ADJ_PVALUE_CUTOFF
    ].copy()

    plot_rows = []
    for (library_name, cell_side, cancer_type), sub_df in plot_source_df_use.groupby(
        ["Library", "CellSide", "CancerType"], sort=False
    ):
        sub_df = sub_df.sort_values(
            by=["Adjusted P-value", "P-value"],
            ascending=[True, True],
            kind="mergesort",
        )
        sub_top = sub_df.head(TOP_TERMS_PER_CANCER_PER_LIBRARY).copy()
        plot_rows.append(sub_top)

    if len(plot_rows) == 0:
        plot_df = pd.DataFrame()
    else:
        plot_df = pd.concat(plot_rows, axis=0, ignore_index=True)

    plot_df.to_csv(plot_table_path, index=False)
    print(f"[Saved dotplot table] {plot_table_path}", flush=True)
    display(plot_df.head(20))

    # ========================================================
    # Draw separate dotplots by CellSide and Library
    # ========================================================
    if plot_df.empty:
        print("[Skip dotplots] plot_df is empty.", flush=True)
    else:
        output_prefix = f"{mi_name_for_out}_{sum_or_mean_tag}_{mode_tag}"
        for lib in ENRICHR_LIBRARIES:
            for cell_side in ["Fibroblast", "TumorCell"]:
                make_dotplot_for_library_and_cell_side(
                    plot_df=plot_df,
                    library_name=lib,
                    cell_side=cell_side,
                    cancer_type_order=cancer_type_order,
                    output_prefix=output_prefix,
                )

gc.collect()

print("\nDone: Fibroblast-side / Tumor-cell-side GO-KEGG enrichment with term reference genes.")
print("Top gene list:", gene_list_path)
print("Enrichr results:", enrichr_result_path)
print("Dotplot table:", plot_table_path)
print("Term reference genes wide:", term_reference_wide_path)
print("Term reference genes long:", term_reference_long_path)


In [ ]:
# ============================================================
# Alternative sender and receiver signed-log2FC analysis
# Recompute gene-expression log2FC for Fibroblast -> tumor MI-4
# ------------------------------------------------------------
# This cell reruns BOTH sides from raw edge-level MI strengths:
#
#   Sending side:
#       Fibroblast cells are split by aggregated outgoing
#       Fibroblast -> tumor MI-4 strength.
#
#   Receiving side:
#       Tumor cells are split by aggregated incoming
#       Fibroblast -> tumor MI-4 strength.
#
# Signed-log2FC display:
#   - NO abs(log2FC)
#   - NO z-score of log2FC
#   - Heatmap directly shows signed log2FC
#   - Gene selection uses max signed log2FC > 0.2
#
# Output:
#   sender_mi_gene_log2fc_df_rerun_signed
#   receiver_mi_gene_log2fc_df_rerun_signed
#   combined Sending + Receiving signed-log2FC heatmap PDF/SVG
# ============================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch, Rectangle

# ============================================================
# User settings
# ============================================================

MI_OI_RERUN = globals().get("MI_OI", "MI4")
MI_OI_INDEX_RERUN = int(str(MI_OI_RERUN).replace("MI", "")) - 1

FIBROBLAST_LABEL = "Fibroblast"
TUMOR_SUFFIX = "-cancercell"

# Cell-level MI aggregation:
#   True  = sum MI-4 strengths per cell
#   False = mean MI-4 strengths per cell
CELL_MI_IF_SUM_RERUN = bool(globals().get("CELL_MI_IF_SUM", True))

# Sending side:
#   keep Fibroblast cells with at least one outgoing Fibroblast -> tumor edge.
# Receiving side:
#   keep tumor cells with at least one incoming Fibroblast -> tumor edge.
KEEP_ONLY_FIBROBLAST_NEIGHBOR_CELLS = True

# Split cells into MI-4-high / MI-4-low within each cancer type and side.
# "rank_half" is robust to many tied MI scores.
MEDIAN_SPLIT_MODE_RERUN = "rank_half"  # "rank_half" or "threshold"

LFC_PSEUDOCOUNT = 1e-6

# Expression source.
# Set to a layer name if needed, e.g. "log1p_norm".
EXPRESSION_LAYER_RERUN = globals().get("EXPRESSION_LAYER", None)

# Top DEG-like columns to show.
# IMPORTANT:
#   This is based on max SIGNED log2FC, not abs(log2FC).
LOG2FC_SHOW_THRESHOLD_RERUN = 0.20
MAX_COLUMNS_PER_CANCER_TYPE = 50

# Direct signed log2FC display range.
# Values outside this range will be clipped only for visualization.
LOG2FC_VMIN = -1.0
LOG2FC_VCENTER = 0.0
LOG2FC_VMAX = 1.0

if not (LOG2FC_VMIN < LOG2FC_VCENTER < LOG2FC_VMAX):
    raise ValueError(
        f"For TwoSlopeNorm, need LOG2FC_VMIN < LOG2FC_VCENTER < LOG2FC_VMAX, "
        f"but got LOG2FC_VMIN={LOG2FC_VMIN}, "
        f"LOG2FC_VCENTER={LOG2FC_VCENTER}, LOG2FC_VMAX={LOG2FC_VMAX}."
    )

SHOW_GENE_LABELS = False
MAX_GENE_LABELS_TO_SHOW = 100

ADD_GAP_BETWEEN_TOPGENE_BLOCKS = True
GAP_SIZE = 2
GAP_COLOR = "#FFFFFF"

# Top annotation colors
SENDING_COLOR = "#7dd2ec"
RECEIVING_COLOR = "#dc7b7f"

CANCER_CMAP_NAME = "tab20"

# Optional cancer-type display order.
if "CANCERTYPE_ORDER" in globals():
    CANCERTYPE_ORDER_RERUN = list(CANCERTYPE_ORDER)
else:
    CANCERTYPE_ORDER_RERUN = [
        "Breast",
        "Colon",
        "Liver",
        "Lung",
        "Melanoma",
        "Ovarian",
        "Prostate",
        "Uterine",
    ]

# Chunk size for expression aggregation.
EXPRESSION_CHUNK_SIZE = 20000

# ============================================================
# Output paths
# ============================================================

sum_or_mean_tag = "sumMI" if CELL_MI_IF_SUM_RERUN else "meanMI"

rerun_lfc_outdir = Path(run_dirs["run_dir"]) / f"{MI_OI_RERUN}_senderReceiver_lfcHM_rerun_signedLog2FC"
rerun_lfc_outdir.mkdir(parents=True, exist_ok=True)

threshold_tag = str(LOG2FC_SHOW_THRESHOLD_RERUN).replace(".", "p")
topn_tag = f"top{MAX_COLUMNS_PER_CANCER_TYPE}"
gap_tag = f"gap{GAP_SIZE}" if ADD_GAP_BETWEEN_TOPGENE_BLOCKS else "nogap"

rerun_prefix = (
    f"{MI_OI_RERUN}_senderReceiver_{sum_or_mean_tag}_"
    f"signedLFC_gt{threshold_tag}_{topn_tag}_{gap_tag}"
)

sender_log2fc_csv_path = rerun_lfc_outdir / f"{rerun_prefix}_sender_log2fc_full.csv"
receiver_log2fc_csv_path = rerun_lfc_outdir / f"{rerun_prefix}_receiver_log2fc_full.csv"
combined_log2fc_csv_path = rerun_lfc_outdir / f"{rerun_prefix}_combined_orig_signed_log2fc.csv"
combined_log2fc_gap_csv_path = rerun_lfc_outdir / f"{rerun_prefix}_combined_signed_log2fc_gap.csv"

cellmeta_csv_path = rerun_lfc_outdir / f"{rerun_prefix}_cellmeta.csv"
group_summary_csv_path = rerun_lfc_outdir / f"{rerun_prefix}_group_summary.csv"
lfc_stats_csv_path = rerun_lfc_outdir / f"{rerun_prefix}_lfc_stats.csv"

col_meta_path = rerun_lfc_outdir / f"{rerun_prefix}_colmeta.csv"
col_meta_with_gaps_path = rerun_lfc_outdir / f"{rerun_prefix}_colmeta_gap.csv"
all_col_meta_path = rerun_lfc_outdir / f"{rerun_prefix}_allcol.csv"
threshold_meta_path = rerun_lfc_outdir / f"{rerun_prefix}_threshold.csv"
row_order_path = rerun_lfc_outdir / f"{rerun_prefix}_row.csv"
cancer_color_map_path = rerun_lfc_outdir / f"{rerun_prefix}_colors.csv"

pdf_path = rerun_lfc_outdir / f"{rerun_prefix}.pdf"
svg_path = rerun_lfc_outdir / f"{rerun_prefix}.svg"

print("Sender + Receiver output directory:")
print(rerun_lfc_outdir)

# ============================================================
# Illustrator-friendly plotting settings
# ============================================================

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "Arial"

# ============================================================
# Helper functions
# ============================================================

def _to_numpy_rerun(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def _get_field_rerun(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    return obj[key]


def _has_field_rerun(obj, key):
    if isinstance(obj, dict):
        return key in obj
    if hasattr(obj, key):
        return True
    try:
        obj[key]
        return True
    except Exception:
        return False


def _edge_index_to_numpy_rerun(edge_index_obj):
    edge_index = _to_numpy_rerun(edge_index_obj).astype(np.int64, copy=False)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(
        f"edge_index should have shape [E, 2] or [2, E], got {edge_index.shape}"
    )


def _edge_count_from_data_rerun(data_obj):
    edge_index = _edge_index_to_numpy_rerun(_get_field_rerun(data_obj, "edge_index"))
    return int(edge_index.shape[0])


def _get_processed_adata_and_data_lists_rerun():
    if "processed" in globals() and processed is not None:
        data_list = processed.spidernet_data

        if hasattr(processed, "adata_list"):
            adata_list_use = processed.adata_list
        elif "adata_list" in globals():
            adata_list_use = adata_list
        else:
            raise NameError("`processed.adata_list` is not available and global `adata_list` was not found.")

        return adata_list_use, data_list

    if "adata_list" in globals() and "spidernet_data" in globals():
        return adata_list, spidernet_data

    raise NameError("Cannot find `processed` or (`adata_list` and `spidernet_data`) in memory.")


def _get_celltypes_rerun(adata, data_obj):
    if adata is not None and hasattr(adata, "obs") and "celltype_final" in adata.obs.columns:
        return adata.obs["celltype_final"].astype(str).to_numpy()

    if _has_field_rerun(data_obj, "cell_class"):
        return _to_numpy_rerun(_get_field_rerun(data_obj, "cell_class")).astype(str)

    raise KeyError(
        "Cannot find celltype labels. Expected adata.obs['celltype_final'] "
        "or data_obj['cell_class']."
    )


def _get_sample_id_rerun(adata, sample_index):
    sample_col_candidates = [
        "SampleID", "sample_id", "sample", "samples", "Sample",
        "slice", "slide", "library_id"
    ]

    if adata is not None and hasattr(adata, "obs"):
        for col in sample_col_candidates:
            if col in adata.obs.columns:
                vals = pd.Series(adata.obs[col]).dropna().astype(str).unique()
                if len(vals) > 0:
                    return str(vals[0])

    return f"sample_{sample_index}"


def _extract_cancer_type_from_tumor_celltype_rerun(celltype):
    s = str(celltype)
    if TUMOR_SUFFIX not in s:
        return None
    return s.replace(TUMOR_SUFFIX, "").strip(" -_")


def _infer_sample_cancer_type_from_celltypes(celltypes):
    tumor_ct = [
        _extract_cancer_type_from_tumor_celltype_rerun(x)
        for x in celltypes
        if str(x).endswith(TUMOR_SUFFIX)
    ]
    tumor_ct = [x for x in tumor_ct if x is not None]

    if len(tumor_ct) == 0:
        return None

    return pd.Series(tumor_ct).value_counts().idxmax()


def _assign_sender_cancer_type_from_edges(src_idx, receiver_ct, fibro_to_tumor_edge_mask, fallback_cancer_type=None):
    """
    Assign each Fibroblast sender cell to the cancer type of its outgoing tumor-cell receivers.
    If a Fibroblast connects to multiple tumor types, use the most frequent receiver cancer type.
    """
    edge_sender = src_idx[fibro_to_tumor_edge_mask]
    edge_receiver_ct = receiver_ct[fibro_to_tumor_edge_mask].astype(str)

    edge_cancer_type = np.asarray(
        [_extract_cancer_type_from_tumor_celltype_rerun(x) for x in edge_receiver_ct],
        dtype=object
    )

    valid = pd.notna(edge_cancer_type)

    if valid.sum() == 0:
        return {}

    tmp = pd.DataFrame({
        "sender_cell_index": edge_sender[valid].astype(int),
        "CancerType": edge_cancer_type[valid].astype(str),
    })

    sender_to_ct = (
        tmp
        .groupby("sender_cell_index")["CancerType"]
        .agg(lambda x: x.value_counts().idxmax())
        .to_dict()
    )

    return sender_to_ct


def _get_barcode_values_rerun(adata, cell_indices):
    barcode_candidates = ["barcode", "Barcode", "cell_id", "cell", "CellID"]

    for col in barcode_candidates:
        if col in adata.obs.columns:
            return adata.obs.iloc[cell_indices][col].astype(str).to_numpy()

    return adata.obs_names[cell_indices].astype(str).to_numpy()


def _get_expr_matrix_rerun(adata):
    if EXPRESSION_LAYER_RERUN is not None:
        if EXPRESSION_LAYER_RERUN not in adata.layers:
            raise KeyError(
                f"EXPRESSION_LAYER_RERUN='{EXPRESSION_LAYER_RERUN}' "
                f"not found in adata.layers."
            )
        return adata.layers[EXPRESSION_LAYER_RERUN]
    return adata.X


def _expr_rows_all_genes_to_numpy_rerun(adata, rows):
    X = _get_expr_matrix_rerun(adata)
    sub = X[rows, :]

    if sparse.issparse(sub):
        sub = sub.toarray()
    else:
        sub = np.asarray(sub)

    if sub.ndim == 1:
        sub = sub.reshape(1, -1)

    return sub.astype(np.float32, copy=False)


def _get_mi_edge_values_for_sample_rerun(factor_mmap, start, end, mi_index, n_total_edges):
    """
    Return MI edge vector for one sample from Factor_envir_use.
    Handles both edge x MI and MI x edge orientation.
    """
    if factor_mmap.shape[0] == n_total_edges:
        return np.asarray(factor_mmap[start:end, mi_index], dtype=np.float32)

    if factor_mmap.shape[1] == n_total_edges:
        return np.asarray(factor_mmap[mi_index, start:end], dtype=np.float32)

    raise ValueError(
        f"Factor_envir_use shape {factor_mmap.shape} is not aligned with total edges {n_total_edges}."
    )


def _assign_rank_half_groups_by_side_rerun(meta_df, score_col="MI_score"):
    """
    Within each MI side and cancer type:
      low  = lower half by cell-level MI score
      high = upper half by cell-level MI score.
    """
    meta_df = meta_df.copy()
    meta_df["MI_group"] = pd.NA
    meta_df["MI_threshold"] = np.nan

    for (mi_side, cancer_type), idx in meta_df.groupby(["MI_side", "CancerType"], sort=False).groups.items():
        idx = np.asarray(list(idx))
        vals = meta_df.loc[idx, score_col].to_numpy(dtype=float)
        finite = np.isfinite(vals)

        if finite.sum() < 2:
            continue

        if MEDIAN_SPLIT_MODE_RERUN == "threshold":
            threshold = float(np.nanmedian(vals[finite]))
            group = np.where(vals > threshold, "High", "Low")

            if len(np.unique(group[finite])) < 2:
                order = np.argsort(vals, kind="mergesort")
                ranks = np.empty_like(order)
                ranks[order] = np.arange(len(vals))
                group = np.where(ranks >= len(vals) // 2, "High", "Low")

        elif MEDIAN_SPLIT_MODE_RERUN == "rank_half":
            threshold = float(np.nanmedian(vals[finite]))
            order = np.argsort(vals, kind="mergesort")
            ranks = np.empty_like(order)
            ranks[order] = np.arange(len(vals))
            group = np.where(ranks >= len(vals) // 2, "High", "Low")

        else:
            raise ValueError("MEDIAN_SPLIT_MODE_RERUN must be 'rank_half' or 'threshold'.")

        meta_df.loc[idx, "MI_group"] = group
        meta_df.loc[idx, "MI_threshold"] = threshold

    meta_df = meta_df.loc[meta_df["MI_group"].isin(["High", "Low"])].copy()
    return meta_df


def _prepare_lfc_df_rerun(df, df_name):
    df = df.copy()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")

    if df.shape[0] < 1 or df.shape[1] < 1:
        raise ValueError(f"{df_name} has no valid rows or columns after removing all-NaN entries.")

    return df


def _build_mixed_top_positive_column_order(
    combined_lfc_df,
    threshold,
    max_columns_per_cancer_type=50,
):
    """
    Mixed Sending + Receiving top DEG-like gene selection using SIGNED log2FC.

    For each combined column, e.g. Sending|gene or Receiving|gene:
      1. compute max signed log2FC across cancer types
      2. find cancer type where signed log2FC is maximal
      3. keep if max signed log2FC > threshold
      4. within each argmax cancer type, keep top N columns by max signed log2FC
      5. sort by argmax cancer type row, max signed log2FC, and original position.

    Important:
      This does NOT use abs(log2FC).
      This selects genes upregulated in MI-4-high cells in at least one cancer type.
    """
    records = []

    for original_col_pos, combined_col in enumerate(combined_lfc_df.columns):
        vals = combined_lfc_df[combined_col].to_numpy(dtype=float)

        if "|" in str(combined_col):
            direction, gene = str(combined_col).split("|", 1)
        else:
            direction, gene = "Unknown", str(combined_col)

        if np.all(np.isnan(vals)):
            argmax_row_index = np.inf
            max_log2fc_value = np.nan
            max_log2fc_cancer_type = None
            pass_threshold = False
        else:
            argmax_row_index = int(np.nanargmax(vals))
            max_log2fc_value = float(vals[argmax_row_index])
            max_log2fc_cancer_type = combined_lfc_df.index[argmax_row_index]
            pass_threshold = bool(max_log2fc_value > threshold)

        records.append({
            "combined_column_id": str(combined_col),
            "Direction": direction,
            "Gene": gene,
            "argmax_row_index_no_row_clustering": argmax_row_index,
            "max_log2fc_value": max_log2fc_value,
            "max_log2fc_cancer_type": max_log2fc_cancer_type,
            "pass_log2fc_threshold": pass_threshold,
            "log2fc_show_threshold": threshold,
            "original_global_col_pos": original_col_pos,
        })

    all_col_meta_df = pd.DataFrame(records)

    threshold_passed_df = all_col_meta_df.loc[
        all_col_meta_df["pass_log2fc_threshold"].eq(True)
    ].copy()

    if threshold_passed_df.empty:
        raise ValueError(
            f"No Sending/Receiving genes passed "
            f"LOG2FC_SHOW_THRESHOLD_RERUN={threshold}. "
            "Please lower the threshold."
        )

    threshold_passed_sorted_df = threshold_passed_df.sort_values(
        by=[
            "argmax_row_index_no_row_clustering",
            "max_log2fc_value",
            "original_global_col_pos",
        ],
        ascending=[True, False, True],
        kind="mergesort",
    ).copy()

    if max_columns_per_cancer_type is not None:
        shown_col_meta_df = (
            threshold_passed_sorted_df
            .groupby(
                "argmax_row_index_no_row_clustering",
                sort=False,
                group_keys=False,
            )
            .head(max_columns_per_cancer_type)
            .copy()
        )
    else:
        shown_col_meta_df = threshold_passed_sorted_df.copy()

    shown_col_meta_df = shown_col_meta_df.sort_values(
        by=[
            "argmax_row_index_no_row_clustering",
            "max_log2fc_value",
            "original_global_col_pos",
        ],
        ascending=[True, False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    shown_col_meta_df["display_col_index"] = np.arange(shown_col_meta_df.shape[0])

    ordered_cols = shown_col_meta_df["combined_column_id"].tolist()

    return ordered_cols, shown_col_meta_df, all_col_meta_df, threshold_passed_sorted_df


def _make_discrete_color_map_rerun(labels, cmap_name="tab20"):
    labels = list(labels)
    n = len(labels)

    base_cmap = plt.get_cmap(cmap_name)

    if n <= 1:
        color_list = [base_cmap(0)]
    else:
        color_list = [base_cmap(i / max(n - 1, 1)) for i in range(n)]

    label_to_color = {
        label: color_list[i]
        for i, label in enumerate(labels)
    }

    color_df = pd.DataFrame({
        "CancerType": labels,
        "color_rgba": [str(label_to_color[label]) for label in labels],
    })

    return label_to_color, color_df


def _insert_gaps_between_column_blocks_rerun(
    data_df,
    col_meta_df,
    block_col="max_log2fc_cancer_type",
    gap_size=2,
    gap_prefix="__TOPGENE_BLOCK_GAP__",
):
    if gap_size is None or gap_size <= 0:
        meta_no_gap = col_meta_df.copy()
        meta_no_gap["is_gap"] = False
        meta_no_gap["display_col_index_with_gap"] = np.arange(meta_no_gap.shape[0])
        return data_df.copy(), meta_no_gap, []

    mat_parts = []
    col_names = []
    meta_rows = []
    gap_positions = []

    prev_block = None
    gap_counter = 0
    n_rows = data_df.shape[0]

    for _, row in col_meta_df.iterrows():
        cur_block = str(row[block_col])
        cur_col = row["combined_column_id"]

        if prev_block is not None and cur_block != prev_block:
            gap_start_pos = len(col_names)
            gap_positions.append(gap_start_pos)

            gap_mat = np.full((n_rows, gap_size), np.nan, dtype=float)
            mat_parts.append(gap_mat)

            for j in range(gap_size):
                gap_col = f"{gap_prefix}_{gap_counter}_{j}"
                col_names.append(gap_col)

                gap_row = {c: np.nan for c in col_meta_df.columns}
                gap_row.update({
                    "combined_column_id": gap_col,
                    "Direction": "Gap",
                    "Gene": "",
                    "max_log2fc_cancer_type": "Gap",
                    "is_gap": True,
                    "display_col_index_with_gap": len(col_names) - 1,
                    "gap_group_index": gap_counter,
                })
                meta_rows.append(gap_row)

            gap_counter += 1

        if cur_col not in data_df.columns:
            raise ValueError(f"Column {cur_col} from col_meta_df is not in data_df.")

        mat_parts.append(data_df[[cur_col]].to_numpy(dtype=float))
        col_names.append(cur_col)

        row_dict = row.to_dict()
        row_dict["is_gap"] = False
        row_dict["display_col_index_with_gap"] = len(col_names) - 1
        row_dict["gap_group_index"] = np.nan
        meta_rows.append(row_dict)

        prev_block = cur_block

    data_mat_with_gaps = np.concatenate(mat_parts, axis=1)

    data_df_with_gaps = pd.DataFrame(
        data_mat_with_gaps,
        index=data_df.index,
        columns=col_names,
    )

    col_meta_with_gaps = pd.DataFrame(meta_rows)

    return data_df_with_gaps, col_meta_with_gaps, gap_positions


def _draw_vector_annotation_bar_rerun(
    ax,
    labels,
    label_to_color,
    ylabel=None,
    gap_color="#FFFFFF",
):
    labels = [str(x) for x in labels]
    n = len(labels)

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.5, 0.5)

    for i, lab in enumerate(labels):
        color = label_to_color.get(lab, gap_color)

        rect = Rectangle(
            (i - 0.5, -0.5),
            1.0,
            1.0,
            facecolor=color,
            edgecolor="none",
            linewidth=0,
            antialiased=False,
        )
        ax.add_patch(rect)

    ax.set_xticks([])
    ax.set_yticks([])

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            rotation=0,
            ha="right",
            va="center",
            fontsize=8,
            labelpad=18,
        )

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_facecolor(gap_color)


# ============================================================
# Step 1. Reload data handles and Factor_envir_use
# ============================================================

adata_list_rerun, spidernet_data_list_rerun = _get_processed_adata_and_data_lists_rerun()

factor_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"

if not factor_path.exists():
    raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {factor_path}")

factor_mmap = np.load(factor_path, mmap_mode="r")

edge_counts = np.asarray(
    [
        _edge_count_from_data_rerun(spidernet_data_list_rerun[i])
        for i in range(len(spidernet_data_list_rerun))
    ],
    dtype=np.int64,
)

edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))
n_total_edges = int(edge_offsets[-1])

if factor_mmap.ndim != 2:
    raise ValueError(f"Factor_envir_use must be 2D, got shape {factor_mmap.shape}")

if not (factor_mmap.shape[0] == n_total_edges or factor_mmap.shape[1] == n_total_edges):
    raise ValueError(
        f"Factor_envir_use shape {factor_mmap.shape} is not aligned with "
        f"total edge count {n_total_edges}."
    )

n_mi_available = factor_mmap.shape[1] if factor_mmap.shape[0] == n_total_edges else factor_mmap.shape[0]

if MI_OI_INDEX_RERUN >= n_mi_available:
    raise ValueError(
        f"MI index {MI_OI_INDEX_RERUN} is out of range for Factor_envir_use "
        f"with {n_mi_available} MI dimensions."
    )

# Check common gene order.
var_names_ref = pd.Index([str(x) for x in adata_list_rerun[0].var_names])
n_genes = len(var_names_ref)

for sample_index, adata in enumerate(adata_list_rerun):
    var_names_cur = pd.Index([str(x) for x in adata.var_names])
    if not var_names_cur.equals(var_names_ref):
        raise ValueError(
            f"adata_list[{sample_index}].var_names differs from adata_list[0]. "
            "This rerun cell expects the same gene order across samples."
        )

# ============================================================
# Step 2. First pass: compute sender Fibroblast and receiver tumor-cell MI metadata
# ============================================================

cell_meta_frames = []

for sample_index, (adata, data_obj) in enumerate(zip(adata_list_rerun, spidernet_data_list_rerun)):
    if sample_index % 10 == 0:
        print(f"[Sender+Receiver first pass] sample {sample_index + 1}/{len(spidernet_data_list_rerun)}")

    edge_index = _edge_index_to_numpy_rerun(_get_field_rerun(data_obj, "edge_index"))
    celltypes = _get_celltypes_rerun(adata, data_obj)

    if edge_index.shape[0] != edge_counts[sample_index]:
        raise ValueError(
            f"Edge count mismatch for sample {sample_index}: "
            f"{edge_index.shape[0]} vs expected {edge_counts[sample_index]}"
        )

    start = int(edge_offsets[sample_index])
    end = int(edge_offsets[sample_index + 1])

    mi_edge_cur = _get_mi_edge_values_for_sample_rerun(
        factor_mmap=factor_mmap,
        start=start,
        end=end,
        mi_index=MI_OI_INDEX_RERUN,
        n_total_edges=n_total_edges,
    )

    src_idx = edge_index[:, 0]
    dst_idx = edge_index[:, 1]

    sender_ct = celltypes[src_idx].astype(str)
    receiver_ct = celltypes[dst_idx].astype(str)

    fibro_to_tumor_edge_mask = (
        (sender_ct == FIBROBLAST_LABEL)
        & np.char.endswith(receiver_ct.astype(str), TUMOR_SUFFIX)
    )

    if not np.any(fibro_to_tumor_edge_mask):
        continue

    n_cells = adata.n_obs
    sample_id = _get_sample_id_rerun(adata, sample_index)
    sample_cancer_type = _infer_sample_cancer_type_from_celltypes(celltypes)

    # -----------------------------
    # Receiver side: tumor cells
    # -----------------------------
    receiver_mi_sum = np.bincount(
        dst_idx[fibro_to_tumor_edge_mask],
        weights=mi_edge_cur[fibro_to_tumor_edge_mask],
        minlength=n_cells,
    ).astype(np.float32)

    receiver_edge_count = np.bincount(
        dst_idx[fibro_to_tumor_edge_mask],
        minlength=n_cells,
    ).astype(np.int32)

    if CELL_MI_IF_SUM_RERUN:
        receiver_mi_score = receiver_mi_sum
    else:
        receiver_mi_score = np.zeros(n_cells, dtype=np.float32)
        valid_count = receiver_edge_count > 0
        receiver_mi_score[valid_count] = (
            receiver_mi_sum[valid_count] / receiver_edge_count[valid_count]
        )

    tumor_cell_mask = np.char.endswith(celltypes.astype(str), TUMOR_SUFFIX)

    if KEEP_ONLY_FIBROBLAST_NEIGHBOR_CELLS:
        receiver_keep_mask = tumor_cell_mask & (receiver_edge_count > 0)
    else:
        receiver_keep_mask = tumor_cell_mask

    receiver_cell_idx = np.where(receiver_keep_mask)[0].astype(np.int64)

    if len(receiver_cell_idx) > 0:
        receiver_celltype_sel = celltypes[receiver_cell_idx].astype(str)
        receiver_cancer_type_sel = np.asarray(
            [_extract_cancer_type_from_tumor_celltype_rerun(x) for x in receiver_celltype_sel],
            dtype=object,
        )

        valid_ct = pd.notna(receiver_cancer_type_sel)

        receiver_cell_idx = receiver_cell_idx[valid_ct]
        receiver_celltype_sel = receiver_celltype_sel[valid_ct]
        receiver_cancer_type_sel = receiver_cancer_type_sel[valid_ct].astype(str)

        if len(receiver_cell_idx) > 0:
            receiver_barcode_sel = _get_barcode_values_rerun(adata, receiver_cell_idx)

            cell_meta_frames.append(pd.DataFrame({
                "MI_side": "Receiving",
                "sample_index": int(sample_index),
                "sample_id": sample_id,
                "barcode": receiver_barcode_sel,
                "cell_index": receiver_cell_idx,
                "celltype_final": receiver_celltype_sel,
                "CancerType": receiver_cancer_type_sel,
                "MI": MI_OI_RERUN,
                "MI_score": receiver_mi_score[receiver_cell_idx],
                "Fibroblast_to_tumor_edge_count": receiver_edge_count[receiver_cell_idx],
                "cell_mi_aggregation": sum_or_mean_tag,
            }))

    # -----------------------------
    # Sending side: Fibroblast cells
    # -----------------------------
    sender_mi_sum = np.bincount(
        src_idx[fibro_to_tumor_edge_mask],
        weights=mi_edge_cur[fibro_to_tumor_edge_mask],
        minlength=n_cells,
    ).astype(np.float32)

    sender_edge_count = np.bincount(
        src_idx[fibro_to_tumor_edge_mask],
        minlength=n_cells,
    ).astype(np.int32)

    if CELL_MI_IF_SUM_RERUN:
        sender_mi_score = sender_mi_sum
    else:
        sender_mi_score = np.zeros(n_cells, dtype=np.float32)
        valid_count = sender_edge_count > 0
        sender_mi_score[valid_count] = (
            sender_mi_sum[valid_count] / sender_edge_count[valid_count]
        )

    fibroblast_cell_mask = celltypes.astype(str) == FIBROBLAST_LABEL

    if KEEP_ONLY_FIBROBLAST_NEIGHBOR_CELLS:
        sender_keep_mask = fibroblast_cell_mask & (sender_edge_count > 0)
    else:
        sender_keep_mask = fibroblast_cell_mask

    sender_cell_idx = np.where(sender_keep_mask)[0].astype(np.int64)

    if len(sender_cell_idx) > 0:
        sender_to_ct = _assign_sender_cancer_type_from_edges(
            src_idx=src_idx,
            receiver_ct=receiver_ct,
            fibro_to_tumor_edge_mask=fibro_to_tumor_edge_mask,
            fallback_cancer_type=sample_cancer_type,
        )

        sender_cancer_type_sel = np.asarray(
            [
                sender_to_ct.get(int(x), sample_cancer_type)
                for x in sender_cell_idx
            ],
            dtype=object,
        )

        valid_ct = pd.notna(sender_cancer_type_sel)

        sender_cell_idx = sender_cell_idx[valid_ct]
        sender_cancer_type_sel = sender_cancer_type_sel[valid_ct].astype(str)

        if len(sender_cell_idx) > 0:
            sender_barcode_sel = _get_barcode_values_rerun(adata, sender_cell_idx)

            cell_meta_frames.append(pd.DataFrame({
                "MI_side": "Sending",
                "sample_index": int(sample_index),
                "sample_id": sample_id,
                "barcode": sender_barcode_sel,
                "cell_index": sender_cell_idx,
                "celltype_final": celltypes[sender_cell_idx].astype(str),
                "CancerType": sender_cancer_type_sel,
                "MI": MI_OI_RERUN,
                "MI_score": sender_mi_score[sender_cell_idx],
                "Fibroblast_to_tumor_edge_count": sender_edge_count[sender_cell_idx],
                "cell_mi_aggregation": sum_or_mean_tag,
            }))

if len(cell_meta_frames) == 0:
    raise ValueError("No Sending/Receiving cells were found for Fibroblast -> tumor MI-4 analysis.")

cell_meta_df = pd.concat(cell_meta_frames, axis=0, ignore_index=True)

cell_meta_df = _assign_rank_half_groups_by_side_rerun(
    cell_meta_df,
    score_col="MI_score",
)

cell_meta_df.to_csv(cellmeta_csv_path, index=False)

group_summary_df = (
    cell_meta_df
    .groupby(["MI_side", "CancerType", "MI_group"], observed=True)
    .agg(
        n_cells=("cell_index", "size"),
        n_samples=("sample_id", "nunique"),
        mean_MI=("MI_score", "mean"),
        median_MI=("MI_score", "median"),
        threshold=("MI_threshold", "median"),
    )
    .reset_index()
)

group_summary_df.to_csv(group_summary_csv_path, index=False)

print("Sender + Receiver MI group summary:")
display(group_summary_df)

# ============================================================
# Step 3. Second pass: aggregate expression sums by side x cancer type x high/low
# ============================================================

expr_sum = {}
expr_count = {}

for mi_side in cell_meta_df["MI_side"].astype(str).unique():
    for cancer_type in cell_meta_df.loc[cell_meta_df["MI_side"].astype(str) == mi_side, "CancerType"].astype(str).unique():
        for group in ["High", "Low"]:
            expr_sum[(mi_side, cancer_type, group)] = np.zeros(n_genes, dtype=np.float64)
            expr_count[(mi_side, cancer_type, group)] = np.zeros(n_genes, dtype=np.float64)

for sample_index, meta_sub in cell_meta_df.groupby("sample_index", sort=True):
    sample_index = int(sample_index)

    if sample_index % 10 == 0:
        print(f"[Sender+Receiver expression pass] sample {sample_index + 1}/{len(spidernet_data_list_rerun)}")

    adata = adata_list_rerun[sample_index]

    meta_sub = meta_sub.reset_index(drop=True)
    rows_all = meta_sub["cell_index"].to_numpy(dtype=np.int64)
    side_arr_all = meta_sub["MI_side"].astype(str).to_numpy()
    cancer_arr_all = meta_sub["CancerType"].astype(str).to_numpy()
    group_arr_all = meta_sub["MI_group"].astype(str).to_numpy()

    for start_row in range(0, len(rows_all), EXPRESSION_CHUNK_SIZE):
        end_row = min(start_row + EXPRESSION_CHUNK_SIZE, len(rows_all))

        rows_chunk = rows_all[start_row:end_row]
        side_chunk = side_arr_all[start_row:end_row]
        cancer_chunk = cancer_arr_all[start_row:end_row]
        group_chunk = group_arr_all[start_row:end_row]

        X_chunk = _expr_rows_all_genes_to_numpy_rerun(adata, rows_chunk)
        X_chunk = X_chunk.astype(np.float64, copy=False)

        for mi_side in np.unique(side_chunk):
            side_mask = side_chunk == mi_side

            for cancer_type in np.unique(cancer_chunk[side_mask]):
                cancer_mask = cancer_chunk == cancer_type

                for group in ["High", "Low"]:
                    mask = side_mask & cancer_mask & (group_chunk == group)

                    if mask.sum() == 0:
                        continue

                    X_sub = X_chunk[mask, :]
                    finite = np.isfinite(X_sub)

                    expr_sum[(mi_side, cancer_type, group)] += np.nansum(X_sub, axis=0)
                    expr_count[(mi_side, cancer_type, group)] += finite.sum(axis=0)

        del X_chunk
        gc.collect()

# ============================================================
# Step 4. Compute high-vs-low log2FC separately for Sending and Receiving
# ============================================================

lfc_by_side = {}
lfc_stat_rows = []

cancer_types_present = cell_meta_df["CancerType"].astype(str).unique().tolist()

cancer_type_order_use = [
    ct for ct in CANCERTYPE_ORDER_RERUN
    if ct in set(cancer_types_present)
]

for ct in sorted(cancer_types_present):
    if ct not in cancer_type_order_use:
        cancer_type_order_use.append(ct)

for mi_side in ["Sending", "Receiving"]:
    lfc_records = {}

    for cancer_type in cancer_type_order_use:
        high_count = expr_count.get((mi_side, cancer_type, "High"), np.zeros(n_genes))
        low_count = expr_count.get((mi_side, cancer_type, "Low"), np.zeros(n_genes))

        high_sum = expr_sum.get((mi_side, cancer_type, "High"), np.zeros(n_genes))
        low_sum = expr_sum.get((mi_side, cancer_type, "Low"), np.zeros(n_genes))

        mean_high = np.full(n_genes, np.nan, dtype=np.float64)
        mean_low = np.full(n_genes, np.nan, dtype=np.float64)

        valid_high = high_count > 0
        valid_low = low_count > 0

        mean_high[valid_high] = high_sum[valid_high] / high_count[valid_high]
        mean_low[valid_low] = low_sum[valid_low] / low_count[valid_low]

        valid_both = valid_high & valid_low

        log2fc = np.full(n_genes, np.nan, dtype=np.float64)
        log2fc[valid_both] = np.log2(
            (mean_high[valid_both] + LFC_PSEUDOCOUNT)
            / (mean_low[valid_both] + LFC_PSEUDOCOUNT)
        )

        lfc_records[cancer_type] = log2fc

        n_high_cells = int(
            cell_meta_df.loc[
                (cell_meta_df["MI_side"].astype(str) == str(mi_side))
                & (cell_meta_df["CancerType"].astype(str) == str(cancer_type))
                & (cell_meta_df["MI_group"].astype(str) == "High")
            ].shape[0]
        )

        n_low_cells = int(
            cell_meta_df.loc[
                (cell_meta_df["MI_side"].astype(str) == str(mi_side))
                & (cell_meta_df["CancerType"].astype(str) == str(cancer_type))
                & (cell_meta_df["MI_group"].astype(str) == "Low")
            ].shape[0]
        )

        lfc_stat_rows.append({
            "MI_side": mi_side,
            "CancerType": cancer_type,
            "n_high_cells": n_high_cells,
            "n_low_cells": n_low_cells,
            "n_genes_valid": int(np.isfinite(log2fc).sum()),
            "max_log2fc": float(np.nanmax(log2fc)) if np.isfinite(log2fc).any() else np.nan,
            "min_log2fc": float(np.nanmin(log2fc)) if np.isfinite(log2fc).any() else np.nan,
            "median_log2fc": float(np.nanmedian(log2fc)) if np.isfinite(log2fc).any() else np.nan,
        })

    lfc_df = pd.DataFrame(
        lfc_records,
        index=var_names_ref,
    ).T

    lfc_df = lfc_df.loc[cancer_type_order_use, :]
    lfc_by_side[mi_side] = lfc_df

sender_mi_gene_log2fc_df_rerun_signed = lfc_by_side["Sending"]
receiver_mi_gene_log2fc_df_rerun_signed = lfc_by_side["Receiving"]

sender_mi_gene_log2fc_df_rerun_signed.to_csv(sender_log2fc_csv_path)
receiver_mi_gene_log2fc_df_rerun_signed.to_csv(receiver_log2fc_csv_path)

lfc_stats_df = pd.DataFrame(lfc_stat_rows)
lfc_stats_df.to_csv(lfc_stats_csv_path, index=False)

print(f"Saved sender-side full signed log2FC matrix: {sender_log2fc_csv_path}")
print(f"Saved receiver-side full signed log2FC matrix: {receiver_log2fc_csv_path}")
print(f"Saved log2FC stats: {lfc_stats_csv_path}")
display(lfc_stats_df)

# ============================================================
# Step 5. Combine Sending and Receiving and select top positive-log2FC genes
# ============================================================

sending_df = _prepare_lfc_df_rerun(
    sender_mi_gene_log2fc_df_rerun_signed,
    "sender_mi_gene_log2fc_df_rerun_signed",
)

receiving_df = _prepare_lfc_df_rerun(
    receiver_mi_gene_log2fc_df_rerun_signed,
    "receiver_mi_gene_log2fc_df_rerun_signed",
)

# Use row union following cancer_type_order_use.
row_order_use = [
    ct for ct in cancer_type_order_use
    if (ct in sending_df.index) or (ct in receiving_df.index)
]

sending_df = sending_df.reindex(row_order_use)
receiving_df = receiving_df.reindex(row_order_use)

row_order_df = pd.DataFrame({
    "CancerType": row_order_use,
    "row_index": np.arange(len(row_order_use)),
})
row_order_df.to_csv(row_order_path, index=False)

sending_prefixed_df = sending_df.copy()
sending_prefixed_df.columns = [f"Sending|{str(g)}" for g in sending_prefixed_df.columns]

receiving_prefixed_df = receiving_df.copy()
receiving_prefixed_df.columns = [f"Receiving|{str(g)}" for g in receiving_prefixed_df.columns]

combined_lfc_df = pd.concat(
    [sending_prefixed_df, receiving_prefixed_df],
    axis=1,
)

combined_lfc_df = combined_lfc_df.dropna(axis=1, how="all")
combined_lfc_df.to_csv(combined_log2fc_csv_path)

ordered_cols, col_meta_df, all_col_meta_df, threshold_passed_df = (
    _build_mixed_top_positive_column_order(
        combined_lfc_df=combined_lfc_df,
        threshold=LOG2FC_SHOW_THRESHOLD_RERUN,
        max_columns_per_cancer_type=MAX_COLUMNS_PER_CANCER_TYPE,
    )
)

combined_plot_df_original = combined_lfc_df.loc[:, ordered_cols].copy()

col_meta_df.to_csv(col_meta_path, index=False)
all_col_meta_df.to_csv(all_col_meta_path, index=False)
threshold_passed_df.to_csv(threshold_meta_path, index=False)

n_send_shown = int((col_meta_df["Direction"] == "Sending").sum())
n_recv_shown = int((col_meta_df["Direction"] == "Receiving").sum())

print(
    f"[Sending+Receiving genes passing max signed log2FC threshold > "
    f"{LOG2FC_SHOW_THRESHOLD_RERUN}] "
    f"{threshold_passed_df.shape[0]} / {combined_lfc_df.shape[1]}"
)
print(
    f"[After keeping at most {MAX_COLUMNS_PER_CANCER_TYPE} columns per cancer type] "
    f"{combined_plot_df_original.shape[1]} columns shown"
)
print(f"  Sending shown: {n_send_shown}")
print(f"  Receiving shown: {n_recv_shown}")

print("\n[Shown columns per cancer type / signed-log2FC argmax row]")
display(
    col_meta_df
    .groupby("max_log2fc_cancer_type")
    .size()
    .rename("n_columns_shown")
    .to_frame()
)

if ADD_GAP_BETWEEN_TOPGENE_BLOCKS:
    combined_plot_df_log2fc_plot, col_meta_plot_df, gap_positions = (
        _insert_gaps_between_column_blocks_rerun(
            data_df=combined_plot_df_original,
            col_meta_df=col_meta_df,
            block_col="max_log2fc_cancer_type",
            gap_size=GAP_SIZE,
            gap_prefix="__SENDER_RECEIVER_TOPGENE_BLOCK_GAP__",
        )
    )
else:
    combined_plot_df_log2fc_plot = combined_plot_df_original.copy()
    col_meta_plot_df = col_meta_df.copy()
    col_meta_plot_df["is_gap"] = False
    col_meta_plot_df["display_col_index_with_gap"] = np.arange(col_meta_plot_df.shape[0])
    gap_positions = []

combined_plot_df_log2fc_plot.to_csv(combined_log2fc_gap_csv_path)
col_meta_plot_df.to_csv(col_meta_with_gaps_path, index=False)

print(f"[Gap columns inserted] {len(gap_positions) * GAP_SIZE}")
print(f"[Gap start positions] {gap_positions}")

# ============================================================
# Step 6. Sender + Receiver signed log2FC heatmap
# ============================================================

display_mat = combined_plot_df_log2fc_plot.to_numpy(dtype=float)

# Clip signed log2FC only for visualization.
display_mat_clipped = np.clip(display_mat, LOG2FC_VMIN, LOG2FC_VMAX)
display_mat_clipped[np.isnan(display_mat)] = np.nan

cancer_bar_labels = (
    col_meta_plot_df["max_log2fc_cancer_type"]
    .astype(str)
    .tolist()
)

cancer_type_order_for_colors = [
    ct for ct in row_order_use
    if ct in set(cancer_bar_labels)
]

for ct in cancer_bar_labels:
    if ct not in cancer_type_order_for_colors and ct != "Gap":
        cancer_type_order_for_colors.append(ct)

cancer_label_to_color, cancer_color_df = _make_discrete_color_map_rerun(
    cancer_type_order_for_colors,
    cmap_name=CANCER_CMAP_NAME,
)

cancer_color_df.to_csv(cancer_color_map_path, index=False)

signed_lfc_cmap = plt.get_cmap("RdBu_r").copy()
signed_lfc_cmap.set_bad(GAP_COLOR)

norm = TwoSlopeNorm(
    vmin=LOG2FC_VMIN,
    vcenter=LOG2FC_VCENTER,
    vmax=LOG2FC_VMAX,
)

masked_mat = np.ma.masked_invalid(display_mat_clipped)

n_rows, n_cols = combined_plot_df_log2fc_plot.shape

fig_width = max(9.0, min(24, 0.18 * n_cols + 5.0))
fig_height = max(5.0, min(11.0, 0.45 * n_rows + 3.8))

plt.close("all")
fig = plt.figure(figsize=(fig_width, fig_height))

gs = GridSpec(
    nrows=3,
    ncols=2,
    height_ratios=[0.22, 0.22, 6.0],
    width_ratios=[6.5, 0.45],
    hspace=0.04,
    wspace=0.06,
)

ax_top_cancer = fig.add_subplot(gs[0, 0])
ax_top_direction = fig.add_subplot(gs[1, 0])
ax_heatmap = fig.add_subplot(gs[2, 0])
ax_cbar = fig.add_subplot(gs[2, 1])

# -----------------------------
# Top annotation bar 1: top cancer type
# -----------------------------
cancer_bar_color_map = {"Gap": GAP_COLOR}
cancer_bar_color_map.update(cancer_label_to_color)

_draw_vector_annotation_bar_rerun(
    ax=ax_top_cancer,
    labels=cancer_bar_labels,
    label_to_color=cancer_bar_color_map,
    ylabel="Top\ncancer",
    gap_color=GAP_COLOR,
)

# -----------------------------
# Top annotation bar 2: MI side
# -----------------------------
direction_bar_labels = (
    col_meta_plot_df["Direction"]
    .astype(str)
    .tolist()
)

direction_bar_color_map = {
    "Gap": GAP_COLOR,
    "Sending": SENDING_COLOR,
    "Receiving": RECEIVING_COLOR,
}

_draw_vector_annotation_bar_rerun(
    ax=ax_top_direction,
    labels=direction_bar_labels,
    label_to_color=direction_bar_color_map,
    ylabel="MI\nside",
    gap_color=GAP_COLOR,
)

# -----------------------------
# Heatmap
# -----------------------------
im = ax_heatmap.imshow(
    masked_mat,
    aspect="auto",
    interpolation="nearest",
    cmap=signed_lfc_cmap,
    norm=norm,
    rasterized=False,
)

ax_heatmap.set_yticks(np.arange(n_rows))
ax_heatmap.set_yticklabels(
    combined_plot_df_log2fc_plot.index.tolist(),
    fontsize=10,
)

if SHOW_GENE_LABELS and n_cols <= MAX_GENE_LABELS_TO_SHOW:
    xlabels = [
        "" if bool(is_gap) else str(gene)
        for is_gap, gene in zip(
            col_meta_plot_df["is_gap"].tolist(),
            col_meta_plot_df["Gene"].tolist(),
        )
    ]
    ax_heatmap.set_xticks(np.arange(n_cols))
    ax_heatmap.set_xticklabels(
        xlabels,
        rotation=90,
        fontsize=7,
    )
else:
    ax_heatmap.set_xticks([])

ax_heatmap.set_xlabel(
    f"Top {MAX_COLUMNS_PER_CANCER_TYPE} Sending/Receiving genes per cancer type "
    f"with max signed log2FC > {LOG2FC_SHOW_THRESHOLD_RERUN}; "
    f"ordered by row of maximal signed log2FC",
    fontsize=12,
)

ax_heatmap.set_ylabel("Cancer type", fontsize=12)
ax_heatmap.tick_params(axis="both", length=0)

for spine in ax_heatmap.spines.values():
    spine.set_visible(False)

# -----------------------------
# Colorbar
# -----------------------------
cbar = fig.colorbar(
    im,
    cax=ax_cbar,
    extend="both",
)

cbar_ticks = np.linspace(LOG2FC_VMIN, LOG2FC_VMAX, 5)
cbar.set_ticks(cbar_ticks)
cbar.set_ticklabels([f"{x:.2g}" for x in cbar_ticks])
cbar.set_label("Signed log2FC\nMI-4-high vs MI-4-low", fontsize=11)
cbar.ax.tick_params(labelsize=9)
cbar.outline.set_visible(True)
cbar.outline.set_linewidth(0.6)

# -----------------------------
# Legends
# -----------------------------
direction_legend_handles = [
    Patch(facecolor=SENDING_COLOR, edgecolor="none", label="Sending"),
    Patch(facecolor=RECEIVING_COLOR, edgecolor="none", label="Receiving"),
]

direction_legend = ax_heatmap.legend(
    handles=direction_legend_handles,
    title="MI side",
    loc="upper left",
    bbox_to_anchor=(1.18, 1.02),
    frameon=False,
    fontsize=9,
    title_fontsize=10,
)

ax_heatmap.add_artist(direction_legend)

cancer_legend_handles = [
    Patch(
        facecolor=cancer_label_to_color[ct],
        edgecolor="none",
        label=ct,
    )
    for ct in cancer_type_order_for_colors
]

ax_heatmap.legend(
    handles=cancer_legend_handles,
    title="Top gene cancer type",
    loc="upper left",
    bbox_to_anchor=(1.18, 0.70),
    frameon=False,
    fontsize=8,
    title_fontsize=10,
)

fig.suptitle(
    f"{MI_OI_RERUN} Fibroblast→tumor median-split gene-expression pattern: "
    f"Sending + Receiving "
    f"(top {MAX_COLUMNS_PER_CANCER_TYPE} per cancer type; shown as signed log2FC)",
    fontsize=13,
    y=1.01,
)

fig.savefig(
    pdf_path,
    format="pdf",
    bbox_inches="tight",
    transparent=True,
)

fig.savefig(
    svg_path,
    format="svg",
    bbox_inches="tight",
    transparent=True,
)

plt.show()
plt.close(fig)

gc.collect()

print("Saved Sender + Receiver signed-log2FC rerun outputs:")
print(sender_log2fc_csv_path)
print(receiver_log2fc_csv_path)
print(combined_log2fc_csv_path)
print(combined_log2fc_gap_csv_path)
print(cellmeta_csv_path)
print(group_summary_csv_path)
print(lfc_stats_csv_path)
print(col_meta_path)
print(col_meta_with_gaps_path)
print(all_col_meta_path)
print(threshold_meta_path)
print(row_order_path)
print(cancer_color_map_path)
print(pdf_path)
print(svg_path)

print("\nFinal signed-log2FC matrix shape without gaps:")
print(combined_plot_df_original.shape)

print("\nFinal signed-log2FC matrix shape with gaps:")
print(combined_plot_df_log2fc_plot.shape)

print("\nShown top positive-log2FC metadata:")
display(col_meta_df.head(50))

print("\nAll candidate metadata:")
display(
    all_col_meta_df
    .sort_values("max_log2fc_value", ascending=False)
    .head(50)
)

print("\nCancer type color map:")
display(cancer_color_df)

# Optional cleanup of large memory map reference.
del factor_mmap
gc.collect()

### Additional GO enrichment of loading-selected regulators and targets

Select genes with normalized loading above 0.4, run Enrichr GO Biological
Process enrichment, and retain predefined MI-4-associated terms when present.
The implementation falls back to the ten most significant terms if none of the
predefined terms is returned. Enrichment results depend on the library version.


In [ ]:
import gseapy as gp

In [ ]:
MI_OI = "MI4"


In [ ]:
## Identify candidate regulator and target genes for knockout
loading_receiver_use = np.load(run_dirs['run_dir'] / "loading_receiver_use.npy")
loading_sender_use = np.load(run_dirs['run_dir'] / "loading_sender_use.npy")
loading_receiver_use_df = pd.DataFrame(loading_receiver_use, index=["MI" + str(i+1) for i in range(loading_receiver_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_sender_use_df = pd.DataFrame(loading_sender_use, index=["MI" + str(i+1) for i in range(loading_sender_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)
loading_receiver_use_df_norm = loading_receiver_use_df.div(loading_receiver_use_df_colsum, axis=1)
loading_sender_use_df_norm = loading_sender_use_df.div(loading_sender_use_df_colsum, axis=1)


In [ ]:
loading_receiver_use_df_norm_choose = loading_receiver_use_df_norm.loc[MI_OI]
loading_sender_use_df_norm_choose = loading_sender_use_df_norm.loc[MI_OI]
loading_receiver_use_df_norm_choose = loading_receiver_use_df_norm_choose.sort_values(ascending=False)
loading_sender_use_df_norm_choose = loading_sender_use_df_norm_choose.sort_values(ascending=False)

In [ ]:
##Get the top regulators and targets based on the loading
top_targetgene_MIOI = loading_receiver_use_df_norm_choose.index[loading_receiver_use_df_norm_choose > 0.4].tolist()
print(top_targetgene_MIOI)
top_regulatorgene_MIOI = loading_sender_use_df_norm_choose.index[loading_sender_use_df_norm_choose > 0.4].tolist()
print(top_regulatorgene_MIOI)

# Release loading matrices after extracting the top gene lists.
release_memory(
    "loading_receiver_use",
    "loading_sender_use",
    "loading_receiver_use_df",
    "loading_sender_use_df",
    "loading_receiver_use_df_colsum",
    "loading_sender_use_df_colsum",
    "loading_receiver_use_df_norm",
    "loading_sender_use_df_norm",
    "loading_receiver_use_df_norm_choose",
    "loading_sender_use_df_norm_choose",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


#### Sender regulators


In [ ]:
go_res = gp.enrichr(
    gene_list=top_regulatorgene_MIOI,
    gene_sets=['GO_Biological_Process_2021'],
    organism="human",
    outdir=str(PANCANCER_OUTPUT / "GO_MIOI_target"),
    cutoff=0.05 
)

df_filter = go_res.results[
    (go_res.results['Adjusted P-value'] < 0.05)]

print(df_filter)

In [ ]:
GO_term_choose = [
    # Fibroblast / CAF sender-side matrix-remodeling programs for MI-4
    "extracellular matrix organization (GO:0030198)",
    "extracellular structure organization (GO:0043062)",
    "external encapsulating structure organization (GO:0045229)",
    "collagen fibril organization (GO:0030199)",
    "collagen metabolic process (GO:0032963)",
    "cell-substrate adhesion (GO:0031589)",
    "wound healing (GO:0042060)",
    "angiogenesis (GO:0001525)",
]

# Keep biologically pre-specified MI-4 sender terms when they are present.
# If none of the exact GO term names are returned by Enrichr, keep the top
# enriched terms so that the downstream MI-4 correlation/logFC cells still run.
df_filter_terms = df_filter["Term"].astype(str)
GO_term_choose_present = [term for term in GO_term_choose if term in set(df_filter_terms)]
if len(GO_term_choose_present) > 0:
    df_filter = df_filter[df_filter["Term"].isin(GO_term_choose_present)].copy()
else:
    print(
        "Warning: none of the pre-specified MI-4 Fibroblast sender GO terms "
        "were found. Falling back to the top 10 enriched GO terms."
    )
    df_filter = df_filter.sort_values("Adjusted P-value", ascending=True).head(10).copy()

print(df_filter)
df_filter.to_csv(str(run_dirs["run_dir"] / "GO_enrichment_topregulator_MI4.csv"), index=False)

# Release sender GO enrichment objects after saving the selected table.
release_memory(
    "go_res",
    "df_filter",
    "df_filter_terms",
    "GO_term_choose",
    "GO_term_choose_present",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


#### Receiver targets


In [ ]:
go_res = gp.enrichr(
    gene_list=top_targetgene_MIOI,
    gene_sets=['GO_Biological_Process_2021'],
    organism="human",
    outdir=str(PANCANCER_OUTPUT / "GO_MIOI_target"), 
    cutoff=0.05
)

df_filter = go_res.results[
    (go_res.results['Adjusted P-value'] < 0.05)]

print(df_filter)

In [ ]:
GO_term_choose = [
    # Cancer-cell receiver-side invasive / ECM / hypoxia / angiogenesis programs for MI-4
    "cell migration (GO:0016477)",
    "epithelial to mesenchymal transition (GO:0001837)",
    "response to hypoxia (GO:0001666)",
    "angiogenesis (GO:0001525)",
    "cell-substrate adhesion (GO:0031589)",
    "extracellular matrix organization (GO:0030198)",
    "extracellular structure organization (GO:0043062)",
    "wound healing (GO:0042060)",
]

# Keep biologically pre-specified MI-4 receiver terms when they are present.
# If none of the exact GO term names are returned by Enrichr, keep the top
# enriched terms so that the downstream MI-4 correlation/logFC cells still run.
df_filter_terms = df_filter["Term"].astype(str)
GO_term_choose_present = [term for term in GO_term_choose if term in set(df_filter_terms)]
if len(GO_term_choose_present) > 0:
    df_filter = df_filter[df_filter["Term"].isin(GO_term_choose_present)].copy()
else:
    print(
        "Warning: none of the pre-specified MI-4 cancer-cell receiver GO terms "
        "were found. Falling back to the top 10 enriched GO terms."
    )
    df_filter = df_filter.sort_values("Adjusted P-value", ascending=True).head(10).copy()

print(df_filter)
df_filter.to_csv(str(run_dirs["run_dir"] / "GO_enrichment_toptarget_MI4.csv"), index=False)

# Release receiver GO enrichment objects after saving the selected table.
release_memory(
    "go_res",
    "df_filter",
    "df_filter_terms",
    "GO_term_choose",
    "GO_term_choose_present",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)


### Additional MI-4 versus GO-program expression analyses

For fibroblast-to-tumor edges, average edge MI-4 activity into sender and
receiver cell scores. Average the observed expression of the genes in each
selected GO term, then compute cancer-specific Pearson/Spearman associations
and median-split expression fold changes. Dot plots and heatmaps provide
alternative displays of these GO-program summaries.


In [ ]:
import gc
import os
import re
from collections import OrderedDict

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import pearsonr, spearmanr


MI_GO_OUTPUT_DIR = run_dirs["run_dir"] / "MI4_fibroblast_to_cancercell_GO_correlation"
MI_GO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

factor_path = run_dirs["run_dir"] / "Factor_envir_use.npy"
sender_go_path = run_dirs["run_dir"] / "GO_enrichment_topregulator_MI4.csv"
receiver_go_path = run_dirs["run_dir"] / "GO_enrichment_toptarget_MI4.csv"

if not factor_path.exists():
    raise FileNotFoundError(f"Could not find: {factor_path}")

if not sender_go_path.exists() or not receiver_go_path.exists():
    raise FileNotFoundError(
        "GO enrichment tables for MI4 were not found. "
        "Please run the GO enrichment cells above before this section."
    )


def go_result_to_gene_table(go_result_df):
    """Expand a GO enrichment result table into a GO-term-to-gene mapping."""
    gene_table = (
        go_result_df.loc[:, ["Term", "Genes"]]
        .assign(
            Gene=lambda d: (
                d["Genes"]
                .fillna("")
                .astype(str)
                .str.split(r"\s*;\s*")
            )
        )
        .explode("Gene")
        .rename(columns={"Term": "GO_Term"})
        .drop(columns=["Genes"])
    )
    gene_table["Gene"] = gene_table["Gene"].astype(str).str.strip()
    gene_table = gene_table[gene_table["Gene"] != ""].drop_duplicates(["Gene", "GO_Term"])
    return gene_table.reset_index(drop=True)


def keep_unique_genesets(go_gene_df):
    """Keep only one GO term for duplicated gene sets while preserving GO-term order."""
    keep_terms = []
    seen_gene_sets = set()

    for go_term in pd.unique(go_gene_df["GO_Term"]):
        genes_cur = (
            go_gene_df.loc[go_gene_df["GO_Term"] == go_term, "Gene"]
            .astype(str)
            .dropna()
            .tolist()
        )
        gene_set_cur = frozenset(g.strip() for g in genes_cur if str(g).strip() != "")
        if gene_set_cur not in seen_gene_sets:
            keep_terms.append(go_term)
            seen_gene_sets.add(gene_set_cur)

    return go_gene_df[go_gene_df["GO_Term"].isin(keep_terms)].copy().reset_index(drop=True)


def safe_corr(x, y):
    """Return both Pearson and Spearman correlations with NaN-safe guards."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    out = {
        "n_cells": int(x.size),
        "pearson_r": np.nan,
        "pearson_p": np.nan,
        "spearman_r": np.nan,
        "spearman_p": np.nan,
        "mean_mi": np.nan,
        "mean_go_expr": np.nan,
    }

    if x.size == 0:
        return out

    out["mean_mi"] = float(np.mean(x))
    out["mean_go_expr"] = float(np.mean(y))

    if x.size < 3:
        return out

    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return out

    try:
        out["pearson_r"], out["pearson_p"] = pearsonr(x, y)
    except Exception:
        pass

    try:
        out["spearman_r"], out["spearman_p"] = spearmanr(x, y)
    except Exception:
        pass

    return out


def ensure_edge_index_e_by_2(edge_index):
    """Convert edge_index to shape (E, 2) if needed."""
    edge_index = np.asarray(edge_index)
    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(np.int64, copy=False)

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def compute_cell_level_mi(mi_edge, edge_index, n_cells, valid_edge_mask=None):
    """
    Aggregate edge-level MI values into cell-level sending and receiving scores.

    If valid_edge_mask is provided, only edges with valid_edge_mask == True
    are used for the aggregation.
    """
    edge_index = np.asarray(edge_index)
    mi_edge = np.asarray(mi_edge, dtype=np.float32)

    if valid_edge_mask is None:
        valid_edge_mask = np.ones(edge_index.shape[0], dtype=bool)
    else:
        valid_edge_mask = np.asarray(valid_edge_mask, dtype=bool)

    src = edge_index[valid_edge_mask, 0]
    dst = edge_index[valid_edge_mask, 1]
    mi_edge_use = mi_edge[valid_edge_mask]

    send_sum = np.bincount(src, weights=mi_edge_use, minlength=n_cells).astype(np.float64, copy=False)
    send_cnt = np.bincount(src, minlength=n_cells).astype(np.float64, copy=False)
    recv_sum = np.bincount(dst, weights=mi_edge_use, minlength=n_cells).astype(np.float64, copy=False)
    recv_cnt = np.bincount(dst, minlength=n_cells).astype(np.float64, copy=False)

    send_score = np.full(n_cells, np.nan, dtype=np.float32)
    recv_score = np.full(n_cells, np.nan, dtype=np.float32)

    np.divide(send_sum, send_cnt, out=send_score, where=send_cnt > 0)
    np.divide(recv_sum, recv_cnt, out=recv_score, where=recv_cnt > 0)

    return send_score, recv_score


def compute_go_average_expression_from_adata(adata_obj, cell_idx, go_gene_table):
    """
    Compute per-cell mean expression for each GO program from the observed expression matrix.
    This matches the definition used in the in-silico perturbation section, but is applied to adata.X.
    """
    gene_names = np.asarray(adata_obj.var_names).astype(str)
    gene_index_map = {gene: idx for idx, gene in enumerate(gene_names)}
    expr_matrix = adata_obj.X
    go_avg = OrderedDict()

    for go_term in pd.unique(go_gene_table["GO_Term"]):
        genes_cur = (
            go_gene_table.loc[go_gene_table["GO_Term"] == go_term, "Gene"]
            .astype(str)
            .tolist()
        )
        gene_idx_cur = [gene_index_map[g] for g in genes_cur if g in gene_index_map]
        if len(gene_idx_cur) == 0:
            continue

        if sparse.issparse(expr_matrix):
            vals = np.asarray(expr_matrix[cell_idx][:, gene_idx_cur].mean(axis=1)).ravel()
        else:
            vals = np.asarray(expr_matrix[np.ix_(cell_idx, gene_idx_cur)].mean(axis=1)).ravel()

        go_avg[go_term] = vals.astype(np.float32, copy=False)

    return go_avg


def append_batch_values(store_dict, cancer_type, go_term, mi_values, expr_values):
    """Append one batch of paired MI and GO-expression values."""
    if cancer_type not in store_dict:
        store_dict[cancer_type] = {}

    if go_term not in store_dict[cancer_type]:
        store_dict[cancer_type][go_term] = {"mi": [], "expr": []}

    store_dict[cancer_type][go_term]["mi"].append(np.asarray(mi_values, dtype=np.float32))
    store_dict[cancer_type][go_term]["expr"].append(np.asarray(expr_values, dtype=np.float32))


def concatenate_store_arrays(store_entry):
    """Concatenate all batch arrays for one cancer type and one GO term."""
    mi_vals = (
        np.concatenate(store_entry["mi"])
        if len(store_entry["mi"]) > 0
        else np.array([], dtype=np.float32)
    )
    expr_vals = (
        np.concatenate(store_entry["expr"])
        if len(store_entry["expr"]) > 0
        else np.array([], dtype=np.float32)
    )
    return mi_vals, expr_vals


def summarize_correlation_store(store_dict, side_label):
    """Build a cancer-type-resolved correlation table with both Pearson and Spearman."""
    rows = []

    for cancer_type in store_dict:
        for go_term in store_dict[cancer_type]:
            mi_vals, expr_vals = concatenate_store_arrays(store_dict[cancer_type][go_term])
            corr_stats = safe_corr(mi_vals, expr_vals)
            rows.append(
                {
                    "Side": side_label,
                    "CancerType": cancer_type,
                    "GO_Term": go_term,
                    **corr_stats,
                }
            )

    return pd.DataFrame(rows)


sender_go_gene_table = keep_unique_genesets(
    go_result_to_gene_table(pd.read_csv(sender_go_path))
)
receiver_go_gene_table = keep_unique_genesets(
    go_result_to_gene_table(pd.read_csv(receiver_go_path))
)

sender_go_terms_order = list(pd.unique(sender_go_gene_table["GO_Term"]))
receiver_go_terms_order = list(pd.unique(receiver_go_gene_table["GO_Term"]))

mi_label = str(MI_OI)
mi_digits = re.findall(r"\d+", mi_label)
if len(mi_digits) == 0:
    raise ValueError(f"Could not parse MI index from MI_OI={MI_OI!r}")
mi_index = int(mi_digits[0]) - 1

batch_sample_ids = [
    str(np.asarray(processed.spidernet_data[i]["sample"]).astype(str)[0])
    for i in range(len(processed.spidernet_data))
]
batch_cancer_types = [sample_id.split("_")[0] for sample_id in batch_sample_ids]
cancer_type_order = list(dict.fromkeys(batch_cancer_types))

edge_counts = np.asarray(
    [
        ensure_edge_index_e_by_2(
            processed.spidernet_data[i]["edge_index"].cpu().numpy()
        ).shape[0]
        for i in range(len(processed.spidernet_data))
    ],
    dtype=np.int64,
)
edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))

Factor_envir_use_mmap = np.load(factor_path, mmap_mode="r")

sender_corr_store = OrderedDict()
receiver_corr_store = OrderedDict()
mi_go_batch_summary = []

for batch_idx in range(len(processed.adata_list)):
    adata_batch = processed.adata_list[batch_idx]
    cell_types_batch = adata_batch.obs[CELL_TYPE_COL].astype(str).to_numpy()
    cell_types_batch_str = cell_types_batch.astype(str)

    # Sender/receiver populations for the MI-4 axis of interest:
    #   Fibroblast --(MI-4)--> cancer cell
    fibro_idx = np.flatnonzero(cell_types_batch_str == "Fibroblast")
    tumor_idx = np.flatnonzero(np.char.endswith(cell_types_batch_str, "-cancercell"))

    sample_id_cur = batch_sample_ids[batch_idx]
    cancer_type_cur = batch_cancer_types[batch_idx]

    if fibro_idx.size == 0 or tumor_idx.size == 0:
        mi_go_batch_summary.append(
            {
                "BatchIndex": batch_idx,
                "SampleID": sample_id_cur,
                "CancerType": cancer_type_cur,
                "n_fibroblast_cells": int(fibro_idx.size),
                "n_tumor_cells": int(tumor_idx.size),
                "n_fibroblast_to_tumor_edges": 0,
                "status": "skipped_missing_fibroblast_or_tumor_cells",
            }
        )
        continue

    edge_index_cur = ensure_edge_index_e_by_2(
        processed.spidernet_data[batch_idx]["edge_index"].cpu().numpy()
    )
    start_idx = edge_offsets[batch_idx]
    end_idx = edge_offsets[batch_idx + 1]
    mi_edge_cur = np.asarray(Factor_envir_use_mmap[start_idx:end_idx, mi_index], dtype=np.float32)

    if mi_edge_cur.shape[0] != edge_index_cur.shape[0]:
        raise ValueError(
            f"Mismatch between edge counts and MI values for batch {batch_idx}: "
            f"{edge_index_cur.shape[0]} edges versus {mi_edge_cur.shape[0]} MI rows."
        )

    # Only keep Fibroblast -> cancer-cell edges when computing cell-level MI-4.
    src_idx = edge_index_cur[:, 0]
    dst_idx = edge_index_cur[:, 1]

    src_is_fibroblast = cell_types_batch_str[src_idx] == "Fibroblast"
    dst_is_tumor = np.char.endswith(cell_types_batch_str[dst_idx], "-cancercell")
    fibroblast_to_tumor_edge_mask = src_is_fibroblast & dst_is_tumor

    if not np.any(fibroblast_to_tumor_edge_mask):
        mi_go_batch_summary.append(
            {
                "BatchIndex": batch_idx,
                "SampleID": sample_id_cur,
                "CancerType": cancer_type_cur,
                "n_fibroblast_cells": int(fibro_idx.size),
                "n_tumor_cells": int(tumor_idx.size),
                "n_fibroblast_to_tumor_edges": 0,
                "status": "skipped_no_fibroblast_to_tumor_edges",
            }
        )
        continue

    sending_mi_cur, receiver_mi_cur = compute_cell_level_mi(
        mi_edge=mi_edge_cur,
        edge_index=edge_index_cur,
        n_cells=adata_batch.n_obs,
        valid_edge_mask=fibroblast_to_tumor_edge_mask,
    )

    # Sender-side values are evaluated in Fibroblast cells.
    # Receiver-side values are evaluated in cancer cells.
    sending_mi_fibroblast = sending_mi_cur[fibro_idx]
    receiver_mi_tumor = receiver_mi_cur[tumor_idx]

    sender_go_avg_cur = compute_go_average_expression_from_adata(
        adata_obj=adata_batch,
        cell_idx=fibro_idx,
        go_gene_table=sender_go_gene_table,
    )
    receiver_go_avg_cur = compute_go_average_expression_from_adata(
        adata_obj=adata_batch,
        cell_idx=tumor_idx,
        go_gene_table=receiver_go_gene_table,
    )

    for go_term, expr_vals in sender_go_avg_cur.items():
        valid_mask = np.isfinite(sending_mi_fibroblast) & np.isfinite(expr_vals)
        if np.any(valid_mask):
            append_batch_values(
                sender_corr_store,
                cancer_type_cur,
                go_term,
                sending_mi_fibroblast[valid_mask],
                expr_vals[valid_mask],
            )

    for go_term, expr_vals in receiver_go_avg_cur.items():
        valid_mask = np.isfinite(receiver_mi_tumor) & np.isfinite(expr_vals)
        if np.any(valid_mask):
            append_batch_values(
                receiver_corr_store,
                cancer_type_cur,
                go_term,
                receiver_mi_tumor[valid_mask],
                expr_vals[valid_mask],
            )

    mi_go_batch_summary.append(
        {
            "BatchIndex": batch_idx,
            "SampleID": sample_id_cur,
            "CancerType": cancer_type_cur,
            "n_fibroblast_cells": int(fibro_idx.size),
            "n_tumor_cells": int(tumor_idx.size),
            "n_fibroblast_to_tumor_edges": int(np.sum(fibroblast_to_tumor_edge_mask)),
            "status": "finished",
        }
    )

    release_memory(
        "adata_batch",
        "cell_types_batch",
        "cell_types_batch_str",
        "fibro_idx",
        "tumor_idx",
        "edge_index_cur",
        "mi_edge_cur",
        "src_idx",
        "dst_idx",
        "src_is_fibroblast",
        "dst_is_tumor",
        "fibroblast_to_tumor_edge_mask",
        "sending_mi_cur",
        "receiver_mi_cur",
        "sending_mi_fibroblast",
        "receiver_mi_tumor",
        "sender_go_avg_cur",
        "receiver_go_avg_cur",
        namespace=globals(),
        run_gc=True,
        clear_cuda=True,
    )

mi_go_batch_summary_df = pd.DataFrame(mi_go_batch_summary)
sender_corr_summary_df = summarize_correlation_store(sender_corr_store, "Sender")
receiver_corr_summary_df = summarize_correlation_store(receiver_corr_store, "Receiver")

mi_go_corr_summary_df = pd.concat(
    [sender_corr_summary_df, receiver_corr_summary_df],
    axis=0,
    ignore_index=True,
)

mi_go_batch_summary_df.to_csv(
    MI_GO_OUTPUT_DIR / f"{mi_label}_celllevel_go_batch_summary.csv",
    index=False,
)
mi_go_corr_summary_df.to_csv(
    MI_GO_OUTPUT_DIR / f"{mi_label}_celllevel_go_correlation_summary_per_cancertype.csv",
    index=False,
)

print(f"Saved batch summary to: {MI_GO_OUTPUT_DIR / f'{mi_label}_celllevel_go_batch_summary.csv'}")
print(
    "Saved cancer-type-resolved correlation summary to: "
    f"{MI_GO_OUTPUT_DIR / f'{mi_label}_celllevel_go_correlation_summary_per_cancertype.csv'}"
)

display(
    mi_go_corr_summary_df
    .sort_values(["Side", "GO_Term", "CancerType"])
    .reset_index(drop=True)
)


In [ ]:
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
    }
)


def wrap_go_label(go_term, width=38):
    """Wrap GO-term text and move the accession to a separate line."""
    go_term = str(go_term).strip()
    m = re.match(r"^(.*?)(\s*\(\s*GO:\d+\s*\))$", go_term)
    if m:
        main_text = m.group(1).strip()
        go_id = m.group(2).strip()
    else:
        main_text = go_term
        go_id = ""

    wrapped = "\n".join(textwrap.wrap(main_text, width=width))
    if go_id != "":
        wrapped = wrapped + "\n" + go_id
    return wrapped


def make_row_label(side, go_term):
    return f"{side} | {go_term}"


def rowwise_minmax_norm(x):
    """
    Row-wise min-max normalization.
    Returns values in [0, 1].
    If a row has constant values or <2 finite values, assign 1.0 to finite entries.
    """
    x = np.asarray(x, dtype=float)
    out = np.full_like(x, np.nan, dtype=float)

    finite_mask = np.isfinite(x)
    if finite_mask.sum() == 0:
        return out

    vals = x[finite_mask]
    vmin = np.min(vals)
    vmax = np.max(vals)

    if np.isclose(vmax, vmin):
        out[finite_mask] = 1.0
    else:
        out[finite_mask] = (vals - vmin) / (vmax - vmin)

    return out


def plot_mi_go_dotplot(
    mi_go_corr_summary_df,
    sender_go_terms_order,
    receiver_go_terms_order,
    mi_label,
    MI_GO_OUTPUT_DIR,
    corr_metric="spearman_r",
):
    """
    Dot plot:
    - color = raw correlation
    - size = row-wise min-max normalized correlation
    - columns sorted by mean normalized correlation
    """
    if corr_metric not in ["spearman_r", "pearson_r"]:
        raise ValueError("corr_metric must be 'spearman_r' or 'pearson_r'")

    plot_rows = []
    for _, row in mi_go_corr_summary_df.iterrows():
        plot_rows.append(
            {
                "Side": row["Side"],
                "CancerType": row["CancerType"],
                "GO_Term": row["GO_Term"],
                "RowLabel": make_row_label(row["Side"], row["GO_Term"]),
                "corr_value": row[corr_metric],
                "n_cells": row["n_cells"],
                "pearson_r": row["pearson_r"],
                "spearman_r": row["spearman_r"],
                "pearson_p": row["pearson_p"],
                "spearman_p": row["spearman_p"],
            }
        )

    mi_go_plot_summary_df = pd.DataFrame(plot_rows)

    plot_df = mi_go_plot_summary_df.copy()
    plot_df = plot_df[plot_df["CancerType"].astype(str) != "Pooled"].copy()
    plot_df = plot_df.dropna(subset=["corr_value"]).copy()

    row_order = (
        [make_row_label("Sender", go_term) for go_term in sender_go_terms_order]
        + [make_row_label("Receiver", go_term) for go_term in receiver_go_terms_order]
    )

    row_to_term = {
        make_row_label("Sender", go_term): go_term for go_term in sender_go_terms_order
    }
    row_to_term.update(
        {make_row_label("Receiver", go_term): go_term for go_term in receiver_go_terms_order}
    )

    plot_df = plot_df[plot_df["RowLabel"].isin(row_order)].copy()

    plot_df["corr_norm_row"] = np.nan
    for row_label in row_order:
        idx = plot_df["RowLabel"] == row_label
        if idx.any():
            plot_df.loc[idx, "corr_norm_row"] = rowwise_minmax_norm(
                plot_df.loc[idx, "corr_value"].to_numpy(dtype=float)
            )

    col_order_df = (
        plot_df.groupby("CancerType", as_index=False)["corr_norm_row"]
        .mean()
        .rename(columns={"corr_norm_row": "mean_normalized_corr"})
        .sort_values("mean_normalized_corr", ascending=False)
        .reset_index(drop=True)
    )
    cancer_type_order_sorted = col_order_df["CancerType"].tolist()

    plot_df["CancerType"] = pd.Categorical(
        plot_df["CancerType"],
        categories=cancer_type_order_sorted,
        ordered=True,
    )
    plot_df["RowLabel"] = pd.Categorical(
        plot_df["RowLabel"],
        categories=row_order,
        ordered=True,
    )
    plot_df = plot_df.sort_values(["RowLabel", "CancerType"]).reset_index(drop=True)

    x_pos = {ct: i for i, ct in enumerate(cancer_type_order_sorted)}
    y_pos = {row_label: i for i, row_label in enumerate(row_order)}

    plot_df["x"] = plot_df["CancerType"].map(x_pos).astype(float)
    plot_df["y"] = plot_df["RowLabel"].map(y_pos).astype(float)

    max_abs_corr = np.nanmax(np.abs(plot_df["corr_value"].to_numpy(dtype=float)))
    if not np.isfinite(max_abs_corr) or max_abs_corr == 0:
        max_abs_corr = 1.0

    norm = TwoSlopeNorm(vmin=-max_abs_corr, vcenter=0.0, vmax=max_abs_corr)

    size_min = 25
    size_max = 320
    plot_df["dot_size"] = size_min + plot_df["corr_norm_row"].fillna(0.0) * (size_max - size_min)

    fig_width = max(8.0, 1.1 * len(cancer_type_order_sorted) + 4.5)
    fig_height = max(6.0, 0.75 * len(row_order) + 2.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    sc = ax.scatter(
        plot_df["x"],
        plot_df["y"],
        s=plot_df["dot_size"],
        c=plot_df["corr_value"],
        cmap="coolwarm",
        norm=norm,
        edgecolors="black",
        linewidths=0.35,
    )

    ax.set_xticks(range(len(cancer_type_order_sorted)))
    ax.set_xticklabels(cancer_type_order_sorted, rotation=45, ha="right", fontsize=10)

    ax.set_yticks(range(len(row_order)))
    ax.set_yticklabels(
        [wrap_go_label(row_to_term[row_label]) for row_label in row_order],
        fontsize=9,
    )

    ax.set_ylim(len(row_order) - 0.5, -0.5)
    ax.set_xlim(-0.5, len(cancer_type_order_sorted) - 0.5)

    for x in np.arange(-0.5, len(cancer_type_order_sorted), 1):
        ax.axvline(x=x + 0.5, color="#E5E5E5", linewidth=0.7, zorder=0)
    for y in np.arange(-0.5, len(row_order), 1):
        ax.axhline(y=y + 0.5, color="#F0F0F0", linewidth=0.7, zorder=0)

    if len(sender_go_terms_order) > 0 and len(receiver_go_terms_order) > 0:
        ax.axhline(y=len(sender_go_terms_order) - 0.5, color="black", linewidth=1.0)

    ax.set_xlabel("Cancer type", fontsize=11)
    ax.set_ylabel("Sender / Receiver GO terms", fontsize=11)

    metric_title = "Spearman correlation" if corr_metric == "spearman_r" else "Pearson correlation"
    ax.set_title(
        f"{mi_label}: cell-level MI versus GO-program expression ({metric_title}, per cancer type)",
        fontsize=12,
        pad=12,
    )

    cbar = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("Correlation coefficient", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    legend_norm_vals = [0.25, 0.50, 0.75, 1.00]
    legend_handles = [
        plt.scatter(
            [],
            [],
            s=size_min + v * (size_max - size_min),
            facecolor="white",
            edgecolor="black",
            linewidth=0.5,
        )
        for v in legend_norm_vals
    ]
    legend_labels = [f"{v:.2f}" for v in legend_norm_vals]

    ax.legend(
        legend_handles,
        legend_labels,
        title="Row-normalized corr",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 0.35),
        borderaxespad=0.0,
    )

    fig.tight_layout()

    plot_base = MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_correlation_dotplot_rowNormSize_{corr_metric}"
    fig.savefig(f"{plot_base}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{plot_base}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    plot_df.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_correlation_dotplot_table_rowNorm_{corr_metric}.csv",
        index=False,
    )
    col_order_df.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_correlation_column_order_by_meanRowNorm_{corr_metric}.csv",
        index=False,
    )

    print(f"Saved dot plot to: {plot_base}.png and {plot_base}.pdf")
    print(
        "Saved row-normalized table to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_correlation_dotplot_table_rowNorm_{corr_metric}.csv'}"
    )
    print(
        "Saved column order to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_correlation_column_order_by_meanRowNorm_{corr_metric}.csv'}"
    )


# ----------------------------------------------------------
# Spearman plot
# ----------------------------------------------------------
plot_mi_go_dotplot(
    mi_go_corr_summary_df=mi_go_corr_summary_df,
    sender_go_terms_order=sender_go_terms_order,
    receiver_go_terms_order=receiver_go_terms_order,
    mi_label=mi_label,
    MI_GO_OUTPUT_DIR=MI_GO_OUTPUT_DIR,
    corr_metric="spearman_r",
)

# ----------------------------------------------------------
# Pearson plot
# ----------------------------------------------------------
plot_mi_go_dotplot(
    mi_go_corr_summary_df=mi_go_corr_summary_df,
    sender_go_terms_order=sender_go_terms_order,
    receiver_go_terms_order=receiver_go_terms_order,
    mi_label=mi_label,
    MI_GO_OUTPUT_DIR=MI_GO_OUTPUT_DIR,
    corr_metric="pearson_r",
)

In [ ]:
## Evaluated by log fold change
import gc
import os
import re
from collections import OrderedDict

import numpy as np
import pandas as pd
from scipy import sparse


MI_GO_OUTPUT_DIR = run_dirs["run_dir"] / "MI4_fibroblast_to_cancercell_GO_correlation"
MI_GO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

factor_path = run_dirs["run_dir"] / "Factor_envir_use.npy"
sender_go_path = run_dirs["run_dir"] / "GO_enrichment_topregulator_MI4.csv"
receiver_go_path = run_dirs["run_dir"] / "GO_enrichment_toptarget_MI4.csv"

if not factor_path.exists():
    raise FileNotFoundError(f"Could not find: {factor_path}")

if not sender_go_path.exists() or not receiver_go_path.exists():
    raise FileNotFoundError(
        "GO enrichment tables for MI4 were not found. "
        "Please run the GO enrichment cells above before this section."
    )


def go_result_to_gene_table(go_result_df):
    """Expand a GO enrichment result table into a GO-term-to-gene mapping."""
    gene_table = (
        go_result_df.loc[:, ["Term", "Genes"]]
        .assign(
            Gene=lambda d: (
                d["Genes"]
                .fillna("")
                .astype(str)
                .str.split(r"\s*;\s*")
            )
        )
        .explode("Gene")
        .rename(columns={"Term": "GO_Term"})
        .drop(columns=["Genes"])
    )
    gene_table["Gene"] = gene_table["Gene"].astype(str).str.strip()
    gene_table = gene_table[gene_table["Gene"] != ""].drop_duplicates(["Gene", "GO_Term"])
    return gene_table.reset_index(drop=True)


def keep_unique_genesets(go_gene_df):
    """Keep only one GO term for duplicated gene sets while preserving GO-term order."""
    keep_terms = []
    seen_gene_sets = set()

    for go_term in pd.unique(go_gene_df["GO_Term"]):
        genes_cur = (
            go_gene_df.loc[go_gene_df["GO_Term"] == go_term, "Gene"]
            .astype(str)
            .dropna()
            .tolist()
        )
        gene_set_cur = frozenset(g.strip() for g in genes_cur if str(g).strip() != "")
        if gene_set_cur not in seen_gene_sets:
            keep_terms.append(go_term)
            seen_gene_sets.add(gene_set_cur)

    return go_gene_df[go_gene_df["GO_Term"].isin(keep_terms)].copy().reset_index(drop=True)


def ensure_edge_index_e_by_2(edge_index):
    """Convert edge_index to shape (E, 2) if needed."""
    edge_index = np.asarray(edge_index)
    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(np.int64, copy=False)

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def compute_cell_level_mi(mi_edge, edge_index, n_cells, valid_edge_mask=None):
    """
    Aggregate edge-level MI values into cell-level sending and receiving scores.

    If valid_edge_mask is provided, only edges with valid_edge_mask == True
    are used for the aggregation.
    """
    edge_index = np.asarray(edge_index)
    mi_edge = np.asarray(mi_edge, dtype=np.float32)

    if valid_edge_mask is None:
        valid_edge_mask = np.ones(edge_index.shape[0], dtype=bool)
    else:
        valid_edge_mask = np.asarray(valid_edge_mask, dtype=bool)

    src = edge_index[valid_edge_mask, 0]
    dst = edge_index[valid_edge_mask, 1]
    mi_edge_use = mi_edge[valid_edge_mask]

    send_sum = np.bincount(src, weights=mi_edge_use, minlength=n_cells).astype(np.float64, copy=False)
    send_cnt = np.bincount(src, minlength=n_cells).astype(np.float64, copy=False)
    recv_sum = np.bincount(dst, weights=mi_edge_use, minlength=n_cells).astype(np.float64, copy=False)
    recv_cnt = np.bincount(dst, minlength=n_cells).astype(np.float64, copy=False)

    send_score = np.full(n_cells, np.nan, dtype=np.float32)
    recv_score = np.full(n_cells, np.nan, dtype=np.float32)

    np.divide(send_sum, send_cnt, out=send_score, where=send_cnt > 0)
    np.divide(recv_sum, recv_cnt, out=recv_score, where=recv_cnt > 0)

    return send_score, recv_score


def compute_go_average_expression_from_adata(adata_obj, cell_idx, go_gene_table):
    """
    Compute per-cell mean expression for each GO program from the observed expression matrix.
    """
    gene_names = np.asarray(adata_obj.var_names).astype(str)
    gene_index_map = {gene: idx for idx, gene in enumerate(gene_names)}
    expr_matrix = adata_obj.X
    go_avg = OrderedDict()

    for go_term in pd.unique(go_gene_table["GO_Term"]):
        genes_cur = (
            go_gene_table.loc[go_gene_table["GO_Term"] == go_term, "Gene"]
            .astype(str)
            .tolist()
        )
        gene_idx_cur = [gene_index_map[g] for g in genes_cur if g in gene_index_map]
        if len(gene_idx_cur) == 0:
            continue

        if sparse.issparse(expr_matrix):
            vals = np.asarray(expr_matrix[cell_idx][:, gene_idx_cur].mean(axis=1)).ravel()
        else:
            vals = np.asarray(expr_matrix[np.ix_(cell_idx, gene_idx_cur)].mean(axis=1)).ravel()

        go_avg[go_term] = vals.astype(np.float32, copy=False)

    return go_avg


def summarize_median_split_logfc(mi_values, expr_values, eps=1e-8):
    """
    Split cells into High/Low groups by the median of MI strength and compute:
        log((mean_high + eps) / (mean_low + eps))

    Cells equal to the median are included in the High group.
    """
    mi_values = np.asarray(mi_values, dtype=float)
    expr_values = np.asarray(expr_values, dtype=float)

    mask = np.isfinite(mi_values) & np.isfinite(expr_values)
    mi_values = mi_values[mask]
    expr_values = expr_values[mask]

    out = {
        "n_cells": int(mi_values.size),
        "n_high": 0,
        "n_low": 0,
        "median_mi": np.nan,
        "mean_mi_high": np.nan,
        "mean_mi_low": np.nan,
        "mean_expr_high": np.nan,
        "mean_expr_low": np.nan,
        "log_fc_high_vs_low": np.nan,
    }

    if mi_values.size == 0:
        return out

    median_mi = np.median(mi_values)
    high_mask = mi_values >= median_mi
    low_mask = mi_values < median_mi

    out["median_mi"] = float(median_mi)
    out["n_high"] = int(np.sum(high_mask))
    out["n_low"] = int(np.sum(low_mask))

    if out["n_high"] > 0:
        out["mean_mi_high"] = float(np.mean(mi_values[high_mask]))
        out["mean_expr_high"] = float(np.mean(expr_values[high_mask]))

    if out["n_low"] > 0:
        out["mean_mi_low"] = float(np.mean(mi_values[low_mask]))
        out["mean_expr_low"] = float(np.mean(expr_values[low_mask]))

    if out["n_high"] > 0 and out["n_low"] > 0:
        out["log_fc_high_vs_low"] = float(
            np.log2((out["mean_expr_high"] + eps) / (out["mean_expr_low"] + eps))
        )

    return out


def summarize_effect_store(store_dict, side_label, eps=1e-8):
    """Build a cancer-type-resolved median-split logFC table."""
    rows = []

    for cancer_type in store_dict:
        for go_term in store_dict[cancer_type]:
            mi_vals = np.asarray(store_dict[cancer_type][go_term]["mi"], dtype=np.float32)
            expr_vals = np.asarray(store_dict[cancer_type][go_term]["expr"], dtype=np.float32)

            stats = summarize_median_split_logfc(mi_vals, expr_vals, eps=eps)
            rows.append(
                {
                    "Side": side_label,
                    "CancerType": cancer_type,
                    "GO_Term": go_term,
                    **stats,
                }
            )

    return pd.DataFrame(rows)


def append_batch_values(store_dict, cancer_type, go_term, mi_values, expr_values):
    """Append one batch of paired MI and GO-expression values."""
    if cancer_type not in store_dict:
        store_dict[cancer_type] = {}

    if go_term not in store_dict[cancer_type]:
        store_dict[cancer_type][go_term] = {"mi": [], "expr": []}

    store_dict[cancer_type][go_term]["mi"].extend(np.asarray(mi_values, dtype=np.float32).tolist())
    store_dict[cancer_type][go_term]["expr"].extend(np.asarray(expr_values, dtype=np.float32).tolist())


sender_go_gene_table = keep_unique_genesets(
    go_result_to_gene_table(pd.read_csv(sender_go_path))
)
receiver_go_gene_table = keep_unique_genesets(
    go_result_to_gene_table(pd.read_csv(receiver_go_path))
)

sender_go_terms_order = list(pd.unique(sender_go_gene_table["GO_Term"]))
receiver_go_terms_order = list(pd.unique(receiver_go_gene_table["GO_Term"]))

mi_label = str(MI_OI)
mi_digits = re.findall(r"\d+", mi_label)
if len(mi_digits) == 0:
    raise ValueError(f"Could not parse MI index from MI_OI={MI_OI!r}")
mi_index = int(mi_digits[0]) - 1

batch_sample_ids = [
    str(np.asarray(processed.spidernet_data[i]["sample"]).astype(str)[0])
    for i in range(len(processed.spidernet_data))
]
batch_cancer_types = [sample_id.split("_")[0] for sample_id in batch_sample_ids]
cancer_type_order = list(dict.fromkeys(batch_cancer_types))

edge_counts = np.asarray(
    [
        ensure_edge_index_e_by_2(
            processed.spidernet_data[i]["edge_index"].cpu().numpy()
        ).shape[0]
        for i in range(len(processed.spidernet_data))
    ],
    dtype=np.int64,
)
edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))

Factor_envir_use_mmap = np.load(factor_path, mmap_mode="r")

sender_effect_store = OrderedDict()
receiver_effect_store = OrderedDict()
mi_go_batch_summary = []

LOGFC_EPS = 1e-8

for batch_idx in range(len(processed.adata_list)):
    adata_batch = processed.adata_list[batch_idx]
    cell_types_batch = adata_batch.obs[CELL_TYPE_COL].astype(str).to_numpy()
    cell_types_batch_str = cell_types_batch.astype(str)

    # Sender/receiver populations for the MI-4 axis of interest:
    #   Fibroblast --(MI-4)--> cancer cell
    fibro_idx = np.flatnonzero(cell_types_batch_str == "Fibroblast")
    tumor_idx = np.flatnonzero(np.char.endswith(cell_types_batch_str, "-cancercell"))

    sample_id_cur = batch_sample_ids[batch_idx]
    cancer_type_cur = batch_cancer_types[batch_idx]

    if fibro_idx.size == 0 or tumor_idx.size == 0:
        mi_go_batch_summary.append(
            {
                "BatchIndex": batch_idx,
                "SampleID": sample_id_cur,
                "CancerType": cancer_type_cur,
                "n_fibroblast_cells": int(fibro_idx.size),
                "n_tumor_cells": int(tumor_idx.size),
                "n_fibroblast_to_tumor_edges": 0,
                "status": "skipped_missing_fibroblast_or_tumor_cells",
            }
        )
        continue

    edge_index_cur = ensure_edge_index_e_by_2(
        processed.spidernet_data[batch_idx]["edge_index"].cpu().numpy()
    )
    start_idx = edge_offsets[batch_idx]
    end_idx = edge_offsets[batch_idx + 1]
    mi_edge_cur = np.asarray(Factor_envir_use_mmap[start_idx:end_idx, mi_index], dtype=np.float32)

    if mi_edge_cur.shape[0] != edge_index_cur.shape[0]:
        raise ValueError(
            f"Mismatch between edge counts and MI values for batch {batch_idx}: "
            f"{edge_index_cur.shape[0]} edges versus {mi_edge_cur.shape[0]} MI rows."
        )

    # Only keep Fibroblast -> cancer-cell edges when computing cell-level MI-4.
    src_idx = edge_index_cur[:, 0]
    dst_idx = edge_index_cur[:, 1]

    src_is_fibroblast = cell_types_batch_str[src_idx] == "Fibroblast"
    dst_is_tumor = np.char.endswith(cell_types_batch_str[dst_idx], "-cancercell")
    fibroblast_to_tumor_edge_mask = src_is_fibroblast & dst_is_tumor

    if not np.any(fibroblast_to_tumor_edge_mask):
        mi_go_batch_summary.append(
            {
                "BatchIndex": batch_idx,
                "SampleID": sample_id_cur,
                "CancerType": cancer_type_cur,
                "n_fibroblast_cells": int(fibro_idx.size),
                "n_tumor_cells": int(tumor_idx.size),
                "n_fibroblast_to_tumor_edges": 0,
                "status": "skipped_no_fibroblast_to_tumor_edges",
            }
        )
        continue

    sending_mi_cur, receiver_mi_cur = compute_cell_level_mi(
        mi_edge=mi_edge_cur,
        edge_index=edge_index_cur,
        n_cells=adata_batch.n_obs,
        valid_edge_mask=fibroblast_to_tumor_edge_mask,
    )

    # Sender-side values are evaluated in Fibroblast cells.
    # Receiver-side values are evaluated in cancer cells.
    sending_mi_fibroblast = sending_mi_cur[fibro_idx]
    receiver_mi_tumor = receiver_mi_cur[tumor_idx]

    sender_go_avg_cur = compute_go_average_expression_from_adata(
        adata_obj=adata_batch,
        cell_idx=fibro_idx,
        go_gene_table=sender_go_gene_table,
    )
    receiver_go_avg_cur = compute_go_average_expression_from_adata(
        adata_obj=adata_batch,
        cell_idx=tumor_idx,
        go_gene_table=receiver_go_gene_table,
    )

    for go_term, expr_vals in sender_go_avg_cur.items():
        valid_mask = np.isfinite(sending_mi_fibroblast) & np.isfinite(expr_vals)
        if np.any(valid_mask):
            append_batch_values(
                sender_effect_store,
                cancer_type_cur,
                go_term,
                sending_mi_fibroblast[valid_mask],
                expr_vals[valid_mask],
            )

    for go_term, expr_vals in receiver_go_avg_cur.items():
        valid_mask = np.isfinite(receiver_mi_tumor) & np.isfinite(expr_vals)
        if np.any(valid_mask):
            append_batch_values(
                receiver_effect_store,
                cancer_type_cur,
                go_term,
                receiver_mi_tumor[valid_mask],
                expr_vals[valid_mask],
            )

    mi_go_batch_summary.append(
        {
            "BatchIndex": batch_idx,
            "SampleID": sample_id_cur,
            "CancerType": cancer_type_cur,
            "n_fibroblast_cells": int(fibro_idx.size),
            "n_tumor_cells": int(tumor_idx.size),
            "n_fibroblast_to_tumor_edges": int(np.sum(fibroblast_to_tumor_edge_mask)),
            "status": "finished",
        }
    )

    release_memory(
        "adata_batch",
        "cell_types_batch",
        "cell_types_batch_str",
        "fibro_idx",
        "tumor_idx",
        "edge_index_cur",
        "mi_edge_cur",
        "src_idx",
        "dst_idx",
        "src_is_fibroblast",
        "dst_is_tumor",
        "fibroblast_to_tumor_edge_mask",
        "sending_mi_cur",
        "receiver_mi_cur",
        "sending_mi_fibroblast",
        "receiver_mi_tumor",
        "sender_go_avg_cur",
        "receiver_go_avg_cur",
        namespace=globals(),
        run_gc=True,
        clear_cuda=True,
    )

mi_go_batch_summary_df = pd.DataFrame(mi_go_batch_summary)
sender_effect_summary_df = summarize_effect_store(
    sender_effect_store,
    side_label="Sender",
    eps=LOGFC_EPS,
)
receiver_effect_summary_df = summarize_effect_store(
    receiver_effect_store,
    side_label="Receiver",
    eps=LOGFC_EPS,
)

mi_go_effect_summary_df = pd.concat(
    [sender_effect_summary_df, receiver_effect_summary_df],
    axis=0,
    ignore_index=True,
)

mi_go_batch_summary_df.to_csv(
    MI_GO_OUTPUT_DIR / f"{mi_label}_celllevel_go_batch_summary.csv",
    index=False,
)
mi_go_effect_summary_df.to_csv(
    MI_GO_OUTPUT_DIR / f"{mi_label}_celllevel_go_medianSplit_logFC_summary_per_cancertype.csv",
    index=False,
)

print(f"Saved batch summary to: {MI_GO_OUTPUT_DIR / f'{mi_label}_celllevel_go_batch_summary.csv'}")
print(
    "Saved cancer-type-resolved median-split logFC summary to: "
    f"{MI_GO_OUTPUT_DIR / f'{mi_label}_celllevel_go_medianSplit_logFC_summary_per_cancertype.csv'}"
)

display(
    mi_go_effect_summary_df
    .sort_values(["Side", "GO_Term", "CancerType"])
    .reset_index(drop=True)
)


In [ ]:
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
    }
)


def wrap_go_label(go_term, width=38):
    """Wrap GO-term text and move the accession to a separate line."""
    go_term = str(go_term).strip()
    m = re.match(r"^(.*?)(\s*\(\s*GO:\d+\s*\))$", go_term)
    if m:
        main_text = m.group(1).strip()
        go_id = m.group(2).strip()
    else:
        main_text = go_term
        go_id = ""

    wrapped = "\n".join(textwrap.wrap(main_text, width=width))
    if go_id != "":
        wrapped = wrapped + "\n" + go_id
    return wrapped


def make_row_label(side, go_term):
    return f"{side} | {go_term}"


def rowwise_minmax_norm(x):
    """
    Row-wise min-max normalization.
    Returns values in [0, 1].
    If a row has constant values or <2 finite values, assign 1.0 to finite entries.
    """
    x = np.asarray(x, dtype=float)
    out = np.full_like(x, np.nan, dtype=float)

    finite_mask = np.isfinite(x)
    if finite_mask.sum() == 0:
        return out

    vals = x[finite_mask]
    vmin = np.min(vals)
    vmax = np.max(vals)

    if np.isclose(vmax, vmin):
        out[finite_mask] = 1.0
    else:
        out[finite_mask] = (vals - vmin) / (vmax - vmin)

    return out


def plot_mi_go_logfc_dotplot(
    mi_go_effect_summary_df,
    sender_go_terms_order,
    receiver_go_terms_order,
    mi_label,
    MI_GO_OUTPUT_DIR,
):
    """
    Dot plot:
    - color = raw logFC
    - size = row-wise min-max normalized logFC
    - columns sorted by mean normalized logFC
    """
    plot_rows = []
    for _, row in mi_go_effect_summary_df.iterrows():
        plot_rows.append(
            {
                "Side": row["Side"],
                "CancerType": row["CancerType"],
                "GO_Term": row["GO_Term"],
                "RowLabel": make_row_label(row["Side"], row["GO_Term"]),
                "effect_value": row["log_fc_high_vs_low"],
                "n_cells": row["n_cells"],
                "n_high": row["n_high"],
                "n_low": row["n_low"],
                "mean_expr_high": row["mean_expr_high"],
                "mean_expr_low": row["mean_expr_low"],
            }
        )

    plot_summary_df = pd.DataFrame(plot_rows)
    plot_df = plot_summary_df.dropna(subset=["effect_value"]).copy()

    row_order = (
        [make_row_label("Sender", go_term) for go_term in sender_go_terms_order]
        + [make_row_label("Receiver", go_term) for go_term in receiver_go_terms_order]
    )

    row_to_term = {
        make_row_label("Sender", go_term): go_term for go_term in sender_go_terms_order
    }
    row_to_term.update(
        {make_row_label("Receiver", go_term): go_term for go_term in receiver_go_terms_order}
    )

    plot_df = plot_df[plot_df["RowLabel"].isin(row_order)].copy()

    plot_df["effect_norm_row"] = np.nan
    for row_label in row_order:
        idx = plot_df["RowLabel"] == row_label
        if idx.any():
            plot_df.loc[idx, "effect_norm_row"] = rowwise_minmax_norm(
                plot_df.loc[idx, "effect_value"].to_numpy(dtype=float)
            )

    col_order_df = (
        plot_df.groupby("CancerType", as_index=False)["effect_norm_row"]
        .mean()
        .rename(columns={"effect_norm_row": "mean_normalized_logFC"})
        .sort_values("mean_normalized_logFC", ascending=False)
        .reset_index(drop=True)
    )
    cancer_type_order_sorted = col_order_df["CancerType"].tolist()

    plot_df["CancerType"] = pd.Categorical(
        plot_df["CancerType"],
        categories=cancer_type_order_sorted,
        ordered=True,
    )
    plot_df["RowLabel"] = pd.Categorical(
        plot_df["RowLabel"],
        categories=row_order,
        ordered=True,
    )
    plot_df = plot_df.sort_values(["RowLabel", "CancerType"]).reset_index(drop=True)

    x_pos = {ct: i for i, ct in enumerate(cancer_type_order_sorted)}
    y_pos = {row_label: i for i, row_label in enumerate(row_order)}

    plot_df["x"] = plot_df["CancerType"].map(x_pos).astype(float)
    plot_df["y"] = plot_df["RowLabel"].map(y_pos).astype(float)

    max_abs_effect = np.nanmax(np.abs(plot_df["effect_value"].to_numpy(dtype=float)))
    if not np.isfinite(max_abs_effect) or max_abs_effect == 0:
        max_abs_effect = 1.0

    norm = TwoSlopeNorm(vmin=-max_abs_effect, vcenter=0.0, vmax=max_abs_effect)

    size_min = 25
    size_max = 320
    plot_df["dot_size"] = size_min + plot_df["effect_norm_row"].fillna(0.0) * (size_max - size_min)

    fig_width = max(8.0, 1.1 * len(cancer_type_order_sorted) + 4.5)
    fig_height = max(6.0, 0.75 * len(row_order) + 2.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    sc = ax.scatter(
        plot_df["x"],
        plot_df["y"],
        s=plot_df["dot_size"],
        c=plot_df["effect_value"],
        cmap="coolwarm",
        norm=norm,
        edgecolors="black",
        linewidths=0.35,
    )

    ax.set_xticks(range(len(cancer_type_order_sorted)))
    ax.set_xticklabels(cancer_type_order_sorted, rotation=45, ha="right", fontsize=10)

    ax.set_yticks(range(len(row_order)))
    ax.set_yticklabels(
        [wrap_go_label(row_to_term[row_label]) for row_label in row_order],
        fontsize=9,
    )

    ax.set_ylim(len(row_order) - 0.5, -0.5)
    ax.set_xlim(-0.5, len(cancer_type_order_sorted) - 0.5)

    for x in np.arange(-0.5, len(cancer_type_order_sorted), 1):
        ax.axvline(x=x + 0.5, color="#E5E5E5", linewidth=0.7, zorder=0)
    for y in np.arange(-0.5, len(row_order), 1):
        ax.axhline(y=y + 0.5, color="#F0F0F0", linewidth=0.7, zorder=0)

    if len(sender_go_terms_order) > 0 and len(receiver_go_terms_order) > 0:
        ax.axhline(y=len(sender_go_terms_order) - 0.5, color="black", linewidth=1.0)

    ax.set_xlabel("Cancer type", fontsize=11)
    ax.set_ylabel("Sender / Receiver GO terms", fontsize=11)
    ax.set_title(
        f"{mi_label}: High-versus-low MI groups and GO-program expression",
        fontsize=12,
        pad=12,
    )

    cbar = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("log(mean_high / mean_low)", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    legend_norm_vals = [0.25, 0.50, 0.75, 1.00]
    legend_handles = [
        plt.scatter(
            [],
            [],
            s=size_min + v * (size_max - size_min),
            facecolor="white",
            edgecolor="black",
            linewidth=0.5,
        )
        for v in legend_norm_vals
    ]
    legend_labels = [f"{v:.2f}" for v in legend_norm_vals]

    ax.legend(
        legend_handles,
        legend_labels,
        title="Row-normalized logFC",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 0.35),
        borderaxespad=0.0,
    )

    fig.tight_layout()

    plot_base = MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot"
    fig.savefig(f"{plot_base}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{plot_base}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    plot_df.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_table_rowNorm.csv",
        index=False,
    )
    col_order_df.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_column_order_by_meanRowNorm.csv",
        index=False,
    )

    print(f"Saved dot plot to: {plot_base}.png and {plot_base}.pdf")
    print(
        "Saved row-normalized table to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_table_rowNorm.csv'}"
    )
    print(
        "Saved column order to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_medianSplit_logFC_column_order_by_meanRowNorm.csv'}"
    )


plot_mi_go_logfc_dotplot(
    mi_go_effect_summary_df=mi_go_effect_summary_df,
    sender_go_terms_order=sender_go_terms_order,
    receiver_go_terms_order=receiver_go_terms_order,
    mi_label=mi_label,
    MI_GO_OUTPUT_DIR=MI_GO_OUTPUT_DIR,
)

In [ ]:
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
    }
)


def wrap_go_label(go_term, width=38):
    """Wrap GO-term text and move the accession to a separate line."""
    go_term = str(go_term).strip()
    m = re.match(r"^(.*?)(\s*\(\s*GO:\d+\s*\))$", go_term)
    if m:
        main_text = m.group(1).strip()
        go_id = m.group(2).strip()
    else:
        main_text = go_term
        go_id = ""

    wrapped = "\n".join(textwrap.wrap(main_text, width=width))
    if go_id != "":
        wrapped = wrapped + "\n" + go_id
    return wrapped


def make_row_label(side, go_term):
    return f"{side} | {go_term}"


def plot_mi_go_logfc_heatmap(
    mi_go_effect_summary_df,
    sender_go_terms_order,
    receiver_go_terms_order,
    mi_label,
    MI_GO_OUTPUT_DIR,
    show_values=True,
    value_fmt=".1f",
):
    """
    Heatmap:
    - rows = sender GO terms + receiver GO terms
    - columns sorted by column sum of raw logFC, descending
    - color = raw logFC
    - text in each cell = raw logFC
    """

    plot_rows = []
    for _, row in mi_go_effect_summary_df.iterrows():
        plot_rows.append(
            {
                "Side": row["Side"],
                "CancerType": row["CancerType"],
                "GO_Term": row["GO_Term"],
                "RowLabel": make_row_label(row["Side"], row["GO_Term"]),
                "effect_value": row["log_fc_high_vs_low"],
                "n_cells": row["n_cells"],
                "n_high": row["n_high"],
                "n_low": row["n_low"],
                "mean_expr_high": row["mean_expr_high"],
                "mean_expr_low": row["mean_expr_low"],
            }
        )

    plot_summary_df = pd.DataFrame(plot_rows)
    plot_df = plot_summary_df.dropna(subset=["effect_value"]).copy()

    row_order = (
        [make_row_label("Sender", go_term) for go_term in sender_go_terms_order]
        + [make_row_label("Receiver", go_term) for go_term in receiver_go_terms_order]
    )

    row_to_term = {
        make_row_label("Sender", go_term): go_term for go_term in sender_go_terms_order
    }
    row_to_term.update(
        {
            make_row_label("Receiver", go_term): go_term
            for go_term in receiver_go_terms_order
        }
    )

    plot_df = plot_df[plot_df["RowLabel"].isin(row_order)].copy()

    # Raw logFC matrix
    raw_matrix = (
        plot_df.pivot_table(
            index="RowLabel",
            columns="CancerType",
            values="effect_value",
            aggfunc="mean",
        )
        .reindex(index=row_order)
    )

    # Sort cancer types by column sum of raw logFC, descending
    col_order_df = (
        raw_matrix.sum(axis=0, skipna=True)
        .sort_values(ascending=False)
        .rename("column_sum_logFC")
        .reset_index()
        .rename(columns={"index": "CancerType"})
    )
    cancer_type_order_sorted = col_order_df["CancerType"].tolist()

    raw_matrix = raw_matrix.reindex(columns=cancer_type_order_sorted)

    heatmap_matrix = raw_matrix.copy()

    max_abs_effect = np.nanmax(np.abs(heatmap_matrix.to_numpy(dtype=float)))
    if not np.isfinite(max_abs_effect) or max_abs_effect == 0:
        max_abs_effect = 1.0

    norm = TwoSlopeNorm(vmin=-max_abs_effect, vcenter=0.0, vmax=max_abs_effect)

    cmap = mpl.cm.get_cmap("coolwarm").copy()
    cmap.set_bad("#F2F2F2")

    fig_width = max(8.5, 1.1 * len(cancer_type_order_sorted) + 5.0)
    fig_height = max(6.5, 0.58 * len(row_order) + 3.0)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    im = ax.imshow(
        heatmap_matrix.to_numpy(dtype=float),
        cmap=cmap,
        norm=norm,
        aspect="auto",
        interpolation="none",
    )

    # Ticks
    ax.set_xticks(np.arange(len(cancer_type_order_sorted)))
    ax.set_xticklabels(cancer_type_order_sorted, rotation=45, ha="right", fontsize=10)

    ax.set_yticks(np.arange(len(row_order)))
    ax.set_yticklabels(
        [wrap_go_label(row_to_term[row_label]) for row_label in row_order],
        fontsize=9,
    )

    # Gridlines
    ax.set_xticks(np.arange(-0.5, len(cancer_type_order_sorted), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
    ax.grid(which="minor", color="#EAEAEA", linestyle="-", linewidth=0.7)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Separate sender and receiver
    if len(sender_go_terms_order) > 0 and len(receiver_go_terms_order) > 0:
        ax.axhline(y=len(sender_go_terms_order) - 0.5, color="black", linewidth=1.0)

    ax.set_xlabel("Cancer type", fontsize=11)
    ax.set_ylabel("Sender / Receiver GO terms", fontsize=11)
    ax.set_title(
        f"{mi_label}: High-versus-low MI groups and GO-program expression",
        fontsize=12,
        pad=12,
    )

    # Add values in cells
    if show_values:
        matrix_for_text = heatmap_matrix.to_numpy(dtype=float)
        for i in range(matrix_for_text.shape[0]):
            for j in range(matrix_for_text.shape[1]):
                val = matrix_for_text[i, j]
                if np.isfinite(val):
                    # Use white text on strong colors, black otherwise
                    text_color = "white" if abs(val) >= 0.55 * max_abs_effect else "black"
                    ax.text(
                        j,
                        i,
                        format(val, value_fmt),
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color=text_color,
                    )

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("log(mean_high / mean_low)", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    fig.tight_layout()

    plot_base = MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_heatmap_colSumSorted"
    fig.savefig(f"{plot_base}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{plot_base}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    heatmap_matrix.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_heatmap_matrix_colSumSorted.csv"
    )
    col_order_df.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_column_order_by_colSum.csv",
        index=False,
    )

    print(f"Saved heatmap to: {plot_base}.png and {plot_base}.pdf")
    print(
        "Saved heatmap matrix to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_medianSplit_logFC_heatmap_matrix_colSumSorted.csv'}"
    )
    print(
        "Saved column order to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_medianSplit_logFC_column_order_by_colSum.csv'}"
    )


# ------------------------------------------------------------------
# Run
# ------------------------------------------------------------------
plot_mi_go_logfc_heatmap(
    mi_go_effect_summary_df=mi_go_effect_summary_df,
    sender_go_terms_order=sender_go_terms_order,
    receiver_go_terms_order=receiver_go_terms_order,
    mi_label=mi_label,
    MI_GO_OUTPUT_DIR=MI_GO_OUTPUT_DIR,
    show_values=True,
    value_fmt=".1f",
)

In [ ]:
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ------------------------------------------------------------------
# Global plotting style
# ------------------------------------------------------------------
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
    }
)


# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
def wrap_go_label(go_term, width=38):
    """Wrap GO-term text and move the accession to a separate line."""
    go_term = str(go_term).strip()
    m = re.match(r"^(.*?)(\s*\(\s*GO:\d+\s*\))$", go_term)
    if m:
        main_text = m.group(1).strip()
        go_id = m.group(2).strip()
    else:
        main_text = go_term
        go_id = ""

    wrapped = "\n".join(textwrap.wrap(main_text, width=width))
    if go_id != "":
        wrapped = wrapped + "\n" + go_id
    return wrapped


def make_row_label(side, go_term):
    return f"{side} | {go_term}"


def rowwise_max_normalize_abs(df):
    """
    Row-wise max normalization on absolute values.
    For each row:
        norm_ij = abs(x_ij) / max_j(abs(x_ij))
    If a row is all zeros or all NaNs, keep as NaN/0 safely.
    """
    abs_df = df.abs()
    row_max = abs_df.max(axis=1, skipna=True).replace(0, np.nan)
    norm_df = abs_df.div(row_max, axis=0)
    return norm_df


# ------------------------------------------------------------------
# Main plotting function
# ------------------------------------------------------------------
def plot_mi_go_logfc_dotplot(
    mi_go_effect_summary_df,
    sender_go_terms_order,
    receiver_go_terms_order,
    mi_label,
    MI_GO_OUTPUT_DIR,
    min_dot_size=20,
    max_dot_size=340,
    sort_columns_by="colsum",   # "colsum" or "none"
    show_grid=True,
):
    """
    Dot plot:
    - rows = sender GO terms + receiver GO terms
    - columns = cancer types
    - color = raw logFC
    - size = row-wise max-normalized abs(logFC)

    Notes:
    - Row-wise normalization is done within each GO-term row across cancer types.
    - Missing values are not plotted.
    """

    # --------------------------------------------------------------
    # 1. Build plotting dataframe
    # --------------------------------------------------------------
    plot_rows = []
    for _, row in mi_go_effect_summary_df.iterrows():
        plot_rows.append(
            {
                "Side": row["Side"],
                "CancerType": row["CancerType"],
                "GO_Term": row["GO_Term"],
                "RowLabel": make_row_label(row["Side"], row["GO_Term"]),
                "effect_value": row["log_fc_high_vs_low"],
                "n_cells": row["n_cells"],
                "n_high": row["n_high"],
                "n_low": row["n_low"],
                "mean_expr_high": row["mean_expr_high"],
                "mean_expr_low": row["mean_expr_low"],
            }
        )

    plot_summary_df = pd.DataFrame(plot_rows)
    plot_df = plot_summary_df.dropna(subset=["effect_value"]).copy()

    # --------------------------------------------------------------
    # 2. Define row order
    # --------------------------------------------------------------
    row_order = (
        [make_row_label("Sender", go_term) for go_term in sender_go_terms_order]
        + [make_row_label("Receiver", go_term) for go_term in receiver_go_terms_order]
    )

    row_to_term = {
        make_row_label("Sender", go_term): go_term for go_term in sender_go_terms_order
    }
    row_to_term.update(
        {
            make_row_label("Receiver", go_term): go_term
            for go_term in receiver_go_terms_order
        }
    )

    plot_df = plot_df[plot_df["RowLabel"].isin(row_order)].copy()

    # --------------------------------------------------------------
    # 3. Construct raw logFC matrix
    # --------------------------------------------------------------
    raw_matrix = (
        plot_df.pivot_table(
            index="RowLabel",
            columns="CancerType",
            values="effect_value",
            aggfunc="mean",
        )
        .reindex(index=row_order)
    )

    # --------------------------------------------------------------
    # 4. Sort cancer types
    # --------------------------------------------------------------
    if sort_columns_by == "colsum":
        col_order_df = (
            raw_matrix.sum(axis=0, skipna=True)
            .sort_values(ascending=False)
            .rename("column_sum_logFC")
            .reset_index()
            .rename(columns={"index": "CancerType"})
        )
        cancer_type_order_sorted = col_order_df["CancerType"].tolist()
    else:
        cancer_type_order_sorted = raw_matrix.columns.tolist()
        col_order_df = pd.DataFrame(
            {
                "CancerType": cancer_type_order_sorted,
                "column_sum_logFC": raw_matrix.sum(axis=0, skipna=True).reindex(cancer_type_order_sorted).values,
            }
        )

    raw_matrix = raw_matrix.reindex(columns=cancer_type_order_sorted)

    # --------------------------------------------------------------
    # 5. Row-wise max-normalized size matrix
    #    size is based on abs(logFC), because dot size cannot be negative
    # --------------------------------------------------------------
    size_matrix = rowwise_max_normalize_abs(raw_matrix)
    size_matrix = np.power(size_matrix,2)

    # --------------------------------------------------------------
    # 6. Prepare long-format dataframe for scatter plotting
    # --------------------------------------------------------------
    plot_long = (
        raw_matrix.stack(dropna=False)
        .rename("logFC")
        .reset_index()
        .rename(columns={"level_0": "RowLabel", "level_1": "CancerType"})
    )

    size_long = (
        size_matrix.stack(dropna=False)
        .rename("size_norm")
        .reset_index()
        .rename(columns={"level_0": "RowLabel", "level_1": "CancerType"})
    )

    plot_long = plot_long.merge(size_long, on=["RowLabel", "CancerType"], how="left")
    plot_long = plot_long.dropna(subset=["logFC"]).copy()

    # Map x/y positions
    x_map = {ct: i for i, ct in enumerate(cancer_type_order_sorted)}
    y_map = {rl: i for i, rl in enumerate(row_order)}

    plot_long["x"] = plot_long["CancerType"].map(x_map)
    plot_long["y"] = plot_long["RowLabel"].map(y_map)

    # Convert normalized size to actual marker size
    plot_long["dot_size"] = (
        min_dot_size
        + (max_dot_size - min_dot_size) * plot_long["size_norm"].fillna(0)
    )

    # --------------------------------------------------------------
    # 7. Colormap normalization
    # --------------------------------------------------------------
    max_abs_effect = np.nanmax(np.abs(raw_matrix.to_numpy(dtype=float)))
    if not np.isfinite(max_abs_effect) or max_abs_effect == 0:
        max_abs_effect = 1.0

    norm = TwoSlopeNorm(vmin=-max_abs_effect, vcenter=0.0, vmax=max_abs_effect)

    cmap = mpl.cm.get_cmap("coolwarm").copy()
    cmap.set_bad("#F2F2F2")

    # --------------------------------------------------------------
    # 8. Figure size
    # --------------------------------------------------------------
    fig_width = max(8.5, 1.0 * len(cancer_type_order_sorted) + 5.0)
    fig_height = max(6.5, 0.55 * len(row_order) + 3.0)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # --------------------------------------------------------------
    # 9. Scatter plot
    # --------------------------------------------------------------
    sc = ax.scatter(
        plot_long["x"],
        plot_long["y"],
        s=plot_long["dot_size"],
        c=plot_long["logFC"],
        cmap=cmap,
        norm=norm,
        edgecolors="none",
    )

    # --------------------------------------------------------------
    # 10. Axes / ticks
    # --------------------------------------------------------------
    ax.set_xticks(np.arange(len(cancer_type_order_sorted)))
    ax.set_xticklabels(cancer_type_order_sorted, rotation=45, ha="right", fontsize=10)

    ax.set_yticks(np.arange(len(row_order)))
    ax.set_yticklabels(
        [wrap_go_label(row_to_term[row_label]) for row_label in row_order],
        fontsize=9,
    )

    ax.set_xlim(-0.5, len(cancer_type_order_sorted) - 0.5)
    ax.set_ylim(len(row_order) - 0.5, -0.5)  # invert y-axis manually

    if show_grid:
        ax.set_xticks(np.arange(-0.5, len(cancer_type_order_sorted), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
        ax.grid(which="minor", color="#EAEAEA", linestyle="-", linewidth=0.7)
        ax.tick_params(which="minor", bottom=False, left=False)

    # Separate sender and receiver blocks
    if len(sender_go_terms_order) > 0 and len(receiver_go_terms_order) > 0:
        ax.axhline(y=len(sender_go_terms_order) - 0.5, color="black", linewidth=1.0)

    ax.set_xlabel("Cancer type", fontsize=11)
    ax.set_ylabel("Sender / Receiver GO terms", fontsize=11)
    ax.set_title(
        f"{mi_label}: High-versus-low MI groups and GO-program expression",
        fontsize=12,
        pad=12,
    )

    # --------------------------------------------------------------
    # 11. Colorbar
    # --------------------------------------------------------------
    cbar = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("log(mean_high / mean_low)", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    # --------------------------------------------------------------
    # 12. Size legend
    # --------------------------------------------------------------
    size_legend_vals = [0.25, 0.5, 0.75, 1.0]
    size_legend_handles = []
    for v in size_legend_vals:
        s = min_dot_size + (max_dot_size - min_dot_size) * v
        h = ax.scatter([], [], s=s, color="gray", edgecolors="none")
        size_legend_handles.append(h)

    legend = ax.legend(
        size_legend_handles,
        [f"{v:.2f}" for v in size_legend_vals],
        title="Row-wise\nmax-normalized\n|logFC|",
        scatterpoints=1,
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        loc="upper left",
        bbox_to_anchor=(1.12, 1.00),
        borderaxespad=0,
    )
    ax.add_artist(legend)

    fig.tight_layout()

    # --------------------------------------------------------------
    # 13. Save outputs
    # --------------------------------------------------------------
    plot_base = MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_rowMaxNorm"

    fig.savefig(f"{plot_base}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{plot_base}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    raw_matrix.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_colorMatrix.csv"
    )
    size_matrix.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_sizeMatrix_rowMaxNorm.csv"
    )
    col_order_df.to_csv(
        MI_GO_OUTPUT_DIR / f"{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_column_order_by_colSum.csv",
        index=False,
    )

    print(f"Saved dot plot to: {plot_base}.png and {plot_base}.pdf")
    print(
        "Saved color matrix to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_colorMatrix.csv'}"
    )
    print(
        "Saved size matrix to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_sizeMatrix_rowMaxNorm.csv'}"
    )
    print(
        "Saved column order to: "
        f"{MI_GO_OUTPUT_DIR / f'{mi_label}_sender_receiver_go_medianSplit_logFC_dotplot_column_order_by_colSum.csv'}"
    )


# ------------------------------------------------------------------
# Run
# ------------------------------------------------------------------
plot_mi_go_logfc_dotplot(
    mi_go_effect_summary_df=mi_go_effect_summary_df,
    sender_go_terms_order=sender_go_terms_order,
    receiver_go_terms_order=receiver_go_terms_order,
    mi_label=mi_label,
    MI_GO_OUTPUT_DIR=MI_GO_OUTPUT_DIR,
    min_dot_size=20,
    max_dot_size=340,
    sort_columns_by="colsum",
    show_grid=True,
)